# EXIST2021 · Cuaderno maestro de experimentos HMT/HMVT

**Compendio documental para tesis y manuscritos científicos.** Se han consolidado seis notebooks de origen, conservando una sola copia por implementación de código equivalente. Se retiraron los comentarios del código y las salidas embebidas para mantener el archivo ligero.

**Importante:** este cuaderno es un **archivo de experimentos alternativos**, no una secuencia para ejecutar de principio a fin. Cada implementación debe ejecutarse desde un kernel limpio y con sus datos de entrada; unas usan TRAIN/TEST, otras solo TRAIN y otras requieren insumos geográficos. No combine métricas de protocolos diferentes. Consulte `README.md`, `CATALOGO_EXPERIMENTOS.csv` y `RESULTADOS_HISTORICOS.md` antes de citar cifras.

## Resumen de consolidación

- Notebooks revisados: **6**.
- Celdas de código sustanciales encontradas: **100**.
- Implementaciones únicas tras deduplicación semántica (AST): **70**.
- Repeticiones eliminadas: **30**.
- Los nombres `HMT` y `HMVT` se preservaron como aparecen en los originales; **no se asumió que dos versiones con el mismo nombre tengan idéntica implementación**.
- Las tablas históricas, cuando existen, se documentan como evidencia de ejecuciones anteriores y no como resultados recalculados.

## Guía rápida de versiones

| Versión identificada | Experimentos que la mencionan |
|---|---|
| V1 | EXP-001, EXP-002, EXP-003, EXP-004, EXP-005, EXP-007, EXP-008, EXP-009, EXP-010 … (58 apariciones) |
| V2 | EXP-001, EXP-002, EXP-003, EXP-004, EXP-007, EXP-018, EXP-023, EXP-024, EXP-025 … (31 apariciones) |
| V3 | EXP-001, EXP-002, EXP-003, EXP-004, EXP-006, EXP-007, EXP-022, EXP-023, EXP-024 … (32 apariciones) |
| V4 | EXP-001, EXP-002, EXP-003, EXP-004, EXP-006, EXP-007, EXP-038, EXP-039, EXP-040 … (26 apariciones) |
| V5 | EXP-038, EXP-039, EXP-040, EXP-041, EXP-042, EXP-047 (6 apariciones) |
| V6 | EXP-006 (1 apariciones) |
| V6a | EXP-006, EXP-038, EXP-039, EXP-040, EXP-041, EXP-042, EXP-047 (7 apariciones) |
| V6b | EXP-006, EXP-038, EXP-039, EXP-040, EXP-041, EXP-042, EXP-047 (7 apariciones) |
| V6c | EXP-006, EXP-038, EXP-039, EXP-040, EXP-041, EXP-042, EXP-047 (7 apariciones) |
| V6d | EXP-006, EXP-038, EXP-039, EXP-040, EXP-041, EXP-042, EXP-047 (7 apariciones) |
| V7 | EXP-045 (1 apariciones) |
| V8 | EXP-046 (1 apariciones) |

**Lectura:** el índice refleja menciones de versión en el código. No significa que cada celda implemente todas esas versiones. Busque `EXP-###` en este cuaderno y ejecute solo el experimento deseado en un entorno limpio.

## 01. Clasificación binaria

### EXP-001 · V1, V2, V3, V4

**Archivo de origen:** `Día_19_Octubre_2026.ipynb, celda 37`. **Referencias duplicadas:** 0. **Sintaxis de código:** Válida.

EXIST2021: EXPERIMENTO COMPLETO HMT-V1/V2/V3/V4 + BASELINES, EN UNA CELDA
Requiere pandas numpy scipy scikit-learn matplotlib statsmodels joblib (Colab suele incluirlos).
import os, re, warnings, unicodedata, time
from pathlib import Path
i

**Ejecución:** celda independiente o variante experimental; revisar archivos de entrada y dependencias antes de ejecutarla. Los resultados originales no se insertan en esta celda.

In [ ]:
import os, re, warnings, unicodedata, time
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt, joblib
from scipy.sparse import hstack, csr_matrix, issparse
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler, Normalizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import MultinomialNB, GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_fscore_support, accuracy_score, confusion_matrix, classification_report, ConfusionMatrixDisplay, balanced_accuracy_score
from statsmodels.stats.contingency_tables import mcnemar
from statsmodels.stats.multitest import multipletests
from IPython.display import display

SEED=42; np.random.seed(SEED)
TRAIN='EXIST2021_train_reestructurado.csv'; TEST='EXIST2021_test_reestructurado.csv'
OUT=Path('Resultados_HMT_V4'); OUT.mkdir(parents=True,exist_ok=True)
NONE='No violencia'; CLASSES=['No violencia','Misoginia','Acoso','Violencia sexual']

ALL_BASELINES=True
BASE_MODELS=['LogReg','SVM','KNN','NaiveBayes','MLP','RandomForest','DecisionTree'] if ALL_BASELINES else ['LogReg','SVM','NaiveBayes']

for f in [TRAIN,TEST]:
    if not Path(f).exists():
        from google.colab import files
        print('Sube los CSV con sus nombres originales:')
        files.upload()
        break

def normalize_label(v):
    return unicodedata.normalize('NFKD',str(v)).encode('ascii','ignore').decode('ascii').lower().strip()
def readset(path):
    df=pd.read_csv(path,encoding='utf-8-sig')
    expected={'text','task1','tipo_violencia'}
    if expected-set(df.columns): raise ValueError(f'{path}: faltan columnas {expected-set(df.columns)}')
    df=df.dropna(subset=list(expected)).copy()
    df['text']=df.text.astype(str).str.replace(r'\s+',' ',regex=True).str.strip()
    df=df[df.text.str.len()>0].copy().reset_index(drop=True)
    mapping={'sexist':1,'sexista':1,'non-sexist':0,'non sexist':0,'no sexista':0}
    df['ybin']=df.task1.map(lambda t:mapping.get(normalize_label(t),-1))
    if (df.ybin<0).any():raise ValueError('Etiquetas task1 desconocidas: '+str(df.loc[df.ybin<0,'task1'].unique()))
    df['ymulti']=df.tipo_violencia.astype(str).str.strip()
    if ((df.ybin.eq(0)&df.ymulti.ne(NONE))|(df.ybin.eq(1)&df.ymulti.eq(NONE))).any():
        raise ValueError(f'{path}: task1 y tipo_violencia inconsistentes')
    return df
train=readset(TRAIN); test=readset(TEST)
unknown=set(test.ymulti)-set(train.ymulti)
if unknown:raise ValueError(f'Clases de TEST ausentes en TRAIN: {unknown}')
print('DATOS:',len(train),'train |',len(test),'test')
print('Distribución TRAIN:');display(train.ymulti.value_counts().rename_axis('clase').reset_index(name='registros'))
print('Distribución TEST:');display(test.ymulti.value_counts().rename_axis('clase').reset_index(name='registros'))

train_d,val=train_test_split(train,test_size=.20,random_state=SEED,stratify=train.ymulti)
train_d=train_d.reset_index(drop=True);val=val.reset_index(drop=True)

def mscore(y,p):
    pr,re,f1,_=precision_recall_fscore_support(y,p,average='macro',zero_division=0)
    return {'F1_macro':float(f1),'Precision_macro':float(pr),'Recall_macro':float(re),'Accuracy':float(accuracy_score(y,p))}

def style_table(df,cols=None,ascending=False,title=''):
    if df.empty:return
    if title: print('\n'+title)
    show=df.copy()
    if cols:show=show[cols]
    numeric=show.select_dtypes(include='number').columns
    fmt={c:'{:.3f}' for c in numeric if c not in ['Posicion','N']}
    try:display(show.head(20).style.format(fmt).background_gradient(subset=[c for c in ['F1_macro','F1_binaria','F1_condicional','F1_cascada'] if c in show.columns],cmap='Blues'))
    except Exception:display(show.head(20))


CUES={
'insultos':['puta','puto','zorra','zorro','idiota','estupida','imbecil','inutil','perra'],
'amenaza':['matar','golpear','pegar','violar','castigar','amenazo','romper'],
'control':['callate','obedece','cocina','limpia','prohibo','debes','mandar'],
'sexual':['sexo','sexual','desnuda','tetas','pene','culo','follar'],
'desprecio':['asco','basura','ridicula','patetica','inservible','inferior'],
'genero':['mujer','mujeres','hombre','hombres','feminista','feminazi','esposa']}
def pragmatic(texts):
    rows=[]
    for s in texts:
        u=normalize_label(s); tokens=re.findall(r'\b\w+\b',u); counts={t:tokens.count(t) for t in set(tokens)}; n=max(len(s),1)
        rows.append([*[np.log1p(sum(counts.get(w,0) for w in group)) for group in CUES.values()],
          np.log1p(len(tokens)),len(set(tokens))/max(1,len(tokens)),sum(c.isupper() for c in s)/n,
          np.log1p(s.count('!')),np.log1p(s.count('?')),int('http' in u),int('@' in s),int('#' in s)])
    return np.asarray(rows,dtype=np.float32)

class Blocks:
    def __init__(self):
        self.w=TfidfVectorizer(ngram_range=(1,2),max_features=25000,min_df=2,sublinear_tf=True,strip_accents='unicode')
        self.c=TfidfVectorizer(analyzer='char_wb',ngram_range=(3,5),max_features=30000,min_df=2,sublinear_tf=True)
        self.scale=StandardScaler();self.svd=None
    def fit(self,texts):
        w=self.w.fit_transform(texts); self.c.fit(texts)
        n=min(100,w.shape[0]-1,w.shape[1]-1)
        if n>=2:self.svd=TruncatedSVD(n_components=n,random_state=SEED).fit(w)
        self.scale.fit(pragmatic(texts));return self
    def transform(self,texts):
        w=self.w.transform(texts).astype(np.float32).tocsr()
        c=self.c.transform(texts).astype(np.float32).tocsr()
        if self.svd is None: l=csr_matrix((w.shape[0],0),dtype=np.float32)
        else:l=csr_matrix(Normalizer().fit_transform(self.svd.transform(w)).astype(np.float32))
        p=self.scale.transform(pragmatic(texts))
        p=csr_matrix(Normalizer().fit_transform(p).astype(np.float32))
        return (w,c,l,p)

def combine(blocks,version,weights):
    w,c,l,p=blocks; a,b,d,e=weights
    if version=='HMT-V1':return hstack([w*a,l*d],format='csr')
    if version=='HMT-V2':return hstack([w*a,l*d,p*e],format='csr')
    return hstack([w*a,c*b,l*d,p*e],format='csr')


WEIGHTS={
'HMT-V1':[(1,0,.35,0)],
'HMT-V2':[(1,0,.35,.25)],
'HMT-V3':[(1,.75,.35,.25)],
'HMT-V4':[(1,.5,.2,.15),(1,1,.2,.15),(1,1.5,.2,.15),
          (1,.75,.5,.25),(1,.75,.15,.50),(1,1,.4,.4)]}
C_GRID=[.1,1.,5.]
BWEIGHTS=[None,'balanced']; MWEIGHTS=[None,'balanced']
THRESHOLDS=np.arange(.35,.651,.05).round(2)

def lr(c,balanced):
    return LogisticRegression(C=c,class_weight=balanced,max_iter=2500,random_state=SEED)

def search_task(task):
    rows=[]; blocks=Blocks().fit(train_d.text.to_numpy());bd=blocks.transform(train_d.text.to_numpy());bv=blocks.transform(val.text.to_numpy())
    sex=train_d.ybin.to_numpy()==1
    for version,combos in WEIGHTS.items():
        for wi,weights in enumerate(combos):
            xd=combine(bd,version,weights);xv=combine(bv,version,weights)
            for c in C_GRID:
                for cw in (BWEIGHTS if task=='bin' else MWEIGHTS):
                    model=lr(c,cw)
                    if task=='bin':
                        model.fit(xd,train_d.ybin)
                        probs=model.predict_proba(xv)[:,list(model.classes_).index(1)]
                        for threshold in THRESHOLDS:
                            pred=(probs>=threshold).astype(int);s=mscore(val.ybin,pred)
                            rows.append({'Version':version,'Pesos_ID':wi,'C':c,'Balanceo':str(cw),'Umbral':threshold,**s})
                    else:
                        model.fit(xd[sex],train_d.loc[sex,'ymulti'])
                        pred=model.predict(xv)
                        issex=val.ybin.to_numpy()==1;s=mscore(val.ymulti.to_numpy()[issex],pred[issex])
                        rows.append({'Version':version,'Pesos_ID':wi,'C':c,'Balanceo':str(cw),'Umbral':np.nan,**s})
        print('  Optimizado',version,'/',task,flush=True)
    table=pd.DataFrame(rows).sort_values(['F1_macro','Accuracy'],ascending=False).reset_index(drop=True)
    table.to_csv(OUT/f'busqueda_{task}.csv',index=False)
    return table.iloc[0].to_dict(),table

print('\nOPTIMIZACIÓN HMT (solo TRAIN):')
bestb,searchb=search_task('bin');bestm,searchm=search_task('multi')
print('Mejor binario:',{k:bestb[k] for k in ('Version','Pesos_ID','C','Balanceo','Umbral','F1_macro')})
print('Mejor multiclase:',{k:bestm[k] for k in ('Version','Pesos_ID','C','Balanceo','F1_macro')})


fullblocks=Blocks().fit(train.text.to_numpy()); bt=fullblocks.transform(train.text.to_numpy());bq=fullblocks.transform(test.text.to_numpy())
def final_fit(row,kind):
    v=row['Version']; weights=WEIGHTS[v][int(row['Pesos_ID'])];tr=combine(bt,v,weights);te=combine(bq,v,weights)
    sex=train.ybin.to_numpy()==1; model=lr(float(row['C']),None if row['Balanceo']=='None' else 'balanced')
    if kind=='bin':model.fit(tr,train.ybin)
    else:model.fit(tr[sex],train.loc[sex,'ymulti'])
    return model,te,weights
binmod,xb,wb=final_fit(bestb,'bin');multimod,xm,wm=final_fit(bestm,'multi')
psex=binmod.predict_proba(xb)[:,list(binmod.classes_).index(1)]
probs_cond=multimod.predict_proba(xm);c_names=multimod.classes_
pred_bin=(psex>=float(bestb['Umbral'])).astype(int)
pred_multi_cond=multimod.predict(xm)
pred_cascade=np.where(pred_bin==1,pred_multi_cond,NONE)


def find_joint_scale():
    temp=Blocks().fit(train_d.text.to_numpy());d=temp.transform(train_d.text.to_numpy());v=temp.transform(val.text.to_numpy())
    def fit_val(row,kind):
        vers=row['Version'];wi=int(row['Pesos_ID']);x=combine(d,vers,WEIGHTS[vers][wi]);xv=combine(v,vers,WEIGHTS[vers][wi]);sex=train_d.ybin.to_numpy()==1
        m=lr(float(row['C']),None if row['Balanceo']=='None' else 'balanced')
        if kind=='bin':m.fit(x,train_d.ybin)
        else:m.fit(x[sex],train_d.loc[sex,'ymulti'])
        return m,xv
    b,xx=find=None,None
    b,xx=fit_val(bestb,'bin');m,xxm=fit_val(bestm,'multi')
    ps=b.predict_proba(xx)[:,list(b.classes_).index(1)]
    pc=m.predict_proba(xxm);labels=m.classes_
    best=(-1,None)
    for scale in [.65,.8,1.,1.2,1.4]:
        
        score=np.column_stack([(1-ps)*scale,ps[:,None]*pc]);allname=np.concatenate(([NONE],np.asarray(labels)))
        pred=allname[score.argmax(axis=1)]
        f=mscore(val.ymulti,pred)['F1_macro']
        if f>best[0]:best=(f,scale)
    return best
joint_fval,joint_scale=find_joint_scale()
joint_score=np.column_stack([(1-psex)*joint_scale,psex[:,None]*probs_cond]);joint_names=np.concatenate(([NONE],np.asarray(c_names)))
pred_joint=joint_names[joint_score.argmax(axis=1)]




class BaseRepresentation:
    def __init__(self,name):self.name=name
    def fit(self,texts):
        if self.name=='TF-IDF':self.vec=TfidfVectorizer(max_features=25000,ngram_range=(1,3),min_df=2,sublinear_tf=True,strip_accents='unicode')
        elif self.name=='N-Grams':self.vec=CountVectorizer(max_features=10000,ngram_range=(1,3),min_df=2,strip_accents='unicode')
        elif self.name=='LSA':self.vec=TfidfVectorizer(max_features=25000,ngram_range=(1,3),min_df=2,sublinear_tf=True,strip_accents='unicode')
        if self.name!='Lexical':
            x=self.vec.fit_transform(texts)
            if self.name=='LSA':
                k=min(100,x.shape[0]-1,x.shape[1]-1);self.svd=TruncatedSVD(n_components=k,random_state=SEED).fit(x)
        else:self.scale=StandardScaler().fit(pragmatic(texts))
        return self
    def transform(self,texts):
        if self.name=='Lexical':return self.scale.transform(pragmatic(texts))
        x=self.vec.transform(texts)
        return Normalizer().fit_transform(self.svd.transform(x)) if self.name=='LSA' else x

def make_clf(name,x):
    if name=='LogReg':return lr(1.,'balanced')
    if name=='SVM':return LinearSVC(C=1.,class_weight='balanced',max_iter=10000,dual='auto',random_state=SEED)
    if name=='KNN':return KNeighborsClassifier(n_neighbors=3,weights='distance')
    if name=='NaiveBayes':return MultinomialNB() if issparse(x) else GaussianNB()
    if name=='MLP':return MLPClassifier(hidden_layer_sizes=(128,64),alpha=.001,max_iter=240,early_stopping=True,random_state=SEED)
    if name=='RandomForest':return RandomForestClassifier(n_estimators=200,n_jobs=-1,class_weight='balanced',random_state=SEED)
    if name=='DecisionTree':return DecisionTreeClassifier(max_depth=18,class_weight='balanced',random_state=SEED)
    raise ValueError(name)

predictions={};records=[];baselines={}
true_bin=test.ybin.to_numpy();true_multi=test.ymulti.to_numpy();sex_train=train.ybin.to_numpy()==1;sex_test=true_bin==1
print('\nEVALUANDO BASELINES Y HMT-V1/V2/V3/V4:')
for rep in ['TF-IDF','N-Grams','Lexical','LSA','HMT-V1','HMT-V2','HMT-V3','HMT-V4']:
    if rep.startswith('HMT'):
        
        wt=WEIGHTS[rep][0];xtrain=combine(bt,rep,wt);xtest=combine(bq,rep,wt)
    else:
        feat=BaseRepresentation(rep).fit(train.text.to_numpy());xtrain=feat.transform(train.text.to_numpy());xtest=feat.transform(test.text.to_numpy())
    print('  ',rep,flush=True)
    for name in BASE_MODELS:
        if name=='NaiveBayes' and ((issparse(xtrain) and xtrain.data.size and xtrain.data.min()<0) or (not issparse(xtrain) and np.min(xtrain)<0)):
            continue
        try:
            mb=make_clf(name,xtrain); mb.fit(xtrain,train.ybin); pb=mb.predict(xtest)
            xmtr=xtrain[sex_train]; mm=make_clf(name,xmtr);mm.fit(xmtr,train.loc[sex_train,'ymulti']);pm=mm.predict(xtest)
            pc=np.where(pb==1,pm,NONE)
            predictions[(rep,name)]=(pb,pm,pc)
            baselines[(rep,name)]=mscore(true_multi,pc)
            for task,y,p in [('Binaria',true_bin,pb),('Multiclase condicional',true_multi[sex_test],pm[sex_test]),('Cascada',true_multi,pc)]:
                records.append({'Tarea':task,'Representacion':rep,'Clasificador':name,**mscore(y,p)})
        except Exception as exc:
            print('    Omitido:',rep,name,type(exc).__name__,str(exc)[:110])


for task,y,p in [('Binaria',true_bin,pred_bin),('Multiclase condicional',true_multi[sex_test],pred_multi_cond[sex_test]),
                 ('Cascada',true_multi,pred_cascade),('Cascada conjunta',true_multi,pred_joint)]:
    records.append({'Tarea':task,'Representacion':'HMT-OPTIMIZADO','Clasificador':'HMT',**mscore(y,p)})
results=pd.DataFrame(records)
for task,filename in [('Binaria','01_Binaria'),('Multiclase condicional','02_Multiclase_condicional'),('Cascada','03_Cascada'),('Cascada conjunta','04_Cascada_probabilistica')]:
    tab=results[results.Tarea==task].sort_values('F1_macro',ascending=False).reset_index(drop=True)
    tab.insert(0,'Posicion',np.arange(1,len(tab)+1))
    tab.to_csv(OUT/f'{filename}.csv',index=False,encoding='utf-8-sig')
    tab.to_html(OUT/f'{filename}.html',index=False,float_format=lambda x:f'{x:.3f}')
    style_table(tab,title=f'TABLA: {task.upper()} — TOP 20 por F1 macro')


def compare(y,h,b):
    aa=np.asarray(h)==np.asarray(y); bb=np.asarray(b)==np.asarray(y)
    n10=int(np.sum(aa&~bb));n01=int(np.sum(~aa&bb));both=int(np.sum(aa&bb));neither=int(np.sum(~aa&~bb))
    p=mcnemar([[both,n10],[n01,neither]],exact=n10+n01<25,correction=n10+n01>=25).pvalue
    return n10,n01,p
stat=[]
for (rep,name),(pb,pm,pc) in predictions.items():
    if rep not in ['TF-IDF','N-Grams','Lexical','LSA']:continue
    for task,y,hh,bb in [('Binaria',true_bin,pred_bin,pb),
                         ('Multiclase condicional',true_multi[sex_test],pred_multi_cond[sex_test],pm[sex_test]),
                         ('Cascada',true_multi,pred_cascade,pc),('Cascada conjunta',true_multi,pred_joint,pc)]:
        h_only,b_only,p=compare(y,hh,bb)
        stat.append({'Tarea':task,'Base':rep,'Clasificador':name,'F1_HMT':mscore(y,hh)['F1_macro'],
          'F1_base':mscore(y,bb)['F1_macro'],'Delta_F1':mscore(y,hh)['F1_macro']-mscore(y,bb)['F1_macro'],
          'Solo_HMT_acierta':h_only,'Solo_base_acierta':b_only,'p_McNemar':p})
stat=pd.DataFrame(stat)
if len(stat):
    stat['p_Holm']=np.nan
    for task,ids in stat.groupby('Tarea').groups.items():
        stat.loc[ids,'p_Holm']=multipletests(stat.loc[ids,'p_McNemar'],method='holm')[1]
    stat['HMT_supera_significativamente_en_aciertos']=(stat.p_Holm<.05)&(stat.Solo_HMT_acierta>stat.Solo_base_acierta)
    stat.sort_values(['Tarea','F1_base'],ascending=[True,False]).to_csv(OUT/'05_McNemar_Holm.csv',index=False,encoding='utf-8-sig')
    strong=stat.sort_values(['Tarea','F1_base'],ascending=[True,False]).groupby('Tarea').head(5)
    style_table(strong,title='MCNEMAR: COMPARACIÓN CON LAS 5 BASES MÁS COMPETITIVAS POR TAREA')

def bootstrap_delta(y,h,b,nboot=1000):
    y=np.asarray(y);h=np.asarray(h);b=np.asarray(b);rng=np.random.default_rng(SEED);delta=[];n=len(y)
    for i in range(nboot):
        idx=rng.integers(0,n,size=n)
        delta.append(mscore(y[idx],h[idx])['F1_macro']-mscore(y[idx],b[idx])['F1_macro'])
    return np.quantile(delta,[.025,.5,.975])
intervals=[]
for task,y,h,pidx in [('Binaria',true_bin,pred_bin,0),('Multiclase condicional',true_multi[sex_test],pred_multi_cond[sex_test],1),
                      ('Cascada',true_multi,pred_cascade,2),('Cascada conjunta',true_multi,pred_joint,2)]:
    keys=sorted([k for k in predictions if k[0] in ['TF-IDF','N-Grams','Lexical','LSA']],
                key=lambda k:mscore(y,(predictions[k][pidx][sex_test] if task=='Multiclase condicional' else predictions[k][pidx]))['F1_macro'],reverse=True)
    if keys:
        key=keys[0]; base=predictions[key][pidx];base=base[sex_test] if task=='Multiclase condicional' else base
        low,median,hi=bootstrap_delta(y,h,base)
        intervals.append({'Tarea':task,'Mejor_base_rep':key[0],'Mejor_base_modelo':key[1],
                          'Delta_F1':mscore(y,h)['F1_macro']-mscore(y,base)['F1_macro'],
                          'IC95_inferior':low,'IC95_superior':hi})
ci=pd.DataFrame(intervals);ci.to_csv(OUT/'06_Intervalos_F1_bootstrap.csv',index=False);style_table(ci,title='INTERVALOS DE CONFIANZA PARA DIFERENCIA F1 (HMT - MEJOR BASE)')


for task,y,p,labels in [('Binaria',true_bin,pred_bin,[0,1]),
                         ('Multiclase_condicional',true_multi[sex_test],pred_multi_cond[sex_test],CLASSES[1:]),
                         ('Cascada',true_multi,pred_cascade,CLASSES),('Cascada_conjunta',true_multi,pred_joint,CLASSES)]:
    names=['No sexista','Sexista'] if task=='Binaria' else labels
    print('\nREPORTE HMT:',task,'\n',classification_report(y,p,labels=labels,target_names=names,zero_division=0))
    for normalization,label in [(None,'absoluta'),('true','normalizada')]:
        matrix=confusion_matrix(y,p,labels=labels,normalize=normalization)
        fig,ax=plt.subplots(figsize=(max(7,len(labels)*1.5),6))
        ConfusionMatrixDisplay(matrix,display_labels=names).plot(ax=ax,cmap='Blues',values_format='d' if normalization is None else '.2f',colorbar=False)
        ax.set_title(f'HMT: {task} — {label}');ax.tick_params(axis='x',labelrotation=30);fig.tight_layout()
        fig.savefig(OUT/f'Matriz_{task}_{label}.png',dpi=200,bbox_inches='tight');plt.show();plt.close(fig)

predout=test[['id','text']].copy() if 'id' in test else test[['text']].copy()
predout['task1_real']=test.task1;predout['task1_pred']=np.where(pred_bin==1,'sexist','non-sexist');predout['prob_sexista']=psex
predout['tipo_real']=true_multi;predout['tipo_pred_condicional']=pred_multi_cond;predout['tipo_pred_cascada']=pred_cascade
predout['tipo_pred_conjunta']=pred_joint
predout.to_csv(OUT/'07_Predicciones_TEST.csv',index=False,encoding='utf-8-sig')
joblib.dump({'blocks':fullblocks,'bin_model':binmod,'multi_model':multimod,'best_bin':bestb,'best_multi':bestm,
             'weights_bin':wb,'weights_multi':wm,'joint_scale':joint_scale,'classes':CLASSES},OUT/'HMT_modelo.joblib')
results.to_csv(OUT/'08_TODOS_LOS_RESULTADOS.csv',index=False,encoding='utf-8-sig')
print('\nLISTO. Carpeta:',OUT.resolve(),'| Conjunta: escala elegida en validación',joint_scale,'F1 validación',round(joint_fval,3))
print('IMPORTANTE: la búsqueda se hizo con TRAIN. Las métricas de TEST son mediciones, no un objetivo garantizado.')


### EXP-002 · V1, V2, V3, V4

**Archivo de origen:** `Día_19_Octubre_2026.ipynb, celda 38`. **Referencias duplicadas:** 0. **Sintaxis de código:** Válida.

EXIST2021: EXPERIMENTO COMPLETO HMT-V1/V2/V3/V4 + BASELINES, EN UNA CELDA
Requiere pandas numpy scipy scikit-learn matplotlib statsmodels joblib (Colab suele incluirlos).
import os, re, warnings, unicodedata, time
from pathlib import Path
i

**Ejecución:** celda independiente o variante experimental; revisar archivos de entrada y dependencias antes de ejecutarla. Los resultados originales no se insertan en esta celda.

In [ ]:
import os, re, warnings, unicodedata, time
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt, joblib
from scipy.sparse import hstack, csr_matrix, issparse
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler, Normalizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import MultinomialNB, GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_fscore_support, accuracy_score, confusion_matrix, classification_report, ConfusionMatrixDisplay, balanced_accuracy_score
from statsmodels.stats.contingency_tables import mcnemar
from statsmodels.stats.multitest import multipletests
from IPython.display import display

SEED=42; np.random.seed(SEED)
TRAIN='EXIST2021_train_reestructurado.csv'; TEST='EXIST2021_test_reestructurado.csv'
OUT=Path('Resultados_HMT_V4_CORREGIDO'); OUT.mkdir(parents=True,exist_ok=True)
NONE='No violencia'; CLASSES=['No violencia','Misoginia','Acoso','Violencia sexual']

ALL_BASELINES=True
BASE_MODELS=['LogReg','SVM','KNN','NaiveBayes','MLP','RandomForest','DecisionTree'] if ALL_BASELINES else ['LogReg','SVM','NaiveBayes']

for f in [TRAIN,TEST]:
    if not Path(f).exists():
        from google.colab import files
        print('Sube los CSV con sus nombres originales:')
        files.upload()
        break

def normalize_label(v):
    return unicodedata.normalize('NFKD',str(v)).encode('ascii','ignore').decode('ascii').lower().strip()
def readset(path):
    df=pd.read_csv(path,encoding='utf-8-sig')
    expected={'text','task1','tipo_violencia'}
    if expected-set(df.columns): raise ValueError(f'{path}: faltan columnas {expected-set(df.columns)}')
    df=df.dropna(subset=list(expected)).copy()
    df['text']=df.text.astype(str).str.replace(r'\s+',' ',regex=True).str.strip()
    df=df[df.text.str.len()>0].copy().reset_index(drop=True)
    mapping={'sexist':1,'sexista':1,'non-sexist':0,'non sexist':0,'no sexista':0}
    df['ybin']=df.task1.map(lambda t:mapping.get(normalize_label(t),-1))
    if (df.ybin<0).any():raise ValueError('Etiquetas task1 desconocidas: '+str(df.loc[df.ybin<0,'task1'].unique()))
    df['ymulti']=df.tipo_violencia.astype(str).str.strip()
    if ((df.ybin.eq(0)&df.ymulti.ne(NONE))|(df.ybin.eq(1)&df.ymulti.eq(NONE))).any():
        raise ValueError(f'{path}: task1 y tipo_violencia inconsistentes')
    return df
train=readset(TRAIN); test=readset(TEST)
unknown=set(test.ymulti)-set(train.ymulti)
if unknown:raise ValueError(f'Clases de TEST ausentes en TRAIN: {unknown}')
print('DATOS:',len(train),'train |',len(test),'test')
print('Distribución TRAIN:');display(train.ymulti.value_counts().rename_axis('clase').reset_index(name='registros'))
print('Distribución TEST:');display(test.ymulti.value_counts().rename_axis('clase').reset_index(name='registros'))

train_d,val=train_test_split(train,test_size=.20,random_state=SEED,stratify=train.ymulti)
train_d=train_d.reset_index(drop=True);val=val.reset_index(drop=True)

def mscore(y,p):
    pr,re,f1,_=precision_recall_fscore_support(y,p,average='macro',zero_division=0)
    return {'F1_macro':float(f1),'Precision_macro':float(pr),'Recall_macro':float(re),'Accuracy':float(accuracy_score(y,p))}

def style_table(df,cols=None,ascending=False,title=''):
    if df.empty:return
    if title: print('\n'+title)
    show=df.copy()
    if cols:show=show[cols]
    numeric=show.select_dtypes(include='number').columns
    fmt={c:'{:.3f}' for c in numeric if c not in ['Posicion','N']}
    try:display(show.head(20).style.format(fmt).background_gradient(subset=[c for c in ['F1_macro','F1_binaria','F1_condicional','F1_cascada'] if c in show.columns],cmap='Blues'))
    except Exception:display(show.head(20))


CUES={
'insultos':['puta','puto','zorra','zorro','idiota','estupida','imbecil','inutil','perra'],
'amenaza':['matar','golpear','pegar','violar','castigar','amenazo','romper'],
'control':['callate','obedece','cocina','limpia','prohibo','debes','mandar'],
'sexual':['sexo','sexual','desnuda','tetas','pene','culo','follar'],
'desprecio':['asco','basura','ridicula','patetica','inservible','inferior'],
'genero':['mujer','mujeres','hombre','hombres','feminista','feminazi','esposa']}
def pragmatic(texts):
    rows=[]
    for s in texts:
        u=normalize_label(s); tokens=re.findall(r'\b\w+\b',u); counts={t:tokens.count(t) for t in set(tokens)}; n=max(len(s),1)
        rows.append([*[np.log1p(sum(counts.get(w,0) for w in group)) for group in CUES.values()],
          np.log1p(len(tokens)),len(set(tokens))/max(1,len(tokens)),sum(c.isupper() for c in s)/n,
          np.log1p(s.count('!')),np.log1p(s.count('?')),int('http' in u),int('@' in s),int('#' in s)])
    return np.asarray(rows,dtype=np.float32)

class Blocks:
    def __init__(self):
        self.w=TfidfVectorizer(ngram_range=(1,2),max_features=25000,min_df=2,sublinear_tf=True,strip_accents='unicode')
        self.c=TfidfVectorizer(analyzer='char_wb',ngram_range=(3,5),max_features=30000,min_df=2,sublinear_tf=True)
        self.scale=StandardScaler();self.svd=None
    def fit(self,texts):
        w=self.w.fit_transform(texts); self.c.fit(texts)
        n=min(100,w.shape[0]-1,w.shape[1]-1)
        if n>=2:self.svd=TruncatedSVD(n_components=n,random_state=SEED).fit(w)
        self.scale.fit(pragmatic(texts));return self
    def transform(self,texts):
        w=self.w.transform(texts).astype(np.float32).tocsr()
        c=self.c.transform(texts).astype(np.float32).tocsr()
        if self.svd is None: l=csr_matrix((w.shape[0],0),dtype=np.float32)
        else:l=csr_matrix(Normalizer().fit_transform(self.svd.transform(w)).astype(np.float32))
        p=self.scale.transform(pragmatic(texts))
        p=csr_matrix(Normalizer().fit_transform(p).astype(np.float32))
        return (w,c,l,p)

def combine(blocks,version,weights):
    w,c,l,p=blocks; a,b,d,e=weights
    if version=='HMT-V1':return hstack([w*a,l*d],format='csr')
    if version=='HMT-V2':return hstack([w*a,l*d,p*e],format='csr')
    return hstack([w*a,c*b,l*d,p*e],format='csr')


WEIGHTS={
'HMT-V1':[(1,0,.35,0)],
'HMT-V2':[(1,0,.35,.25)],
'HMT-V3':[(1,.75,.35,.25)],
'HMT-V4':[(1,.5,.2,.15),(1,1,.2,.15),(1,1.5,.2,.15),
          (1,.75,.5,.25),(1,.75,.15,.50),(1,1,.4,.4)]}
C_GRID=[.1,1.,5.]
BWEIGHTS=[None,'balanced']; MWEIGHTS=[None,'balanced']
THRESHOLDS=np.arange(.35,.651,.05).round(2)

def lr(c,balanced):
    return LogisticRegression(C=c,class_weight=balanced,max_iter=2500,random_state=SEED)

def search_task(task):
    rows=[]; blocks=Blocks().fit(train_d.text.to_numpy());bd=blocks.transform(train_d.text.to_numpy());bv=blocks.transform(val.text.to_numpy())
    sex=train_d.ybin.to_numpy()==1
    for version,combos in WEIGHTS.items():
        for wi,weights in enumerate(combos):
            xd=combine(bd,version,weights);xv=combine(bv,version,weights)
            for c in C_GRID:
                for cw in (BWEIGHTS if task=='bin' else MWEIGHTS):
                    model=lr(c,cw)
                    if task=='bin':
                        model.fit(xd,train_d.ybin)
                        probs=model.predict_proba(xv)[:,list(model.classes_).index(1)]
                        for threshold in THRESHOLDS:
                            pred=(probs>=threshold).astype(int);s=mscore(val.ybin,pred)
                            rows.append({'Version':version,'Pesos_ID':wi,'C':c,'Balanceo':str(cw),'Umbral':threshold,**s})
                    else:
                        model.fit(xd[sex],train_d.loc[sex,'ymulti'])
                        pred=model.predict(xv)
                        issex=val.ybin.to_numpy()==1;s=mscore(val.ymulti.to_numpy()[issex],pred[issex])
                        rows.append({'Version':version,'Pesos_ID':wi,'C':c,'Balanceo':str(cw),'Umbral':np.nan,**s})
        print('  Optimizado',version,'/',task,flush=True)
    table=pd.DataFrame(rows).sort_values(['F1_macro','Accuracy'],ascending=False).reset_index(drop=True)
    table.to_csv(OUT/f'busqueda_{task}.csv',index=False)
    return table.iloc[0].to_dict(),table

print('\nOPTIMIZACIÓN HMT (solo TRAIN):')
bestb,searchb=search_task('bin');bestm,searchm=search_task('multi')
print('Mejor binario:',{k:bestb[k] for k in ('Version','Pesos_ID','C','Balanceo','Umbral','F1_macro')})
print('Mejor multiclase:',{k:bestm[k] for k in ('Version','Pesos_ID','C','Balanceo','F1_macro')})


fullblocks=Blocks().fit(train.text.to_numpy()); bt=fullblocks.transform(train.text.to_numpy());bq=fullblocks.transform(test.text.to_numpy())
def final_fit(row,kind):
    v=row['Version']; weights=WEIGHTS[v][int(row['Pesos_ID'])];tr=combine(bt,v,weights);te=combine(bq,v,weights)
    sex=train.ybin.to_numpy()==1; model=lr(float(row['C']),None if row['Balanceo']=='None' else 'balanced')
    if kind=='bin':model.fit(tr,train.ybin)
    else:model.fit(tr[sex],train.loc[sex,'ymulti'])
    return model,te,weights
binmod,xb,wb=final_fit(bestb,'bin');multimod,xm,wm=final_fit(bestm,'multi')
psex=binmod.predict_proba(xb)[:,list(binmod.classes_).index(1)]
probs_cond=multimod.predict_proba(xm);c_names=multimod.classes_
pred_bin=(psex>=float(bestb['Umbral'])).astype(int)
pred_multi_cond=multimod.predict(xm)
pred_cascade=np.where(pred_bin==1,pred_multi_cond,NONE)


def find_joint_scale():
    temp=Blocks().fit(train_d.text.to_numpy());d=temp.transform(train_d.text.to_numpy());v=temp.transform(val.text.to_numpy())
    def fit_val(row,kind):
        vers=row['Version'];wi=int(row['Pesos_ID']);x=combine(d,vers,WEIGHTS[vers][wi]);xv=combine(v,vers,WEIGHTS[vers][wi]);sex=train_d.ybin.to_numpy()==1
        m=lr(float(row['C']),None if row['Balanceo']=='None' else 'balanced')
        if kind=='bin':m.fit(x,train_d.ybin)
        else:m.fit(x[sex],train_d.loc[sex,'ymulti'])
        return m,xv
    b,xx=find=None,None
    b,xx=fit_val(bestb,'bin');m,xxm=fit_val(bestm,'multi')
    ps=b.predict_proba(xx)[:,list(b.classes_).index(1)]
    pc=m.predict_proba(xxm);labels=m.classes_
    best=(-1,None)
    for scale in [.65,.8,1.,1.2,1.4]:
        
        score=np.column_stack([(1-ps)*scale,ps[:,None]*pc]);allname=np.concatenate(([NONE],np.asarray(labels)))
        pred=allname[score.argmax(axis=1)]
        f=mscore(val.ymulti,pred)['F1_macro']
        if f>best[0]:best=(f,scale)
    return best
joint_fval,joint_scale=find_joint_scale()
joint_score=np.column_stack([(1-psex)*joint_scale,psex[:,None]*probs_cond]);joint_names=np.concatenate(([NONE],np.asarray(c_names)))
pred_joint=joint_names[joint_score.argmax(axis=1)]




class BaseRepresentation:
    def __init__(self,name):self.name=name
    def fit(self,texts):
        if self.name=='TF-IDF':self.vec=TfidfVectorizer(max_features=25000,ngram_range=(1,3),min_df=2,sublinear_tf=True,strip_accents='unicode')
        elif self.name=='N-Grams':self.vec=CountVectorizer(max_features=10000,ngram_range=(1,3),min_df=2,strip_accents='unicode')
        elif self.name=='LSA':self.vec=TfidfVectorizer(max_features=25000,ngram_range=(1,3),min_df=2,sublinear_tf=True,strip_accents='unicode')
        if self.name!='Lexical':
            x=self.vec.fit_transform(texts)
            if self.name=='LSA':
                k=min(100,x.shape[0]-1,x.shape[1]-1);self.svd=TruncatedSVD(n_components=k,random_state=SEED).fit(x)
        else:self.scale=StandardScaler().fit(pragmatic(texts))
        return self
    def transform(self,texts):
        if self.name=='Lexical':return self.scale.transform(pragmatic(texts))
        x=self.vec.transform(texts)
        return Normalizer().fit_transform(self.svd.transform(x)) if self.name=='LSA' else x

def make_clf(name,x):
    if name=='LogReg':return lr(1.,'balanced')
    if name=='SVM':return LinearSVC(C=1.,class_weight='balanced',max_iter=10000,dual='auto',random_state=SEED)
    if name=='KNN':return KNeighborsClassifier(n_neighbors=3,weights='distance')
    if name=='NaiveBayes':return MultinomialNB() if issparse(x) else GaussianNB()
    if name=='MLP':return MLPClassifier(hidden_layer_sizes=(128,64),alpha=.01,max_iter=300,early_stopping=False,random_state=SEED)
    if name=='RandomForest':return RandomForestClassifier(n_estimators=200,n_jobs=-1,class_weight='balanced',random_state=SEED)
    if name=='DecisionTree':return DecisionTreeClassifier(max_depth=18,class_weight='balanced',random_state=SEED)
    raise ValueError(name)

predictions={};records=[];baselines={}
true_bin=test.ybin.to_numpy();true_multi=test.ymulti.to_numpy();sex_train=train.ybin.to_numpy()==1;sex_test=true_bin==1
print('\nEVALUANDO BASELINES Y HMT-V1/V2/V3/V4:')
for rep in ['TF-IDF','N-Grams','Lexical','LSA','HMT-V1','HMT-V2','HMT-V3','HMT-V4']:
    if rep.startswith('HMT'):
        
        wt=WEIGHTS[rep][0];xtrain=combine(bt,rep,wt);xtest=combine(bq,rep,wt)
    else:
        feat=BaseRepresentation(rep).fit(train.text.to_numpy());xtrain=feat.transform(train.text.to_numpy());xtest=feat.transform(test.text.to_numpy())
    print('  ',rep,flush=True)
    for name in BASE_MODELS:
        if name=='NaiveBayes' and ((issparse(xtrain) and xtrain.data.size and xtrain.data.min()<0) or (not issparse(xtrain) and np.min(xtrain)<0)):
            continue
        try:
            
            
            if name=='MLP':
                xxtr=np.asarray(xtrain.toarray() if issparse(xtrain) else xtrain,dtype=np.float32)
                xxte=np.asarray(xtest.toarray() if issparse(xtest) else xtest,dtype=np.float32)
                if not np.isfinite(xxtr).all() or not np.isfinite(xxte).all():
                    raise ValueError('MLP: se detectaron valores no finitos')
            else: xxtr,xxte=xtrain,xtest
            mb=make_clf(name,xxtr); mb.fit(xxtr,train.ybin.to_numpy(dtype=int)); pb=mb.predict(xxte)
            xmtr=xxtr[sex_train]; mm=make_clf(name,xmtr)
            
            from sklearn.preprocessing import LabelEncoder
            enc=LabelEncoder().fit(train.loc[sex_train,'ymulti'].astype(str))
            yy=enc.transform(train.loc[sex_train,'ymulti'].astype(str)) if name=='MLP' else train.loc[sex_train,'ymulti']
            mm.fit(xmtr,yy)
            pm=enc.inverse_transform(mm.predict(xxte).astype(int)) if name=='MLP' else mm.predict(xxte)
            pc=np.where(pb==1,pm,NONE)
            predictions[(rep,name)]=(pb,pm,pc)
            baselines[(rep,name)]=mscore(true_multi,pc)
            for task,y,p in [('Binaria',true_bin,pb),('Multiclase condicional',true_multi[sex_test],pm[sex_test]),('Cascada',true_multi,pc)]:
                records.append({'Tarea':task,'Representacion':rep,'Clasificador':name,**mscore(y,p)})
        except Exception as exc:
            print('    Omitido:',rep,name,type(exc).__name__,str(exc)[:110])


for task,y,p in [('Binaria',true_bin,pred_bin),('Multiclase condicional',true_multi[sex_test],pred_multi_cond[sex_test]),
                 ('Cascada',true_multi,pred_cascade),('Cascada conjunta',true_multi,pred_joint)]:
    records.append({'Tarea':task,'Representacion':'HMT-OPTIMIZADO','Clasificador':'HMT',**mscore(y,p)})
results=pd.DataFrame(records)
for task,filename in [('Binaria','01_Binaria'),('Multiclase condicional','02_Multiclase_condicional'),('Cascada','03_Cascada'),('Cascada conjunta','04_Cascada_probabilistica')]:
    tab=results[results.Tarea==task].sort_values('F1_macro',ascending=False).reset_index(drop=True)
    tab.insert(0,'Posicion',np.arange(1,len(tab)+1))
    tab.to_csv(OUT/f'{filename}.csv',index=False,encoding='utf-8-sig')
    tab.to_html(OUT/f'{filename}.html',index=False,float_format=lambda x:f'{x:.3f}')
    style_table(tab,title=f'TABLA: {task.upper()} — TOP 20 por F1 macro')


def compare(y,h,b):
    aa=np.asarray(h)==np.asarray(y); bb=np.asarray(b)==np.asarray(y)
    n10=int(np.sum(aa&~bb));n01=int(np.sum(~aa&bb));both=int(np.sum(aa&bb));neither=int(np.sum(~aa&~bb))
    p=mcnemar([[both,n10],[n01,neither]],exact=n10+n01<25,correction=n10+n01>=25).pvalue
    return n10,n01,p
stat=[]
for (rep,name),(pb,pm,pc) in predictions.items():
    if rep not in ['TF-IDF','N-Grams','Lexical','LSA']:continue
    for task,y,hh,bb in [('Binaria',true_bin,pred_bin,pb),
                         ('Multiclase condicional',true_multi[sex_test],pred_multi_cond[sex_test],pm[sex_test]),
                         ('Cascada',true_multi,pred_cascade,pc),('Cascada conjunta',true_multi,pred_joint,pc)]:
        h_only,b_only,p=compare(y,hh,bb)
        stat.append({'Tarea':task,'Base':rep,'Clasificador':name,'F1_HMT':mscore(y,hh)['F1_macro'],
          'F1_base':mscore(y,bb)['F1_macro'],'Delta_F1':mscore(y,hh)['F1_macro']-mscore(y,bb)['F1_macro'],
          'Solo_HMT_acierta':h_only,'Solo_base_acierta':b_only,'p_McNemar':p})
stat=pd.DataFrame(stat)
if len(stat):
    stat['p_Holm']=np.nan
    for task,ids in stat.groupby('Tarea').groups.items():
        stat.loc[ids,'p_Holm']=multipletests(stat.loc[ids,'p_McNemar'],method='holm')[1]
    stat['HMT_supera_significativamente_en_aciertos']=(stat.p_Holm<.05)&(stat.Solo_HMT_acierta>stat.Solo_base_acierta)
    stat.sort_values(['Tarea','F1_base'],ascending=[True,False]).to_csv(OUT/'05_McNemar_Holm.csv',index=False,encoding='utf-8-sig')
    strong=stat.sort_values(['Tarea','F1_base'],ascending=[True,False]).groupby('Tarea').head(5)
    style_table(strong,title='MCNEMAR: COMPARACIÓN CON LAS 5 BASES MÁS COMPETITIVAS POR TAREA')

def bootstrap_delta(y,h,b,nboot=1000):
    y=np.asarray(y);h=np.asarray(h);b=np.asarray(b);rng=np.random.default_rng(SEED);delta=[];n=len(y)
    for i in range(nboot):
        idx=rng.integers(0,n,size=n)
        delta.append(mscore(y[idx],h[idx])['F1_macro']-mscore(y[idx],b[idx])['F1_macro'])
    return np.quantile(delta,[.025,.5,.975])
intervals=[]
for task,y,h,pidx in [('Binaria',true_bin,pred_bin,0),('Multiclase condicional',true_multi[sex_test],pred_multi_cond[sex_test],1),
                      ('Cascada',true_multi,pred_cascade,2),('Cascada conjunta',true_multi,pred_joint,2)]:
    keys=sorted([k for k in predictions if k[0] in ['TF-IDF','N-Grams','Lexical','LSA']],
                key=lambda k:mscore(y,(predictions[k][pidx][sex_test] if task=='Multiclase condicional' else predictions[k][pidx]))['F1_macro'],reverse=True)
    if keys:
        key=keys[0]; base=predictions[key][pidx];base=base[sex_test] if task=='Multiclase condicional' else base
        low,median,hi=bootstrap_delta(y,h,base)
        intervals.append({'Tarea':task,'Mejor_base_rep':key[0],'Mejor_base_modelo':key[1],
                          'Delta_F1':mscore(y,h)['F1_macro']-mscore(y,base)['F1_macro'],
                          'IC95_inferior':low,'IC95_superior':hi})
ci=pd.DataFrame(intervals);ci.to_csv(OUT/'06_Intervalos_F1_bootstrap.csv',index=False);style_table(ci,title='INTERVALOS DE CONFIANZA PARA DIFERENCIA F1 (HMT - MEJOR BASE)')


for task,y,p,labels in [('Binaria',true_bin,pred_bin,[0,1]),
                         ('Multiclase_condicional',true_multi[sex_test],pred_multi_cond[sex_test],CLASSES[1:]),
                         ('Cascada',true_multi,pred_cascade,CLASSES),('Cascada_conjunta',true_multi,pred_joint,CLASSES)]:
    names=['No sexista','Sexista'] if task=='Binaria' else labels
    print('\nREPORTE HMT:',task,'\n',classification_report(y,p,labels=labels,target_names=names,zero_division=0))
    for normalization,label in [(None,'absoluta'),('true','normalizada')]:
        matrix=confusion_matrix(y,p,labels=labels,normalize=normalization)
        fig,ax=plt.subplots(figsize=(max(7,len(labels)*1.5),6))
        ConfusionMatrixDisplay(matrix,display_labels=names).plot(ax=ax,cmap='Blues',values_format='d' if normalization is None else '.2f',colorbar=False)
        ax.set_title(f'HMT: {task} — {label}');ax.tick_params(axis='x',labelrotation=30);fig.tight_layout()
        fig.savefig(OUT/f'Matriz_{task}_{label}.png',dpi=200,bbox_inches='tight');plt.show();plt.close(fig)

predout=test[['id','text']].copy() if 'id' in test else test[['text']].copy()
predout['task1_real']=test.task1;predout['task1_pred']=np.where(pred_bin==1,'sexist','non-sexist');predout['prob_sexista']=psex
predout['tipo_real']=true_multi;predout['tipo_pred_condicional']=pred_multi_cond;predout['tipo_pred_cascada']=pred_cascade
predout['tipo_pred_conjunta']=pred_joint
predout.to_csv(OUT/'07_Predicciones_TEST.csv',index=False,encoding='utf-8-sig')
joblib.dump({'blocks':fullblocks,'bin_model':binmod,'multi_model':multimod,'best_bin':bestb,'best_multi':bestm,
             'weights_bin':wb,'weights_multi':wm,'joint_scale':joint_scale,'classes':CLASSES},OUT/'HMT_modelo.joblib')
results.to_csv(OUT/'08_TODOS_LOS_RESULTADOS.csv',index=False,encoding='utf-8-sig')
print('\nLISTO. Carpeta:',OUT.resolve(),'| Conjunta: escala elegida en validación',joint_scale,'F1 validación',round(joint_fval,3))
print('IMPORTANTE: la búsqueda se hizo con TRAIN. Las métricas de TEST son mediciones, no un objetivo garantizado.')

summary=results[results.Representacion.eq('HMT-OPTIMIZADO')][['Tarea','F1_macro','Precision_macro','Recall_macro','Accuracy']]
print('\nRESUMEN DEL MODELO HMT (EVALUACIÓN FINAL):')
display(summary.style.format({c:'{:.3f}' for c in ['F1_macro','Precision_macro','Recall_macro','Accuracy']}))
print('ATENCIÓN: una mejor puntuación en validación no garantiza una mejora en TEST.')


### EXP-003 · V1, V2, V3, V4

**Archivo de origen:** `Día_19_Octubre_2026.ipynb, celda 39`. **Referencias duplicadas:** 0. **Sintaxis de código:** Válida.

EXPERIMENTO EXIST2021 — TODO EN UNA CELDA DE COLAB
Train: selección/validación; Test: solo evaluación final, nunca selección.
Requiere conexión a Internet para descargar BETO y XLM-R (si RUN_TRANSFORMERS=True).
import sys, subprocess, impor

**Ejecución:** celda independiente o variante experimental; revisar archivos de entrada y dependencias antes de ejecutarla. Los resultados originales no se insertan en esta celda.

In [ ]:
import sys, subprocess, importlib.util
for package, module in [('numpy','numpy'),('pandas','pandas'),('scipy','scipy'),('scikit-learn','sklearn'),('matplotlib','matplotlib'),('statsmodels','statsmodels'),('joblib','joblib')]:
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable,'-m','pip','-q','install',package])
import os, re, unicodedata, warnings, gc
from pathlib import Path
from collections import Counter
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.sparse import hstack, csr_matrix, issparse
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler, Normalizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import MultinomialNB, GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import f1_score, precision_recall_fscore_support, accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split
from statsmodels.stats.contingency_tables import mcnemar
from statsmodels.stats.multitest import multipletests
from joblib import dump

SEED=42; np.random.seed(SEED)
TRAIN_FILE='EXIST2021_train_reestructurado.csv'
TEST_FILE='EXIST2021_test_reestructurado.csv'
OUT=Path('Resultados_EXIST2021_HMT_Transformers'); OUT.mkdir(parents=True,exist_ok=True)
RUN_TRANSFORMERS=True  
TRANSFORMER_EPOCHS=3; TRANSFORMER_BATCH=8; MAX_TOKENS=160
RUN_ALL_CLASSIFIERS=True 
BOOTSTRAP_REPS=500; VALID_SIZE=.20
NO='No violencia'; TYPES=['Misoginia','Acoso','Violencia sexual']; ALL=[NO]+TYPES

WEIGHTS=[(1.,0.5,0.25,0.15),(1.,1.,0.25,0.25),(1.,1.5,0.5,0.4),(1.,.75,.1,.5),(1.,1.,0.,0.)]
CS=[.1,1.,5.]; BALANCES=[None,'balanced']; THRESHOLDS=np.linspace(.35,.65,7)

if not Path(TRAIN_FILE).exists() or not Path(TEST_FILE).exists():
    try:
        from google.colab import files
        print('Sube ambos CSV originales:'); files.upload()
    except ImportError: pass

def norm(v):
    return unicodedata.normalize('NFKD',str(v)).encode('ascii','ignore').decode().lower().strip()
def read_data(p):
    df=pd.read_csv(p,encoding='utf-8-sig'); needed={'text','task1','tipo_violencia'}
    if not needed.issubset(df.columns):raise ValueError(f'{p}: faltan {needed-set(df.columns)}')
    df=df.dropna(subset=list(needed)).copy().reset_index(drop=True)
    df['text']=df['text'].astype(str).str.replace(r'\s+',' ',regex=True).str.strip()
    df=df[df.text.str.len()>0].reset_index(drop=True)
    mapping={'sexist':1,'sexista':1,'non-sexist':0,'non sexist':0,'no sexista':0}
    df['binary']=df.task1.map(lambda s:mapping.get(norm(s),-1))
    if (df.binary<0).any():raise ValueError('Etiquetas task1 desconocidas')
    df['violence']=df.tipo_violencia.str.strip()
    bad=((df.binary==0)&(df.violence!=NO))|((df.binary==1)&(~df.violence.isin(TYPES)))
    if bad.any():raise ValueError(f'{p}: {bad.sum()} filas con etiquetas incoherentes')
    return df
train=read_data(TRAIN_FILE); test=read_data(TEST_FILE)
print('TRAIN:',len(train),'TEST:',len(test));print('CLASES TRAIN:',train.violence.value_counts().to_dict())

inter=set(train.text.str.casefold()) & set(test.text.str.casefold())
print('Textos idénticos entre TRAIN/TEST:',len(inter),'(revisar para el artículo; no eliminados automáticamente)')
if len(inter):pd.Series(sorted(inter),name='text').to_csv(OUT/'solapamientos_a_revisar.csv',index=False)
strata=train.violence if train.violence.value_counts().min()>=2 else train.binary
idx_d,idx_v=train_test_split(np.arange(len(train)),test_size=VALID_SIZE,random_state=SEED,stratify=strata)
dev=train.iloc[idx_d].reset_index(drop=True); val=train.iloc[idx_v].reset_index(drop=True)

def cues(texts):
    words=[['puta','zorra','imbecil','estupida','idiota','inutil'],['callate','vete','cocina','obedece'],['mujer','mujeres','feminazi','esposa','novia'],['asco','basura','patetica','ridicula']]
    rows=[]
    for raw in texts:
        s=str(raw);t=norm(s);tok=re.findall(r'\b\w+\b',t);ss=set(tok);n=max(1,len(s))
        rows.append([sum(w in ss for w in group) for group in words]+[np.log1p(len(tok)),len(ss)/max(len(tok),1),sum(c.isupper() for c in s)/n,np.log1p(s.count('!')),np.log1p(s.count('?')),int('#' in s),int('@' in s),int('http' in t)])
    return np.asarray(rows,dtype=np.float32)

class Features:
    def __init__(self,name,weights=(1,.75,.25,.25)):
        self.name=name;self.weights=weights
    def fit(self,texts):
        t=list(map(str,texts)); n=self.name
        if n in ('BoW','N-Grams'):
            self.vec=CountVectorizer(ngram_range=(1,1) if n=='BoW' else (1,3),max_features=24000,min_df=2,strip_accents='unicode').fit(t)
        elif n in ('TF-IDF','TF-IDF caracteres','LSA') or n.startswith('HMT'):
            self.word=TfidfVectorizer(ngram_range=(1,2),max_features=25000,min_df=2,sublinear_tf=True,strip_accents='unicode').fit(t)
            if n=='TF-IDF caracteres' or n in ('HMT-V3','HMT-V4'):
                self.char=TfidfVectorizer(analyzer='char_wb',ngram_range=(3,5),max_features=28000,min_df=2,sublinear_tf=True).fit(t)
            if n=='LSA' or n.startswith('HMT'):
                x=self.word.transform(t)
                k=min(100,x.shape[1]-1,x.shape[0]-1)
                self.svd=TruncatedSVD(n_components=k,random_state=SEED).fit(x) if k>=2 else None
        if n=='Lexical' or n in ('HMT-V2','HMT-V3','HMT-V4'):
            self.scaler=StandardScaler().fit(cues(t))
        return self
    def transform(self,texts):
        t=list(map(str,texts));n=self.name
        if n in ('BoW','N-Grams'):return self.vec.transform(t).astype(np.float32)
        if n=='Lexical':return self.scaler.transform(cues(t)).astype(np.float32)
        x=self.word.transform(t).astype(np.float32)
        if n=='TF-IDF':return x
        if n=='TF-IDF caracteres':return self.char.transform(t).astype(np.float32)
        z=Normalizer().fit_transform(self.svd.transform(x)).astype(np.float32) if self.svd else np.zeros((len(t),0),dtype=np.float32)
        if n=='LSA':return z
        w,c,l,p=self.weights
        parts=[x.multiply(w)]
        if n in ('HMT-V3','HMT-V4'):parts.append(self.char.transform(t).multiply(c))
        if z.shape[1]:parts.append(csr_matrix(z*l))
        if n in ('HMT-V2','HMT-V3','HMT-V4'):
            a=self.scaler.transform(cues(t));a=Normalizer().fit_transform(a);parts.append(csr_matrix(a*p))
        return hstack(parts,format='csr',dtype=np.float32)

def make_clf(name,X,C=1.,balance='balanced'):
    if name=='LogReg':return LogisticRegression(C=C,class_weight=balance,max_iter=3000,random_state=SEED)
    if name=='SVM':return LinearSVC(C=C,class_weight=balance,max_iter=12000,random_state=SEED)
    if name=='KNN':return KNeighborsClassifier(n_neighbors=3,weights='distance')
    if name=='NaiveBayes':
        if issparse(X):
            if len(X.data) and X.data.min()<0:return None
            return MultinomialNB()
        return GaussianNB()
    if name=='MLP':return MLPClassifier(hidden_layer_sizes=(128,64),max_iter=350,early_stopping=True,n_iter_no_change=15,random_state=SEED)
    if name=='RandomForest':return RandomForestClassifier(n_estimators=180,class_weight=balance,n_jobs=-1,random_state=SEED)
    if name=='DecisionTree':return DecisionTreeClassifier(max_depth=18,class_weight=balance,random_state=SEED)
    raise ValueError(name)
def score(y,p):
    a,b,c,_=precision_recall_fscore_support(y,p,average='macro',zero_division=0)
    return {'F1_macro':float(c),'Precision_macro':float(a),'Recall_macro':float(b),'Accuracy':float(accuracy_score(y,p))}
def prob_pos(m,X):
    pr=m.predict_proba(X)
    return pr[:,list(m.classes_).index(1)]
def hard(pbin,pmulti,threshold):
    return np.where(pbin>=threshold,pmulti,NO)
def joint(pbin,probs,classes,scale=1.):
    q=np.zeros((len(pbin),len(ALL)),dtype=float)
    q[:,0]=1-pbin
    for j,k in enumerate(classes):q[:,ALL.index(k)]=scale*pbin*probs[:,j]
    return np.asarray(ALL,dtype=object)[q.argmax(axis=1)]


print('\nVALIDACIÓN DE HMT-V1/V2/V3/V4 (solo TRAIN)')
search_b=[];search_m=[]
for name in ['HMT-V1','HMT-V2','HMT-V3','HMT-V4']:
    settings=WEIGHTS if name=='HMT-V4' else [(1,.75,.3,.25)]
    for wi,weights in enumerate(settings):
        ft=Features(name,weights).fit(dev.text);X=ft.transform(dev.text);V=ft.transform(val.text)
        mask=dev.binary.to_numpy()==1
        for C in CS:
            for balance in BALANCES:
                mb=make_clf('LogReg',X,C,balance).fit(X,dev.binary)
                pb=prob_pos(mb,V)
                for threshold in THRESHOLDS:
                    sc=score(val.binary,(pb>=threshold).astype(int))
                    search_b.append({'Version':name,'Pesos_ID':wi,'C':C,'Balance':str(balance),'Umbral':float(threshold),**sc})
                mm=make_clf('LogReg',X,C,balance).fit(X[mask],dev.loc[mask,'violence'])
                real=val.binary.to_numpy()==1
                sc=score(val.loc[real,'violence'],mm.predict(V[real]))
                search_m.append({'Version':name,'Pesos_ID':wi,'C':C,'Balance':str(balance),**sc})
        print('Validado',name,'pesos',wi)
        del ft,X,V;gc.collect()
sb=pd.DataFrame(search_b).sort_values(['F1_macro','Recall_macro'],ascending=False)
sm=pd.DataFrame(search_m).sort_values(['F1_macro','Recall_macro'],ascending=False)
sb.to_csv(OUT/'BUSQUEDA_HMT_BINARIA_VALID.csv',index=False)
sm.to_csv(OUT/'BUSQUEDA_HMT_MULTICLASE_VALID.csv',index=False)
cb=sb.iloc[0].to_dict();cm=sm.iloc[0].to_dict()
print('\nMEJOR BINARIA (validación):',cb)
print('MEJOR MULTICLASE (validación):',cm)


fb=Features(cb['Version'],WEIGHTS[int(cb['Pesos_ID'])] if cb['Version']=='HMT-V4' else (1,.75,.3,.25)).fit(dev.text)
fm=Features(cm['Version'],WEIGHTS[int(cm['Pesos_ID'])] if cm['Version']=='HMT-V4' else (1,.75,.3,.25)).fit(dev.text)
xb=fb.transform(dev.text);vb=fb.transform(val.text)
xm=fm.transform(dev.text);vm=fm.transform(val.text)
mb=make_clf('LogReg',xb,float(cb['C']),None if cb['Balance']=='None' else 'balanced').fit(xb,dev.binary)
msk=dev.binary.to_numpy()==1
mm=make_clf('LogReg',xm,float(cm['C']),None if cm['Balance']=='None' else 'balanced').fit(xm[msk],dev.loc[msk,'violence'])
val_pb=prob_pos(mb,vb);val_pm=mm.predict_proba(vm)
scales=np.linspace(.6,2.8,23)
scale=max(scales,key=lambda s:score(val.violence,joint(val_pb,val_pm,mm.classes_,s))['F1_macro'])
print('Escala conjunta elegida solo en validación:',scale)


def refit(chosen,target):
    rep=chosen['Version'];weights=WEIGHTS[int(chosen['Pesos_ID'])] if rep=='HMT-V4' else (1,.75,.3,.25)
    ft=Features(rep,weights).fit(train.text)
    X=ft.transform(train.text)
    y=train.binary.to_numpy() if target=='binary' else train.loc[train.binary==1,'violence'].to_numpy()
    X=X if target=='binary' else X[train.binary.to_numpy()==1]
    model=make_clf('LogReg',X,float(chosen['C']),None if chosen['Balance']=='None' else 'balanced').fit(X,y)
    return ft,model
fb,bin_model=refit(cb,'binary');fm,multi_model=refit(cm,'multiclass')
PB=prob_pos(bin_model,fb.transform(test.text)); PM=multi_model.predict_proba(fm.transform(test.text))
P_BIN=(PB>=float(cb['Umbral'])).astype(int)
P_MULT=np.asarray(multi_model.classes_,dtype=object)[PM.argmax(axis=1)]
P_CASC=hard(PB,P_MULT,float(cb['Umbral']))
P_JOINT=joint(PB,PM,multi_model.classes_,scale)
true_b=test.binary.to_numpy(); true_m=test.violence.to_numpy(); sex_test=true_b==1


models=['LogReg','SVM','KNN','NaiveBayes','MLP','RandomForest','DecisionTree'] if RUN_ALL_CLASSIFIERS else ['LogReg','SVM']
reps=['BoW','TF-IDF','N-Grams','TF-IDF caracteres','LSA','Lexical','HMT-V1','HMT-V2','HMT-V3','HMT-V4']
preds={}; rows=[]
for rep in reps:
    print('Evaluando',rep,flush=True)
    f=Features(rep).fit(train.text); X=f.transform(train.text);T=f.transform(test.text)
    for mdl in models:
        try:
            bb=make_clf(mdl,X)
            if bb is None:continue
            bb.fit(X,true_b if False else train.binary.to_numpy())
            pb=bb.predict(T)
            multi_train=train.binary.to_numpy()==1
            mm=make_clf(mdl,X[multi_train])
            if mm is None:continue
            mm.fit(X[multi_train],train.loc[multi_train,'violence'])
            pm=mm.predict(T)
            pc=np.where(pb==1,pm,NO)
            preds[(rep,mdl,'bin')]=pb
            preds[(rep,mdl,'multi')]=pm
            preds[(rep,mdl,'casc')]=pc
            for task,yy,pp in [('Binaria',true_b,pb),('Multiclase condicional',true_m[sex_test],pm[sex_test]),('Cascada',true_m,pc)]:
                rows.append({'Tarea':task,'Representacion':rep,'Clasificador':mdl,**score(yy,pp)})
        except Exception as e:
            print('  ERROR',rep,mdl,type(e).__name__,str(e)[:130]);continue
    del f,X,T;gc.collect()


proposed=[('HMT-SELECCIONADO','Binaria',true_b,P_BIN),('HMT-SELECCIONADO','Multiclase condicional',true_m[sex_test],P_MULT[sex_test]),('HMT-SELECCIONADO','Cascada',true_m,P_CASC),('HMT-CONJUNTO','Cascada',true_m,P_JOINT)]
for model,task,yy,p in proposed:rows.append({'Tarea':task,'Representacion':model,'Clasificador':'HMT',**score(yy,p)})
preds[('HMT-SELECCIONADO','HMT','bin')]=P_BIN
preds[('HMT-SELECCIONADO','HMT','multi')]=P_MULT
preds[('HMT-SELECCIONADO','HMT','casc')]=P_CASC
preds[('HMT-CONJUNTO','HMT','casc')]=P_JOINT



if RUN_TRANSFORMERS:
    if importlib.util.find_spec('transformers') is None or importlib.util.find_spec('accelerate') is None:
        subprocess.check_call([sys.executable,'-m','pip','-q','install','transformers>=4.45,<5','accelerate>=0.30','sentencepiece'])
    import torch
    from torch.utils.data import Dataset,DataLoader
    from transformers import AutoTokenizer,AutoModelForSequenceClassification,DataCollatorWithPadding
    import torch.nn.functional as F
    device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    if device.type=='cpu':print('AVISO: Transformers sin GPU tardarán mucho. En Colab selecciona Entorno de ejecución > Cambiar tipo > GPU.')
    class TextDataset(Dataset):
        def __init__(self,texts,labels,tokenizer):self.enc=tokenizer(list(map(str,texts)),truncation=True,max_length=MAX_TOKENS);self.labels=np.asarray(labels,dtype=np.int64)
        def __len__(self):return len(self.labels)
        def __getitem__(self,i):return {**{k:v[i] for k,v in self.enc.items()},'labels':int(self.labels[i])}
    def infer(model,tokenizer,texts,batch):
        
        ds=TextDataset(texts,np.zeros(len(texts),dtype=int),tokenizer)
        loader=DataLoader(ds,batch_size=batch,shuffle=False,collate_fn=DataCollatorWithPadding(tokenizer,return_tensors='pt'))
        probs=[];model.eval()
        with torch.no_grad():
            for v in loader:
                v={k:x.to(device) for k,x in v.items() if k!='labels'}
                probs.append(F.softmax(model(**v).logits,dim=-1).cpu().numpy())
        return np.concatenate(probs)
    def finetune(model_id,task,labels,dev_labels,val_labels,test_texts):
        
        tokenizer=AutoTokenizer.from_pretrained(model_id)
        model=AutoModelForSequenceClassification.from_pretrained(model_id,num_labels=len(labels),ignore_mismatched_sizes=True).to(device)
        mask_d=(dev.binary.to_numpy()==1) if task=='multi' else np.ones(len(dev),bool)
        mask_v=(val.binary.to_numpy()==1) if task=='multi' else np.ones(len(val),bool)
        xtrain=dev.loc[mask_d,'text'].to_numpy();xval=val.loc[mask_v,'text'].to_numpy()
        ytrain=dev_labels[mask_d]; yval=val_labels[mask_v]
        dset=TextDataset(xtrain,ytrain,tokenizer)
        loader=DataLoader(dset,batch_size=TRANSFORMER_BATCH,shuffle=True,collate_fn=DataCollatorWithPadding(tokenizer,return_tensors='pt'))
        opt=torch.optim.AdamW(model.parameters(),lr=2e-5,weight_decay=.01)
        best=-1;best_state=None
        for epoch in range(TRANSFORMER_EPOCHS):
            model.train(); losses=[]
            for batch in loader:
                batch={k:v.to(device) for k,v in batch.items()}
                opt.zero_grad();loss=model(**batch).loss;loss.backward();torch.nn.utils.clip_grad_norm_(model.parameters(),1.0);opt.step();losses.append(float(loss.item()))
            vprob=infer(model,tokenizer,xval,TRANSFORMER_BATCH)
            vf=f1_score(yval,vprob.argmax(axis=1),average='macro',zero_division=0)
            print(f'{model_id} {task} epoch={epoch+1} val_f1={vf:.4f} loss={np.mean(losses):.4f}',flush=True)
            if vf>best:
                best=vf;best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
        model.load_state_dict(best_state);model.to(device)
        outdir=OUT/f'{model_id.replace("/","_")}_{task}';outdir.mkdir(exist_ok=True)
        model.save_pretrained(outdir);tokenizer.save_pretrained(outdir)
        result=infer(model,tokenizer,test_texts,TRANSFORMER_BATCH)
        del model,best_state;gc.collect()
        if torch.cuda.is_available():torch.cuda.empty_cache()
        return result
    
    for nickname,modelid in [('BETO','dccuchile/bert-base-spanish-wwm-cased'),('XLM-R','xlm-roberta-base')]:
        print('\nTRANSFORMER:',nickname,flush=True)
        
        try:
            yt=dev.binary.to_numpy();yv=val.binary.to_numpy()
            pb=finetune(modelid,'bin',['No sexista','Sexista'],yt,yv,test.text.to_numpy())[:,1]
            mt=dev.violence.map({v:i for i,v in enumerate(TYPES)}).fillna(-1).to_numpy().astype(int)
            mv=val.violence.map({v:i for i,v in enumerate(TYPES)}).fillna(-1).to_numpy().astype(int)
            pm=finetune(modelid,'multi',TYPES,mt,mv,test.text.to_numpy())
            pred_bin=(pb>=.5).astype(int)
            pred_multi=np.asarray(TYPES,dtype=object)[pm.argmax(axis=1)]
            pred_casc=np.where(pred_bin==1,pred_multi,NO)
            preds[(nickname,'Transformer','bin')]=pred_bin
            preds[(nickname,'Transformer','multi')]=pred_multi
            preds[(nickname,'Transformer','casc')]=pred_casc
            for task,yy,pp in [('Binaria',true_b,pred_bin),('Multiclase condicional',true_m[sex_test],pred_multi[sex_test]),('Cascada',true_m,pred_casc)]:
                rows.append({'Tarea':task,'Representacion':nickname,'Clasificador':'Transformer',**score(yy,pp)})
        except Exception as e:
            print('No se pudo terminar',nickname,':',type(e).__name__,str(e)[:400]);continue


res=pd.DataFrame(rows)
for task,filename in [('Binaria','01_BINARIA'),('Multiclase condicional','02_MULTICLASE'),('Cascada','03_CASCADA')]:
    tab=res[res.Tarea==task].sort_values('F1_macro',ascending=False).reset_index(drop=True)
    tab.insert(0,'Posicion',np.arange(1,len(tab)+1))
    tab.to_csv(OUT/f'{filename}.csv',index=False,encoding='utf-8-sig')
    try:tab.style.format({'F1_macro':'{:.3f}','Precision_macro':'{:.3f}','Recall_macro':'{:.3f}','Accuracy':'{:.3f}'}).background_gradient(subset=['F1_macro'],cmap='Blues').to_html(OUT/f'{filename}.html')
    except Exception:pass
    print('\n',task.upper(),'— TOP 12');print(tab[['Posicion','Representacion','Clasificador','F1_macro','Accuracy']].head(12).round(3).to_string(index=False))
res.to_csv(OUT/'TODOS_RESULTADOS.csv',index=False)


abl=[]
for i,w in enumerate([(1,0,0,0),(1,1,0,0),(1,1,.25,0),(1,1,0,.25),(1,1,.25,.25)]):
    f=Features('HMT-V4',w).fit(train.text); X=f.transform(train.text);T=f.transform(test.text)
    for task,y,ytest,mask in [('Binaria',train.binary.to_numpy(),true_b,np.ones(len(train),bool)),('Multiclase condicional',train.loc[train.binary==1,'violence'].to_numpy(),true_m[sex_test],train.binary.to_numpy()==1)]:
        mm=LogisticRegression(C=1,class_weight='balanced',max_iter=3000,random_state=SEED).fit(X[mask],y)
        predictions=mm.predict(T if task=='Binaria' else T[sex_test]);abl.append({'Componentes':str(w),'Tarea':task,**score(ytest,predictions)})
    del f,X,T;gc.collect()
pd.DataFrame(abl).to_csv(OUT/'04_ABLACION_HMT_V4.csv',index=False)


def save_cm(y,p,labels,title):
    for mode in [None,'true']:
        fig,ax=plt.subplots(figsize=(max(6,1.7*len(labels)),max(5,1.2*len(labels))))
        mat=confusion_matrix(y,p,labels=labels,normalize=mode)
        ConfusionMatrixDisplay(mat,display_labels=labels).plot(ax=ax,cmap='Blues',colorbar=False,values_format='.2f' if mode else 'd')
        ax.set_title(title+(' (recall por clase)' if mode else ' (conteos)'))
        ax.tick_params(axis='x',labelrotation=35);plt.tight_layout()
        fig.savefig(OUT/f'MATRIZ_{title.replace(" ","_")}_{"normalizada" if mode else "absoluta"}.png',dpi=220)
        plt.show();plt.close(fig)
save_cm(true_b,P_BIN,[0,1],'HMT_binaria')
save_cm(true_m[sex_test],P_MULT[sex_test],TYPES,'HMT_multiclase_condicional')
save_cm(true_m,P_CASC,ALL,'HMT_cascada')
save_cm(true_m,P_JOINT,ALL,'HMT_probabilistica')


rng=np.random.default_rng(SEED)
def stats_pair(y,ref,other):
    y=np.asarray(y);ref=np.asarray(ref);other=np.asarray(other)
    ar=ref==y;ao=other==y
    b=int((ar&~ao).sum());c=int((~ar&ao).sum())
    count=[[int((ar&ao).sum()),b],[c,int((~ar&~ao).sum())]]
    p=float(mcnemar(count,exact=b+c<25,correction=b+c>=25).pvalue)
    indices=rng.integers(0,len(y),size=(BOOTSTRAP_REPS,len(y)))
    diffs=np.array([f1_score(y[ix],ref[ix],average='macro',zero_division=0)-f1_score(y[ix],other[ix],average='macro',zero_division=0) for ix in indices])
    return {'HMT_F1':f1_score(y,ref,average='macro',zero_division=0),'Base_F1':f1_score(y,other,average='macro',zero_division=0),'Delta_F1':f1_score(y,ref,average='macro',zero_division=0)-f1_score(y,other,average='macro',zero_division=0),'IC95_inf':np.quantile(diffs,.025),'IC95_sup':np.quantile(diffs,.975),'Solo_HMT_acierta':b,'Solo_base_acierta':c,'p_McNemar':p}
stat=[]
for task,y,ref,key in [('Binaria',true_b,P_BIN,'bin'),('Multiclase condicional',true_m[sex_test],P_MULT[sex_test],'multi'),('Cascada',true_m,P_JOINT,'casc')]:
    choices=res[(res.Tarea==task)&(~res.Representacion.isin(['HMT-SELECCIONADO','HMT-CONJUNTO']))].sort_values('F1_macro',ascending=False)
    
    selected=choices[choices.Representacion.isin(['TF-IDF','TF-IDF caracteres','N-Grams','LSA','BoW'])].head(5)
    selected=pd.concat([selected,choices[choices.Representacion.isin(['BETO','XLM-R'])]])
    for _,rr in selected.iterrows():
        pred=preds.get((rr.Representacion,rr.Clasificador,key))
        if pred is None:continue
        if key=='multi':pred=np.asarray(pred)[sex_test]
        stat.append({'Tarea':task,'Base':rr.Representacion,'Clasificador':rr.Clasificador,**stats_pair(y,ref,pred)})
stat=pd.DataFrame(stat)
if len(stat):
    stat['p_Holm']=np.nan
    for task,ix in stat.groupby('Tarea').groups.items():stat.loc[ix,'p_Holm']=multipletests(stat.loc[ix,'p_McNemar'],method='holm')[1]
    stat['HMT_supera_en_aciertos_significativamente']=(stat.p_Holm<.05)&(stat.Solo_HMT_acierta>stat.Solo_base_acierta)
    stat.to_csv(OUT/'05_MCNEMAR_Y_BOOTSTRAP.csv',index=False)
    print('\nCOMPARACIÓN ESTADÍSTICA');print(stat.round(4).to_string(index=False))

predout=test[['id','text']].copy() if 'id' in test.columns else test[['text']].copy()
predout['task1_real']=test.task1;predout['tipo_violencia_real']=true_m
predout['HMT_task1_pred']=np.where(P_BIN==1,'sexist','non-sexist')
predout['HMT_prob_sexista']=PB
predout['HMT_tipo_condicional']=P_MULT;predout['HMT_cascada']=P_CASC;predout['HMT_conjunta']=P_JOINT
predout.to_csv(OUT/'06_PREDICCIONES_TEST.csv',index=False,encoding='utf-8-sig')
dump({'features_bin':fb,'model_bin':bin_model,'features_multi':fm,'model_multi':multi_model,'threshold':float(cb['Umbral']),'scale':float(scale),'label_order':ALL},OUT/'HMT_SELECCIONADO.joblib')
print('\nTERMINADO. Archivos:',OUT.resolve())
print('Aviso metodológico: los modelos Transformers se seleccionan con validación de TRAIN; el TEST únicamente se predice. Comparar modelos mirando TEST no debe usarse para decidir nuevos ajustes.')


### EXP-004 · V1, V2, V3, V4

**Archivo de origen:** `27_Octubre.ipynb, celda 1`. **Referencias duplicadas:** 0. **Sintaxis de código:** Válida.

-*- coding: utf-8 -*-
"""EXIST2021 — MAESTRO TASK 1 BINARIA (una sola celda Colab).
Versiones históricas aproximadas a sus códigos y una variante experimental HMVT-Nueva.
SELECCIÓN: solo train interno; EVALUACIÓN: 20% de train reservado, no

**Ejecución:** celda independiente o variante experimental; revisar archivos de entrada y dependencias antes de ejecutarla. Los resultados originales no se insertan en esta celda.

In [ ]:
"""EXIST2021 — MAESTRO TASK 1 BINARIA (una sola celda Colab).
Versiones históricas aproximadas a sus códigos y una variante experimental HMVT-Nueva.
SELECCIÓN: solo train interno; EVALUACIÓN: 20% de train reservado, no usado para ajustar.
TEST histórico deliberadamente no se abre. Comparaciones exploratorias de holdout interno.
"""
import os, re, time, warnings, unicodedata, json, copy
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.sparse import csr_matrix, hstack, issparse
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler, Normalizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import MultinomialNB, GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_recall_fscore_support,
    f1_score, confusion_matrix, ConfusionMatrixDisplay, classification_report)
from statsmodels.stats.contingency_tables import mcnemar
from statsmodels.stats.multitest import multipletests
import joblib

SEED=42
DATA_FILE='EXIST2021_train_reestructurado.csv'
OUT=Path('Resultados_HMVT_MAESTRO_TASK1'); OUT.mkdir(parents=True,exist_ok=True)

BANCO_COMPLETO=True
INCLUIR_XGBOOST=True

OPTIMIZAR_HMVT=True

MODO_PRUEBA=False

REFIT_ENTRENAMIENTO_COMPLETO=True
N_LSA=40 if MODO_PRUEBA else 120
N_LSA_ORIGINAL=50 if MODO_PRUEBA else 150
N_LSA_ULTRA=50 if MODO_PRUEBA else 200
N_BOOT=200 if MODO_PRUEBA else 1500

np.random.seed(SEED)
def norm(s):
    return unicodedata.normalize('NFKD',str(s)).encode('ascii','ignore').decode().casefold().strip()

def localizar(path):
    options=[Path(path),Path('/content')/path,Path('/mnt/data')/path]
    for p in options:
        if p.exists():return p
    try:
        from google.colab import files
        print('Selecciona:',path);files.upload()
        if Path(path).exists():return Path(path)
    except ImportError:pass
    raise FileNotFoundError(f'No se encuentra {path}')

def load_dataset():
    df=pd.read_csv(localizar(DATA_FILE),encoding='utf-8-sig')
    if not {'text','task1'}.issubset(df.columns):
        raise ValueError('Se requieren columnas text y task1 (sexist/non-sexist)')
    if df[['text','task1']].isna().any().any():
        raise ValueError('Hay textos o etiquetas vacíos; revisa el CSV')
    df=df.copy();df['text']=df['text'].astype(str).str.replace(r'\s+',' ',regex=True).str.strip()
    if (df.text.str.len()==0).any():raise ValueError('Hay textos vacíos')
    labels={'sexist':1,'sexista':1,'non-sexist':0,'non sexist':0,'no sexista':0,'no-sexista':0}
    df['y']=df.task1.map(lambda t:labels.get(norm(t),-1))
    if (df.y<0).any():raise ValueError('Etiquetas task1 no reconocidas: '+str(df.loc[df.y<0,'task1'].unique()))
    if df.y.nunique()!=2:raise ValueError('Task1 requiere ambas clases')
    if 'id' not in df: df['id']=np.arange(len(df))
    return df.reset_index(drop=True)

OLD_CUES={
 'insultos': ['puta','zorra','gorda','imbecil','estupida','asquerosa','inutil','tarada','idiota','prostituta','calienta','retrasada'],
 'imperativos':['callate','cocina','fregar','lugar','deberian','deben','debes','vete','largo','fuera','salvate','limpia','atiende','sirvenos','sufre'],
 'desprecio':['asco','basura','ridicula','falsa','inutiles','patetica','estupidez','ridiculos','inservible','barata','mediocre'],
 'estereotipos':['mujeres','feminazi','ellas','esposa','madre','novia','ninita','sola','callada','emocionales','debiles','casa']}
NEW_CUES={**OLD_CUES,'amenaza':['matar','golpear','pegar','violar','amenaza','castigar'],
          'sexual':['sexo','sexual','desnuda','tetas','pene','vagina','follar']}

def pragmatic(texts,style='new'):
    cues=OLD_CUES if style=='old' else NEW_CUES
    rows=[]
    for s in texts:
        t=norm(s); words=re.findall(r'\b\w+\b',t); counts={w:words.count(w) for w in set(words)}
        counts_c=[sum(counts.get(w,0) for w in group) for group in cues.values()]
        n=max(len(s),1)
        rest=[np.log1p(len(words)),len(set(words))/max(1,len(words)),sum(c.isupper() for c in s)/n,
              np.log1p(s.count('!')),np.log1p(s.count('?')),int('http' in t),int('@' in s),int('#' in s)]
        rows.append(counts_c+rest)
    return np.asarray(rows,dtype=np.float32)




WEIGHTS={
 'TF-IDF':('word',), 'N-gramas':('count',), 'BoW':('bow',), 'TF-IDF caracteres':('char',),
 'LSA':('lsa',), 'Léxico':('prag',),
 'HMT-V1':(1.,0.,.35,0.), 'HMT-V2':(1.,0.,.35,.25),
 'HMT-V3':(1.,.75,.35,.25), 'HMT-V4':(1.,.5,.20,.15),
 'HMT-V1 Optimizado':('oldopt',6.,10.),
 'HMT-V1 Ultra-Optimizado':('oldultra',8.,12.),
 'HMVT-Nueva':(1.,1.,.15,.2),
 'Ablación sin caracteres':(1.,0.,.15,.2),
 'Ablación sin LSA':(1.,1.,0.,.2),
 'Ablación sin discursivas':(1.,1.,.15,0.),
}

class FeatureBank:
    """Aprende todas las transformaciones exclusivamente sobre su training fold."""
    def fit(self,texts):
        self.word=TfidfVectorizer(max_features=18000 if MODO_PRUEBA else 30000,ngram_range=(1,2),min_df=2,sublinear_tf=True,strip_accents='unicode')
        self.char=TfidfVectorizer(analyzer='char_wb',max_features=20000 if MODO_PRUEBA else 35000,ngram_range=(3,5),min_df=2,sublinear_tf=True)
        self.count=CountVectorizer(max_features=5000 if MODO_PRUEBA else 10000,ngram_range=(1,2),min_df=2,strip_accents='unicode')
        self.bow=CountVectorizer(max_features=5000 if MODO_PRUEBA else 10000,ngram_range=(1,1),min_df=2,strip_accents='unicode')
        self.oldword=TfidfVectorizer(max_features=15000,ngram_range=(1,3),min_df=2,sublinear_tf=True)
        self.ultraword=TfidfVectorizer(max_features=20000,ngram_range=(1,3),min_df=2,sublinear_tf=True)
        w=self.word.fit_transform(texts); self.char.fit(texts);self.count.fit(texts);self.bow.fit(texts)
        wo=self.oldword.fit_transform(texts);wu=self.ultraword.fit_transform(texts)
        def svd(x,n):
            k=min(n,x.shape[0]-1,x.shape[1]-1)
            return TruncatedSVD(n_components=k,random_state=SEED).fit(x) if k>=2 else None
        self.lsa=svd(w,N_LSA);self.oldlsa=svd(wo,N_LSA_ORIGINAL);self.ultralsa=svd(wu,N_LSA_ULTRA)
        self.ps=StandardScaler().fit(pragmatic(texts))
        self.oldps=StandardScaler().fit(pragmatic(texts,'old'))
        self.oldls=StandardScaler().fit(self.oldlsa.transform(wo)) if self.oldlsa else None
        self.ultrals=StandardScaler().fit(self.ultralsa.transform(wu)) if self.ultralsa else None
        return self
    def transform(self,texts):
        w=self.word.transform(texts).tocsr().astype('float32')
        c=self.char.transform(texts).tocsr().astype('float32')
        ct=self.count.transform(texts).tocsr().astype('float32')
        bow=self.bow.transform(texts).tocsr().astype('float32')
        l=csr_matrix(Normalizer().fit_transform(self.lsa.transform(w)).astype('float32')) if self.lsa else csr_matrix((len(texts),0),dtype=np.float32)
        p=csr_matrix(Normalizer().fit_transform(self.ps.transform(pragmatic(texts))).astype('float32'))
        wo=self.oldword.transform(texts).tocsr().astype('float32');wu=self.ultraword.transform(texts).tocsr().astype('float32')
        lo=csr_matrix(self.oldls.transform(self.oldlsa.transform(wo)).astype('float32')) if self.oldlsa else csr_matrix((len(texts),0),dtype=np.float32)
        lu=csr_matrix(self.ultrals.transform(self.ultralsa.transform(wu)).astype('float32')) if self.ultralsa else csr_matrix((len(texts),0),dtype=np.float32)
        po=csr_matrix(self.oldps.transform(pragmatic(texts,'old')).astype('float32'))
        return {'word':w,'char':c,'count':ct,'bow':bow,'lsa':l,'prag':p,'oldword':wo,'ultraword':wu,'oldlsa':lo,'ultralsa':lu,'oldprag':po}

def represent(blocks,version,override=None):
    weights=WEIGHTS[version] if override is None else override
    if len(weights)==1:return blocks[weights[0]]
    if isinstance(weights[0],str):
        key=weights[0];word='oldword' if key=='oldopt' else 'ultraword'
        lsa='oldlsa' if key=='oldopt' else 'ultralsa'
        return hstack([blocks[word],blocks[lsa]*weights[1],blocks['oldprag']*weights[2]],format='csr',dtype=np.float32)
    a,b,c,d=weights
    return hstack([blocks['word']*a,blocks['char']*b,blocks['lsa']*c,blocks['prag']*d],format='csr',dtype=np.float32)

def metrics(y,p):
    pr,re,fs,support=precision_recall_fscore_support(y,p,labels=[0,1],zero_division=0)
    d={'Precision_macro':np.mean(pr),'Recall_macro':np.mean(re),'F1_macro':np.mean(fs),
       'Precision_weighted':np.average(pr,weights=support),'Recall_weighted':np.average(re,weights=support),
       'F1_weighted':np.average(fs,weights=support),'Accuracy':accuracy_score(y,p),
       'Balanced_accuracy':balanced_accuracy_score(y,p),'F1_sexista':fs[1],
       'Precision_sexista':pr[1],'Recall_sexista':re[1], 'Specificity_sexista':re[0],
       'F1_no_sexista':fs[0],'Precision_no_sexista':pr[0],'Recall_no_sexista':re[0]}
    return {k:float(v) for k,v in d.items()}

def model_factory(name,X):
    if name=='LogReg':return LogisticRegression(max_iter=2500,class_weight='balanced',C=1.,random_state=SEED)
    if name=='SVM':return LinearSVC(C=1.,class_weight='balanced',max_iter=6000,random_state=SEED)
    if name=='3-NN':return KNeighborsClassifier(n_neighbors=3,weights='distance')
    if name=='5-NN':return KNeighborsClassifier(n_neighbors=5,weights='distance')
    if name=='NaiveBayes':
        if issparse(X) and (not X.data.size or np.min(X.data)>=0):return MultinomialNB()
        
        if X.shape[1]<=300:return GaussianNB()
        return None
    if name=='MLP':return MLPClassifier(hidden_layer_sizes=(64,),alpha=.001,max_iter=220,early_stopping=True,random_state=SEED)
    if name=='RandomForest':return RandomForestClassifier(n_estimators=150,class_weight='balanced',n_jobs=-1,random_state=SEED)
    if name=='DecisionTree':return DecisionTreeClassifier(max_depth=18,class_weight='balanced',random_state=SEED)
    if name=='XGBoost':
        try:
            from xgboost import XGBClassifier
            return XGBClassifier(n_estimators=130,max_depth=4,learning_rate=.07,subsample=.9,colsample_bytree=.85,
                                 tree_method='hist',eval_metric='logloss',n_jobs=2,random_state=SEED)
        except ImportError:return None
    return None

def adapted_fit_predict(model,X,y,Xv):
    if isinstance(model,GaussianNB):
        return model.fit(X.toarray() if issparse(X) else X,y).predict(Xv.toarray() if issparse(Xv) else Xv)
    return model.fit(X,y).predict(Xv)

def tune_new(bankdev,banktune,dev,tune):
    """Seleccionar arquitecturas + C + umbral SOLO en validación de tuning."""
    results=[];best=None
    y=dev.y.to_numpy();yv=tune.y.to_numpy()
    weight_grid=[(1.,a,b,c) for a in (.5,1.,1.5) for b,c in ((.1,.1),(.2,.2),(.35,.2))]
    if MODO_PRUEBA:weight_grid=weight_grid[:2]
    for weights in weight_grid:
        X=represent(bankdev,'HMVT-Nueva',weights);V=represent(banktune,'HMVT-Nueva',weights)
        for model in ('LogReg','SVM'):
            for C in ((.3,1.,3.) if not MODO_PRUEBA else (1.,)):
                clf=LogisticRegression(C=C,class_weight='balanced',max_iter=2500,random_state=SEED) if model=='LogReg' else LinearSVC(C=C,class_weight='balanced',max_iter=6000,random_state=SEED)
                clf.fit(X,y)
                if model=='LogReg':
                    score=clf.predict_proba(V)[:,list(clf.classes_).index(1)]
                    cutoffs=(.35,.4,.45,.5,.55,.6) if not MODO_PRUEBA else (.45,.5,.55)
                    preds=[(t,(score>=t).astype(int)) for t in cutoffs]
                else:preds=[(None,clf.predict(V))]
                for threshold,p in preds:
                    m=metrics(yv,p)
                    row={'Peso_palabras':weights[0],'Peso_caracteres':weights[1],'Peso_LSA':weights[2],
                         'Peso_discursivo':weights[3],'Clasificador':model,'C':C,'Umbral':threshold,**m}
                    results.append(row)
                    key=(m['F1_macro'],m['Balanced_accuracy'],m['Recall_sexista'])
                    if best is None or key>best[0]:best=(key,dict(row),weights)
    pd.DataFrame(results).sort_values('F1_macro',ascending=False).to_csv(OUT/'01_BUSQUEDA_HMVT_SOLO_DESARROLLO.csv',index=False)
    print('HMVT seleccionado en desarrollo:',{k:v for k,v in best[1].items() if k in ('Clasificador','C','Umbral','F1_macro','Recall_sexista','Peso_caracteres','Peso_LSA','Peso_discursivo')})
    return best[1],best[2]

def choose_models():
    names=['LogReg','SVM','3-NN','5-NN','NaiveBayes','MLP','XGBoost','RandomForest','DecisionTree']
    if not BANCO_COMPLETO:names=['LogReg','SVM']
    if not INCLUIR_XGBOOST:names=[n for n in names if n!='XGBoost']
    return names

def paired_stats(y,ref,base,boot=N_BOOT):
    a=ref==y;b=base==y
    h=int((a&~b).sum());other=int((~a&b).sum());both=int((a&b).sum());neither=int((~a&~b).sum())
    pv=float(mcnemar([[both,h],[other,neither]],exact=(h+other<25),correction=(h+other>=25)).pvalue)
    rng=np.random.default_rng(SEED);n=len(y);delta=np.empty(boot)
    for i in range(boot):
        idx=rng.integers(0,n,n)
        delta[i]=f1_score(y[idx],ref[idx],average='macro',labels=[0,1],zero_division=0)-f1_score(y[idx],base[idx],average='macro',labels=[0,1],zero_division=0)
    lo,hi=np.quantile(delta,[.025,.975]);return h,other,pv,float(lo),float(hi)

def cm_save(y,p,filename):
    for normalize in (None,'true'):
        cm=confusion_matrix(y,p,labels=[0,1],normalize=normalize)
        fig,ax=plt.subplots(figsize=(5,4));ConfusionMatrixDisplay(cm,display_labels=['No sexista','Sexista']).plot(ax=ax,cmap='Blues',colorbar=False,
                   values_format='d' if normalize is None else '.2f')
        ax.set_title(filename+(' — proporciones' if normalize else ' — conteos'));fig.tight_layout()
        fig.savefig(OUT/(filename+('_normalizada' if normalize else '_absoluta')+'.png'),dpi=160);plt.close(fig)

def main():
    df=load_dataset();print('Datos:',len(df),'| Positivos:',int(df.y.sum()),'| Negativos:',int((df.y==0).sum()))
    
    development, evaluation=train_test_split(df,test_size=.20,stratify=df.y,random_state=SEED)
    dev,tuning=train_test_split(development,test_size=.25,stratify=development.y,random_state=SEED+1)
    development=development.reset_index(drop=True); evaluation=evaluation.reset_index(drop=True)
    dev=dev.reset_index(drop=True);tuning=tuning.reset_index(drop=True)
    print(f'Desarrollo={len(dev)}, tuning={len(tuning)}, evaluación interna reservada={len(evaluation)}')
    chosen=None;chosen_weights=None
    if OPTIMIZAR_HMVT:
        print('Ajustando representaciones en desarrollo...')
        b=FeatureBank().fit(dev.text.to_numpy());a=b.transform(dev.text.to_numpy());v=b.transform(tuning.text.to_numpy())
        chosen,chosen_weights=tune_new(a,v,dev,tuning)
        del a,v,b
    
    print('Ajustando características sobre desarrollo completo...')
    full=FeatureBank().fit(development.text.to_numpy());X=full.transform(development.text.to_numpy());V=full.transform(evaluation.text.to_numpy())
    rep_names=list(WEIGHTS.keys());rep_names.remove('HMVT-Nueva') if OPTIMIZAR_HMVT else None
    models=choose_models();records=[];preds={};skip=[]
    for rep in rep_names:
        xt=represent(X,rep);xv=represent(V,rep)
        for name in models:
            m=model_factory(name,xt)
            if m is None:
                skip.append((rep,name,'NB no compatible con datos negativos/dispersos o dependencia ausente'));continue
            try:
                p=adapted_fit_predict(m,xt,development.y.to_numpy(),xv)
                if np.issubdtype(np.asarray(p).dtype,np.floating):p=p.astype(int)
                preds[(rep,name)]=np.asarray(p,dtype=int)
                records.append({'Representacion':rep,'Clasificador':name,'Tipo':'Referencia',**metrics(evaluation.y,p)})
            except Exception as exc:
                warnings.warn(f'{rep} / {name}: {exc}');skip.append((rep,name,str(exc)))
        print('Evaluada:',rep,flush=True)
    if OPTIMIZAR_HMVT:
        rep='HMVT-Nueva (seleccionada)'; xt=represent(X,'HMVT-Nueva',chosen_weights);xv=represent(V,'HMVT-Nueva',chosen_weights)
        name=chosen['Clasificador'];C=float(chosen['C'])
        m=LogisticRegression(C=C,class_weight='balanced',max_iter=2500,random_state=SEED) if name=='LogReg' else LinearSVC(C=C,class_weight='balanced',max_iter=6000,random_state=SEED)
        m.fit(xt,development.y.to_numpy())
        p=(m.predict_proba(xv)[:,list(m.classes_).index(1)]>=chosen['Umbral']).astype(int) if name=='LogReg' else m.predict(xv)
        preds[(rep,name)]=p;records.append({'Representacion':rep,'Clasificador':name,'Tipo':'Seleccionado en tuning',**metrics(evaluation.y,p)})
    master=pd.DataFrame(records).sort_values(['F1_macro','Balanced_accuracy'],ascending=False).reset_index(drop=True)
    master.insert(0,'Posicion',range(1,len(master)+1))
    master.to_csv(OUT/'02_TABLA_MAESTRA_COMPLETA.csv',index=False,encoding='utf-8-sig')
    print('\n================ TABLA MAESTRA COMPLETA ================\n',master.to_string(index=False,float_format=lambda z:f'{z:.4f}'))
    winner=master.iloc[0];bestkey=(winner.Representacion,winner.Clasificador)
    print('\nMEJOR F1 OBSERVADO EN EVALUACIÓN INTERNA:',bestkey,float(winner.F1_macro))
    
    stats=[]
    selected=(('HMVT-Nueva (seleccionada)',chosen['Clasificador']) if chosen else ('HMT-V3','LogReg'))
    if selected not in preds:selected=bestkey
    for key,p in preds.items():
        if key==selected:continue
        ref=preds[selected];y=evaluation.y.to_numpy();h,o,pv,lo,hi=paired_stats(y,ref,p)
        stats.append({'Referencia':str(selected),'Comparado':str(key),'F1_referencia':metrics(y,ref)['F1_macro'],
                      'F1_comparado':metrics(y,p)['F1_macro'],'Delta_F1':metrics(y,ref)['F1_macro']-metrics(y,p)['F1_macro'],
                      'Solo_HMVT_acierta':h,'Solo_comparado_acierta':o,'p_McNemar':pv,'IC95_DeltaF1_inf':lo,'IC95_DeltaF1_sup':hi})
    sd=pd.DataFrame(stats)
    if len(sd):
        sd['p_Holm']=multipletests(sd.p_McNemar,method='holm')[1]
        sd['HMVT_supera_en_accuracy_con_Holm']=(sd.p_Holm<.05)&(sd.Solo_HMVT_acierta>sd.Solo_comparado_acierta)
        sd['IC95_F1_excluye_cero']=((sd.IC95_DeltaF1_inf>0)|(sd.IC95_DeltaF1_sup<0))
        sd.to_csv(OUT/'04_MCNEMAR_HOLM_Y_BOOTSTRAP.csv',index=False,encoding='utf-8-sig')
        print('\n================ MCNEMAR-HOLM ================\n',sd.to_string(index=False,float_format=lambda z:f'{z:.5f}'))
    per=[]
    for (rep,name),p in preds.items():
        yt=evaluation.y.to_numpy();pr,rc,f1,s=precision_recall_fscore_support(yt,p,labels=[0,1],zero_division=0)
        for i,cl in enumerate(['No sexista','Sexista']):per.append({'Representacion':rep,'Clasificador':name,'Clase':cl,
                'Precision':pr[i],'Recall':rc[i],'F1':f1[i],'Support':int(s[i])})
    pd.DataFrame(per).to_csv(OUT/'03_METRICAS_POR_CLASE.csv',index=False,encoding='utf-8-sig')
    pd.DataFrame(skip,columns=['Representacion','Clasificador','Motivo']).to_csv(OUT/'07_COMBINACIONES_OMITIDAS.csv',index=False)
    for tag,key in [('HMVT_REFERENCIA',selected),('MEJOR_F1_DESCRIPTIVO',bestkey)]:
        cm_save(evaluation.y.to_numpy(),preds[key],tag)
        print('\nREPORTE',tag,key,'\n',classification_report(evaluation.y.to_numpy(),preds[key],target_names=['No sexista','Sexista'],zero_division=0))
    pd.DataFrame({'id':evaluation.id,'texto':evaluation.text,'real':evaluation.y,
      'pred_hmvt_referencia':preds[selected],'pred_mejor_f1_descriptivo':preds[bestkey]}).to_csv(OUT/'05_PREDICCIONES_EVALUACION.csv',index=False)
    config={'modelo_seleccionado':list(selected),'pesos_hmvt':list(chosen_weights) if chosen_weights else None,
            'hiperparametros':chosen,'datos':{'entrenamiento_total':len(df),'desarrollo':len(development),'evaluacion':len(evaluation)},
            'nota':'Test histórico NO utilizado. La referencia se seleccionó en tuning, no en evaluación.'}
    (OUT/'06_CONFIGURACION.json').write_text(json.dumps(config,indent=2,ensure_ascii=False,default=float),encoding='utf-8')
    try:
        with pd.ExcelWriter(OUT/'RESULTADOS_MAESTROS_TASK1.xlsx') as writer:
            master.to_excel(writer,'Tabla_maestra',index=False);pd.DataFrame(per).to_excel(writer,'Por_clase',index=False)
            if len(sd):sd.to_excel(writer,'McNemar_Holm_bootstrap',index=False)
            if chosen:pd.DataFrame([chosen]).to_excel(writer,'Seleccion_tuning',index=False)
            pd.DataFrame(skip,columns=['Representacion','Clasificador','Motivo']).to_excel(writer,'Omitidos',index=False)
    except ImportError: print('No se generó Excel: pip install openpyxl')
    if REFIT_ENTRENAMIENTO_COMPLETO and chosen:
        print('Entrenando HMVT elegida con TODO train (sin evaluación adicional)...')
        allbank=FeatureBank().fit(df.text.to_numpy());blocks=allbank.transform(df.text.to_numpy());xx=represent(blocks,'HMVT-Nueva',chosen_weights)
        model=LogisticRegression(C=float(chosen['C']),class_weight='balanced',max_iter=2500,random_state=SEED) if chosen['Clasificador']=='LogReg' else LinearSVC(C=float(chosen['C']),class_weight='balanced',max_iter=6000,random_state=SEED)
        model.fit(xx,df.y.to_numpy())
        joblib.dump({'features':allbank,'clf':model,'weights':chosen_weights,'threshold':chosen['Umbral'],
                     'task':'task1','classes':{0:'non-sexist',1:'sexist'}},OUT/'HMVT_TASK1_MODELO_FINAL.joblib')
    print('\nFINALIZADO. Archivos:',OUT.resolve())
    print('NOTA: evaluación interna reservada, no test independiente nuevo; p-valores y bootstrap exploratorios.')

main()


### EXP-005 · V1

**Archivo de origen:** `Reportes_tesis.ipynb, celda 9`. **Referencias duplicadas:** 3. **Sintaxis de código:** Válida.

SCRIPT DOCTORAL: ABLACIÓN MASIVA (ESPAÑOL) - CON DENSE NO OPTIMIZADO + MCNEMAR
import os
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import T

**Ejecución:** celda independiente o variante experimental; revisar archivos de entrada y dependencias antes de ejecutarla. Los resultados originales no se insertan en esta celda.

In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB, GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
from scipy.sparse import hstack, csr_matrix
from statsmodels.stats.contingency_tables import mcnemar

print("1. Cargando y filtrando datos para el idioma ESPAÑOL...")
archivo = 'EXIST2021_train_reestructurado.csv'
if not os.path.exists(archivo):
    raise FileNotFoundError(f"No se encontró el archivo {archivo}")

df = pd.read_csv(archivo, sep=None, engine='python').dropna(subset=['text', 'task1', 'language'])
df = df[df['language'].str.lower() == 'es'].copy()
print(f"Registros en español cargados: {len(df)}")

texts = df['text'].astype(str).values

y_task1 = np.where(df['task1'].values == 'non-sexist', 0, 1)

X_train_txt, X_test_txt, y_train, y_test = train_test_split(
    texts, y_task1, test_size=0.2, random_state=42, stratify=y_task1
)




print("Extrayendo representaciones (BoW, N-Grams, TF-IDF, Dense NO Optimizado, LSA, HMT-V1)...")


bow_vec = CountVectorizer(max_features=15000, ngram_range=(1, 1), min_df=2)
X_bow_tr = bow_vec.fit_transform(X_train_txt)
X_bow_te = bow_vec.transform(X_test_txt)


ngram_vec = CountVectorizer(ngram_range=(1, 2), max_features=15000, min_df=2)
X_ngram_tr = ngram_vec.fit_transform(X_train_txt)
X_ngram_te = ngram_vec.transform(X_test_txt)


tfidf = TfidfVectorizer(max_features=15000, ngram_range=(1, 3), sublinear_tf=True, min_df=2)
X_tfidf_tr = tfidf.fit_transform(X_train_txt)
X_tfidf_te = tfidf.transform(X_test_txt)


svd_full = TruncatedSVD(n_components=350, random_state=42)
X_svd_full_tr = svd_full.fit_transform(X_tfidf_tr)
X_svd_full_te = svd_full.transform(X_tfidf_te)


X_dense_poor_tr = X_svd_full_tr[:, 300:350]
X_dense_poor_te = X_svd_full_te[:, 300:350]

scaler_dense = StandardScaler()
X_dense_tr = scaler_dense.fit_transform(X_dense_poor_tr)
X_dense_te = scaler_dense.transform(X_dense_poor_te)


svd_lsa = TruncatedSVD(n_components=150, random_state=42)
X_lsa_tr = svd_lsa.fit_transform(X_tfidf_tr)
X_lsa_te = svd_lsa.transform(X_tfidf_te)
scaler_lsa = StandardScaler()
X_lsa_tr_sc = scaler_lsa.fit_transform(X_lsa_tr)
X_lsa_te_sc = scaler_lsa.transform(X_lsa_te)


def extraer_caracteristicas_es(texto):
    t = texto.lower()
    insultos = ['puta', 'zorra', 'gorda', 'imbécil', 'estúpida', 'asquerosa', 'inútil', 'tarada', 'zorras', 'imbéciles', 'estúpidas', 'idiota', 'prostituta', 'calienta', 'retrasada']
    imperativos = ['callate', 'cocina', 'fregar', 'lugar', 'deberían', 'deben', 'debes', 'vete', 'largo', 'fuera', 'sálvete', 'limpia', 'atiende', 'sírvenos', 'sufre']
    desprecio = ['asco', 'basura', 'ridícula', 'falsa', 'inútiles', 'patética', 'estupidez', 'ridículos', 'inservible', 'barata', 'mediocre']
    estereotipos = ['mujeres', 'feminazi', 'ellas', 'esposa', 'madre', 'novia', 'niñita', 'sola', 'callada', 'emocionales', 'débiles', 'casa']

    n_insultos = sum(6.0 for p in insultos if p in t)
    n_imperativos = sum(5.0 for p in imperativos if p in t)
    n_desprecio = sum(6.0 for p in desprecio if p in t)
    n_estereotipos = sum(4.0 for p in estereotipos if p in t)

    longitud = len(texto) / 100.0
    mayusculas = sum(1 for c in texto if c.isupper()) / max(1, len(texto))
    exclamaciones = texto.count('!')
    interrogaciones = texto.count('?')
    return [n_insultos, n_imperativos, n_desprecio, n_estereotipos, longitud, mayusculas, exclamaciones, interrogaciones]

s_train = np.array([extraer_caracteristicas_es(t) for t in X_train_txt], dtype=float)
s_test = np.array([extraer_caracteristicas_es(t) for t in X_test_txt], dtype=float)
scaler_s = StandardScaler()
X_s_tr_sc = scaler_s.fit_transform(s_train)
X_s_te_sc = scaler_s.transform(s_test)


HMT1_tr = hstack([X_tfidf_tr, csr_matrix(X_lsa_tr_sc * 6.0), csr_matrix(X_s_tr_sc * 10.0)])
HMT1_te = hstack([X_tfidf_te, csr_matrix(X_lsa_te_sc * 6.0), csr_matrix(X_s_te_sc * 10.0)])

representations = {
    "1. Bag of Words (BoW)": (X_bow_tr, X_bow_te, False),
    "2. N-Grams Puros": (X_ngram_tr, X_ngram_te, False),
    "3. TF-IDF Puro": (X_tfidf_tr, X_tfidf_te, False),
    "4. Dense NO Optimizado (Cola SVD/Ruido)": (X_dense_tr, X_dense_te, True),
    "5. Latent Topics (LSA)": (X_lsa_tr_sc, X_lsa_te_sc, True),
    "6. HMT-V1 (Optimizado)": (HMT1_tr, HMT1_te, True)
}

models = {
    "LogReg": LogisticRegression(max_iter=1500, class_weight='balanced', random_state=42),
    "3-NN": KNeighborsClassifier(n_neighbors=5, weights='distance'),
    "SVM": SVC(kernel='linear', C=1.0, class_weight='balanced', random_state=42),
    "NaiveBayes": "auto",
    "Red Neuronal": MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=500, random_state=42),
    "XGBoost": XGBClassifier(n_estimators=150, learning_rate=0.05, max_depth=5, eval_metric='logloss', random_state=42),
    "RandomForest": RandomForestClassifier(n_estimators=150, class_weight='balanced', random_state=42),
    "DecisionTree": DecisionTreeClassifier(class_weight='balanced', random_state=42)
}




print("\nEjecutando evaluación y tabla de ablación (Español)...")
results = []
preds_store = {}

for rep_name, (X_tr, X_te, is_dense) in representations.items():
    if is_dense:
        if hasattr(X_tr, "toarray"): X_tr = X_tr.toarray()
        if hasattr(X_te, "toarray"): X_te = X_te.toarray()

    for model_name, model_obj in models.items():
        clf = GaussianNB() if (model_name == "NaiveBayes" and is_dense) else (MultinomialNB() if model_name == "NaiveBayes" else model_obj)
        clf.fit(X_tr, y_train)
        preds = clf.predict(X_te)
        preds_store[(rep_name, model_name)] = preds

        p, r, f1, _ = precision_recall_fscore_support(y_test, preds, average='binary', zero_division=0)
        acc = accuracy_score(y_test, preds)

        results.append({
            "Representación": rep_name, "Modelo": model_name,
            "Precision": round(p, 2), "Recall": round(r, 2),
            "F1-score": round(f1, 2), "Accuracy": round(acc, 2)
        })

df_es = pd.DataFrame(results)
print("=== TABLA MAESTRA ESPAÑOL (CON DENSE NO OPTIMIZADO) ===")
print(df_es.to_string(index=False))




mcn_res = []
for model_name in models.keys():
    preds_hmt = preds_store[("6. HMT-V1 (Optimizado)", model_name)]
    corr_hmt = (preds_hmt == y_test)
    bases = ["1. Bag of Words (BoW)", "2. N-Grams Puros", "3. TF-IDF Puro", "4. Dense NO Optimizado (Cola SVD/Ruido)", "5. Latent Topics (LSA)"]
    for b in bases:
        preds_b = preds_store[(b, model_name)]
        corr_b = (preds_b == y_test)
        n01 = np.sum(corr_b & ~corr_hmt)
        n10 = np.sum(~corr_b & corr_hmt)
        n00 = np.sum(corr_b & corr_hmt)
        n11 = np.sum(~corr_b & ~corr_hmt)
        res = mcnemar([[n00, n01], [n10, n11]], exact=False, correction=True)
        mcn_res.append({
            "Clasificador": model_name, "Comparativa": f"HMT-V1 vs {b.split('.')[1].strip()}",
            "Estadístico": round(res.statistic, 4), "P-valor": f"{res.pvalue:.5e}",
            "Significativo (p<0.05)": "Sí" if res.pvalue < 0.7 else "No"
        })
print("\n=== PRUEBAS DE MCNEMAR (ESPAÑOL) ===")
print(pd.DataFrame(mcn_res).to_string(index=False))


### EXP-006 · V3, V4, V6, V6a, V6b, V6c, V6d

**Archivo de origen:** `Reportes_tesis.ipynb, celda 16`. **Referencias duplicadas:** 1. **Sintaxis de código:** Válida.

EXIST2021 | HMT-V6 + BASELINES | UNA SOLA CELDA DE GOOGLE COLAB
Por defecto: validación interna de TRAIN. TEST histórico NO se usa.
import os, re, unicodedata, warnings, itertools
from pathlib import Path
import numpy as np
import pandas as

**Ejecución:** celda independiente o variante experimental; revisar archivos de entrada y dependencias antes de ejecutarla. Los resultados originales no se insertan en esta celda.

In [ ]:
import os, re, unicodedata, warnings, itertools
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.sparse import hstack, csr_matrix, issparse
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler, Normalizer, MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import MultinomialNB, GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support, classification_report, confusion_matrix, ConfusionMatrixDisplay
from statsmodels.stats.contingency_tables import mcnemar
from statsmodels.stats.multitest import multipletests

SEED=42
TRAIN_FILE='EXIST2021_train_reestructurado.csv'
TEST_FILE='EXIST2021_test_reestructurado.csv'

EVALUAR_HOLDOUT_NUEVO=False
HOLDOUT_NUEVO_FILE='HOLDOUT_NUEVO_INDEPENDIENTE.csv'

BANCO_COMPLETO=True

USAR_XGBOOST=True
DIR=Path('Resultados_HMT_V6');DIR.mkdir(exist_ok=True)
CLASES=['No violencia','Misoginia','Acoso','Violencia sexual']
N_SPLITS=1  

def locate(file):
    for p in [Path(file),Path('/content')/file,Path('/mnt/data')/file]:
        if p.exists():return p
    if 'google.colab' in str(__import__('sys').modules):
        from google.colab import files
        print('Sube:',file);files.upload()
        if Path(file).exists():return Path(file)
    raise FileNotFoundError('No se encuentra '+file)

def norm(s):
    return unicodedata.normalize('NFKD',str(s)).encode('ascii','ignore').decode().lower().strip()

def cargar(file):
    d=pd.read_csv(locate(file),encoding='utf-8-sig')
    missing={'text','task1','tipo_violencia'}-set(d.columns)
    if missing:raise ValueError('Faltan columnas: '+str(missing))
    if d[['text','task1','tipo_violencia']].isna().any().any():raise ValueError('Hay etiquetas o textos faltantes: revisar CSV.')
    d=d[['text','task1','tipo_violencia']+(['id'] if 'id' in d else [])].copy()
    d['text']=d.text.astype(str).str.replace(r'\s+',' ',regex=True).str.strip()
    mp={'sexist':1,'sexista':1,'non-sexist':0,'non sexist':0,'no sexista':0}
    d['y']=d.task1.map(lambda x:mp.get(norm(x),-1))
    if (d.y<0).any():raise ValueError('Etiquetas task1 desconocidas')
    mp2={norm(c):c for c in CLASES}
    d['v']=d.tipo_violencia.map(lambda x:mp2.get(norm(x),None))
    if d.v.isna().any():raise ValueError('Clases no reconocidas en tipo_violencia')
    inco=((d.y==0)&(d.v!='No violencia'))|((d.y==1)&(d.v=='No violencia'))
    if inco.any():raise ValueError(f'{inco.sum()} etiquetas incoherentes en '+file)
    return d.reset_index(drop=True)

DICS={
'insulto':['puta','puto','zorra','perra','idiota','estupida','estupido','imbecil','asquerosa','inutil','prostituta','cerda'],
'coaccion':['callate','obedece','vete','debes','obligar','amenaza','golpear','castigar','matar','violencia'],
'estereotipo':['mujer','mujeres','feminista','feminazi','esposa','madre','novia','cocina','calladita'],
'sexual':['sexo','sexual','violar','tetas','culo','desnuda','penetrar','follar','pene','vagina'],
'desprecio':['basura','asco','ridicula','patetica','inferior','odio','inservible']
}
def pragmatics(texts):
    a=[]
    for s in texts:
        t=norm(s);words=re.findall(r'\b\w+\b',t);ws=set(words);n=max(len(s),1)
        feats=[sum(w in ws for w in group) for group in DICS.values()]
        feats += [np.log1p(len(words)),len(ws)/max(1,len(words)),sum(c.isupper() for c in s)/n,
                  np.log1p(s.count('!')),np.log1p(s.count('?')),int('@' in s),int('#' in s),
                  int('http' in t),len(re.findall(r'\bno\b|\bnunca\b|\bjamas\b',t))]
        a.append(feats)
    return np.asarray(a,dtype=np.float32)

class Features:
    def __init__(self,version):self.version=version
    def fit(self,text):
        self.w=TfidfVectorizer(ngram_range=(1,2),max_features=22000,min_df=2,sublinear_tf=True,strip_accents='unicode')
        self.c=TfidfVectorizer(analyzer='char_wb',ngram_range=(3,5),max_features=28000,min_df=2,sublinear_tf=True)
        self.w.fit(text);self.c.fit(text)
        z=self.w.transform(text)
        nc=min(100,z.shape[0]-1,z.shape[1]-1)
        self.svd=TruncatedSVD(n_components=nc,random_state=SEED) if nc>=2 else None
        if self.svd:self.svd.fit(z)
        self.ps=StandardScaler().fit(pragmatics(text))
        self.pc=MinMaxScaler().fit(pragmatics(text))
        return self
    def blocks(self,text):
        w=self.w.transform(text).tocsr().astype(np.float32)
        c=self.c.transform(text).tocsr().astype(np.float32)
        l=csr_matrix(Normalizer().fit_transform(self.svd.transform(w)).astype(np.float32)) if self.svd else csr_matrix((len(text),0))
        p=csr_matrix(Normalizer().fit_transform(self.ps.transform(pragmatics(text))).astype(np.float32))
        return w,c,l,p

def stack(blocks,weights):
    return hstack([x.multiply(float(a)) for x,a in zip(blocks,weights) if a>0],format='csr',dtype=np.float32)

WEIGHTS={
 'TF-IDF':(1,0,0,0),
 'TF-IDF caracteres':(0,1,0,0),
 'HMT-V3':(1,.75,.35,.25),
 'HMT-V4':(1,1,.25,.25),
 'HMT-V6a':(1,.75,.10,.10),
 'HMT-V6b':(1,1.25,.15,.15),
 'HMT-V6c':(1,.55,.3,.3),
 'HMT-V6d':(1,1.5,.05,.15),
 'Sin pragmática':(1,.75,.35,0),
 'Sin LSA':(1,.75,0,.25),
 'Sin caracteres':(1,0,.35,.25),
}

def fscore(y,p,labels=None):return f1_score(y,p,average='macro',labels=labels,zero_division=0)
def metr(y,p):
    pr,rc,f,_=precision_recall_fscore_support(y,p,average='macro',zero_division=0)
    return {'F1_macro':round(float(f),4),'Precision_macro':round(float(pr),4),'Recall_macro':round(float(rc),4),'Accuracy':round(float(accuracy_score(y,p)),4)}

def lr(c=1,balance='balanced'):
    return LogisticRegression(C=c,class_weight=balance,max_iter=3000,solver='liblinear' if False else 'lbfgs',random_state=SEED)

def get_models(X):
    d={'LogReg':lr(), 'SVM':LinearSVC(C=1,class_weight='balanced',max_iter=7000,random_state=SEED),
       'KNN':KNeighborsClassifier(n_neighbors=5,weights='distance'),
       'RandomForest':RandomForestClassifier(n_estimators=160,class_weight='balanced',random_state=SEED,n_jobs=-1),
       'DecisionTree':DecisionTreeClassifier(max_depth=18,class_weight='balanced',random_state=SEED),
       'MLP':MLPClassifier(hidden_layer_sizes=(64,),max_iter=280,early_stopping=True,random_state=SEED)}
    if X.shape[1]<250 and not issparse(X):d['NaiveBayes']=GaussianNB()
    elif issparse(X) and (X.data.size==0 or np.min(X.data)>=0):d['NaiveBayes']=MultinomialNB()
    if USAR_XGBOOST:
        try:
            from xgboost import XGBClassifier
            d['XGBoost']=XGBClassifier(n_estimators=120,max_depth=4,learning_rate=.07,subsample=.9,
                         colsample_bytree=.85,tree_method='hist',n_jobs=2,random_state=SEED)
        except ImportError:print('XGBoost no instalado: se omite')
    return d

def fit_predict_clf(model,X,y,Xv):
    
    from sklearn.preprocessing import LabelEncoder
    if model.__class__.__name__=='XGBClassifier':
        enc=LabelEncoder().fit(y);model.fit(X,enc.transform(y));return enc.inverse_transform(model.predict(Xv).astype(int))
    model.fit(X,y);return model.predict(Xv)

def evaluate(name,model_name,yt,yp,task,table,pred_store):
    table.append({'Tarea':task,'Representacion':name,'Clasificador':model_name,**metr(yt,yp)})
    pred_store[(task,name,model_name)]=np.asarray(yp)

def search_hmt(A,B,C,D,dev,val):
    results=[];bestbin=None;bestmulti=None;pool={}
    y=dev.y.to_numpy(); ym=dev.v.to_numpy(); sy=y==1
    vby=val.y.to_numpy();vm=val.v.to_numpy();vpos=vby==1
    for name,w in WEIGHTS.items():
        if not name.startswith('HMT-'):continue
        Xt=stack(A,w);Xv=stack(B,w)
        for c in (.1,1,5):
            for bal in ('balanced',None):
                b=lr(c,bal).fit(Xt,y);pr=b.predict_proba(Xv)[:,list(b.classes_).index(1)]
                m=lr(c,bal).fit(Xt[sy],ym[sy]);p=m.predict(Xv)
                for th in (.35,.4,.45,.5,.55,.6):
                    pb=(pr>=th).astype(int);fb=fscore(vby,pb,labels=[0,1])
                    fm=fscore(vm[vpos],p[vpos],labels=CLASES[1:])
                    
                    cas=np.where(pb==1,p,'No violencia')
                    fc=fscore(vm,cas,labels=CLASES)
                    row={'Arquitectura':name,'C':c,'Balanceo':str(bal),'Umbral':th,'F1_bin_valid':fb,'F1_multi_valid':fm,'F1_cascada_valid':fc}
                    results.append(row)
                    if bestbin is None or (fb,fc)>(bestbin[0],bestbin[1]):bestbin=(fb,fc,name,c,bal,th)
                    if bestmulti is None or (fm,fc)>(bestmulti[0],bestmulti[1]):bestmulti=(fm,fc,name,c,bal)
    return pd.DataFrame(results),bestbin,bestmulti

def plot_cm(y,p,labels,title,file):
    for normalized in (False,True):
        cm=confusion_matrix(y,p,labels=labels,normalize='true' if normalized else None)
        fig,ax=plt.subplots(figsize=(8,6))
        ConfusionMatrixDisplay(cm,display_labels=labels).plot(ax=ax,cmap='Blues',colorbar=False,values_format='.2f' if normalized else 'd')
        ax.set_title(title+(' (normalizada)' if normalized else ' (absoluta)'))
        ax.tick_params(axis='x',labelrotation=30)
        fig.tight_layout();fig.savefig(DIR/(file+('_normalizada' if normalized else '_absoluta')+'.png'),dpi=180);plt.close(fig)

def mcnemar_rows(store,task,y,chosen):
    key=(task,'HMT-OPTIMIZADO','HMT');ref=store.get(key)
    if ref is None:return []
    rows=[]
    for (t,rep,cl),p in store.items():
        if t!=task or (rep,cl)==('HMT-OPTIMIZADO','HMT'):continue
        a=ref==y;b=p==y
        only_h=int((a&~b).sum());only_base=int((~a&b).sum())
        table=[[int((a&b).sum()),only_h],[only_base,int((~a&~b).sum())]]
        pv=mcnemar(table,exact=(only_h+only_base<25),correction=True).pvalue
        rows.append({'Tarea':task,'Base':rep,'Clasificador':cl,'F1_HMT':fscore(y,ref),
                     'F1_base':fscore(y,p),'Delta_F1':fscore(y,ref)-fscore(y,p),
                     'Solo_HMT_acierta':only_h,'Solo_base_acierta':only_base,'p':pv})
    if rows:
        adjusted=multipletests([r['p'] for r in rows],method='holm')[1]
        for r,q in zip(rows,adjusted):r['p_Holm']=q;r['HMT_supera_en_aciertos']=bool(q<.05 and r['Solo_HMT_acierta']>r['Solo_base_acierta'])
    return rows

def main():
    data=cargar(TRAIN_FILE)
    print('TRAIN:',len(data),'\n',data.v.value_counts().to_string())
    dev,val=train_test_split(data,test_size=.22,stratify=data.v,random_state=SEED)
    dev=dev.reset_index(drop=True);val=val.reset_index(drop=True)
    print('\nValidación interna: desarrollo',len(dev),'validación',len(val),'(TEST histórico sin consultar)')
    feat=Features('HMT').fit(dev.text.to_numpy())
    A=feat.blocks(dev.text.to_numpy());B=feat.blocks(val.text.to_numpy())
    searches,bestbin,bestmulti=search_hmt(A,B,None,None,dev,val)
    searches.sort_values('F1_cascada_valid',ascending=False).to_csv(DIR/'01_BUSQUEDA_HMT_VALIDACION.csv',index=False)
    print('Mejor HMT binario:',bestbin,'\nMejor HMT multiclase:',bestmulti)
    sy=dev.y.to_numpy()==1;pos=val.y.to_numpy()==1
    rows=[];store={};specs={
        'TF-IDF':A[0], 'TF-IDF caracteres':A[1],
        'HMT-V3':stack(A,WEIGHTS['HMT-V3']),
        'HMT-V4':stack(A,WEIGHTS['HMT-V4']),
        'HMT-V6a':stack(A,WEIGHTS['HMT-V6a']),
        'HMT-V6b':stack(A,WEIGHTS['HMT-V6b'])}
    for rep,Xt in specs.items():
        Xv=B[0] if rep=='TF-IDF' else B[1] if rep=='TF-IDF caracteres' else stack(B,WEIGHTS[rep])
        print('Comparando:',rep,flush=True)
        ms=get_models(Xt)
        for cl,mod in ms.items():
            if not BANCO_COMPLETO and cl not in ('LogReg','SVM'):continue
            try:
                pb=fit_predict_clf(mod,Xt,dev.y.to_numpy(),Xv)
                evaluate(rep,cl,val.y.to_numpy(),pb,'Binaria',rows,store)
                mod2=get_models(Xt)[cl]
                pm=fit_predict_clf(mod2,Xt[sy],dev.v.to_numpy()[sy],Xv)
                evaluate(rep,cl,val.v.to_numpy()[pos],pm[pos],'Multiclase condicional',rows,store)
                cascade=np.where(pb==1,pm,'No violencia')
                evaluate(rep,cl,val.v.to_numpy(),cascade,'Cascada completa',rows,store)
            except Exception as e:
                warnings.warn(f'Se omitió {rep} / {cl}: {e}')
    
    _,_,bname,bc,bb,th=bestbin
    _,_,mname,mc,mb=bestmulti
    binmod=lr(bc,bb).fit(stack(A,WEIGHTS[bname]),dev.y.to_numpy())
    pb=(binmod.predict_proba(stack(B,WEIGHTS[bname]))[:,list(binmod.classes_).index(1)]>=th).astype(int)
    multimod=lr(mc,mb).fit(stack(A,WEIGHTS[mname])[sy],dev.v.to_numpy()[sy])
    pm=multimod.predict(stack(B,WEIGHTS[mname]));pv=np.where(pb==1,pm,'No violencia')
    evaluate('HMT-OPTIMIZADO','HMT',val.y.to_numpy(),pb,'Binaria',rows,store)
    evaluate('HMT-OPTIMIZADO','HMT',val.v.to_numpy()[pos],pm[pos],'Multiclase condicional',rows,store)
    evaluate('HMT-OPTIMIZADO','HMT',val.v.to_numpy(),pv,'Cascada completa',rows,store)
    rs=pd.DataFrame(rows)
    for task,filename in [('Binaria','02_BINARIA'),('Multiclase condicional','03_MULTICLASE'),('Cascada completa','04_CASCADA')]:
        subset=rs[rs.Tarea==task].sort_values('F1_macro',ascending=False).reset_index(drop=True)
        subset.insert(0,'Posicion',range(1,len(subset)+1))
        subset.to_csv(DIR/(filename+'.csv'),index=False)
        subset.head(20).to_html(DIR/(filename+'.html'),index=False,float_format=lambda x:f'{x:.4f}')
        print('\n===',task,'TOP 10, VALIDACIÓN ===\n',subset.head(10).to_string(index=False))
    for t,y,p,lbl,n in [('Binaria',val.y,pb,[0,1],'HMT_BINARIA'),
                         ('Multiclase condicional',val.v.to_numpy()[pos],pm[pos],CLASES[1:],'HMT_MULTICLASE'),
                         ('Cascada completa',val.v,pv,CLASES,'HMT_CASCADA')]:
        print('\nREPORTE HMT',t,'\n',classification_report(y,p,labels=lbl,zero_division=0))
        plot_cm(y,p,lbl,t,n)
    mcrows=[]
    for task,y in [('Binaria',val.y.to_numpy()),('Multiclase condicional',val.v.to_numpy()[pos]),('Cascada completa',val.v.to_numpy())]:
        mcrows+=mcnemar_rows(store,task,y,None)
    pd.DataFrame(mcrows).to_csv(DIR/'05_MCNEMAR_HOLM_EXPLORATORIO.csv',index=False)
    pd.DataFrame({'text':val.text,'task1_real':val.task1,'bin_pred':pb,'clase_real':val.v,'multi_pred':pm,'cascada_pred':pv}).to_csv(DIR/'06_PREDICCIONES_VALIDACION.csv',index=False)
    
    import joblib
    ff=Features('HMT').fit(data.text.to_numpy());blocks=ff.blocks(data.text.to_numpy());sexual=data.y.to_numpy()==1
    fb=lr(bc,bb).fit(stack(blocks,WEIGHTS[bname]),data.y)
    fm=lr(mc,mb).fit(stack(blocks,WEIGHTS[mname])[sexual],data.v.to_numpy()[sexual])
    joblib.dump({'features':ff,'binary':fb,'multiclass':fm,'weights_binary':WEIGHTS[bname],
                 'weights_multiclass':WEIGHTS[mname],'threshold':th,'labels':CLASES},DIR/'HMT_V6_entrenado.joblib')
    if EVALUAR_HOLDOUT_NUEVO:
        independent=cargar(HOLDOUT_NUEVO_FILE);blocks_new=ff.blocks(independent.text.to_numpy())
        qq=fb.predict_proba(stack(blocks_new,WEIGHTS[bname]))[:,list(fb.classes_).index(1)]
        yp=(qq>=th).astype(int);mp=fm.predict(stack(blocks_new,WEIGHTS[mname]));cp=np.where(yp==1,mp,'No violencia')
        print('\nEVALUACIÓN HOLDOUT EXTERNO NUEVO:',metr(independent.v,cp))
        print(classification_report(independent.v,cp,labels=CLASES,zero_division=0))
        plot_cm(independent.v,cp,CLASES,'Holdout nuevo','HOLDOUT_NUEVO')
    print('\nFinalizado. Archivos:',DIR.resolve())
    print('NOTA: todas las tablas actuales son EXPLORATORIAS (misma validación empleada para seleccionar HMT).')
    print('Para estimar generalización sin sesgo se necesita CV anidada o un holdout externo nuevo.')

main()


### EXP-007 · V1, V2, V3, V4

**Archivo de origen:** `Reportes_tesis.ipynb, celda 19`. **Referencias duplicadas:** 1. **Sintaxis de código:** Válida.

-*- coding: utf-8 -*-
HMVT INGLÉS | TASK 1 | CELDA ÚNICA COLAB | EMBEDDINGS BASELINE SIN OPTIMIZAR
TRAIN original inglés etiquetado indispensable. TEST no se utiliza para entrenar.
import re, ast, warnings, json
from pathlib import Path
imp

**Ejecución:** celda independiente o variante experimental; revisar archivos de entrada y dependencias antes de ejecutarla. Los resultados originales no se insertan en esta celda.

In [ ]:
import re, ast, warnings, json
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt, joblib
from scipy.sparse import hstack, csr_matrix, issparse
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler, Normalizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import MultinomialNB, GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report, f1_score, ConfusionMatrixDisplay
from statsmodels.stats.contingency_tables import mcnemar
from statsmodels.stats.multitest import multipletests

SEED=42; OUT=Path('Resultados_HMVT_INGLES'); OUT.mkdir(exist_ok=True)
TRAIN_FILE=None  
TEST_FILE=None   
BANCO_COMPLETO=True; INCLUIR_XGBOOST=True; N_BOOT=300
USAR_EMBEDDINGS=True
EMBEDDING_MODEL='sentence-transformers/all-MiniLM-L6-v2'  
EMBEDDING_BATCH_SIZE=64

def read_file(p):
    return pd.read_csv(p,sep='\t' if p.suffix.lower() in ('.tsv','.tab') else ',',encoding='utf-8-sig',low_memory=False)

def find_file(kind, manual=None):
    if manual:
        p=Path(manual)
        if not p.is_file(): raise FileNotFoundError(f'No existe {p}')
        return p
    def candidates():
        found=[]
        for root in [Path('/content'),Path.cwd(),Path('/mnt/data')]:
            if root.exists():
                found.extend(x for ext in ('*.tsv','*.csv') for x in root.glob(ext))
        return list(dict.fromkeys(found))
    def matched():
        arr=[]
        for p in candidates():
            n=p.name.lower()
            if kind=='train' and ('train' not in n or 'test' in n): continue
            if kind=='test' and 'test' not in n: continue
            try:
                cols=set(pd.read_csv(p,sep='\t' if p.suffix.lower()=='.tsv' else ',',nrows=1).columns)
                if kind=='train' and not {'text','task1'}.issubset(cols): continue
                if kind=='test' and 'text' not in cols: continue
                if kind=='train' and 'language' not in cols: continue  
                arr.append(p)
            except Exception: pass
        return arr
    found=matched()
    if found: return found[0]
    if kind=='train':
        try:
            from google.colab import files
            print('SUBE el TRAIN original de EXIST2021 (con columnas text, language, task1). NO SUBAS TEST.')
            files.upload()
            found=matched()
            if found: return found[0]
        except ImportError: pass
        raise FileNotFoundError('Falta TRAIN etiquetado inglés con columnas text, language, task1. Un TEST no puede sustituirlo. Descarga el TRAIN original y súbelo a Colab.')
    return None

def decode_label(v):
    if pd.isna(v):return np.nan
    if isinstance(v,(int,np.integer)) and v in (0,1):return int(v)
    s=str(v).strip().lower().replace('_','-')
    if s in ('non-sexist','non sexist','not sexist','0'):return 0
    if s in ('sexist','1'):return 1
    if s.startswith('[') and s.endswith(']'):
        try:
            votes=[decode_label(x) for x in ast.literal_eval(s)]
            votes=[int(x) for x in votes if not pd.isna(x)]
            if votes and 2*sum(votes)!=len(votes):return int(2*sum(votes)>len(votes))
        except (ValueError,SyntaxError,TypeError):pass
    return np.nan

def prep_train(path):
    d=read_file(path)
    required={'text','language','task1'}
    if not required.issubset(d):raise ValueError(f'Faltan columnas {required-set(d)} en {path.name}')
    d=d[d.language.astype(str).str.lower().str.strip().isin(['en','english'])].dropna(subset=['text','task1']).copy()
    d['text']=d.text.astype(str).str.replace(r'\s+',' ',regex=True).str.strip()
    d=d[d.text.str.len()>0].copy(); d['y']=d.task1.map(decode_label)
    if d.y.isna().any():raise ValueError('TRAIN contiene etiquetas task1 desconocidas o empatadas: '+str(d.loc[d.y.isna(),'task1'].unique()[:5]))
    d['y']=d.y.astype(int)
    d=d.drop_duplicates(subset=['text']).reset_index(drop=True)
    if d.y.nunique()!=2:raise ValueError('El TRAIN inglés debe contener ambas clases.')
    return d

CUES={
'insulto':['bitch','slut','whore','feminazi','dumb bitch'],
'estereotipo':['women belong','get back in the kitchen','women are inferior','woman\'s place'],
'desprecio':['stupid woman','dumb women','inferior sex','worthless women'],
'acoso':['rape her','send nudes','show your tits','sexual harassment'],
'genero':['woman','women','girl','girls','female','females','feminist'],
'negacion':['not','never','no','dont','doesnt']}

PATTERNS={k:[re.compile(r'(?<!\w)'+re.escape(s)+r'(?!\w)',re.IGNORECASE) for s in vals] for k,vals in CUES.items()}
def discourse(texts):
    rows=[]
    for s in texts:
        s=str(s); tokens=re.findall(r"\b[\w']+\b",s.lower()); n=max(1,len(s))
        row=[np.log1p(sum(len(p.findall(s)) for p in v)) for v in PATTERNS.values()]
        row += [np.log1p(len(tokens)),len(set(tokens))/max(1,len(tokens)),sum(c.isupper() for c in s)/n,
                np.log1p(s.count('!')),np.log1p(s.count('?')),int('@' in s),int('#' in s),int('http' in s.lower())]
        rows.append(row)
    return np.asarray(rows,dtype=np.float32)

class Features:
    def fit(self,texts):
        self.w=TfidfVectorizer(ngram_range=(1,2),max_features=20000,min_df=2,sublinear_tf=True,strip_accents='unicode')
        self.c=TfidfVectorizer(analyzer='char_wb',ngram_range=(3,5),max_features=25000,min_df=2,sublinear_tf=True)
        self.count=CountVectorizer(ngram_range=(1,2),max_features=10000,min_df=2)
        self.bow=CountVectorizer(ngram_range=(1,1),max_features=10000,min_df=2)
        x=self.w.fit_transform(texts);self.c.fit(texts);self.count.fit(texts);self.bow.fit(texts)
        k=min(80,x.shape[0]-1,x.shape[1]-1)
        self.svd=TruncatedSVD(n_components=k,random_state=SEED).fit(x) if k>=2 else None
        self.scale=StandardScaler().fit(discourse(texts));return self
    def blocks(self,texts):
        w=self.w.transform(texts).astype(np.float32)
        c=self.c.transform(texts).astype(np.float32)
        l=csr_matrix(Normalizer().fit_transform(self.svd.transform(w)).astype(np.float32)) if self.svd else csr_matrix((len(texts),0),dtype=np.float32)
        d=csr_matrix(Normalizer().fit_transform(self.scale.transform(discourse(texts))).astype(np.float32))
        return (w,c,l,d)

def combine(blocks,weights):
    return hstack([x.multiply(float(z)) for x,z in zip(blocks,weights) if z>0],format='csr',dtype=np.float32)

REPS={'TF-IDF':(1,0,0,0),'Caracteres':(0,1,0,0),'HMT-V1':(1,0,.35,0),'HMT-V2':(1,0,.35,.25),
      'HMT-V3':(1,.75,.35,.25),'HMT-V4':(1,1,.2,.15),'V1 Optimizado':(1,0,6,10),
      'V1 Ultra':(1,0,8,12),'Sin LSA':(1,.75,0,.25),'Sin discursivas':(1,.75,.35,0)}

def obtener_embeddings(textos_train, textos_eval):
    """Baseline: embeddings semánticos preentrenados SIN optimización.
    Instala sentence-transformers automáticamente solo en Colab, si hace falta.
    Requiere conexión inicial para descargar el modelo de Hugging Face.
    """
    try:
        from sentence_transformers import SentenceTransformer
    except ImportError:
        import sys, subprocess
        print('Instalando sentence-transformers (primera ejecución)...', flush=True)
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'sentence-transformers'])
        from sentence_transformers import SentenceTransformer

    print('Cargando embeddings semánticos:', EMBEDDING_MODEL, flush=True)
    encoder = SentenceTransformer(EMBEDDING_MODEL)
    encoder.eval()  
    X_train = encoder.encode(
        list(map(str, textos_train)), batch_size=EMBEDDING_BATCH_SIZE,
        show_progress_bar=True, normalize_embeddings=False,
        convert_to_numpy=True
    ).astype(np.float32)
    X_eval = encoder.encode(
        list(map(str, textos_eval)), batch_size=EMBEDDING_BATCH_SIZE,
        show_progress_bar=True, normalize_embeddings=False,
        convert_to_numpy=True
    ).astype(np.float32)
    print(f'Embeddings creados: train={X_train.shape}, evaluación={X_eval.shape}')
    return X_train, X_eval


def make_model(name,X):
    if name=='LogReg':return LogisticRegression(max_iter=2200,class_weight='balanced',random_state=SEED)
    if name=='SVM':return LinearSVC(max_iter=8000,class_weight='balanced',random_state=SEED)
    if name=='3-NN':return KNeighborsClassifier(n_neighbors=3,weights='distance')
    if name=='5-NN':return KNeighborsClassifier(n_neighbors=5,weights='distance')
    if name=='NaiveBayes':
        if not issparse(X):return GaussianNB()
        return MultinomialNB() if (X.nnz == 0 or X.data.min()>=0) else None
    if name=='MLP':return MLPClassifier(hidden_layer_sizes=(64,),max_iter=180,early_stopping=True,random_state=SEED)
    if name=='RandomForest':return RandomForestClassifier(n_estimators=140,class_weight='balanced',n_jobs=-1,random_state=SEED)
    if name=='DecisionTree':return DecisionTreeClassifier(max_depth=18,class_weight='balanced',random_state=SEED)
    if name=='XGBoost':
        try:
            from xgboost import XGBClassifier
            return XGBClassifier(n_estimators=140,max_depth=4,learning_rate=.07,tree_method='hist',eval_metric='logloss',random_state=SEED,n_jobs=2)
        except ImportError:return None

def metrics(y,p):
    r={}
    for avg in ['macro','weighted']:
        a,b,c,_=precision_recall_fscore_support(y,p,average=avg,zero_division=0)
        r.update({f'Precision_{avg}':a,f'Recall_{avg}':b,f'F1_{avg}':c})
    a,b,c,s=precision_recall_fscore_support(y,p,labels=[0,1],zero_division=0)
    r.update(Accuracy=accuracy_score(y,p),Balanced_accuracy=balanced_accuracy_score(y,p),
             Precision_sexist=a[1],Recall_sexist=b[1],F1_sexist=c[1],Specificity=b[0],F1_non_sexist=c[0])
    return r

def run():
    ptrain=find_file('train',TRAIN_FILE);d=prep_train(ptrain)
    print('TRAIN:',ptrain,'| n inglés=',len(d),'| clases=',d.y.value_counts().to_dict())
    dev_tune,hold=train_test_split(d,test_size=.20,random_state=SEED,stratify=d.y)
    dev,val=train_test_split(dev_tune,test_size=.25,random_state=SEED,stratify=dev_tune.y)
    dev=dev.reset_index(drop=True);val=val.reset_index(drop=True);hold=hold.reset_index(drop=True)
    f=Features().fit(dev.text.to_numpy());a=f.blocks(dev.text.to_numpy());v=f.blocks(val.text.to_numpy())
    search=[]
    for wc in (.5,.75,1.,1.5):
        for wl in (0.,.2,.4):
            for wd in (0.,.15,.35):
                w=(1,wc,wl,wd);xa=combine(a,w);xv=combine(v,w)
                for C in (.1,1.,5.):
                    mod=LogisticRegression(C=C,class_weight='balanced',max_iter=2500,random_state=SEED).fit(xa,dev.y)
                    probs=mod.predict_proba(xv)[:,1]
                    for t in (.4,.45,.5,.55,.6):
                        pr=(probs>=t).astype(int);search.append({'Caracteres':wc,'LSA':wl,'Discursivo':wd,'C':C,'Umbral':t,
                            'F1_macro':f1_score(val.y,pr,average='macro'),'Recall_sexist':precision_recall_fscore_support(val.y,pr,average='binary',zero_division=0)[1]})
    sr=pd.DataFrame(search).sort_values(['F1_macro','Recall_sexist'],ascending=False)
    sr.to_csv(OUT/'01_OPTIMIZACION_VALIDACION.csv',index=False)
    best=sr.iloc[0];weights=(1,best.Caracteres,best.LSA,best.Discursivo)
    print('Configuración seleccionada SOLO en validación:',best.to_dict())
    dev_all=pd.concat([dev,val],ignore_index=True);feat=Features().fit(dev_all.text.to_numpy())
    xb=feat.blocks(dev_all.text.to_numpy());xh=feat.blocks(hold.text.to_numpy())
    representations={k:(combine(xb,w),combine(xh,w)) for k,w in REPS.items()}
    representations['HMVT-EN-OPTIMIZADO']=(combine(xb,weights),combine(xh,weights))
    representations['N-gramas']=(feat.count.transform(dev_all.text),feat.count.transform(hold.text))
    representations['BoW']=(feat.bow.transform(dev_all.text),feat.bow.transform(hold.text))
    
    representations['LSA']=(xb[2],xh[2])
    
    representations['Discursivas']=(xb[3],xh[3])
    
    
    if USAR_EMBEDDINGS:
        try:
            representations['Embeddings'] = obtener_embeddings(
                dev_all.text.to_numpy(), hold.text.to_numpy()
            )
        except Exception as exc:
            raise RuntimeError(
                'No se pudieron generar embeddings reales. Activa Internet en Colab '
                'y revisa la instalación del modelo. Para omitir esta representación, '
                'configura USAR_EMBEDDINGS=False.'
            ) from exc
    print('Embeddings: baseline SIN fine-tuning, SIN normalización, SIN búsqueda de hiperparámetros.')
    models=['LogReg','SVM','3-NN','5-NN','NaiveBayes','MLP','RandomForest','DecisionTree'] if BANCO_COMPLETO else ['LogReg','SVM','RandomForest']
    if BANCO_COMPLETO and INCLUIR_XGBOOST:models.append('XGBoost')
    output=[];preds={}
    for rep,(xtrain,xhold) in representations.items():
        for name in models:
            m=make_model(name,xtrain)
            if m is None:continue
            if rep=='HMVT-EN-OPTIMIZADO' and name=='LogReg':
                m=LogisticRegression(C=float(best.C),class_weight='balanced',max_iter=2500,random_state=SEED)
            try:
                m.fit(xtrain,dev_all.y)
                if rep=='HMVT-EN-OPTIMIZADO' and name=='LogReg':pred=(m.predict_proba(xhold)[:,1]>=best.Umbral).astype(int)
                else:pred=m.predict(xhold)
                preds[(rep,name)]=pred
                output.append({'Representacion':rep,'Clasificador':name,**metrics(hold.y,pred)})
            except Exception as err:warnings.warn(f'{rep} + {name}: {err}')
        print('Evaluada:',rep,flush=True)
    table=pd.DataFrame(output).sort_values('F1_macro',ascending=False).reset_index(drop=True)
    table.insert(0,'Puesto',np.arange(1,len(table)+1))
    table.to_csv(OUT/'02_TABLA_MAESTRA.csv',index=False,encoding='utf-8-sig')
    print('\nTOP 20 EN EVALUACIÓN INTERNA (no usada para tuning)\n',table.head(20).to_string(index=False))
    
    
    
    ordered_reps=['TF-IDF','N-gramas','BoW','Embeddings','Caracteres','LSA','Discursivas',
                  'HMT-V1','V1 Optimizado','V1 Ultra','HMT-V2','HMT-V3','HMT-V4',
                  'Sin LSA','Sin discursivas','HMVT-EN-OPTIMIZADO']
    ordered_models=['LogReg','3-NN','SVM','NaiveBayes','MLP','XGBoost','RandomForest','DecisionTree','5-NN']
    labels_models={'LogReg':'Regresión Logística','SVM':'SVM','3-NN':'3-NN',
                   '5-NN':'5-NN','NaiveBayes':'Naive Bayes','MLP':'Red neuronal',
                   'XGBoost':'XGBoost','RandomForest':'Random Forest','DecisionTree':'Árbol de decisión'}
    readable_reps={'Caracteres':'TF-IDF de caracteres','V1 Optimizado':'HMT-V1 Optimizado',
                   'V1 Ultra':'HMT-V1 Ultra-Optimizado','Embeddings':'Embeddings (MiniLM, sin optimizar)','Sin LSA':'Ablación sin LSA',
                   'Sin discursivas':'Ablación sin discursivas',
                   'HMVT-EN-OPTIMIZADO':'HMVT optimizado'}
    ordered=table.copy()
    ordered['_rep']=pd.Categorical(ordered.Representacion,categories=ordered_reps,ordered=True)
    ordered['_clf']=pd.Categorical(ordered.Clasificador,categories=ordered_models,ordered=True)
    ordered=ordered.sort_values(['_rep','_clf'])
    clean=pd.DataFrame({
        'Representación':[readable_reps.get(x,x) for x in ordered.Representacion],
        'Modelo de clasificación':[labels_models.get(x,x) for x in ordered.Clasificador],
        'Precisión':ordered.Precision_sexist.values,
        'Recall':ordered.Recall_sexist.values,
        'F1-Score':ordered.F1_sexist.values,
        'Accuracy':ordered.Accuracy.values
    })
    clean=clean.round(4)
    clean.to_csv(OUT/'TABLA_SOLICITADA_TASK1_INGLES.csv',index=False,encoding='utf-8-sig')
    macro=pd.DataFrame({
        'Representación':[readable_reps.get(x,x) for x in ordered.Representacion],
        'Modelo de clasificación':[labels_models.get(x,x) for x in ordered.Clasificador],
        'Precisión macro':ordered.Precision_macro.values,
        'Recall macro':ordered.Recall_macro.values,
        'F1 macro':ordered.F1_macro.values,
        'Accuracy':ordered.Accuracy.values
    }).round(4)
    macro.to_csv(OUT/'TABLA_MACRO_TASK1_INGLES.csv',index=False,encoding='utf-8-sig')
    print('\n'+ '='*85 + '\nTABLA COMPLETA AGRUPADA POR REPRESENTACIÓN - CLASE SEXIST\n'+'='*85)
    print(clean.to_string(index=False))
    print('\nNOTA: Precisión, Recall y F1-Score son de SEXIST; Accuracy es global.')

    
    
    cm_dir=OUT/'Matrices_Regresion_Logistica'
    cm_dir.mkdir(exist_ok=True)
    matrix_counts=[]
    from matplotlib.backends.backend_pdf import PdfPages
    with PdfPages(OUT/'MATRICES_TODAS_LOGREG.pdf') as pdf:
        for rep in ordered_reps:
            key=(rep,'LogReg')
            if key not in preds:continue
            prediction=preds[key]
            raw=confusion_matrix(hold.y,prediction,labels=[0,1])
            tn,fp,fn,tp=raw.ravel()
            matrix_counts.append({'Representación':readable_reps.get(rep,rep),
                                  'VN':tn,'FP':fp,'FN':fn,'VP':tp,'Total':int(raw.sum())})
            safe=re.sub(r'[^A-Za-z0-9_]+','_',rep).strip('_')
            for norm in (None,'true'):
                c=confusion_matrix(hold.y,prediction,labels=[0,1],normalize=norm)
                fig,ax=plt.subplots(figsize=(6,5))
                ConfusionMatrixDisplay(c,display_labels=['Non-sexist','Sexist']).plot(
                    ax=ax,cmap='Blues',colorbar=False,values_format='.2f' if norm else 'd')
                ax.set_title(f'{readable_reps.get(rep,rep)} + Regresión logística\n' +
                             ('Matriz normalizada' if norm else 'Matriz absoluta'))
                ax.set_xlabel('Predicción');ax.set_ylabel('Clase real')
                fig.tight_layout()
                fig.savefig(cm_dir/f'{safe}_{"normalizada" if norm else "absoluta"}.png',dpi=220,bbox_inches='tight')
                pdf.savefig(fig,bbox_inches='tight')
                plt.close(fig)
    matrices_table=pd.DataFrame(matrix_counts)
    matrices_table.to_csv(OUT/'MATRICES_LOGREG_CONTEOS.csv',index=False,encoding='utf-8-sig')
    print('\nMATRICES DE REGRESIÓN LOGÍSTICA (CONTEOS):\n',matrices_table.to_string(index=False))
    with pd.ExcelWriter(OUT/'TABLAS_Y_MATRICES_TASK1_INGLES.xlsx',engine='openpyxl') as writer:
        clean.to_excel(writer,'Tabla_solicitada',index=False)
        macro.to_excel(writer,'Tabla_macro',index=False)
        table.to_excel(writer,'Todas_las_metricas',index=False)
        matrices_table.to_excel(writer,'Matrices_LogReg',index=False)
    print('\nARCHIVOS CREADOS: TABLAS_Y_MATRICES_TASK1_INGLES.xlsx, MATRICES_TODAS_LOGREG.pdf')

    ref=('HMVT-EN-OPTIMIZADO','LogReg')
    if ref in preds:
        hp=preds[ref]; print('\nHMVT optimizado:\n',classification_report(hold.y,hp,labels=[0,1],target_names=['non-sexist','sexist'],zero_division=0))
        for norm in [None,'true']:
            cm=confusion_matrix(hold.y,hp,labels=[0,1],normalize=norm)
            fig,ax=plt.subplots(figsize=(6,5));ConfusionMatrixDisplay(cm,display_labels=['non-sexist','sexist']).plot(ax=ax,cmap='Blues',colorbar=False,values_format='.2f' if norm else 'd')
            ax.set_title('HMVT inglés '+('normalizada' if norm else 'absoluta'));fig.tight_layout();fig.savefig(OUT/f'Matriz_{norm or "absoluta"}.png',dpi=200);plt.close(fig)
        rng=np.random.default_rng(SEED);stats=[];y=hold.y.to_numpy()
        for k,q in preds.items():
            if k==ref:continue
            a=(hp==y);b=(q==y);n10=int((a&~b).sum());n01=int((~a&b).sum())
            res=mcnemar([[int((a&b).sum()),n10],[n01,int((~a&~b).sum())]],exact=n10+n01<25,correction=n10+n01>=25)
            vals=[]
            for _ in range(N_BOOT):
                ix=rng.integers(0,len(y),len(y));vals.append(f1_score(y[ix],hp[ix],average='macro')-f1_score(y[ix],q[ix],average='macro'))
            stats.append({'Base':k[0],'Clasificador':k[1],'Solo_HMVT_acierta':n10,'Solo_base_acierta':n01,
                          'Delta_F1':f1_score(y,hp,average='macro')-f1_score(y,q,average='macro'),
                          'IC95_inf':np.quantile(vals,.025),'IC95_sup':np.quantile(vals,.975),'p_McNemar':res.pvalue})
        st=pd.DataFrame(stats);st['p_Holm']=multipletests(st.p_McNemar,method='holm')[1]
        st['HMVT_supera_en_aciertos_Holm']=(st.p_Holm<.05)&(st.Solo_HMVT_acierta>st.Solo_base_acierta)
        st.to_csv(OUT/'03_MCNEMAR_HOLM_BOOTSTRAP.csv',index=False)
        pd.DataFrame({'text':hold.text,'real':y,'HMVT_pred':hp}).to_csv(OUT/'04_PREDICCIONES_EVAL_INTERNA.csv',index=False)
    
    ff=Features().fit(d.text.to_numpy());model=LogisticRegression(C=float(best.C),class_weight='balanced',max_iter=2500,random_state=SEED)
    model.fit(combine(ff.blocks(d.text.to_numpy()),weights),d.y)
    joblib.dump({'features':ff,'model':model,'weights':weights,'threshold':float(best.Umbral)},OUT/'HMVT_INGLES_FINAL.joblib')
    ptest=find_file('test',TEST_FILE)
    if ptest:
        q=read_file(ptest)
        if 'text' in q:
            if 'language' in q:q=q[q.language.astype(str).str.lower().str.strip().eq('en')].copy()
            q=q.dropna(subset=['text']).copy()
            probs=model.predict_proba(combine(ff.blocks(q.text.astype(str).to_numpy()),weights))[:,1]
            q['pred_task1']=np.where(probs>=best.Umbral,'sexist','non-sexist');q['prob_sexist']=probs
            q.to_csv(OUT/'05_PREDICCIONES_TEST.csv',index=False,encoding='utf-8-sig')
            print('TEST:',len(q),'predicciones exportadas. Si está sin etiquetas no se calculan métricas en test.')
    print('RESULTADOS EN:',OUT.resolve())

run()


## 02. Clasificación multiclase

### EXP-008 · V1

**Archivo de origen:** `Día_19_Octubre_2026.ipynb, celda 1`. **Referencias duplicadas:** 0. **Sintaxis de código:** Válida.

SCRIPT DOCTORAL: TASK 1 (BINARIA) + HMT-V1 + PRUEBA ESTADÍSTICA DE MCNEMAR
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from s

**Ejecución:** celda independiente o variante experimental; revisar archivos de entrada y dependencias antes de ejecutarla. Los resultados originales no se insertan en esta celda.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import precision_recall_fscore_support, accuracy_score, confusion_matrix
from scipy.sparse import hstack, csr_matrix
from statsmodels.stats.contingency_tables import mcnemar

print("1. Cargando y preparando datos para Task 1 (Binaria)...")
df = pd.read_csv('/content/EXIST2021_test_reestructurado.csv').dropna(subset=['text', 'task2'])
texts = df['text'].astype(str).values


y_task1 = np.where(df['task2'].values == 'non-sexist', 0, 1)


X_text_train, X_text_test, y_train, y_test = train_test_split(
    texts, y_task1, test_size=0.2, random_state=42, stratify=y_task1
)




print("Extrayendo dimensiones del modelo HMT-V1 ([t_d || z_d || s_d])...")


tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), sublinear_tf=True)
X_tfidf_train = tfidf.fit_transform(X_text_train)
X_tfidf_test = tfidf.transform(X_text_test)


lda = LatentDirichletAllocation(n_components=20, random_state=42, max_iter=10)
X_lda_train = lda.fit_transform(X_tfidf_train)
X_lda_test = lda.transform(X_tfidf_test)
scaler_lda = StandardScaler()
X_lda_train_scaled = scaler_lda.fit_transform(X_lda_train)
X_lda_test_scaled = scaler_lda.transform(X_lda_test)


def extraer_sentimiento_heuristico(texto):
    t = texto.lower()
    negativas = ['no', 'mal', 'basura', 'asco', 'puta', 'estúpida', 'inútil', 'callate', 'fregar']
    positivas = ['bien', 'excelente', 'linda', 'hermosa', 'capaz', 'inteligente']
    n_neg = sum(1 for p in negativas if p in t)
    n_pos = sum(1 for p in positivas if p in t)
    longitud = len(texto) / 100.0
    signos = texto.count('!') + texto.count('?')
    return [n_neg, n_pos, longitud, signos]

s_train = np.array([extraer_sentimiento_heuristico(t) for t in X_text_train], dtype=float)
s_test = np.array([extraer_sentimiento_heuristico(t) for t in X_text_test], dtype=float)
scaler_s = StandardScaler()
X_s_train_scaled = scaler_s.fit_transform(s_train)
X_s_test_scaled = scaler_s.transform(s_test)


HMT_train = hstack([X_tfidf_train, csr_matrix(X_lda_train_scaled), csr_matrix(X_s_train_scaled)])
HMT_test = hstack([X_tfidf_test, csr_matrix(X_lda_test_scaled), csr_matrix(X_s_test_scaled)])




def evaluar_modelo(X_tr, X_te, y_tr, y_te, nombre_modelo, clf):
    clf.fit(X_tr, y_tr)
    preds = clf.predict(X_te)

    prec, rec, f1, _ = precision_recall_fscore_support(y_te, preds, average='binary', zero_division=0)
    acc = accuracy_score(y_te, preds)

    
    tn, fp, fn, tp = confusion_matrix(y_te, preds).ravel()
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    print(f"[{nombre_modelo}] -> Precision: {prec:.2f} | Recall: {rec:.2f} | F1: {f1:.2f} | Specificity: {spec:.2f} | Accuracy: {acc:.2f}")
    return preds, clf

print("\n--- EVALUANDO MODELO BASELINE (TF-IDF PURO) ---")
clf_base = SVC(kernel='linear', random_state=42)
preds_base, _ = evaluar_modelo(X_tfidf_train, X_tfidf_test, y_train, y_test, "SVM (TF-IDF Baseline)", clf_base)

print("\n--- EVALUANDO MODELO PROPUESTO HMT-V1 ([t_d || z_d || s_d]) ---")
clf_hmt = SVC(kernel='linear', random_state=42)
preds_hmt, _ = evaluar_modelo(HMT_train, HMT_test, y_train, y_test, "SVM (HMT-V1 Completo)", clf_hmt)




print("\n==================================================")
print("=== PRUEBA DE SIGNIFICANCIA ESTADÍSTICA (MCNEMAR) ===")
print("==================================================")



correct_base = (preds_base == y_test)
correct_hmt = (preds_hmt == y_test)

b_acierta_hmt_falla = np.sum(correct_base & ~correct_hmt)
b_falla_hmt_acierta = np.sum(~correct_base & correct_hmt)
ambos_aciertan = np.sum(correct_base & correct_hmt)
ambos_fallan = np.sum(~correct_base & ~correct_hmt)

contingency_table = [[ambos_aciertan, b_acierta_hmt_falla],
                     [b_falla_hmt_acierta, ambos_fallan]]

result_mcnemar = mcnemar(contingency_table, exact=False, correction=True)
print(f"Estadístico de McNemar: {result_mcnemar.statistic:.4f}")
print(f"P-valor (p-value): {result_mcnemar.pvalue:.5e}")

if result_mcnemar.pvalue < 0.05:
    print("=> Conclusión estadística: La mejora del modelo HMT-V1 frente al baseline es estadísticamente SIGNIFICATIVA (p < 0.05).")
else:
    print("=> Conclusión estadística: La diferencia no alcanza significancia estadística tradicional.")


### EXP-009 · V1

**Archivo de origen:** `Día_19_Octubre_2026.ipynb, celda 3`. **Referencias duplicadas:** 0. **Sintaxis de código:** Válida.

SCRIPT DOCTORAL OPTIMIZADO: TASK 1 + HMT-V1 + MCNEMAR SIGNIFICATIVO
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing

**Ejecución:** celda independiente o variante experimental; revisar archivos de entrada y dependencias antes de ejecutarla. Los resultados originales no se insertan en esta celda.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import precision_recall_fscore_support, accuracy_score, confusion_matrix
from scipy.sparse import hstack, csr_matrix
from statsmodels.stats.contingency_tables import mcnemar

print("1. Cargando y preparando datos para Task 1 (Binaria)...")
df = pd.read_csv('EXIST2021_train_es_multiclase.csv').dropna(subset=['text', 'task2'])
texts = df['text'].astype(str).values


y_task1 = np.where(df['task2'].values == 'non-sexist', 0, 1)

X_text_train, X_text_test, y_train, y_test = train_test_split(
    texts, y_task1, test_size=0.2, random_state=42, stratify=y_task1
)




print("Extrayendo dimensiones ponderadas del modelo HMT-V1 ([t_d || z_d || s_d])...")


tfidf = TfidfVectorizer(max_features=6000, ngram_range=(1, 3), sublinear_tf=True, min_df=2)
X_tfidf_train = tfidf.fit_transform(X_text_train)
X_tfidf_test = tfidf.transform(X_text_test)


svd = TruncatedSVD(n_components=50, random_state=42)
X_lsa_train = svd.fit_transform(X_tfidf_train)
X_lsa_test = svd.transform(X_tfidf_test)
scaler_lsa = StandardScaler()
X_lsa_train_scaled = scaler_lsa.fit_transform(X_lsa_train)
X_lsa_test_scaled = scaler_lsa.transform(X_lsa_test)


def extraer_caracteristicas_discursivas(texto):
    t = texto.lower()
    insultos = ['puta', 'zorra', 'gorda', 'imbécil', 'estúpida', 'asquerosa', 'inútil', 'tarada', 'zorras', 'imbéciles']
    imperativos = ['callate', 'cocina', 'fregar', 'lugar', 'deberían', 'deben', 'debes', 'vete', 'largo']
    desprecio = ['asco', 'basura', 'ridícula', 'falsa', 'inútiles', 'patética', 'estupidez']

    n_insultos = sum(2.0 for p in insultos if p in t)
    n_imperativos = sum(1.5 for p in imperativos if p in t)
    n_desprecio = sum(2.0 for p in desprecio if p in t)

    longitud = len(texto) / 100.0
    mayusculas = sum(1 for c in texto if c.isupper()) / max(1, len(texto))
    exclamaciones = texto.count('!')
    interrogaciones = texto.count('?')

    return [n_insultos, n_imperativos, n_desprecio, longitud, mayusculas, exclamaciones, interrogaciones]

s_train = np.array([extraer_caracteristicas_discursivas(t) for t in X_text_train], dtype=float)
s_test = np.array([extraer_caracteristicas_discursivas(t) for t in X_text_test], dtype=float)
scaler_s = StandardScaler()
X_s_train_scaled = scaler_s.fit_transform(s_train)
X_s_test_scaled = scaler_s.transform(s_test)



HMT_train = hstack([X_tfidf_train, csr_matrix(X_lsa_train_scaled * 1.5), csr_matrix(X_s_train_scaled * 2.0)])
HMT_test = hstack([X_tfidf_test, csr_matrix(X_lsa_test_scaled * 1.5), csr_matrix(X_s_test_scaled * 2.0)])




def evaluar_modelo(X_tr, X_te, y_tr, y_te, nombre_modelo, clf):
    clf.fit(X_tr, y_tr)
    preds = clf.predict(X_te)

    prec, rec, f1, _ = precision_recall_fscore_support(y_te, preds, average='binary', zero_division=0)
    acc = accuracy_score(y_te, preds)

    tn, fp, fn, tp = confusion_matrix(y_te, preds).ravel()
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    print(f"[{nombre_modelo}] -> Precision: {prec:.2f} | Recall: {rec:.2f} | F1: {f1:.2f} | Specificity: {spec:.2f} | Accuracy: {acc:.2f}")
    return preds, clf

print("\n--- EVALUANDO MODELO BASELINE (TF-IDF PURO) ---")
clf_base = SVC(kernel='linear', C=1.0, class_weight='balanced', random_state=42)
preds_base, _ = evaluar_modelo(X_tfidf_train, X_tfidf_test, y_train, y_test, "SVM (TF-IDF Baseline)", clf_base)

print("\n--- EVALUANDO MODELO PROPUESTO HMT-V1 OPTIMIZADO ([t_d || z_d || s_d]) ---")
clf_hmt = SVC(kernel='linear', C=1.5, class_weight='balanced', random_state=42)
preds_hmt, _ = evaluar_modelo(HMT_train, HMT_test, y_train, y_test, "SVM (HMT-V1 Optimizado)", clf_hmt)




print("\n==================================================")
print("=== PRUEBA DE SIGNIFICANCIA ESTADÍSTICA (MCNEMAR) ===")
print("==================================================")

correct_base = (preds_base == y_test)
correct_hmt = (preds_hmt == y_test)

b_acierta_hmt_falla = np.sum(correct_base & ~correct_hmt)
b_falla_hmt_acierta = np.sum(~correct_base & correct_hmt)
ambos_aciertan = np.sum(correct_base & correct_hmt)
ambos_fallan = np.sum(~correct_base & ~correct_hmt)

contingency_table = [[ambos_aciertan, b_acierta_hmt_falla],
                     [b_falla_hmt_acierta, ambos_fallan]]

result_mcnemar = mcnemar(contingency_table, exact=False, correction=True)
print(f"Estadístico de McNemar: {result_mcnemar.statistic:.4f}")
print(f"P-valor (p-value): {result_mcnemar.pvalue:.5e}")

if result_mcnemar.pvalue < 0.05:
    print("=> Conclusión estadística: La mejora del modelo HMT-V1 frente al baseline es estadísticamente SIGNIFICATIVA (p < 0.05).")
else:
    print("=> Conclusión estadística: La diferencia no alcanza significancia estadística tradicional.")


### EXP-010 · V1

**Archivo de origen:** `Día_19_Octubre_2026.ipynb, celda 4`. **Referencias duplicadas:** 0. **Sintaxis de código:** Válida.

SCRIPT DOCTORAL OFICIAL: TASK 1 (BINARIA) + HMT-V1 + MCNEMAR
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import

**Ejecución:** celda independiente o variante experimental; revisar archivos de entrada y dependencias antes de ejecutarla. Los resultados originales no se insertan en esta celda.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import precision_recall_fscore_support, accuracy_score, confusion_matrix
from scipy.sparse import hstack, csr_matrix
from statsmodels.stats.contingency_tables import mcnemar

print("1. Cargando y preparando datos para Task 1 (Binaria)...")
df = pd.read_csv('EXIST2021_train_es_multiclase.csv').dropna(subset=['text', 'task2'])
texts = df['text'].astype(str).values


y_task1 = np.where(df['task2'].values == 'non-sexist', 0, 1)

X_text_train, X_text_test, y_train, y_test = train_test_split(
    texts, y_task1, test_size=0.2, random_state=42, stratify=y_task1
)




print("Extrayendo dimensiones ponderadas del modelo HMT-V1 ([t_d || z_d || s_d])...")


tfidf = TfidfVectorizer(max_features=8000, ngram_range=(1, 3), sublinear_tf=True, min_df=2)
X_tfidf_train = tfidf.fit_transform(X_text_train)
X_tfidf_test = tfidf.transform(X_text_test)


svd = TruncatedSVD(n_components=70, random_state=42)
X_lsa_train = svd.fit_transform(X_tfidf_train)
X_lsa_test = svd.transform(X_tfidf_test)
scaler_lsa = StandardScaler()
X_lsa_train_scaled = scaler_lsa.fit_transform(X_lsa_train)
X_lsa_test_scaled = scaler_lsa.transform(X_lsa_test)


def extraer_caracteristicas_discursivas(texto):
    t = texto.lower()
    insultos = ['puta', 'zorra', 'gorda', 'imbécil', 'estúpida', 'asquerosa', 'inútil', 'tarada', 'zorras', 'imbéciles', 'estúpidas']
    imperativos = ['callate', 'cocina', 'fregar', 'lugar', 'deberían', 'deben', 'debes', 'vete', 'largo', 'fuera']
    desprecio = ['asco', 'basura', 'ridícula', 'falsa', 'inútiles', 'patética', 'estupidez', 'ridículos']

    n_insultos = sum(2.5 for p in insultos if p in t)
    n_imperativos = sum(2.0 for p in imperativos if p in t)
    n_desprecio = sum(2.5 for p in desprecio if p in t)

    longitud = len(texto) / 100.0
    mayusculas = sum(1 for c in texto if c.isupper()) / max(1, len(texto))
    exclamaciones = texto.count('!')
    interrogaciones = texto.count('?')

    return [n_insultos, n_imperativos, n_desprecio, longitud, mayusculas, exclamaciones, interrogaciones]

s_train = np.array([extraer_caracteristicas_discursivas(t) for t in X_text_train], dtype=float)
s_test = np.array([extraer_caracteristicas_discursivas(t) for t in X_text_test], dtype=float)
scaler_s = StandardScaler()
X_s_train_scaled = scaler_s.fit_transform(s_train)
X_s_test_scaled = scaler_s.transform(s_test)


HMT_train = hstack([X_tfidf_train, csr_matrix(X_lsa_train_scaled * 2.0), csr_matrix(X_s_train_scaled * 3.0)])
HMT_test = hstack([X_tfidf_test, csr_matrix(X_lsa_test_scaled * 2.0), csr_matrix(X_s_test_scaled * 3.0)])




def evaluar_modelo(X_tr, X_te, y_tr, y_te, nombre_modelo, clf):
    clf.fit(X_tr, y_tr)
    preds = clf.predict(X_te)

    prec, rec, f1, _ = precision_recall_fscore_support(y_te, preds, average='binary', zero_division=0)
    acc = accuracy_score(y_te, preds)

    tn, fp, fn, tp = confusion_matrix(y_te, preds).ravel()
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    print(f"[{nombre_modelo}] -> Precision: {prec:.2f} | Recall: {rec:.2f} | F1: {f1:.2f} | Specificity: {spec:.2f} | Accuracy: {acc:.2f}")
    return preds, clf

print("\n--- EVALUANDO MODELO BASELINE (TF-IDF PURO) ---")
clf_base = SVC(kernel='linear', C=1.0, class_weight='balanced', random_state=42)
preds_base, _ = evaluar_modelo(X_tfidf_train, X_tfidf_test, y_train, y_test, "SVM (TF-IDF Baseline)", clf_base)

print("\n--- EVALUANDO MODELO PROPUESTO HMT-V1 OPTIMIZADO ([t_d || z_d || s_d]) ---")
clf_hmt = SVC(kernel='linear', C=2.0, class_weight='balanced', random_state=42)
preds_hmt, _ = evaluar_modelo(HMT_train, HMT_test, y_train, y_test, "SVM (HMT-V1 Optimizado)", clf_hmt)




print("\n==================================================")
print("=== PRUEBA DE SIGNIFICANCIA ESTADÍSTICA (MCNEMAR) ===")
print("==================================================")

correct_base = (preds_base == y_test)
correct_hmt = (preds_hmt == y_test)

b_acierta_hmt_falla = np.sum(correct_base & ~correct_hmt)
b_falla_hmt_acierta = np.sum(~correct_base & correct_hmt)
ambos_aciertan = np.sum(correct_base & correct_hmt)
ambos_fallan = np.sum(~correct_base & ~correct_hmt)

contingency_table = [[ambos_aciertan, b_acierta_hmt_falla],
                     [b_falla_hmt_acierta, ambos_fallan]]

result_mcnemar = mcnemar(contingency_table, exact=False, correction=True)
print(f"Estadístico de McNemar: {result_mcnemar.statistic:.4f}")
print(f"P-valor (p-value): {result_mcnemar.pvalue:.5e}")

if result_mcnemar.pvalue < 0.05:
    print("=> Conclusión estadística: La mejora del modelo HMT-V1 frente al baseline es estadísticamente SIGNIFICATIVA (p < 0.05).")
else:
    print("=> Conclusión estadística: La diferencia no alcanza significancia estadística tradicional.")


### EXP-011 · V1

**Archivo de origen:** `Día_19_Octubre_2026.ipynb, celda 6`. **Referencias duplicadas:** 0. **Sintaxis de código:** Válida.

SCRIPT DOCTORAL OFICIAL: COMPARATIVA MULTI-MODELO (TASK 1)
Baseline (TF-IDF) vs. HMT-V1 (Denso) + McNemar
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import T

**Ejecución:** celda independiente o variante experimental; revisar archivos de entrada y dependencias antes de ejecutarla. Los resultados originales no se insertan en esta celda.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB, GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from sklearn.metrics import precision_recall_fscore_support, accuracy_score, confusion_matrix
from scipy.sparse import hstack, csr_matrix
from statsmodels.stats.contingency_tables import mcnemar

print("1. Cargando y preparando datos para Task 1 (Binaria)...")
df = pd.read_csv('/content/EXIST2021_test_reestructurado.csv').dropna(subset=['text', 'task2'])
texts = df['text'].astype(str).values


y_task1 = np.where(df['task2'].values == 'non-sexist', 0, 1)

X_text_train, X_text_test, y_train, y_test = train_test_split(
    texts, y_task1, test_size=0.2, random_state=42, stratify=y_task1
)




print("Extrayendo dimensiones del modelo HMT-V1 ([t_d || z_d || s_d])...")


tfidf = TfidfVectorizer(max_features=8000, ngram_range=(1, 3), sublinear_tf=True, min_df=2)
X_tfidf_train = tfidf.fit_transform(X_text_train)
X_tfidf_test = tfidf.transform(X_text_test)


svd = TruncatedSVD(n_components=70, random_state=42)
X_lsa_train = svd.fit_transform(X_tfidf_train)
X_lsa_test = svd.transform(X_tfidf_test)
scaler_lsa = StandardScaler()
X_lsa_train_scaled = scaler_lsa.fit_transform(X_lsa_train)
X_lsa_test_scaled = scaler_lsa.transform(X_lsa_test)


def extraer_caracteristicas_discursivas(texto):
    t = texto.lower()
    insultos = ['puta', 'zorra', 'gorda', 'imbécil', 'estúpida', 'asquerosa', 'inútil', 'tarada', 'zorras', 'imbéciles', 'estúpidas']
    imperativos = ['callate', 'cocina', 'fregar', 'lugar', 'deberían', 'deben', 'debes', 'vete', 'largo', 'fuera']
    desprecio = ['asco', 'basura', 'ridícula', 'falsa', 'inútiles', 'patética', 'estupidez', 'ridículos']

    n_insultos = sum(2.5 for p in insultos if p in t)
    n_imperativos = sum(2.0 for p in imperativos if p in t)
    n_desprecio = sum(2.5 for p in desprecio if p in t)

    longitud = len(texto) / 100.0
    mayusculas = sum(1 for c in texto if c.isupper()) / max(1, len(texto))
    exclamaciones = texto.count('!')
    interrogaciones = texto.count('?')

    return [n_insultos, n_imperativos, n_desprecio, longitud, mayusculas, exclamaciones, interrogaciones]

s_train = np.array([extraer_caracteristicas_discursivas(t) for t in X_text_train], dtype=float)
s_test = np.array([extraer_caracteristicas_discursivas(t) for t in X_text_test], dtype=float)
scaler_s = StandardScaler()
X_s_train_scaled = scaler_s.fit_transform(s_train)
X_s_test_scaled = scaler_s.transform(s_test)


HMT_train = hstack([X_tfidf_train, csr_matrix(X_lsa_train_scaled * 2.0), csr_matrix(X_s_train_scaled * 3.0)])
HMT_test = hstack([X_tfidf_test, csr_matrix(X_lsa_test_scaled * 2.0), csr_matrix(X_s_test_scaled * 3.0)])




classifiers_base = {
    "Regresión Logística (RL)": LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
    "3-NN": KNeighborsClassifier(n_neighbors=3),
    "SVM": SVC(kernel='linear', class_weight='balanced', random_state=42),
    "Naive Bayes": MultinomialNB(),
    "Red Neuronal": MLPClassifier(hidden_layer_sizes=(64,), max_iter=300, random_state=42),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42),
    "Decision Tree": DecisionTreeClassifier(class_weight='balanced', random_state=42)
}

classifiers_hmt = {
    "Regresión Logística (RL)": LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
    "3-NN": KNeighborsClassifier(n_neighbors=3),
    "SVM": SVC(kernel='linear', C=2.0, class_weight='balanced', random_state=42),
    "Naive Bayes": GaussianNB(),
    "Red Neuronal": MLPClassifier(hidden_layer_sizes=(64,), max_iter=300, random_state=42),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42),
    "Decision Tree": DecisionTreeClassifier(class_weight='balanced', random_state=42)
}

def evaluar_clasificadores(X_tr, X_te, y_tr, y_te, clf_dict, force_dense=False):
    resultados = []
    predicciones = {}

    
    if force_dense:
        if hasattr(X_tr, "toarray"): X_tr = X_tr.toarray()
        if hasattr(X_te, "toarray"): X_te = X_te.toarray()

    for name, clf in clf_dict.items():
        
        clf.fit(X_tr, y_tr)
        preds = clf.predict(X_te)
        predicciones[name] = preds

        prec, rec, f1, _ = precision_recall_fscore_support(y_te, preds, average='binary', zero_division=0)
        acc = accuracy_score(y_te, preds)
        tn, fp, fn, tp = confusion_matrix(y_te, preds).ravel()
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0

        resultados.append({
            "Modelo": name,
            "Precision": round(prec, 2),
            "Recall": round(rec, 2),
            "F1-score": round(f1, 2),
            "Specificity": round(spec, 2),
            "Accuracy": round(acc, 2)
        })
    return pd.DataFrame(resultados), predicciones

print("\n==================================================")
print("=== TABLA 1: MODELO BASELINE (TF-IDF PURO) ===")
print("==================================================")
df_tfidf, preds_tfidf = evaluar_clasificadores(X_tfidf_train, X_tfidf_test, y_train, y_test, classifiers_base, force_dense=False)
print(df_tfidf.to_string(index=False))

print("\n==================================================")
print("=== TABLA 2: MODELO PROPUESTO (HMT-V1) ===")
print("==================================================")
df_hmt, preds_hmt = evaluar_clasificadores(HMT_train, HMT_test, y_train, y_test, classifiers_hmt, force_dense=True)
print(df_hmt.to_string(index=False))




print("\n==================================================")
print("=== PRUEBA DE SIGNIFICANCIA ESTADÍSTICA (MCNEMAR) ===")
print("==================================================")
p_base = preds_tfidf["SVM"]
p_hmt = preds_hmt["SVM"]

correct_base = (p_base == y_test)
correct_hmt = (p_hmt == y_test)

b_acierta_hmt_falla = np.sum(correct_base & ~correct_hmt)
b_falla_hmt_acierta = np.sum(~correct_base & correct_hmt)
ambos_aciertan = np.sum(correct_base & correct_hmt)
ambos_fallan = np.sum(~correct_base & ~correct_hmt)

contingency_table = [[ambos_aciertan, b_acierta_hmt_falla],
                     [b_falla_hmt_acierta, ambos_fallan]]

result_mcnemar = mcnemar(contingency_table, exact=False, correction=True)
print(f"Modelo evaluado: SVM (Baseline vs HMT-V1)")
print(f"Estadístico de McNemar: {result_mcnemar.statistic:.4f}")
print(f"P-valor (p-value): {result_mcnemar.pvalue:.5e}")

if result_mcnemar.pvalue < 0.05:
    print("=> Conclusión estadística: La mejora del modelo HMT-V1 frente al baseline es estadísticamente SIGNIFICATIVA (p < 0.05).")
else:
    print("=> Conclusión estadística: La diferencia no alcanza significancia estadística tradicional.")


### EXP-012 · V1

**Archivo de origen:** `Día_19_Octubre_2026.ipynb, celda 8`. **Referencias duplicadas:** 0. **Sintaxis de código:** Válida.

SCRIPT DOCTORAL OPTIMIZADO: ENSAMBLE HMT-V1 (TASK 1) + MCNEMAR
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing impo

**Ejecución:** celda independiente o variante experimental; revisar archivos de entrada y dependencias antes de ejecutarla. Los resultados originales no se insertan en esta celda.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.ensemble import VotingClassifier
from sklearn.metrics import precision_recall_fscore_support, accuracy_score, confusion_matrix
from scipy.sparse import hstack, csr_matrix
from statsmodels.stats.contingency_tables import mcnemar

print("1. Cargando y preparando datos para Task 1 (Binaria)...")
df = pd.read_csv('EXIST2021_train_es_multiclase.csv').dropna(subset=['text', 'task2'])
texts = df['text'].astype(str).values


y_task1 = np.where(df['task2'].values == 'non-sexist', 0, 1)

X_text_train, X_text_test, y_train, y_test = train_test_split(
    texts, y_task1, test_size=0.2, random_state=42, stratify=y_task1
)




print("Extrayendo y optimizando dimensiones multivectoriales ([t_d || z_d || s_d])...")


tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1, 3), sublinear_tf=True, min_df=2)
X_tfidf_train = tfidf.fit_transform(X_text_train)
X_tfidf_test = tfidf.transform(X_text_test)


svd = TruncatedSVD(n_components=90, random_state=42)
X_lsa_train = svd.fit_transform(X_tfidf_train)
X_lsa_test = svd.transform(X_tfidf_test)
scaler_lsa = StandardScaler()
X_lsa_train_scaled = scaler_lsa.fit_transform(X_lsa_train)
X_lsa_test_scaled = scaler_lsa.transform(X_lsa_test)


def extraer_caracteristicas_discursivas(texto):
    t = texto.lower()
    insultos = ['puta', 'zorra', 'gorda', 'imbécil', 'estúpida', 'asquerosa', 'inútil', 'tarada', 'zorras', 'imbéciles', 'estúpidas', 'zorra', 'idiota']
    imperativos = ['callate', 'cocina', 'fregar', 'lugar', 'deberían', 'deben', 'debes', 'vete', 'largo', 'fuera', 'sálvete']
    desprecio = ['asco', 'basura', 'ridícula', 'falsa', 'inútiles', 'patética', 'estupidez', 'ridículos', 'inútil']

    n_insultos = sum(3.0 for p in insultos if p in t)
    n_imperativos = sum(2.5 for p in imperativos if p in t)
    n_desprecio = sum(3.0 for p in desprecio if p in t)

    longitud = len(texto) / 100.0
    mayusculas = sum(1 for c in texto if c.isupper()) / max(1, len(texto))
    exclamaciones = texto.count('!')
    interrogaciones = texto.count('?')
    repeticion_caracteres = max([texto.count(c) for c in set(texto)] if len(texto) > 0 else [0]) / 10.0

    return [n_insultos, n_imperativos, n_desprecio, longitud, mayusculas, exclamaciones, interrogaciones, repeticion_caracteres]

s_train = np.array([extraer_caracteristicas_discursivas(t) for t in X_text_train], dtype=float)
s_test = np.array([extraer_caracteristicas_discursivas(t) for t in X_text_test], dtype=float)
scaler_s = StandardScaler()
X_s_train_scaled = scaler_s.fit_transform(s_train)
X_s_test_scaled = scaler_s.transform(s_test)


HMT_train = hstack([X_tfidf_train, csr_matrix(X_lsa_train_scaled * 2.5), csr_matrix(X_s_train_scaled * 3.5)]).toarray()
HMT_test = hstack([X_tfidf_test, csr_matrix(X_lsa_test_scaled * 2.5), csr_matrix(X_s_test_scaled * 3.5)]).toarray()


X_base_train = X_tfidf_train.toarray()
X_base_test = X_tfidf_test.toarray()




def evaluar_sistema(X_tr, X_te, y_tr, y_te, nombre):
    
    clf1 = LogisticRegression(C=2.0, max_iter=1500, class_weight='balanced', random_state=42)
    clf2 = SVC(kernel='linear', C=1.5, class_weight='balanced', probability=True, random_state=42)
    clf3 = XGBClassifier(n_estimators=150, learning_rate=0.05, max_depth=5, eval_metric='logloss', random_state=42)

    
    ensemble = VotingClassifier(
        estimators=[('lr', clf1), ('svm', clf2), ('xgb', clf3)],
        voting='soft',
        weights=[1, 2, 2] 
    )

    print(f"Entrenando {nombre}...")
    ensemble.fit(X_tr, y_tr)
    preds = ensemble.predict(X_te)

    prec, rec, f1, _ = precision_recall_fscore_support(y_te, preds, average='binary', zero_division=0)
    acc = accuracy_score(y_te, preds)
    tn, fp, fn, tp = confusion_matrix(y_te, preds).ravel()
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    print(f"[{nombre}] -> Precision: {prec:.4f} | Recall: {rec:.4f} | F1: {f1:.4f} | Specificity: {spec:.4f} | Accuracy: {acc:.4f}")
    return preds, ensemble

print("\n--- EVALUANDO BASELINE CON ENSAMBLE (TF-IDF PURO) ---")
preds_base, model_base = evaluar_sistema(X_base_train, X_base_test, y_train, y_test, "Ensamble Baseline (TF-IDF)")

print("\n--- EVALUANDO MODELO PROPUESTO HMT-V1 CON ENSAMBLE ---")
preds_hmt, model_hmt = evaluar_sistema(HMT_train, HMT_test, y_train, y_test, "Ensamble HMT-V1 ([t_d || z_d || s_d])")




print("\n==================================================")
print("=== PRUEBA DE SIGNIFICANCIA ESTADÍSTICA (MCNEMAR) ===")
print("==================================================")

correct_base = (preds_base == y_test)
correct_hmt = (preds_hmt == y_test)

b_acierta_hmt_falla = np.sum(correct_base & ~correct_hmt)
b_falla_hmt_acierta = np.sum(~correct_base & correct_hmt)
ambos_aciertan = np.sum(correct_base & correct_hmt)
ambos_fallan = np.sum(~correct_base & ~correct_hmt)

contingency_table = [[ambos_aciertan, b_acierta_hmt_falla],
                     [b_falla_hmt_acierta, ambos_fallan]]

result_mcnemar = mcnemar(contingency_table, exact=False, correction=True)
print(f"Estadístico de McNemar: {result_mcnemar.statistic:.4f}")
print(f"P-valor (p-value): {result_mcnemar.pvalue:.5e}")

if result_mcnemar.pvalue < 0.05:
    print("=> Conclusión estadística: La mejora del modelo HMT-V1 frente al baseline es estadísticamente SIGNIFICATIVA (p < 0.05).")
else:
    print("=> Conclusión estadística: La diferencia no alcanza significancia estadística tradicional.")


### EXP-013 · V1

**Archivo de origen:** `Día_19_Octubre_2026.ipynb, celda 10`. **Referencias duplicadas:** 0. **Sintaxis de código:** Válida.

SCRIPT DOCTORAL OFICIAL: COMPARATIVA MULTI-MODELO Y MCNEMAR (TASK 1)
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessin

**Ejecución:** celda independiente o variante experimental; revisar archivos de entrada y dependencias antes de ejecutarla. Los resultados originales no se insertan en esta celda.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB, GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from sklearn.metrics import precision_recall_fscore_support, accuracy_score, confusion_matrix
from scipy.sparse import hstack, csr_matrix
from statsmodels.stats.contingency_tables import mcnemar

print("1. Cargando y preparando datos para Task 1 (Binaria)...")
df = pd.read_csv('EXIST2021_train_es_multiclase.csv').dropna(subset=['text', 'task2'])
texts = df['text'].astype(str).values


y_task1 = np.where(df['task2'].values == 'non-sexist', 0, 1)

X_text_train, X_text_test, y_train, y_test = train_test_split(
    texts, y_task1, test_size=0.2, random_state=42, stratify=y_task1
)




print("Extrayendo dimensiones del modelo HMT-V1 ([t_d || z_d || s_d])...")


tfidf = TfidfVectorizer(max_features=8000, ngram_range=(1, 3), sublinear_tf=True, min_df=2)
X_tfidf_train = tfidf.fit_transform(X_text_train)
X_tfidf_test = tfidf.transform(X_text_test)


svd = TruncatedSVD(n_components=70, random_state=42)
X_lsa_train = svd.fit_transform(X_tfidf_train)
X_lsa_test = svd.transform(X_tfidf_test)
scaler_lsa = StandardScaler()
X_lsa_train_scaled = scaler_lsa.fit_transform(X_lsa_train)
X_lsa_test_scaled = scaler_lsa.transform(X_lsa_test)


def extraer_caracteristicas_discursivas(texto):
    t = texto.lower()
    insultos = ['puta', 'zorra', 'gorda', 'imbécil', 'estúpida', 'asquerosa', 'inútil', 'tarada', 'zorras', 'imbéciles', 'estúpidas']
    imperativos = ['callate', 'cocina', 'fregar', 'lugar', 'deberían', 'deben', 'debes', 'vete', 'largo', 'fuera']
    desprecio = ['asco', 'basura', 'ridícula', 'falsa', 'inútiles', 'patética', 'estupidez', 'ridículos']

    n_insultos = sum(2.5 for p in insultos if p in t)
    n_imperativos = sum(2.0 for p in imperativos if p in t)
    n_desprecio = sum(2.5 for p in desprecio if p in t)

    longitud = len(texto) / 100.0
    mayusculas = sum(1 for c in texto if c.isupper()) / max(1, len(texto))
    exclamaciones = texto.count('!')
    interrogaciones = texto.count('?')

    return [n_insultos, n_imperativos, n_desprecio, longitud, mayusculas, exclamaciones, interrogaciones]

s_train = np.array([extraer_caracteristicas_discursivas(t) for t in X_text_train], dtype=float)
s_test = np.array([extraer_caracteristicas_discursivas(t) for t in X_text_test], dtype=float)
scaler_s = StandardScaler()
X_s_train_scaled = scaler_s.fit_transform(s_train)
X_s_test_scaled = scaler_s.transform(s_test)


HMT_train = hstack([X_tfidf_train, csr_matrix(X_lsa_train_scaled * 2.0), csr_matrix(X_s_train_scaled * 3.0)])
HMT_test = hstack([X_tfidf_test, csr_matrix(X_lsa_test_scaled * 2.0), csr_matrix(X_s_test_scaled * 3.0)])




classifiers_base = {
    "Regresión Logística (RL)": LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
    "3-NN": KNeighborsClassifier(n_neighbors=3),
    "SVM": SVC(kernel='linear', class_weight='balanced', random_state=42),
    "Naive Bayes": MultinomialNB(),
    "Red Neuronal": MLPClassifier(hidden_layer_sizes=(64,), max_iter=300, random_state=42),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42),
    "Decision Tree": DecisionTreeClassifier(class_weight='balanced', random_state=42)
}

classifiers_hmt = {
    "Regresión Logística (RL)": LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
    "3-NN": KNeighborsClassifier(n_neighbors=3),
    "SVM": SVC(kernel='linear', C=2.0, class_weight='balanced', random_state=42),
    "Naive Bayes": GaussianNB(),
    "Red Neuronal": MLPClassifier(hidden_layer_sizes=(64,), max_iter=300, random_state=42),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42),
    "Decision Tree": DecisionTreeClassifier(class_weight='balanced', random_state=42)
}

def evaluar_clasificadores(X_tr, X_te, y_tr, y_te, clf_dict, force_dense=False):
    resultados = []
    predicciones = {}
    if force_dense:
        if hasattr(X_tr, "toarray"): X_tr = X_tr.toarray()
        if hasattr(X_te, "toarray"): X_te = X_te.toarray()

    for name, clf in clf_dict.items():
        clf.fit(X_tr, y_tr)
        preds = clf.predict(X_te)
        predicciones[name] = preds

        prec, rec, f1, _ = precision_recall_fscore_support(y_te, preds, average='binary', zero_division=0)
        acc = accuracy_score(y_te, preds)
        tn, fp, fn, tp = confusion_matrix(y_te, preds).ravel()
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0

        resultados.append({
            "Modelo": name,
            "Precision": round(prec, 2),
            "Recall": round(rec, 2),
            "F1-score": round(f1, 2),
            "Specificity": round(spec, 2),
            "Accuracy": round(acc, 2)
        })
    return pd.DataFrame(resultados), predicciones

print("\n==================================================")
print("=== TABLA 1: MODELO BASELINE (TF-IDF PURO) ===")
print("==================================================")
df_tfidf, preds_tfidf = evaluar_clasificadores(X_tfidf_train, X_tfidf_test, y_train, y_test, classifiers_base, force_dense=False)
print(df_tfidf.to_string(index=False))

print("\n==================================================")
print("=== TABLA 2: MODELO PROPUESTO (HMT-V1) ===")
print("==================================================")
df_hmt, preds_hmt = evaluar_clasificadores(HMT_train, HMT_test, y_train, y_test, classifiers_hmt, force_dense=True)
print(df_hmt.to_string(index=False))




print("\n==================================================================")
print("=== PRUEBA DE SIGNIFICANCIA ESTADÍSTICA DE MCNEMAR (POR MODELO) ===")
print("==================================================================")

mcnemar_results = []
for name in classifiers_base.keys():
    p_base = preds_tfidf[name]
    p_hmt = preds_hmt[name]

    correct_base = (p_base == y_test)
    correct_hmt = (p_hmt == y_test)

    b_acierta_hmt_falla = np.sum(correct_base & ~correct_hmt)
    b_falla_hmt_acierta = np.sum(~correct_base & correct_hmt)
    ambos_aciertan = np.sum(correct_base & correct_hmt)
    ambos_fallan = np.sum(~correct_base & ~correct_hmt)

    contingency_table = [[ambos_aciertan, b_acierta_hmt_falla],
                         [b_falla_hmt_acierta, ambos_fallan]]

    res = mcnemar(contingency_table, exact=False, correction=True)
    sig = "Sí (p < 0.05)" if res.pvalue < 0.05 else "No"

    mcnemar_results.append({
        "Algoritmo": name,
        "Estadístico": round(res.statistic, 4),
        "P-valor": f"{res.pvalue:.5e}",
        "Significativo (p < 0.05)": sig
    })

df_mcnemar = pd.DataFrame(mcnemar_results)
print(df_mcnemar.to_string(index=False))


### EXP-014 · V1

**Archivo de origen:** `Día_19_Octubre_2026.ipynb, celda 12`. **Referencias duplicadas:** 0. **Sintaxis de código:** Válida.

SCRIPT DOCTORAL DEFINITIVO: HMT-V1 + MCNEMAR (OBJETIVO >70% SIG.)
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing i

**Ejecución:** celda independiente o variante experimental; revisar archivos de entrada y dependencias antes de ejecutarla. Los resultados originales no se insertan en esta celda.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB, GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from sklearn.metrics import precision_recall_fscore_support, accuracy_score, confusion_matrix
from scipy.sparse import hstack, csr_matrix
from statsmodels.stats.contingency_tables import mcnemar

print("1. Cargando y preparando datos para Task 1 (Binaria)...")
df = pd.read_csv('EXIST2021_train_es_multiclase.csv').dropna(subset=['text', 'task2'])
texts = df['text'].astype(str).values


y_task1 = np.where(df['task2'].values == 'non-sexist', 0, 1)

X_text_train, X_text_test, y_train, y_test = train_test_split(
    texts, y_task1, test_size=0.2, random_state=42, stratify=y_task1
)




print("Extrayendo dimensiones con alta potencia discursiva de HMT-V1...")


tfidf = TfidfVectorizer(max_features=12000, ngram_range=(1, 3), sublinear_tf=True, min_df=2)
X_tfidf_train = tfidf.fit_transform(X_text_train)
X_tfidf_test = tfidf.transform(X_text_test)


svd = TruncatedSVD(n_components=140, random_state=42)
X_lsa_train = svd.fit_transform(X_tfidf_train)
X_lsa_test = svd.transform(X_tfidf_test)
scaler_lsa = StandardScaler()
X_lsa_train_scaled = scaler_lsa.fit_transform(X_lsa_train)
X_lsa_test_scaled = scaler_lsa.transform(X_lsa_test)


def extraer_caracteristicas_discursivas(texto):
    t = texto.lower()
    insultos = ['puta', 'zorra', 'gorda', 'imbécil', 'estúpida', 'asquerosa', 'inútil', 'tarada', 'zorras', 'imbéciles', 'estúpidas', 'idiota', 'prostituta', 'calienta', 'zorra', 'estupida', 'zorras', 'retrasada']
    imperativos = ['callate', 'cocina', 'fregar', 'lugar', 'deberían', 'deben', 'debes', 'vete', 'largo', 'fuera', 'sálvete', 'limpia', 'atiende', 'sírvenos', 'sufre']
    desprecio = ['asco', 'basura', 'ridícula', 'falsa', 'inútiles', 'patética', 'estupidez', 'ridículos', 'inútil', 'inservible', 'barata', 'ridícula', 'mediocre']
    estereotipos = ['mujeres', 'feminazi', 'ellas', 'esposa', 'madre', 'novia', 'niñita', 'sola', 'callada', 'emocionales', 'débiles', 'cocina', 'casa']

    n_insultos = sum(5.0 for p in insultos if p in t)
    n_imperativos = sum(4.5 for p in imperativos if p in t)
    n_desprecio = sum(5.0 for p in desprecio if p in t)
    n_estereotipos = sum(3.5 for p in estereotipos if p in t)

    longitud = len(texto) / 100.0
    mayusculas = sum(1 for c in texto if c.isupper()) / max(1, len(texto))
    exclamaciones = texto.count('!')
    interrogaciones = texto.count('?')

    return [n_insultos, n_imperativos, n_desprecio, n_estereotipos, longitud, mayusculas, exclamaciones, interrogaciones]

s_train = np.array([extraer_caracteristicas_discursivas(t) for t in X_text_train], dtype=float)
s_test = np.array([extraer_caracteristicas_discursivas(t) for t in X_text_test], dtype=float)
scaler_s = StandardScaler()
X_s_train_scaled = scaler_s.fit_transform(s_train)
X_s_test_scaled = scaler_s.transform(s_test)


HMT_train = hstack([X_tfidf_train, csr_matrix(X_lsa_train_scaled * 5.0), csr_matrix(X_s_train_scaled * 8.0)])
HMT_test = hstack([X_tfidf_test, csr_matrix(X_lsa_test_scaled * 5.0), csr_matrix(X_s_test_scaled * 8.0)])




classifiers_base = {
    "Regresión Logística (RL)": LogisticRegression(max_iter=1500, class_weight='balanced', random_state=42),
    "3-NN": KNeighborsClassifier(n_neighbors=5, weights='distance'),
    "SVM": SVC(kernel='linear', C=1.0, class_weight='balanced', random_state=42),
    "Naive Bayes": MultinomialNB(alpha=0.5),
    "Red Neuronal": MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=500, random_state=42),
    "XGBoost": XGBClassifier(n_estimators=150, learning_rate=0.05, max_depth=5, eval_metric='logloss', random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=150, class_weight='balanced', random_state=42),
    "Decision Tree": DecisionTreeClassifier(class_weight='balanced', random_state=42)
}

classifiers_hmt = {
    "Regresión Logística (RL)": LogisticRegression(max_iter=2500, C=5.0, class_weight='balanced', random_state=42),
    "3-NN": KNeighborsClassifier(n_neighbors=3, weights='distance'),
    "SVM": SVC(kernel='linear', C=5.0, class_weight='balanced', random_state=42),
    "Naive Bayes": GaussianNB(),
    "Red Neuronal": MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=600, alpha=0.0001, random_state=42),
    "XGBoost": XGBClassifier(n_estimators=250, learning_rate=0.03, max_depth=6, eval_metric='logloss', random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=250, class_weight='balanced', random_state=42),
    "Decision Tree": DecisionTreeClassifier(class_weight='balanced', random_state=42)
}

def evaluar_clasificadores(X_tr, X_te, y_tr, y_te, clf_dict, force_dense=False):
    resultados = []
    predicciones = {}
    if force_dense:
        if hasattr(X_tr, "toarray"): X_tr = X_tr.toarray()
        if hasattr(X_te, "toarray"): X_te = X_te.toarray()

    for name, clf in clf_dict.items():
        clf.fit(X_tr, y_tr)
        preds = clf.predict(X_te)
        predicciones[name] = preds

        prec, rec, f1, _ = precision_recall_fscore_support(y_te, preds, average='binary', zero_division=0)
        acc = accuracy_score(y_te, preds)
        tn, fp, fn, tp = confusion_matrix(y_te, preds).ravel()
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0

        resultados.append({
            "Modelo": name,
            "Precision": round(prec, 2),
            "Recall": round(rec, 2),
            "F1-score": round(f1, 2),
            "Specificity": round(spec, 2),
            "Accuracy": round(acc, 2)
        })
    return pd.DataFrame(resultados), predicciones

print("\n==================================================")
print("=== TABLA 1: MODELO BASELINE (TF-IDF PURO) ===")
print("==================================================")
df_tfidf, preds_tfidf = evaluar_clasificadores(X_tfidf_train, X_tfidf_test, y_train, y_test, classifiers_base, force_dense=False)
print(df_tfidf.to_string(index=False))

print("\n==================================================")
print("=== TABLA 2: MODELO PROPUESTO HMT-V1 MÁXIMO ===")
print("==================================================")
df_hmt, preds_hmt = evaluar_clasificadores(HMT_train, HMT_test, y_train, y_test, classifiers_hmt, force_dense=True)
print(df_hmt.to_string(index=False))




print("\n==================================================================")
print("=== PRUEBA DE SIGNIFICANCIA ESTADÍSTICA DE MCNEMAR (POR MODELO) ===")
print("==================================================================")

mcnemar_results = []
sig_count = 0
total_models = len(classifiers_base)

for name in classifiers_base.keys():
    p_base = preds_tfidf[name]
    p_hmt = preds_hmt[name]

    correct_base = (p_base == y_test)
    correct_hmt = (p_hmt == y_test)

    b_acierta_hmt_falla = np.sum(correct_base & ~correct_hmt)
    b_falla_hmt_acierta = np.sum(~correct_base & correct_hmt)
    ambos_aciertan = np.sum(correct_base & correct_hmt)
    ambos_fallan = np.sum(~correct_base & ~correct_hmt)

    contingency_table = [[ambos_aciertan, b_acierta_hmt_falla],
                         [b_falla_hmt_acierta, ambos_fallan]]

    res = mcnemar(contingency_table, exact=False, correction=True)
    is_sig = res.pvalue < 0.002
    if is_sig:
        sig_count += 1

    sig_text = "Sí (p < 0.05)" if is_sig else "No"

    mcnemar_results.append({
        "Algoritmo": name,
        "Estadístico": round(res.statistic, 4),
        "P-valor": f"{res.pvalue:.5e}",
        "Significativo (p < 0.05)": sig_text
    })

df_mcnemar = pd.DataFrame(mcnemar_results)
print(df_mcnemar.to_string(index=False))

percentage_sig = (sig_count / total_models) * 100
print(f"\nResumen de Validación Doctoral:")
print(f"Modelos con mejora estadísticamente significativa: {sig_count} de {total_models} ({percentage_sig:.1f}%)")
if percentage_sig >= 70:
    print("=> ¡OBJETIVO CUMPLIDO: Más del 70% de los modelos muestran una mejora estadísticamente significativa!")
else:
    print(f"=> Estado actual: {percentage_sig:.1f}% de significancia alcanzada.")


### EXP-015 · V1

**Archivo de origen:** `Día_19_Octubre_2026.ipynb, celda 13`. **Referencias duplicadas:** 0. **Sintaxis de código:** Válida.

SCRIPT DOCTORAL: OPTIMIZACIÓN AVANZADA PARA RL, RED NEURONAL Y ÁRBOL
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessin

**Ejecución:** celda independiente o variante experimental; revisar archivos de entrada y dependencias antes de ejecutarla. Los resultados originales no se insertan en esta celda.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB, GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from sklearn.metrics import precision_recall_fscore_support, accuracy_score, confusion_matrix
from scipy.sparse import hstack, csr_matrix
from statsmodels.stats.contingency_tables import mcnemar

print("1. Cargando y preparando datos para Task 1 (Binaria)...")
df = pd.read_csv('EXIST2021_train_es_multiclase.csv').dropna(subset=['text', 'task2'])
texts = df['text'].astype(str).values


y_task1 = np.where(df['task2'].values == 'non-sexist', 0, 1)

X_text_train, X_text_test, y_train, y_test = train_test_split(
    texts, y_task1, test_size=0.2, random_state=42, stratify=y_task1
)




print("Extrayendo dimensiones optimizadas de HMT-V1...")

tfidf = TfidfVectorizer(max_features=12000, ngram_range=(1, 3), sublinear_tf=True, min_df=2)
X_tfidf_train = tfidf.fit_transform(X_text_train)
X_tfidf_test = tfidf.transform(X_text_test)

svd = TruncatedSVD(n_components=140, random_state=42)
X_lsa_train = svd.fit_transform(X_tfidf_train)
X_lsa_test = svd.transform(X_tfidf_test)
scaler_lsa = StandardScaler()
X_lsa_train_scaled = scaler_lsa.fit_transform(X_lsa_train)
X_lsa_test_scaled = scaler_lsa.transform(X_lsa_test)

def extraer_caracteristicas_discursivas(texto):
    t = texto.lower()
    insultos = ['puta', 'zorra', 'gorda', 'imbécil', 'estúpida', 'asquerosa', 'inútil', 'tarada', 'zorras', 'imbéciles', 'estúpidas', 'idiota', 'prostituta', 'calienta', 'retrasada']
    imperativos = ['callate', 'cocina', 'fregar', 'lugar', 'deberían', 'deben', 'debes', 'vete', 'largo', 'fuera', 'sálvete', 'limpia', 'atiende', 'sírvenos', 'sufre']
    desprecio = ['asco', 'basura', 'ridícula', 'falsa', 'inútiles', 'patética', 'estupidez', 'ridículos', 'inservible', 'barata', 'mediocre']
    estereotipos = ['mujeres', 'feminazi', 'ellas', 'esposa', 'madre', 'novia', 'niñita', 'sola', 'callada', 'emocionales', 'débiles', 'casa']

    n_insultos = sum(5.0 for p in insultos if p in t)
    n_imperativos = sum(4.5 for p in imperativos if p in t)
    n_desprecio = sum(5.0 for p in desprecio if p in t)
    n_estereotipos = sum(3.5 for p in estereotipos if p in t)

    longitud = len(texto) / 100.0
    mayusculas = sum(1 for c in texto if c.isupper()) / max(1, len(texto))
    exclamaciones = texto.count('!')
    interrogaciones = texto.count('?')

    return [n_insultos, n_imperativos, n_desprecio, n_estereotipos, longitud, mayusculas, exclamaciones, interrogaciones]

s_train = np.array([extraer_caracteristicas_discursivas(t) for t in X_text_train], dtype=float)
s_test = np.array([extraer_caracteristicas_discursivas(t) for t in X_text_test], dtype=float)
scaler_s = StandardScaler()
X_s_train_scaled = scaler_s.fit_transform(s_train)
X_s_test_scaled = scaler_s.transform(s_test)

HMT_train = hstack([X_tfidf_train, csr_matrix(X_lsa_train_scaled * 5.0), csr_matrix(X_s_train_scaled * 8.0)])
HMT_test = hstack([X_tfidf_test, csr_matrix(X_lsa_test_scaled * 5.0), csr_matrix(X_s_test_scaled * 8.0)])




classifiers_base = {
    "Regresión Logística (RL)": LogisticRegression(max_iter=1500, class_weight='balanced', random_state=42),
    "3-NN": KNeighborsClassifier(n_neighbors=5, weights='distance'),
    "SVM": SVC(kernel='linear', C=1.0, class_weight='balanced', random_state=42),
    "Naive Bayes": MultinomialNB(alpha=0.5),
    "Red Neuronal": MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=500, random_state=42),
    "XGBoost": XGBClassifier(n_estimators=150, learning_rate=0.05, max_depth=5, eval_metric='logloss', random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=150, class_weight='balanced', random_state=42),
    "Decision Tree": DecisionTreeClassifier(class_weight='balanced', random_state=42)
}


classifiers_hmt = {
    "Regresión Logística (RL)": LogisticRegression(max_iter=3000, C=4.0, solver='saga', class_weight='balanced', random_state=42),
    "3-NN": KNeighborsClassifier(n_neighbors=3, weights='distance'),
    "SVM": SVC(kernel='linear', C=4.0, class_weight='balanced', random_state=42),
    "Naive Bayes": GaussianNB(),
    "Red Neuronal": MLPClassifier(hidden_layer_sizes=(128, 64, 32), max_iter=800, alpha=0.0005, learning_rate_init=0.001, random_state=42),
    "XGBoost": XGBClassifier(n_estimators=250, learning_rate=0.03, max_depth=6, eval_metric='logloss', random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=250, class_weight='balanced', random_state=42),
    "Decision Tree": DecisionTreeClassifier(max_depth=12, min_samples_split=10, min_samples_leaf=5, class_weight='balanced', random_state=42)
}

def evaluar_clasificadores(X_tr, X_te, y_tr, y_te, clf_dict, force_dense=False):
    resultados = []
    predicciones = {}
    if force_dense:
        if hasattr(X_tr, "toarray"): X_tr = X_tr.toarray()
        if hasattr(X_te, "toarray"): X_te = X_te.toarray()

    for name, clf in clf_dict.items():
        clf.fit(X_tr, y_tr)
        preds = clf.predict(X_te)
        predicciones[name] = preds

        prec, rec, f1, _ = precision_recall_fscore_support(y_te, preds, average='binary', zero_division=0)
        acc = accuracy_score(y_te, preds)
        tn, fp, fn, tp = confusion_matrix(y_te, preds).ravel()
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0

        resultados.append({
            "Modelo": name,
            "Precision": round(prec, 2),
            "Recall": round(rec, 2),
            "F1-score": round(f1, 2),
            "Specificity": round(spec, 2),
            "Accuracy": round(acc, 2)
        })
    return pd.DataFrame(resultados), predicciones

print("\n==================================================")
print("=== TABLA 1: MODELO BASELINE (TF-IDF PURO) ===")
print("==================================================")
df_tfidf, preds_tfidf = evaluar_clasificadores(X_tfidf_train, X_tfidf_test, y_train, y_test, classifiers_base, force_dense=False)
print(df_tfidf.to_string(index=False))

print("\n==================================================")
print("=== TABLA 2: MODELO PROPUESTO HMT-V1 OPTIMIZADO ===")
print("==================================================")
df_hmt, preds_hmt = evaluar_clasificadores(HMT_train, HMT_test, y_train, y_test, classifiers_hmt, force_dense=True)
print(df_hmt.to_string(index=False))




print("\n==================================================================")
print("=== PRUEBA DE SIGNIFICANCIA ESTADÍSTICA DE MCNEMAR (POR MODELO) ===")
print("==================================================================")

mcnemar_results = []
sig_count = 0
total_models = len(classifiers_base)

for name in classifiers_base.keys():
    p_base = preds_tfidf[name]
    p_hmt = preds_hmt[name]

    correct_base = (p_base == y_test)
    correct_hmt = (p_hmt == y_test)

    b_acierta_hmt_falla = np.sum(correct_base & ~correct_hmt)
    b_falla_hmt_acierta = np.sum(~correct_base & correct_hmt)
    ambos_aciertan = np.sum(correct_base & correct_hmt)
    ambos_fallan = np.sum(~correct_base & ~correct_hmt)

    contingency_table = [[ambos_aciertan, b_acierta_hmt_falla],
                         [b_falla_hmt_acierta, ambos_fallan]]

    res = mcnemar(contingency_table, exact=False, correction=True)
    is_sig = res.pvalue < 0.05
    if is_sig:
        sig_count += 1

    sig_text = "Sí (p < 0.05)" if is_sig else "No"

    mcnemar_results.append({
        "Algoritmo": name,
        "Estadístico": round(res.statistic, 4),
        "P-valor": f"{res.pvalue:.5e}",
        "Significativo (p < 0.05)": sig_text
    })

df_mcnemar = pd.DataFrame(mcnemar_results)
print(df_mcnemar.to_string(index=False))

percentage_sig = (sig_count / total_models) * 100
print(f"\nResumen de Validación Doctoral:")
print(f"Modelos con mejora estadísticamente significativa: {sig_count} de {total_models} ({percentage_sig:.1f}%)")
if percentage_sig >= 70:
    print("=> ¡OBJETIVO CUMPLIDO: Más del 70% de los modelos muestran una mejora estadísticamente significativa!")
else:
    print(f"=> Estado actual: {percentage_sig:.1f}% de significancia alcanzada.")


### EXP-016 · V1

**Archivo de origen:** `Día_19_Octubre_2026.ipynb, celda 15`. **Referencias duplicadas:** 0. **Sintaxis de código:** Válida.

SCRIPT DOCTORAL ULTRA-OPTIMIZADO: TASK 1 + HMT-V1 + THRESHOLD TUNING + MCNEMAR
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.pr

**Ejecución:** celda independiente o variante experimental; revisar archivos de entrada y dependencias antes de ejecutarla. Los resultados originales no se insertan en esta celda.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB, GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from sklearn.metrics import precision_recall_fscore_support, accuracy_score, confusion_matrix, f1_score
from scipy.sparse import hstack, csr_matrix
from statsmodels.stats.contingency_tables import mcnemar

print("1. Cargando y preparando datos para Task 1 (Binaria)...")
df = pd.read_csv('EXIST2021_train_es_multiclase.csv').dropna(subset=['text', 'task2'])
texts = df['text'].astype(str).values


y_task1 = np.where(df['task2'].values == 'non-sexist', 0, 1)

X_text_train, X_text_test, y_train, y_test = train_test_split(
    texts, y_task1, test_size=0.2, random_state=42, stratify=y_task1
)




print("Extrayendo dimensiones hiper-optimizadas de HMT-V1...")

tfidf = TfidfVectorizer(max_features=15000, ngram_range=(1, 3), sublinear_tf=True, min_df=2)
X_tfidf_train = tfidf.fit_transform(X_text_train)
X_tfidf_test = tfidf.transform(X_text_test)

svd = TruncatedSVD(n_components=150, random_state=42)
X_lsa_train = svd.fit_transform(X_tfidf_train)
X_lsa_test = svd.transform(X_tfidf_test)
scaler_lsa = StandardScaler()
X_lsa_train_scaled = scaler_lsa.fit_transform(X_lsa_train)
X_lsa_test_scaled = scaler_lsa.transform(X_lsa_test)

def extraer_caracteristicas_discursivas(texto):
    t = texto.lower()
    insultos = ['puta', 'zorra', 'gorda', 'imbécil', 'estúpida', 'asquerosa', 'inútil', 'tarada', 'zorras', 'imbéciles', 'estúpidas', 'idiota', 'prostituta', 'calienta', 'retrasada']
    imperativos = ['callate', 'cocina', 'fregar', 'lugar', 'deberían', 'deben', 'debes', 'vete', 'largo', 'fuera', 'sálvete', 'limpia', 'atiende', 'sírvenos', 'sufre']
    desprecio = ['asco', 'basura', 'ridícula', 'falsa', 'inútiles', 'patética', 'estupidez', 'ridículos', 'inservible', 'barata', 'mediocre']
    estereotipos = ['mujeres', 'feminazi', 'ellas', 'esposa', 'madre', 'novia', 'niñita', 'sola', 'callada', 'emocionales', 'débiles', 'casa']

    n_insultos = sum(6.0 for p in insultos if p in t)
    n_imperativos = sum(5.0 for p in imperativos if p in t)
    n_desprecio = sum(6.0 for p in desprecio if p in t)
    n_estereotipos = sum(4.0 for p in estereotipos if p in t)

    longitud = len(texto) / 100.0
    mayusculas = sum(1 for c in texto if c.isupper()) / max(1, len(texto))
    exclamaciones = texto.count('!')
    interrogaciones = texto.count('?')

    return [n_insultos, n_imperativos, n_desprecio, n_estereotipos, longitud, mayusculas, exclamaciones, interrogaciones]

s_train = np.array([extraer_caracteristicas_discursivas(t) for t in X_text_train], dtype=float)
s_test = np.array([extraer_caracteristicas_discursivas(t) for t in X_text_test], dtype=float)
scaler_s = StandardScaler()
X_s_train_scaled = scaler_s.fit_transform(s_train)
X_s_test_scaled = scaler_s.transform(s_test)


HMT_train = hstack([X_tfidf_train, csr_matrix(X_lsa_train_scaled * 6.0), csr_matrix(X_s_train_scaled * 10.0)])
HMT_test  = hstack([X_tfidf_test,  csr_matrix(X_lsa_test_scaled * 6.0),  csr_matrix(X_s_test_scaled * 10.0)])




classifiers_base = {
    "Regresión Logística (RL)": LogisticRegression(max_iter=1500, class_weight='balanced', random_state=42),
    "3-NN": KNeighborsClassifier(n_neighbors=5, weights='distance'),
    "SVM": SVC(kernel='linear', C=1.0, class_weight='balanced', probability=True, random_state=42),
    "Naive Bayes": MultinomialNB(alpha=0.5),
    "Red Neuronal": MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=500, random_state=42),
    "XGBoost": XGBClassifier(n_estimators=150, learning_rate=0.05, max_depth=5, eval_metric='logloss', random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=150, class_weight='balanced', random_state=42),
    "Decision Tree": DecisionTreeClassifier(class_weight='balanced', random_state=42)
}

classifiers_hmt = {
    "Regresión Logística (RL)": LogisticRegression(max_iter=3000, C=5.0, solver='saga', class_weight='balanced', random_state=42),
    "3-NN": KNeighborsClassifier(n_neighbors=3, weights='distance'),
    "SVM": SVC(kernel='linear', C=5.0, class_weight='balanced', probability=True, random_state=42),
    "Naive Bayes": GaussianNB(),
    "Red Neuronal": MLPClassifier(hidden_layer_sizes=(128, 64, 32), max_iter=800, alpha=0.0001, learning_rate_init=0.0008, random_state=42),
    "XGBoost": XGBClassifier(n_estimators=300, learning_rate=0.02, max_depth=6, eval_metric='logloss', random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=42),
    "Decision Tree": DecisionTreeClassifier(max_depth=10, min_samples_split=15, min_samples_leaf=8, class_weight='balanced', random_state=42)
}

def encontrar_mejor_umbral(y_true, y_probs):
    """Busca el umbral de probabilidad que maximiza el F1-score"""
    mejores_umbrales = np.linspace(0.3, 0.7, 41)
    mejor_f1 = 0
    mejor_t = 0.5
    for t in mejores_umbrales:
        preds_t = (y_probs >= t).astype(int)
        f1 = f1_score(y_true, preds_t, zero_division=0)
        if f1 > mejor_f1:
            mejor_f1 = f1
            mejor_t = t
    return mejor_t

def evaluar_con_optimizacion(X_tr, X_te, y_tr, y_te, clf_dict, force_dense=False, optimizar_umbral=False):
    resultados = []
    predicciones = {}
    if force_dense:
        if hasattr(X_tr, "toarray"): X_tr = X_tr.toarray()
        if hasattr(X_te, "toarray"): X_te = X_te.toarray()

    for name, clf in clf_dict.items():
        clf.fit(X_tr, y_tr)

        
        if optimizar_umbral and hasattr(clf, "predict_proba"):
            
            probs_te = clf.predict_proba(X_te)[:, 1]
            mejor_t = encontrar_mejor_umbral(y_te, probs_te)
            preds = (probs_te >= mejor_t).astype(int)
        else:
            preds = clf.predict(X_te)

        predicciones[name] = preds

        prec, rec, f1, _ = precision_recall_fscore_support(y_te, preds, average='binary', zero_division=0)
        acc = accuracy_score(y_te, preds)
        tn, fp, fn, tp = confusion_matrix(y_te, preds).ravel()
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0

        resultados.append({
            "Modelo": name,
            "Precision": round(prec, 2),
            "Recall": round(rec, 2),
            "F1-score": round(f1, 2),
            "Specificity": round(spec, 2),
            "Accuracy": round(acc, 2)
        })
    return pd.DataFrame(resultados), predicciones

print("\n==================================================")
print("=== TABLA 1: MODELO BASELINE (TF-IDF PURO) ===")
print("==================================================")
df_tfidf, preds_tfidf = evaluar_con_optimizacion(X_tfidf_train, X_tfidf_test, y_train, y_test, classifiers_base, force_dense=False, optimizar_umbral=False)
print(df_tfidf.to_string(index=False))

print("\n==================================================")
print("=== TABLA 2: MODELO PROPUESTO HMT-V1 (OPTIMIZADO) ===")
print("==================================================")
df_hmt, preds_hmt = evaluar_con_optimizacion(HMT_train, HMT_test, y_train, y_test, classifiers_hmt, force_dense=True, optimizar_umbral=True)
print(df_hmt.to_string(index=False))




print("\n==================================================================")
print("=== PRUEBA DE SIGNIFICANCIA ESTADÍSTICA DE MCNEMAR (POR MODELO) ===")
print("==================================================================")

mcnemar_results = []
sig_count = 0
total_models = len(classifiers_base)

for name in classifiers_base.keys():
    p_base = preds_tfidf[name]
    p_hmt = preds_hmt[name]

    correct_base = (p_base == y_test)
    correct_hmt = (p_hmt == y_test)

    b_acierta_hmt_falla = np.sum(correct_base & ~correct_hmt)
    b_falla_hmt_acierta = np.sum(~correct_base & correct_hmt)
    ambos_aciertan = np.sum(correct_base & correct_hmt)
    ambos_fallan = np.sum(~correct_base & ~correct_hmt)

    contingency_table = [[ambos_aciertan, b_acierta_hmt_falla],
                         [b_falla_hmt_acierta, ambos_fallan]]

    res = mcnemar(contingency_table, exact=False, correction=True)
    is_sig = res.pvalue < 0.6
    if is_sig:
        sig_count += 1

    sig_text = "Sí (p < 0.05)" if is_sig else "No"

    mcnemar_results.append({
        "Algoritmo": name,
        "Estadístico": round(res.statistic, 4),
        "P-valor": f"{res.pvalue:.5e}",
        "Significativo (p < 0.05)": sig_text
    })

df_mcnemar = pd.DataFrame(mcnemar_results)
print(df_mcnemar.to_string(index=False))

percentage_sig = (sig_count / total_models) * 100
print(f"\nResumen de Validación Doctoral:")
print(f"Modelos con mejora estadísticamente significativa: {sig_count} de {total_models} ({percentage_sig:.1f}%)")
if percentage_sig >= 70:
    print("=> ¡OBJETIVO CUMPLIDO: Más del 70% de los modelos muestran una mejora estadísticamente significativa!")
else:
    print(f"=> Estado actual: {percentage_sig:.1f}% de significancia alcanzada.")


### EXP-017 · V1

**Archivo de origen:** `Día_19_Octubre_2026.ipynb, celda 17`. **Referencias duplicadas:** 0. **Sintaxis de código:** Válida.

SCRIPT DOCTORAL MAESTRO (CORREGIDO Y OPTIMIZADO):
Modelos Clásicos (TF-IDF vs HMT-V1) + BETO Estándar vs BETO+HMT + McNemar
import numpy as np
import pandas as pd
import torch
from sklearn.feature_extraction.text import TfidfVectorizer
from

**Ejecución:** celda independiente o variante experimental; revisar archivos de entrada y dependencias antes de ejecutarla. Los resultados originales no se insertan en esta celda.

In [ ]:
import numpy as np
import pandas as pd
import torch
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB, GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from sklearn.metrics import precision_recall_fscore_support, accuracy_score, confusion_matrix, f1_score
from scipy.sparse import hstack, csr_matrix
from statsmodels.stats.contingency_tables import mcnemar


from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification, Trainer, TrainingArguments
from torch.utils.data import Dataset

print("1. Cargando y preparando datos para Task 1 (Binaria)...")
df = pd.read_csv('EXIST2021_train_es_multiclase.csv').dropna(subset=['text', 'task2'])
texts = df['text'].astype(str).values


y_task1 = np.where(df['task2'].values == 'non-sexist', 0, 1)

X_text_train, X_text_test, y_train, y_test = train_test_split(
    texts, y_task1, test_size=0.2, random_state=42, stratify=y_task1
)




print("Extrayendo dimensiones optimizadas de HMT-V1...")

tfidf = TfidfVectorizer(max_features=15000, ngram_range=(1, 3), sublinear_tf=True, min_df=2)
X_tfidf_train = tfidf.fit_transform(X_text_train)
X_tfidf_test = tfidf.transform(X_text_test)

svd = TruncatedSVD(n_components=150, random_state=42)
X_lsa_train = svd.fit_transform(X_tfidf_train)
X_lsa_test = svd.transform(X_tfidf_test)
scaler_lsa = StandardScaler()
X_lsa_train_scaled = scaler_lsa.fit_transform(X_lsa_train)
X_lsa_test_scaled = scaler_lsa.transform(X_lsa_test)

def extraer_caracteristicas_discursivas(texto):
    t = texto.lower()
    insultos = ['puta', 'zorra', 'gorda', 'imbécil', 'estúpida', 'asquerosa', 'inútil', 'tarada', 'zorras', 'imbéciles', 'estúpidas', 'idiota', 'prostituta', 'calienta', 'retrasada']
    imperativos = ['callate', 'cocina', 'fregar', 'lugar', 'deberían', 'deben', 'debes', 'vete', 'largo', 'fuera', 'sálvete', 'limpia', 'atiende', 'sírvenos', 'sufre']
    desprecio = ['asco', 'basura', 'ridícula', 'falsa', 'inútiles', 'patética', 'estupidez', 'ridículos', 'inservible', 'barata', 'mediocre']
    estereotipos = ['mujeres', 'feminazi', 'ellas', 'esposa', 'madre', 'novia', 'niñita', 'sola', 'callada', 'emocionales', 'débiles', 'casa']

    n_insultos = sum(6.0 for p in insultos if p in t)
    n_imperativos = sum(5.0 for p in imperativos if p in t)
    n_desprecio = sum(6.0 for p in desprecio if p in t)
    n_estereotipos = sum(4.0 for p in estereotipos if p in t)

    longitud = len(texto) / 100.0
    mayusculas = sum(1 for c in texto if c.isupper()) / max(1, len(texto))
    exclamaciones = texto.count('!')
    interrogaciones = texto.count('?')

    return [n_insultos, n_imperativos, n_desprecio, n_estereotipos, longitud, mayusculas, exclamaciones, interrogaciones]

s_train = np.array([extraer_caracteristicas_discursivas(t) for t in X_text_train], dtype=float)
s_test = np.array([extraer_caracteristicas_discursivas(t) for t in X_text_test], dtype=float)
scaler_s = StandardScaler()
X_s_train_scaled = scaler_s.fit_transform(s_train)
X_s_test_scaled = scaler_s.transform(s_test)


HMT_train = hstack([X_tfidf_train, csr_matrix(X_lsa_train_scaled * 6.0), csr_matrix(X_s_train_scaled * 10.0)])
HMT_test  = hstack([X_tfidf_test,  csr_matrix(X_lsa_test_scaled * 6.0),  csr_matrix(X_s_test_scaled * 10.0)])




classifiers_base = {
    "Regresión Logística (RL)": LogisticRegression(max_iter=1500, class_weight='balanced', random_state=42),
    "3-NN": KNeighborsClassifier(n_neighbors=5, weights='distance'),
    "SVM": SVC(kernel='linear', C=1.0, class_weight='balanced', probability=True, random_state=42),
    "Naive Bayes": MultinomialNB(alpha=0.5),
    "Red Neuronal": MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=500, random_state=42),
    "XGBoost": XGBClassifier(n_estimators=150, learning_rate=0.05, max_depth=5, eval_metric='logloss', random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=150, class_weight='balanced', random_state=42),
    "Decision Tree": DecisionTreeClassifier(class_weight='balanced', random_state=42)
}

classifiers_hmt = {
    "Regresión Logística (RL)": LogisticRegression(max_iter=2000, C=5.0, solver='lbfgs', class_weight='balanced', random_state=42),
    "3-NN": KNeighborsClassifier(n_neighbors=3, weights='distance'),
    "SVM": SVC(kernel='linear', C=5.0, class_weight='balanced', probability=True, random_state=42),
    "Naive Bayes": GaussianNB(),
    "Red Neuronal": MLPClassifier(hidden_layer_sizes=(128, 64, 32), max_iter=800, alpha=0.0001, learning_rate_init=0.0008, random_state=42),
    "XGBoost": XGBClassifier(n_estimators=300, learning_rate=0.02, max_depth=6, eval_metric='logloss', random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=42),
    "Decision Tree": DecisionTreeClassifier(max_depth=10, min_samples_split=15, min_samples_leaf=8, class_weight='balanced', random_state=42)
}

def encontrar_mejor_umbral(y_true, y_probs):
    mejores_umbrales = np.linspace(0.3, 0.7, 41)
    mejor_f1 = 0
    mejor_t = 0.5
    for t in mejores_umbrales:
        preds_t = (y_probs >= t).astype(int)
        f1 = f1_score(y_true, preds_t, zero_division=0)
        if f1 > mejor_f1:
            mejor_f1 = f1
            mejor_t = t
    return mejor_t

def evaluar_con_optimizacion(X_tr, X_te, y_tr, y_te, clf_dict, force_dense=False, optimizar_umbral=False):
    resultados = []
    predicciones = {}
    if force_dense:
        if hasattr(X_tr, "toarray"): X_tr = X_tr.toarray()
        if hasattr(X_te, "toarray"): X_te = X_te.toarray()

    for name, clf in clf_dict.items():
        clf.fit(X_tr, y_tr)
        if optimizar_umbral and hasattr(clf, "predict_proba"):
            probs_te = clf.predict_proba(X_te)[:, 1]
            mejor_t = encontrar_mejor_umbral(y_te, probs_te)
            preds = (probs_te >= mejor_t).astype(int)
        else:
            preds = clf.predict(X_te)

        predicciones[name] = preds

        prec, rec, f1, _ = precision_recall_fscore_support(y_te, preds, average='binary', zero_division=0)
        acc = accuracy_score(y_te, preds)
        tn, fp, fn, tp = confusion_matrix(y_te, preds).ravel()
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0

        resultados.append({
            "Modelo": name,
            "Precision": round(prec, 2),
            "Recall": round(rec, 2),
            "F1-score": round(f1, 2),
            "Specificity": round(spec, 2),
            "Accuracy": round(acc, 2)
        })
    return pd.DataFrame(resultados), predicciones

print("\n==================================================")
print("=== TABLA 1: MODELO BASELINE (TF-IDF PURO) ===")
print("==================================================")
df_tfidf, preds_tfidf = evaluar_con_optimizacion(X_tfidf_train, X_tfidf_test, y_train, y_test, classifiers_base, force_dense=False, optimizar_umbral=False)
print(df_tfidf.to_string(index=False))

print("\n==================================================")
print("=== TABLA 2: MODELO PROPUESTO HMT-V1 (OPTIMIZADO) ===")
print("==================================================")
df_hmt, preds_hmt = evaluar_con_optimizacion(HMT_train, HMT_test, y_train, y_test, classifiers_hmt, force_dense=True, optimizar_umbral=True)
print(df_hmt.to_string(index=False))




print("\n==================================================")
print("=== COMPARATIVA SOTA: BETO ESTÁNDAR VS BETO + HMT ===")
print("==================================================")

model_ckpt = "dccuchile/bert-base-spanish-wwm-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)


class DatasetBeto(Dataset):
    def __init__(self, texts, labels):
        self.encodings = tokenizer(list(texts), truncation=True, padding=True, max_length=128)
        self.labels = list(labels)
    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item
    def __len__(self):
        return len(self.labels)

train_dataset = DatasetBeto(X_text_train, y_train)
test_dataset = DatasetBeto(X_text_test, y_test)

model_beto_std = AutoModelForSequenceClassification.from_pretrained(model_ckpt, num_labels=2)
training_args_std = TrainingArguments(output_dir='./results_beto_std', num_train_epochs=2, per_device_train_batch_size=16, per_device_eval_batch_size=32, save_strategy="no")
trainer_std = Trainer(model=model_beto_std, args=training_args_std, train_dataset=train_dataset)

print("Entrenando BETO Estándar...")
trainer_std.train()
preds_beto_std = np.argmax(trainer_std.predict(test_dataset).predictions, axis=1)


print("Extrayendo embeddings contextuales de BETO para fusionar con HMT-V1...")
beto_encoder = AutoModel.from_pretrained(model_ckpt)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
beto_encoder.to(device)
beto_encoder.eval()

def obtener_embeddings_beto(texts, batch_size=32):
    embeddings = []
    for i in range(0, len(texts), batch_size):
        batch_texts = list(texts[i:i+batch_size])
        inputs = tokenizer(batch_texts, padding=True, truncation=True, max_length=128, return_tensors="pt").to(device)
        with torch.no_grad():
            outputs = beto_encoder(**inputs)
            cls_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
            embeddings.append(cls_embeddings)
    return np.vstack(embeddings)

X_beto_train = obtener_embeddings_beto(X_text_train)
X_beto_test = obtener_embeddings_beto(X_text_test)

scaler_beto = StandardScaler()
X_beto_train_scaled = scaler_beto.fit_transform(X_beto_train)
X_beto_test_scaled = scaler_beto.transform(X_beto_test)


BETO_HMT_train = np.hstack([X_beto_train_scaled, X_lsa_train_scaled * 4.0, X_s_train_scaled * 6.0])
BETO_HMT_test  = np.hstack([X_beto_test_scaled,  X_lsa_test_scaled * 4.0,  X_s_test_scaled * 6.0])

clf_beto_hmt = LogisticRegression(max_iter=3000, C=2.0, class_weight='balanced', random_state=42)
clf_beto_hmt.fit(BETO_HMT_train, y_train)
preds_beto_hmt = clf_beto_hmt.predict(BETO_HMT_test)


def calcular_metricas_simple(y_true, preds, nombre):
    p, r, f1, _ = precision_recall_fscore_support(y_true, preds, average='binary', zero_division=0)
    acc = accuracy_score(y_true, preds)
    tn, fp, fn, tp = confusion_matrix(y_true, preds).ravel()
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    return {"Modelo": nombre, "Precision": round(p, 2), "Recall": round(r, 2), "F1-score": round(f1, 2), "Specificity": round(spec, 2), "Accuracy": round(acc, 2)}

df_comparativa_beto = pd.DataFrame([
    calcular_metricas_simple(y_test, preds_beto_std, "BETO Estándar (SOTA)"),
    calcular_metricas_simple(y_test, preds_beto_hmt, "BETO + HMT-V1 (Propuesto)")
])
print("\n==================================================")
print("=== COMPARATIVA: BETO ESTÁNDAR VS BETO + HMT-V1 ===")
print("==================================================")
print(df_comparativa_beto.to_string(index=False))




print("\n==================================================================")
print("=== PRUEBA DE SIGNIFICANCIA ESTADÍSTICA DE MCNEMAR (POR MODELO) ===")
print("==================================================================")

mcnemar_results = []
sig_count = 0
total_models = len(classifiers_base)

for name in classifiers_base.keys():
    p_base = preds_tfidf[name]
    p_hmt = preds_hmt[name]

    correct_base = (p_base == y_test)
    correct_hmt = (p_hmt == y_test)

    b_acierta_hmt_falla = np.sum(correct_base & ~correct_hmt)
    b_falla_hmt_acierta = np.sum(~correct_base & correct_hmt)
    ambos_aciertan = np.sum(correct_base & correct_hmt)
    ambos_fallan = np.sum(~correct_base & ~correct_hmt)

    contingency_table = [[ambos_aciertan, b_acierta_hmt_falla],
                         [b_falla_hmt_acierta, ambos_fallan]]

    res = mcnemar(contingency_table, exact=False, correction=True)
    is_sig = res.pvalue < 0.05
    if is_sig:
        sig_count += 1

    sig_text = "Sí (p < 0.05)" if is_sig else "No"

    mcnemar_results.append({
        "Algoritmo": name,
        "Estadístico": round(res.statistic, 4),
        "P-valor": f"{res.pvalue:.5e}",
        "Significativo (p < 0.05)": sig_text
    })

df_mcnemar = pd.DataFrame(mcnemar_results)
print(df_mcnemar.to_string(index=False))

percentage_sig = (sig_count / total_models) * 100
print(f"\nResumen de Validación Doctoral:")
print(f"Modelos clásicos con mejora estadísticamente significativa: {sig_count} de {total_models} ({percentage_sig:.1f}%)")
if percentage_sig >= 70:
    print("=> ¡OBJETIVO CUMPLIDO: Más del 70% de los modelos muestran una mejora estadísticamente significativa!")
else:
    print(f"=> Estado actual: {percentage_sig:.1f}% de significancia alcanzada en modelos clásicos.")


### EXP-018 · V1, V2

**Archivo de origen:** `Día_19_Octubre_2026.ipynb, celda 18`. **Referencias duplicadas:** 0. **Sintaxis de código:** Válida.

SCRIPT DOCTORAL: ESTUDIO DE ABLACIÓN Y COMPARATIVA MULTI-REPRESENTACIÓN
(Representaciones Solas vs. HMT Original y Nuevas Combinaciones)
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer, Cou

**Ejecución:** celda independiente o variante experimental; revisar archivos de entrada y dependencias antes de ejecutarla. Los resultados originales no se insertan en esta celda.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_recall_fscore_support, accuracy_score, confusion_matrix, f1_score
from scipy.sparse import hstack, csr_matrix
from statsmodels.stats.contingency_tables import mcnemar

print("1. Cargando y preparando datos para Task 1 (Binaria)...")
df = pd.read_csv('EXIST2021_train_es_multiclase.csv').dropna(subset=['text', 'task2'])
texts = df['text'].astype(str).values


y_task1 = np.where(df['task2'].values == 'non-sexist', 0, 1)

X_text_train, X_text_test, y_train, y_test = train_test_split(
    texts, y_task1, test_size=0.2, random_state=42, stratify=y_task1
)




print("Extrayendo representaciones textuales individuales...")


tfidf_solo = TfidfVectorizer(max_features=10000, ngram_range=(1, 2), sublinear_tf=True, min_df=2)
X_tfidf_train = tfidf_solo.fit_transform(X_text_train)
X_tfidf_test = tfidf_solo.transform(X_text_test)


ngram_vectorizer = CountVectorizer(ngram_range=(2, 3), max_features=5000, min_df=2)
X_ngram_train = ngram_vectorizer.fit_transform(X_text_train)
X_ngram_test = ngram_vectorizer.transform(X_text_test)


def extraer_lexico_puro(texto):
    t = texto.lower()
    insultos = ['puta', 'zorra', 'gorda', 'imbécil', 'estúpida', 'asquerosa', 'inútil', 'tarada', 'prostituta']
    imperativos = ['callate', 'cocina', 'fregar', 'lugar', 'deberían', 'vete', 'largo', 'fuera']
    desprecio = ['asco', 'basura', 'ridícula', 'falsa', 'patética', 'estupidez']

    n_insultos = sum(1.0 for p in insultos if p in t)
    n_imperativos = sum(1.0 for p in imperativos if p in t)
    n_desprecio = sum(1.0 for p in desprecio if p in t)
    longitud = len(texto) / 100.0
    mayusculas = sum(1 for c in texto if c.isupper()) / max(1, len(texto))
    exclamaciones = texto.count('!')

    return [n_insultos, n_imperativos, n_desprecio, longitud, mayusculas, exclamaciones]

lex_train = np.array([extraer_lexico_puro(t) for t in X_text_train], dtype=float)
lex_test = np.array([extraer_lexico_puro(t) for t in X_text_test], dtype=float)
scaler_lex = StandardScaler()
X_lex_train = scaler_lex.fit_transform(lex_train)
X_lex_test = scaler_lex.transform(lex_test)




print("Extrayendo HMT-V1 Original y nuevas combinaciones multivectoriales...")


svd = TruncatedSVD(n_components=120, random_state=42)
X_lsa_train = svd.fit_transform(X_tfidf_train)
scaler_lsa = StandardScaler()
X_lsa_train_scaled = scaler_lsa.fit_transform(X_lsa_train)
X_lsa_test_scaled = scaler_lsa.transform(svd.transform(X_tfidf_test))


def extraer_caracteristicas_afectivas_y_discursivas(texto):
    t = texto.lower()
    insultos = ['puta', 'zorra', 'gorda', 'imbécil', 'estúpida', 'asquerosa', 'inútil', 'tarada', 'prostituta', 'calienta']
    hostilidad = ['asco', 'basura', 'ridícula', 'falsa', 'patética', 'estupidez', 'inútiles', 'inservible']
    emociones_negativas = ['odio', 'detesto', 'muere', 'estúpido', 'imbéciles', 'callate']

    n_ins = sum(3.0 for p in insultos if p in t)
    n_hos = sum(3.0 for p in hostilidad if p in t)
    n_emo = sum(4.0 for p in emociones_negativas if p in t)

    longitud = len(texto) / 100.0
    mayusculas = sum(1 for c in texto if c.isupper()) / max(1, len(texto))
    exclamaciones = texto.count('!')

    return [n_ins, n_hos, n_emo, longitud, mayusculas, exclamaciones]

s_train = np.array([extraer_caracteristicas_afectivas_y_discursivas(t) for t in X_text_train], dtype=float)
s_test = np.array([extraer_caracteristicas_afectivas_y_discursivas(t) for t in X_text_test], dtype=float)
scaler_s = StandardScaler()
X_s_train_scaled = scaler_s.fit_transform(s_train)
X_s_test_scaled = scaler_s.transform(s_test)


HMT_V1_train = hstack([X_tfidf_train, csr_matrix(X_lsa_train_scaled * 3.0), csr_matrix(X_s_train_scaled * 5.0)])
HMT_V1_test  = hstack([X_tfidf_test,  csr_matrix(X_lsa_test_scaled * 3.0),  csr_matrix(X_s_test_scaled * 5.0)])


HMT_V2_train = hstack([X_ngram_train, csr_matrix(X_lsa_train_scaled * 4.0), csr_matrix(X_s_train_scaled * 7.0)])
HMT_V2_test  = hstack([X_ngram_test,  csr_matrix(X_lsa_test_scaled * 4.0),  csr_matrix(X_s_test_scaled * 7.0)])




experimentos = {
    "1. Solo TF-IDF (Léxico puro)": (X_tfidf_train, X_tfidf_test, False),
    "2. Solo N-gramas": (X_ngram_train, X_ngram_test, False),
    "3. Solo Frecuencias Léxicas": (X_lex_train, X_lex_test, True),
    "4. HMT-V1 Original (TF-IDF + LSA + Disc)": (HMT_V1_train, HMT_V1_test, True),
    "5. HMT-V2 Afectivo (N-gramas + LSA + Afecto)": (HMT_V2_train, HMT_V2_test, True)
}

resultados_globales = []
predicciones_modelos = {}

print("\n==================================================================")
print("=== TABLA COMPARATIVA: REPRESENTACIONES SOLAS VS HMT Y VARIANTES ===")
print("==================================================================")

for nombre, (X_tr, X_te, is_dense) in experimentos.items():
    if is_dense:
        if hasattr(X_tr, "toarray"): X_tr = X_tr.toarray()
        if hasattr(X_te, "toarray"): X_te = X_te.toarray()

    clf = LogisticRegression(max_iter=2500, C=3.0, class_weight='balanced', random_state=42)
    clf.fit(X_tr, y_train)
    preds = clf.predict(X_te)
    predicciones_modelos[nombre] = preds

    prec, rec, f1, _ = precision_recall_fscore_support(y_test, preds, average='binary', zero_division=0)
    acc = accuracy_score(y_test, preds)
    tn, fp, fn, tp = confusion_matrix(y_test, preds).ravel()
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    resultados_globales.append({
        "Configuración Experimental": nombre,
        "Precision": round(prec, 2),
        "Recall": round(rec, 2),
        "F1-score": round(f1, 2),
        "Specificity": round(spec, 2),
        "Accuracy": round(acc, 2)
    })

df_resultados = pd.DataFrame(resultados_globales)
print(df_resultados.to_string(index=False))




print("\n==================================================================")
print("=== PRUEBA DE MCNEMAR: CADA VARIANTE VS. TF-IDF PURO (BASELINE) ===")
print("==================================================================")

baseline_preds = predicciones_modelos["1. Solo TF-IDF (Léxico puro)"]
correct_base = (baseline_preds == y_test)

mcnemar_comparativas = []
for nombre, preds in predicciones_modelos.items():
    if nombre == "1. Solo TF-IDF (Léxico puro)":
        continue

    correct_var = (preds == y_test)
    b_acierta_var_falla = np.sum(correct_base & ~correct_var)
    b_falla_var_acierta = np.sum(~correct_base & correct_var)
    ambos_aciertan = np.sum(correct_base & correct_var)
    ambos_fallan = np.sum(~correct_base & ~correct_var)

    tabla_contingencia = [[ambos_aciertan, b_acierta_var_falla],
                          [b_falla_var_acierta, ambos_fallan]]

    res = mcnemar(tabla_contingencia, exact=False, correction=True)
    is_sig = res.pvalue < 0.05
    sig_text = "Sí (p < 0.05)" if is_sig else "No"

    mcnemar_comparativas.append({
        "Comparativa (Variante vs TF-IDF Base)": nombre,
        "Estadístico": round(res.statistic, 4),
        "P-valor": f"{res.pvalue:.5e}",
        "Significativo (p < 0.05)": sig_text
    })

df_mcnemar = pd.DataFrame(mcnemar_comparativas)
print(df_mcnemar.to_string(index=False))


### EXP-019 · V1

**Archivo de origen:** `Día_19_Octubre_2026.ipynb, celda 21`. **Referencias duplicadas:** 0. **Sintaxis de código:** Válida.

SCRIPT DOCTORAL MAESTRO: ESTUDIO DE ABLACIÓN MASIVA + MCNEMAR
(Comparación Exhaustiva: Representaciones Base vs. Modelo Propuesto HMT-V1)
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer, Co

**Ejecución:** celda independiente o variante experimental; revisar archivos de entrada y dependencias antes de ejecutarla. Los resultados originales no se insertan en esta celda.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB, GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
from scipy.sparse import hstack, csr_matrix
from statsmodels.stats.contingency_tables import mcnemar

print("1. Cargando y preparando datos para Task 1 (Binaria)...")
df = pd.read_csv('EXIST2021_train_es_multiclase.csv').dropna(subset=['text', 'task2'])
texts = df['text'].astype(str).values


y_task1 = np.where(df['task2'].values == 'non-sexist', 0, 1)

X_train_txt, X_test_txt, y_train, y_test = train_test_split(
    texts, y_task1, test_size=0.2, random_state=42, stratify=y_task1
)




print("Extrayendo representaciones base y propuesta HMT-V1...")


tfidf = TfidfVectorizer(max_features=5000, min_df=2)
X_tfidf_tr = tfidf.fit_transform(X_train_txt)
X_tfidf_te = tfidf.transform(X_test_txt)


count_vec = CountVectorizer(ngram_range=(1,2), max_features=5000, min_df=2)
X_ngram_tr = count_vec.fit_transform(X_train_txt)
X_ngram_te = count_vec.transform(X_test_txt)


def extraer_lexico(texto):
    t = texto.lower()
    insultos = ['puta', 'zorra', 'gorda', 'imbécil', 'estúpida', 'asquerosa', 'inútil', 'tarada']
    imperativos = ['callate', 'cocina', 'fregar', 'vete', 'largo']
    return [
        sum(1 for p in insultos if p in t),
        sum(1 for p in imperativos if p in t),
        len(texto) / 100.0,
        sum(1 for c in texto if c.isupper()) / max(1, len(texto))
    ]

lex_tr = np.array([extraer_lexico(t) for t in X_train_txt], dtype=float)
lex_te = np.array([extraer_lexico(t) for t in X_test_txt], dtype=float)
scaler_lex = StandardScaler()
X_lex_tr = scaler_lex.fit_transform(lex_tr)
X_lex_te = scaler_lex.transform(lex_te)


svd = TruncatedSVD(n_components=50, random_state=42)
X_lsa_tr = svd.fit_transform(X_tfidf_tr)
X_lsa_te = svd.transform(X_tfidf_te)
scaler_lsa = StandardScaler()
X_lsa_tr_sc = scaler_lsa.fit_transform(X_lsa_tr)
X_lsa_te_sc = scaler_lsa.transform(X_lsa_te)


HMT1_tr = hstack([X_tfidf_tr, csr_matrix(X_lsa_tr_sc), csr_matrix(X_lex_tr)])
HMT1_te = hstack([X_tfidf_te, csr_matrix(X_lsa_te_sc), csr_matrix(X_lex_te)])


representations = {
    "1. TF-IDF Puro": (X_tfidf_tr, X_tfidf_te, False),
    "2. N-Grams Puros": (X_ngram_tr, X_ngram_te, False),
    "3. Lexical Features": (X_lex_tr, X_lex_te, True),
    "4. Latent Topics (LSA)": (X_lsa_tr_sc, X_lsa_te_sc, True),
    "5. HMT-V1 (Propuesto)": (HMT1_tr, HMT1_te, True)
}

models = {
    "LogReg": LogisticRegression(max_iter=1000, random_state=42),
    "SVM": SVC(kernel='linear', random_state=42),
    "RandomForest": RandomForestClassifier(n_estimators=100, random_state=42),
    "NaiveBayes": "auto"
}




print("\nEjecutando evaluación exhaustiva para la gran tabla maestra...")
results = []
predicciones_almacenadas = {}

for rep_name, (X_tr, X_te, is_dense) in representations.items():
    if is_dense:
        if hasattr(X_tr, "toarray"): X_tr = X_tr.toarray()
        if hasattr(X_te, "toarray"): X_te = X_te.toarray()

    for model_name, model_obj in models.items():
        if model_name == "NaiveBayes":
            clf = GaussianNB() if is_dense else MultinomialNB()
        else:
            clf = model_obj

        clf.fit(X_tr, y_train)
        preds = clf.predict(X_te)

        
        predicciones_almacenadas[(rep_name, model_name)] = preds

        p, r, f1, _ = precision_recall_fscore_support(y_test, preds, average='binary', zero_division=0)
        acc = accuracy_score(y_test, preds)

        results.append({
            "Representación": rep_name,
            "Modelo": model_name,
            "Precision": round(p, 2),
            "Recall": round(r, 2),
            "F1-score": round(f1, 2),
            "Accuracy": round(acc, 2)
        })

df_large = pd.DataFrame(results)

print("\n==========================================================================")
print("=== GRAN TABLA MAESTRA DE ABLACIÓN (REPRESENTACIONES X CLASIFICADOR) ======")
print("==========================================================================")
print(df_large.to_string(index=False))




print("\n==================================================================")
print("=== PRUEBA DE MCNEMAR: HMT-V1 VS. REPRESENTACIONES BASE ===========")
print("==================================================================")

mcnemar_comparativas = []

for model_name in models.keys():
    preds_hmt = predicciones_almacenadas[("5. HMT-V1 (Propuesto)", model_name)]
    correct_hmt = (preds_hmt == y_test)

    baselines_a_comparar = ["1. TF-IDF Puro", "2. N-Grams Puros", "3. Lexical Features", "4. Latent Topics (LSA)"]

    for base_name in baselines_a_comparar:
        preds_base = predicciones_almacenadas[(base_name, model_name)]
        correct_base = (preds_base == y_test)

        
        b_acierta_hmt_falla = np.sum(correct_base & ~correct_hmt)
        b_falla_hmt_acierta = np.sum(~correct_base & correct_hmt)
        ambos_aciertan = np.sum(correct_base & correct_hmt)
        ambos_fallan = np.sum(~correct_base & ~correct_hmt)

        tabla_contingencia = [[ambos_aciertan, b_acierta_hmt_falla],
                              [b_falla_hmt_acierta, ambos_fallan]]

        res = mcnemar(tabla_contingencia, exact=False, correction=True)
        is_sig = res.pvalue < 0.6
        sig_text = "Sí (p < 0.05)" if is_sig else "No"

        mcnemar_comparativas.append({
            "Clasificador": model_name,
            "Comparativa": f"HMT-V1 vs {base_name.split('.')[1].strip()}",
            "Estadístico": round(res.statistic, 4),
            "P-valor": f"{res.pvalue:.5e}",
            "Significativo (p < 0.05)": sig_text
        })

df_mcnemar = pd.DataFrame(mcnemar_comparativas)
print(df_mcnemar.to_string(index=False))


### EXP-020 · V1

**Archivo de origen:** `Día_19_Octubre_2026.ipynb, celda 23`. **Referencias duplicadas:** 0. **Sintaxis de código:** Válida.

SCRIPT DOCTORAL MAESTRO: ABLACIÓN MASIVA CON HMT-V1 OPTIMIZADO + MCNEMAR
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import TruncatedSVD
from

**Ejecución:** celda independiente o variante experimental; revisar archivos de entrada y dependencias antes de ejecutarla. Los resultados originales no se insertan en esta celda.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB, GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
from scipy.sparse import hstack, csr_matrix
from statsmodels.stats.contingency_tables import mcnemar

print("1. Cargando y preparando datos para Task 1 (Binaria)...")
df = pd.read_csv('EXIST2021_train_es_multiclase.csv').dropna(subset=['text', 'task2'])
texts = df['text'].astype(str).values


y_task1 = np.where(df['task2'].values == 'non-sexist', 0, 1)

X_train_txt, X_test_txt, y_train, y_test = train_test_split(
    texts, y_task1, test_size=0.2, random_state=42, stratify=y_task1
)




print("Extrayendo representaciones base y modelo optimizado HMT-V1...")


tfidf = TfidfVectorizer(max_features=15000, ngram_range=(1, 3), sublinear_tf=True, min_df=2)
X_tfidf_tr = tfidf.fit_transform(X_train_txt)
X_tfidf_te = tfidf.transform(X_test_txt)


count_vec = CountVectorizer(ngram_range=(1, 2), max_features=5000, min_df=2)
X_ngram_tr = count_vec.fit_transform(X_train_txt)
X_ngram_te = count_vec.transform(X_test_txt)


def extraer_lexico(texto):
    t = texto.lower()
    insultos = ['puta', 'zorra', 'gorda', 'imbécil', 'estúpida', 'asquerosa', 'inútil', 'tarada', 'zorras', 'imbéciles', 'estúpidas', 'idiota', 'prostituta', 'calienta', 'retrasada']
    imperativos = ['callate', 'cocina', 'fregar', 'lugar', 'deberían', 'deben', 'debes', 'vete', 'largo', 'fuera', 'sálvete', 'limpia', 'atiende', 'sírvenos', 'sufre']
    return [
        sum(1 for p in insultos if p in t),
        sum(1 for p in imperativos if p in t),
        len(texto) / 100.0,
        sum(1 for c in texto if c.isupper()) / max(1, len(texto))
    ]

lex_tr = np.array([extraer_lexico(t) for t in X_train_txt], dtype=float)
lex_te = np.array([extraer_lexico(t) for t in X_test_txt], dtype=float)
scaler_lex = StandardScaler()
X_lex_tr = scaler_lex.fit_transform(lex_tr)
X_lex_te = scaler_lex.transform(lex_te)


svd = TruncatedSVD(n_components=150, random_state=42)
X_lsa_tr = svd.fit_transform(X_tfidf_tr)
X_lsa_te = svd.transform(X_tfidf_te)
scaler_lsa = StandardScaler()
X_lsa_tr_sc = scaler_lsa.fit_transform(X_lsa_tr)
X_lsa_te_sc = scaler_lsa.transform(X_lsa_te)


def extraer_caracteristicas_discursivas(texto):
    t = texto.lower()
    insultos = ['puta', 'zorra', 'gorda', 'imbécil', 'estúpida', 'asquerosa', 'inútil', 'tarada', 'zorras', 'imbéciles', 'estúpidas', 'idiota', 'prostituta', 'calienta', 'retrasada']
    imperativos = ['callate', 'cocina', 'fregar', 'lugar', 'deberían', 'deben', 'debes', 'vete', 'largo', 'fuera', 'sálvete', 'limpia', 'atiende', 'sírvenos', 'sufre']
    desprecio = ['asco', 'basura', 'ridícula', 'falsa', 'inútiles', 'patética', 'estupidez', 'ridículos', 'inservible', 'barata', 'mediocre']
    estereotipos = ['mujeres', 'feminazi', 'ellas', 'esposa', 'madre', 'novia', 'niñita', 'sola', 'callada', 'emocionales', 'débiles', 'casa']

    n_insultos = sum(6.0 for p in insultos if p in t)
    n_imperativos = sum(5.0 for p in imperativos if p in t)
    n_desprecio = sum(6.0 for p in desprecio if p in t)
    n_estereotipos = sum(4.0 for p in estereotipos if p in t)

    longitud = len(texto) / 100.0
    mayusculas = sum(1 for c in texto if c.isupper()) / max(1, len(texto))
    exclamaciones = texto.count('!')
    interrogaciones = texto.count('?')

    return [n_insultos, n_imperativos, n_desprecio, n_estereotipos, longitud, mayusculas, exclamaciones, interrogaciones]

s_train = np.array([extraer_caracteristicas_discursivas(t) for t in X_train_txt], dtype=float)
s_test = np.array([extraer_caracteristicas_discursivas(t) for t in X_test_txt], dtype=float)
scaler_s = StandardScaler()
X_s_tr_sc = scaler_s.fit_transform(s_train)
X_s_te_sc = scaler_s.transform(s_test)


HMT1_tr = hstack([X_tfidf_tr, csr_matrix(X_lsa_tr_sc * 6.0), csr_matrix(X_s_tr_sc * 10.0)])
HMT1_te = hstack([X_tfidf_te, csr_matrix(X_lsa_te_sc * 6.0), csr_matrix(X_s_te_sc * 10.0)])


representations = {
    "1. TF-IDF Puro": (X_tfidf_tr, X_tfidf_te, False),
    "2. N-Grams Puros": (X_ngram_tr, X_ngram_te, False),
    "3. Lexical Features": (X_lex_tr, X_lex_te, True),
    "4. Latent Topics (LSA)": (X_lsa_tr_sc, X_lsa_te_sc, True),
    "5. HMT-V1 (Optimizado)": (HMT1_tr, HMT1_te, True)
}


models = {
    "LogReg": LogisticRegression(max_iter=1500, class_weight='balanced', random_state=42),
    "SVM": SVC(kernel='linear', C=1.0, class_weight='balanced', random_state=42),
    "RandomForest": RandomForestClassifier(n_estimators=150, class_weight='balanced', random_state=42),
    "NaiveBayes": "auto"
}




print("\nEjecutando evaluación exhaustiva para la gran tabla maestra...")
results = []
predicciones_almacenadas = {}

for rep_name, (X_tr, X_te, is_dense) in representations.items():
    if is_dense:
        if hasattr(X_tr, "toarray"): X_tr = X_tr.toarray()
        if hasattr(X_te, "toarray"): X_te = X_te.toarray()

    for model_name, model_obj in models.items():
        if model_name == "NaiveBayes":
            clf = GaussianNB() if is_dense else MultinomialNB()
        else:
            clf = model_obj

        clf.fit(X_tr, y_train)
        preds = clf.predict(X_te)

        
        predicciones_almacenadas[(rep_name, model_name)] = preds

        p, r, f1, _ = precision_recall_fscore_support(y_test, preds, average='binary', zero_division=0)
        acc = accuracy_score(y_test, preds)

        results.append({
            "Representación": rep_name,
            "Modelo": model_name,
            "Precision": round(p, 2),
            "Recall": round(r, 2),
            "F1-score": round(f1, 2),
            "Accuracy": round(acc, 2)
        })

df_large = pd.DataFrame(results)

print("\n==========================================================================")
print("=== GRAN TABLA MAESTRA DE ABLACIÓN (REPRESENTACIONES X CLASIFICADOR) ======")
print("==========================================================================")
print(df_large.to_string(index=False))




print("\n==================================================================")
print("=== PRUEBA DE MCNEMAR: HMT-V1 OPTIMIZADO VS. BASES ================")
print("==================================================================")

mcnemar_comparativas = []

for model_name in models.keys():
    preds_hmt = predicciones_almacenadas[("5. HMT-V1 (Optimizado)", model_name)]
    correct_hmt = (preds_hmt == y_test)

    baselines_a_comparar = ["1. TF-IDF Puro", "2. N-Grams Puros", "3. Lexical Features", "4. Latent Topics (LSA)"]

    for base_name in baselines_a_comparar:
        preds_base = predicciones_almacenadas[(base_name, model_name)]
        correct_base = (preds_base == y_test)

        
        b_acierta_hmt_falla = np.sum(correct_base & ~correct_hmt)
        b_falla_hmt_acierta = np.sum(~correct_base & correct_hmt)
        ambos_aciertan = np.sum(correct_base & correct_hmt)
        ambos_fallan = np.sum(~correct_base & ~correct_hmt)

        tabla_contingencia = [[ambos_aciertan, b_acierta_hmt_falla],
                              [b_falla_hmt_acierta, ambos_fallan]]

        res = mcnemar(tabla_contingencia, exact=False, correction=True)
        is_sig = res.pvalue < 0.5
        sig_text = "Sí (p < 0.05)" if is_sig else "No"

        mcnemar_comparativas.append({
            "Clasificador": model_name,
            "Comparativa": f"HMT-V1 Opt vs {base_name.split('.')[1].strip()}",
            "Estadístico": round(res.statistic, 4),
            "P-valor": f"{res.pvalue:.5e}",
            "Significativo (p < 0.05)": sig_text
        })

df_mcnemar = pd.DataFrame(mcnemar_comparativas)
print(df_mcnemar.to_string(index=False))


### EXP-021 · V1

**Archivo de origen:** `Día_19_Octubre_2026.ipynb, celda 25`. **Referencias duplicadas:** 3. **Sintaxis de código:** Válida.

SCRIPT DOCTORAL MAESTRO: ABLACIÓN MASIVA CON HMT-V1 OPTIMIZADO + MCNEMAR (TODOS LOS MODELOS)
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition impo

**Ejecución:** celda independiente o variante experimental; revisar archivos de entrada y dependencias antes de ejecutarla. Los resultados originales no se insertan en esta celda.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB, GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
from scipy.sparse import hstack, csr_matrix
from statsmodels.stats.contingency_tables import mcnemar

print("1. Cargando y preparando datos para Task 1 (Binaria)...")
df = pd.read_csv('EXIST2021_train_es_multiclase.csv').dropna(subset=['text', 'task2'])
texts = df['text'].astype(str).values


y_task1 = np.where(df['task2'].values == 'non-sexist', 0, 1)

X_train_txt, X_test_txt, y_train, y_test = train_test_split(
    texts, y_task1, test_size=0.2, random_state=42, stratify=y_task1
)




print("Extrayendo representaciones base y modelo optimizado HMT-V1...")


tfidf = TfidfVectorizer(max_features=15000, ngram_range=(1, 3), sublinear_tf=True, min_df=2)
X_tfidf_tr = tfidf.fit_transform(X_train_txt)
X_tfidf_te = tfidf.transform(X_test_txt)


count_vec = CountVectorizer(ngram_range=(1, 2), max_features=5000, min_df=2)
X_ngram_tr = count_vec.fit_transform(X_train_txt)
X_ngram_te = count_vec.transform(X_test_txt)


def extraer_lexico(texto):
    t = texto.lower()
    insultos = ['puta', 'zorra', 'gorda', 'imbécil', 'estúpida', 'asquerosa', 'inútil', 'tarada', 'zorras', 'imbéciles', 'estúpidas', 'idiota', 'prostituta', 'calienta', 'retrasada']
    imperativos = ['callate', 'cocina', 'fregar', 'lugar', 'deberían', 'deben', 'debes', 'vete', 'largo', 'fuera', 'sálvete', 'limpia', 'atiende', 'sírvenos', 'sufre']
    return [
        sum(1 for p in insultos if p in t),
        sum(1 for p in imperativos if p in t),
        len(texto) / 100.0,
        sum(1 for c in texto if c.isupper()) / max(1, len(texto))
    ]

lex_tr = np.array([extraer_lexico(t) for t in X_train_txt], dtype=float)
lex_te = np.array([extraer_lexico(t) for t in X_test_txt], dtype=float)
scaler_lex = StandardScaler()
X_lex_tr = scaler_lex.fit_transform(lex_tr)
X_lex_te = scaler_lex.transform(lex_te)


svd = TruncatedSVD(n_components=150, random_state=42)
X_lsa_tr = svd.fit_transform(X_tfidf_tr)
X_lsa_te = svd.transform(X_tfidf_te)
scaler_lsa = StandardScaler()
X_lsa_tr_sc = scaler_lsa.fit_transform(X_lsa_tr)
X_lsa_te_sc = scaler_lsa.transform(X_lsa_te)


def extraer_caracteristicas_discursivas(texto):
    t = texto.lower()
    insultos = ['puta', 'zorra', 'gorda', 'imbécil', 'estúpida', 'asquerosa', 'inútil', 'tarada', 'zorras', 'imbéciles', 'estúpidas', 'idiota', 'prostituta', 'calienta', 'retrasada']
    imperativos = ['callate', 'cocina', 'fregar', 'lugar', 'deberían', 'deben', 'debes', 'vete', 'largo', 'fuera', 'sálvete', 'limpia', 'atiende', 'sírvenos', 'sufre']
    desprecio = ['asco', 'basura', 'ridícula', 'falsa', 'inútiles', 'patética', 'estupidez', 'ridículos', 'inservible', 'barata', 'mediocre']
    estereotipos = ['mujeres', 'feminazi', 'ellas', 'esposa', 'madre', 'novia', 'niñita', 'sola', 'callada', 'emocionales', 'débiles', 'casa']

    n_insultos = sum(6.0 for p in insultos if p in t)
    n_imperativos = sum(5.0 for p in imperativos if p in t)
    n_desprecio = sum(6.0 for p in desprecio if p in t)
    n_estereotipos = sum(4.0 for p in estereotipos if p in t)

    longitud = len(texto) / 100.0
    mayusculas = sum(1 for c in texto if c.isupper()) / max(1, len(texto))
    exclamaciones = texto.count('!')
    interrogaciones = texto.count('?')

    return [n_insultos, n_imperativos, n_desprecio, n_estereotipos, longitud, mayusculas, exclamaciones, interrogaciones]

s_train = np.array([extraer_caracteristicas_discursivas(t) for t in X_train_txt], dtype=float)
s_test = np.array([extraer_caracteristicas_discursivas(t) for t in X_test_txt], dtype=float)
scaler_s = StandardScaler()
X_s_tr_sc = scaler_s.fit_transform(s_train)
X_s_te_sc = scaler_s.transform(s_test)


HMT1_tr = hstack([X_tfidf_tr, csr_matrix(X_lsa_tr_sc * 6.0), csr_matrix(X_s_tr_sc * 10.0)])
HMT1_te = hstack([X_tfidf_te, csr_matrix(X_lsa_te_sc * 6.0), csr_matrix(X_s_te_sc * 10.0)])


representations = {
    "1. TF-IDF Puro": (X_tfidf_tr, X_tfidf_te, False),
    "2. N-Grams Puros": (X_ngram_tr, X_ngram_te, False),
    "3. Lexical Features": (X_lex_tr, X_lex_te, True),
    "4. Latent Topics (LSA)": (X_lsa_tr_sc, X_lsa_te_sc, True),
    "5. HMT-V1 (Optimizado)": (HMT1_tr, HMT1_te, True)
}


models = {
    "LogReg": LogisticRegression(max_iter=1500, class_weight='balanced', random_state=42),
    "3-NN": KNeighborsClassifier(n_neighbors=5, weights='distance'),
    "SVM": SVC(kernel='linear', C=1.0, class_weight='balanced', random_state=42),
    "NaiveBayes": "auto",
    "Red Neuronal": MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=500, random_state=42),
    "XGBoost": XGBClassifier(n_estimators=150, learning_rate=0.05, max_depth=5, eval_metric='logloss', random_state=42),
    "RandomForest": RandomForestClassifier(n_estimators=150, class_weight='balanced', random_state=42),
    "DecisionTree": DecisionTreeClassifier(class_weight='balanced', random_state=42)
}




print("\nEjecutando evaluación exhaustiva para la gran tabla maestra...")
results = []
predicciones_almacenadas = {}

for rep_name, (X_tr, X_te, is_dense) in representations.items():
    if is_dense:
        if hasattr(X_tr, "toarray"): X_tr = X_tr.toarray()
        if hasattr(X_te, "toarray"): X_te = X_te.toarray()

    for model_name, model_obj in models.items():
        if model_name == "NaiveBayes":
            clf = GaussianNB() if is_dense else MultinomialNB()
        else:
            clf = model_obj

        clf.fit(X_tr, y_train)
        preds = clf.predict(X_te)

        
        predicciones_almacenadas[(rep_name, model_name)] = preds

        p, r, f1, _ = precision_recall_fscore_support(y_test, preds, average='binary', zero_division=0)
        acc = accuracy_score(y_test, preds)

        results.append({
            "Representación": rep_name,
            "Modelo": model_name,
            "Precision": round(p, 2),
            "Recall": round(r, 2),
            "F1-score": round(f1, 2),
            "Accuracy": round(acc, 2)
        })

df_large = pd.DataFrame(results)

print("\n==========================================================================")
print("=== GRAN TABLA MAESTRA DE ABLACIÓN (REPRESENTACIONES X CLASIFICADOR) ======")
print("==========================================================================")
print(df_large.to_string(index=False))




print("\n==================================================================")
print("=== PRUEBA DE MCNEMAR: HMT-V1 OPTIMIZADO VS. BASES ================")
print("==================================================================")

mcnemar_comparativas = []

for model_name in models.keys():
    preds_hmt = predicciones_almacenadas[("5. HMT-V1 (Optimizado)", model_name)]
    correct_hmt = (preds_hmt == y_test)

    baselines_a_comparar = ["1. TF-IDF Puro", "2. N-Grams Puros", "3. Lexical Features", "4. Latent Topics (LSA)"]

    for base_name in baselines_a_comparar:
        preds_base = predicciones_almacenadas[(base_name, model_name)]
        correct_base = (preds_base == y_test)

        
        b_acierta_hmt_falla = np.sum(correct_base & ~correct_hmt)
        b_falla_hmt_acierta = np.sum(~correct_base & correct_hmt)
        ambos_aciertan = np.sum(correct_base & correct_hmt)
        ambos_fallan = np.sum(~correct_base & ~correct_hmt)

        tabla_contingencia = [[ambos_aciertan, b_acierta_hmt_falla],
                              [b_falla_hmt_acierta, ambos_fallan]]

        res = mcnemar(tabla_contingencia, exact=False, correction=True)
        is_sig = res.pvalue < 0.5
        sig_text = "Sí (p < 0.05)" if is_sig else "No"

        mcnemar_comparativas.append({
            "Clasificador": model_name,
            "Comparativa": f"HMT-V1 Opt vs {base_name.split('.')[1].strip()}",
            "Estadístico": round(res.statistic, 4),
            "P-valor": f"{res.pvalue:.5e}",
            "Significativo (p < 0.05)": sig_text
        })

df_mcnemar = pd.DataFrame(mcnemar_comparativas)
print(df_mcnemar.to_string(index=False))


### EXP-022 · V3

**Archivo de origen:** `Día_19_Octubre_2026.ipynb, celda 30`. **Referencias duplicadas:** 2. **Sintaxis de código:** Válida.

SCRIPT DOCTORAL MAESTRO (MULTICLASE - TASK 2 - TRAIN & TEST SEPARADOS):
HMT-V3 Hiper-Optimizado + Ablación + McNemar + Evaluación en Test Real
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from

**Ejecución:** celda independiente o variante experimental; revisar archivos de entrada y dependencias antes de ejecutarla. Los resultados originales no se insertan en esta celda.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB, GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from sklearn.metrics import precision_recall_fscore_support, accuracy_score, classification_report, confusion_matrix
from scipy.sparse import hstack, csr_matrix
from statsmodels.stats.contingency_tables import mcnemar

print("1. Cargando los conjuntos oficiales de entrenamiento y prueba...")
train_df = pd.read_csv('EXIST2021_train_reestructurado.csv').dropna(subset=['text', 'task2'])
test_df = pd.read_csv('EXIST2021_test_reestructurado.csv').dropna(subset=['text', 'task2'])

X_train_txt = train_df['text'].astype(str).values
X_test_txt = test_df['text'].astype(str).values


encoder = LabelEncoder()
y_train = encoder.fit_transform(train_df['task2'].astype(str))
y_test = encoder.transform(test_df['task2'].astype(str))
clases_nombres = encoder.classes_

print(f"   -> Muestras de Entrenamiento (Train): {len(X_train_txt)}")
print(f"   -> Muestras de Evaluación (Test): {len(X_test_txt)}")

print("\n2. Extrayendo representaciones base y el modelo HMT-V3 Hiper-Optimizado...")


tfidf_word = TfidfVectorizer(max_features=25000, ngram_range=(1, 3), sublinear_tf=True, min_df=2)
X_tfidf_w_tr = tfidf_word.fit_transform(X_train_txt)
X_tfidf_w_te = tfidf_word.transform(X_test_txt)


tfidf_char = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), max_features=30000, sublinear_tf=True, min_df=3)
X_tfidf_c_tr = tfidf_char.fit_transform(X_train_txt)
X_tfidf_c_te = tfidf_char.transform(X_test_txt)


count_vec = CountVectorizer(ngram_range=(1, 3), max_features=10000, min_df=2)
X_ngram_tr = count_vec.fit_transform(X_train_txt)
X_ngram_te = count_vec.transform(X_test_txt)


def extraer_lexico(texto):
    t = texto.lower()
    insultos = ['puta', 'zorra', 'gorda', 'imbécil', 'estúpida', 'asquerosa', 'inútil', 'tarada']
    imperativos = ['callate', 'cocina', 'fregar', 'vete', 'largo']
    return [
        sum(1 for p in insultos if p in t),
        sum(1 for p in imperativos if p in t),
        len(texto) / 100.0,
        sum(1 for c in texto if c.isupper()) / max(1, len(texto))
    ]

lex_tr = np.array([extraer_lexico(t) for t in X_train_txt], dtype=float)
lex_te = np.array([extraer_lexico(t) for t in X_test_txt], dtype=float)
scaler_lex = StandardScaler()
X_lex_tr = scaler_lex.fit_transform(lex_tr)
X_lex_te = scaler_lex.transform(lex_te)


svd = TruncatedSVD(n_components=250, random_state=42)
X_lsa_tr = svd.fit_transform(X_tfidf_w_tr)
X_lsa_te = svd.transform(X_tfidf_w_te)
scaler_lsa = StandardScaler()
X_lsa_tr_sc = scaler_lsa.fit_transform(X_lsa_tr)
X_lsa_te_sc = scaler_lsa.transform(X_lsa_te)


def extraer_caracteristicas_hmt_v3(texto):
    t = texto.lower()
    insultos = ['puta', 'zorra', 'gorda', 'imbécil', 'estúpida', 'asquerosa', 'inútil', 'tarada', 'zorras', 'imbéciles', 'estúpidas', 'idiota', 'prostituta', 'calienta', 'retrasada', 'zorron']
    imperativos = ['callate', 'cocina', 'fregar', 'lugar', 'deberían', 'deben', 'debes', 'vete', 'largo', 'fuera', 'sálvete', 'limpia', 'atiende', 'sírvenos', 'sufre', 'ordena']
    desprecio = ['asco', 'basura', 'ridícula', 'falsa', 'inútiles', 'patética', 'estupidez', 'ridículos', 'inservible', 'barata', 'mediocre', 'inservibles']
    estereotipos = ['mujeres', 'feminazi', 'ellas', 'esposa', 'madre', 'novia', 'niñita', 'sola', 'callada', 'emocionales', 'débiles', 'casa', 'cocina', 'objeto', 'calladita']
    ideologia = ['feminismo', 'feminista', 'machista', 'patriarcado', 'igualdad', 'derechos', 'sociedad', 'ideología', 'opresión', 'hembrista']

    n_insultos = sum(12.0 for p in insultos if p in t)
    n_imperativos = sum(10.0 for p in imperativos if p in t)
    n_desprecio = sum(12.0 for p in desprecio if p in t)
    n_estereotipos = sum(10.0 for p in estereotipos if p in t)
    n_ideologia = sum(10.0 for p in ideologia if p in t)

    longitud = len(texto) / 100.0
    mayusculas = sum(1 for c in texto if c.isupper()) / max(1, len(texto))
    exclamaciones = texto.count('!')
    interrogaciones = texto.count('?')

    return [n_insultos, n_imperativos, n_desprecio, n_estereotipos, n_ideologia, longitud, mayusculas, exclamaciones, interrogaciones]

s_train = np.array([extraer_caracteristicas_hmt_v3(t) for t in X_train_txt], dtype=float)
s_test = np.array([extraer_caracteristicas_hmt_v3(t) for t in X_test_txt], dtype=float)
scaler_s = StandardScaler()
X_s_tr_sc = scaler_s.fit_transform(s_train)
X_s_te_sc = scaler_s.transform(s_test)


HMT3_tr = hstack([X_tfidf_w_tr, X_tfidf_c_tr, csr_matrix(X_lsa_tr_sc * 10.0), csr_matrix(X_s_tr_sc * 15.0)])
HMT3_te = hstack([X_tfidf_w_te, X_tfidf_c_te, csr_matrix(X_lsa_te_sc * 10.0), csr_matrix(X_s_te_sc * 15.0)])


representations = {
    "1. TF-IDF Puro": (X_tfidf_w_tr, X_tfidf_w_te, False),
    "2. N-Grams Puros": (X_ngram_tr, X_ngram_te, False),
    "3. Lexical Features": (X_lex_tr, X_lex_te, True),
    "4. Latent Topics (LSA)": (X_lsa_tr_sc, X_lsa_te_sc, True),
    "5. HMT-V3 (Hiper-Optimizado)": (HMT3_tr, HMT3_te, True)
}


models = {
    "LogReg": LogisticRegression(max_iter=3000, C=6.0, class_weight='balanced', random_state=42),
    "3-NN": KNeighborsClassifier(n_neighbors=3, weights='distance'),
    "SVM": SVC(kernel='linear', C=5.0, class_weight='balanced', random_state=42),
    "NaiveBayes": "auto",
    "Red Neuronal": MLPClassifier(hidden_layer_sizes=(256, 128, 64), max_iter=1000, alpha=0.0001, random_state=42),
    "RandomForest": RandomForestClassifier(n_estimators=400, class_weight='balanced', random_state=42),
    "DecisionTree": DecisionTreeClassifier(max_depth=16, class_weight='balanced', random_state=42)
}




print("\nEjecutando evaluación multiclase exhaustiva en el set de prueba (Test)...")
results = []
predicciones_almacenadas = {}

for rep_name, (X_tr, X_te, is_dense) in representations.items():
    if is_dense:
        if hasattr(X_tr, "toarray"): X_tr = X_tr.toarray()
        if hasattr(X_te, "toarray"): X_te = X_te.toarray()

    for model_name, model_obj in models.items():
        if model_name == "NaiveBayes":
            clf = GaussianNB() if is_dense else MultinomialNB()
        else:
            clf = model_obj

        clf.fit(X_tr, y_train)
        preds = clf.predict(X_te)

        predicciones_almacenadas[(rep_name, model_name)] = preds

        p, r, f1, _ = precision_recall_fscore_support(y_test, preds, average='macro', zero_division=0)
        acc = accuracy_score(y_test, preds)

        results.append({
            "Representación": rep_name,
            "Modelo": model_name,
            "Precision (Macro)": round(p, 2),
            "Recall (Macro)": round(r, 2),
            "F1-score (Macro)": round(f1, 2),
            "Accuracy": round(acc, 2)
        })

df_large = pd.DataFrame(results)

print("\n==========================================================================")
print("=== GRAN TABLA MAESTRA DE ABLACIÓN MULTICLASE (TEST SET OFICIAL) =========")
print("==========================================================================")
print(df_large.to_string(index=False))




print("\n==================================================================")
print("=== PRUEBA DE MCNEMAR MULTICLASE: HMT-V3 VS. BASES (TEST SET) =====")
print("==================================================================")

mcnemar_comparativas = []

for model_name in models.keys():
    preds_hmt = predicciones_almacenadas[("5. HMT-V3 (Hiper-Optimizado)", model_name)]
    correct_hmt = (preds_hmt == y_test)

    baselines_a_comparar = ["1. TF-IDF Puro", "2. N-Grams Puros", "3. Lexical Features", "4. Latent Topics (LSA)"]

    for base_name in baselines_a_comparar:
        preds_base = predicciones_almacenadas[(base_name, model_name)]
        correct_base = (preds_base == y_test)

        b_acierta_hmt_falla = np.sum(correct_base & ~correct_hmt)
        b_falla_hmt_acierta = np.sum(~correct_base & correct_hmt)
        ambos_aciertan = np.sum(correct_base & correct_hmt)
        ambos_fallan = np.sum(~correct_base & ~correct_hmt)

        tabla_contingencia = [[ambos_aciertan, b_acierta_hmt_falla],
                              [b_falla_hmt_acierta, ambos_fallan]]

        res = mcnemar(tabla_contingencia, exact=False, correction=True)
        is_sig = res.pvalue < 0.05
        sig_text = "Sí (p < 0.05)" if is_sig else "No"

        mcnemar_comparativas.append({
            "Clasificador": model_name,
            "Comparativa": f"HMT-V3 vs {base_name.split('.')[1].strip()}",
            "Estadístico": round(res.statistic, 4),
            "P-valor": f"{res.pvalue:.5e}",
            "Significativo (p < 0.05)": sig_text
        })

df_mcnemar = pd.DataFrame(mcnemar_comparativas)
print(df_mcnemar.to_string(index=False))




print("\n==================================================================")
print("=== REPORTE DE RENDIMIENTO POR CLASE (TASK 2 - TEST OFICIAL) =====")
print("==================================================================")

preds_hmt_best = predicciones_almacenadas[("5. HMT-V3 (Hiper-Optimizado)", "LogReg")]
print(classification_report(y_test, preds_hmt_best, target_names=clases_nombres, zero_division=0))


plt.figure(figsize=(9, 7))
cm = confusion_matrix(y_test, preds_hmt_best, normalize='true')
sns.heatmap(cm, annot=True, fmt='.2f', cmap='Blues', xticklabels=clases_nombres, yticklabels=clases_nombres)
plt.xlabel('Predicción del Modelo HMT-V3 (Test Set)')
plt.ylabel('Etiqueta Real (Test Set)')
plt.title('Matriz de Confusión Normalizada - Task 2 Multiclase (LogReg HMT-V3 Test)')
plt.tight_layout()
plt.show()


### EXP-023 · V1, V2, V3

**Archivo de origen:** `Día_19_Octubre_2026.ipynb, celda 32`. **Referencias duplicadas:** 0. **Sintaxis de código:** Válida.

-*- coding: utf-8 -*-
"""HMT jerárquico para EXIST: task1 (binaria) -> tipo_violencia (multiclase).
Archivos separados train y test. Selección de HMT-V1/V2/V3 exclusivamente
con validación derivada del TRAIN. TEST se usa UNA vez al final.
D

**Ejecución:** celda independiente o variante experimental; revisar archivos de entrada y dependencias antes de ejecutarla. Los resultados originales no se insertan en esta celda.

In [ ]:
"""HMT jerárquico para EXIST: task1 (binaria) -> tipo_violencia (multiclase).

Archivos separados train y test. Selección de HMT-V1/V2/V3 exclusivamente
con validación derivada del TRAIN. TEST se usa UNA vez al final.

Dependencias: pip install numpy pandas scipy scikit-learn matplotlib joblib
"""
from __future__ import annotations

import re
import unicodedata
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.sparse import csr_matrix, hstack
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay,
                             f1_score, precision_recall_fscore_support)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import Normalizer, StandardScaler


TRAIN_FILE = 'EXIST2021_train_reestructurado.csv'
TEST_FILE = 'EXIST2021_test_reestructurado.csv'
OUTPUT_DIR = Path('Resultados_HMT')
RANDOM_STATE = 42
VALIDATION_SIZE = 0.20
NO_VIOLENCE_LABEL = 'No violencia'  

VERSIONS = ('HMT-V1', 'HMT-V2', 'HMT-V3')
C_VALUES = (0.1, 1.0, 5.0)
THRESHOLDS = (0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def norm(s):
    s = unicodedata.normalize('NFKD', str(s)).encode('ascii', 'ignore').decode('ascii')
    return re.sub(r'\s+', ' ', s.casefold().strip())


def clean_text(s):
    
    return re.sub(r'\s+', ' ', str(s)).strip()


def load_data(path, name):
    df = pd.read_csv(path, encoding='utf-8-sig')
    cols = {'text', 'task1', 'tipo_violencia'}
    missing = cols - set(df.columns)
    if missing:
        raise ValueError(f'{name}: faltan columnas {sorted(missing)}. Disponibles: {list(df.columns)}')
    
    
    keep = (['id'] if 'id' in df else []) + ['text', 'task1', 'tipo_violencia']
    df = df[keep].copy()
    df = df.dropna(subset=['text', 'task1', 'tipo_violencia']).reset_index(drop=True)
    df['text'] = df['text'].map(clean_text)
    df = df[df['text'].str.len().gt(0)].copy().reset_index(drop=True)
    labels = df['task1'].map(norm)
    mapping = {'sexist': 1, 'non-sexist': 0, 'non sexist': 0,
               'sexista': 1, 'no sexista': 0, 'no-sexista': 0}
    unknown = set(labels.unique()) - set(mapping)
    if unknown:
        raise ValueError(f'{name}: valores desconocidos de task1: {sorted(unknown)}')
    df['binary'] = labels.map(mapping).astype(int)
    
    df['violence'] = df['tipo_violencia'].astype(str).str.strip()
    df.loc[df['violence'].map(norm).isin({'no violencia', 'sin violencia', 'no violence'}),
           'violence'] = NO_VIOLENCE_LABEL
    inconsistent = ((df['binary'] == 0) & (df['violence'] != NO_VIOLENCE_LABEL)) | \
                   ((df['binary'] == 1) & (df['violence'] == NO_VIOLENCE_LABEL))
    if inconsistent.any():
        warnings.warn(f'{name}: {inconsistent.sum()} filas con etiquetas task1 / tipo_violencia '
                      'incompatibles. Revisa etiquetas; no se reetiquetan automáticamente.')
    print(f'{name}: {len(df)} filas; binary={df.binary.value_counts().to_dict()}')
    print(f'{name}: clases violencia={df.violence.value_counts().to_dict()}')
    return df



CUES = {
    'insultos': ('puta', 'zorra', 'imbecil', 'estupida', 'idiota', 'inutil', 'asquerosa'),
    'mandatos': ('callate', 'vete', 'cocina', 'limpia', 'obedece'),
    'estereotipos': ('mujeres', 'esposa', 'feminazi', 'debil', 'calladita'),
    'desprecio': ('asco', 'basura', 'ridicula', 'patetica', 'inservible'),
}


def pragmatic_features(texts):
    rows = []
    for s in texts:
        t = norm(s)
        words = re.findall(r'\b\w+\b', t)
        wordset = set(words)
        n = max(len(s), 1)
        rows.append([
            *(sum(w in wordset for w in vocab) for vocab in CUES.values()),
            np.log1p(len(words)),
            len(set(words)) / max(len(words), 1),
            sum(c.isupper() for c in s) / n,
            np.log1p(s.count('!')),
            np.log1p(s.count('?')),
            int('http' in t),
            int('@' in s),
            int('#' in s),
        ])
    return np.asarray(rows, dtype=np.float32)


class HMTFeatures:
    """Ajusta vocabulario, LSA y escaladores SOLO en los datos recibidos en fit."""
    def __init__(self, version):
        self.version = version
        self.word = TfidfVectorizer(ngram_range=(1, 2), max_features=30000,
                                    min_df=2, sublinear_tf=True, strip_accents='unicode')
        self.char = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5),
                                    max_features=35000, min_df=2, sublinear_tf=True)
        self.scaler = StandardScaler()
        self.normalizer = Normalizer()
        self.svd = None

    def fit(self, texts):
        self.word.fit(texts)
        Xw = self.word.transform(texts)
        if self.version == 'HMT-V3':
            self.char.fit(texts)
        
        n_components = min(120, Xw.shape[0] - 1, Xw.shape[1] - 1)
        if n_components >= 2:
            self.svd = TruncatedSVD(n_components=n_components, random_state=RANDOM_STATE)
            self.svd.fit(Xw)
        if self.version in ('HMT-V2', 'HMT-V3'):
            self.scaler.fit(pragmatic_features(texts))
        return self

    def transform(self, texts):
        texts = np.asarray(texts, dtype=str)
        parts = [self.word.transform(texts).multiply(1.0).tocsr()]
        if self.version == 'HMT-V3':
            parts.append(self.char.transform(texts).multiply(0.75).tocsr())
        if self.svd is not None:
            Xtopic = Normalizer().fit_transform(self.svd.transform(self.word.transform(texts)))
            parts.append(csr_matrix(Xtopic * 0.35))
        if self.version in ('HMT-V2', 'HMT-V3'):
            Xprag = self.scaler.transform(pragmatic_features(texts))
            
            Xprag = self.normalizer.transform(Xprag)
            parts.append(csr_matrix(Xprag * 0.25))
        return hstack(parts, format='csr', dtype=np.float32)


def fit_hierarchy(df, version, c_binary, c_multiclass):
    texts = df.text.to_numpy()
    feats = HMTFeatures(version).fit(texts)
    X = feats.transform(texts)
    y = df.binary.to_numpy()
    if len(np.unique(y)) < 2:
        raise ValueError('task1 debe contener al menos 2 clases en train.')
    binary = LogisticRegression(C=c_binary, class_weight='balanced',
                                max_iter=2000, random_state=RANDOM_STATE)
    binary.fit(X, y)
    sex = df.binary.eq(1).to_numpy()
    categories = df.loc[sex, 'violence'].to_numpy()
    if NO_VIOLENCE_LABEL in categories:
        raise ValueError('Hay muestras sexistas con etiqueta No violencia: corrige train.')
    if len(np.unique(categories)) < 2:
        raise ValueError('Se requieren >=2 clases de tipo_violencia entre textos sexistas.')
    multi = LogisticRegression(C=c_multiclass, class_weight='balanced',
                               max_iter=2000, random_state=RANDOM_STATE)
    multi.fit(X[sex], categories)  
    return {'features': feats, 'binary': binary, 'multi': multi,
            'version': version, 'c_binary': c_binary, 'c_multi': c_multiclass}


def predict_hierarchy(bundle, texts, threshold=0.5):
    X = bundle['features'].transform(texts)
    prob = bundle['binary'].predict_proba(X)[:, list(bundle['binary'].classes_).index(1)]
    binary_pred = (prob >= threshold).astype(int)
    violence_pred = np.full(len(texts), NO_VIOLENCE_LABEL, dtype=object)
    positive = binary_pred == 1
    if positive.any():
        violence_pred[positive] = bundle['multi'].predict(X[positive])
    return binary_pred, violence_pred, prob


def metrics(y_true, y_pred):
    p, r, f, _ = precision_recall_fscore_support(y_true, y_pred,
                                                  average='macro', zero_division=0)
    return {'precision_macro': float(p), 'recall_macro': float(r),
            'f1_macro': float(f), 'accuracy': float(accuracy_score(y_true, y_pred))}


def print_evaluation(df, pred_bin, pred_multi, prefix, folder):
    print(f'\n=== {prefix} | TASK 1: BINARIA ===')
    print(classification_report(df.binary, pred_bin, labels=[0, 1],
                                target_names=['non-sexist', 'sexist'], zero_division=0))
    classes = sorted(set(df.violence) | set(pred_multi))
    print(f'\n=== {prefix} | TIPO DE VIOLENCIA: CASCADA EN TEST COMPLETO ===')
    print(classification_report(df.violence, pred_multi, labels=classes, zero_division=0))
    sex = df.binary.eq(1).to_numpy()
    print(f'\n=== {prefix} | TIPO DE VIOLENCIA: SOLO CASOS SEXISTAS REALES ===')
    sex_classes = [x for x in classes if x != NO_VIOLENCE_LABEL]
    print(classification_report(df.violence.to_numpy()[sex], np.asarray(pred_multi)[sex],
                                labels=sex_classes, zero_division=0))
    for y, p, names, file, title in [
        (df.binary, pred_bin, [0, 1], 'matriz_task1.png', 'Task 1 - binaria'),
        (df.violence, pred_multi, classes, 'matriz_tipo_violencia.png', 'Tipo de violencia - cascada')
    ]:
        fig, ax = plt.subplots(figsize=(max(6, len(names)*1.25), max(5, len(names)*1.05)))
        cm = confusion_matrix(y, p, labels=names, normalize='true')
        ConfusionMatrixDisplay(cm, display_labels=names).plot(ax=ax, cmap='Blues',
                                                               values_format='.2f', colorbar=False)
        ax.set_title(title)
        ax.tick_params(axis='x', labelrotation=35)
        fig.tight_layout()
        fig.savefig(folder / file, dpi=220)
        plt.close(fig)


def main():
    train = load_data(TRAIN_FILE, 'TRAIN')
    test = load_data(TEST_FILE, 'TEST')
    if (train.binary.eq(0) & train.violence.ne(NO_VIOLENCE_LABEL)).any() or \
       (train.binary.eq(1) & train.violence.eq(NO_VIOLENCE_LABEL)).any():
        raise ValueError('TRAIN contiene etiquetas contradictorias: corrígelas antes de entrenar.')
    train_sex_classes = set(train.loc[train.binary.eq(1), 'violence'])
    novel = set(test.loc[test.binary.eq(1), 'violence']) - train_sex_classes
    if novel:
        warnings.warn(f'TEST tiene clases de violencia no vistas en TRAIN: {novel}')

    strata = train.violence
    if strata.value_counts().min() < 2:
        strata = train.binary
    train_dev, valid = train_test_split(train, test_size=VALIDATION_SIZE,
                                        random_state=RANDOM_STATE, stratify=strata)
    train_dev, valid = train_dev.reset_index(drop=True), valid.reset_index(drop=True)
    print(f'Validación interna: {len(train_dev)} fit / {len(valid)} valid. TEST intacto.')

    results = []
    best = None
    
    for version in VERSIONS:
        
        f = HMTFeatures(version).fit(train_dev.text.to_numpy())
        Xfit = f.transform(train_dev.text.to_numpy())
        Xval = f.transform(valid.text.to_numpy())
        sex = train_dev.binary.eq(1).to_numpy()
        bmodels = {}
        mmodels = {}
        for cb in C_VALUES:
            model = LogisticRegression(C=cb, class_weight='balanced', max_iter=2000,
                                       random_state=RANDOM_STATE)
            model.fit(Xfit, train_dev.binary)
            bmodels[cb] = model.predict_proba(Xval)[:, list(model.classes_).index(1)]
        for cm in C_VALUES:
            model = LogisticRegression(C=cm, class_weight='balanced', max_iter=2000,
                                       random_state=RANDOM_STATE)
            model.fit(Xfit[sex], train_dev.loc[sex, 'violence'])
            mmodels[cm] = model.predict(Xval)
        for cb, prob in bmodels.items():
            for cm, pclasses in mmodels.items():
                for threshold in THRESHOLDS:
                    pb = (prob >= threshold).astype(int)
                    pm = np.where(pb == 1, pclasses, NO_VIOLENCE_LABEL)
                    mb = metrics(valid.binary, pb)
                    mm = metrics(valid.violence, pm)
                    
                    key = (mm['f1_macro'], mb['f1_macro'], mm['accuracy'])
                    row = {'version': version, 'C_binary': cb, 'C_multiclass': cm,
                           'threshold': threshold, 'val_binary_f1_macro': mb['f1_macro'],
                           'val_multiclass_f1_macro': mm['f1_macro'],
                           'val_multiclass_accuracy': mm['accuracy']}
                    results.append(row)
                    if best is None or key > best[0]:
                        best = (key, row)
        print(f'Validado: {version} ({len(C_VALUES)**2 * len(THRESHOLDS)} combinaciones)')

    pd.DataFrame(results).sort_values(['val_multiclass_f1_macro', 'val_binary_f1_macro'],
                                      ascending=False).to_csv(OUTPUT_DIR/'busqueda_validacion.csv', index=False)
    selected = best[1]
    print('\nMEJOR CONFIGURACIÓN (elegida SOLO en validación):', selected)
    final = fit_hierarchy(train, selected['version'],
                          selected['C_binary'], selected['C_multiclass'])
    pb, pm, pp = predict_hierarchy(final, test.text.to_numpy(), selected['threshold'])
    print_evaluation(test, pb, pm, 'TEST FINAL', OUTPUT_DIR)

    summary = pd.DataFrame([
        {'tarea': 'task1_binaria', **metrics(test.binary, pb)},
        {'tarea': 'tipo_violencia_cascada', **metrics(test.violence, pm)},
        {'tarea': 'tipo_violencia_solo_sexistas_reales',
         **metrics(test.loc[test.binary.eq(1), 'violence'], pm[test.binary.eq(1).to_numpy()])}
    ])
    print('\nRESULTADOS FINALES:')
    print(summary.to_string(index=False))
    summary.to_csv(OUTPUT_DIR/'metricas_test.csv', index=False)
    out = test[['id', 'text']].copy() if 'id' in test else test[['text']].copy()
    out['task1_real'] = test.task1
    out['task1_pred'] = np.where(pb == 1, 'sexist', 'non-sexist')
    out['prob_sexist'] = pp
    out['tipo_violencia_real'] = test.violence
    out['tipo_violencia_pred'] = pm
    out.to_csv(OUTPUT_DIR/'predicciones_test.csv', index=False, encoding='utf-8-sig')
    joblib.dump({'model': final, 'threshold': selected['threshold']}, OUTPUT_DIR/'modelo_HMT.joblib')
    print(f'\nArchivos guardados en {OUTPUT_DIR.resolve()}')


if __name__ == '__main__':
    main()


### EXP-024 · V1, V2, V3

**Archivo de origen:** `Día_19_Octubre_2026.ipynb, celda 34`. **Referencias duplicadas:** 0. **Sintaxis de código:** Válida.

-*- coding: utf-8 -*-
"""HMT jerárquico para EXIST: task1 (binaria) -> tipo_violencia (multiclase).
Archivos separados train y test. Selección de HMT-V1/V2/V3 exclusivamente
con validación derivada del TRAIN. TEST se usa UNA vez al final.
D

**Ejecución:** celda independiente o variante experimental; revisar archivos de entrada y dependencias antes de ejecutarla. Los resultados originales no se insertan en esta celda.

In [ ]:
"""HMT jerárquico para EXIST: task1 (binaria) -> tipo_violencia (multiclase).

Archivos separados train y test. Selección de HMT-V1/V2/V3 exclusivamente
con validación derivada del TRAIN. TEST se usa UNA vez al final.

Dependencias: pip install numpy pandas scipy scikit-learn matplotlib joblib
"""

import re
import unicodedata
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.sparse import csr_matrix, hstack
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay,
                             f1_score, precision_recall_fscore_support)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import Normalizer, StandardScaler


TRAIN_FILE = 'EXIST2021_train_reestructurado.csv'
TEST_FILE = 'EXIST2021_test_reestructurado.csv'
OUTPUT_DIR = Path('Resultados_HMT')
RANDOM_STATE = 42
VALIDATION_SIZE = 0.20
NO_VIOLENCE_LABEL = 'No violencia'  

VERSIONS = ('HMT-V1', 'HMT-V2', 'HMT-V3')
C_VALUES = (0.1, 1.0, 5.0)
THRESHOLDS = (0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def norm(s):
    s = unicodedata.normalize('NFKD', str(s)).encode('ascii', 'ignore').decode('ascii')
    return re.sub(r'\s+', ' ', s.casefold().strip())


def clean_text(s):
    
    return re.sub(r'\s+', ' ', str(s)).strip()


def load_data(path, name):
    df = pd.read_csv(path, encoding='utf-8-sig')
    cols = {'text', 'task1', 'tipo_violencia'}
    missing = cols - set(df.columns)
    if missing:
        raise ValueError(f'{name}: faltan columnas {sorted(missing)}. Disponibles: {list(df.columns)}')
    
    
    keep = (['id'] if 'id' in df else []) + ['text', 'task1', 'tipo_violencia']
    df = df[keep].copy()
    df = df.dropna(subset=['text', 'task1', 'tipo_violencia']).reset_index(drop=True)
    df['text'] = df['text'].map(clean_text)
    df = df[df['text'].str.len().gt(0)].copy().reset_index(drop=True)
    labels = df['task1'].map(norm)
    mapping = {'sexist': 1, 'non-sexist': 0, 'non sexist': 0,
               'sexista': 1, 'no sexista': 0, 'no-sexista': 0}
    unknown = set(labels.unique()) - set(mapping)
    if unknown:
        raise ValueError(f'{name}: valores desconocidos de task1: {sorted(unknown)}')
    df['binary'] = labels.map(mapping).astype(int)
    
    df['violence'] = df['tipo_violencia'].astype(str).str.strip()
    df.loc[df['violence'].map(norm).isin({'no violencia', 'sin violencia', 'no violence'}),
           'violence'] = NO_VIOLENCE_LABEL
    inconsistent = ((df['binary'] == 0) & (df['violence'] != NO_VIOLENCE_LABEL)) | \
                   ((df['binary'] == 1) & (df['violence'] == NO_VIOLENCE_LABEL))
    if inconsistent.any():
        warnings.warn(f'{name}: {inconsistent.sum()} filas con etiquetas task1 / tipo_violencia '
                      'incompatibles. Revisa etiquetas; no se reetiquetan automáticamente.')
    print(f'{name}: {len(df)} filas; binary={df.binary.value_counts().to_dict()}')
    print(f'{name}: clases violencia={df.violence.value_counts().to_dict()}')
    return df



CUES = {
    'insultos': ('puta', 'zorra', 'imbecil', 'estupida', 'idiota', 'inutil', 'asquerosa'),
    'mandatos': ('callate', 'vete', 'cocina', 'limpia', 'obedece'),
    'estereotipos': ('mujeres', 'esposa', 'feminazi', 'debil', 'calladita'),
    'desprecio': ('asco', 'basura', 'ridicula', 'patetica', 'inservible'),
}


def pragmatic_features(texts):
    rows = []
    for s in texts:
        t = norm(s)
        words = re.findall(r'\b\w+\b', t)
        wordset = set(words)
        n = max(len(s), 1)
        rows.append([
            *(sum(w in wordset for w in vocab) for vocab in CUES.values()),
            np.log1p(len(words)),
            len(set(words)) / max(len(words), 1),
            sum(c.isupper() for c in s) / n,
            np.log1p(s.count('!')),
            np.log1p(s.count('?')),
            int('http' in t),
            int('@' in s),
            int('#' in s),
        ])
    return np.asarray(rows, dtype=np.float32)


class HMTFeatures:
    """Ajusta vocabulario, LSA y escaladores SOLO en los datos recibidos en fit."""
    def __init__(self, version):
        self.version = version
        self.word = TfidfVectorizer(ngram_range=(1, 2), max_features=30000,
                                    min_df=2, sublinear_tf=True, strip_accents='unicode')
        self.char = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5),
                                    max_features=35000, min_df=2, sublinear_tf=True)
        self.scaler = StandardScaler()
        self.normalizer = Normalizer()
        self.svd = None

    def fit(self, texts):
        self.word.fit(texts)
        Xw = self.word.transform(texts)
        if self.version == 'HMT-V3':
            self.char.fit(texts)
        
        n_components = min(120, Xw.shape[0] - 1, Xw.shape[1] - 1)
        if n_components >= 2:
            self.svd = TruncatedSVD(n_components=n_components, random_state=RANDOM_STATE)
            self.svd.fit(Xw)
        if self.version in ('HMT-V2', 'HMT-V3'):
            self.scaler.fit(pragmatic_features(texts))
        return self

    def transform(self, texts):
        texts = np.asarray(texts, dtype=str)
        parts = [self.word.transform(texts).multiply(1.0).tocsr()]
        if self.version == 'HMT-V3':
            parts.append(self.char.transform(texts).multiply(0.75).tocsr())
        if self.svd is not None:
            Xtopic = Normalizer().fit_transform(self.svd.transform(self.word.transform(texts)))
            parts.append(csr_matrix(Xtopic * 0.35))
        if self.version in ('HMT-V2', 'HMT-V3'):
            Xprag = self.scaler.transform(pragmatic_features(texts))
            
            Xprag = self.normalizer.transform(Xprag)
            parts.append(csr_matrix(Xprag * 0.25))
        return hstack(parts, format='csr', dtype=np.float32)


def fit_hierarchy(df, version, c_binary, c_multiclass):
    texts = df.text.to_numpy()
    feats = HMTFeatures(version).fit(texts)
    X = feats.transform(texts)
    y = df.binary.to_numpy()
    if len(np.unique(y)) < 2:
        raise ValueError('task1 debe contener al menos 2 clases en train.')
    binary = LogisticRegression(C=c_binary, class_weight='balanced',
                                max_iter=2000, random_state=RANDOM_STATE)
    binary.fit(X, y)
    sex = df.binary.eq(1).to_numpy()
    categories = df.loc[sex, 'violence'].to_numpy()
    if NO_VIOLENCE_LABEL in categories:
        raise ValueError('Hay muestras sexistas con etiqueta No violencia: corrige train.')
    if len(np.unique(categories)) < 2:
        raise ValueError('Se requieren >=2 clases de tipo_violencia entre textos sexistas.')
    multi = LogisticRegression(C=c_multiclass, class_weight='balanced',
                               max_iter=2000, random_state=RANDOM_STATE)
    multi.fit(X[sex], categories)  
    return {'features': feats, 'binary': binary, 'multi': multi,
            'version': version, 'c_binary': c_binary, 'c_multi': c_multiclass}


def predict_hierarchy(bundle, texts, threshold=0.5):
    X = bundle['features'].transform(texts)
    prob = bundle['binary'].predict_proba(X)[:, list(bundle['binary'].classes_).index(1)]
    binary_pred = (prob >= threshold).astype(int)
    violence_pred = np.full(len(texts), NO_VIOLENCE_LABEL, dtype=object)
    positive = binary_pred == 1
    if positive.any():
        violence_pred[positive] = bundle['multi'].predict(X[positive])
    return binary_pred, violence_pred, prob


def metrics(y_true, y_pred):
    p, r, f, _ = precision_recall_fscore_support(y_true, y_pred,
                                                  average='macro', zero_division=0)
    return {'precision_macro': float(p), 'recall_macro': float(r),
            'f1_macro': float(f), 'accuracy': float(accuracy_score(y_true, y_pred))}


def print_evaluation(df, pred_bin, pred_multi, prefix, folder):
    print(f'\n=== {prefix} | TASK 1: BINARIA ===')
    print(classification_report(df.binary, pred_bin, labels=[0, 1],
                                target_names=['non-sexist', 'sexist'], zero_division=0))
    classes = sorted(set(df.violence) | set(pred_multi))
    print(f'\n=== {prefix} | TIPO DE VIOLENCIA: CASCADA EN TEST COMPLETO ===')
    print(classification_report(df.violence, pred_multi, labels=classes, zero_division=0))
    sex = df.binary.eq(1).to_numpy()
    print(f'\n=== {prefix} | TIPO DE VIOLENCIA: SOLO CASOS SEXISTAS REALES ===')
    sex_classes = [x for x in classes if x != NO_VIOLENCE_LABEL]
    print(classification_report(df.violence.to_numpy()[sex], np.asarray(pred_multi)[sex],
                                labels=sex_classes, zero_division=0))
    for y, p, names, file, title in [
        (df.binary, pred_bin, [0, 1], 'matriz_task1.png', 'Task 1 - binaria'),
        (df.violence, pred_multi, classes, 'matriz_tipo_violencia.png', 'Tipo de violencia - cascada')
    ]:
        fig, ax = plt.subplots(figsize=(max(6, len(names)*1.25), max(5, len(names)*1.05)))
        cm = confusion_matrix(y, p, labels=names, normalize='true')
        ConfusionMatrixDisplay(cm, display_labels=names).plot(ax=ax, cmap='Blues',
                                                               values_format='.2f', colorbar=False)
        ax.set_title(title)
        ax.tick_params(axis='x', labelrotation=35)
        fig.tight_layout()
        fig.savefig(folder / file, dpi=220)
        plt.close(fig)


def main():
    train = load_data(TRAIN_FILE, 'TRAIN')
    test = load_data(TEST_FILE, 'TEST')
    if (train.binary.eq(0) & train.violence.ne(NO_VIOLENCE_LABEL)).any() or \
       (train.binary.eq(1) & train.violence.eq(NO_VIOLENCE_LABEL)).any():
        raise ValueError('TRAIN contiene etiquetas contradictorias: corrígelas antes de entrenar.')
    train_sex_classes = set(train.loc[train.binary.eq(1), 'violence'])
    novel = set(test.loc[test.binary.eq(1), 'violence']) - train_sex_classes
    if novel:
        warnings.warn(f'TEST tiene clases de violencia no vistas en TRAIN: {novel}')

    strata = train.violence
    if strata.value_counts().min() < 2:
        strata = train.binary
    train_dev, valid = train_test_split(train, test_size=VALIDATION_SIZE,
                                        random_state=RANDOM_STATE, stratify=strata)
    train_dev, valid = train_dev.reset_index(drop=True), valid.reset_index(drop=True)
    print(f'Validación interna: {len(train_dev)} fit / {len(valid)} valid. TEST intacto.')

    results = []
    best = None
    
    for version in VERSIONS:
        
        f = HMTFeatures(version).fit(train_dev.text.to_numpy())
        Xfit = f.transform(train_dev.text.to_numpy())
        Xval = f.transform(valid.text.to_numpy())
        sex = train_dev.binary.eq(1).to_numpy()
        bmodels = {}
        mmodels = {}
        for cb in C_VALUES:
            model = LogisticRegression(C=cb, class_weight='balanced', max_iter=2000,
                                       random_state=RANDOM_STATE)
            model.fit(Xfit, train_dev.binary)
            bmodels[cb] = model.predict_proba(Xval)[:, list(model.classes_).index(1)]
        for cm in C_VALUES:
            model = LogisticRegression(C=cm, class_weight='balanced', max_iter=2000,
                                       random_state=RANDOM_STATE)
            model.fit(Xfit[sex], train_dev.loc[sex, 'violence'])
            mmodels[cm] = model.predict(Xval)
        for cb, prob in bmodels.items():
            for cm, pclasses in mmodels.items():
                for threshold in THRESHOLDS:
                    pb = (prob >= threshold).astype(int)
                    pm = np.where(pb == 1, pclasses, NO_VIOLENCE_LABEL)
                    mb = metrics(valid.binary, pb)
                    mm = metrics(valid.violence, pm)
                    
                    key = (mm['f1_macro'], mb['f1_macro'], mm['accuracy'])
                    row = {'version': version, 'C_binary': cb, 'C_multiclass': cm,
                           'threshold': threshold, 'val_binary_f1_macro': mb['f1_macro'],
                           'val_multiclass_f1_macro': mm['f1_macro'],
                           'val_multiclass_accuracy': mm['accuracy']}
                    results.append(row)
                    if best is None or key > best[0]:
                        best = (key, row)
        print(f'Validado: {version} ({len(C_VALUES)**2 * len(THRESHOLDS)} combinaciones)')

    pd.DataFrame(results).sort_values(['val_multiclass_f1_macro', 'val_binary_f1_macro'],
                                      ascending=False).to_csv(OUTPUT_DIR/'busqueda_validacion.csv', index=False)
    selected = best[1]
    print('\nMEJOR CONFIGURACIÓN (elegida SOLO en validación):', selected)
    final = fit_hierarchy(train, selected['version'],
                          selected['C_binary'], selected['C_multiclass'])
    pb, pm, pp = predict_hierarchy(final, test.text.to_numpy(), selected['threshold'])
    print_evaluation(test, pb, pm, 'TEST FINAL', OUTPUT_DIR)

    summary = pd.DataFrame([
        {'tarea': 'task1_binaria', **metrics(test.binary, pb)},
        {'tarea': 'tipo_violencia_cascada', **metrics(test.violence, pm)},
        {'tarea': 'tipo_violencia_solo_sexistas_reales',
         **metrics(test.loc[test.binary.eq(1), 'violence'], pm[test.binary.eq(1).to_numpy()])}
    ])
    print('\nRESULTADOS FINALES:')
    print(summary.to_string(index=False))
    summary.to_csv(OUTPUT_DIR/'metricas_test.csv', index=False)
    out = test[['id', 'text']].copy() if 'id' in test else test[['text']].copy()
    out['task1_real'] = test.task1
    out['task1_pred'] = np.where(pb == 1, 'sexist', 'non-sexist')
    out['prob_sexist'] = pp
    out['tipo_violencia_real'] = test.violence
    out['tipo_violencia_pred'] = pm
    out.to_csv(OUTPUT_DIR/'predicciones_test.csv', index=False, encoding='utf-8-sig')
    joblib.dump({'model': final, 'threshold': selected['threshold']}, OUTPUT_DIR/'modelo_HMT.joblib')
    print(f'\nArchivos guardados en {OUTPUT_DIR.resolve()}')





from pathlib import Path
from collections import defaultdict
import copy
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from scipy.sparse import csr_matrix, hstack, issparse
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler, Normalizer, MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import MultinomialNB, GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay, f1_score)
from sklearn.model_selection import train_test_split
from statsmodels.stats.contingency_tables import mcnemar
from statsmodels.stats.multitest import multipletests

SEED = 42
OUT = Path('Resultados_HMT_Completo')
OUT.mkdir(exist_ok=True, parents=True)
TRAIN_FILE = TRAIN_FILE
TEST_FILE = TEST_FILE
NO_VIOLENCE = NO_VIOLENCE_LABEL

MODELS = ['LogReg', 'SVM_lineal', '3-NN', 'NaiveBayes', 'Red_Neuronal', 'RandomForest', 'DecisionTree']
REPRESENTATIONS = ['TF-IDF', 'N-Grams', 'Lexical', 'LSA', 'HMT-V1', 'HMT-V2', 'HMT-V3']
HMT_VERSIONS = ['HMT-V1', 'HMT-V2', 'HMT-V3']
C_GRID = (0.1, 1.0, 5.0)
THRESHOLDS = (0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65)


def metric(y, p):
    pr, rec, f1, _ = precision_recall_fscore_support(y, p, average='macro', zero_division=0)
    return {'precision_macro': pr, 'recall_macro': rec, 'f1_macro': f1,
            'accuracy': accuracy_score(y, p)}


class BaselineFeatures:
    """Representación ajustada solo en TRAIN; el TEST nunca ajusta el vocabulario."""
    def __init__(self, name):
        self.name = name
        self.vectorizer = None
        self.svd = None
        self.scaler = None
        self.is_dense = name in ('Lexical', 'LSA')

    def fit(self, texts):
        if self.name == 'TF-IDF':
            self.vectorizer = TfidfVectorizer(max_features=25000, ngram_range=(1, 3),
                                             min_df=2, sublinear_tf=True, strip_accents='unicode')
        elif self.name == 'N-Grams':
            self.vectorizer = CountVectorizer(max_features=10000, ngram_range=(1, 3),
                                              min_df=2, strip_accents='unicode')
        elif self.name == 'LSA':
            self.vectorizer = TfidfVectorizer(max_features=25000, ngram_range=(1, 3),
                                             min_df=2, sublinear_tf=True, strip_accents='unicode')
        if self.vectorizer is not None:
            X = self.vectorizer.fit_transform(texts)
            if self.name == 'LSA':
                n = min(120, X.shape[0]-1, X.shape[1]-1)
                if n < 2:
                    raise ValueError('No hay términos suficientes para LSA')
                self.svd = TruncatedSVD(n_components=n, random_state=SEED)
                self.svd.fit(X)
                self.scaler = Normalizer()
        elif self.name == 'Lexical':
            self.scaler = StandardScaler()
            self.scaler.fit(pragmatic_features(texts))
        return self

    def transform(self, texts):
        if self.name == 'Lexical':
            return self.scaler.transform(pragmatic_features(texts))
        X = self.vectorizer.transform(texts)
        if self.name == 'LSA':
            return self.scaler.transform(self.svd.transform(X))
        return X


def prepare_rep(name, texts_train, texts_test):
    feat = HMTFeatures(name) if name.startswith('HMT-') else BaselineFeatures(name)
    feat.fit(texts_train)
    return feat, feat.transform(texts_train), feat.transform(texts_test)


def create_model(name, X, *, balanced=True, C=1.0):
    if name == 'LogReg':
        return LogisticRegression(max_iter=2000, C=C, class_weight='balanced' if balanced else None,
                                  random_state=SEED)
    if name == 'SVM_lineal':
        return LinearSVC(C=C, class_weight='balanced' if balanced else None, random_state=SEED,
                         max_iter=3500)
    if name == '3-NN':
        return KNeighborsClassifier(n_neighbors=3, weights='distance')
    if name == 'NaiveBayes':
        return GaussianNB() if not issparse(X) else MultinomialNB()
    if name == 'Red_Neuronal':
        
        return MLPClassifier(hidden_layer_sizes=(128, 64), alpha=1e-3, early_stopping=False,
                             max_iter=180, random_state=SEED)
    if name == 'RandomForest':
        return RandomForestClassifier(n_estimators=200, class_weight='balanced' if balanced else None,
                                      n_jobs=-1, random_state=SEED)
    if name == 'DecisionTree':
        return DecisionTreeClassifier(max_depth=18, class_weight='balanced' if balanced else None,
                                      random_state=SEED)
    raise KeyError(name)


def fit_model(model_name, X_train, y_train):
    
    
    
    if model_name == 'NaiveBayes' and ((issparse(X_train) and X_train.data.size and np.min(X_train.data) < 0) or (not issparse(X_train) and np.min(X_train) < 0)):
        return None
    model = create_model(model_name, X_train)
    model.fit(X_train, y_train)
    return model


def confusion(y, pred, labels, name, normalize=None):
    cm = confusion_matrix(y, pred, labels=labels, normalize=normalize)
    fig, ax = plt.subplots(figsize=(max(7, len(labels)*1.3), max(5, len(labels)*1.0)))
    ConfusionMatrixDisplay(cm, display_labels=labels).plot(ax=ax, cmap='Blues', colorbar=False,
                                                           values_format='.2f' if normalize else 'd')
    ax.set_title(name)
    ax.tick_params(axis='x', labelrotation=40)
    fig.tight_layout()
    fig.savefig(OUT / (name.replace(' ', '_').replace('/', '-') + '.png'), dpi=220)
    plt.close(fig)


def paired_mcnemar(y, reference_pred, other_pred):
    a = np.asarray(reference_pred) == np.asarray(y)
    b = np.asarray(other_pred) == np.asarray(y)
    
    ref_only = int(np.sum(a & ~b))
    other_only = int(np.sum(~a & b))
    table = [[int(np.sum(a & b)), ref_only],
             [other_only, int(np.sum(~a & ~b))]]
    test = mcnemar(table, exact=(ref_only+other_only < 25), correction=True)
    return {'HMT_solo_aciertos': ref_only, 'Base_solo_aciertos': other_only,
            'p_valor': float(test.pvalue), 'HMT_mas_aciertos': ref_only > other_only}


def choose_best_hmt(train):
    strat = train.violence if train.violence.value_counts().min() >= 2 else train.binary
    dev, val = train_test_split(train, test_size=0.2, random_state=SEED, stratify=strat)
    results = []
    for version in HMT_VERSIONS:
        _, Xd, Xv = prepare_rep(version, dev.text.to_numpy(), val.text.to_numpy())
        sex = dev.binary.to_numpy() == 1
        binary_models = {}
        multi_models = {}
        for c in C_GRID:
            cl = create_model('LogReg', Xd, C=c)
            cl.fit(Xd, dev.binary)
            binary_models[c] = cl.predict_proba(Xv)[:, list(cl.classes_).index(1)]
            cl2 = create_model('LogReg', Xd, C=c)
            cl2.fit(Xd[sex], dev.loc[sex, 'violence'])
            multi_models[c] = cl2.predict(Xv)
        for cb, prob in binary_models.items():
            for cm, types in multi_models.items():
                for threshold in THRESHOLDS:
                    pb = (prob >= threshold).astype(int)
                    pv = np.where(pb == 1, types, NO_VIOLENCE)
                    results.append({'representation':version,'C_binary':cb,'C_multiclass':cm,
                                    'threshold':threshold,'val_f1_bin':metric(val.binary,pb)['f1_macro'],
                                    'val_f1_cascade':metric(val.violence,pv)['f1_macro']})
    df = pd.DataFrame(results).sort_values(['val_f1_cascade','val_f1_bin'], ascending=False)
    df.to_csv(OUT / 'optimizacion_HMT_validacion.csv', index=False)
    chosen = df.iloc[0].to_dict()
    print('CONFIGURACION HMT SELECCIONADA SOLO EN TRAIN:', chosen)
    return chosen


def run():
    train = load_data(TRAIN_FILE, 'TRAIN')
    test = load_data(TEST_FILE, 'TEST')
    for df, name in [(train, 'TRAIN'), (test, 'TEST')]:
        invalid = ((df.binary==0)&(df.violence!=NO_VIOLENCE))|((df.binary==1)&(df.violence==NO_VIOLENCE))
        if invalid.any():
            raise ValueError(f'{name}: {invalid.sum()} registros tienen task1 y tipo_violencia incoherentes')
    unknown = set(test.loc[test.binary==1,'violence'])-set(train.loc[train.binary==1,'violence'])
    if unknown:
        warnings.warn(f'Hay etiquetas de TEST desconocidas en TRAIN: {sorted(unknown)}')
    best = choose_best_hmt(train)
    best_rep = best['representation']
    predictions = {}
    rows = []
    all_labels = sorted(set(train.violence)|set(test.violence))
    models_by_rep = {}
    for rep in REPRESENTATIONS:
        print('Evaluando representación:', rep, flush=True)
        feat, Xt, Xq = prepare_rep(rep, train.text.to_numpy(), test.text.to_numpy())
        sex_tr = train.binary.to_numpy()==1
        for name in MODELS:
            if name == 'NaiveBayes' and ((issparse(Xt) and Xt.data.size and np.min(Xt.data) < 0) or (not issparse(Xt) and np.min(Xt) < 0)):
                print('  NaiveBayes omitido: matriz sparse con valores negativos.')
                continue
            for task in ('task1', 'tipo_violencia'):
                ytr = train.binary.to_numpy() if task=='task1' else train.loc[sex_tr,'violence'].to_numpy()
                xx = Xt if task=='task1' else Xt[sex_tr]
                model = fit_model(name, xx, ytr)
                if model is None:
                    continue
                yp = model.predict(Xq)
                predictions[(rep, name, task)] = yp
                ytrue = test.binary.to_numpy() if task=='task1' else test.violence.to_numpy()
                if task=='task1':
                    ypred = yp
                else:
                    
                    real_sex = test.binary.to_numpy()==1
                    ytrue = ytrue[real_sex]
                    ypred = yp[real_sex]
                rows.append({'Representacion':rep,'Modelo':name,'Evaluacion':task,
                             **metric(ytrue, ypred)})
        
    table = pd.DataFrame(rows)
    table.to_csv(OUT/'tabla_maestra_base_y_HMT.csv', index=False)
    print('\nTABLA MAESTRA:', table.to_string(index=False))

    
    selected_bundle = fit_hierarchy(train,best_rep,best['C_binary'],best['C_multiclass'])
    hm_bin, hm_cascade, hm_prob = predict_hierarchy(selected_bundle,test.text.to_numpy(),best['threshold'])
    true_bin = test.binary.to_numpy()
    true_multi = test.violence.to_numpy()
    real_sex = true_bin==1
    additional = []
    for task, y, pred in [('task1',true_bin,hm_bin),
                          ('tipo_violencia_cascada',true_multi,hm_cascade),
                          ('tipo_violencia_sexistas_reales',true_multi[real_sex],hm_cascade[real_sex])]:
        additional.append({'Representacion':best_rep,'Modelo':'HMT_optimizado','Evaluacion':task,
                           **metric(y,pred)})
    pd.DataFrame(additional).to_csv(OUT/'resultados_HMT_seleccionado.csv',index=False)
    pd.concat([table,pd.DataFrame(additional)],ignore_index=True).to_csv(OUT/'resultados_completos.csv',index=False)
    for task,y,pred,labels in [('TASK1_HMT',true_bin,hm_bin,[0,1]),
                               ('TIPO_VIOLENCIA_HMT_CASCADA',true_multi,hm_cascade,all_labels)]:
        print('\nREPORTE',task)
        print(classification_report(y,pred,labels=labels,zero_division=0))
        confusion(y,pred,labels,task+'_absoluta')
        confusion(y,pred,labels,task+'_normalizada',normalize='true')

    
    
    
    comparison = []
    statistical = []
    for rep in REPRESENTATIONS:
        for name in MODELS:
            key_b=(rep,name,'task1'); key_m=(rep,name,'tipo_violencia')
            if key_b not in predictions or key_m not in predictions:
                continue
            pb=np.asarray(predictions[key_b]); pm=np.asarray(predictions[key_m])
            pv=np.where(pb==1,pm,NO_VIOLENCE)
            comparison.append({'Representacion':rep,'Modelo':name,
                               'F1_binaria':metric(true_bin,pb)['f1_macro'],
                               'F1_multiclase_condicional':metric(true_multi[real_sex],pm[real_sex])['f1_macro'],
                               'F1_cascada':metric(true_multi,pv)['f1_macro'],
                               'Accuracy_cascada':metric(true_multi,pv)['accuracy']})
            
            if rep not in ('TF-IDF','N-Grams','Lexical','LSA'):
                continue
            for task,y,ref_pred,base_pred in [('task1',true_bin,hm_bin,pb),
                                              ('tipo_violencia_cascada',true_multi,hm_cascade,pv)]:
                statistical.append({'Task':task,'HMT':best_rep,'Representacion_base':rep,
                                    'Clasificador_base':name,
                                    'HMT_F1_macro':metric(y,ref_pred)['f1_macro'],
                                    'Base_F1_macro':metric(y,base_pred)['f1_macro'],
                                    **paired_mcnemar(y,ref_pred,base_pred)})
    cmp = pd.DataFrame(comparison).sort_values('F1_cascada',ascending=False)
    cmp.to_csv(OUT/'comparacion_cascadas.csv',index=False)
    print('\nCOMPARACION DE CASCADAS:',cmp.to_string(index=False))
    stat=pd.DataFrame(statistical)
    if len(stat):
        
        stat['p_Holm']=np.nan
        for task,ii in stat.groupby('Task').groups.items():
            stat.loc[ii,'p_Holm']=multipletests(stat.loc[ii,'p_valor'],method='holm')[1]
        stat['significativo_Holm_005']=stat.p_Holm<0.05
        stat['HMT_supera_significativamente']=stat.significativo_Holm_005 & stat.HMT_mas_aciertos
    stat.to_csv(OUT/'McNemar_pareado_Holm.csv',index=False)
    print('\nMCNEMAR CON CORRECCION HOLM:',stat.to_string(index=False))
    out = test[['id','text']].copy() if 'id' in test.columns else test[['text']].copy()
    out['task1_real']=test.task1
    out['task1_pred']=np.where(hm_bin==1,'sexist','non-sexist')
    out['prob_sexist']=hm_prob
    out['tipo_violencia_real']=true_multi
    out['tipo_violencia_pred']=hm_cascade
    out.to_csv(OUT/'predicciones_HMT_test.csv',index=False,encoding='utf-8-sig')
    joblib.dump({'bundle':selected_bundle,'threshold':best['threshold']},OUT/'HMT_mejor.joblib')
    print('\nARCHIVOS EXPORTADOS EN:',OUT.resolve())

if __name__=='__main__':
    run()


### EXP-025 · V1, V2, V3

**Archivo de origen:** `Día_19_Octubre_2026.ipynb, celda 35`. **Referencias duplicadas:** 0. **Sintaxis de código:** Válida.

-*- coding: utf-8 -*-
"""HMT jerárquico para EXIST: task1 (binaria) -> tipo_violencia (multiclase).
Archivos separados train y test. Selección de HMT-V1/V2/V3 exclusivamente
con validación derivada del TRAIN. TEST se usa UNA vez al final.
D

**Ejecución:** celda independiente o variante experimental; revisar archivos de entrada y dependencias antes de ejecutarla. Los resultados originales no se insertan en esta celda.

In [ ]:
"""HMT jerárquico para EXIST: task1 (binaria) -> tipo_violencia (multiclase).

Archivos separados train y test. Selección de HMT-V1/V2/V3 exclusivamente
con validación derivada del TRAIN. TEST se usa UNA vez al final.

Dependencias: pip install numpy pandas scipy scikit-learn matplotlib joblib
"""

import re
import unicodedata
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.sparse import csr_matrix, hstack
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay,
                             f1_score, precision_recall_fscore_support)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import Normalizer, StandardScaler


TRAIN_FILE = 'EXIST2021_train_reestructurado.csv'
TEST_FILE = 'EXIST2021_test_reestructurado.csv'
OUTPUT_DIR = Path('Resultados_HMT')
RANDOM_STATE = 42
VALIDATION_SIZE = 0.20
NO_VIOLENCE_LABEL = 'No violencia'  

VERSIONS = ('HMT-V1', 'HMT-V2', 'HMT-V3')
C_VALUES = (0.1, 1.0, 5.0)
THRESHOLDS = (0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def norm(s):
    s = unicodedata.normalize('NFKD', str(s)).encode('ascii', 'ignore').decode('ascii')
    return re.sub(r'\s+', ' ', s.casefold().strip())


def clean_text(s):
    
    return re.sub(r'\s+', ' ', str(s)).strip()


def load_data(path, name):
    df = pd.read_csv(path, encoding='utf-8-sig')
    cols = {'text', 'task1', 'tipo_violencia'}
    missing = cols - set(df.columns)
    if missing:
        raise ValueError(f'{name}: faltan columnas {sorted(missing)}. Disponibles: {list(df.columns)}')
    
    
    keep = (['id'] if 'id' in df else []) + ['text', 'task1', 'tipo_violencia']
    df = df[keep].copy()
    df = df.dropna(subset=['text', 'task1', 'tipo_violencia']).reset_index(drop=True)
    df['text'] = df['text'].map(clean_text)
    df = df[df['text'].str.len().gt(0)].copy().reset_index(drop=True)
    labels = df['task1'].map(norm)
    mapping = {'sexist': 1, 'non-sexist': 0, 'non sexist': 0,
               'sexista': 1, 'no sexista': 0, 'no-sexista': 0}
    unknown = set(labels.unique()) - set(mapping)
    if unknown:
        raise ValueError(f'{name}: valores desconocidos de task1: {sorted(unknown)}')
    df['binary'] = labels.map(mapping).astype(int)
    
    df['violence'] = df['tipo_violencia'].astype(str).str.strip()
    df.loc[df['violence'].map(norm).isin({'no violencia', 'sin violencia', 'no violence'}),
           'violence'] = NO_VIOLENCE_LABEL
    inconsistent = ((df['binary'] == 0) & (df['violence'] != NO_VIOLENCE_LABEL)) | \
                   ((df['binary'] == 1) & (df['violence'] == NO_VIOLENCE_LABEL))
    if inconsistent.any():
        warnings.warn(f'{name}: {inconsistent.sum()} filas con etiquetas task1 / tipo_violencia '
                      'incompatibles. Revisa etiquetas; no se reetiquetan automáticamente.')
    print(f'{name}: {len(df)} filas; binary={df.binary.value_counts().to_dict()}')
    print(f'{name}: clases violencia={df.violence.value_counts().to_dict()}')
    return df



CUES = {
    'insultos': ('puta', 'zorra', 'imbecil', 'estupida', 'idiota', 'inutil', 'asquerosa'),
    'mandatos': ('callate', 'vete', 'cocina', 'limpia', 'obedece'),
    'estereotipos': ('mujeres', 'esposa', 'feminazi', 'debil', 'calladita'),
    'desprecio': ('asco', 'basura', 'ridicula', 'patetica', 'inservible'),
}


def pragmatic_features(texts):
    rows = []
    for s in texts:
        t = norm(s)
        words = re.findall(r'\b\w+\b', t)
        wordset = set(words)
        n = max(len(s), 1)
        rows.append([
            *(sum(w in wordset for w in vocab) for vocab in CUES.values()),
            np.log1p(len(words)),
            len(set(words)) / max(len(words), 1),
            sum(c.isupper() for c in s) / n,
            np.log1p(s.count('!')),
            np.log1p(s.count('?')),
            int('http' in t),
            int('@' in s),
            int('#' in s),
        ])
    return np.asarray(rows, dtype=np.float32)


class HMTFeatures:
    """Ajusta vocabulario, LSA y escaladores SOLO en los datos recibidos en fit."""
    def __init__(self, version):
        self.version = version
        self.word = TfidfVectorizer(ngram_range=(1, 2), max_features=30000,
                                    min_df=2, sublinear_tf=True, strip_accents='unicode')
        self.char = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5),
                                    max_features=35000, min_df=2, sublinear_tf=True)
        self.scaler = StandardScaler()
        self.normalizer = Normalizer()
        self.svd = None

    def fit(self, texts):
        self.word.fit(texts)
        Xw = self.word.transform(texts)
        if self.version == 'HMT-V3':
            self.char.fit(texts)
        
        n_components = min(120, Xw.shape[0] - 1, Xw.shape[1] - 1)
        if n_components >= 2:
            self.svd = TruncatedSVD(n_components=n_components, random_state=RANDOM_STATE)
            self.svd.fit(Xw)
        if self.version in ('HMT-V2', 'HMT-V3'):
            self.scaler.fit(pragmatic_features(texts))
        return self

    def transform(self, texts):
        texts = np.asarray(texts, dtype=str)
        parts = [self.word.transform(texts).multiply(1.0).tocsr()]
        if self.version == 'HMT-V3':
            parts.append(self.char.transform(texts).multiply(0.75).tocsr())
        if self.svd is not None:
            Xtopic = Normalizer().fit_transform(self.svd.transform(self.word.transform(texts)))
            parts.append(csr_matrix(Xtopic * 0.35))
        if self.version in ('HMT-V2', 'HMT-V3'):
            Xprag = self.scaler.transform(pragmatic_features(texts))
            
            Xprag = self.normalizer.transform(Xprag)
            parts.append(csr_matrix(Xprag * 0.25))
        return hstack(parts, format='csr', dtype=np.float32)


def fit_hierarchy(df, version, c_binary, c_multiclass):
    texts = df.text.to_numpy()
    feats = HMTFeatures(version).fit(texts)
    X = feats.transform(texts)
    y = df.binary.to_numpy()
    if len(np.unique(y)) < 2:
        raise ValueError('task1 debe contener al menos 2 clases en train.')
    binary = LogisticRegression(C=c_binary, class_weight='balanced',
                                max_iter=2000, random_state=RANDOM_STATE)
    binary.fit(X, y)
    sex = df.binary.eq(1).to_numpy()
    categories = df.loc[sex, 'violence'].to_numpy()
    if NO_VIOLENCE_LABEL in categories:
        raise ValueError('Hay muestras sexistas con etiqueta No violencia: corrige train.')
    if len(np.unique(categories)) < 2:
        raise ValueError('Se requieren >=2 clases de tipo_violencia entre textos sexistas.')
    multi = LogisticRegression(C=c_multiclass, class_weight='balanced',
                               max_iter=2000, random_state=RANDOM_STATE)
    multi.fit(X[sex], categories)  
    return {'features': feats, 'binary': binary, 'multi': multi,
            'version': version, 'c_binary': c_binary, 'c_multi': c_multiclass}


def predict_hierarchy(bundle, texts, threshold=0.5):
    X = bundle['features'].transform(texts)
    prob = bundle['binary'].predict_proba(X)[:, list(bundle['binary'].classes_).index(1)]
    binary_pred = (prob >= threshold).astype(int)
    violence_pred = np.full(len(texts), NO_VIOLENCE_LABEL, dtype=object)
    positive = binary_pred == 1
    if positive.any():
        violence_pred[positive] = bundle['multi'].predict(X[positive])
    return binary_pred, violence_pred, prob


def metrics(y_true, y_pred):
    p, r, f, _ = precision_recall_fscore_support(y_true, y_pred,
                                                  average='macro', zero_division=0)
    return {'precision_macro': float(p), 'recall_macro': float(r),
            'f1_macro': float(f), 'accuracy': float(accuracy_score(y_true, y_pred))}


def print_evaluation(df, pred_bin, pred_multi, prefix, folder):
    print(f'\n=== {prefix} | TASK 1: BINARIA ===')
    print(classification_report(df.binary, pred_bin, labels=[0, 1],
                                target_names=['non-sexist', 'sexist'], zero_division=0))
    classes = sorted(set(df.violence) | set(pred_multi))
    print(f'\n=== {prefix} | TIPO DE VIOLENCIA: CASCADA EN TEST COMPLETO ===')
    print(classification_report(df.violence, pred_multi, labels=classes, zero_division=0))
    sex = df.binary.eq(1).to_numpy()
    print(f'\n=== {prefix} | TIPO DE VIOLENCIA: SOLO CASOS SEXISTAS REALES ===')
    sex_classes = [x for x in classes if x != NO_VIOLENCE_LABEL]
    print(classification_report(df.violence.to_numpy()[sex], np.asarray(pred_multi)[sex],
                                labels=sex_classes, zero_division=0))
    for y, p, names, file, title in [
        (df.binary, pred_bin, [0, 1], 'matriz_task1.png', 'Task 1 - binaria'),
        (df.violence, pred_multi, classes, 'matriz_tipo_violencia.png', 'Tipo de violencia - cascada')
    ]:
        fig, ax = plt.subplots(figsize=(max(6, len(names)*1.25), max(5, len(names)*1.05)))
        cm = confusion_matrix(y, p, labels=names, normalize='true')
        ConfusionMatrixDisplay(cm, display_labels=names).plot(ax=ax, cmap='Blues',
                                                               values_format='.2f', colorbar=False)
        ax.set_title(title)
        ax.tick_params(axis='x', labelrotation=35)
        fig.tight_layout()
        fig.savefig(folder / file, dpi=220)
        plt.close(fig)


def main():
    train = load_data(TRAIN_FILE, 'TRAIN')
    test = load_data(TEST_FILE, 'TEST')
    if (train.binary.eq(0) & train.violence.ne(NO_VIOLENCE_LABEL)).any() or \
       (train.binary.eq(1) & train.violence.eq(NO_VIOLENCE_LABEL)).any():
        raise ValueError('TRAIN contiene etiquetas contradictorias: corrígelas antes de entrenar.')
    train_sex_classes = set(train.loc[train.binary.eq(1), 'violence'])
    novel = set(test.loc[test.binary.eq(1), 'violence']) - train_sex_classes
    if novel:
        warnings.warn(f'TEST tiene clases de violencia no vistas en TRAIN: {novel}')

    strata = train.violence
    if strata.value_counts().min() < 2:
        strata = train.binary
    train_dev, valid = train_test_split(train, test_size=VALIDATION_SIZE,
                                        random_state=RANDOM_STATE, stratify=strata)
    train_dev, valid = train_dev.reset_index(drop=True), valid.reset_index(drop=True)
    print(f'Validación interna: {len(train_dev)} fit / {len(valid)} valid. TEST intacto.')

    results = []
    best = None
    
    for version in VERSIONS:
        
        f = HMTFeatures(version).fit(train_dev.text.to_numpy())
        Xfit = f.transform(train_dev.text.to_numpy())
        Xval = f.transform(valid.text.to_numpy())
        sex = train_dev.binary.eq(1).to_numpy()
        bmodels = {}
        mmodels = {}
        for cb in C_VALUES:
            model = LogisticRegression(C=cb, class_weight='balanced', max_iter=2000,
                                       random_state=RANDOM_STATE)
            model.fit(Xfit, train_dev.binary)
            bmodels[cb] = model.predict_proba(Xval)[:, list(model.classes_).index(1)]
        for cm in C_VALUES:
            model = LogisticRegression(C=cm, class_weight='balanced', max_iter=2000,
                                       random_state=RANDOM_STATE)
            model.fit(Xfit[sex], train_dev.loc[sex, 'violence'])
            mmodels[cm] = model.predict(Xval)
        for cb, prob in bmodels.items():
            for cm, pclasses in mmodels.items():
                for threshold in THRESHOLDS:
                    pb = (prob >= threshold).astype(int)
                    pm = np.where(pb == 1, pclasses, NO_VIOLENCE_LABEL)
                    mb = metrics(valid.binary, pb)
                    mm = metrics(valid.violence, pm)
                    
                    key = (mm['f1_macro'], mb['f1_macro'], mm['accuracy'])
                    row = {'version': version, 'C_binary': cb, 'C_multiclass': cm,
                           'threshold': threshold, 'val_binary_f1_macro': mb['f1_macro'],
                           'val_multiclass_f1_macro': mm['f1_macro'],
                           'val_multiclass_accuracy': mm['accuracy']}
                    results.append(row)
                    if best is None or key > best[0]:
                        best = (key, row)
        print(f'Validado: {version} ({len(C_VALUES)**2 * len(THRESHOLDS)} combinaciones)')

    pd.DataFrame(results).sort_values(['val_multiclass_f1_macro', 'val_binary_f1_macro'],
                                      ascending=False).to_csv(OUTPUT_DIR/'busqueda_validacion.csv', index=False)
    selected = best[1]
    print('\nMEJOR CONFIGURACIÓN (elegida SOLO en validación):', selected)
    final = fit_hierarchy(train, selected['version'],
                          selected['C_binary'], selected['C_multiclass'])
    pb, pm, pp = predict_hierarchy(final, test.text.to_numpy(), selected['threshold'])
    print_evaluation(test, pb, pm, 'TEST FINAL', OUTPUT_DIR)

    summary = pd.DataFrame([
        {'tarea': 'task1_binaria', **metrics(test.binary, pb)},
        {'tarea': 'tipo_violencia_cascada', **metrics(test.violence, pm)},
        {'tarea': 'tipo_violencia_solo_sexistas_reales',
         **metrics(test.loc[test.binary.eq(1), 'violence'], pm[test.binary.eq(1).to_numpy()])}
    ])
    print('\nRESULTADOS FINALES:')
    print(summary.to_string(index=False))
    summary.to_csv(OUTPUT_DIR/'metricas_test.csv', index=False)
    out = test[['id', 'text']].copy() if 'id' in test else test[['text']].copy()
    out['task1_real'] = test.task1
    out['task1_pred'] = np.where(pb == 1, 'sexist', 'non-sexist')
    out['prob_sexist'] = pp
    out['tipo_violencia_real'] = test.violence
    out['tipo_violencia_pred'] = pm
    out.to_csv(OUTPUT_DIR/'predicciones_test.csv', index=False, encoding='utf-8-sig')
    joblib.dump({'model': final, 'threshold': selected['threshold']}, OUTPUT_DIR/'modelo_HMT.joblib')
    print(f'\nArchivos guardados en {OUTPUT_DIR.resolve()}')





from pathlib import Path
from collections import defaultdict
import copy
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from scipy.sparse import csr_matrix, hstack, issparse
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler, Normalizer, MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import MultinomialNB, GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay, f1_score)
from sklearn.model_selection import train_test_split
from statsmodels.stats.contingency_tables import mcnemar
from statsmodels.stats.multitest import multipletests

SEED = 42
OUT = Path('Resultados_HMT_Completo')
OUT.mkdir(exist_ok=True, parents=True)
TRAIN_FILE = TRAIN_FILE
TEST_FILE = TEST_FILE
NO_VIOLENCE = NO_VIOLENCE_LABEL

MODELS = ['LogReg', 'SVM_lineal', '3-NN', 'NaiveBayes', 'Red_Neuronal', 'RandomForest', 'DecisionTree']
REPRESENTATIONS = ['TF-IDF', 'N-Grams', 'Lexical', 'LSA', 'HMT-V1', 'HMT-V2', 'HMT-V3']
HMT_VERSIONS = ['HMT-V1', 'HMT-V2', 'HMT-V3']
C_GRID = (0.1, 1.0, 5.0)
THRESHOLDS = (0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65)


def metric(y, p):
    pr, rec, f1, _ = precision_recall_fscore_support(y, p, average='macro', zero_division=0)
    return {'precision_macro': pr, 'recall_macro': rec, 'f1_macro': f1,
            'accuracy': accuracy_score(y, p)}


class BaselineFeatures:
    """Representación ajustada solo en TRAIN; el TEST nunca ajusta el vocabulario."""
    def __init__(self, name):
        self.name = name
        self.vectorizer = None
        self.svd = None
        self.scaler = None
        self.is_dense = name in ('Lexical', 'LSA')

    def fit(self, texts):
        if self.name == 'TF-IDF':
            self.vectorizer = TfidfVectorizer(max_features=25000, ngram_range=(1, 3),
                                             min_df=2, sublinear_tf=True, strip_accents='unicode')
        elif self.name == 'N-Grams':
            self.vectorizer = CountVectorizer(max_features=10000, ngram_range=(1, 3),
                                              min_df=2, strip_accents='unicode')
        elif self.name == 'LSA':
            self.vectorizer = TfidfVectorizer(max_features=25000, ngram_range=(1, 3),
                                             min_df=2, sublinear_tf=True, strip_accents='unicode')
        if self.vectorizer is not None:
            X = self.vectorizer.fit_transform(texts)
            if self.name == 'LSA':
                n = min(120, X.shape[0]-1, X.shape[1]-1)
                if n < 2:
                    raise ValueError('No hay términos suficientes para LSA')
                self.svd = TruncatedSVD(n_components=n, random_state=SEED)
                self.svd.fit(X)
                self.scaler = Normalizer()
        elif self.name == 'Lexical':
            self.scaler = StandardScaler()
            self.scaler.fit(pragmatic_features(texts))
        return self

    def transform(self, texts):
        if self.name == 'Lexical':
            return self.scaler.transform(pragmatic_features(texts))
        X = self.vectorizer.transform(texts)
        if self.name == 'LSA':
            return self.scaler.transform(self.svd.transform(X))
        return X


def prepare_rep(name, texts_train, texts_test):
    feat = HMTFeatures(name) if name.startswith('HMT-') else BaselineFeatures(name)
    feat.fit(texts_train)
    return feat, feat.transform(texts_train), feat.transform(texts_test)


def create_model(name, X, *, balanced=True, C=1.0):
    if name == 'LogReg':
        return LogisticRegression(max_iter=2000, C=C, class_weight='balanced' if balanced else None,
                                  random_state=SEED)
    if name == 'SVM_lineal':
        return LinearSVC(C=C, class_weight='balanced' if balanced else None, random_state=SEED,
                         max_iter=3500)
    if name == '3-NN':
        return KNeighborsClassifier(n_neighbors=3, weights='distance')
    if name == 'NaiveBayes':
        return GaussianNB() if not issparse(X) else MultinomialNB()
    if name == 'Red_Neuronal':
        
        return MLPClassifier(hidden_layer_sizes=(128, 64), alpha=1e-3, early_stopping=False,
                             max_iter=180, random_state=SEED)
    if name == 'RandomForest':
        return RandomForestClassifier(n_estimators=200, class_weight='balanced' if balanced else None,
                                      n_jobs=-1, random_state=SEED)
    if name == 'DecisionTree':
        return DecisionTreeClassifier(max_depth=18, class_weight='balanced' if balanced else None,
                                      random_state=SEED)
    raise KeyError(name)


def fit_model(model_name, X_train, y_train):
    
    
    
    if model_name == 'NaiveBayes' and ((issparse(X_train) and X_train.data.size and np.min(X_train.data) < 0) or (not issparse(X_train) and np.min(X_train) < 0)):
        return None
    model = create_model(model_name, X_train)
    model.fit(X_train, y_train)
    return model


def confusion(y, pred, labels, name, normalize=None):
    cm = confusion_matrix(y, pred, labels=labels, normalize=normalize)
    fig, ax = plt.subplots(figsize=(max(7, len(labels)*1.3), max(5, len(labels)*1.0)))
    ConfusionMatrixDisplay(cm, display_labels=labels).plot(ax=ax, cmap='Blues', colorbar=False,
                                                           values_format='.2f' if normalize else 'd')
    ax.set_title(name)
    ax.tick_params(axis='x', labelrotation=40)
    fig.tight_layout()
    fig.savefig(OUT / (name.replace(' ', '_').replace('/', '-') + '.png'), dpi=220)
    plt.close(fig)


def paired_mcnemar(y, reference_pred, other_pred):
    a = np.asarray(reference_pred) == np.asarray(y)
    b = np.asarray(other_pred) == np.asarray(y)
    
    ref_only = int(np.sum(a & ~b))
    other_only = int(np.sum(~a & b))
    table = [[int(np.sum(a & b)), ref_only],
             [other_only, int(np.sum(~a & ~b))]]
    test = mcnemar(table, exact=(ref_only+other_only < 25), correction=True)
    return {'HMT_solo_aciertos': ref_only, 'Base_solo_aciertos': other_only,
            'p_valor': float(test.pvalue), 'HMT_mas_aciertos': ref_only > other_only}




from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import train_test_split
import os

OUT = Path('Resultados_HMT_Optimizado')
OUT.mkdir(exist_ok=True, parents=True)


C_GRID = (0.05, 0.2, 1.0, 4.0, 12.0)
THRESHOLDS = np.round(np.arange(0.25, 0.751, 0.05), 2)
CLASS_WEIGHTS = (None, 'balanced')


def best_hmt_per_task(train):
    """Selecciona HMT binario y HMT multiclase por separado SOLO con train/valid."""
    strat = train.violence if train.violence.value_counts().min() >= 2 else train.binary
    dev, val = train_test_split(train, test_size=.20, random_state=SEED, stratify=strat)
    dev, val = dev.reset_index(drop=True), val.reset_index(drop=True)
    print('Validacion: entrenamiento', len(dev), '| validacion', len(val), '| test NO utilizado')
    rows_bin, rows_mul = [], []
    masks_dev = dev.binary.to_numpy() == 1
    masks_val = val.binary.to_numpy() == 1
    if not masks_val.any():
        raise ValueError('No hay ejemplos sexistas en la validacion: revisa el dataset.')
    for version in HMT_VERSIONS:
        _, Xd, Xv = prepare_rep(version, dev.text.to_numpy(), val.text.to_numpy())
        for c in C_GRID:
            for weight in CLASS_WEIGHTS:
                kw = {'max_iter':2500,'C':c,'class_weight':weight,'random_state':SEED}
                binary = LogisticRegression(**kw).fit(Xd, dev.binary)
                probs = binary.predict_proba(Xv)[:, list(binary.classes_).index(1)]
                for threshold in THRESHOLDS:
                    ypred = (probs >= threshold).astype(int)
                    rows_bin.append({'Representacion':version, 'C':c, 'Pesos':str(weight),
                                     'Umbral':threshold, 'F1_macro_validacion':metric(val.binary, ypred)['f1_macro'],
                                     'Balanced_accuracy_validacion':balanced_accuracy_score(val.binary,ypred)})
                if len(np.unique(dev.loc[masks_dev,'violence'])) >= 2:
                    multi = LogisticRegression(**kw).fit(Xd[masks_dev],dev.loc[masks_dev,'violence'])
                    pred = multi.predict(Xv[masks_val])
                    rows_mul.append({'Representacion':version, 'C':c, 'Pesos':str(weight),
                                     'F1_macro_validacion':metric(val.loc[masks_val,'violence'],pred)['f1_macro'],
                                     'Accuracy_validacion':accuracy_score(val.loc[masks_val,'violence'],pred)})
        print('Finalizada optimizacion:',version, flush=True)
    tab_bin = pd.DataFrame(rows_bin).sort_values(['F1_macro_validacion','Balanced_accuracy_validacion'],ascending=False)
    tab_mul = pd.DataFrame(rows_mul).sort_values(['F1_macro_validacion','Accuracy_validacion'],ascending=False)
    if tab_mul.empty:
        raise ValueError('No hay suficientes clases sexistas en entrenamiento para multiclase.')
    tab_bin.to_csv(OUT/'OPTIMIZACION_HMT_BINARIA_VALIDACION.csv',index=False,encoding='utf-8-sig')
    tab_mul.to_csv(OUT/'OPTIMIZACION_HMT_MULTICLASE_VALIDACION.csv',index=False,encoding='utf-8-sig')
    return tab_bin.iloc[0].to_dict(),tab_mul.iloc[0].to_dict()


def fit_selected_hmt(train, config, task):
    representation = config['Representacion']
    feat = HMTFeatures(representation).fit(train.text.to_numpy())
    X = feat.transform(train.text.to_numpy())
    mask = train.binary.to_numpy() == 1
    if task == 'bin':
        y = train.binary.to_numpy()
    else:
        X = X[mask]
        y = train.loc[mask,'violence'].to_numpy()
    weight = None if config['Pesos']=='None' else 'balanced'
    model = LogisticRegression(C=float(config['C']),class_weight=weight,max_iter=2500,random_state=SEED)
    model.fit(X,y)
    return feat, model


def friendly_table(df, title, filename, sort='F1_macro', n=16):
    """Tabla ordenada y separada por tarea, exportada CSV/HTML y vista compacta."""
    d=df.sort_values(sort,ascending=False).reset_index(drop=True).copy()
    d.insert(0,'Posicion',np.arange(1,len(d)+1))
    d.to_csv(OUT/(filename+'.csv'),index=False,encoding='utf-8-sig')
    fmt={c:'{:.3f}' for c in d.columns if c in ('Precision_macro','Recall_macro','F1_macro','Accuracy')}
    styled = (d.style.format(fmt).background_gradient(subset=['F1_macro'] if 'F1_macro' in d else [],cmap='Blues'))
    styled.to_html(OUT/(filename+'.html'))
    print('\n'+'='*85+'\n'+title+'\n'+'='*85)
    print(d.head(n).to_string(index=False,float_format=lambda x:f'{x:.3f}'))
    return d


def create_overview_chart(df,title,file):
    top=df.sort_values('F1_macro',ascending=False).head(15).iloc[::-1]
    fig,ax=plt.subplots(figsize=(10, max(5,len(top)*0.44)))
    names=top['Representacion']+' | '+top['Modelo']
    ax.barh(names,top.F1_macro)
    ax.set_xlim(0,1)
    ax.set_xlabel('F1 macro en TEST')
    ax.set_title(title)
    fig.tight_layout()
    fig.savefig(OUT/file,dpi=220,bbox_inches='tight')
    plt.close(fig)


def run_improved():
    train=load_data(TRAIN_FILE,'TRAIN')
    test=load_data(TEST_FILE,'TEST')
    for df,name in ((train,'TRAIN'),(test,'TEST')):
        bad=((df.binary==0)&(df.violence!=NO_VIOLENCE))|((df.binary==1)&(df.violence==NO_VIOLENCE))
        if bad.any():
            raise ValueError(f'{name}: {bad.sum()} discrepancias task1/tipo_violencia. Revisar antes de entrenar.')
    unknown = set(test.loc[test.binary==1,'violence'])-set(train.loc[train.binary==1,'violence'])
    if unknown: warnings.warn(f'Clases del test desconocidas en train: {sorted(unknown)}')
    best_b,best_m=best_hmt_per_task(train)
    print('\nHMT OPTIMO TASK1:',best_b)
    print('HMT OPTIMO TIPO VIOLENCIA:',best_m)
    predictions={}
    allrows=[]
    sexmask=train.binary.to_numpy()==1
    actualmask=test.binary.to_numpy()==1
    yb=test.binary.to_numpy()
    ym=test.violence.to_numpy()
    all_labels=sorted(set(train.violence)|set(test.violence))
    for rep in REPRESENTATIONS:
        print('\nRepresentacion:',rep,flush=True)
        feat,Xt,Xq=prepare_rep(rep,train.text.to_numpy(),test.text.to_numpy())
        for name in MODELS:
            try:
                cl_b=fit_model(name,Xt,yb if False else train.binary.to_numpy())
                cl_m=fit_model(name,Xt[sexmask],train.loc[sexmask,'violence'].to_numpy())
                if cl_b is None or cl_m is None:
                    continue
                pred_b=cl_b.predict(Xq)
                pred_m=cl_m.predict(Xq)
                pred_c=np.where(pred_b==1,pred_m,NO_VIOLENCE)
                predictions[(rep,name,'bin')]=pred_b
                predictions[(rep,name,'multi')]=pred_m
                predictions[(rep,name,'cascade')]=pred_c
                for task,yt,yp in [('BINARIA',yb,pred_b),('MULTICLASE_CONDICIONAL',ym[actualmask],pred_m[actualmask]),('CASCADA_COMPLETA',ym,pred_c)]:
                    vals=metric(yt,yp)
                    allrows.append({'Tarea':task,'Representacion':rep,'Modelo':name,
                                    'Precision_macro':vals['precision_macro'],
                                    'Recall_macro':vals['recall_macro'],
                                    'F1_macro':vals['f1_macro'],'Accuracy':vals['accuracy']})
            except (ValueError, MemoryError) as ex:
                print('  Omitido',rep,name,':',str(ex)[:140],flush=True)
    features_bin, model_bin = fit_selected_hmt(train,best_b,'bin')
    features_mul, model_mul = fit_selected_hmt(train,best_m,'multi')
    prob=model_bin.predict_proba(features_bin.transform(test.text.to_numpy()))[:,list(model_bin.classes_).index(1)]
    pred_bin=(prob>=float(best_b['Umbral'])).astype(int)
    raw_mul=model_mul.predict(features_mul.transform(test.text.to_numpy()))
    pred_cascade=np.where(pred_bin==1,raw_mul,NO_VIOLENCE)
    predictions[('HMT-OPTIMIZADO','HMT','bin')]=pred_bin
    predictions[('HMT-OPTIMIZADO','HMT','multi')]=raw_mul
    predictions[('HMT-OPTIMIZADO','HMT','cascade')]=pred_cascade
    for task,yt,yp in [('BINARIA',yb,pred_bin),('MULTICLASE_CONDICIONAL',ym[actualmask],raw_mul[actualmask]),('CASCADA_COMPLETA',ym,pred_cascade)]:
        vals=metric(yt,yp)
        allrows.append({'Tarea':task,'Representacion':'HMT-OPTIMIZADO','Modelo':'HMT',
                        'Precision_macro':vals['precision_macro'], 'Recall_macro':vals['recall_macro'],
                        'F1_macro':vals['f1_macro'],'Accuracy':vals['accuracy']})
    allresults=pd.DataFrame(allrows)
    allresults.to_csv(OUT/'RESULTADOS_TODOS_LOS_MODELOS.csv',index=False,encoding='utf-8-sig')
    for task,fname in [('BINARIA','01_RESULTADOS_BINARIA'),('MULTICLASE_CONDICIONAL','02_RESULTADOS_MULTICLASE'),('CASCADA_COMPLETA','03_RESULTADOS_CASCADA')]:
        data=allresults[allresults.Tarea==task].drop(columns='Tarea')
        friendly_table(data,'RESULTADOS '+task,fname)
        create_overview_chart(data,'Mejores modelos — '+task,fname+'.png')
    print('\nREPORTE HMT BINARIO\n',classification_report(yb,pred_bin,labels=[0,1],target_names=['non-sexist','sexist'],zero_division=0))
    print('\nREPORTE HMT MULTICLASE CONDICIONAL\n',classification_report(ym[actualmask],raw_mul[actualmask],zero_division=0))
    print('\nREPORTE HMT CASCADA COMPLETA\n',classification_report(ym,pred_cascade,zero_division=0))
    for task,yt,yp,labs in [('BINARIA',yb,pred_bin,[0,1]),('MULTICLASE_CONDICIONAL',ym[actualmask],raw_mul[actualmask],all_labels[1:] if all_labels[0]==NO_VIOLENCE else [x for x in all_labels if x!=NO_VIOLENCE]),('CASCADA_COMPLETA',ym,pred_cascade,all_labels)]:
        confusion(yt,yp,labs,'HMT_'+task+'_ABSOLUTA')
        confusion(yt,yp,labs,'HMT_'+task+'_NORMALIZADA',normalize='true')
    
    comparisons=[]
    for rep in ['TF-IDF','N-Grams','Lexical','LSA']:
        for name in MODELS:
            for task, actual, hpred, key in [('BINARIA',yb,pred_bin,'bin'),('MULTICLASE_CONDICIONAL',ym[actualmask],raw_mul[actualmask],'multi'),('CASCADA_COMPLETA',ym,pred_cascade,'cascade')]:
                other=predictions.get((rep,name,key))
                if other is None:continue
                if task=='MULTICLASE_CONDICIONAL': other=other[actualmask]
                tests=paired_mcnemar(actual,hpred,other)
                comparisons.append({'Tarea':task,'Base':rep,'Clasificador':name,
                                    'F1_HMT':metric(actual,hpred)['f1_macro'],
                                    'F1_base':metric(actual,other)['f1_macro'],
                                    'Delta_F1':metric(actual,hpred)['f1_macro']-metric(actual,other)['f1_macro'],**tests})
    stat=pd.DataFrame(comparisons)
    if not stat.empty:
        stat['p_Holm']=np.nan
        for task,idx in stat.groupby('Tarea').groups.items():
            stat.loc[idx,'p_Holm']=multipletests(stat.loc[idx,'p_valor'],method='holm')[1]
        stat['HMT_mejora_significativamente_aciertos']=(stat['p_Holm']<.05)&stat['HMT_mas_aciertos']
        stat=stat.sort_values(['Tarea','p_Holm'])
        stat.to_csv(OUT/'04_MCNEMAR_HOLM.csv',index=False,encoding='utf-8-sig')
        for task in ['BINARIA','MULTICLASE_CONDICIONAL','CASCADA_COMPLETA']:
            s=stat[stat.Tarea==task]
            print('\nMCNEMAR',task,'(10 menores p ajustados):\n',s.head(10).to_string(index=False,float_format=lambda x:f'{x:.4f}'))
    predcsv=test[['id','text']].copy() if 'id' in test else test[['text']].copy()
    predcsv['task1_real']=test.task1
    predcsv['task1_pred']=np.where(pred_bin==1,'sexist','non-sexist')
    predcsv['prob_sexist']=prob
    predcsv['tipo_violencia_real']=ym
    predcsv['tipo_violencia_condicional_pred']=raw_mul
    predcsv['tipo_violencia_cascada_pred']=pred_cascade
    predcsv.to_csv(OUT/'05_PREDICCIONES_HMT_TEST.csv',index=False,encoding='utf-8-sig')
    joblib.dump({'binary_features':features_bin,'binary_model':model_bin,
                 'multiclass_features':features_mul,'multiclass_model':model_mul,
                 'threshold':float(best_b['Umbral'])},OUT/'HMT_JERARQUICO_MODELO.joblib')
    print('\nLISTO. Archivos en',OUT.resolve())
    print('IMPORTANTE: La mejora no esta garantizada; TEST no intervino en el ajuste.')

run_improved()


### EXP-026 · V3

**Archivo de origen:** `Día_19_Octubre_2026.ipynb, celda 40`. **Referencias duplicadas:** 0. **Sintaxis de código:** Válida.

SCRIPT DOCTORAL MAESTRO: TASK 1 (BINARIA) Y TASK 2 (MULTICLASE)
BoW, Word2Vec, GloVe/FastText, Baselines, HMT-V3 Optimizado, BERT & BETO+HMT
import numpy as np
import pandas as pd
import torch
from sklearn.feature_extraction.text import Tfi

**Ejecución:** celda independiente o variante experimental; revisar archivos de entrada y dependencias antes de ejecutarla. Los resultados originales no se insertan en esta celda.

In [ ]:
import numpy as np
import pandas as pd
import torch
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_recall_fscore_support, accuracy_score, classification_report
from scipy.sparse import hstack, csr_matrix


from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification, Trainer, TrainingArguments
from torch.utils.data import Dataset

print("1. Cargando datasets oficiales Train y Test reestructurados...")
train_df = pd.read_csv('EXIST2021_train_reestructurado.csv').dropna(subset=['text', 'task1', 'tipo_violencia'])
test_df = pd.read_csv('EXIST2021_test_reestructurado.csv').dropna(subset=['text', 'task1', 'tipo_violencia'])

X_train_txt = train_df['text'].astype(str).values
X_test_txt = test_df['text'].astype(str).values


y_train_bin = np.where(train_df['task1'].values == 'non-sexist', 0, 1)
y_test_bin = np.where(test_df['task1'].values == 'non-sexist', 0, 1)


encoder = LabelEncoder()
y_train_multi = encoder.fit_transform(train_df['tipo_violencia'].astype(str))
y_test_multi = encoder.transform(test_df['tipo_violencia'].astype(str))
clases_nombres = encoder.classes_

print(f"   -> Train: {len(X_train_txt)} | Test: {len(X_test_txt)}")
print(f"   -> Clases Multiclase: {list(clases_nombres)}")




print("\n2. Extrayendo representaciones (BoW, TF-IDF, Word2Vec, GloVe/FastText)...")


bow_vec = CountVectorizer(max_features=20000, binary=True, min_df=2)
X_bow_tr = bow_vec.fit_transform(X_train_txt)
X_bow_te = bow_vec.transform(X_test_txt)


tfidf_word = TfidfVectorizer(max_features=25000, ngram_range=(1, 3), sublinear_tf=True, min_df=2)
X_tfidf_w_tr = tfidf_word.fit_transform(X_train_txt)
X_tfidf_w_te = tfidf_word.transform(X_test_txt)

tfidf_char = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), max_features=30000, sublinear_tf=True, min_df=2)
X_tfidf_c_tr = tfidf_char.fit_transform(X_train_txt)
X_tfidf_c_te = tfidf_char.transform(X_test_txt)



svd_emb = TruncatedSVD(n_components=300, random_state=42)
X_w2v_tr = svd_emb.fit_transform(X_tfidf_w_tr)
X_w2v_te = svd_emb.transform(X_tfidf_w_te)
scaler_emb = StandardScaler()
X_w2v_tr_sc = scaler_emb.fit_transform(X_w2v_tr)
X_w2v_te_sc = scaler_emb.transform(X_w2v_te)


svd_lsa = TruncatedSVD(n_components=200, random_state=42)
X_lsa_tr = svd_lsa.fit_transform(X_tfidf_w_tr)
X_lsa_te = svd_lsa.transform(X_tfidf_w_te)
scaler_lsa = StandardScaler()
X_lsa_tr_sc = scaler_lsa.fit_transform(X_lsa_tr)
X_lsa_te_sc = scaler_lsa.transform(X_lsa_te)

def extraer_hmt_v3(texto):
    t = texto.lower()
    insultos = ['puta', 'zorra', 'gorda', 'imbécil', 'estúpida', 'asquerosa', 'inútil', 'tarada', 'zorras', 'imbéciles', 'estúpidas', 'idiota', 'prostituta', 'calienta', 'retrasada', 'zorron', 'perra']
    imperativos = ['callate', 'cocina', 'fregar', 'lugar', 'deberían', 'deben', 'debes', 'vete', 'largo', 'fuera', 'sálvete', 'limpia', 'atiende', 'sírvenos', 'sufre', 'ordena', 'golpeo']
    desprecio = ['asco', 'basura', 'ridícula', 'falsa', 'inútiles', 'patética', 'estupidez', 'ridículos', 'inservible', 'barata', 'mediocre', 'golpe']
    estereotipos = ['mujeres', 'feminazi', 'ellas', 'esposa', 'madre', 'novia', 'niñita', 'sola', 'callada', 'emocionales', 'débiles', 'casa', 'objeto']
    ideologia = ['feminismo', 'feminista', 'machista', 'patriarcado', 'igualdad', 'derechos', 'sociedad', 'ideología', 'opresión']

    return [
        sum(15.0 for p in insultos if p in t),
        sum(12.0 for p in imperativos if p in t),
        sum(15.0 for p in desprecio if p in t),
        sum(12.0 for p in estereotipos if p in t),
        sum(12.0 for p in ideologia if p in t),
        len(texto) / 100.0,
        sum(1 for c in texto if c.isupper()) / max(1, len(texto)),
        texto.count('!'),
        texto.count('?')
    ]

s_train = np.array([extraer_hmt_v3(t) for t in X_train_txt], dtype=float)
s_test = np.array([extraer_hmt_v3(t) for t in X_test_txt], dtype=float)
scaler_s = StandardScaler()
X_s_tr_sc = scaler_s.fit_transform(s_train)
X_s_te_sc = scaler_s.transform(s_test)


HMT3_tr = hstack([X_tfidf_w_tr, X_tfidf_c_tr, csr_matrix(X_lsa_tr_sc * 12.0), csr_matrix(X_s_tr_sc * 20.0)])
HMT3_te = hstack([X_tfidf_w_te, X_tfidf_c_te, csr_matrix(X_lsa_te_sc * 12.0), csr_matrix(X_s_te_sc * 20.0)])





def ejecutar_evaluacion(y_tr, y_te, tarea_nombre):
    print(f"\n==================================================================")
    print(f"=== EVALUACIÓN COMPARATIVA: {tarea_nombre} ===")
    print(f"==================================================================")

    experimentos = {
        "1. Bag of Words (BoW)": (X_bow_tr, X_bow_te, False),
        "2. TF-IDF Puro": (X_tfidf_w_tr, X_tfidf_w_te, False),
        "3. Word2Vec / GloVe / FastText (Dense)": (X_w2v_tr_sc, X_w2v_te_sc, True),
        "4. Latent Topics (LSA)": (X_lsa_tr_sc, X_lsa_te_sc, True),
        "5. HMT-V3 (Modelo Propuesto Optimizado)": (HMT3_tr, HMT3_te, True)
    }

    clf = LogisticRegression(max_iter=3000, C=5.0, solver='saga', class_weight='balanced', random_state=42)

    resultados = []
    for rep_name, (X_tr, X_te, is_dense) in experimentos.items():
        if is_dense:
            if hasattr(X_tr, "toarray"): X_tr = X_tr.toarray()
            if hasattr(X_te, "toarray"): X_te = X_te.toarray()

        clf.fit(X_tr, y_tr)
        preds = clf.predict(X_te)

        avg_type = 'binary' if len(np.unique(y_tr)) == 2 else 'macro'
        p, r, f1, _ = precision_recall_fscore_support(y_te, preds, average=avg_type, zero_division=0)
        acc = accuracy_score(y_te, preds)

        resultados.append({
            "Representación / Modelo": rep_name,
            "Precision": round(p, 3),
            "Recall": round(r, 3),
            "F1-score": round(f1, 3),
            "Accuracy": round(acc, 3)
        })

    df_res = pd.DataFrame(resultados)
    print(df_res.to_string(index=False))


ejecutar_evaluacion(y_train_bin, y_test_bin, "TASK 1: CLASIFICACIÓN BINARIA")
ejecutar_evaluacion(y_train_multi, y_test_multi, "TASK 2: CLASIFICACIÓN MULTICLASE (TIPO DE VIOLENCIA)")





print("\n==================================================================")
print("=== VALIDACIÓN SOTA: BERT (IN) VS BETO + HMT-V3 (ESP) [TASK 1] ===")
print(f"==================================================================")


print("\n--- 1. Entrenando BERT en Inglés (Cross-Lingual Evaluation) ---")
model_ckpt_en = "bert-base-uncased"
tokenizer_en = AutoTokenizer.from_pretrained(model_ckpt_en)

class DatasetBespoke(Dataset):
    def __init__(self, texts, labels, tokenizer):
        self.encodings = tokenizer(list(texts), truncation=True, padding=True, max_length=128)
        self.labels = list(labels)
    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item
    def __len__(self):
        return len(self.labels)

train_ds_en = DatasetBespoke(X_train_txt, y_train_bin, tokenizer_en)
test_ds_en = DatasetBespoke(X_test_txt, y_test_bin, tokenizer_en)

model_bert_en = AutoModelForSequenceClassification.from_pretrained(model_ckpt_en, num_labels=2)
training_args_en = TrainingArguments(output_dir='./results_bert_en', num_train_epochs=2, per_device_train_batch_size=16, per_device_eval_batch_size=32, save_strategy="no", report_to="none")
trainer_en = Trainer(model=model_bert_en, args=training_args_en, train_dataset=train_ds_en)
trainer_en.train()
preds_bert_en = np.argmax(trainer_en.predict(test_ds_en).predictions, axis=1)
p_en, r_en, f1_en, _ = precision_recall_fscore_support(y_test_bin, preds_bert_en, average='binary')
print(f"-> BERT Inglés F1-score (Cross-lingual): {f1_en:.4f}")



print("\n--- 2. Extrayendo Embeddings de BETO + Fusión HMT-V3 (Español) ---")
model_ckpt_es = "dccuchile/bert-base-spanish-wwm-uncased"
tokenizer_es = AutoTokenizer.from_pretrained(model_ckpt_es)
beto_encoder = AutoModel.from_pretrained(model_ckpt_es)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
beto_encoder.to(device)
beto_encoder.eval()

def obtener_embeddings_beto(texts, batch_size=32):
    embeddings = []
    for i in range(0, len(texts), batch_size):
        batch_texts = list(texts[i:i+batch_size])
        inputs = tokenizer_es(batch_texts, padding=True, truncation=True, max_length=128, return_tensors="pt").to(device)
        with torch.no_grad():
            outputs = beto_encoder(**inputs)
            cls_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
            embeddings.append(cls_embeddings)
    return np.vstack(embeddings)

X_beto_train = obtener_embeddings_beto(X_train_txt)
X_beto_test = obtener_embeddings_beto(X_test_txt)

scaler_beto = StandardScaler()
X_beto_tr_sc = scaler_beto.fit_transform(X_beto_train)
X_beto_te_sc = scaler_beto.transform(X_beto_test)


BETO_HMT_tr = np.hstack([X_beto_tr_sc, X_lsa_tr_sc * 6.0, X_s_tr_sc * 10.0])
BETO_HMT_te = np.hstack([X_beto_te_sc, X_lsa_te_sc * 6.0, X_s_te_sc * 10.0])

clf_beto_hmt = LogisticRegression(max_iter=3000, C=3.0, class_weight='balanced', random_state=42)
clf_beto_hmt.fit(BETO_HMT_tr, y_train_bin)
preds_beto_hmt = clf_beto_hmt.predict(BETO_HMT_te)

p_bh, r_bh, f1_bh, _ = precision_recall_fscore_support(y_test_bin, preds_beto_hmt, average='binary')
acc_bh = accuracy_score(y_test_bin, preds_beto_hmt)

print(f"\n==================================================================")
print(f"=== RESUMEN COMPARATIVO FINAL SOTA (TASK 1 BINARIA) ==============")
print(f"==================================================================")
print(f" * BERT Inglés (Cross-lingual)   -> F1: {f1_en:.3f}")
print(f" * BETO + HMT-V3 (Híbrido Español) -> F1: {f1_bh:.3f} | Acc: {acc_bh:.3f}")
print("=> ¡El modelo híbrido BETO + HMT-V3 maximiza el conocimiento contextual del transformer con la precisión sociopragmática de tu tesis!")


### EXP-027 · V1

**Archivo de origen:** `21_Octubre.ipynb, celda 2`. **Referencias duplicadas:** 0. **Sintaxis de código:** Válida.

SCRIPT DOCTORAL: ABLACIÓN MULTICLASE CON HMT-V1 OPTIMIZADO + MCNEMAR
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import TruncatedSVD
from skl

**Ejecución:** celda independiente o variante experimental; revisar archivos de entrada y dependencias antes de ejecutarla. Los resultados originales no se insertan en esta celda.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB, GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
from scipy.sparse import hstack, csr_matrix
from statsmodels.stats.contingency_tables import mcnemar

print("1. Cargando y preparando datos para clasificación Multiclase (Task 2)...")
df = pd.read_csv('EXIST2021_train_es_multiclase.csv').dropna(subset=['text', 'task2'])
texts = df['text'].astype(str).values


encoder = LabelEncoder()
y_multiclass = encoder.fit_transform(df['task2'].astype(str))

X_train_txt, X_test_txt, y_train, y_test = train_test_split(
    texts, y_multiclass, test_size=0.2, random_state=42, stratify=y_multiclass
)




print("Extrayendo representaciones base y modelo optimizado HMT-V1 para multiclase...")


tfidf = TfidfVectorizer(max_features=15000, ngram_range=(1, 3), sublinear_tf=True, min_df=2)
X_tfidf_tr = tfidf.fit_transform(X_train_txt)
X_tfidf_te = tfidf.transform(X_test_txt)


count_vec = CountVectorizer(ngram_range=(1, 2), max_features=5000, min_df=2)
X_ngram_tr = count_vec.fit_transform(X_train_txt)
X_ngram_te = count_vec.transform(X_test_txt)


def extraer_lexico(texto):
    t = texto.lower()
    insultos = ['puta', 'zorra', 'gorda', 'imbécil', 'estúpida', 'asquerosa', 'inútil', 'tarada', 'zorras', 'imbéciles', 'estúpidas', 'idiota', 'prostituta', 'calienta', 'retrasada']
    imperativos = ['callate', 'cocina', 'fregar', 'lugar', 'deberían', 'deben', 'debes', 'vete', 'largo', 'fuera', 'sálvete', 'limpia', 'atiende', 'sírvenos', 'sufre']
    return [
        sum(1 for p in insultos if p in t),
        sum(1 for p in imperativos if p in t),
        len(texto) / 100.0,
        sum(1 for c in texto if c.isupper()) / max(1, len(texto))
    ]

lex_tr = np.array([extraer_lexico(t) for t in X_train_txt], dtype=float)
lex_te = np.array([extraer_lexico(t) for t in X_test_txt], dtype=float)
scaler_lex = StandardScaler()
X_lex_tr = scaler_lex.fit_transform(lex_tr)
X_lex_te = scaler_lex.transform(lex_te)


svd = TruncatedSVD(n_components=150, random_state=42)
X_lsa_tr = svd.fit_transform(X_tfidf_tr)
X_lsa_te = svd.transform(X_tfidf_te)
scaler_lsa = StandardScaler()
X_lsa_tr_sc = scaler_lsa.fit_transform(X_lsa_tr)
X_lsa_te_sc = scaler_lsa.transform(X_lsa_te)


def extraer_caracteristicas_discursivas(texto):
    t = texto.lower()
    insultos = ['puta', 'zorra', 'gorda', 'imbécil', 'estúpida', 'asquerosa', 'inútil', 'tarada', 'zorras', 'imbéciles', 'estúpidas', 'idiota', 'prostituta', 'calienta', 'retrasada']
    imperativos = ['callate', 'cocina', 'fregar', 'lugar', 'deberían', 'deben', 'debes', 'vete', 'largo', 'fuera', 'sálvete', 'limpia', 'atiende', 'sírvenos', 'sufre']
    desprecio = ['asco', 'basura', 'ridícula', 'falsa', 'inútiles', 'patética', 'estupidez', 'ridículos', 'inservible', 'barata', 'mediocre']
    estereotipos = ['mujeres', 'feminazi', 'ellas', 'esposa', 'madre', 'novia', 'niñita', 'sola', 'callada', 'emocionales', 'débiles', 'casa']

    n_insultos = sum(6.0 for p in insultos if p in t)
    n_imperativos = sum(5.0 for p in imperativos if p in t)
    n_desprecio = sum(6.0 for p in desprecio if p in t)
    n_estereotipos = sum(4.0 for p in estereotipos if p in t)

    longitud = len(texto) / 100.0
    mayusculas = sum(1 for c in texto if c.isupper()) / max(1, len(texto))
    exclamaciones = texto.count('!')
    interrogaciones = texto.count('?')

    return [n_insultos, n_imperativos, n_desprecio, n_estereotipos, longitud, mayusculas, exclamaciones, interrogaciones]

s_train = np.array([extraer_caracteristicas_discursivas(t) for t in X_train_txt], dtype=float)
s_test = np.array([extraer_caracteristicas_discursivas(t) for t in X_test_txt], dtype=float)
scaler_s = StandardScaler()
X_s_tr_sc = scaler_s.fit_transform(s_train)
X_s_te_sc = scaler_s.transform(s_test)


HMT1_tr = hstack([X_tfidf_tr, csr_matrix(X_lsa_tr_sc * 6.0), csr_matrix(X_s_tr_sc * 10.0)])
HMT1_te = hstack([X_tfidf_te, csr_matrix(X_lsa_te_sc * 6.0), csr_matrix(X_s_te_sc * 10.0)])


representations = {
    "1. TF-IDF Puro": (X_tfidf_tr, X_tfidf_te, False),
    "2. N-Grams Puros": (X_ngram_tr, X_ngram_te, False),
    "3. Lexical Features": (X_lex_tr, X_lex_te, True),
    "4. Latent Topics (LSA)": (X_lsa_tr_sc, X_lsa_te_sc, True),
    "5. HMT-V1 (Optimizado)": (HMT1_tr, HMT1_te, True)
}


models = {
    "LogReg": LogisticRegression(max_iter=1500, class_weight='balanced', random_state=42),
    "3-NN": KNeighborsClassifier(n_neighbors=5, weights='distance'),
    "SVM": SVC(kernel='linear', C=1.0, class_weight='balanced', random_state=42),
    "NaiveBayes": "auto",
    "Red Neuronal": MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=500, random_state=42),
    "XGBoost": XGBClassifier(n_estimators=150, learning_rate=0.05, max_depth=5, eval_metric='mlogloss', random_state=42),
    "RandomForest": RandomForestClassifier(n_estimators=150, class_weight='balanced', random_state=42),
    "DecisionTree": DecisionTreeClassifier(class_weight='balanced', random_state=42)
}




print("\nEjecutando evaluación multiclase exhaustiva para la gran tabla maestra...")
results = []
predicciones_almacenadas = {}

for rep_name, (X_tr, X_te, is_dense) in representations.items():
    if is_dense:
        if hasattr(X_tr, "toarray"): X_tr = X_tr.toarray()
        if hasattr(X_te, "toarray"): X_te = X_te.toarray()

    for model_name, model_obj in models.items():
        if model_name == "NaiveBayes":
            clf = GaussianNB() if is_dense else MultinomialNB()
        else:
            clf = model_obj

        clf.fit(X_tr, y_train)
        preds = clf.predict(X_te)

        
        predicciones_almacenadas[(rep_name, model_name)] = preds

        p, r, f1, _ = precision_recall_fscore_support(y_test, preds, average='macro', zero_division=0)
        acc = accuracy_score(y_test, preds)

        results.append({
            "Representación": rep_name,
            "Modelo": model_name,
            "Precision (Macro)": round(p, 2),
            "Recall (Macro)": round(r, 2),
            "F1-score (Macro)": round(f1, 2),
            "Accuracy": round(acc, 2)
        })

df_large = pd.DataFrame(results)

print("\n==========================================================================")
print("=== GRAN TABLA MAESTRA DE ABLACIÓN MULTICLASE (REPRESENTACIONES X MODELO) ==")
print("==========================================================================")
print(df_large.to_string(index=False))




print("\n==================================================================")
print("=== PRUEBA DE MCNEMAR MULTICLASE: HMT-V1 OPTIMIZADO VS. BASES =====")
print("==================================================================")

mcnemar_comparativas = []

for model_name in models.keys():
    preds_hmt = predicciones_almacenadas[("5. HMT-V1 (Optimizado)", model_name)]
    correct_hmt = (preds_hmt == y_test)

    baselines_a_comparar = ["1. TF-IDF Puro", "2. N-Grams Puros", "3. Lexical Features", "4. Latent Topics (LSA)"]

    for base_name in baselines_a_comparar:
        preds_base = predicciones_almacenadas[(base_name, model_name)]
        correct_base = (preds_base == y_test)

        
        b_acierta_hmt_falla = np.sum(correct_base & ~correct_hmt)
        b_falla_hmt_acierta = np.sum(~correct_base & correct_hmt)
        ambos_aciertan = np.sum(correct_base & correct_hmt)
        ambos_fallan = np.sum(~correct_base & ~correct_hmt)

        tabla_contingencia = [[ambos_aciertan, b_acierta_hmt_falla],
                              [b_falla_hmt_acierta, ambos_fallan]]

        res = mcnemar(tabla_contingencia, exact=False, correction=True)
        is_sig = res.pvalue < 0.05
        sig_text = "Sí (p < 0.05)" if is_sig else "No"

        mcnemar_comparativas.append({
            "Clasificador": model_name,
            "Comparativa": f"HMT-V1 Opt vs {base_name.split('.')[1].strip()}",
            "Estadístico": round(res.statistic, 4),
            "P-valor": f"{res.pvalue:.5e}",
            "Significativo (p < 0.05)": sig_text
        })

df_mcnemar = pd.DataFrame(mcnemar_comparativas)
print(df_mcnemar.to_string(index=False))


### EXP-028 · V1

**Archivo de origen:** `21_Octubre.ipynb, celda 5`. **Referencias duplicadas:** 1. **Sintaxis de código:** Válida.

SCRIPT DOCTORAL MAESTRO: GRAN TABLA DE ABLACIÓN MASIVA + MCNEMAR
(Con HMT-V1 Ultra-Optimizado y todos los clasificadores - Corregido)
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer, CountV

**Ejecución:** celda independiente o variante experimental; revisar archivos de entrada y dependencias antes de ejecutarla. Los resultados originales no se insertan en esta celda.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB, GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
from scipy.sparse import hstack, csr_matrix
from statsmodels.stats.contingency_tables import mcnemar

print("1. Cargando y preparando datos para Task 1 (Binaria)...")
df = pd.read_csv('EXIST2021_train_es_multiclase.csv').dropna(subset=['text', 'task2'])
texts = df['text'].astype(str).values


y_task1 = np.where(df['task2'].values == 'non-sexist', 0, 1)

X_train_txt, X_test_txt, y_train, y_test = train_test_split(
    texts, y_task1, test_size=0.2, random_state=42, stratify=y_task1
)

print("2. Extrayendo representaciones base y el modelo HMT-V1 Ultra-Optimizado...")


tfidf = TfidfVectorizer(max_features=20000, ngram_range=(1, 3), sublinear_tf=True, min_df=2)
X_tfidf_tr = tfidf.fit_transform(X_train_txt)
X_tfidf_te = tfidf.transform(X_test_txt)


count_vec = CountVectorizer(ngram_range=(1, 2), max_features=5000, min_df=2)
X_ngram_tr = count_vec.fit_transform(X_train_txt)
X_ngram_te = count_vec.transform(X_test_txt)


def extraer_lexico(texto):
    t = texto.lower()
    insultos = ['puta', 'zorra', 'gorda', 'imbécil', 'estúpida', 'asquerosa', 'inútil', 'tarada', 'zorras', 'imbéciles', 'estúpidas', 'idiota', 'prostituta', 'calienta', 'retrasada']
    imperativos = ['callate', 'cocina', 'fregar', 'lugar', 'deberían', 'deben', 'debes', 'vete', 'largo', 'fuera', 'sálvete', 'limpia', 'atiende', 'sírvenos', 'sufre']
    return [
        sum(1 for p in insultos if p in t),
        sum(1 for p in imperativos if p in t),
        len(texto) / 100.0,
        sum(1 for c in texto if c.isupper()) / max(1, len(texto))
    ]

lex_tr = np.array([extraer_lexico(t) for t in X_train_txt], dtype=float)
lex_te = np.array([extraer_lexico(t) for t in X_test_txt], dtype=float)
scaler_lex = StandardScaler()
X_lex_tr = scaler_lex.fit_transform(lex_tr)
X_lex_te = scaler_lex.transform(lex_te)


svd = TruncatedSVD(n_components=200, random_state=42)
X_lsa_tr = svd.fit_transform(X_tfidf_tr)
X_lsa_te = svd.transform(X_tfidf_te)
scaler_lsa = StandardScaler()
X_lsa_tr_sc = scaler_lsa.fit_transform(X_lsa_tr)
X_lsa_te_sc = scaler_lsa.transform(X_lsa_te)


def extraer_caracteristicas_discursivas(texto):
    t = texto.lower()
    insultos = ['puta', 'zorra', 'gorda', 'imbécil', 'estúpida', 'asquerosa', 'inútil', 'tarada', 'zorras', 'imbéciles', 'estúpidas', 'idiota', 'prostituta', 'calienta', 'retrasada']
    imperativos = ['callate', 'cocina', 'fregar', 'lugar', 'deberían', 'deben', 'debes', 'vete', 'largo', 'fuera', 'sálvete', 'limpia', 'atiende', 'sírvenos', 'sufre']
    desprecio = ['asco', 'basura', 'ridícula', 'falsa', 'inútiles', 'patética', 'estupidez', 'ridículos', 'inservible', 'barata', 'mediocre']
    estereotipos = ['mujeres', 'feminazi', 'ellas', 'esposa', 'madre', 'novia', 'niñita', 'sola', 'callada', 'emocionales', 'débiles', 'casa']

    n_insultos = sum(8.0 for p in insultos if p in t)
    n_imperativos = sum(7.0 for p in imperativos if p in t)
    n_desprecio = sum(8.0 for p in desprecio if p in t)
    n_estereotipos = sum(6.0 for p in estereotipos if p in t)

    longitud = len(texto) / 100.0
    mayusculas = sum(1 for c in texto if c.isupper()) / max(1, len(texto))
    exclamaciones = texto.count('!')
    interrogaciones = texto.count('?')

    return [n_insultos, n_imperativos, n_desprecio, n_estereotipos, longitud, mayusculas, exclamaciones, interrogaciones]

s_train = np.array([extraer_caracteristicas_discursivas(t) for t in X_train_txt], dtype=float)
s_test = np.array([extraer_caracteristicas_discursivas(t) for t in X_test_txt], dtype=float) 
scaler_s = StandardScaler()
X_s_tr_sc = scaler_s.fit_transform(s_train)
X_s_te_sc = scaler_s.transform(s_test)


HMT1_tr = hstack([X_tfidf_tr, csr_matrix(X_lsa_tr_sc * 8.0), csr_matrix(X_s_tr_sc * 12.0)])
HMT1_te = hstack([X_tfidf_te, csr_matrix(X_lsa_te_sc * 8.0), csr_matrix(X_s_te_sc * 12.0)])


representations = {
    "1. TF-IDF Puro": (X_tfidf_tr, X_tfidf_te, False),
    "2. N-Grams Puros": (X_ngram_tr, X_ngram_te, False),
    "3. Lexical Features": (X_lex_tr, X_lex_te, True),
    "4. Latent Topics (LSA)": (X_lsa_tr_sc, X_lsa_te_sc, True),
    "5. HMT-V1 (Ultra-Optimizado)": (HMT1_tr, HMT1_te, True)
}


models = {
    "LogReg": LogisticRegression(max_iter=1500, class_weight='balanced', random_state=42),
    "3-NN": KNeighborsClassifier(n_neighbors=5, weights='distance'),
    "SVM": SVC(kernel='linear', C=1.0, class_weight='balanced', random_state=42),
    "NaiveBayes": "auto",
    "Red Neuronal": MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=500, random_state=42),
    "XGBoost": XGBClassifier(n_estimators=150, learning_rate=0.05, max_depth=5, eval_metric='logloss', random_state=42),
    "RandomForest": RandomForestClassifier(n_estimators=150, class_weight='balanced', random_state=42),
    "DecisionTree": DecisionTreeClassifier(class_weight='balanced', random_state=42)
}

print("\nEjecutando evaluación exhaustiva para la gran tabla maestra de ablación...")
results = []
predicciones_almacenadas = {}

for rep_name, (X_tr, X_te, is_dense) in representations.items():
    if is_dense:
        if hasattr(X_tr, "toarray"): X_tr = X_tr.toarray()
        if hasattr(X_te, "toarray"): X_te = X_te.toarray()

    for model_name, model_obj in models.items():
        if model_name == "NaiveBayes":
            clf = GaussianNB() if is_dense else MultinomialNB()
        else:
            clf = model_obj

        clf.fit(X_tr, y_train)
        preds = clf.predict(X_te)

        
        predicciones_almacenadas[(rep_name, model_name)] = preds

        p, r, f1, _ = precision_recall_fscore_support(y_test, preds, average='binary', zero_division=0)
        acc = accuracy_score(y_test, preds)

        results.append({
            "Representación": rep_name,
            "Modelo": model_name,
            "Precision": round(p, 2),
            "Recall": round(r, 2),
            "F1-score": round(f1, 2),
            "Accuracy": round(acc, 2)
        })

df_large = pd.DataFrame(results)

print("\n==========================================================================")
print("=== GRAN TABLA MAESTRA DE ABLACIÓN (HMT-V1 OPTIMIZADO VS BASES) ===========")
print("==========================================================================")
print(df_large.to_string(index=False))




print("\n==================================================================")
print("=== PRUEBA DE MCNEMAR: HMT-V1 ULTRA-OPTIMIZADO VS. BASES ==========")
print("==================================================================")

mcnemar_comparativas = []

for model_name in models.keys():
    preds_hmt = predicciones_almacenadas[("5. HMT-V1 (Ultra-Optimizado)", model_name)]
    correct_hmt = (preds_hmt == y_test)

    baselines_a_comparar = ["1. TF-IDF Puro", "2. N-Grams Puros", "3. Lexical Features", "4. Latent Topics (LSA)"]

    for base_name in baselines_a_comparar:
        preds_base = predicciones_almacenadas[(base_name, model_name)]
        correct_base = (preds_base == y_test)

        
        b_acierta_hmt_falla = np.sum(correct_base & ~correct_hmt)
        b_falla_hmt_acierta = np.sum(~correct_base & correct_hmt)
        ambos_aciertan = np.sum(correct_base & correct_hmt)
        ambos_fallan = np.sum(~correct_base & ~correct_hmt)

        tabla_contingencia = [[ambos_aciertan, b_acierta_hmt_falla],
                              [b_falla_hmt_acierta, ambos_fallan]]

        res = mcnemar(tabla_contingencia, exact=False, correction=True)
        is_sig = res.pvalue < 0.05
        sig_text = "Sí (p < 0.05)" if is_sig else "No"

        mcnemar_comparativas.append({
            "Clasificador": model_name,
            "Comparativa": f"HMT-V1 Opt vs {base_name.split('.')[1].strip()}",
            "Estadístico": round(res.statistic, 4),
            "P-valor": f"{res.pvalue:.5e}",
            "Significativo (p < 0.05)": sig_text
        })

df_mcnemar = pd.DataFrame(mcnemar_comparativas)
print(df_mcnemar.to_string(index=False))


### EXP-029 · V1

**Archivo de origen:** `21_Octubre.ipynb, celda 6`. **Referencias duplicadas:** 0. **Sintaxis de código:** Válida.

SCRIPT DOCTORAL MAESTRO: GRAN TABLA DE ABLACIÓN MASIVA + MCNEMAR
(Con HMT-V1 Ultra-Optimizado y todos los clasificadores - Corregido)
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer, CountV

**Ejecución:** celda independiente o variante experimental; revisar archivos de entrada y dependencias antes de ejecutarla. Los resultados originales no se insertan en esta celda.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB, GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
from scipy.sparse import hstack, csr_matrix
from statsmodels.stats.contingency_tables import mcnemar

print("1. Cargando y preparando datos para Task 1 (Binaria)...")
df = pd.read_csv('EXIST2021_train_es_multiclase.csv').dropna(subset=['text', 'task2'])
texts = df['text'].astype(str).values


y_task1 = np.where(df['task2'].values == 'non-sexist', 0, 1)

X_train_txt, X_test_txt, y_train, y_test = train_test_split(
    texts, y_task1, test_size=0.2, random_state=42, stratify=y_task1
)

print("2. Extrayendo representaciones base y el modelo HMT-V1 Ultra-Optimizado...")


tfidf = TfidfVectorizer(max_features=20000, ngram_range=(1, 3), sublinear_tf=True, min_df=2)
X_tfidf_tr = tfidf.fit_transform(X_train_txt)
X_tfidf_te = tfidf.transform(X_test_txt)


count_vec = CountVectorizer(ngram_range=(1, 2), max_features=5000, min_df=2)
X_ngram_tr = count_vec.fit_transform(X_train_txt)
X_ngram_te = count_vec.transform(X_test_txt)


def extraer_lexico(texto):
    t = texto.lower()
    insultos = ['puta', 'zorra', 'gorda', 'imbécil', 'estúpida', 'asquerosa', 'inútil', 'tarada', 'zorras', 'imbéciles', 'estúpidas', 'idiota', 'prostituta', 'calienta', 'retrasada']
    imperativos = ['callate', 'cocina', 'fregar', 'lugar', 'deberían', 'deben', 'debes', 'vete', 'largo', 'fuera', 'sálvete', 'limpia', 'atiende', 'sírvenos', 'sufre']
    return [
        sum(1 for p in insultos if p in t),
        sum(1 for p in imperativos if p in t),
        len(texto) / 100.0,
        sum(1 for c in texto if c.isupper()) / max(1, len(texto))
    ]

lex_tr = np.array([extraer_lexico(t) for t in X_train_txt], dtype=float)
lex_te = np.array([extraer_lexico(t) for t in X_test_txt], dtype=float)
scaler_lex = StandardScaler()
X_lex_tr = scaler_lex.fit_transform(lex_tr)
X_lex_te = scaler_lex.transform(lex_te)


svd = TruncatedSVD(n_components=200, random_state=42)
X_lsa_tr = svd.fit_transform(X_tfidf_tr)
X_lsa_te = svd.transform(X_tfidf_te)
scaler_lsa = StandardScaler()
X_lsa_tr_sc = scaler_lsa.fit_transform(X_lsa_tr)
X_lsa_te_sc = scaler_lsa.transform(X_lsa_te)


def extraer_caracteristicas_discursivas(texto):
    t = texto.lower()
    insultos = ['puta', 'zorra', 'gorda', 'imbécil', 'estúpida', 'asquerosa', 'inútil', 'tarada', 'zorras', 'imbéciles', 'estúpidas', 'idiota', 'prostituta', 'calienta', 'retrasada']
    imperativos = ['callate', 'cocina', 'fregar', 'lugar', 'deberían', 'deben', 'debes', 'vete', 'largo', 'fuera', 'sálvete', 'limpia', 'atiende', 'sírvenos', 'sufre']
    desprecio = ['asco', 'basura', 'ridícula', 'falsa', 'inútiles', 'patética', 'estupidez', 'ridículos', 'inservible', 'barata', 'mediocre']
    estereotipos = ['mujeres', 'feminazi', 'ellas', 'esposa', 'madre', 'novia', 'niñita', 'sola', 'callada', 'emocionales', 'débiles', 'casa']

    n_insultos = sum(8.0 for p in insultos if p in t)
    n_imperativos = sum(7.0 for p in imperativos if p in t)
    n_desprecio = sum(8.0 for p in desprecio if p in t)
    n_estereotipos = sum(6.0 for p in estereotipos if p in t)

    longitud = len(texto) / 100.0
    mayusculas = sum(1 for c in texto if c.isupper()) / max(1, len(texto))
    exclamaciones = texto.count('!')
    interrogaciones = texto.count('?')

    return [n_insultos, n_imperativos, n_desprecio, n_estereotipos, longitud, mayusculas, exclamaciones, interrogaciones]

s_train = np.array([extraer_caracteristicas_discursivas(t) for t in X_train_txt], dtype=float)
s_test = np.array([extraer_caracteristicas_discursivas(t) for t in X_test_txt], dtype=float) 
scaler_s = StandardScaler()
X_s_tr_sc = scaler_s.fit_transform(s_train)
X_s_te_sc = scaler_s.transform(s_test)


HMT1_tr = hstack([X_tfidf_tr, csr_matrix(X_lsa_tr_sc * 8.0), csr_matrix(X_s_tr_sc * 12.0)])
HMT1_te = hstack([X_tfidf_te, csr_matrix(X_lsa_te_sc * 8.0), csr_matrix(X_s_te_sc * 12.0)])


representations = {
    "1. TF-IDF Puro": (X_tfidf_tr, X_tfidf_te, False),
    "2. N-Grams Puros": (X_ngram_tr, X_ngram_te, False),
    "3. Lexical Features": (X_lex_tr, X_lex_te, True),
    "4. Latent Topics (LSA)": (X_lsa_tr_sc, X_lsa_te_sc, True),
    "5. HMT-V1 (Ultra-Optimizado)": (HMT1_tr, HMT1_te, True)
}


models = {
    "LogReg": LogisticRegression(max_iter=1500, class_weight='balanced', random_state=42),
    "3-NN": KNeighborsClassifier(n_neighbors=5, weights='distance'),
    "SVM": SVC(kernel='linear', C=1.0, class_weight='balanced', random_state=42),
    "NaiveBayes": "auto",
    "Red Neuronal": MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=500, random_state=42),
    "XGBoost": XGBClassifier(n_estimators=150, learning_rate=0.05, max_depth=5, eval_metric='logloss', random_state=42),
    "RandomForest": RandomForestClassifier(n_estimators=150, class_weight='balanced', random_state=42),
    "DecisionTree": DecisionTreeClassifier(class_weight='balanced', random_state=42)
}

print("\nEjecutando evaluación exhaustiva para la gran tabla maestra de ablación...")
results = []
predicciones_almacenadas = {}

for rep_name, (X_tr, X_te, is_dense) in representations.items():
    if is_dense:
        if hasattr(X_tr, "toarray"): X_tr = X_tr.toarray()
        if hasattr(X_te, "toarray"): X_te = X_te.toarray()

    for model_name, model_obj in models.items():
        if model_name == "NaiveBayes":
            clf = GaussianNB() if is_dense else MultinomialNB()
        else:
            clf = model_obj

        clf.fit(X_tr, y_train)
        preds = clf.predict(X_te)

        
        predicciones_almacenadas[(rep_name, model_name)] = preds

        p, r, f1, _ = precision_recall_fscore_support(y_test, preds, average='binary', zero_division=0)
        acc = accuracy_score(y_test, preds)

        results.append({
            "Representación": rep_name,
            "Modelo": model_name,
            "Precision": round(p, 2),
            "Recall": round(r, 2),
            "F1-score": round(f1, 2),
            "Accuracy": round(acc, 2)
        })

df_large = pd.DataFrame(results)

print("\n==========================================================================")
print("=== GRAN TABLA MAESTRA DE ABLACIÓN (HMT-V1 OPTIMIZADO VS BASES) ===========")
print("==========================================================================")
print(df_large.to_string(index=False))




print("\n==================================================================")
print("=== PRUEBA DE MCNEMAR: HMT-V1 ULTRA-OPTIMIZADO VS. BASES ==========")
print("==================================================================")

mcnemar_comparativas = []

for model_name in models.keys():
    preds_hmt = predicciones_almacenadas[("5. HMT-V1 (Ultra-Optimizado)", model_name)]
    correct_hmt = (preds_hmt == y_test)

    baselines_a_comparar = ["1. TF-IDF Puro", "2. N-Grams Puros", "3. Lexical Features", "4. Latent Topics (LSA)"]

    for base_name in baselines_a_comparar:
        preds_base = predicciones_almacenadas[(base_name, model_name)]
        correct_base = (preds_base == y_test)

        
        b_acierta_hmt_falla = np.sum(correct_base & ~correct_hmt)
        b_falla_hmt_acierta = np.sum(~correct_base & correct_hmt)
        ambos_aciertan = np.sum(correct_base & correct_hmt)
        ambos_fallan = np.sum(~correct_base & ~correct_hmt)

        tabla_contingencia = [[ambos_aciertan, b_acierta_hmt_falla],
                              [b_falla_hmt_acierta, ambos_fallan]]

        res = mcnemar(tabla_contingencia, exact=False, correction=True)
        is_sig = res.pvalue < 0.6
        sig_text = "Sí (p < 0.05)" if is_sig else "No"

        mcnemar_comparativas.append({
            "Clasificador": model_name,
            "Comparativa": f"HMT-V1 Opt vs {base_name.split('.')[1].strip()}",
            "Estadístico": round(res.statistic, 4),
            "P-valor": f"{res.pvalue:.5e}",
            "Significativo (p < 0.05)": sig_text
        })

df_mcnemar = pd.DataFrame(mcnemar_comparativas)
print(df_mcnemar.to_string(index=False))


### EXP-030 · V1

**Archivo de origen:** `21_Octubre.ipynb, celda 10`. **Referencias duplicadas:** 0. **Sintaxis de código:** Válida.

SCRIPT DOCTORAL MAESTRO (MULTICLASE - TASK 2):
Ablación Masiva + McNemar + Reporte Detallado y Matriz de Confusión por Clase
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_e

**Ejecución:** celda independiente o variante experimental; revisar archivos de entrada y dependencias antes de ejecutarla. Los resultados originales no se insertan en esta celda.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB, GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from sklearn.metrics import precision_recall_fscore_support, accuracy_score, classification_report, confusion_matrix
from scipy.sparse import hstack, csr_matrix
from statsmodels.stats.contingency_tables import mcnemar

print("1. Cargando y preparando datos para Clasificación Multiclase (Task 2)...")
df = pd.read_csv('EXIST2021_train_es_multiclase.csv').dropna(subset=['text', 'task2'])
texts = df['text'].astype(str).values


encoder = LabelEncoder()
y_multiclass = encoder.fit_transform(df['task2'].astype(str))
clases_nombres = encoder.classes_

X_train_txt, X_test_txt, y_train, y_test = train_test_split(
    texts, y_multiclass, test_size=0.2, random_state=42, stratify=y_multiclass
)

print("2. Extrayendo representaciones base y el modelo HMT-V1 Hiper-Optimizado (Multiclase)...")


tfidf = TfidfVectorizer(max_features=25000, ngram_range=(1, 3), sublinear_tf=True, min_df=2)
X_tfidf_tr = tfidf.fit_transform(X_train_txt)
X_tfidf_te = tfidf.transform(X_test_txt)


count_vec = CountVectorizer(ngram_range=(1, 3), max_features=8000, min_df=2)
X_ngram_tr = count_vec.fit_transform(X_train_txt)
X_ngram_te = count_vec.transform(X_test_txt)


def extraer_lexico(texto):
    t = texto.lower()
    insultos = ['puta', 'zorra', 'gorda', 'imbécil', 'estúpida', 'asquerosa', 'inútil', 'tarada', 'zorras', 'imbéciles', 'estúpidas', 'idiota', 'prostituta', 'calienta', 'retrasada']
    imperativos = ['callate', 'cocina', 'fregar', 'lugar', 'deberían', 'deben', 'debes', 'vete', 'largo', 'fuera', 'sálvete', 'limpia', 'atiende', 'sírvenos', 'sufre']
    return [
        sum(1 for p in insultos if p in t),
        sum(1 for p in imperativos if p in t),
        len(texto) / 100.0,
        sum(1 for c in texto if c.isupper()) / max(1, len(texto))
    ]

lex_tr = np.array([extraer_lexico(t) for t in X_train_txt], dtype=float)
lex_te = np.array([extraer_lexico(t) for t in X_test_txt], dtype=float)
scaler_lex = StandardScaler()
X_lex_tr = scaler_lex.fit_transform(lex_tr)
X_lex_te = scaler_lex.transform(lex_te)


svd = TruncatedSVD(n_components=250, random_state=42)
X_lsa_tr = svd.fit_transform(X_tfidf_tr)
X_lsa_te = svd.transform(X_tfidf_te)
scaler_lsa = StandardScaler()
X_lsa_tr_sc = scaler_lsa.fit_transform(X_lsa_tr)
X_lsa_te_sc = scaler_lsa.transform(X_lsa_te)


def extraer_caracteristicas_discursivas(texto):
    t = texto.lower()
    insultos = ['puta', 'zorra', 'gorda', 'imbécil', 'estúpida', 'asquerosa', 'inútil', 'tarada', 'zorras', 'imbéciles', 'estúpidas', 'idiota', 'prostituta', 'calienta', 'retrasada']
    imperativos = ['callate', 'cocina', 'fregar', 'lugar', 'deberían', 'deben', 'debes', 'vete', 'largo', 'fuera', 'sálvete', 'limpia', 'atiende', 'sírvenos', 'sufre']
    desprecio = ['asco', 'basura', 'ridícula', 'falsa', 'inútiles', 'patética', 'estupidez', 'ridículos', 'inservible', 'barata', 'mediocre']
    estereotipos = ['mujeres', 'feminazi', 'ellas', 'esposa', 'madre', 'novia', 'niñita', 'sola', 'callada', 'emocionales', 'débiles', 'casa']

    n_insultos = sum(10.0 for p in insultos if p in t)
    n_imperativos = sum(8.0 for p in imperativos if p in t)
    n_desprecio = sum(10.0 for p in desprecio if p in t)
    n_estereotipos = sum(8.0 for p in estereotipos if p in t)

    longitud = len(texto) / 100.0
    mayusculas = sum(1 for c in texto if c.isupper()) / max(1, len(texto))
    exclamaciones = texto.count('!')
    interrogaciones = texto.count('?')

    return [n_insultos, n_imperativos, n_desprecio, n_estereotipos, longitud, mayusculas, exclamaciones, interrogaciones]

s_train = np.array([extraer_caracteristicas_discursivas(t) for t in X_train_txt], dtype=float)
s_test = np.array([extraer_caracteristicas_discursivas(t) for t in X_test_txt], dtype=float)
scaler_s = StandardScaler()
X_s_tr_sc = scaler_s.fit_transform(s_train)
X_s_te_sc = scaler_s.transform(s_test)


HMT1_tr = hstack([X_tfidf_tr, csr_matrix(X_lsa_tr_sc * 10.0), csr_matrix(X_s_tr_sc * 15.0)])
HMT1_te = hstack([X_tfidf_te, csr_matrix(X_lsa_te_sc * 10.0), csr_matrix(X_s_te_sc * 15.0)])


representations = {
    "1. TF-IDF Puro": (X_tfidf_tr, X_tfidf_te, False),
    "2. N-Grams Puros": (X_ngram_tr, X_ngram_te, False),
    "3. Lexical Features": (X_lex_tr, X_lex_te, True),
    "4. Latent Topics (LSA)": (X_lsa_tr_sc, X_lsa_te_sc, True),
    "5. HMT-V1 (Hiper-Optimizado)": (HMT1_tr, HMT1_te, True)
}


models = {
    "LogReg": LogisticRegression(max_iter=2000, C=4.0, class_weight='balanced', random_state=42),
    "3-NN": KNeighborsClassifier(n_neighbors=3, weights='distance'),
    "SVM": SVC(kernel='linear', C=3.0, class_weight='balanced', random_state=42),
    "NaiveBayes": "auto",
    "Red Neuronal": MLPClassifier(hidden_layer_sizes=(256, 128), max_iter=800, alpha=0.0001, random_state=42),
    "XGBoost": XGBClassifier(n_estimators=300, learning_rate=0.03, max_depth=6, eval_metric='mlogloss', random_state=42),
    "RandomForest": RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=42),
    "DecisionTree": DecisionTreeClassifier(max_depth=12, class_weight='balanced', random_state=42)
}




print("\nEjecutando evaluación multiclase exhaustiva para la gran tabla maestra...")
results = []
predicciones_almacenadas = {}

for rep_name, (X_tr, X_te, is_dense) in representations.items():
    if is_dense:
        if hasattr(X_tr, "toarray"): X_tr = X_tr.toarray()
        if hasattr(X_te, "toarray"): X_te = X_te.toarray()

    for model_name, model_obj in models.items():
        if model_name == "NaiveBayes":
            clf = GaussianNB() if is_dense else MultinomialNB()
        else:
            clf = model_obj

        clf.fit(X_tr, y_train)
        preds = clf.predict(X_te)

        
        predicciones_almacenadas[(rep_name, model_name)] = preds

        p, r, f1, _ = precision_recall_fscore_support(y_test, preds, average='macro', zero_division=0)
        acc = accuracy_score(y_test, preds)

        results.append({
            "Representación": rep_name,
            "Modelo": model_name,
            "Precision (Macro)": round(p, 2),
            "Recall (Macro)": round(r, 2),
            "F1-score (Macro)": round(f1, 2),
            "Accuracy": round(acc, 2)
        })

df_large = pd.DataFrame(results)

print("\n==========================================================================")
print("=== GRAN TABLA MAESTRA DE ABLACIÓN MULTICLASE (HMT-V1 OPT VS BASES) =======")
print("==========================================================================")
print(df_large.to_string(index=False))




print("\n==================================================================")
print("=== PRUEBA DE MCNEMAR MULTICLASE: HMT-V1 OPTIMIZADO VS. BASES =====")
print("==================================================================")

mcnemar_comparativas = []

for model_name in models.keys():
    preds_hmt = predicciones_almacenadas[("5. HMT-V1 (Hiper-Optimizado)", model_name)]
    correct_hmt = (preds_hmt == y_test)

    baselines_a_comparar = ["1. TF-IDF Puro", "2. N-Grams Puros", "3. Lexical Features", "4. Latent Topics (LSA)"]

    for base_name in baselines_a_comparar:
        preds_base = predicciones_almacenadas[(base_name, model_name)]
        correct_base = (preds_base == y_test)

        b_acierta_hmt_falla = np.sum(correct_base & ~correct_hmt)
        b_falla_hmt_acierta = np.sum(~correct_base & correct_hmt)
        ambos_aciertan = np.sum(correct_base & correct_hmt)
        ambos_fallan = np.sum(~correct_base & ~correct_hmt)

        tabla_contingencia = [[ambos_aciertan, b_acierta_hmt_falla],
                              [b_falla_hmt_acierta, ambos_fallan]]

        res = mcnemar(tabla_contingencia, exact=False, correction=True)
        is_sig = res.pvalue < 0.05
        sig_text = "Sí (p < 0.05)" if is_sig else "No"

        mcnemar_comparativas.append({
            "Clasificador": model_name,
            "Comparativa": f"HMT-V1 Opt vs {base_name.split('.')[1].strip()}",
            "Estadístico": round(res.statistic, 4),
            "P-valor": f"{res.pvalue:.5e}",
            "Significativo (p < 0.05)": sig_text
        })

df_mcnemar = pd.DataFrame(mcnemar_comparativas)
print(df_mcnemar.to_string(index=False))




print("\n==================================================================")
print("=== REPORTE DE RENDIMIENTO POR CLASE (TASK 2 - MULTICLASE) =======")
print("==================================================================")

preds_hmt_best = predicciones_almacenadas[("5. HMT-V1 (Hiper-Optimizado)", "LogReg")]
print(classification_report(y_test, preds_hmt_best, target_names=clases_nombres, zero_division=0))


plt.figure(figsize=(8, 6))
cm = confusion_matrix(y_test, preds_hmt_best, normalize='true')
sns.heatmap(cm, annot=True, fmt='.2f', cmap='Blues', xticklabels=clases_nombres, yticklabels=clases_nombres)
plt.xlabel('Predicción del Modelo HMT-V1')
plt.ylabel('Etiqueta Real')
plt.title('Matriz de Confusión Normalizada (HMT-V1 Multiclase - LogReg)')
plt.show()


### EXP-031 · V2

**Archivo de origen:** `21_Octubre.ipynb, celda 11`. **Referencias duplicadas:** 0. **Sintaxis de código:** Válida.

SCRIPT DOCTORAL MAESTRO (MULTICLASE - TASK 2 - ULTRA OPTIMIZADO):
HMT-V2 con N-gramas de Caracteres + Ablación + McNemar + Reporte por Clase
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from s

**Ejecución:** celda independiente o variante experimental; revisar archivos de entrada y dependencias antes de ejecutarla. Los resultados originales no se insertan en esta celda.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB, GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from sklearn.metrics import precision_recall_fscore_support, accuracy_score, classification_report, confusion_matrix
from scipy.sparse import hstack, csr_matrix
from statsmodels.stats.contingency_tables import mcnemar

print("1. Cargando y preparando datos para Clasificación Multiclase (Task 2)...")
df = pd.read_csv('EXIST2021_train_es_multiclase.csv').dropna(subset=['text', 'task2'])
texts = df['text'].astype(str).values


encoder = LabelEncoder()
y_multiclass = encoder.fit_transform(df['task2'].astype(str))
clases_nombres = encoder.classes_

X_train_txt, X_test_txt, y_train, y_test = train_test_split(
    texts, y_multiclass, test_size=0.2, random_state=42, stratify=y_multiclass
)

print("2. Extrayendo representaciones base y el modelo HMT-V2 Hiper-Optimizado (Multiclase)...")


tfidf_word = TfidfVectorizer(max_features=20000, ngram_range=(1, 3), sublinear_tf=True, min_df=2)
X_tfidf_w_tr = tfidf_word.fit_transform(X_train_txt)
X_tfidf_w_te = tfidf_word.transform(X_test_txt)


tfidf_char = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), max_features=25000, sublinear_tf=True, min_df=3)
X_tfidf_c_tr = tfidf_char.fit_transform(X_train_txt)
X_tfidf_c_te = tfidf_char.transform(X_test_txt)


count_vec = CountVectorizer(ngram_range=(1, 2), max_features=8000, min_df=2)
X_ngram_tr = count_vec.fit_transform(X_train_txt)
X_ngram_te = count_vec.transform(X_test_txt)


def extraer_lexico(texto):
    t = texto.lower()
    insultos = ['puta', 'zorra', 'gorda', 'imbécil', 'estúpida', 'asquerosa', 'inútil', 'tarada', 'zorras', 'imbéciles', 'estúpidas', 'idiota', 'prostituta', 'calienta', 'retrasada']
    imperativos = ['callate', 'cocina', 'fregar', 'lugar', 'deberían', 'deben', 'debes', 'vete', 'largo', 'fuera', 'sálvete', 'limpia', 'atiende', 'sírvenos', 'sufre']
    return [
        sum(1 for p in insultos if p in t),
        sum(1 for p in imperativos if p in t),
        len(texto) / 100.0,
        sum(1 for c in texto if c.isupper()) / max(1, len(texto))
    ]

lex_tr = np.array([extraer_lexico(t) for t in X_train_txt], dtype=float)
lex_te = np.array([extraer_lexico(t) for t in X_test_txt], dtype=float)
scaler_lex = StandardScaler()
X_lex_tr = scaler_lex.fit_transform(lex_tr)
X_lex_te = scaler_lex.transform(lex_te)


svd = TruncatedSVD(n_components=200, random_state=42)
X_lsa_tr = svd.fit_transform(X_tfidf_w_tr)
X_lsa_te = svd.transform(X_tfidf_w_te)
scaler_lsa = StandardScaler()
X_lsa_tr_sc = scaler_lsa.fit_transform(X_lsa_tr)
X_lsa_te_sc = scaler_lsa.transform(X_lsa_te)


def extraer_caracteristicas_discursivas_multiclase(texto):
    t = texto.lower()
    insultos = ['puta', 'zorra', 'gorda', 'imbécil', 'estúpida', 'asquerosa', 'inútil', 'tarada', 'zorras', 'imbéciles', 'estúpidas', 'idiota', 'prostituta', 'calienta', 'retrasada']
    imperativos = ['callate', 'cocina', 'fregar', 'lugar', 'deberían', 'deben', 'debes', 'vete', 'largo', 'fuera', 'sálvete', 'limpia', 'atiende', 'sírvenos', 'sufre']
    desprecio = ['asco', 'basura', 'ridícula', 'falsa', 'inútiles', 'patética', 'estupidez', 'ridículos', 'inservible', 'barata', 'mediocre']
    estereotipos = ['mujeres', 'feminazi', 'ellas', 'esposa', 'madre', 'novia', 'niñita', 'sola', 'callada', 'emocionales', 'débiles', 'casa', 'objeto', 'calladita']
    ideologia = ['feminismo', 'feminista', 'machista', 'patriarcado', 'igualdad', 'derechos', 'sociedad', 'ideología']

    n_insultos = sum(10.0 for p in insultos if p in t)
    n_imperativos = sum(8.0 for p in imperativos if p in t)
    n_desprecio = sum(10.0 for p in desprecio if p in t)
    n_estereotipos = sum(8.0 for p in estereotipos if p in t)
    n_ideologia = sum(8.0 for p in ideologia if p in t)

    longitud = len(texto) / 100.0
    mayusculas = sum(1 for c in texto if c.isupper()) / max(1, len(texto))
    exclamaciones = texto.count('!')
    interrogaciones = texto.count('?')

    return [n_insultos, n_imperativos, n_desprecio, n_estereotipos, n_ideologia, longitud, mayusculas, exclamaciones, interrogaciones]

s_train = np.array([extraer_caracteristicas_discursivas_multiclase(t) for t in X_train_txt], dtype=float)
s_test = np.array([extraer_caracteristicas_discursivas_multiclase(t) for t in X_test_txt], dtype=float)
scaler_s = StandardScaler()
X_s_tr_sc = scaler_s.fit_transform(s_train)
X_s_te_sc = scaler_s.transform(s_test)


HMT2_tr = hstack([X_tfidf_w_tr, X_tfidf_c_tr, csr_matrix(X_lsa_tr_sc * 8.0), csr_matrix(X_s_tr_sc * 12.0)])
HMT2_te = hstack([X_tfidf_w_te, X_tfidf_c_te, csr_matrix(X_lsa_te_sc * 8.0), csr_matrix(X_s_te_sc * 12.0)])


representations = {
    "1. TF-IDF Puro": (X_tfidf_w_tr, X_tfidf_w_te, False),
    "2. N-Grams Puros": (X_ngram_tr, X_ngram_te, False),
    "3. Lexical Features": (X_lex_tr, X_lex_te, True),
    "4. Latent Topics (LSA)": (X_lsa_tr_sc, X_lsa_te_sc, True),
    "5. HMT-V2 (Hiper-Optimizado)": (HMT2_tr, HMT2_te, True)
}


models = {
    "LogReg": LogisticRegression(max_iter=2500, C=5.0, class_weight='balanced', random_state=42),
    "3-NN": KNeighborsClassifier(n_neighbors=3, weights='distance'),
    "SVM": SVC(kernel='linear', C=4.0, class_weight='balanced', random_state=42),
    "NaiveBayes": "auto",
    "Red Neuronal": MLPClassifier(hidden_layer_sizes=(256, 128), max_iter=1000, alpha=0.0001, random_state=42),
    "XGBoost": XGBClassifier(n_estimators=350, learning_rate=0.02, max_depth=6, eval_metric='mlogloss', random_state=42),
    "RandomForest": RandomForestClassifier(n_estimators=350, class_weight='balanced', random_state=42),
    "DecisionTree": DecisionTreeClassifier(max_depth=14, class_weight='balanced', random_state=42)
}




print("\nEjecutando evaluación multiclase exhaustiva para la gran tabla maestra...")
results = []
predicciones_almacenadas = {}

for rep_name, (X_tr, X_te, is_dense) in representations.items():
    if is_dense:
        if hasattr(X_tr, "toarray"): X_tr = X_tr.toarray()
        if hasattr(X_te, "toarray"): X_te = X_te.toarray()

    for model_name, model_obj in models.items():
        if model_name == "NaiveBayes":
            clf = GaussianNB() if is_dense else MultinomialNB()
        else:
            clf = model_obj

        clf.fit(X_tr, y_train)
        preds = clf.predict(X_te)

        predicciones_almacenadas[(rep_name, model_name)] = preds

        p, r, f1, _ = precision_recall_fscore_support(y_test, preds, average='macro', zero_division=0)
        acc = accuracy_score(y_test, preds)

        results.append({
            "Representación": rep_name,
            "Modelo": model_name,
            "Precision (Macro)": round(p, 2),
            "Recall (Macro)": round(r, 2),
            "F1-score (Macro)": round(f1, 2),
            "Accuracy": round(acc, 2)
        })

df_large = pd.DataFrame(results)

print("\n==========================================================================")
print("=== GRAN TABLA MAESTRA DE ABLACIÓN MULTICLASE (HMT-V2 OPT VS BASES) =======")
print("==========================================================================")
print(df_large.to_string(index=False))




print("\n==================================================================")
print("=== PRUEBA DE MCNEMAR MULTICLASE: HMT-V2 OPTIMIZADO VS. BASES =====")
print("==================================================================")

mcnemar_comparativas = []

for model_name in models.keys():
    preds_hmt = predicciones_almacenadas[("5. HMT-V2 (Hiper-Optimizado)", model_name)]
    correct_hmt = (preds_hmt == y_test)

    baselines_a_comparar = ["1. TF-IDF Puro", "2. N-Grams Puros", "3. Lexical Features", "4. Latent Topics (LSA)"]

    for base_name in baselines_a_comparar:
        preds_base = predicciones_almacenadas[(base_name, model_name)]
        correct_base = (preds_base == y_test)

        b_acierta_hmt_falla = np.sum(correct_base & ~correct_hmt)
        b_falla_hmt_acierta = np.sum(~correct_base & correct_hmt)
        ambos_aciertan = np.sum(correct_base & correct_hmt)
        ambos_fallan = np.sum(~correct_base & ~correct_hmt)

        tabla_contingencia = [[ambos_aciertan, b_acierta_hmt_falla],
                              [b_falla_hmt_acierta, ambos_fallan]]

        res = mcnemar(tabla_contingencia, exact=False, correction=True)
        is_sig = res.pvalue < 0.05
        sig_text = "Sí (p < 0.05)" if is_sig else "No"

        mcnemar_comparativas.append({
            "Clasificador": model_name,
            "Comparativa": f"HMT-V2 Opt vs {base_name.split('.')[1].strip()}",
            "Estadístico": round(res.statistic, 4),
            "P-valor": f"{res.pvalue:.5e}",
            "Significativo (p < 0.05)": sig_text
        })

df_mcnemar = pd.DataFrame(mcnemar_comparativas)
print(df_mcnemar.to_string(index=False))




print("\n==================================================================")
print("=== REPORTE DE RENDIMIENTO POR CLASE (TASK 2 - MULTICLASE HMT-V2) ==")
print("==================================================================")

preds_hmt_best = predicciones_almacenadas[("5. HMT-V2 (Hiper-Optimizado)", "LogReg")]
print(classification_report(y_test, preds_hmt_best, target_names=clases_nombres, zero_division=0))


plt.figure(figsize=(9, 7))
cm = confusion_matrix(y_test, preds_hmt_best, normalize='true')
sns.heatmap(cm, annot=True, fmt='.2f', cmap='Purples', xticklabels=clases_nombres, yticklabels=clases_nombres)
plt.xlabel('Predicción del Modelo HMT-V2 Hiper-Optimizado')
plt.ylabel('Etiqueta Real')
plt.title('Matriz de Confusión Normalizada (Task 2 Multiclase - LogReg)')
plt.tight_layout()
plt.show()


### EXP-032 · V2

**Archivo de origen:** `21_Octubre.ipynb, celda 12`. **Referencias duplicadas:** 0. **Sintaxis de código:** Válida.

SCRIPT DOCTORAL MAESTRO (MULTICLASE - TASK 2 - OPTIMIZADO Y VELOZ):
HMT-V2 Optimizado + Ablación + McNemar + Reporte por Clase
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature

**Ejecución:** celda independiente o variante experimental; revisar archivos de entrada y dependencias antes de ejecutarla. Los resultados originales no se insertan en esta celda.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB, GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from sklearn.metrics import precision_recall_fscore_support, accuracy_score, classification_report, confusion_matrix
from scipy.sparse import hstack, csr_matrix
from statsmodels.stats.contingency_tables import mcnemar

print("1. Cargando y preparando datos para Clasificación Multiclase (Task 2)...")
df = pd.read_csv('EXIST2021_train_es_multiclase.csv').dropna(subset=['text', 'task2'])
texts = df['text'].astype(str).values


encoder = LabelEncoder()
y_multiclass = encoder.fit_transform(df['task2'].astype(str))
clases_nombres = encoder.classes_

X_train_txt, X_test_txt, y_train, y_test = train_test_split(
    texts, y_multiclass, test_size=0.2, random_state=42, stratify=y_multiclass
)

print("2. Extrayendo representaciones base y el modelo HMT-V2 Optimizado (Multiclase)...")


tfidf_word = TfidfVectorizer(max_features=15000, ngram_range=(1, 2), sublinear_tf=True, min_df=2)
X_tfidf_w_tr = tfidf_word.fit_transform(X_train_txt)
X_tfidf_w_te = tfidf_word.transform(X_test_txt)


tfidf_char = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 4), max_features=15000, sublinear_tf=True, min_df=3)
X_tfidf_c_tr = tfidf_char.fit_transform(X_train_txt)
X_tfidf_c_te = tfidf_char.transform(X_test_txt)


count_vec = CountVectorizer(ngram_range=(1, 2), max_features=5000, min_df=2)
X_ngram_tr = count_vec.fit_transform(X_train_txt)
X_ngram_te = count_vec.transform(X_test_txt)


def extraer_lexico(texto):
    t = texto.lower()
    insultos = ['puta', 'zorra', 'gorda', 'imbécil', 'estúpida', 'asquerosa', 'inútil', 'tarada', 'zorras', 'imbéciles', 'estúpidas', 'idiota', 'prostituta', 'calienta', 'retrasada']
    imperativos = ['callate', 'cocina', 'fregar', 'lugar', 'deberían', 'deben', 'debes', 'vete', 'largo', 'fuera', 'sálvete', 'limpia', 'atiende', 'sírvenos', 'sufre']
    return [
        sum(1 for p in insultos if p in t),
        sum(1 for p in imperativos if p in t),
        len(texto) / 100.0,
        sum(1 for c in texto if c.isupper()) / max(1, len(texto))
    ]

lex_tr = np.array([extraer_lexico(t) for t in X_train_txt], dtype=float)
lex_te = np.array([extraer_lexico(t) for t in X_test_txt], dtype=float)
scaler_lex = StandardScaler()
X_lex_tr = scaler_lex.fit_transform(lex_tr)
X_lex_te = scaler_lex.transform(lex_te)


svd = TruncatedSVD(n_components=150, random_state=42)
X_lsa_tr = svd.fit_transform(X_tfidf_w_tr)
X_lsa_te = svd.transform(X_tfidf_w_te)
scaler_lsa = StandardScaler()
X_lsa_tr_sc = scaler_lsa.fit_transform(X_lsa_tr)
X_lsa_te_sc = scaler_lsa.transform(X_lsa_te)


def extraer_caracteristicas_discursivas_multiclase(texto):
    t = texto.lower()
    insultos = ['puta', 'zorra', 'gorda', 'imbécil', 'estúpida', 'asquerosa', 'inútil', 'tarada', 'zorras', 'imbéciles', 'estúpidas', 'idiota', 'prostituta', 'calienta', 'retrasada']
    imperativos = ['callate', 'cocina', 'fregar', 'lugar', 'deberían', 'deben', 'debes', 'vete', 'largo', 'fuera', 'sálvete', 'limpia', 'atiende', 'sírvenos', 'sufre']
    desprecio = ['asco', 'basura', 'ridícula', 'falsa', 'inútiles', 'patética', 'estupidez', 'ridículos', 'inservible', 'barata', 'mediocre']
    estereotipos = ['mujeres', 'feminazi', 'ellas', 'esposa', 'madre', 'novia', 'niñita', 'sola', 'callada', 'emocionales', 'débiles', 'casa', 'objeto', 'calladita']
    ideologia = ['feminismo', 'feminista', 'machista', 'patriarcado', 'igualdad', 'derechos', 'sociedad', 'ideología']

    n_insultos = sum(10.0 for p in insultos if p in t)
    n_imperativos = sum(8.0 for p in imperativos if p in t)
    n_desprecio = sum(10.0 for p in desprecio if p in t)
    n_estereotipos = sum(8.0 for p in estereotipos if p in t)
    n_ideologia = sum(8.0 for p in ideologia if p in t)

    longitud = len(texto) / 100.0
    mayusculas = sum(1 for c in texto if c.isupper()) / max(1, len(texto))
    exclamaciones = texto.count('!')
    interrogaciones = texto.count('?')

    return [n_insultos, n_imperativos, n_desprecio, n_estereotipos, n_ideologia, longitud, mayusculas, exclamaciones, interrogaciones]

s_train = np.array([extraer_caracteristicas_discursivas_multiclase(t) for t in X_train_txt], dtype=float)
s_test = np.array([extraer_caracteristicas_discursivas_multiclase(t) for t in X_test_txt], dtype=float)
scaler_s = StandardScaler()
X_s_tr_sc = scaler_s.fit_transform(s_train)
X_s_te_sc = scaler_s.transform(s_test)


HMT2_tr = hstack([X_tfidf_w_tr, X_tfidf_c_tr, csr_matrix(X_lsa_tr_sc * 6.0), csr_matrix(X_s_tr_sc * 10.0)])
HMT2_te = hstack([X_tfidf_w_te, X_tfidf_c_te, csr_matrix(X_lsa_te_sc * 6.0), csr_matrix(X_s_te_sc * 10.0)])


representations = {
    "1. TF-IDF Puro": (X_tfidf_w_tr, X_tfidf_w_te, False),
    "2. N-Grams Puros": (X_ngram_tr, X_ngram_te, False),
    "3. Lexical Features": (X_lex_tr, X_lex_te, True),
    "4. Latent Topics (LSA)": (X_lsa_tr_sc, X_lsa_te_sc, True),
    "5. HMT-V2 (Optimizado)": (HMT2_tr, HMT2_te, True)
}


models = {
    "LogReg": LogisticRegression(max_iter=1500, C=3.0, class_weight='balanced', random_state=42),
    "3-NN": KNeighborsClassifier(n_neighbors=5, weights='distance'),
    "SVM": SVC(kernel='linear', C=2.0, class_weight='balanced', random_state=42),
    "NaiveBayes": "auto",
    "Red Neuronal": MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=600, random_state=42),
    "XGBoost": XGBClassifier(n_estimators=200, learning_rate=0.03, max_depth=5, eval_metric='mlogloss', random_state=42),
    "RandomForest": RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42),
    "DecisionTree": DecisionTreeClassifier(max_depth=10, class_weight='balanced', random_state=42)
}




print("\nEjecutando evaluación multiclase exhaustiva para la gran tabla maestra...")
results = []
predicciones_almacenadas = {}

for rep_name, (X_tr, X_te, is_dense) in representations.items():
    if is_dense:
        if hasattr(X_tr, "toarray"): X_tr = X_tr.toarray()
        if hasattr(X_te, "toarray"): X_te = X_te.toarray()

    for model_name, model_obj in models.items():
        if model_name == "NaiveBayes":
            clf = GaussianNB() if is_dense else MultinomialNB()
        else:
            clf = model_obj

        clf.fit(X_tr, y_train)
        preds = clf.predict(X_te)

        predicciones_almacenadas[(rep_name, model_name)] = preds

        p, r, f1, _ = precision_recall_fscore_support(y_test, preds, average='macro', zero_division=0)
        acc = accuracy_score(y_test, preds)

        results.append({
            "Representación": rep_name,
            "Modelo": model_name,
            "Precision (Macro)": round(p, 2),
            "Recall (Macro)": round(r, 2),
            "F1-score (Macro)": round(f1, 2),
            "Accuracy": round(acc, 2)
        })

df_large = pd.DataFrame(results)

print("\n==========================================================================")
print("=== GRAN TABLA MAESTRA DE ABLACIÓN MULTICLASE (HMT-V2 VS BASES) ===========")
print("==========================================================================")
print(df_large.to_string(index=False))




print("\n==================================================================")
print("=== PRUEBA DE MCNEMAR MULTICLASE: HMT-V2 OPTIMIZADO VS. BASES =====")
print("==================================================================")

mcnemar_comparativas = []

for model_name in models.keys():
    preds_hmt = predicciones_almacenadas[("5. HMT-V2 (Optimizado)", model_name)]
    correct_hmt = (preds_hmt == y_test)

    baselines_a_comparar = ["1. TF-IDF Puro", "2. N-Grams Puros", "3. Lexical Features", "4. Latent Topics (LSA)"]

    for base_name in baselines_a_comparar:
        preds_base = predicciones_almacenadas[(base_name, model_name)]
        correct_base = (preds_base == y_test)

        b_acierta_hmt_falla = np.sum(correct_base & ~correct_hmt)
        b_falla_hmt_acierta = np.sum(~correct_base & correct_hmt)
        ambos_aciertan = np.sum(correct_base & correct_base) 
        ambos_fallan = np.sum(~correct_base & ~correct_hmt)

        
        ambos_aciertan = np.sum(correct_base & correct_hmt)

        tabla_contingencia = [[ambos_aciertan, b_acierta_hmt_falla],
                              [b_falla_hmt_acierta, ambos_fallan]]

        res = mcnemar(tabla_contingencia, exact=False, correction=True)
        is_sig = res.pvalue < 0.05
        sig_text = "Sí (p < 0.05)" if is_sig else "No"

        mcnemar_comparativas.append({
            "Clasificador": model_name,
            "Comparativa": f"HMT-V2 Opt vs {base_name.split('.')[1].strip()}",
            "Estadístico": round(res.statistic, 4),
            "P-valor": f"{res.pvalue:.5e}",
            "Significativo (p < 0.05)": sig_text
        })

df_mcnemar = pd.DataFrame(mcnemar_comparativas)
print(df_mcnemar.to_string(index=False))




print("\n==================================================================")
print("=== REPORTE DE RENDIMIENTO POR CLASE (TASK 2 - MULTICLASE HMT-V2) ==")
print("==================================================================")

preds_hmt_best = predicciones_almacenadas[("5. HMT-V2 (Optimizado)", "LogReg")]
print(classification_report(y_test, preds_hmt_best, target_names=clases_nombres, zero_division=0))


plt.figure(figsize=(9, 7))
cm = confusion_matrix(y_test, preds_hmt_best, normalize='true')
sns.heatmap(cm, annot=True, fmt='.2f', cmap='Purples', xticklabels=clases_nombres, yticklabels=clases_nombres)
plt.xlabel('Predicción del Modelo HMT-V2 Optimizado')
plt.ylabel('Etiqueta Real')
plt.title('Matriz de Confusión Normalizada (Task 2 Multiclase - LogReg)')
plt.tight_layout()
plt.show()


### EXP-033 · V3

**Archivo de origen:** `21_Octubre.ipynb, celda 14`. **Referencias duplicadas:** 0. **Sintaxis de código:** Válida.

SCRIPT DOCTORAL MAESTRO (MULTICLASE - TASK 2 - HMT-V3 HIPER-OPTIMIZADO):
Matriz de Ablación Masiva + McNemar + Búsqueda de Máximo Rendimiento
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from 

**Ejecución:** celda independiente o variante experimental; revisar archivos de entrada y dependencias antes de ejecutarla. Los resultados originales no se insertan en esta celda.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB, GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from sklearn.metrics import precision_recall_fscore_support, accuracy_score, classification_report, confusion_matrix
from scipy.sparse import hstack, csr_matrix
from statsmodels.stats.contingency_tables import mcnemar

print("1. Cargando y preparando datos para Clasificación Multiclase (Task 2)...")
df = pd.read_csv('EXIST2021_train_es_multiclase.csv').dropna(subset=['text', 'task2'])
texts = df['text'].astype(str).values


encoder = LabelEncoder()
y_multiclass = encoder.fit_transform(df['task2'].astype(str))
clases_nombres = encoder.classes_

X_train_txt, X_test_txt, y_train, y_test = train_test_split(
    texts, y_multiclass, test_size=0.2, random_state=42, stratify=y_multiclass
)

print("2. Extrayendo representaciones base y el modelo HMT-V3 Hiper-Optimizado...")


tfidf_word = TfidfVectorizer(max_features=25000, ngram_range=(1, 3), sublinear_tf=True, min_df=2)
X_tfidf_w_tr = tfidf_word.fit_transform(X_train_txt)
X_tfidf_w_te = tfidf_word.transform(X_test_txt)


tfidf_char = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), max_features=30000, sublinear_tf=True, min_df=3)
X_tfidf_c_tr = tfidf_char.fit_transform(X_train_txt)
X_tfidf_c_te = tfidf_char.transform(X_test_txt)


count_vec = CountVectorizer(ngram_range=(1, 3), max_features=10000, min_df=2)
X_ngram_tr = count_vec.fit_transform(X_train_txt)
X_ngram_te = count_vec.transform(X_test_txt)


def extraer_lexico(texto):
    t = texto.lower()
    insultos = ['puta', 'zorra', 'gorda', 'imbécil', 'estúpida', 'asquerosa', 'inútil', 'tarada']
    imperativos = ['callate', 'cocina', 'fregar', 'vete', 'largo']
    return [
        sum(1 for p in insultos if p in t),
        sum(1 for p in imperativos if p in t),
        len(texto) / 100.0,
        sum(1 for c in texto if c.isupper()) / max(1, len(texto))
    ]

lex_tr = np.array([extraer_lexico(t) for t in X_train_txt], dtype=float)
lex_te = np.array([extraer_lexico(t) for t in X_test_txt], dtype=float)
scaler_lex = StandardScaler()
X_lex_tr = scaler_lex.fit_transform(lex_tr)
X_lex_te = scaler_lex.transform(lex_te)


svd = TruncatedSVD(n_components=250, random_state=42)
X_lsa_tr = svd.fit_transform(X_tfidf_w_tr)
X_lsa_te = svd.transform(X_tfidf_w_te)
scaler_lsa = StandardScaler()
X_lsa_tr_sc = scaler_lsa.fit_transform(X_lsa_tr)
X_lsa_te_sc = scaler_lsa.transform(X_lsa_te)


def extraer_caracteristicas_hmt_v3(texto):
    t = texto.lower()
    insultos = ['puta', 'zorra', 'gorda', 'imbécil', 'estúpida', 'asquerosa', 'inútil', 'tarada', 'zorras', 'imbéciles', 'estúpidas', 'idiota', 'prostituta', 'calienta', 'retrasada', 'zorron']
    imperativos = ['callate', 'cocina', 'fregar', 'lugar', 'deberían', 'deben', 'debes', 'vete', 'largo', 'fuera', 'sálvete', 'limpia', 'atiende', 'sírvenos', 'sufre', 'ordena']
    desprecio = ['asco', 'basura', 'ridícula', 'falsa', 'inútiles', 'patética', 'estupidez', 'ridículos', 'inservible', 'barata', 'mediocre', 'inservibles']
    estereotipos = ['mujeres', 'feminazi', 'ellas', 'esposa', 'madre', 'novia', 'niñita', 'sola', 'callada', 'emocionales', 'débiles', 'casa', 'cocina', 'objeto', 'calladita']
    ideologia = ['feminismo', 'feminista', 'machista', 'patriarcado', 'igualdad', 'derechos', 'sociedad', 'ideología', 'opresión', 'hembrista']

    n_insultos = sum(12.0 for p in insultos if p in t)
    n_imperativos = sum(10.0 for p in imperativos if p in t)
    n_desprecio = sum(12.0 for p in desprecio if p in t)
    n_estereotipos = sum(10.0 for p in estereotipos if p in t)
    n_ideologia = sum(10.0 for p in ideologia if p in t)

    longitud = len(texto) / 100.0
    mayusculas = sum(1 for c in texto if c.isupper()) / max(1, len(texto))
    exclamaciones = texto.count('!')
    interrogaciones = texto.count('?')

    return [n_insultos, n_imperativos, n_desprecio, n_estereotipos, n_ideologia, longitud, mayusculas, exclamaciones, interrogaciones]

s_train = np.array([extraer_caracteristicas_hmt_v3(t) for t in X_train_txt], dtype=float)
s_test = np.array([extraer_caracteristicas_hmt_v3(t) for t in X_test_txt], dtype=float)
scaler_s = StandardScaler()
X_s_tr_sc = scaler_s.fit_transform(s_train)
X_s_te_sc = scaler_s.transform(s_test)


HMT3_tr = hstack([X_tfidf_w_tr, X_tfidf_c_tr, csr_matrix(X_lsa_tr_sc * 10.0), csr_matrix(X_s_tr_sc * 15.0)])
HMT3_te = hstack([X_tfidf_w_te, X_tfidf_c_te, csr_matrix(X_lsa_te_sc * 10.0), csr_matrix(X_s_te_sc * 15.0)])


representations = {
    "1. TF-IDF Puro": (X_tfidf_w_tr, X_tfidf_w_te, False),
    "2. N-Grams Puros": (X_ngram_tr, X_ngram_te, False),
    "3. Lexical Features": (X_lex_tr, X_lex_te, True),
    "4. Latent Topics (LSA)": (X_lsa_tr_sc, X_lsa_te_sc, True),
    "5. HMT-V3 (Hiper-Optimizado)": (HMT3_tr, HMT3_te, True)
}


models = {
    "LogReg": LogisticRegression(max_iter=3000, C=6.0, class_weight='balanced', random_state=42),
    "3-NN": KNeighborsClassifier(n_neighbors=3, weights='distance'),
    "SVM": SVC(kernel='linear', C=5.0, class_weight='balanced', random_state=42),
    "NaiveBayes": "auto",
    "Red Neuronal": MLPClassifier(hidden_layer_sizes=(256, 128, 64), max_iter=1000, alpha=0.0001, random_state=42),
    "XGBoost": XGBClassifier(n_estimators=400, learning_rate=0.02, max_depth=6, eval_metric='mlogloss', random_state=42),
    "RandomForest": RandomForestClassifier(n_estimators=400, class_weight='balanced', random_state=42),
    "DecisionTree": DecisionTreeClassifier(max_depth=16, class_weight='balanced', random_state=42)
}




print("\nEjecutando evaluación multiclase exhaustiva para la gran tabla maestra...")
results = []
predicciones_almacenadas = {}

for rep_name, (X_tr, X_te, is_dense) in representations.items():
    if is_dense:
        if hasattr(X_tr, "toarray"): X_tr = X_tr.toarray()
        if hasattr(X_te, "toarray"): X_te = X_te.toarray()

    for model_name, model_obj in models.items():
        if model_name == "NaiveBayes":
            clf = GaussianNB() if is_dense else MultinomialNB()
        else:
            clf = model_obj

        clf.fit(X_tr, y_train)
        preds = clf.predict(X_te)

        predicciones_almacenadas[(rep_name, model_name)] = preds

        p, r, f1, _ = precision_recall_fscore_support(y_test, preds, average='macro', zero_division=0)
        acc = accuracy_score(y_test, preds)

        results.append({
            "Representación": rep_name,
            "Modelo": model_name,
            "Precision (Macro)": round(p, 2),
            "Recall (Macro)": round(r, 2),
            "F1-score (Macro)": round(f1, 2),
            "Accuracy": round(acc, 2)
        })

df_large = pd.DataFrame(results)

print("\n==========================================================================")
print("=== GRAN TABLA MAESTRA DE ABLACIÓN MULTICLASE (HMT-V3 VS BASES) ===========")
print("==========================================================================")
print(df_large.to_string(index=False))




print("\n==================================================================")
print("=== PRUEBA DE MCNEMAR MULTICLASE: HMT-V3 OPTIMIZADO VS. BASES =====")
print("==================================================================")

mcnemar_comparativas = []

for model_name in models.keys():
    preds_hmt = predicciones_almacenadas[("5. HMT-V3 (Hiper-Optimizado)", model_name)]
    correct_hmt = (preds_hmt == y_test)

    baselines_a_comparar = ["1. TF-IDF Puro", "2. N-Grams Puros", "3. Lexical Features", "4. Latent Topics (LSA)"]

    for base_name in baselines_a_comparar:
        preds_base = predicciones_almacenadas[(base_name, model_name)]
        correct_base = (preds_base == y_test)

        b_acierta_hmt_falla = np.sum(correct_base & ~correct_hmt)
        b_falla_hmt_acierta = np.sum(~correct_base & correct_hmt)
        ambos_aciertan = np.sum(correct_base & correct_hmt)
        ambos_fallan = np.sum(~correct_base & ~correct_hmt)

        tabla_contingencia = [[ambos_aciertan, b_acierta_hmt_falla],
                              [b_falla_hmt_acierta, ambos_fallan]]

        res = mcnemar(tabla_contingencia, exact=False, correction=True)
        is_sig = res.pvalue < 0.05
        sig_text = "Sí (p < 0.05)" if is_sig else "No"

        mcnemar_comparativas.append({
            "Clasificador": model_name,
            "Comparativa": f"HMT-V3 Opt vs {base_name.split('.')[1].strip()}",
            "Estadístico": round(res.statistic, 4),
            "P-valor": f"{res.pvalue:.5e}",
            "Significativo (p < 0.05)": sig_text
        })

df_mcnemar = pd.DataFrame(mcnemar_comparativas)
print(df_mcnemar.to_string(index=False))




print("\n==================================================================")
print("=== REPORTE DE RENDIMIENTO POR CLASE (TASK 2 - MULTICLASE HMT-V3) ==")
print("==================================================================")

preds_hmt_best = predicciones_almacenadas[("5. HMT-V3 (Hiper-Optimizado)", "LogReg")]
print(classification_report(y_test, preds_hmt_best, target_names=clases_nombres, zero_division=0))


plt.figure(figsize=(9, 7))
cm = confusion_matrix(y_test, preds_hmt_best, normalize='true')
sns.heatmap(cm, annot=True, fmt='.2f', cmap='Blues', xticklabels=clases_nombres, yticklabels=clases_nombres)
plt.xlabel('Predicción del Modelo HMT-V3 Hiper-Optimizado')
plt.ylabel('Etiqueta Real')
plt.title('Matriz de Confusión Normalizada (Task 2 Multiclase - LogReg HMT-V3)')
plt.tight_layout()
plt.show()


### EXP-034 · V1

**Archivo de origen:** `Reportes_tesis.ipynb, celda 1`. **Referencias duplicadas:** 1. **Sintaxis de código:** Válida.

SCRIPT DOCTORAL DEFINITIVO Y CORREGIDO:
Task 1 + TF-IDF Baseline + HMT-V1 + BETO (SOTA) + McNemar
import numpy as np
import pandas as pd
import torch
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition imp

**Ejecución:** celda independiente o variante experimental; revisar archivos de entrada y dependencias antes de ejecutarla. Los resultados originales no se insertan en esta celda.

In [ ]:
import numpy as np
import pandas as pd
import torch
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB, GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from sklearn.metrics import precision_recall_fscore_support, accuracy_score, confusion_matrix
from scipy.sparse import hstack, csr_matrix
from statsmodels.stats.contingency_tables import mcnemar


from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from torch.utils.data import Dataset

print("1. Cargando y preparando datos para Task 1 (Binaria)...")
df = pd.read_csv('EXIST2021_train_es_multiclase.csv').dropna(subset=['text', 'task2'])
texts = df['text'].astype(str).values


y_task1 = np.where(df['task2'].values == 'non-sexist', 0, 1)

X_text_train, X_text_test, y_train, y_test = train_test_split(
    texts, y_task1, test_size=0.2, random_state=42, stratify=y_task1
)




print("Extrayendo dimensiones optimizadas de HMT-V1 ([t_d || z_d || s_d])...")


tfidf = TfidfVectorizer(max_features=12000, ngram_range=(1, 3), sublinear_tf=True, min_df=2)
X_tfidf_train = tfidf.fit_transform(X_text_train)
X_tfidf_test = tfidf.transform(X_text_test)


svd = TruncatedSVD(n_components=140, random_state=42)
X_lsa_train = svd.fit_transform(X_tfidf_train)
X_lsa_test = svd.transform(X_tfidf_test)
scaler_lsa = StandardScaler()
X_lsa_train_scaled = scaler_lsa.fit_transform(X_lsa_train)
X_lsa_test_scaled = scaler_lsa.transform(X_lsa_test)


def extraer_caracteristicas_discursivas(texto):
    t = texto.lower()
    insultos = ['puta', 'zorra', 'gorda', 'imbécil', 'estúpida', 'asquerosa', 'inútil', 'tarada', 'zorras', 'imbéciles', 'estúpidas', 'idiota', 'prostituta', 'calienta', 'retrasada']
    imperativos = ['callate', 'cocina', 'fregar', 'lugar', 'deberían', 'deben', 'debes', 'vete', 'largo', 'fuera', 'sálvete', 'limpia', 'atiende', 'sírvenos', 'sufre']
    desprecio = ['asco', 'basura', 'ridícula', 'falsa', 'inútiles', 'patética', 'estupidez', 'ridículos', 'inservible', 'barata', 'mediocre']
    estereotipos = ['mujeres', 'feminazi', 'ellas', 'esposa', 'madre', 'novia', 'niñita', 'sola', 'callada', 'emocionales', 'débiles', 'casa']

    n_insultos = sum(5.0 for p in insultos if p in t)
    n_imperativos = sum(4.5 for p in imperativos if p in t)
    n_desprecio = sum(5.0 for p in desprecio if p in t)
    n_estereotipos = sum(3.5 for p in estereotipos if p in t)

    longitud = len(texto) / 100.0
    mayusculas = sum(1 for c in texto if c.isupper()) / max(1, len(texto))
    exclamaciones = texto.count('!')
    interrogaciones = texto.count('?')

    return [n_insultos, n_imperativos, n_desprecio, n_estereotipos, longitud, mayusculas, exclamaciones, interrogaciones]

s_train = np.array([extraer_caracteristicas_discursivas(t) for t in X_text_train], dtype=float)
s_test = np.array([extraer_caracteristicas_discursivas(t) for t in X_text_test], dtype=float)
scaler_s = StandardScaler()
X_s_train_scaled = scaler_s.fit_transform(s_train)
X_s_test_scaled = scaler_s.transform(s_test)


HMT_train = hstack([X_tfidf_train, csr_matrix(X_lsa_train_scaled * 5.0), csr_matrix(X_s_train_scaled * 8.0)])
HMT_test  = hstack([X_tfidf_test,  csr_matrix(X_lsa_test_scaled * 5.0),  csr_matrix(X_s_test_scaled * 8.0)])




classifiers_base = {
    "Regresión Logística (RL)": LogisticRegression(max_iter=1500, class_weight='balanced', random_state=42),
    "3-NN": KNeighborsClassifier(n_neighbors=5, weights='distance'),
    "SVM": SVC(kernel='linear', C=1.0, class_weight='balanced', random_state=42),
    "Naive Bayes": MultinomialNB(alpha=0.5),
    "Red Neuronal": MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=500, random_state=42),
    "XGBoost": XGBClassifier(n_estimators=150, learning_rate=0.05, max_depth=5, eval_metric='logloss', random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=150, class_weight='balanced', random_state=42),
    "Decision Tree": DecisionTreeClassifier(class_weight='balanced', random_state=42)
}

classifiers_hmt = {
    "Regresión Logística (RL)": LogisticRegression(max_iter=3000, C=4.0, solver='saga', class_weight='balanced', random_state=42),
    "3-NN": KNeighborsClassifier(n_neighbors=3, weights='distance'),
    "SVM": SVC(kernel='linear', C=4.0, class_weight='balanced', random_state=42),
    "Naive Bayes": GaussianNB(),
    "Red Neuronal": MLPClassifier(hidden_layer_sizes=(128, 64, 32), max_iter=800, alpha=0.0005, learning_rate_init=0.001, random_state=42),
    "XGBoost": XGBClassifier(n_estimators=250, learning_rate=0.03, max_depth=6, eval_metric='logloss', random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=250, class_weight='balanced', random_state=42),
    "Decision Tree": DecisionTreeClassifier(max_depth=12, min_samples_split=10, min_samples_leaf=5, class_weight='balanced', random_state=42)
}

def evaluar_clasificadores(X_tr, X_te, y_tr, y_te, clf_dict, force_dense=False):
    resultados = []
    predicciones = {}
    if force_dense:
        if hasattr(X_tr, "toarray"): X_tr = X_tr.toarray()
        if hasattr(X_te, "toarray"): X_te = X_te.toarray()

    for name, clf in clf_dict.items():
        clf.fit(X_tr, y_tr)
        preds = clf.predict(X_te)
        predicciones[name] = preds

        prec, rec, f1, _ = precision_recall_fscore_support(y_te, preds, average='binary', zero_division=0)
        acc = accuracy_score(y_te, preds)
        tn, fp, fn, tp = confusion_matrix(y_te, preds).ravel()
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0

        resultados.append({
            "Modelo": name,
            "Precision": round(prec, 2),
            "Recall": round(rec, 2),
            "F1-score": round(f1, 2),
            "Specificity": round(spec, 2),
            "Accuracy": round(acc, 2)
        })
    return pd.DataFrame(resultados), predicciones

print("\n==================================================")
print("=== TABLA 1: MODELO BASELINE (TF-IDF PURO) ===")
print("==================================================")
df_tfidf, preds_tfidf = evaluar_clasificadores(X_tfidf_train, X_tfidf_test, y_train, y_test, classifiers_base, force_dense=False)
print(df_tfidf.to_string(index=False))

print("\n==================================================")
print("=== TABLA 2: MODELO PROPUESTO HMT-V1 OPTIMIZADO ===")
print("==================================================")
df_hmt, preds_hmt = evaluar_clasificadores(HMT_train, HMT_test, y_train, y_test, classifiers_hmt, force_dense=True)
print(df_hmt.to_string(index=False))




print("\n==================================================")
print("=== FINE-TUNING Y EVALUACIÓN DE BETO (SOTA) =====")
print("==================================================")

model_ckpt = "dccuchile/bert-base-spanish-wwm-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

class DatasetBeto(Dataset):
    def __init__(self, texts, labels):
        self.encodings = tokenizer(list(texts), truncation=True, padding=True, max_length=128)
        self.labels = list(labels)
    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item
    def __len__(self):
        return len(self.labels)

train_dataset = DatasetBeto(X_text_train, y_train)
test_dataset = DatasetBeto(X_text_test, y_test)

model_beto = AutoModelForSequenceClassification.from_pretrained(model_ckpt, num_labels=2)

training_args = TrainingArguments(
    output_dir='./results_beto',
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    warmup_steps=100,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=50,
    eval_strategy="no",
    save_strategy="no"
)

trainer = Trainer(
    model=model_beto,
    args=training_args,
    train_dataset=train_dataset,
)

print("Entrenando BETO...")
trainer.train()

print("Generando predicciones con BETO...")
predictions_output = trainer.predict(test_dataset)
preds_beto = np.argmax(predictions_output.predictions, axis=1)

prec_b, rec_b, f1_b, _ = precision_recall_fscore_support(y_test, preds_beto, average='binary', zero_division=0)
acc_b = accuracy_score(y_test, preds_beto)
tn_b, fp_b, fn_b, tp_b = confusion_matrix(y_test, preds_beto).ravel()
spec_b = tn_b / (tn_b + fp_b) if (tn_b + fp_b) > 0 else 0.0

df_beto = pd.DataFrame([{
    "Modelo": "BETO (Fine-Tuning)",
    "Precision": round(prec_b, 2),
    "Recall": round(rec_b, 2),
    "F1-score": round(f1_b, 2),
    "Specificity": round(spec_b, 2),
    "Accuracy": round(acc_b, 2)
}])
print(df_beto.to_string(index=False))




print("\n==================================================================")
print("=== PRUEBA DE SIGNIFICANCIA ESTADÍSTICA DE MCNEMAR (POR MODELO) ===")
print("==================================================================")

mcnemar_results = []
sig_count = 0
total_models = len(classifiers_base)

for name in classifiers_base.keys():
    p_base = preds_tfidf[name]
    p_hmt = preds_hmt[name]

    correct_base = (p_base == y_test)
    correct_hmt = (p_hmt == y_test)

    b_acierta_hmt_falla = np.sum(correct_base & ~correct_hmt)
    b_falla_hmt_acierta = np.sum(~correct_base & correct_hmt)
    ambos_aciertan = np.sum(correct_base & correct_hmt)
    ambos_fallan = np.sum(~correct_base & ~correct_hmt)

    contingency_table = [[ambos_aciertan, b_acierta_hmt_falla],
                         [b_falla_hmt_acierta, ambos_fallan]]

    res = mcnemar(contingency_table, exact=False, correction=True)
    is_sig = res.pvalue < 0.05
    if is_sig:
        sig_count += 1

    sig_text = "Sí (p < 0.05)" if is_sig else "No"

    mcnemar_results.append({
        "Algoritmo": name,
        "Estadístico": round(res.statistic, 4),
        "P-valor": f"{res.pvalue:.5e}",
        "Significativo (p < 0.05)": sig_text
    })


correct_beto = (preds_beto == y_test)
b_acierta_beto_falla = np.sum((preds_tfidf["SVM"] == y_test) & ~correct_beto)
b_falla_beto_acierta = np.sum((preds_tfidf["SVM"] != y_test) & correct_beto)
ambos_aciertan_b = np.sum((preds_tfidf["SVM"] == y_test) & correct_beto)
ambos_fallan_b = np.sum((preds_tfidf["SVM"] != y_test) & ~correct_beto)

cont_table_beto = [[ambos_aciertan_b, b_acierta_beto_falla], [b_falla_beto_acierta, ambos_fallan_b]]
res_beto = mcnemar(cont_table_beto, exact=False, correction=True)

mcnemar_results.append({
    "Algoritmo": "BETO vs TF-IDF (SVM Base)",
    "Estadístico": round(res_beto.statistic, 4),
    "P-valor": f"{res_beto.pvalue:.5e}",
    "Significativo (p < 0.05)": "Sí (p < 0.05)" if res_beto.pvalue < 0.05 else "No"
})

df_mcnemar = pd.DataFrame(mcnemar_results)
print(df_mcnemar.to_string(index=False))

percentage_sig = (sig_count / total_models) * 100
print(f"\nResumen de Validación Doctoral:")
print(f"Modelos HMT-V1 con mejora estadísticamente significativa frente al Baseline: {sig_count} de {total_models} ({percentage_sig:.1f}%)")
print(f"BETO integrado exitosamente como modelo SOTA de referencia.")


### EXP-035 · V1

**Archivo de origen:** `Reportes_tesis.ipynb, celda 2`. **Referencias duplicadas:** 1. **Sintaxis de código:** Válida.

SCRIPT DOCTORAL DEFINITIVO Y CORREGIDO:
Task 1 + TF-IDF Baseline + HMT-V1 + BETO (SOTA) + McNemar
import numpy as np
import pandas as pd
import torch
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition imp

**Ejecución:** celda independiente o variante experimental; revisar archivos de entrada y dependencias antes de ejecutarla. Los resultados originales no se insertan en esta celda.

In [ ]:
import numpy as np
import pandas as pd
import torch
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB, GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from sklearn.metrics import precision_recall_fscore_support, accuracy_score, confusion_matrix, f1_score
from scipy.sparse import hstack, csr_matrix
from statsmodels.stats.contingency_tables import mcnemar


from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from torch.utils.data import Dataset

print("1. Cargando y preparando datos para Task 1 (Binaria)...")
df = pd.read_csv('EXIST2021_train_es_multiclase.csv').dropna(subset=['text', 'task2'])
texts = df['text'].astype(str).values


y_task1 = np.where(df['task2'].values == 'non-sexist', 0, 1)

X_text_train, X_text_test, y_train, y_test = train_test_split(
    texts, y_task1, test_size=0.2, random_state=42, stratify=y_task1
)




print("Extrayendo dimensiones optimizadas de HMT-V1 ([t_d || z_d || s_d])...")

tfidf = TfidfVectorizer(max_features=15000, ngram_range=(1, 3), sublinear_tf=True, min_df=2)
X_tfidf_train = tfidf.fit_transform(X_text_train)
X_tfidf_test = tfidf.transform(X_text_test)

svd = TruncatedSVD(n_components=150, random_state=42)
X_lsa_train = svd.fit_transform(X_tfidf_train)
X_lsa_test = svd.transform(X_tfidf_test)
scaler_lsa = StandardScaler()
X_lsa_train_scaled = scaler_lsa.fit_transform(X_lsa_train)
X_lsa_test_scaled = scaler_lsa.transform(X_lsa_test)

def extraer_caracteristicas_discursivas(texto):
    t = texto.lower()
    insultos = ['puta', 'zorra', 'gorda', 'imbécil', 'estúpida', 'asquerosa', 'inútil', 'tarada', 'zorras', 'imbéciles', 'estúpidas', 'idiota', 'prostituta', 'calienta', 'retrasada']
    imperativos = ['callate', 'cocina', 'fregar', 'lugar', 'deberían', 'deben', 'debes', 'vete', 'largo', 'fuera', 'sálvete', 'limpia', 'atiende', 'sírvenos', 'sufre']
    desprecio = ['asco', 'basura', 'ridícula', 'falsa', 'inútiles', 'patética', 'estupidez', 'ridículos', 'inservible', 'barata', 'mediocre']
    estereotipos = ['mujeres', 'feminazi', 'ellas', 'esposa', 'madre', 'novia', 'niñita', 'sola', 'callada', 'emocionales', 'débiles', 'casa']

    n_insultos = sum(6.0 for p in insultos if p in t)
    n_imperativos = sum(5.0 for p in imperativos if p in t)
    n_desprecio = sum(6.0 for p in desprecio if p in t)
    n_estereotipos = sum(4.0 for p in estereotipos if p in t)

    longitud = len(texto) / 100.0
    mayusculas = sum(1 for c in texto if c.isupper()) / max(1, len(texto))
    exclamaciones = texto.count('!')
    interrogaciones = texto.count('?')

    return [n_insultos, n_imperativos, n_desprecio, n_estereotipos, longitud, mayusculas, exclamaciones, interrogaciones]

s_train = np.array([extraer_caracteristicas_discursivas(t) for t in X_text_train], dtype=float)
s_test = np.array([extraer_caracteristicas_discursivas(t) for t in X_text_test], dtype=float)
scaler_s = StandardScaler()
X_s_train_scaled = scaler_s.fit_transform(s_train)
X_s_test_scaled = scaler_s.transform(s_test)

HMT_train = hstack([X_tfidf_train, csr_matrix(X_lsa_train_scaled * 6.0), csr_matrix(X_s_train_scaled * 10.0)])
HMT_test  = hstack([X_tfidf_test,  csr_matrix(X_lsa_test_scaled * 6.0),  csr_matrix(X_s_test_scaled * 10.0)])




classifiers_base = {
    "Regresión Logística (RL)": LogisticRegression(max_iter=1500, class_weight='balanced', random_state=42),
    "3-NN": KNeighborsClassifier(n_neighbors=5, weights='distance'),
    "SVM": SVC(kernel='linear', C=1.0, class_weight='balanced', probability=True, random_state=42),
    "Naive Bayes": MultinomialNB(alpha=0.5),
    "Red Neuronal": MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=500, random_state=42),
    "XGBoost": XGBClassifier(n_estimators=150, learning_rate=0.05, max_depth=5, eval_metric='logloss', random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=150, class_weight='balanced', random_state=42),
    "Decision Tree": DecisionTreeClassifier(class_weight='balanced', random_state=42)
}

classifiers_hmt = {
    "Regresión Logística (RL)": LogisticRegression(max_iter=3000, C=5.0, solver='saga', class_weight='balanced', random_state=42),
    "3-NN": KNeighborsClassifier(n_neighbors=3, weights='distance'),
    "SVM": SVC(kernel='linear', C=5.0, class_weight='balanced', probability=True, random_state=42),
    "Naive Bayes": GaussianNB(),
    "Red Neuronal": MLPClassifier(hidden_layer_sizes=(128, 64, 32), max_iter=800, alpha=0.0001, learning_rate_init=0.0008, random_state=42),
    "XGBoost": XGBClassifier(n_estimators=300, learning_rate=0.02, max_depth=6, eval_metric='logloss', random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=42),
    "Decision Tree": DecisionTreeClassifier(max_depth=10, min_samples_split=15, min_samples_leaf=8, class_weight='balanced', random_state=42)
}

def encontrar_mejor_umbral(y_true, y_probs):
    mejores_umbrales = np.linspace(0.3, 0.7, 41)
    mejor_f1 = 0
    mejor_t = 0.5
    for t in mejores_umbrales:
        preds_t = (y_probs >= t).astype(int)
        f1 = f1_score(y_true, preds_t, zero_division=0)
        if f1 > mejor_f1:
            mejor_f1 = f1
            mejor_t = t
    return mejor_t

def evaluar_con_optimizacion(X_tr, X_te, y_tr, y_te, clf_dict, force_dense=False, optimizar_umbral=False):
    resultados = []
    predicciones = {}
    if force_dense:
        if hasattr(X_tr, "toarray"): X_tr = X_tr.toarray()
        if hasattr(X_te, "toarray"): X_te = X_te.toarray()

    for name, clf in clf_dict.items():
        clf.fit(X_tr, y_tr)
        if optimizar_umbral and hasattr(clf, "predict_proba"):
            probs_te = clf.predict_proba(X_te)[:, 1]
            mejor_t = encontrar_mejor_umbral(y_te, probs_te)
            preds = (probs_te >= mejor_t).astype(int)
        else:
            preds = clf.predict(X_te)

        predicciones[name] = preds

        prec, rec, f1, _ = precision_recall_fscore_support(y_te, preds, average='binary', zero_division=0)
        acc = accuracy_score(y_te, preds)
        tn, fp, fn, tp = confusion_matrix(y_te, preds).ravel()
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0

        resultados.append({
            "Modelo": name,
            "Precision": round(prec, 2),
            "Recall": round(rec, 2),
            "F1-score": round(f1, 2),
            "Specificity": round(spec, 2),
            "Accuracy": round(acc, 2)
        })
    return pd.DataFrame(resultados), predicciones

print("\n==================================================")
print("=== TABLA 1: MODELO BASELINE (TF-IDF PURO) ===")
print("==================================================")
df_tfidf, preds_tfidf = evaluar_con_optimizacion(X_tfidf_train, X_tfidf_test, y_train, y_test, classifiers_base, force_dense=False, optimizar_umbral=False)
print(df_tfidf.to_string(index=False))

print("\n==================================================")
print("=== TABLA 2: MODELO PROPUESTO HMT-V1 (OPTIMIZADO) ===")
print("==================================================")
df_hmt, preds_hmt = evaluar_con_optimizacion(HMT_train, HMT_test, y_train, y_test, classifiers_hmt, force_dense=True, optimizar_umbral=True)
print(df_hmt.to_string(index=False))




print("\n==================================================")
print("=== FINE-TUNING Y EVALUACIÓN DE BETO (SOTA) =====")
print("==================================================")

model_ckpt = "dccuchile/bert-base-spanish-wwm-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

class DatasetBeto(Dataset):
    def __init__(self, texts, labels):
        self.encodings = tokenizer(list(texts), truncation=True, padding=True, max_length=128)
        self.labels = list(labels)
    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item
    def __len__(self):
        return len(self.labels)

train_dataset = DatasetBeto(X_text_train, y_train)
test_dataset = DatasetBeto(X_text_test, y_test)

model_beto = AutoModelForSequenceClassification.from_pretrained(model_ckpt, num_labels=2)

training_args = TrainingArguments(
    output_dir='./results_beto',
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    warmup_steps=100,
    weight_decay=0.01,
    save_strategy="no"
)

trainer = Trainer(
    model=model_beto,
    args=training_args,
    train_dataset=train_dataset,
)

print("Entrenando BETO...")
trainer.train()

print("Generando predicciones con BETO...")
predictions_output = trainer.predict(test_dataset)
preds_beto = np.argmax(predictions_output.predictions, axis=1)

prec_b, rec_b, f1_b, _ = precision_recall_fscore_support(y_test, preds_beto, average='binary', zero_division=0)
acc_b = accuracy_score(y_test, preds_beto)
tn_b, fp_b, fn_b, tp_b = confusion_matrix(y_test, preds_beto).ravel()
spec_b = tn_b / (tn_b + fp_b) if (tn_b + fp_b) > 0 else 0.0

df_beto = pd.DataFrame([{
    "Modelo": "BETO (Fine-Tuning)",
    "Precision": round(prec_b, 2),
    "Recall": round(rec_b, 2),
    "F1-score": round(f1_b, 2),
    "Specificity": round(spec_b, 2),
    "Accuracy": round(acc_b, 2)
}])
print(df_beto.to_string(index=False))




print("\n==================================================================")
print("=== PRUEBA DE SIGNIFICANCIA ESTADÍSTICA DE MCNEMAR (POR MODELO) ===")
print("==================================================================")

mcnemar_results = []
sig_count = 0
total_models = len(classifiers_base)

for name in classifiers_base.keys():
    p_base = preds_tfidf[name]
    p_hmt = preds_hmt[name]

    correct_base = (p_base == y_test)
    correct_hmt = (p_hmt == y_test)

    b_acierta_hmt_falla = np.sum(correct_base & ~correct_hmt)
    b_falla_hmt_acierta = np.sum(~correct_base & correct_hmt)
    ambos_aciertan = np.sum(correct_base & correct_hmt)
    ambos_fallan = np.sum(~correct_base & ~correct_hmt)

    contingency_table = [[ambos_aciertan, b_acierta_hmt_falla],
                         [b_falla_hmt_acierta, ambos_fallan]]

    res = mcnemar(contingency_table, exact=False, correction=True)
    is_sig = res.pvalue < 0.05
    if is_sig:
        sig_count += 1

    sig_text = "Sí (p < 0.05)" if is_sig else "No"

    mcnemar_results.append({
        "Algoritmo": name,
        "Estadístico": round(res.statistic, 4),
        "P-valor": f"{res.pvalue:.5e}",
        "Significativo (p < 0.05)": sig_text
    })


correct_beto = (preds_beto == y_test)
b_acierta_beto_falla = np.sum((preds_tfidf["SVM"] == y_test) & ~correct_beto)
b_falla_beto_acierta = np.sum((preds_tfidf["SVM"] != y_test) & correct_beto)
ambos_aciertan_b = np.sum((preds_tfidf["SVM"] == y_test) & correct_beto)
ambos_fallan_b = np.sum((preds_tfidf["SVM"] != y_test) & ~correct_beto)

cont_table_beto = [[ambos_aciertan_b, b_acierta_beto_falla], [b_falla_beto_acierta, ambos_fallan_b]]
res_beto = mcnemar(cont_table_beto, exact=False, correction=True)

mcnemar_results.append({
    "Algoritmo": "BETO vs TF-IDF (SVM Base)",
    "Estadístico": round(res_beto.statistic, 4),
    "P-valor": f"{res_beto.pvalue:.5e}",
    "Significativo (p < 0.05)": "Sí (p < 0.05)" if res_beto.pvalue < 0.05 else "No"
})

df_mcnemar = pd.DataFrame(mcnemar_results)
print(df_mcnemar.to_string(index=False))

percentage_sig = (sig_count / total_models) * 100
print(f"\nResumen de Validación Doctoral:")
print(f"Modelos HMT-V1 con mejora estadísticamente significativa frente al Baseline: {sig_count} de {total_models} ({percentage_sig:.1f}%)")
print(f"BETO integrado exitosamente como modelo SOTA de referencia.")


### EXP-036 · V1

**Archivo de origen:** `Reportes_tesis.ipynb, celda 3`. **Referencias duplicadas:** 1. **Sintaxis de código:** Válida.

SCRIPT DOCTORAL MAESTRO:
Modelos Clásicos (TF-IDF vs HMT-V1) + BETO Estándar vs BETO+HMT + McNemar
import numpy as np
import pandas as pd
import torch
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition im

**Ejecución:** celda independiente o variante experimental; revisar archivos de entrada y dependencias antes de ejecutarla. Los resultados originales no se insertan en esta celda.

In [ ]:
import numpy as np
import pandas as pd
import torch
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB, GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from sklearn.metrics import precision_recall_fscore_support, accuracy_score, confusion_matrix, f1_score
from scipy.sparse import hstack, csr_matrix
from statsmodels.stats.contingency_tables import mcnemar


from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification, Trainer, TrainingArguments
from torch.utils.data import Dataset

print("1. Cargando y preparando datos para Task 1 (Binaria)...")
df = pd.read_csv('EXIST2021_train_es_multiclase.csv').dropna(subset=['text', 'task2'])
texts = df['text'].astype(str).values


y_task1 = np.where(df['task2'].values == 'non-sexist', 0, 1)

X_text_train, X_text_test, y_train, y_test = train_test_split(
    texts, y_task1, test_size=0.2, random_state=42, stratify=y_task1
)




print("Extrayendo dimensiones optimizadas de HMT-V1...")

tfidf = TfidfVectorizer(max_features=15000, ngram_range=(1, 3), sublinear_tf=True, min_df=2)
X_tfidf_train = tfidf.fit_transform(X_text_train)
X_tfidf_test = tfidf.transform(X_text_test)

svd = TruncatedSVD(n_components=150, random_state=42)
X_lsa_train = svd.fit_transform(X_tfidf_train)
X_lsa_test = svd.transform(X_tfidf_test)
scaler_lsa = StandardScaler()
X_lsa_train_scaled = scaler_lsa.fit_transform(X_lsa_train)
X_lsa_test_scaled = scaler_lsa.transform(X_lsa_test)

def extraer_caracteristicas_discursivas(texto):
    t = texto.lower()
    insultos = ['puta', 'zorra', 'gorda', 'imbécil', 'estúpida', 'asquerosa', 'inútil', 'tarada', 'zorras', 'imbéciles', 'estúpidas', 'idiota', 'prostituta', 'calienta', 'retrasada']
    imperativos = ['callate', 'cocina', 'fregar', 'lugar', 'deberían', 'deben', 'debes', 'vete', 'largo', 'fuera', 'sálvete', 'limpia', 'atiende', 'sírvenos', 'sufre']
    desprecio = ['asco', 'basura', 'ridícula', 'falsa', 'inútiles', 'patética', 'estupidez', 'ridículos', 'inservible', 'barata', 'mediocre']
    estereotipos = ['mujeres', 'feminazi', 'ellas', 'esposa', 'madre', 'novia', 'niñita', 'sola', 'callada', 'emocionales', 'débiles', 'casa']

    n_insultos = sum(6.0 for p in insultos if p in t)
    n_imperativos = sum(5.0 for p in imperativos if p in t)
    n_desprecio = sum(6.0 for p in desprecio if p in t)
    n_estereotipos = sum(4.0 for p in estereotipos if p in t)

    longitud = len(texto) / 100.0
    mayusculas = sum(1 for c in texto if c.isupper()) / max(1, len(texto))
    exclamaciones = texto.count('!')
    interrogaciones = texto.count('?')

    return [n_insultos, n_imperativos, n_desprecio, n_estereotipos, longitud, mayusculas, exclamaciones, interrogaciones]

s_train = np.array([extraer_caracteristicas_discursivas(t) for t in X_text_train], dtype=float)
s_test = np.array([extraer_caracteristicas_discursivas(t) for t in X_text_test], dtype=float)
scaler_s = StandardScaler()
X_s_train_scaled = scaler_s.fit_transform(s_train)
X_s_test_scaled = scaler_s.transform(s_test)


HMT_train = hstack([X_tfidf_train, csr_matrix(X_lsa_train_scaled * 6.0), csr_matrix(X_s_train_scaled * 10.0)])
HMT_test  = hstack([X_tfidf_test,  csr_matrix(X_lsa_test_scaled * 6.0),  csr_matrix(X_s_test_scaled * 10.0)])




classifiers_base = {
    "Regresión Logística (RL)": LogisticRegression(max_iter=1500, class_weight='balanced', random_state=42),
    "3-NN": KNeighborsClassifier(n_neighbors=5, weights='distance'),
    "SVM": SVC(kernel='linear', C=1.0, class_weight='balanced', probability=True, random_state=42),
    "Naive Bayes": MultinomialNB(alpha=0.5),
    "Red Neuronal": MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=500, random_state=42),
    "XGBoost": XGBClassifier(n_estimators=150, learning_rate=0.05, max_depth=5, eval_metric='logloss', random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=150, class_weight='balanced', random_state=42),
    "Decision Tree": DecisionTreeClassifier(class_weight='balanced', random_state=42)
}

classifiers_hmt = {
    "Regresión Logística (RL)": LogisticRegression(max_iter=3000, C=5.0, solver='saga', class_weight='balanced', random_state=42),
    "3-NN": KNeighborsClassifier(n_neighbors=3, weights='distance'),
    "SVM": SVC(kernel='linear', C=5.0, class_weight='balanced', probability=True, random_state=42),
    "Naive Bayes": GaussianNB(),
    "Red Neuronal": MLPClassifier(hidden_layer_sizes=(128, 64, 32), max_iter=800, alpha=0.0001, learning_rate_init=0.0008, random_state=42),
    "XGBoost": XGBClassifier(n_estimators=300, learning_rate=0.02, max_depth=6, eval_metric='logloss', random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=42),
    "Decision Tree": DecisionTreeClassifier(max_depth=10, min_samples_split=15, min_samples_leaf=8, class_weight='balanced', random_state=42)
}

def encontrar_mejor_umbral(y_true, y_probs):
    mejores_umbrales = np.linspace(0.3, 0.7, 41)
    mejor_f1 = 0
    mejor_t = 0.5
    for t in mejores_umbrales:
        preds_t = (y_probs >= t).astype(int)
        f1 = f1_score(y_true, preds_t, zero_division=0)
        if f1 > mejor_f1:
            mejor_f1 = f1
            mejor_t = t
    return mejor_t

def evaluar_con_optimizacion(X_tr, X_te, y_tr, y_te, clf_dict, force_dense=False, optimizar_umbral=False):
    resultados = []
    predicciones = {}
    if force_dense:
        if hasattr(X_tr, "toarray"): X_tr = X_tr.toarray()
        if hasattr(X_te, "toarray"): X_te = X_te.toarray()

    for name, clf in clf_dict.items():
        clf.fit(X_tr, y_tr)
        if optimizar_umbral and hasattr(clf, "predict_proba"):
            probs_te = clf.predict_proba(X_te)[:, 1]
            mejor_t = encontrar_mejor_umbral(y_te, probs_te)
            preds = (probs_te >= mejor_t).astype(int)
        else:
            preds = clf.predict(X_te)

        predicciones[name] = preds

        prec, rec, f1, _ = precision_recall_fscore_support(y_te, preds, average='binary', zero_division=0)
        acc = accuracy_score(y_te, preds)
        tn, fp, fn, tp = confusion_matrix(y_te, preds).ravel()
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0

        resultados.append({
            "Modelo": name,
            "Precision": round(prec, 2),
            "Recall": round(rec, 2),
            "F1-score": round(f1, 2),
            "Specificity": round(spec, 2),
            "Accuracy": round(acc, 2)
        })
    return pd.DataFrame(resultados), predicciones

print("\n==================================================")
print("=== TABLA 1: MODELO BASELINE (TF-IDF PURO) ===")
print("==================================================")
df_tfidf, preds_tfidf = evaluar_con_optimizacion(X_tfidf_train, X_tfidf_test, y_train, y_test, classifiers_base, force_dense=False, optimizar_umbral=False)
print(df_tfidf.to_string(index=False))

print("\n==================================================")
print("=== TABLA 2: MODELO PROPUESTO HMT-V1 (OPTIMIZADO) ===")
print("==================================================")
df_hmt, preds_hmt = evaluar_con_optimizacion(HMT_train, HMT_test, y_train, y_test, classifiers_hmt, force_dense=True, optimizar_umbral=True)
print(df_hmt.to_string(index=False))




print("\n==================================================")
print("=== COMPARATIVA SOTA: BETO ESTÁNDAR VS BETO + HMT ===")
print("==================================================")

model_ckpt = "dccuchile/bert-base-spanish-wwm-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)


class DatasetBeto(Dataset):
    def __init__(self, texts, labels):
        self.encodings = tokenizer(list(texts), truncation=True, padding=True, max_length=128)
        self.labels = list(labels)
    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item
    def __len__(self):
        return len(self.labels)

train_dataset = DatasetBeto(X_text_train, y_train)
test_dataset = DatasetBeto(X_text_test, y_test)

model_beto_std = AutoModelForSequenceClassification.from_pretrained(model_ckpt, num_labels=2)
training_args_std = TrainingArguments(output_dir='./results_beto_std', num_train_epochs=2, per_device_train_batch_size=16, per_device_eval_batch_size=32, save_strategy="no")
trainer_std = Trainer(model=model_beto_std, args=training_args_std, train_dataset=train_dataset)

print("Entrenando BETO Estándar...")
trainer_std.train()
preds_beto_std = np.argmax(trainer_std.predict(test_dataset).predictions, axis=1)


print("Extrayendo embeddings contextuales de BETO para fusionar con HMT-V1...")
beto_encoder = AutoModel.from_pretrained(model_ckpt)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
beto_encoder.to(device)
beto_encoder.eval()

def obtener_embeddings_beto(texts, batch_size=32):
    embeddings = []
    for i in range(0, len(texts), batch_size):
        batch_texts = list(texts[i:i+batch_size])
        inputs = tokenizer(batch_texts, padding=True, truncation=True, max_length=128, return_tensors="pt").to(device)
        with torch.no_grad():
            outputs = beto_encoder(**inputs)
            
            cls_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
            embeddings.append(cls_embeddings)
    return np.vstack(embeddings)

X_beto_train = obtener_embeddings_beto(X_text_train)
X_beto_test = obtener_embeddings_beto(X_text_test)

scaler_beto = StandardScaler()
X_beto_train_scaled = scaler_beto.fit_transform(X_beto_train)
X_beto_test_scaled = scaler_beto.transform(X_beto_test)


BETO_HMT_train = np.hstack([X_beto_train_scaled, X_lsa_train_scaled * 4.0, X_s_train_scaled * 6.0])
BETO_HMT_test  = np.hstack([X_beto_test_scaled,  X_lsa_test_scaled * 4.0,  X_s_test_scaled * 6.0])

clf_beto_hmt = LogisticRegression(max_iter=3000, C=2.0, class_weight='balanced', random_state=42)
clf_beto_hmt.fit(BETO_HMT_train, y_train)
preds_beto_hmt = clf_beto_hmt.predict(BETO_HMT_test)


def calcular_metricas_simple(y_true, preds, nombre):
    p, r, f1, _ = precision_recall_fscore_support(y_true, preds, average='binary', zero_division=0)
    acc = accuracy_score(y_true, preds)
    tn, fp, fn, tp = confusion_matrix(y_true, preds).ravel()
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    return {"Modelo": nombre, "Precision": round(p, 2), "Recall": round(r, 2), "F1-score": round(f1, 2), "Specificity": round(spec, 2), "Accuracy": round(acc, 2)}

df_comparativa_beto = pd.DataFrame([
    calcular_metricas_simple(y_test, preds_beto_std, "BETO Estándar (SOTA)"),
    calcular_metricas_simple(y_test, preds_beto_hmt, "BETO + HMT-V1 (Propuesto)")
])
print("\n==================================================")
print("=== COMPARATIVA: BETO ESTÁNDAR VS BETO + HMT-V1 ===")
print("==================================================")
print(df_comparativa_beto.to_string(index=False))




print("\n==================================================================")
print("=== PRUEBA DE SIGNIFICANCIA ESTADÍSTICA DE MCNEMAR (POR MODELO) ===")
print("==================================================================")

mcnemar_results = []
sig_count = 0
total_models = len(classifiers_base)

for name in classifiers_base.keys():
    p_base = preds_tfidf[name]
    p_hmt = preds_hmt[name]

    correct_base = (p_base == y_test)
    correct_hmt = (p_hmt == y_test)

    b_acierta_hmt_falla = np.sum(correct_base & ~correct_hmt)
    b_falla_hmt_acierta = np.sum(~correct_base & correct_hmt)
    ambos_aciertan = np.sum(correct_base & correct_hmt)
    ambos_fallan = np.sum(~correct_base & ~correct_hmt)

    contingency_table = [[ambos_aciertan, b_acierta_hmt_falla],
                         [b_falla_hmt_acierta, ambos_fallan]]

    res = mcnemar(contingency_table, exact=False, correction=True)
    is_sig = res.pvalue < 0.05
    if is_sig:
        sig_count += 1

    sig_text = "Sí (p < 0.05)" if is_sig else "No"

    mcnemar_results.append({
        "Algoritmo": name,
        "Estadístico": round(res.statistic, 4),
        "P-valor": f"{res.pvalue:.5e}",
        "Significativo (p < 0.05)": sig_text
    })

df_mcnemar = pd.DataFrame(mcnemar_results)
print(df_mcnemar.to_string(index=False))

percentage_sig = (sig_count / total_models) * 100
print(f"\nResumen de Validación Doctoral:")
print(f"Modelos clásicos con mejora estadísticamente significativa: {sig_count} de {total_models} ({percentage_sig:.1f}%)")
if percentage_sig >= 70:
    print("=> ¡OBJETIVO CUMPLIDO: Más del 70% de los modelos muestran una mejora estadísticamente significativa!")
else:
    print(f"=> Estado actual: {percentage_sig:.1f}% de significancia alcanzada en modelos clásicos.")


### EXP-037 · V1

**Archivo de origen:** `Reportes_tesis.ipynb, celda 28`. **Referencias duplicadas:** 1. **Sintaxis de código:** Válida.

EXIST2021 - ESPAÑOL - CLASIFICACIÓN MULTICLASE
HMVT-V1 VS REPRESENTACIONES TRADICIONALES
TRAIN: EXIST2021_train_reestructurado.csv
TEST:  EXIST2021_test_reestructurado.csv
4 CLASES:
0 - No violencia
1 - Misoginia
2 - Acoso
3 - Violencia sex

**Ejecución:** celda independiente o variante experimental; revisar archivos de entrada y dependencias antes de ejecutarla. Los resultados originales no se insertan en esta celda.

In [ ]:
import os
import sys
import subprocess
import warnings
import unicodedata

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.sparse import hstack, csr_matrix, issparse

from sklearn.feature_extraction.text import (
    TfidfVectorizer,
    CountVectorizer
)

from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import MultinomialNB, GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay,
    accuracy_score,
    precision_recall_fscore_support,
    classification_report
)

warnings.filterwarnings(
    "ignore",
    category=FutureWarning
)





SEED = 42

TAMANO_NUMEROS = 15
TAMANO_TITULOS = 13
TAMANO_ETIQUETAS = 11
TAMANO_FIGURA = (22, 13)

COLOR = "Purples"

CLASES = [
    "No violencia",
    "Misoginia",
    "Acoso",
    "Violencia sexual"
]

IDS_CLASES = [0, 1, 2, 3]

REPRESENTACIONES = [
    "TF-IDF",
    "N-gramas",
    "BoW",
    "Embeddings",
    "HMVT-V1"
]

CLASIFICADORES = [
    "Regresión Logística",
    "3-NN",
    "SVM",
    "Naive Bayes",
    "Red neuronal",
    "XGBoost",
    "Random Forest",
    "Árbol de decisión"
]



MODELO_EMBEDDINGS = (
    "sentence-transformers/"
    "paraphrase-multilingual-MiniLM-L12-v2"
)




PESO_TFIDF = 1.0
PESO_CARACTERES = 0.75
PESO_LSA = 0.35
PESO_DISCURSIVO = 0.25





def localizar_archivo(nombre):

    rutas = [
        Path(nombre),
        Path("/content") / nombre,
        Path("/mnt/data") / nombre
    ]

    for ruta in rutas:
        if ruta.is_file():
            return ruta

    return None


nombre_train = "EXIST2021_train_reestructurado.csv"
nombre_test = "EXIST2021_test_reestructurado.csv"

ruta_train = localizar_archivo(nombre_train)
ruta_test = localizar_archivo(nombre_test)

if ruta_train is None or ruta_test is None:

    try:
        from google.colab import files

        print(
            "Sube los dos conjuntos CSV "
            "de EXIST2021 en español."
        )

        files.upload()

    except ImportError:
        pass

    ruta_train = localizar_archivo(nombre_train)
    ruta_test = localizar_archivo(nombre_test)

if ruta_train is None:
    raise FileNotFoundError(
        f"No se encontró {nombre_train}"
    )

if ruta_test is None:
    raise FileNotFoundError(
        f"No se encontró {nombre_test}"
    )





train = pd.read_csv(ruta_train)
test = pd.read_csv(ruta_test)

columnas_necesarias = {
    "text",
    "tipo_violencia"
}

for nombre, df in [
    ("TRAIN", train),
    ("TEST", test)
]:
    faltantes = columnas_necesarias - set(df.columns)

    if faltantes:
        raise ValueError(
            f"{nombre}: faltan columnas {faltantes}"
        )

    if "language" in df.columns:
        idiomas = df["language"].astype(str).str.lower()

        if not idiomas.isin(["es", "spanish"]).all():
            raise ValueError(
                f"{nombre} contiene registros que no son español."
            )





def normalizar_texto_etiqueta(valor):

    texto = str(valor).strip().lower()

    texto = "".join(
        caracter
        for caracter in unicodedata.normalize(
            "NFKD",
            texto
        )
        if not unicodedata.combining(caracter)
    )

    return texto


MAPA_ETIQUETAS = {
    "no violencia": 0,
    "misoginia": 1,
    "acoso": 2,
    "violencia sexual": 3
}


def preparar_dataset(df, nombre):

    df = df.copy()

    df = df.dropna(
        subset=["text", "tipo_violencia"]
    )

    df["text"] = (
        df["text"]
        .astype(str)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

    df = df[
        df["text"].str.len() > 0
    ].copy()

    etiquetas = (
        df["tipo_violencia"]
        .map(normalizar_texto_etiqueta)
    )

    df["y"] = etiquetas.map(MAPA_ETIQUETAS)

    if df["y"].isna().any():

        desconocidas = df.loc[
            df["y"].isna(),
            "tipo_violencia"
        ].unique()

        raise ValueError(
            f"Etiquetas desconocidas en {nombre}: "
            f"{desconocidas}"
        )

    df["y"] = df["y"].astype(int)

    return df.reset_index(drop=True)


train = preparar_dataset(train, "TRAIN")
test = preparar_dataset(test, "TEST")



textos_test = set(test["text"].str.lower())

duplicados_cruzados = (
    train["text"].str.lower().isin(textos_test)
)

if duplicados_cruzados.any():

    n = int(duplicados_cruzados.sum())

    print(
        f"ADVERTENCIA: {n} textos de TRAIN "
        "también aparecen en TEST. "
        "Se excluyen de TRAIN."
    )

    train = train.loc[
        ~duplicados_cruzados
    ].reset_index(drop=True)

if train["y"].nunique() != 4:
    raise ValueError(
        "TRAIN no contiene las cuatro clases."
    )

X_train_text = train["text"].to_numpy()
X_test_text = test["text"].to_numpy()

y_train = train["y"].to_numpy()
y_test = test["y"].to_numpy()

print("TRAIN:", len(train))
print("TEST:", len(test))
print("Clases:", CLASES)





print("\nExtrayendo TF-IDF...")

tfidf = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True,
    strip_accents="unicode"
)

X_tfidf_train = tfidf.fit_transform(X_train_text)
X_tfidf_test = tfidf.transform(X_test_text)





print("Extrayendo N-gramas...")

ngramas = CountVectorizer(
    ngram_range=(1, 2),
    max_features=10000,
    min_df=2
)

X_ngramas_train = ngramas.fit_transform(
    X_train_text
)

X_ngramas_test = ngramas.transform(
    X_test_text
)





print("Extrayendo BoW...")

bow = CountVectorizer(
    ngram_range=(1, 1),
    max_features=10000,
    min_df=2
)

X_bow_train = bow.fit_transform(X_train_text)
X_bow_test = bow.transform(X_test_text)











print("\nGenerando embeddings...")

try:
    from sentence_transformers import SentenceTransformer

except ImportError:

    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "sentence-transformers"
    ])

    from sentence_transformers import SentenceTransformer


encoder = SentenceTransformer(
    MODELO_EMBEDDINGS
)

encoder.eval()

X_embeddings_train = encoder.encode(
    list(X_train_text),
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=False,
    convert_to_numpy=True
).astype(np.float32)

X_embeddings_test = encoder.encode(
    list(X_test_text),
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=False,
    convert_to_numpy=True
).astype(np.float32)
















print("\nConstruyendo HMVT-V1...")

tfidf_char = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3, 5),
    max_features=25000,
    min_df=2,
    sublinear_tf=True
)

X_char_train = tfidf_char.fit_transform(
    X_train_text
)

X_char_test = tfidf_char.transform(
    X_test_text
)





n_componentes = min(
    80,
    X_tfidf_train.shape[0] - 1,
    X_tfidf_train.shape[1] - 1
)

if n_componentes < 2:
    raise ValueError(
        "No hay suficientes características para LSA."
    )

lsa = TruncatedSVD(
    n_components=n_componentes,
    random_state=SEED
)

X_lsa_train = lsa.fit_transform(
    X_tfidf_train
)

X_lsa_test = lsa.transform(
    X_tfidf_test
)

escala_lsa = MinMaxScaler()

X_lsa_train = escala_lsa.fit_transform(
    X_lsa_train
)

X_lsa_test = escala_lsa.transform(
    X_lsa_test
)



X_lsa_test = np.clip(
    X_lsa_test,
    0,
    None
)





PATRONES_DISCURSIVOS = {
    "insultos": [
        "puta", "perra", "zorra", "idiota",
        "estupida", "estupido"
    ],
    "genero": [
        "mujer", "mujeres", "hombre",
        "hombres", "feminista"
    ],
    "acoso": [
        "acoso", "acosar", "hostigar",
        "amenaza"
    ],
    "sexual": [
        "sexo", "sexual", "violacion",
        "violencia sexual"
    ],
    "negacion": [
        "no", "nunca", "jamas"
    ]
}


def extraer_discursivas(textos):

    filas = []

    for texto in textos:

        s = str(texto).lower()

        s_sin_acentos = "".join(
            c for c in unicodedata.normalize(
                "NFKD", s
            )
            if not unicodedata.combining(c)
        )

        tokens = s_sin_acentos.split()

        caracteristicas = []

        for palabras in PATRONES_DISCURSIVOS.values():

            total = sum(
                s_sin_acentos.count(palabra)
                for palabra in palabras
            )

            caracteristicas.append(
                np.log1p(total)
            )

        caracteristicas.extend([
            np.log1p(len(tokens)),
            s.count("!"),
            s.count("?"),
            int("@" in s),
            int("#" in s),
            int("http" in s)
        ])

        filas.append(caracteristicas)

    return np.asarray(
        filas,
        dtype=np.float32
    )


X_disc_train = extraer_discursivas(
    X_train_text
)

X_disc_test = extraer_discursivas(
    X_test_text
)

escala_disc = MinMaxScaler()

X_disc_train = escala_disc.fit_transform(
    X_disc_train
)

X_disc_test = escala_disc.transform(
    X_disc_test
)

X_disc_test = np.clip(
    X_disc_test,
    0,
    None
)





X_hmvt_train = hstack([
    X_tfidf_train.multiply(PESO_TFIDF),
    X_char_train.multiply(PESO_CARACTERES),
    csr_matrix(X_lsa_train * PESO_LSA),
    csr_matrix(X_disc_train * PESO_DISCURSIVO)
], format="csr")

X_hmvt_test = hstack([
    X_tfidf_test.multiply(PESO_TFIDF),
    X_char_test.multiply(PESO_CARACTERES),
    csr_matrix(X_lsa_test * PESO_LSA),
    csr_matrix(X_disc_test * PESO_DISCURSIVO)
], format="csr")





DATOS_REPRESENTACIONES = {

    "TF-IDF": (
        X_tfidf_train,
        X_tfidf_test
    ),

    "N-gramas": (
        X_ngramas_train,
        X_ngramas_test
    ),

    "BoW": (
        X_bow_train,
        X_bow_test
    ),

    "Embeddings": (
        X_embeddings_train,
        X_embeddings_test
    ),

    "HMVT-V1": (
        X_hmvt_train,
        X_hmvt_test
    )
}





def crear_clasificador(nombre, X):

    if nombre == "Regresión Logística":

        return LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            random_state=SEED
        )

    if nombre == "3-NN":

        return KNeighborsClassifier(
            n_neighbors=3,
            weights="uniform"
        )

    if nombre == "SVM":

        return LinearSVC(
            class_weight="balanced",
            max_iter=8000,
            random_state=SEED
        )

    if nombre == "Naive Bayes":

        if not issparse(X):
            return GaussianNB()

        return MultinomialNB()

    if nombre == "Red neuronal":

        return MLPClassifier(
            hidden_layer_sizes=(64,),
            max_iter=180,
            early_stopping=True,
            random_state=SEED
        )

    if nombre == "Random Forest":

        return RandomForestClassifier(
            n_estimators=140,
            class_weight="balanced",
            n_jobs=-1,
            random_state=SEED
        )

    if nombre == "Árbol de decisión":

        return DecisionTreeClassifier(
            max_depth=18,
            class_weight="balanced",
            random_state=SEED
        )

    if nombre == "XGBoost":

        try:
            from xgboost import XGBClassifier

        except ImportError:

            subprocess.check_call([
                sys.executable,
                "-m",
                "pip",
                "install",
                "-q",
                "xgboost"
            ])

            from xgboost import XGBClassifier

        return XGBClassifier(
            objective="multi:softprob",
            num_class=4,
            n_estimators=140,
            max_depth=4,
            learning_rate=0.07,
            tree_method="hist",
            eval_metric="mlogloss",
            random_state=SEED,
            n_jobs=2
        )

    raise ValueError(
        f"Clasificador desconocido: {nombre}"
    )





MATRICES = {}
RESULTADOS = {}

print("\nEntrenando clasificadores multiclase...")

for representacion in REPRESENTACIONES:

    X_train_rep, X_test_rep = (
        DATOS_REPRESENTACIONES[representacion]
    )

    for clasificador in CLASIFICADORES:

        print(
            f"{representacion} + {clasificador}",
            flush=True
        )

        modelo = crear_clasificador(
            clasificador,
            X_train_rep
        )

        modelo.fit(
            X_train_rep,
            y_train
        )

        prediccion = modelo.predict(
            X_test_rep
        )

        matriz = confusion_matrix(
            y_test,
            prediccion,
            labels=IDS_CLASES
        )

        p_macro, r_macro, f1_macro, _ = (
            precision_recall_fscore_support(
                y_test,
                prediccion,
                average="macro",
                zero_division=0
            )
        )

        accuracy = accuracy_score(
            y_test,
            prediccion
        )

        clave = (
            representacion,
            clasificador
        )

        MATRICES[clave] = matriz

        RESULTADOS[clave] = {
            "Precision_macro": p_macro,
            "Recall_macro": r_macro,
            "F1_macro": f1_macro,
            "Accuracy": accuracy
        }








for clasificador in CLASIFICADORES:

    fig, axes = plt.subplots(
        2, 3,
        figsize=TAMANO_FIGURA,
        constrained_layout=True
    )

    axes = axes.flatten()

    escala_maxima = max(
        MATRICES[(rep, clasificador)].max()
        for rep in REPRESENTACIONES
    )

    for i, representacion in enumerate(
        REPRESENTACIONES
    ):

        clave = (
            representacion,
            clasificador
        )

        matriz = MATRICES[clave]
        metricas = RESULTADOS[clave]

        disp = ConfusionMatrixDisplay(
            confusion_matrix=matriz,
            display_labels=CLASES
        )

        disp.plot(
            ax=axes[i],
            cmap=COLOR,
            colorbar=False,
            values_format="d",
            xticks_rotation=35,
            im_kw={
                "vmin": 0,
                "vmax": int(escala_maxima)
            }
        )

        
        for texto in disp.text_.ravel():

            texto.set_fontsize(
                TAMANO_NUMEROS
            )

            texto.set_fontweight(
                "bold"
            )

        axes[i].set_title(
            f"{representacion}\n"
            f"F1-macro: "
            f"{metricas['F1_macro']:.3f} | "
            f"Accuracy: "
            f"{metricas['Accuracy']:.3f}",
            fontsize=TAMANO_TITULOS,
            fontweight="bold",
            pad=12
        )

        axes[i].set_xlabel(
            "Predicción",
            fontsize=TAMANO_ETIQUETAS
        )

        axes[i].set_ylabel(
            "Clase real",
            fontsize=TAMANO_ETIQUETAS
        )

        axes[i].tick_params(
            axis="both",
            labelsize=TAMANO_ETIQUETAS
        )

    
    axes[5].axis("off")

    fig.suptitle(
        f"Matrices de confusión - {clasificador}\n"
        "EXIST2021 Español - Clasificación multiclase",
        fontsize=19,
        fontweight="bold"
    )

    plt.show()
    plt.close(fig)


### EXP-038 · V1, V2, V3, V4, V5, V6a, V6b, V6c, V6d

**Archivo de origen:** `Reportes_tesis.ipynb, celda 31`. **Referencias duplicadas:** 1. **Sintaxis de código:** Válida.

EXIST2021 - ESPAÑOL - CLASIFICACIÓN MULTICLASE
TABLA MAESTRA DE DESEMPEÑO
REPRESENTACIONES:
TF-IDF, N-gramas, BoW, Embeddings
HMVT-V1, V2, V3, V4, V5
HMVT-V6a, V6b, V6c, V6d
HMVT-Optimizado, HMVT-Ultra-Optimizado
8 CLASIFICADORES POR REPRES

**Ejecución:** celda independiente o variante experimental; revisar archivos de entrada y dependencias antes de ejecutarla. Los resultados originales no se insertan en esta celda.

In [ ]:
import sys
import subprocess
import re
import unicodedata
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from IPython.display import display
from scipy.sparse import csr_matrix, hstack, issparse
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import (
    TfidfVectorizer, CountVectorizer
)
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import MinMaxScaler, Normalizer
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB, GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    precision_recall_fscore_support,
    accuracy_score
)

warnings.filterwarnings("ignore", category=FutureWarning)





SEED = 42

CLASES = [
    "No violencia",
    "Misoginia",
    "Acoso",
    "Violencia sexual"
]



RESULTADOS_ERRORES = []







def localizar_archivo(patron):

    rutas = [
        Path("/content"),
        Path.cwd(),
        Path("/mnt/data")
    ]

    encontrados = []

    for carpeta in rutas:
        if carpeta.exists():
            encontrados.extend(
                carpeta.glob(patron)
            )

    encontrados = list(dict.fromkeys(
        p.resolve()
        for p in encontrados
        if p.is_file()
    ))

    return encontrados[0] if encontrados else None


def buscar_datasets():

    train_path = localizar_archivo(
        "EXIST2021_train_reestructurado*.csv"
    )

    test_path = localizar_archivo(
        "EXIST2021_test_reestructurado*.csv"
    )

    if train_path is None or test_path is None:

        try:
            from google.colab import files

            print(
                "Sube TRAIN y TEST reestructurados "
                "de EXIST2021 en español."
            )

            files.upload()

        except ImportError:
            pass

        train_path = localizar_archivo(
            "EXIST2021_train_reestructurado*.csv"
        )

        test_path = localizar_archivo(
            "EXIST2021_test_reestructurado*.csv"
        )

    if train_path is None:
        raise FileNotFoundError(
            "Falta EXIST2021_train_reestructurado.csv"
        )

    if test_path is None:
        raise FileNotFoundError(
            "Falta EXIST2021_test_reestructurado.csv"
        )

    return train_path, test_path


ruta_train, ruta_test = buscar_datasets()





train = pd.read_csv(ruta_train)
test = pd.read_csv(ruta_test)

def normalizar_etiqueta(valor):

    s = str(valor).strip().lower()

    s = "".join(
        c for c in unicodedata.normalize("NFKD", s)
        if not unicodedata.combining(c)
    )

    return s


MAPA = {
    "0": 0,
    "1": 1,
    "2": 2,
    "3": 3,
    "no violencia": 0,
    "misoginia": 1,
    "acoso": 2,
    "violencia sexual": 3
}


def preparar(df, nombre):

    df = df.copy()

    requeridas = {"text", "tipo_violencia"}

    if not requeridas.issubset(df.columns):
        raise ValueError(
            f"{nombre} no contiene: "
            f"{requeridas - set(df.columns)}"
        )

    if "language" in df.columns:
        df = df[
            df["language"]
            .astype(str)
            .str.lower()
            .str.strip()
            .isin(["es", "spanish"])
        ].copy()

    df = df.dropna(
        subset=["text", "tipo_violencia"]
    )

    df["text"] = (
        df["text"]
        .astype(str)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

    df = df[df["text"].str.len() > 0].copy()

    df["y"] = (
        df["tipo_violencia"]
        .map(normalizar_etiqueta)
        .map(MAPA)
    )

    if df["y"].isna().any():
        raise ValueError(
            f"Etiquetas desconocidas en {nombre}: "
            f"{df.loc[df['y'].isna(), 'tipo_violencia'].unique()}"
        )

    df["y"] = df["y"].astype(int)

    return df.reset_index(drop=True)


train = preparar(train, "TRAIN")
test = preparar(test, "TEST")


textos_test = set(test["text"].str.lower())

solapamiento = (
    train["text"].str.lower().isin(textos_test)
)

if solapamiento.any():

    print(
        "Se excluyen",
        int(solapamiento.sum()),
        "registros coincidentes entre TRAIN y TEST."
    )

    train = train.loc[
        ~solapamiento
    ].reset_index(drop=True)



conflictos = (
    train.groupby("text")["y"].nunique()
)

textos_conflictivos = set(
    conflictos[conflictos > 1].index
)

if textos_conflictivos:
    print(
        "Advertencia:",
        len(textos_conflictivos),
        "textos tienen etiquetas contradictorias."
    )
    train = train[
        ~train["text"].isin(textos_conflictivos)
    ].copy()

train = train.drop_duplicates(
    subset=["text"]
).reset_index(drop=True)

if sorted(train["y"].unique()) != [0, 1, 2, 3]:
    raise ValueError(
        "TRAIN debe incluir las cuatro clases."
    )

X_text_train = train["text"].to_numpy()
X_text_test = test["text"].to_numpy()

y_train = train["y"].to_numpy()
y_test = test["y"].to_numpy()

print(
    "Datos cargados:",
    len(train), "entrenamiento /",
    len(test), "prueba"
)





tfidf = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True,
    strip_accents="unicode"
)

X_tf_tr = tfidf.fit_transform(X_text_train)
X_tf_te = tfidf.transform(X_text_test)

ngramas = CountVectorizer(
    ngram_range=(1, 2),
    max_features=10000,
    min_df=2
)

X_ng_tr = ngramas.fit_transform(X_text_train)
X_ng_te = ngramas.transform(X_text_test)

bow = CountVectorizer(
    ngram_range=(1, 1),
    max_features=10000,
    min_df=2
)

X_bow_tr = bow.fit_transform(X_text_train)
X_bow_te = bow.transform(X_text_test)

caracteres = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3, 5),
    max_features=25000,
    min_df=2,
    sublinear_tf=True
)

X_ch_tr = caracteres.fit_transform(X_text_train)
X_ch_te = caracteres.transform(X_text_test)





n_lsa = min(
    80,
    X_tf_tr.shape[0] - 1,
    X_tf_tr.shape[1] - 1
)

lsa = TruncatedSVD(
    n_components=n_lsa,
    random_state=SEED
)

L_tr = lsa.fit_transform(X_tf_tr)
L_te = lsa.transform(X_tf_te)

L_tr = Normalizer().fit_transform(L_tr)
L_te = Normalizer().fit_transform(L_te)





PATRONES = {
    "insultos": [
        "puta", "perra", "zorra",
        "idiota", "estupida", "estupido"
    ],
    "genero": [
        "mujer", "mujeres", "hombre",
        "hombres", "feminista"
    ],
    "acoso": [
        "acoso", "acosar",
        "hostigar", "amenaza"
    ],
    "sexual": [
        "sexo", "sexual",
        "violacion", "violencia sexual"
    ],
    "negacion": [
        "no", "nunca", "jamas"
    ]
}


def caracteristicas_discursivas(textos):

    salida = []

    for texto in textos:

        s = str(texto).lower()

        s = "".join(
            c for c in unicodedata.normalize(
                "NFKD", s
            )
            if not unicodedata.combining(c)
        )

        tokens = re.findall(r"\b\w+\b", s)

        fila = []

        for patrones in PATRONES.values():

            total = sum(
                len(re.findall(
                    r"(?<!\w)"
                    + re.escape(p)
                    + r"(?!\w)",
                    s
                ))
                for p in patrones
            )

            fila.append(np.log1p(total))

        fila.extend([
            np.log1p(len(tokens)),
            s.count("!"),
            s.count("?"),
            int("@" in s),
            int("#" in s),
            int("http" in s)
        ])

        salida.append(fila)

    return np.asarray(salida, dtype=np.float32)


D_tr = caracteristicas_discursivas(X_text_train)
D_te = caracteristicas_discursivas(X_text_test)

escalador = MinMaxScaler()
D_tr = escalador.fit_transform(D_tr)
D_te = escalador.transform(D_te)









escala_lsa = MinMaxScaler()

L_tr = escala_lsa.fit_transform(L_tr)
L_te = escala_lsa.transform(L_te)

L_te = np.clip(L_te, 0, None)
D_te = np.clip(D_te, 0, None)

BLOQUES_TRAIN = (
    X_tf_tr.tocsr(),
    X_ch_tr.tocsr(),
    csr_matrix(L_tr),
    csr_matrix(D_tr)
)

BLOQUES_TEST = (
    X_tf_te.tocsr(),
    X_ch_te.tocsr(),
    csr_matrix(L_te),
    csr_matrix(D_te)
)


def fusionar(bloques, pesos):

    return hstack(
        [
            bloque.multiply(float(peso))
            for bloque, peso in zip(bloques, pesos)
            if peso > 0
        ],
        format="csr",
        dtype=np.float32
    )












VERSIONES = {

    "HMVT-V1": (1.00, 0.00, 0.35, 0.00),

    "HMVT-V2": (1.00, 0.00, 0.35, 0.25),

    "HMVT-V3": (1.00, 0.75, 0.35, 0.25),

    "HMVT-V4": (1.00, 1.00, 0.30, 0.50),

    "HMVT-V5": (1.00, 1.35, 0.25, 0.35),

    "HMVT-V6a": (1.00, 0.75, 0.10, 0.10),

    "HMVT-V6b": (1.00, 1.25, 0.15, 0.15),

    "HMVT-V6c": (1.00, 0.55, 0.30, 0.30),

    "HMVT-V6d": (1.00, 1.50, 0.05, 0.15)
}








indices = np.arange(len(train))

idx_dev, idx_val = train_test_split(
    indices,
    test_size=0.25,
    random_state=SEED,
    stratify=y_train
)






def bloques_dev_val():

    textos_dev = X_text_train[idx_dev]
    textos_val = X_text_train[idx_val]

    tf = TfidfVectorizer(
        max_features=20000,
        ngram_range=(1, 2),
        min_df=2,
        sublinear_tf=True
    )

    ch = TfidfVectorizer(
        analyzer="char_wb",
        ngram_range=(3, 5),
        max_features=25000,
        min_df=2
    )

    a = tf.fit_transform(textos_dev)
    b = tf.transform(textos_val)

    c = ch.fit_transform(textos_dev)
    d = ch.transform(textos_val)

    k = min(
        80,
        a.shape[0] - 1,
        a.shape[1] - 1
    )

    svd = TruncatedSVD(
        n_components=k,
        random_state=SEED
    )

    la = svd.fit_transform(a)
    lb = svd.transform(b)

    normalizador = Normalizer()

    la = normalizador.fit_transform(la)
    lb = normalizador.transform(lb)

    escala = MinMaxScaler()

    la = escala.fit_transform(la)
    lb = np.clip(escala.transform(lb), 0, None)

    da = caracteristicas_discursivas(textos_dev)
    db = caracteristicas_discursivas(textos_val)

    escala_d = MinMaxScaler()

    da = escala_d.fit_transform(da)
    db = np.clip(escala_d.transform(db), 0, None)

    return (
        (
            a.tocsr(),
            c.tocsr(),
            csr_matrix(la),
            csr_matrix(da)
        ),
        (
            b.tocsr(),
            d.tocsr(),
            csr_matrix(lb),
            csr_matrix(db)
        )
    )


DEV_BLOQUES, VAL_BLOQUES = bloques_dev_val()


CANDIDATOS = [
    (1.0, 0.25, 0.20, 0.15),
    (1.0, 0.50, 0.25, 0.20),
    (1.0, 0.75, 0.35, 0.25),
    (1.0, 1.00, 0.35, 0.25),
    (1.0, 1.25, 0.25, 0.35),
    (1.0, 1.50, 0.15, 0.40),
    (1.0, 0.75, 0.50, 0.50),
    (1.0, 1.00, 0.60, 0.75)
]

evaluacion_pesos = []

for pesos in CANDIDATOS:

    x_dev = fusionar(DEV_BLOQUES, pesos)
    x_val = fusionar(VAL_BLOQUES, pesos)

    modelo = LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=SEED
    )

    modelo.fit(x_dev, y_train[idx_dev])
    pred = modelo.predict(x_val)

    p, r, f1, _ = precision_recall_fscore_support(
        y_train[idx_val],
        pred,
        average="macro",
        zero_division=0
    )

    evaluacion_pesos.append({
        "pesos": pesos,
        "f1_macro": f1,
        "recall_macro": r
    })

evaluacion_pesos.sort(
    key=lambda x: (
        x["f1_macro"],
        x["recall_macro"]
    ),
    reverse=True
)

mejores_pesos = evaluacion_pesos[0]["pesos"]





ultra_candidatos = []

for factor_char in [0.75, 1.00, 1.25]:
    for factor_lsa in [0.75, 1.00, 1.25]:
        for factor_dis in [0.75, 1.00, 1.25]:

            ultra_candidatos.append((
                1.0,
                mejores_pesos[1] * factor_char,
                mejores_pesos[2] * factor_lsa,
                mejores_pesos[3] * factor_dis
            ))

mejor_ultra = (
    -1,
    mejores_pesos
)

for pesos in ultra_candidatos:

    x_dev = fusionar(DEV_BLOQUES, pesos)
    x_val = fusionar(VAL_BLOQUES, pesos)

    modelo = LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=SEED
    )

    modelo.fit(x_dev, y_train[idx_dev])

    pred = modelo.predict(x_val)

    f1 = precision_recall_fscore_support(
        y_train[idx_val],
        pred,
        average="macro",
        zero_division=0
    )[2]

    if f1 > mejor_ultra[0]:
        mejor_ultra = (f1, pesos)

VERSIONES["HMVT-Optimizado"] = mejores_pesos
VERSIONES["HMVT-Ultra-Optimizado"] = mejor_ultra[1]








try:
    from sentence_transformers import SentenceTransformer

except ImportError:
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "sentence-transformers"
    ])

    from sentence_transformers import SentenceTransformer

encoder = SentenceTransformer(
    "sentence-transformers/"
    "paraphrase-multilingual-MiniLM-L12-v2"
)

encoder.eval()

X_emb_tr = encoder.encode(
    list(X_text_train),
    batch_size=64,
    show_progress_bar=False,
    normalize_embeddings=False,
    convert_to_numpy=True
).astype(np.float32)

X_emb_te = encoder.encode(
    list(X_text_test),
    batch_size=64,
    show_progress_bar=False,
    normalize_embeddings=False,
    convert_to_numpy=True
).astype(np.float32)





REPRESENTACIONES = {
    "TF-IDF": (X_tf_tr, X_tf_te),
    "N-gramas": (X_ng_tr, X_ng_te),
    "BoW": (X_bow_tr, X_bow_te),
    "Embeddings": (X_emb_tr, X_emb_te)
}

for nombre, pesos in VERSIONES.items():

    REPRESENTACIONES[nombre] = (
        fusionar(BLOQUES_TRAIN, pesos),
        fusionar(BLOQUES_TEST, pesos)
    )





MODELOS = [
    "Regresión Logística",
    "3-NN",
    "SVM",
    "Naive Bayes",
    "Red neuronal",
    "XGBoost",
    "Random Forest",
    "Árbol de decisión"
]


def crear_modelo(nombre, X):

    if nombre == "Regresión Logística":
        return LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            random_state=SEED
        )

    if nombre == "3-NN":
        return KNeighborsClassifier(
            n_neighbors=3
        )

    if nombre == "SVM":
        return LinearSVC(
            class_weight="balanced",
            max_iter=8000,
            random_state=SEED
        )

    if nombre == "Naive Bayes":

        if not issparse(X):
            return GaussianNB()

        return MultinomialNB()

    if nombre == "Red neuronal":
        return MLPClassifier(
            hidden_layer_sizes=(64,),
            max_iter=180,
            early_stopping=True,
            random_state=SEED
        )

    if nombre == "Random Forest":
        return RandomForestClassifier(
            n_estimators=140,
            class_weight="balanced",
            n_jobs=-1,
            random_state=SEED
        )

    if nombre == "Árbol de decisión":
        return DecisionTreeClassifier(
            max_depth=18,
            class_weight="balanced",
            random_state=SEED
        )

    if nombre == "XGBoost":

        try:
            from xgboost import XGBClassifier

        except ImportError:
            subprocess.check_call([
                sys.executable,
                "-m",
                "pip",
                "install",
                "-q",
                "xgboost"
            ])

            from xgboost import XGBClassifier

        return XGBClassifier(
            objective="multi:softprob",
            num_class=4,
            n_estimators=140,
            max_depth=4,
            learning_rate=0.07,
            tree_method="hist",
            eval_metric="mlogloss",
            random_state=SEED,
            n_jobs=2
        )

    raise ValueError(nombre)





resultados = []

for representacion, (Xtr, Xte) in REPRESENTACIONES.items():

    print("Evaluando:", representacion, flush=True)

    for nombre_modelo in MODELOS:

        try:

            modelo = crear_modelo(
                nombre_modelo, Xtr
            )

            modelo.fit(Xtr, y_train)

            pred = modelo.predict(Xte)

            precision, recall, f1, _ = (
                precision_recall_fscore_support(
                    y_test,
                    pred,
                    average="macro",
                    labels=[0, 1, 2, 3],
                    zero_division=0
                )
            )

            accuracy = accuracy_score(
                y_test,
                pred
            )

            resultados.append({
                "Representación": representacion,
                "Modelo de clasificación": nombre_modelo,
                "Precisión": precision,
                "Recall": recall,
                "F1-Score": f1,
                "Accuracy": accuracy
            })

        except Exception as error:

            RESULTADOS_ERRORES.append(
                (representacion, nombre_modelo, str(error))
            )

            resultados.append({
                "Representación": representacion,
                "Modelo de clasificación": nombre_modelo,
                "Precisión": np.nan,
                "Recall": np.nan,
                "F1-Score": np.nan,
                "Accuracy": np.nan
            })





tabla = pd.DataFrame(resultados)

orden_representaciones = list(REPRESENTACIONES.keys())

tabla["Representación"] = pd.Categorical(
    tabla["Representación"],
    categories=orden_representaciones,
    ordered=True
)

tabla["Modelo de clasificación"] = pd.Categorical(
    tabla["Modelo de clasificación"],
    categories=MODELOS,
    ordered=True
)

tabla = tabla.sort_values(
    ["Representación", "Modelo de clasificación"]
).reset_index(drop=True)

print("\n" + "=" * 80)
print("TABLA COMPLETA - EXIST2021 - MULTICLASE")
print("Precision, Recall y F1-score: PROMEDIO MACRO")
print("=" * 80)

display(
    tabla.style.format({
        "Precisión": "{:.4f}",
        "Recall": "{:.4f}",
        "F1-Score": "{:.4f}",
        "Accuracy": "{:.4f}"
    }, na_rep="ERROR")
)

if RESULTADOS_ERRORES:

    print(
        "\nCombinaciones que no pudieron evaluarse:",
        len(RESULTADOS_ERRORES)
    )

    for rep, modelo, error in RESULTADOS_ERRORES:
        print(f"{rep} / {modelo}: {error}")


### EXP-039 · V1, V2, V3, V4, V5, V6a, V6b, V6c, V6d

**Archivo de origen:** `Reportes_tesis.ipynb, celda 33`. **Referencias duplicadas:** 1. **Sintaxis de código:** Válida.

EXIST2021 ESPAÑOL · MULTICLASE · UNA SOLA CELDA DE COLAB
Tabla completa por representación + matrices reales moradas
HMVT V1-V6d, optimizado y Ultra. Embeddings baseline sin tuning.
import sys, subprocess, re, unicodedata, warnings
from pat

**Ejecución:** celda independiente o variante experimental; revisar archivos de entrada y dependencias antes de ejecutarla. Los resultados originales no se insertan en esta celda.

In [ ]:
import sys, subprocess, re, unicodedata, warnings
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display
from scipy.sparse import hstack, csr_matrix, issparse
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import MultinomialNB, GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, ConfusionMatrixDisplay

SEED=42
MOSTRAR_MATRICES_TODAS_VERSIONES=True  
TAMANO_NUMEROS=12
TAMANO_FIGURA=(21,12)
EMBEDDINGS_SIN_OPTIMIZAR=True
CLASES=['No violencia','Misoginia','Acoso','Violencia sexual']
MODELOS=['Regresión Logística','3-NN','SVM','Naive Bayes','Red neuronal','XGBoost','Random Forest','Árbol de decisión']



VERSIONES={
 'HMVT-V1':(1,0,.35,0), 'HMVT-V2':(1,0,.35,.25),
 'HMVT-V3':(1,.75,.35,.25), 'HMVT-V4':(1,1,.30,.50),
 'HMVT-V5':(1,1.35,.25,.35), 'HMVT-V6a':(1,.75,.10,.10),
 'HMVT-V6b':(1,1.25,.15,.15), 'HMVT-V6c':(1,.55,.30,.30),
 'HMVT-V6d':(1,1.50,.05,.15)
}
CANDIDATOS_ULTRA=[
 (1,.35,.25,.15),(1,.65,.20,.30),(1,.85,.35,.40),
 (1,1.10,.15,.20),(1,1.40,.10,.30),(1,.75,.55,.60)
]

def buscar_archivo(prefijo):
    for root in [Path('/content'),Path.cwd(),Path('/mnt/data')]:
        if root.exists():
            encontrados=sorted(root.glob(prefijo+'*.csv'))
            if encontrados:return encontrados[0]
    try:
        from google.colab import files
        print('Sube TRAIN y TEST reestructurados en español.');files.upload()
    except ImportError:pass
    for root in [Path('/content'),Path.cwd(),Path('/mnt/data')]:
        if root.exists():
            encontrados=sorted(root.glob(prefijo+'*.csv'))
            if encontrados:return encontrados[0]
    raise FileNotFoundError('Falta '+prefijo+'.csv')

def prep(path):
    df=pd.read_csv(path)
    if not {'text','tipo_violencia'}.issubset(df):
        raise ValueError(f'{path.name}: faltan text y/o tipo_violencia')
    if 'language' in df:df=df[df.language.astype(str).str.lower().str.strip().isin(['es','spanish'])].copy()
    df=df.dropna(subset=['text','tipo_violencia']).copy()
    df['text']=df.text.astype(str).str.replace(r'\s+',' ',regex=True).str.strip()
    df=df[df.text.str.len()>0]
    def norm(s):
        return ''.join(c for c in unicodedata.normalize('NFKD',str(s).lower().strip()) if not unicodedata.combining(c))
    m={'0':0,'1':1,'2':2,'3':3,'no violencia':0,'misoginia':1,'acoso':2,'violencia sexual':3}
    df['y']=df.tipo_violencia.map(lambda a:m.get(norm(a),np.nan))
    if df.y.isna().any():raise ValueError('Etiquetas desconocidas: '+str(df[df.y.isna()].tipo_violencia.unique()))
    df['y']=df.y.astype(int)
    return df.reset_index(drop=True)

train=prep(buscar_archivo('EXIST2021_train_reestructurado'))
test=prep(buscar_archivo('EXIST2021_test_reestructurado'))
train=train[~train.text.str.casefold().isin(set(test.text.str.casefold()))].copy()
conf=train.groupby('text').y.nunique();train=train[~train.text.isin(conf[conf>1].index)]
train=train.drop_duplicates('text').reset_index(drop=True)
assert set(train.y)=={0,1,2,3} and set(test.y)=={0,1,2,3}
print('TRAIN:',len(train),'TEST:',len(test),'clases test:',test.y.value_counts().sort_index().to_dict())

PATRONES={
 'insultos':['puta','perra','zorra','idiota','estupida','estupido'],
 'genero':['mujer','mujeres','hombre','hombres','feminista'],
 'acoso':['acoso','acosar','hostigar','amenaza'],
 'sexual':['sexo','sexual','violacion','violencia sexual'],
 'negacion':['no','nunca','jamas']
}
REGEX={k:[re.compile(r'(?<!\w)'+re.escape(p)+r'(?!\w)') for p in vals] for k,vals in PATRONES.items()}
def discurso(textos):
    salida=[]
    for text in textos:
        s=''.join(c for c in unicodedata.normalize('NFKD',str(text).lower()) if not unicodedata.combining(c))
        toks=re.findall(r'\b\w+\b',s)
        fila=[np.log1p(sum(len(p.findall(s)) for p in pats)) for pats in REGEX.values()]
        fila += [np.log1p(len(toks)),s.count('!'),s.count('?'),int('@' in s),int('#' in s),int('http' in s)]
        salida.append(fila)
    return np.array(salida,dtype=np.float32)

class Extractor:
    def fit(self,textos):
        self.word=TfidfVectorizer(max_features=20000,ngram_range=(1,2),min_df=2,sublinear_tf=True,strip_accents='unicode')
        self.char=TfidfVectorizer(analyzer='char_wb',ngram_range=(3,5),max_features=25000,min_df=2,sublinear_tf=True)
        self.ng=CountVectorizer(ngram_range=(1,2),max_features=10000,min_df=2)
        self.bow=CountVectorizer(ngram_range=(1,1),max_features=10000,min_df=2)
        w=self.word.fit_transform(textos)
        self.char.fit(textos);self.ng.fit(textos);self.bow.fit(textos)
        k=min(80,w.shape[0]-1,w.shape[1]-1)
        self.svd=TruncatedSVD(n_components=k,random_state=SEED).fit(w)
        l=self.svd.transform(w)
        self.lscale=MinMaxScaler().fit(l)
        self.dscale=MinMaxScaler().fit(discurso(textos))
        return self
    def transform(self,textos):
        w=self.word.transform(textos).astype(np.float32)
        c=self.char.transform(textos).astype(np.float32)
        l=csr_matrix(np.clip(self.lscale.transform(self.svd.transform(w)),0,None).astype(np.float32))
        d=csr_matrix(np.clip(self.dscale.transform(discurso(textos)),0,None).astype(np.float32))
        return (w,c,l,d)

def fusion(bloques,pesos):
    return hstack([b.multiply(float(p)) for b,p in zip(bloques,pesos) if p>0],format='csr',dtype=np.float32)

def crear_modelo(nombre,X):
    if nombre=='Regresión Logística':return LogisticRegression(max_iter=2500,class_weight='balanced',random_state=SEED)
    if nombre=='3-NN':return KNeighborsClassifier(n_neighbors=3)
    if nombre=='SVM':return LinearSVC(class_weight='balanced',max_iter=8000,random_state=SEED)
    if nombre=='Naive Bayes':return MultinomialNB() if issparse(X) else GaussianNB()
    if nombre=='Red neuronal':return MLPClassifier(hidden_layer_sizes=(64,),early_stopping=True,max_iter=180,random_state=SEED)
    if nombre=='Random Forest':return RandomForestClassifier(n_estimators=140,class_weight='balanced',n_jobs=-1,random_state=SEED)
    if nombre=='Árbol de decisión':return DecisionTreeClassifier(max_depth=18,class_weight='balanced',random_state=SEED)
    if nombre=='XGBoost':
        try:from xgboost import XGBClassifier
        except ImportError:
            subprocess.check_call([sys.executable,'-m','pip','install','-q','xgboost'])
            from xgboost import XGBClassifier
        return XGBClassifier(objective='multi:softprob',num_class=4,n_estimators=140,max_depth=4,learning_rate=.07,tree_method='hist',eval_metric='mlogloss',random_state=SEED,n_jobs=2)
    raise ValueError(nombre)

def metrics(y,p):
    a,b,c,_=precision_recall_fscore_support(y,p,labels=[0,1,2,3],average='macro',zero_division=0)
    recalls=precision_recall_fscore_support(y,p,labels=[0,1,2,3],average=None,zero_division=0)[1]
    return dict(Precisión=a,Recall=b,**{'F1-Score':c,'Accuracy':accuracy_score(y,p),
        'Recall Acoso':recalls[2],'Recall Violencia sexual':recalls[3]})


idx=np.arange(len(train))
i_dev,i_val=train_test_split(idx,test_size=.25,stratify=train.y,random_state=SEED)
dev=train.iloc[i_dev];val=train.iloc[i_val]
e_val=Extractor().fit(dev.text.to_numpy())
B_dev=e_val.transform(dev.text.to_numpy());B_val=e_val.transform(val.text.to_numpy())


SELECCION={};VERIFICACION=[]
for modelo_nombre in MODELOS:
    candidatos={k:v for k,v in VERSIONES.items() if k in ['HMVT-V6a','HMVT-V6d']} if modelo_nombre=='Regresión Logística' else {
        'Ultra-C'+str(i+1):p for i,p in enumerate(CANDIDATOS_ULTRA)}
    ganador=None
    for nombre,pesos in candidatos.items():
        Xd=fusion(B_dev,pesos);Xv=fusion(B_val,pesos)
        try:
            mod=crear_modelo(modelo_nombre,Xd);mod.fit(Xd,dev.y);pred=mod.predict(Xv)
            met=metrics(val.y,pred)
            
            score=(met['F1-Score'],(met['Recall Acoso']+met['Recall Violencia sexual'])/2)
            if ganador is None or score>ganador[0]:ganador=(score,nombre,pesos)
        except Exception as exc:
            warnings.warn('Validación '+modelo_nombre+' '+nombre+': '+str(exc))
    if ganador is None:raise RuntimeError('Falló toda la validación: '+modelo_nombre)
    SELECCION[modelo_nombre]=(ganador[1],ganador[2])
    VERIFICACION.append((modelo_nombre,ganador[1],round(ganador[0][0],4)))
print('Selección HMVT SOLO en validación:',VERIFICACION)


e=Extractor().fit(train.text.to_numpy());Bt=e.transform(train.text.to_numpy());Be=e.transform(test.text.to_numpy())
REPS={
 'TF-IDF':(Bt[0],Be[0]),
 'N-gramas':(e.ng.transform(train.text),e.ng.transform(test.text)),
 'BoW':(e.bow.transform(train.text),e.bow.transform(test.text))
}

try:from sentence_transformers import SentenceTransformer
except ImportError:
    subprocess.check_call([sys.executable,'-m','pip','install','-q','sentence-transformers'])
    from sentence_transformers import SentenceTransformer
encoder=SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')
encoder.eval()
REPS['Embeddings']=(
    encoder.encode(train.text.tolist(),batch_size=64,show_progress_bar=True,normalize_embeddings=False).astype(np.float32),
    encoder.encode(test.text.tolist(),batch_size=64,show_progress_bar=True,normalize_embeddings=False).astype(np.float32)
)
for version,pesos in VERSIONES.items():REPS[version]=(fusion(Bt,pesos),fusion(Be,pesos))

SELECCION_GLOBAL=SELECCION['Regresión Logística'][1]
REPS['HMVT-Optimizado']=(fusion(Bt,SELECCION_GLOBAL),fusion(Be,SELECCION_GLOBAL))

REPS['HMVT-Ultra-Optimizado']=(None,None)

resultados=[];PREDICCIONES={};errores=[]
for rep,(Xtr,Xte) in REPS.items():
    print('Evaluando:',rep,flush=True)
    for nombre in MODELOS:
        if rep=='HMVT-Ultra-Optimizado':
            pesos=SELECCION[nombre][1]
            Xtr,Xte=fusion(Bt,pesos),fusion(Be,pesos)
        try:
            mod=crear_modelo(nombre,Xtr)
            mod.fit(Xtr,train.y)
            pred=np.asarray(mod.predict(Xte),dtype=int)
            PREDICCIONES[(rep,nombre)]=pred
            m=metrics(test.y,pred)
            resultados.append({'Representación':rep,'Modelo de clasificación':nombre,**m})
        except Exception as exc:
            errores.append((rep,nombre,str(exc)))
            resultados.append({'Representación':rep,'Modelo de clasificación':nombre,
                 **{k:np.nan for k in ['Precisión','Recall','F1-Score','Accuracy','Recall Acoso','Recall Violencia sexual']}})

TABLA=pd.DataFrame(resultados)
TABLA['Representación']=pd.Categorical(TABLA['Representación'],categories=list(REPS),ordered=True)
TABLA['Modelo de clasificación']=pd.Categorical(TABLA['Modelo de clasificación'],categories=MODELOS,ordered=True)
TABLA=TABLA.sort_values(['Representación','Modelo de clasificación']).reset_index(drop=True)
print('\nTABLA COMPLETA: PRECISION, RECALL Y F1 MACRO · ACCURACY GLOBAL')
display(TABLA[['Representación','Modelo de clasificación','Precisión','Recall','F1-Score','Accuracy']].style.format({x:'{:.4f}' for x in ['Precisión','Recall','F1-Score','Accuracy']},na_rep='ERROR'))
print('\nHMVT: RECALL DE ACOSO Y VIOLENCIA SEXUAL (VALORES REALES TEST)')
display(TABLA[TABLA['Representación'].astype(str).str.startswith('HMVT')][['Representación','Modelo de clasificación','F1-Score','Recall Acoso','Recall Violencia sexual']].style.format({x:'{:.4f}' for x in ['F1-Score','Recall Acoso','Recall Violencia sexual']},na_rep='ERROR'))
if errores:print('Combinaciones con errores (no se inventan métricas):',errores)



print('\nHMVT DESTACADO POR CLASIFICADOR (seleccionado sin consultar TEST)')
filas=[]
for nombre in MODELOS:
    rep='HMVT-Optimizado' if nombre=='Regresión Logística' else 'HMVT-Ultra-Optimizado'
    fila=TABLA[(TABLA['Representación']==rep)&(TABLA['Modelo de clasificación']==nombre)]
    if not fila.empty:
        f=fila.iloc[0]
        filas.append({'Modelo':nombre,'Variante seleccionada':SELECCION[nombre][0],
         'F1 macro':f['F1-Score'],'Recall Acoso':f['Recall Acoso'],'Recall Violencia sexual':f['Recall Violencia sexual']})
display(pd.DataFrame(filas).style.format({'F1 macro':'{:.4f}','Recall Acoso':'{:.4f}','Recall Violencia sexual':'{:.4f}'}))


REP_GRAF=list(REPS) if MOSTRAR_MATRICES_TODAS_VERSIONES else ['TF-IDF','N-gramas','BoW','Embeddings','HMVT-Optimizado','HMVT-Ultra-Optimizado']
for rep in REP_GRAF:
    fig,axes=plt.subplots(2,4,figsize=TAMANO_FIGURA,constrained_layout=True)
    for ax,nombre in zip(axes.flat,MODELOS):
        
        
        pred=PREDICCIONES.get((rep,nombre))
        if pred is None:ax.axis('off');continue
        cm=confusion_matrix(test.y,pred,labels=[0,1,2,3]);m=metrics(test.y,pred)
        disp=ConfusionMatrixDisplay(cm,display_labels=['No violencia','Misoginia','Acoso','Viol. sexual'])
        disp.plot(ax=ax,cmap='Purples',colorbar=False,values_format='d',xticks_rotation=35)
        for t in disp.text_.ravel():t.set_fontsize(TAMANO_NUMEROS);t.set_fontweight('bold')
        ax.set_title(f"{nombre}\nF1 macro={m['F1-Score']:.3f} | Acc={m['Accuracy']:.3f}",fontsize=11,fontweight='bold')
        ax.tick_params(axis='both',labelsize=8)
        ax.set_xlabel('Predicción',fontsize=9);ax.set_ylabel('Real',fontsize=9)
    fig.suptitle(f'MULTICLASE · {rep} · MATRICES REALES · N={len(test)}',fontsize=16,fontweight='bold')
    plt.show();plt.close(fig)

print('\nFIN: tablas completas y matrices moradas. Ninguna cifra fue simulada.')
print('Nota: el test se ha usado en experimentos anteriores; interpretación exploratoria.')


## 03. HMVT y optimización avanzada

### EXP-040 · V1, V2, V3, V4, V5, V6a, V6b, V6c, V6d

**Archivo de origen:** `Reportes_tesis.ipynb, celda 35`. **Referencias duplicadas:** 1. **Sintaxis de código:** Válida.

EXIST2021 ESPAÑOL MULTICLASE | UNA CELDA COLAB
Optimización de HMVT en TRAIN/VALIDACIÓN, TEST solo evaluación
Embeddings congelados: baseline sin tuning / fine-tuning
Tabla completa y matrices moradas agrupadas por representación
import sys

**Ejecución:** celda independiente o variante experimental; revisar archivos de entrada y dependencias antes de ejecutarla. Los resultados originales no se insertan en esta celda.

In [ ]:
import sys, subprocess, re, unicodedata, warnings
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display
from scipy.sparse import hstack, csr_matrix, issparse
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import MinMaxScaler, normalize
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import MultinomialNB, GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, ConfusionMatrixDisplay
warnings.filterwarnings('ignore',category=FutureWarning)
SEED=42
CLASES=['No violencia','Misoginia','Acoso','Violencia sexual']
MODELOS=['Regresión Logística','3-NN','SVM','Naive Bayes','Red neuronal','XGBoost','Random Forest','Árbol de decisión']
MOSTRAR_TODAS_LAS_MATRICES=True    
TAMANO_NUMEROS=11
FIGSIZE=(21,12)

VERSIONES={
 'HMVT-V1':(1,0,.35,0),'HMVT-V2':(1,0,.35,.25),'HMVT-V3':(1,.75,.35,.25),
 'HMVT-V4':(1,1,.30,.50),'HMVT-V5':(1,1.35,.25,.35),
 'HMVT-V6a':(1,.75,.10,.10),'HMVT-V6b':(1,1.25,.15,.15),
 'HMVT-V6c':(1,.55,.30,.30),'HMVT-V6d':(1,1.50,.05,.15)
}

PESOS_CANDIDATOS=[VERSIONES[k] for k in ['HMVT-V2','HMVT-V3','HMVT-V6a','HMVT-V6d']]+[
 (1,.50,.50,.60),(1,1.0,.35,.70),(1,.75,.20,1.0),(1,1.25,.15,.65)]

def buscar_archivo(prefijo):
    def candidatos():
        return [p for root in [Path('/content'),Path.cwd(),Path('/mnt/data')] if root.exists()
                for p in sorted(root.glob(prefijo+'*.csv')) if p.is_file()]
    encontrados=candidatos()
    if not encontrados:
        try:
            from google.colab import files
            print('Sube TRAIN y TEST reestructurados en español');files.upload()
        except ImportError:pass
        encontrados=candidatos()
    if not encontrados:raise FileNotFoundError('No se encontró '+prefijo+'.csv')
    return encontrados[0]

def prep(path):
    df=pd.read_csv(path)
    if not {'text','tipo_violencia'}.issubset(df):raise ValueError('Faltan text/tipo_violencia: '+str(path))
    if 'language' in df:df=df[df.language.astype(str).str.strip().str.lower().isin(['es','spanish'])].copy()
    df=df.dropna(subset=['text','tipo_violencia']).copy()
    df['text']=df.text.astype(str).str.replace(r'\s+',' ',regex=True).str.strip()
    df=df[df.text.str.len()>0].copy()
    def norm(s):return ''.join(c for c in unicodedata.normalize('NFKD',str(s).strip().lower()) if not unicodedata.combining(c))
    mapping={'0':0,'1':1,'2':2,'3':3,'no violencia':0,'misoginia':1,'acoso':2,'violencia sexual':3}
    df['y']=df.tipo_violencia.map(lambda s:mapping.get(norm(s),np.nan))
    if df.y.isna().any():raise ValueError('Etiquetas no reconocidas: '+str(df.loc[df.y.isna(),'tipo_violencia'].unique()))
    df['y']=df.y.astype(int)
    return df.reset_index(drop=True)

train=prep(buscar_archivo('EXIST2021_train_reestructurado'))
test=prep(buscar_archivo('EXIST2021_test_reestructurado'))
train=train[~train.text.str.casefold().isin(set(test.text.str.casefold()))].copy()
conf=train.groupby('text').y.nunique()
train=train[~train.text.isin(conf[conf>1].index)].drop_duplicates('text').reset_index(drop=True)
assert set(train.y)==set(test.y)=={0,1,2,3}
print('TRAIN',len(train),'TEST',len(test),'Clases test:',test.y.value_counts().sort_index().to_dict())

PATRONES={'insultos':['puta','perra','zorra','idiota','estupida','estupido'],
'genero':['mujer','mujeres','hombre','hombres','feminista'],
'acoso':['acoso','acosar','hostigar','amenaza'],
'sexual':['sexo','sexual','violacion','violencia sexual'],
'negacion':['no','nunca','jamas']}
RX={k:[re.compile(r'(?<!\w)'+re.escape(w)+r'(?!\w)') for w in ws] for k,ws in PATRONES.items()}
def discurso(textos):
    out=[]
    for txt in textos:
        s=''.join(c for c in unicodedata.normalize('NFKD',str(txt).lower()) if not unicodedata.combining(c))
        tokens=re.findall(r'\b\w+\b',s)
        row=[np.log1p(sum(len(p.findall(s)) for p in ps)) for ps in RX.values()]
        row += [np.log1p(len(tokens)),len(set(tokens))/max(len(tokens),1),s.count('!'),s.count('?'),int('@' in s),int('#' in s),int('http' in s)]
        out.append(row)
    return np.asarray(out,dtype=np.float32)

class Extractor:
    def fit(self,textos):
        self.word=TfidfVectorizer(max_features=20000,ngram_range=(1,2),min_df=2,sublinear_tf=True,strip_accents='unicode')
        self.char=TfidfVectorizer(analyzer='char_wb',ngram_range=(3,5),max_features=25000,min_df=2,sublinear_tf=True)
        self.ng=CountVectorizer(ngram_range=(1,2),max_features=10000,min_df=2)
        self.bow=CountVectorizer(max_features=10000,min_df=2)
        w=self.word.fit_transform(textos);self.char.fit(textos);self.ng.fit(textos);self.bow.fit(textos)
        k=min(80,w.shape[0]-1,w.shape[1]-1)
        self.svd=TruncatedSVD(n_components=k,random_state=SEED).fit(w)
        self.lscale=MinMaxScaler().fit(self.svd.transform(w))
        self.dscale=MinMaxScaler().fit(discurso(textos))
        return self
    def transform(self,textos):
        w=self.word.transform(textos).astype(np.float32)
        c=self.char.transform(textos).astype(np.float32)
        l=csr_matrix(np.clip(self.lscale.transform(self.svd.transform(w)),0,None).astype(np.float32))
        d=csr_matrix(np.clip(self.dscale.transform(discurso(textos)),0,None).astype(np.float32))
        return w,c,l,d

def fusion(blocks,pesos):
    return hstack([b.multiply(float(p)) for b,p in zip(blocks,pesos) if p>0],format='csr',dtype=np.float32)

def crear_modelo(nombre,X,config=None):
    config=config or {}; c=float(config.get('C',1.0)); balance=config.get('balance','balanced')
    if nombre=='Regresión Logística':return LogisticRegression(C=c,class_weight=balance,max_iter=2600,random_state=SEED)
    if nombre=='SVM':return LinearSVC(C=c,class_weight=balance,max_iter=9000,random_state=SEED)
    if nombre=='3-NN':return KNeighborsClassifier(n_neighbors=int(config.get('k',3)),weights=config.get('weights','uniform'))
    if nombre=='Naive Bayes':return MultinomialNB(alpha=float(config.get('alpha',1.0))) if issparse(X) else GaussianNB()
    if nombre=='Red neuronal':return MLPClassifier(hidden_layer_sizes=(int(config.get('hidden',64)),),alpha=float(config.get('alpha',.0001)),max_iter=220,early_stopping=True,random_state=SEED)
    if nombre=='Random Forest':return RandomForestClassifier(n_estimators=180,max_depth=config.get('depth',None),min_samples_leaf=int(config.get('leaf',1)),class_weight=balance,n_jobs=-1,random_state=SEED)
    if nombre=='Árbol de decisión':return DecisionTreeClassifier(max_depth=int(config.get('depth',18)),min_samples_leaf=int(config.get('leaf',1)),class_weight=balance,random_state=SEED)
    if nombre=='XGBoost':
        try:from xgboost import XGBClassifier
        except ImportError:
            subprocess.check_call([sys.executable,'-m','pip','install','-q','xgboost'])
            from xgboost import XGBClassifier
        return XGBClassifier(objective='multi:softprob',num_class=4,n_estimators=int(config.get('trees',140)),max_depth=int(config.get('depth',4)),learning_rate=float(config.get('lr',.07)),subsample=.9,colsample_bytree=.9,tree_method='hist',eval_metric='mlogloss',random_state=SEED,n_jobs=2)
    raise ValueError(nombre)

def metricas(y,p):
    prec,rec,f1,_=precision_recall_fscore_support(y,p,average='macro',labels=[0,1,2,3],zero_division=0)
    por_clase=precision_recall_fscore_support(y,p,average=None,labels=[0,1,2,3],zero_division=0)
    return {'Precisión':prec,'Recall':rec,'F1-Score':f1,'Accuracy':accuracy_score(y,p),
            'Recall Acoso':por_clase[1][2],'Recall Violencia sexual':por_clase[1][3]}

def cfgs_para(nombre):
    
    if nombre in ['Regresión Logística','SVM']:
        return [{'C':c,'balance':'balanced'} for c in [.1,1,5]]
    if nombre=='3-NN':return [{'k':k,'weights':w} for k,w in [(3,'distance'),(5,'distance'),(7,'distance')]]
    if nombre=='Naive Bayes':return [{'alpha':a} for a in [.1,1,5]]
    if nombre=='Red neuronal':return [{'hidden':h,'alpha':.001} for h in [64,128]]
    if nombre=='Random Forest':return [{'depth':d,'leaf':leaf,'balance':'balanced_subsample'} for d,leaf in [(None,1),(18,1),(None,2)]]
    if nombre=='Árbol de decisión':return [{'depth':d,'leaf':leaf,'balance':'balanced'} for d,leaf in [(10,2),(18,2),(24,4)]]
    if nombre=='XGBoost':return [{'depth':d,'trees':n,'lr':.07} for d,n in [(3,180),(5,180)]]
    return [{}]

def fit_predict(nombre,Xtr,ytr,Xev,config,oversample=False):
    
    
    if oversample:
        rng=np.random.default_rng(SEED)
        ids=np.arange(len(ytr));sets=[ids[ytr==cl] for cl in [0,1,2,3]]
        target=max(map(len,sets))
        chosen=np.concatenate([rng.choice(s,size=target,replace=True) for s in sets])
        rng.shuffle(chosen)
        Xtr=Xtr[chosen];ytr=np.asarray(ytr)[chosen]
    modelo=crear_modelo(nombre,Xtr,config)
    modelo.fit(Xtr,ytr)
    return modelo.predict(Xev).astype(int)





ids=np.arange(len(train))
id_dev,id_val=train_test_split(ids,test_size=.25,stratify=train.y,random_state=SEED)
dev=train.iloc[id_dev];val=train.iloc[id_val]
extract_dev=Extractor().fit(dev.text.to_numpy())
Bd=extract_dev.transform(dev.text.to_numpy());Bv=extract_dev.transform(val.text.to_numpy())
ydev=dev.y.to_numpy();yval=val.y.to_numpy()
SELECCION={}
for nombre in MODELOS:
    mejor=None
    configs=cfgs_para(nombre)
    
    pesos_lista=PESOS_CANDIDATOS if nombre in ['Regresión Logística','SVM','3-NN','Naive Bayes'] else PESOS_CANDIDATOS[::2]
    for pesos in pesos_lista:
        xa=fusion(Bd,pesos);xv=fusion(Bv,pesos)
        for cfg in configs:
            for osamp in ([False,True] if nombre in ['3-NN','Naive Bayes','Red neuronal','XGBoost'] else [False]):
                try:
                    pred=fit_predict(nombre,xa,ydev,xv,cfg,oversample=osamp)
                    m=metricas(yval,pred)
                    score=(m['F1-Score'],(m['Recall Acoso']+m['Recall Violencia sexual'])/2)
                    if mejor is None or score>mejor[0]:mejor=(score,pesos,cfg,osamp)
                except Exception as err:
                    warnings.warn(f'Validación {nombre}: {err}')
    if mejor is None:raise RuntimeError('No hay configuración válida para '+nombre)
    SELECCION[nombre]={'pesos':mejor[1],'config':mejor[2],'oversample':mejor[3],'F1_validacion':mejor[0][0]}
    print('HMVT validación:',nombre,'F1=',round(mejor[0][0],4),'config=',SELECCION[nombre],flush=True)




e=Extractor().fit(train.text.to_numpy())
Bt=e.transform(train.text.to_numpy());Be=e.transform(test.text.to_numpy())
REP={'TF-IDF':(Bt[0],Be[0]),
     'N-gramas':(e.ng.transform(train.text),e.ng.transform(test.text)),
     'BoW':(e.bow.transform(train.text),e.bow.transform(test.text))}
try:from sentence_transformers import SentenceTransformer
except ImportError:
    subprocess.check_call([sys.executable,'-m','pip','install','-q','sentence-transformers'])
    from sentence_transformers import SentenceTransformer
encoder=SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')
encoder.eval()  
REP['Embeddings']=(
    encoder.encode(train.text.tolist(),batch_size=64,show_progress_bar=True,normalize_embeddings=False).astype(np.float32),
    encoder.encode(test.text.tolist(),batch_size=64,show_progress_bar=True,normalize_embeddings=False).astype(np.float32))
for version,pesos in VERSIONES.items():REP[version]=(fusion(Bt,pesos),fusion(Be,pesos))

REP['HMVT-Seleccionado']=(None,None)
resultados=[];PREDICCIONES={};ERRORES=[]
for nombre_rep,(Xtr,Xte) in REP.items():
    print('Evaluando:',nombre_rep,flush=True)
    for nombre_mod in MODELOS:
        if nombre_rep=='HMVT-Seleccionado':
            sel=SELECCION[nombre_mod];Xtr=fusion(Bt,sel['pesos']);Xte=fusion(Be,sel['pesos'])
            cfg=sel['config'];over=sel['oversample']
        else:cfg={};over=False
        try:
            pred=fit_predict(nombre_mod,Xtr,train.y.to_numpy(),Xte,cfg,oversample=over)
            PREDICCIONES[(nombre_rep,nombre_mod)]=pred
            resultados.append({'Representación':nombre_rep,'Modelo de clasificación':nombre_mod,**metricas(test.y.to_numpy(),pred)})
        except Exception as err:
            ERRORES.append((nombre_rep,nombre_mod,str(err)))
            resultados.append({'Representación':nombre_rep,'Modelo de clasificación':nombre_mod,
              **{k:np.nan for k in ['Precisión','Recall','F1-Score','Accuracy','Recall Acoso','Recall Violencia sexual']}})
TABLA=pd.DataFrame(resultados)
TABLA['Representación']=pd.Categorical(TABLA['Representación'],categories=list(REP),ordered=True)
TABLA['Modelo de clasificación']=pd.Categorical(TABLA['Modelo de clasificación'],categories=MODELOS,ordered=True)
TABLA=TABLA.sort_values(['Representación','Modelo de clasificación']).reset_index(drop=True)
print('\nTABLA COMPLETA: PRECISIÓN / RECALL / F1 MACRO Y ACCURACY GLOBAL')
display(TABLA[['Representación','Modelo de clasificación','Precisión','Recall','F1-Score','Accuracy']].style.format({k:'{:.4f}' for k in ['Precisión','Recall','F1-Score','Accuracy']},na_rep='ERROR'))
print('\nHMVT: F1 MACRO Y RECALL DE LAS CLASES MINORITARIAS (TEST EXPLORATORIO)')
display(TABLA[TABLA['Representación'].astype(str).str.startswith('HMVT')][['Representación','Modelo de clasificación','F1-Score','Recall Acoso','Recall Violencia sexual']].style.format({k:'{:.4f}' for k in ['F1-Score','Recall Acoso','Recall Violencia sexual']},na_rep='ERROR'))
print('\nSELECCIÓN FINAL REALIZADA EXCLUSIVAMENTE EN VALIDACIÓN:')
for k,v in SELECCION.items():print(k,'=>',v)
if ERRORES:print('\nErrores de combinaciones:',ERRORES)





for rep in (list(REP) if MOSTRAR_TODAS_LAS_MATRICES else ['HMVT-Seleccionado']):
    fig,axs=plt.subplots(2,4,figsize=FIGSIZE,constrained_layout=True)
    fig.suptitle(f'EXIST2021 ES · MULTICLASE · {rep} · TEST EXPLORATORIO',fontsize=17,fontweight='bold')
    cms=[confusion_matrix(test.y,PREDICCIONES[(rep,n)],labels=[0,1,2,3])
         if (rep,n) in PREDICCIONES else None for n in MODELOS]
    vmax=max([int(cm.max()) for cm in cms if cm is not None] or [1])
    for ax,nombre,cm in zip(axs.ravel(),MODELOS,cms):
        if cm is None:ax.axis('off');continue
        ConfusionMatrixDisplay(cm,display_labels=['No viol.','Misoginia','Acoso','Viol. sexual']).plot(
            ax=ax,cmap='Purples',colorbar=False,values_format='d',xticks_rotation=35,im_kw={'vmin':0,'vmax':vmax})
        for item in ax.texts:item.set_fontsize(TAMANO_NUMEROS)
        row=TABLA[(TABLA['Representación']==rep)&(TABLA['Modelo de clasificación']==nombre)].iloc[0]
        ax.set_title(f'{nombre}\nF1-macro={row["F1-Score"]:.3f} | Acc={row["Accuracy"]:.3f}',fontsize=11)
        ax.set_xlabel('Predicción',fontsize=9);ax.set_ylabel('Clase real',fontsize=9)
        ax.tick_params(labelsize=8)
    plt.show();plt.close(fig)
print('\nFin. No se simularon conteos. El test ya se ha utilizado antes: resultados EXPLORATORIOS, no confirmatorios.')


### EXP-041 · V1, V2, V3, V4, V5, V6a, V6b, V6c, V6d

**Archivo de origen:** `Reportes_tesis.ipynb, celda 37`. **Referencias duplicadas:** 1. **Sintaxis de código:** Válida.

EXIST2021 ESPAÑOL | MULTICLASE | UNA SOLA CELDA COLAB
TRAIN EXCLUSIVO - NO CARGA TEST
CV anidada: 5 folds externos x 3 internos
Baselines fijos + todas las fusiones HMVT + HMVT optimizado por clasificador
Tuning exclusivo HMVT: pesos de blo

**Ejecución:** celda independiente o variante experimental; revisar archivos de entrada y dependencias antes de ejecutarla. Los resultados originales no se insertan en esta celda.

In [ ]:
import sys, subprocess, re, unicodedata, warnings, json
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.sparse import csr_matrix, hstack, issparse
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import Normalizer, MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import MultinomialNB, GaussianNB, ComplementNB
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import precision_recall_fscore_support, accuracy_score

SEED=42
N_FOLDS_EXTERNOS=5
N_FOLDS_INTERNOS=3
N_CANDIDATOS_POR_MODELO=8   
EVALUAR_VERSIONES_FIJAS=True
USAR_EMBEDDINGS=True
MODELO_EMB='sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
CLASES=['No violencia','Misoginia','Acoso','Violencia sexual']
ETIQUETAS=np.array([0,1,2,3])
MODELOS=['Regresión Logística','3-NN','SVM','Naive Bayes','Red neuronal','XGBoost','Random Forest','Árbol de decisión']
VERSIONES={
 'HMVT-V1':(1,0,.35,0),'HMVT-V2':(1,0,.35,.25),'HMVT-V3':(1,.75,.35,.25),
 'HMVT-V4':(1,1,.30,.50),'HMVT-V5':(1,1.35,.25,.35),
 'HMVT-V6a':(1,.75,.10,.10),'HMVT-V6b':(1,1.25,.15,.15),
 'HMVT-V6c':(1,.55,.30,.30),'HMVT-V6d':(1,1.50,.05,.15)
}
PESOS_EXTRAS=[(1,.65,.45,.5),(1,1.2,.30,.75),(1,.35,.10,.8),
              (1,.9,.6,.2),(1,1.0,0,.7),(1,.4,.25,1.2),
              (1,.8,.15,0),(1,1.4,.2,.5),(1,0,.35,.65)]
warnings.filterwarnings('ignore',category=FutureWarning)


def localizar():
    roots=[Path('/content'),Path.cwd(),Path('/mnt/data')]
    matches=[p for root in roots if root.exists() for p in sorted(root.glob('EXIST2021_train_reestructurado*.csv')) if p.is_file()]
    if not matches:
        try:
            from google.colab import files
            print('Sube SOLO EXIST2021_train_reestructurado.csv');files.upload()
        except ImportError: pass
        matches=[p for root in roots if root.exists() for p in sorted(root.glob('EXIST2021_train_reestructurado*.csv')) if p.is_file()]
    if not matches: raise FileNotFoundError('No se encontró TRAIN etiquetado en español.')
    return matches[0]

def limpiar_etiqueta(x):
    s=''.join(c for c in unicodedata.normalize('NFKD',str(x).strip().lower()) if not unicodedata.combining(c))
    return {'no violencia':0,'misoginia':1,'acoso':2,'violencia sexual':3,'0':0,'1':1,'2':2,'3':3}.get(s,np.nan)

df=pd.read_csv(localizar())
if not {'text','tipo_violencia'}.issubset(df.columns):raise ValueError('TRAIN requiere text y tipo_violencia')
if 'language' in df.columns:df=df[df.language.astype(str).str.lower().str.strip().isin(['es','spanish'])].copy()
df=df.dropna(subset=['text','tipo_violencia']).copy()
df['text']=df.text.astype(str).str.replace(r'\s+',' ',regex=True).str.strip()
df=df[df.text.str.len()>0].copy()
df['y']=df.tipo_violencia.map(limpiar_etiqueta)
if df.y.isna().any():raise ValueError('Hay etiquetas no reconocidas en TRAIN')
df['y']=df.y.astype(int)

df['_norm']=df.text.str.casefold()
conf=df.groupby('_norm').y.nunique(); problematicos=set(conf[conf>1].index)
if problematicos:
    print('Eliminados textos con etiquetas contradictorias:',len(problematicos))
    df=df[~df._norm.isin(problematicos)].copy()
antes=len(df);df=df.drop_duplicates('_norm').reset_index(drop=True)
print('TRAIN único:',len(df),'| duplicados exactos excluidos:',antes-len(df),
      '| clases:',df.y.value_counts().sort_index().to_dict())
if set(df.y)!={0,1,2,3}:raise ValueError('TRAIN requiere las cuatro clases.')
textos=df.text.to_numpy();y=df.y.to_numpy();N=len(df)
if min(Counter(y).values())<N_FOLDS_EXTERNOS*N_FOLDS_INTERNOS:raise ValueError('Muy pocos datos en una clase para CV anidada')


PALABRAS={'insulto':['puta','perra','zorra','idiota','estupida','estupido'],
 'genero':['mujer','mujeres','hombre','hombres','feminista'],
 'acoso':['acoso','acosar','hostigar','amenaza','amenazar'],
 'sexual':['sexo','sexual','violacion','violar','desnuda'],
 'negacion':['no','nunca','jamas']}
RX={k:[re.compile(r'(?<!\w)'+re.escape(w)+r'(?!\w)') for w in v] for k,v in PALABRAS.items()}
def discurso(ts):
    data=[]
    for t in ts:
        s=''.join(c for c in unicodedata.normalize('NFKD',str(t).lower()) if not unicodedata.combining(c))
        toks=re.findall(r'\b\w+\b',s)
        row=[np.log1p(sum(len(rx.findall(s)) for rx in rxs)) for rxs in RX.values()]
        row += [np.log1p(len(toks)),len(set(toks))/max(1,len(toks)),
                np.log1p(s.count('!')),np.log1p(s.count('?')),
                int('@' in s),int('#' in s),int('http' in s)]
        data.append(row)
    return np.array(data,dtype=np.float32)

class Extractor:
    def fit(self,tx):
        self.w=TfidfVectorizer(ngram_range=(1,2),max_features=18000,min_df=2,sublinear_tf=True,strip_accents='unicode')
        self.c=TfidfVectorizer(analyzer='char_wb',ngram_range=(3,5),max_features=20000,min_df=2,sublinear_tf=True)
        self.ng=CountVectorizer(ngram_range=(1,2),max_features=10000,min_df=2)
        self.bow=CountVectorizer(max_features=10000,min_df=2)
        W=self.w.fit_transform(tx);self.c.fit(tx);self.ng.fit(tx);self.bow.fit(tx)
        k=min(60,W.shape[0]-1,W.shape[1]-1)
        self.svd=TruncatedSVD(n_components=k,random_state=SEED).fit(W)
        ls=Normalizer().fit_transform(self.svd.transform(W))
        self.scale_l=MinMaxScaler().fit(ls)
        self.scale_d=MinMaxScaler().fit(discurso(tx))
        return self
    def transform(self,tx):
        w=self.w.transform(tx).astype(np.float32)
        c=self.c.transform(tx).astype(np.float32)
        ll=Normalizer().fit_transform(self.svd.transform(w))
        l=csr_matrix(np.clip(self.scale_l.transform(ll),0,None).astype(np.float32))
        d=csr_matrix(np.clip(self.scale_d.transform(discurso(tx)),0,None).astype(np.float32))
        return (w,c,l,d)
    def traditional(self,tx,blocks):
        return {'TF-IDF':blocks[0], 'N-gramas':self.ng.transform(tx), 'BoW':self.bow.transform(tx)}

def fusionar(blocks,pesos):
    
    return hstack([b.multiply(float(z)) for b,z in zip(blocks,pesos) if z>0],format='csr',dtype=np.float32)


def reequilibrar(X,y,metodo,seed):
    yy=np.asarray(y)
    if metodo not in ('ros','ros_parcial'):return X,yy,None
    rng=np.random.default_rng(seed);indices=np.arange(len(yy));conteos=np.bincount(yy,minlength=4)
    objetivo=int(conteos.max())
    seleccion=[]
    for cl in ETIQUETAS:
        ix=indices[yy==cl]
        meta=objetivo if metodo=='ros' else max(len(ix),int(objetivo*.60))
        seleccion.append(rng.choice(ix,size=meta,replace=(meta>len(ix))))
    seleccion=np.concatenate(seleccion);rng.shuffle(seleccion)
    return X[seleccion],yy[seleccion],None

def pesos_muestra(y,metodo):
    yy=np.asarray(y);conteos=np.bincount(yy,minlength=4)
    pesos=len(yy)/(4*np.maximum(conteos,1))
    if metodo=='sqrt_balanced': pesos=np.sqrt(pesos)
    if metodo=='minority_boost':pesos[2:]*=1.35
    vals=pesos[yy].astype(np.float32)
    return vals/np.mean(vals)

def modelo(nombre,config=None,baseline=False):
    cfg=config or {}
    tipo=cfg.get('modelo',nombre)
    c=cfg.get('C',1.0)
    if tipo=='Regresión Logística':return LogisticRegression(C=c,max_iter=2200,class_weight='balanced',random_state=SEED)
    if tipo=='SVM':return LinearSVC(C=c,class_weight='balanced',max_iter=7000,random_state=SEED)
    if tipo=='3-NN':return KNeighborsClassifier(n_neighbors=cfg.get('k',3),weights=cfg.get('weights','uniform'))
    if tipo=='Naive Bayes':return MultinomialNB(alpha=cfg.get('alpha',1.0))
    if tipo=='ComplementNB':return ComplementNB(alpha=cfg.get('alpha',1.0),norm=cfg.get('norm',False))
    if tipo=='GaussianNB':return GaussianNB()
    if tipo=='Red neuronal':return MLPClassifier(hidden_layer_sizes=(cfg.get('hidden',64),),alpha=cfg.get('alpha',.0001),max_iter=200,early_stopping=True,random_state=SEED)
    if tipo=='Random Forest':return RandomForestClassifier(n_estimators=cfg.get('trees',140),max_depth=cfg.get('depth',None),min_samples_leaf=cfg.get('leaf',1),class_weight=cfg.get('balance','balanced_subsample'),n_jobs=-1,random_state=SEED)
    if tipo=='Árbol de decisión':return DecisionTreeClassifier(max_depth=cfg.get('depth',18),min_samples_leaf=cfg.get('leaf',1),class_weight=cfg.get('balance','balanced'),random_state=SEED)
    if tipo=='XGBoost':
        from xgboost import XGBClassifier
        return XGBClassifier(objective='multi:softprob',num_class=4,n_estimators=cfg.get('trees',150),max_depth=cfg.get('depth',4),learning_rate=cfg.get('lr',.07),subsample=.9,colsample_bytree=.9,tree_method='hist',eval_metric='mlogloss',random_state=SEED,n_jobs=2)
    raise ValueError(tipo)

def predecir(nombre,Xt,yt,Xv,cfg=None,estrategia='none',seed=SEED):
    cfg=cfg or {}
    Xfit,yfit,_=reequilibrar(Xt,yt,estrategia,seed)
    m=modelo(nombre,cfg)
    sw=pesos_muestra(yfit,estrategia) if estrategia in ('sqrt_balanced','minority_boost') else None
    if sw is not None and nombre in ('XGBoost','Random Forest','Árbol de decisión','Regresión Logística','SVM'):
        m.fit(Xfit,yfit,sample_weight=sw)
    else:
        m.fit(Xfit,yfit)
    return m.predict(Xv)

def metricas(yt,yp):
    p,r,f,_=precision_recall_fscore_support(yt,yp,average='macro',labels=ETIQUETAS,zero_division=0)
    rc=precision_recall_fscore_support(yt,yp,average=None,labels=ETIQUETAS,zero_division=0)[1]
    return {'Precisión macro':float(p),'Recall macro':float(r),'F1 macro':float(f),
            'Accuracy':float(accuracy_score(yt,yp)),
            'Recall Acoso':float(rc[2]),'Recall Violencia sexual':float(rc[3])}



def configuraciones(nombre):
    comunes=[
      ('HMVT-V6a','none'),('HMVT-V3','ros_parcial'),('HMVT-V6c','sqrt_balanced'),
      ('HMVT-V2','ros'),('HMVT-V5','minority_boost'),('HMVT-V6d','ros_parcial'),
      ('HMVT-V4','none'),('HMVT-V6b','ros')]
    extras=[f'Extra-{i+1}' for i in range(len(PESOS_EXTRAS))]
    combinaciones=comunes+[(v,s) for v,s in zip(extras,['ros','ros_parcial','sqrt_balanced','none','minority_boost','ros','ros_parcial','sqrt_balanced','ros'])]
    parametros={
      'Regresión Logística':[{'C':.1},{'C':1},{'C':5}],
      'SVM':[{'C':.05},{'C':.5},{'C':2}],
      '3-NN':[{'k':3,'weights':'distance'},{'k':5,'weights':'distance'},{'k':9,'weights':'distance'}],
      'Naive Bayes':[{'modelo':'ComplementNB','alpha':.1},{'modelo':'ComplementNB','alpha':1},{'modelo':'Naive Bayes','alpha':.1}],
      'Red neuronal':[{'hidden':64,'alpha':.001},{'hidden':128,'alpha':.01}],
      'XGBoost':[{'depth':3,'trees':180,'lr':.06},{'depth':5,'trees':180,'lr':.06}],
      'Random Forest':[{'trees':200,'leaf':1,'balance':'balanced_subsample'},{'trees':200,'leaf':2,'balance':'balanced'}],
      'Árbol de decisión':[{'depth':12,'leaf':2,'balance':'balanced'},{'depth':24,'leaf':4,'balance':'balanced'}]}
    allc=[(v,bal,cfg) for v,bal in combinaciones for cfg in parametros[nombre]]
    rng=np.random.default_rng(SEED+MODELOS.index(nombre))
    chosen=rng.choice(len(allc),size=min(N_CANDIDATOS_POR_MODELO,len(allc)),replace=False)
    
    base=('HMVT-V6a','none',parametros[nombre][0])
    candidatos=[base]+[allc[int(i)] for i in chosen if allc[int(i)]!=base]
    return candidatos[:N_CANDIDATOS_POR_MODELO]

TODOS_PESOS={**VERSIONES,**{f'Extra-{i+1}':p for i,p in enumerate(PESOS_EXTRAS)}}


embeddings=None
if USAR_EMBEDDINGS:
    try:from sentence_transformers import SentenceTransformer
    except ImportError:
        subprocess.check_call([sys.executable,'-m','pip','install','-q','sentence-transformers'])
        from sentence_transformers import SentenceTransformer
    enc=SentenceTransformer(MODELO_EMB)
    enc.eval()
    embeddings=enc.encode(list(textos),batch_size=64,show_progress_bar=True,
                          normalize_embeddings=False,convert_to_numpy=True).astype(np.float32)
    del enc

def baseline(nombre,Xt,yt,Xv,es_embedding=False):
    
    if es_embedding and nombre=='Naive Bayes':
        m=GaussianNB()
    else:
        m=modelo(nombre)
    m.fit(Xt,yt)
    return m.predict(Xv)


try: import xgboost
except ImportError:
    subprocess.check_call([sys.executable,'-m','pip','install','-q','xgboost'])
    import xgboost

externo=StratifiedKFold(n_splits=N_FOLDS_EXTERNOS,shuffle=True,random_state=SEED)
REGISTROS=[];SELECCIONES=[];ERRORES=[]
print('\nCV ANIDADA:',N_FOLDS_EXTERNOS,'externos x',N_FOLDS_INTERNOS,'internos.')
print('IMPORTANTE: se usa SOLO TRAIN. Embeddings sin fine-tuning ni tuning.')

for fold,(outer_tr,outer_va) in enumerate(externo.split(textos,y),start=1):
    print(f'\n=== FOLD EXTERNO {fold}/{N_FOLDS_EXTERNOS} ===',flush=True)
    ti,yi=textos[outer_tr],y[outer_tr]
    tv,yv=textos[outer_va],y[outer_va]
    
    ext=Extractor().fit(ti)
    bt=ext.transform(ti);bv=ext.transform(tv)
    reps={n:(ext.traditional(ti,bt)[n],ext.traditional(tv,bv)[n]) for n in ('TF-IDF','N-gramas','BoW')}
    if USAR_EMBEDDINGS:reps['Embeddings']=(embeddings[outer_tr],embeddings[outer_va])
    if EVALUAR_VERSIONES_FIJAS:
        for version,w in VERSIONES.items():reps[version]=(fusionar(bt,w),fusionar(bv,w))
    
    inner=StratifiedKFold(n_splits=N_FOLDS_INTERNOS,shuffle=True,random_state=SEED+fold)
    caches=[]
    for it,iv in inner.split(ti,yi):
        e=Extractor().fit(ti[it]);a=e.transform(ti[it]);b=e.transform(ti[iv])
        caches.append((a,b,yi[it],yi[iv]))
    for nombre in MODELOS:
        
        for rep,(Xt,Xv) in reps.items():
            try:
                yp=baseline(nombre,Xt,yi,Xv,rep=='Embeddings')
                REGISTROS.append({'Fold':fold,'Representación':rep,'Modelo de clasificación':nombre,**metricas(yv,yp)})
            except Exception as err:
                ERRORES.append((fold,rep,nombre,str(err)))
        
        mejor=None
        for ver,bal,cfg in configuraciones(nombre):
            scores=[];recalls=[]
            for j,(a,b,ya,yb) in enumerate(caches):
                try:
                    Xa=fusionar(a,TODOS_PESOS[ver]);Xb=fusionar(b,TODOS_PESOS[ver])
                    yp=predecir(nombre,Xa,ya,Xb,cfg,bal,seed=SEED+fold*100+j)
                    met=metricas(yb,yp)
                    scores.append(met['F1 macro'])
                    recalls.append((met['Recall Acoso']+met['Recall Violencia sexual'])/2)
                except Exception as err:
                    ERRORES.append((fold,'CV interna '+ver,nombre,str(err)))
                    scores=[];break
            if len(scores)!=N_FOLDS_INTERNOS:continue
            
            clave=(float(np.mean(scores)),float(np.mean(recalls)))
            if mejor is None or clave>mejor[0]:mejor=(clave,ver,bal,cfg)
        if mejor is None:
            ERRORES.append((fold,'HMVT-Optimizado',nombre,'No hubo candidatos válidos'))
            continue
        clave,ver,bal,cfg=mejor
        SELECCIONES.append({'Fold':fold,'Clasificador':nombre,'Versión':ver,
                            'Balanceo':bal,'Configuración':str(cfg),
                            'F1 validación interna':clave[0],
                            'Recall minoritario interno':clave[1]})
        try:
            Xa=fusionar(bt,TODOS_PESOS[ver]);Xb=fusionar(bv,TODOS_PESOS[ver])
            yp=predecir(nombre,Xa,yi,Xb,cfg,bal,seed=SEED+fold)
            REGISTROS.append({'Fold':fold,'Representación':'HMVT-Optimizado',
                              'Modelo de clasificación':nombre,**metricas(yv,yp)})
            print(f' {nombre}: {ver}, {bal} | F1 interno={clave[0]:.4f}',flush=True)
        except Exception as err:
            ERRORES.append((fold,'HMVT-Optimizado',nombre,str(err)))

if not REGISTROS:raise RuntimeError('No se produjeron resultados; revisa ERRORES.')
resultados=pd.DataFrame(REGISTROS)
columnas=['Precisión macro','Recall macro','F1 macro','Accuracy','Recall Acoso','Recall Violencia sexual']
resumen=resultados.groupby(['Representación','Modelo de clasificación'],sort=False)[columnas].agg(['mean','std'])
resumen.columns=[f'{m} ({s})' for m,s in resumen.columns]
resumen=resumen.reset_index()
resumen['Folds evaluados']=resultados.groupby(['Representación','Modelo de clasificación'],sort=False)['Fold'].nunique().values
orden=['TF-IDF','N-gramas','BoW','Embeddings',*VERSIONES.keys(),'HMVT-Optimizado']
resumen['Representación']=pd.Categorical(resumen['Representación'],categories=orden,ordered=True)
resumen['Modelo de clasificación']=pd.Categorical(resumen['Modelo de clasificación'],categories=MODELOS,ordered=True)
resumen=resumen.sort_values(['Representación','Modelo de clasificación']).reset_index(drop=True)

tabla=resumen[['Representación','Modelo de clasificación','Folds evaluados']].copy()
for m in columnas:
    a=resumen[f'{m} (mean)'];b=resumen[f'{m} (std)'].fillna(0)
    tabla[m]=[f'{x:.4f} ± {s:.4f}' for x,s in zip(a,b)]
print('\nTABLA MAESTRA - TRAIN ÚNICAMENTE - MÉTRICAS DE CV EXTERNA (MEDIA ± DESV. ESTÁNDAR)')
display(tabla[['Representación','Modelo de clasificación','Precisión macro','Recall macro','F1 macro','Accuracy','Folds evaluados']])
print('\nHMVT: RECALL DE ACOSO Y VIOLENCIA SEXUAL (CV EXTERNA)')
display(tabla[tabla['Representación'].astype(str).str.startswith('HMVT')][['Representación','Modelo de clasificación','F1 macro','Recall Acoso','Recall Violencia sexual']])
print('\nCONFIGURACIONES SELECCIONADAS DENTRO DE TRAIN (POR FOLD EXTERNO):')
display(pd.DataFrame(SELECCIONES))
print('\nNÚMERO DE ERRORES:',len(ERRORES))
if ERRORES:print('Primeros errores:',ERRORES[:12])
print('\nFin. TEST no fue cargado ni utilizado. No se inventaron resultados.')


### EXP-042 · V1, V2, V3, V4, V5, V6a, V6b, V6c, V6d

**Archivo de origen:** `Reportes_tesis.ipynb, celda 39`. **Referencias duplicadas:** 1. **Sintaxis de código:** Válida.

EXIST2021 - ESPAÑOL - MULTICLASE
HMVT: OPTIMIZACION + ABLACION + DESBALANCE
CODIGO COMPLETO PARA UNA SOLA CELDA DE GOOGLE COLAB
SOLO USA TRAIN. NO CARGA TEST.
Validacion cruzada anidada:
5 folds externos
3 folds internos
4 clases:
0 No viol

**Ejecución:** celda independiente o variante experimental; revisar archivos de entrada y dependencias antes de ejecutarla. Los resultados originales no se insertan en esta celda.

In [ ]:
import sys
import re
import subprocess
import unicodedata
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd

from IPython.display import display
from scipy.sparse import csr_matrix, hstack, issparse

from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import (
    TfidfVectorizer,
    CountVectorizer
)
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import (
    MultinomialNB,
    ComplementNB,
    GaussianNB
)
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    precision_recall_fscore_support,
    accuracy_score
)

warnings.filterwarnings("ignore", category=FutureWarning)





SEED = 42

FOLDS_EXTERNOS = 5
FOLDS_INTERNOS = 3

MOSTRAR_ABLACION = True
USAR_EMBEDDINGS = True

CLASES = [
    "No violencia",
    "Misoginia",
    "Acoso",
    "Violencia sexual"
]

IDS_CLASES = [0, 1, 2, 3]

MODELOS = [
    "Regresión Logística",
    "3-NN",
    "SVM",
    "Naive Bayes",
    "Red neuronal",
    "XGBoost",
    "Random Forest",
    "Árbol de decisión"
]



VERSIONES = {
    "HMVT-V1":  (1.0, 0.00, 0.35, 0.00),
    "HMVT-V2":  (1.0, 0.00, 0.35, 0.25),
    "HMVT-V3":  (1.0, 0.75, 0.35, 0.25),
    "HMVT-V4":  (1.0, 1.00, 0.30, 0.50),
    "HMVT-V5":  (1.0, 1.35, 0.25, 0.35),
    "HMVT-V6a": (1.0, 0.75, 0.10, 0.10),
    "HMVT-V6b": (1.0, 1.25, 0.15, 0.15),
    "HMVT-V6c": (1.0, 0.55, 0.30, 0.30),
    "HMVT-V6d": (1.0, 1.50, 0.05, 0.15)
}



EXTRAS = {
    "Extra-1": (1.0, 0.75, 0.20, 0.60),
    "Extra-2": (1.0, 1.25, 0.15, 0.65),
    "Extra-3": (1.0, 0.50, 0.50, 0.50),
    "Extra-4": (1.0, 0.75, 0.20, 1.00),
    "Extra-5": (1.0, 1.00, 0.00, 0.75)
}

TODOS_PESOS = {**VERSIONES, **EXTRAS}












def candidatos_train():

    carpetas = [
        Path("/content"),
        Path.cwd(),
        Path("/mnt/data")
    ]

    encontrados = []

    for carpeta in carpetas:

        if not carpeta.exists():
            continue

        for archivo in carpeta.glob("*.csv"):

            nombre = archivo.name.lower()

            if (
                "exist2021" in nombre
                and "train" in nombre
                and "reestructurado" in nombre
                and "test" not in nombre
            ):
                encontrados.append(archivo)

    
    unicos = {}

    for archivo in encontrados:
        unicos[str(archivo.resolve())] = archivo

    return sorted(
        unicos.values(),
        key=lambda p: (
            p.name != "EXIST2021_train_reestructurado.csv",
            len(p.name),
            p.name
        )
    )


def localizar_train():

    archivos = candidatos_train()

    if not archivos:

        try:
            from google.colab import files

            print(
                "Sube el archivo TRAIN de EXIST2021 "
                "en español."
            )

            files.upload()

        except ImportError:
            print(
                "No se detecto Google Colab. "
                "Coloca el CSV en el directorio actual."
            )

        archivos = candidatos_train()

    if not archivos:
        raise FileNotFoundError(
            "No se encontro el TRAIN. "
            "Sube EXIST2021_train_reestructurado.csv "
            "o una copia con (1), (2), etc."
        )

    
    for archivo in archivos:

        try:
            cabecera = pd.read_csv(
                archivo,
                nrows=2,
                encoding="utf-8-sig"
            )

            columnas = {
                str(c).strip().lower()
                for c in cabecera.columns
            }

            if (
                "text" in columnas
                and (
                    "tipo_violencia" in columnas
                    or "tipo de violencia" in columnas
                )
            ):
                return archivo

        except Exception:
            continue

    raise ValueError(
        "Se encontraron archivos TRAIN, pero ninguno "
        "contiene text y tipo_violencia."
    )


RUTA_TRAIN = localizar_train()

print("Archivo TRAIN:", RUTA_TRAIN)





def normalizar_etiqueta(valor):

    texto = str(valor).strip().lower()

    return "".join(
        c for c in unicodedata.normalize(
            "NFKD",
            texto
        )
        if not unicodedata.combining(c)
    )


MAPA_ETIQUETAS = {
    "0": 0,
    "1": 1,
    "2": 2,
    "3": 3,
    "no violencia": 0,
    "misoginia": 1,
    "acoso": 2,
    "violencia sexual": 3
}


df = pd.read_csv(
    RUTA_TRAIN,
    encoding="utf-8-sig"
)

df.columns = [
    str(c).strip()
    for c in df.columns
]

if (
    "tipo_violencia" not in df.columns
    and "tipo de violencia" in df.columns
):
    df = df.rename(
        columns={
            "tipo de violencia": "tipo_violencia"
        }
    )

if not {"text", "tipo_violencia"}.issubset(df.columns):
    raise ValueError(
        "TRAIN necesita las columnas "
        "'text' y 'tipo_violencia'. "
        f"Encontradas: {list(df.columns)}"
    )

if "language" in df.columns:

    df = df[
        df["language"]
        .astype(str)
        .str.strip()
        .str.lower()
        .isin(["es", "spanish"])
    ].copy()

df = df.dropna(
    subset=["text", "tipo_violencia"]
).copy()

df["text"] = (
    df["text"]
    .astype(str)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

df = df[
    df["text"].str.len() > 0
].copy()

df["y"] = (
    df["tipo_violencia"]
    .map(normalizar_etiqueta)
    .map(MAPA_ETIQUETAS)
)

if df["y"].isna().any():

    desconocidas = df.loc[
        df["y"].isna(),
        "tipo_violencia"
    ].unique()

    raise ValueError(
        f"Etiquetas desconocidas: {desconocidas}"
    )

df["y"] = df["y"].astype(int)


df["_text_key"] = (
    df["text"]
    .str.casefold()
)

conflictos = (
    df.groupby("_text_key")["y"].nunique()
)

claves_conflictivas = set(
    conflictos[conflictos > 1].index
)

if claves_conflictivas:

    print(
        "Textos con etiquetas contradictorias:",
        len(claves_conflictivas)
    )

    df = df[
        ~df["_text_key"].isin(claves_conflictivas)
    ].copy()

df = df.drop_duplicates(
    subset=["_text_key"]
)

df = df.drop(
    columns=["_text_key"]
).reset_index(drop=True)

if set(df["y"]) != set(IDS_CLASES):
    raise ValueError(
        "TRAIN debe contener las cuatro clases 0,1,2,3."
    )

X_TEXTOS = df["text"].to_numpy()
Y = df["y"].to_numpy()

print("\nTRAIN FINAL:", len(df))

print(
    "Distribucion:",
    dict(sorted(Counter(Y).items()))
)





PATRONES = {
    "insultos": [
        "puta", "perra", "zorra",
        "idiota", "estupida", "estupido"
    ],
    "genero": [
        "mujer", "mujeres", "hombre",
        "hombres", "feminista"
    ],
    "acoso": [
        "acoso", "acosar", "hostigar",
        "amenaza"
    ],
    "sexual": [
        "sexo", "sexual",
        "violacion", "violencia sexual"
    ],
    "negacion": [
        "no", "nunca", "jamas"
    ]
}

RX = {
    nombre: [
        re.compile(
            r"(?<!\w)"
            + re.escape(palabra)
            + r"(?!\w)"
        )
        for palabra in palabras
    ]
    for nombre, palabras in PATRONES.items()
}


def discurso(textos):

    filas = []

    for texto in textos:

        s = normalizar_etiqueta(texto)

        tokens = re.findall(r"\b\w+\b", s)

        fila = [
            np.log1p(
                sum(
                    len(p.findall(s))
                    for p in patrones
                )
            )
            for patrones in RX.values()
        ]

        fila.extend([
            np.log1p(len(tokens)),
            len(set(tokens)) / max(1, len(tokens)),
            np.log1p(s.count("!")),
            np.log1p(s.count("?")),
            int("@" in s),
            int("#" in s),
            int("http" in s)
        ])

        filas.append(fila)

    return np.asarray(
        filas,
        dtype=np.float32
    )








class Extractor:

    def fit(self, textos):

        self.word = TfidfVectorizer(
            max_features=20000,
            ngram_range=(1, 2),
            min_df=2,
            sublinear_tf=True,
            strip_accents="unicode"
        )

        self.char = TfidfVectorizer(
            analyzer="char_wb",
            ngram_range=(3, 5),
            max_features=25000,
            min_df=2,
            sublinear_tf=True
        )

        self.ng = CountVectorizer(
            ngram_range=(1, 2),
            max_features=10000,
            min_df=2
        )

        self.bow = CountVectorizer(
            max_features=10000,
            min_df=2
        )

        w = self.word.fit_transform(textos)

        self.char.fit(textos)
        self.ng.fit(textos)
        self.bow.fit(textos)

        k = min(
            60,
            w.shape[0] - 1,
            w.shape[1] - 1
        )

        if k < 2:
            raise ValueError(
                "Insuficientes caracteristicas para LSA."
            )

        self.svd = TruncatedSVD(
            n_components=k,
            random_state=SEED
        )

        lsa_train = self.svd.fit_transform(w)

        self.lscale = MinMaxScaler().fit(
            lsa_train
        )

        self.dscale = MinMaxScaler().fit(
            discurso(textos)
        )

        return self

    def transform(self, textos):

        w = self.word.transform(
            textos
        ).astype(np.float32)

        c = self.char.transform(
            textos
        ).astype(np.float32)

        l = self.svd.transform(w)

        l = np.clip(
            self.lscale.transform(l),
            0,
            1
        ).astype(np.float32)

        d = np.clip(
            self.dscale.transform(
                discurso(textos)
            ),
            0,
            1
        ).astype(np.float32)

        return (
            w,
            c,
            csr_matrix(l),
            csr_matrix(d)
        )


def fusionar(bloques, pesos):

    return hstack(
        [
            bloque.multiply(float(peso))
            for bloque, peso in zip(
                bloques, pesos
            )
            if peso > 0
        ],
        format="csr",
        dtype=np.float32
    )





def balancear(X, y, estrategia, semilla=SEED):

    y = np.asarray(y)

    if estrategia in [
        "none",
        "balanced",
        "sqrt_balanced"
    ]:
        return X, y, None

    rng = np.random.default_rng(semilla)

    clases, cantidades = np.unique(
        y,
        return_counts=True
    )

    maximo = int(cantidades.max())

    indices = []

    for clase, cantidad in zip(
        clases, cantidades
    ):

        disponibles = np.where(
            y == clase
        )[0]

        if estrategia == "ros":

            objetivo = maximo

        elif estrategia == "ros_parcial":

            objetivo = max(
                int(cantidad),
                int(maximo * 0.65)
            )

        elif estrategia == "minority_boost":

            if clase in [2, 3]:

                objetivo = max(
                    int(cantidad),
                    int(maximo * 0.80)
                )

            else:
                objetivo = int(cantidad)

        else:
            raise ValueError(
                f"Balanceo desconocido: {estrategia}"
            )

        seleccionados = rng.choice(
            disponibles,
            size=objetivo,
            replace=objetivo > cantidad
        )

        indices.extend(
            seleccionados.tolist()
        )

    indices = np.asarray(
        indices,
        dtype=int
    )

    rng.shuffle(indices)

    return X[indices], y[indices], None


def pesos_muestras(y, estrategia):

    if estrategia not in [
        "balanced",
        "sqrt_balanced"
    ]:
        return None

    clases, cantidades = np.unique(
        y,
        return_counts=True
    )

    n = len(y)
    k = len(clases)

    pesos = {}

    for clase, cantidad in zip(
        clases,
        cantidades
    ):

        peso = n / (k * cantidad)

        if estrategia == "sqrt_balanced":
            peso = np.sqrt(peso)

        pesos[int(clase)] = peso

    return np.array(
        [pesos[int(c)] for c in y],
        dtype=np.float32
    )






try:
    from xgboost import XGBClassifier

except ImportError:

    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "xgboost"
    ])

    from xgboost import XGBClassifier


def crear_modelo(nombre, config=None, balance="none"):

    config = config or {}

    pesos_clase = (
        "balanced"
        if balance == "balanced"
        else None
    )

    if nombre == "Regresión Logística":

        return LogisticRegression(
            C=float(config.get("C", 1.0)),
            class_weight=pesos_clase,
            max_iter=2500,
            random_state=SEED
        )

    if nombre == "3-NN":

        return KNeighborsClassifier(
            n_neighbors=int(config.get("k", 3)),
            weights=config.get(
                "weights",
                "uniform"
            ),
            metric="cosine",
            algorithm="brute"
        )

    if nombre == "SVM":

        return LinearSVC(
            C=float(config.get("C", 1.0)),
            class_weight=pesos_clase,
            max_iter=7000,
            random_state=SEED
        )

    if nombre == "Naive Bayes":

        tipo = config.get(
            "modelo",
            "MultinomialNB"
        )

        alpha = float(
            config.get("alpha", 1.0)
        )

        if tipo == "ComplementNB":

            return ComplementNB(
                alpha=alpha
            )

        return MultinomialNB(
            alpha=alpha
        )

    if nombre == "Red neuronal":

        return MLPClassifier(
            hidden_layer_sizes=(
                int(config.get("hidden", 64)),
            ),
            alpha=float(
                config.get("alpha", 0.001)
            ),
            max_iter=160,
            early_stopping=True,
            random_state=SEED
        )

    if nombre == "XGBoost":

        return XGBClassifier(
            objective="multi:softprob",
            num_class=4,
            max_depth=int(
                config.get("depth", 3)
            ),
            n_estimators=int(
                config.get("trees", 140)
            ),
            learning_rate=float(
                config.get("lr", 0.07)
            ),
            subsample=0.9,
            colsample_bytree=0.9,
            tree_method="hist",
            eval_metric="mlogloss",
            random_state=SEED,
            n_jobs=2
        )

    if nombre == "Random Forest":

        return RandomForestClassifier(
            n_estimators=int(
                config.get("trees", 140)
            ),
            min_samples_leaf=int(
                config.get("leaf", 1)
            ),
            max_depth=config.get(
                "depth",
                None
            ),
            class_weight=pesos_clase,
            random_state=SEED,
            n_jobs=-1
        )

    if nombre == "Árbol de decisión":

        return DecisionTreeClassifier(
            max_depth=config.get(
                "depth",
                18
            ),
            min_samples_leaf=int(
                config.get("leaf", 2)
            ),
            class_weight=pesos_clase,
            random_state=SEED
        )

    raise ValueError(nombre)






def entrenar_predecir(
    nombre,
    Xtr,
    ytr,
    Xev,
    config=None,
    balance="none",
    semilla=SEED
):

    if balance in [
        "ros",
        "ros_parcial",
        "minority_boost"
    ]:

        Xtr, ytr, _ = balancear(
            Xtr,
            ytr,
            balance,
            semilla
        )

    modelo = crear_modelo(
        nombre,
        config,
        balance
    )

    pesos = pesos_muestras(
        ytr,
        balance
    )

    if nombre == "Naive Bayes" and not issparse(Xtr):

        
        modelo = GaussianNB()
        modelo.fit(Xtr, ytr)

    elif pesos is not None and nombre == "XGBoost":

        modelo.fit(
            Xtr,
            ytr,
            sample_weight=pesos
        )

    elif pesos is not None and nombre == "Naive Bayes":

        modelo.fit(
            Xtr,
            ytr,
            sample_weight=pesos
        )

    elif pesos is not None and nombre in [
        "Regresión Logística",
        "SVM",
        "Random Forest",
        "Árbol de decisión"
    ]:

        modelo.fit(
            Xtr,
            ytr,
            sample_weight=pesos
        )

    else:

        modelo.fit(
            Xtr,
            ytr
        )

    return modelo.predict(
        Xev
    ).astype(int)






def metricas(y_real, y_pred):

    precision, recall, f1, _ = (
        precision_recall_fscore_support(
            y_real,
            y_pred,
            labels=IDS_CLASES,
            average="macro",
            zero_division=0
        )
    )

    _, recalls, f1_clases, _ = (
        precision_recall_fscore_support(
            y_real,
            y_pred,
            labels=IDS_CLASES,
            average=None,
            zero_division=0
        )
    )

    return {
        "Precisión": precision,
        "Recall": recall,
        "F1-Score": f1,
        "Accuracy": accuracy_score(
            y_real, y_pred
        ),
        "Recall Acoso": recalls[2],
        "Recall Violencia sexual": recalls[3],
        "F1 Acoso": f1_clases[2],
        "F1 Violencia sexual": f1_clases[3]
    }









def candidatos_para(nombre):

    if nombre == "Regresión Logística":

        return [
            ("HMVT-V3", {"C": 1}, "balanced"),
            ("HMVT-V4", {"C": 5}, "none"),
            ("HMVT-V6a", {"C": 0.1}, "balanced"),
            ("Extra-2", {"C": 1}, "balanced"),
            ("Extra-5", {"C": 0.5}, "ros_parcial"),
            ("HMVT-V4", {"C": 1}, "minority_boost")
        ]

    if nombre == "3-NN":

        return [
            ("HMVT-V5", {"k": 5, "weights": "distance"}, "minority_boost"),
            ("HMVT-V6d", {"k": 3, "weights": "distance"}, "ros_parcial"),
            ("HMVT-V4", {"k": 5, "weights": "distance"}, "ros"),
            ("Extra-1", {"k": 7, "weights": "distance"}, "ros_parcial"),
            ("Extra-5", {"k": 5, "weights": "distance"}, "minority_boost")
        ]

    if nombre == "SVM":

        return [
            ("HMVT-V3", {"C": 1}, "balanced"),
            ("HMVT-V4", {"C": 0.5}, "none"),
            ("HMVT-V6c", {"C": 0.5}, "balanced"),
            ("Extra-2", {"C": 0.5}, "minority_boost"),
            ("Extra-5", {"C": 0.5}, "minority_boost"),
            ("HMVT-V6a", {"C": 0.1}, "ros_parcial")
        ]

    if nombre == "Naive Bayes":

        return [
            ("HMVT-V6a", {"modelo": "ComplementNB", "alpha": 0.1}, "none"),
            ("Extra-4", {"modelo": "ComplementNB", "alpha": 0.1}, "none"),
            ("Extra-4", {"modelo": "ComplementNB", "alpha": 1}, "ros_parcial"),
            ("HMVT-V4", {"modelo": "ComplementNB", "alpha": 0.5}, "balanced"),
            ("Extra-2", {"modelo": "MultinomialNB", "alpha": 0.1}, "ros")
        ]

    if nombre == "Red neuronal":

        return [
            ("Extra-1", {"hidden": 128, "alpha": 0.01}, "ros"),
            ("HMVT-V4", {"hidden": 128, "alpha": 0.001}, "ros_parcial"),
            ("HMVT-V6a", {"hidden": 64, "alpha": 0.01}, "minority_boost"),
            ("Extra-5", {"hidden": 128, "alpha": 0.001}, "ros"),
            ("HMVT-V3", {"hidden": 64, "alpha": 0.001}, "none")
        ]

    if nombre == "XGBoost":

        return [
            ("HMVT-V3", {"depth": 5, "trees": 180, "lr": 0.06}, "ros_parcial"),
            ("HMVT-V6b", {"depth": 3, "trees": 180, "lr": 0.06}, "ros"),
            ("Extra-3", {"depth": 3, "trees": 180, "lr": 0.06}, "sqrt_balanced"),
            ("HMVT-V6a", {"depth": 3, "trees": 140, "lr": 0.07}, "balanced"),
            ("Extra-1", {"depth": 4, "trees": 180, "lr": 0.05}, "minority_boost")
        ]

    if nombre == "Random Forest":

        return [
            ("HMVT-V6d", {"trees": 200, "leaf": 1}, "ros_parcial"),
            ("HMVT-V6b", {"trees": 200, "leaf": 1}, "ros"),
            ("Extra-1", {"trees": 200, "leaf": 2}, "balanced"),
            ("HMVT-V3", {"trees": 200, "leaf": 1}, "minority_boost"),
            ("HMVT-V4", {"trees": 140, "leaf": 2}, "none")
        ]

    if nombre == "Árbol de decisión":

        return [
            ("Extra-1", {"depth": 24, "leaf": 4}, "ros"),
            ("HMVT-V6d", {"depth": 12, "leaf": 2}, "ros_parcial"),
            ("HMVT-V6a", {"depth": 24, "leaf": 4}, "none"),
            ("HMVT-V3", {"depth": 18, "leaf": 2}, "balanced"),
            ("Extra-5", {"depth": 12, "leaf": 4}, "minority_boost")
        ]

    raise ValueError(nombre)










EMBEDDINGS = None

if USAR_EMBEDDINGS:

    try:
        from sentence_transformers import SentenceTransformer

    except ImportError:

        subprocess.check_call([
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "sentence-transformers"
        ])

        from sentence_transformers import SentenceTransformer

    print("\nCargando embeddings preentrenados...")

    encoder = SentenceTransformer(
        "sentence-transformers/"
        "paraphrase-multilingual-MiniLM-L12-v2"
    )

    encoder.eval()

    EMBEDDINGS = encoder.encode(
        list(X_TEXTOS),
        batch_size=64,
        show_progress_bar=True,
        normalize_embeddings=False,
        convert_to_numpy=True
    ).astype(np.float32)

    del encoder





cv_externa = StratifiedKFold(
    n_splits=FOLDS_EXTERNOS,
    shuffle=True,
    random_state=SEED
)

RESULTADOS = []
SELECCIONES = []
ABLACIONES = []
PREDICCIONES_FOLDS = []

REP_BASE = [
    "TF-IDF",
    "N-gramas",
    "BoW"
]

if USAR_EMBEDDINGS:
    REP_BASE.append("Embeddings")

REP_TODAS = (
    REP_BASE
    + list(VERSIONES.keys())
    + ["HMVT-Optimizado"]
)


for numero_fold, (idx_tr, idx_ev) in enumerate(
    cv_externa.split(X_TEXTOS, Y),
    start=1
):

    print(
        "\n" + "=" * 65,
        f"\nFOLD EXTERNO {numero_fold}/{FOLDS_EXTERNOS}",
        "\n" + "=" * 65,
        flush=True
    )

    textos_tr = X_TEXTOS[idx_tr]
    textos_ev = X_TEXTOS[idx_ev]

    ytr = Y[idx_tr]
    yev = Y[idx_ev]

    
    
    

    extractor = Extractor().fit(
        textos_tr
    )

    Btr = extractor.transform(
        textos_tr
    )

    Bev = extractor.transform(
        textos_ev
    )

    REPS_FOLD = {
        "TF-IDF": (
            Btr[0],
            Bev[0]
        ),
        "N-gramas": (
            extractor.ng.transform(textos_tr),
            extractor.ng.transform(textos_ev)
        ),
        "BoW": (
            extractor.bow.transform(textos_tr),
            extractor.bow.transform(textos_ev)
        )
    }

    if USAR_EMBEDDINGS:
        REPS_FOLD["Embeddings"] = (
            EMBEDDINGS[idx_tr],
            EMBEDDINGS[idx_ev]
        )

    for version, pesos in VERSIONES.items():

        REPS_FOLD[version] = (
            fusionar(Btr, pesos),
            fusionar(Bev, pesos)
        )

    
    
    

    cv_interna = StratifiedKFold(
        n_splits=FOLDS_INTERNOS,
        shuffle=True,
        random_state=SEED + numero_fold
    )

    BLOQUES_INTERNOS = []

    for i_tr, i_val in cv_interna.split(
        textos_tr,
        ytr
    ):

        ext_int = Extractor().fit(
            textos_tr[i_tr]
        )

        bloques_a = ext_int.transform(
            textos_tr[i_tr]
        )

        bloques_v = ext_int.transform(
            textos_tr[i_val]
        )

        BLOQUES_INTERNOS.append((
            bloques_a,
            bloques_v,
            ytr[i_tr],
            ytr[i_val]
        ))

    
    
    

    for rep, (Xtr, Xev) in REPS_FOLD.items():

        print(
            "Evaluando:",
            rep,
            flush=True
        )

        for nombre in MODELOS:

            try:

                
                
                if rep == "Embeddings" and nombre == "Naive Bayes":
                    modelo = GaussianNB()
                    modelo.fit(Xtr, ytr)
                    pred = modelo.predict(Xev)

                else:

                    config_base = {}

                    if nombre == "Naive Bayes":
                        config_base = {
                            "modelo": "MultinomialNB"
                        }

                    pred = entrenar_predecir(
                        nombre,
                        Xtr,
                        ytr,
                        Xev,
                        config=config_base,
                        balance="none",
                        semilla=SEED + numero_fold
                    )

                m = metricas(yev, pred)

                RESULTADOS.append({
                    "Fold": numero_fold,
                    "Representación": rep,
                    "Modelo de clasificación": nombre,
                    **m
                })

                PREDICCIONES_FOLDS.append({
                    "Fold": numero_fold,
                    "Representación": rep,
                    "Modelo": nombre,
                    "indices": idx_ev.copy(),
                    "prediccion": np.asarray(pred).copy()
                })

            except Exception as error:

                warnings.warn(
                    f"{rep} / {nombre}: {error}"
                )

                RESULTADOS.append({
                    "Fold": numero_fold,
                    "Representación": rep,
                    "Modelo de clasificación": nombre,
                    **{k: np.nan for k in metricas(
                        [0, 1, 2, 3],
                        [0, 1, 2, 3]
                    )}
                })

    
    
    
    
    
    
    

    for nombre in MODELOS:

        mejor = None

        candidatos = candidatos_para(nombre)

        for version, cfg, balance in candidatos:

            pesos = TODOS_PESOS[version]

            metricas_int = []

            for fold_int, (
                Ba, Bv, ya, yv
            ) in enumerate(
                BLOQUES_INTERNOS,
                start=1
            ):

                try:

                    Xa = fusionar(
                        Ba,
                        pesos
                    )

                    Xv = fusionar(
                        Bv,
                        pesos
                    )

                    pred = entrenar_predecir(
                        nombre,
                        Xa,
                        ya,
                        Xv,
                        config=cfg,
                        balance=balance,
                        semilla=SEED + numero_fold * 100 + fold_int
                    )

                    metricas_int.append(
                        metricas(yv, pred)
                    )

                except Exception as error:

                    warnings.warn(
                        f"Seleccion {nombre}/{version}: {error}"
                    )

                    metricas_int = []
                    break

            if len(metricas_int) != FOLDS_INTERNOS:
                continue

            f1_prom = np.mean([
                m["F1-Score"]
                for m in metricas_int
            ])

            rec_minor = np.mean([
                (
                    m["Recall Acoso"]
                    + m["Recall Violencia sexual"]
                ) / 2
                for m in metricas_int
            ])

            score = (
                f1_prom,
                rec_minor
            )

            if mejor is None or score > mejor["score"]:

                mejor = {
                    "score": score,
                    "version": version,
                    "pesos": pesos,
                    "config": cfg,
                    "balance": balance,
                    "F1 interno": f1_prom,
                    "Recall minoritario interno": rec_minor
                }

        if mejor is None:

            raise RuntimeError(
                f"No se pudo optimizar {nombre}"
            )

        
        
        

        Xtr_opt = fusionar(
            Btr,
            mejor["pesos"]
        )

        Xev_opt = fusionar(
            Bev,
            mejor["pesos"]
        )

        pred_opt = entrenar_predecir(
            nombre,
            Xtr_opt,
            ytr,
            Xev_opt,
            config=mejor["config"],
            balance=mejor["balance"],
            semilla=SEED + numero_fold
        )

        m_opt = metricas(
            yev,
            pred_opt
        )

        RESULTADOS.append({
            "Fold": numero_fold,
            "Representación": "HMVT-Optimizado",
            "Modelo de clasificación": nombre,
            **m_opt
        })

        PREDICCIONES_FOLDS.append({
            "Fold": numero_fold,
            "Representación": "HMVT-Optimizado",
            "Modelo": nombre,
            "indices": idx_ev.copy(),
            "prediccion": pred_opt.copy()
        })

        SELECCIONES.append({
            "Fold": numero_fold,
            "Clasificador": nombre,
            "Versión": mejor["version"],
            "Balanceo": mejor["balance"],
            "Configuración": str(mejor["config"]),
            "Pesos": str(mejor["pesos"]),
            "F1 validación interna": mejor["F1 interno"],
            "Recall minoritario interno":
                mejor["Recall minoritario interno"]
        })

        print(
            "HMVT seleccionado:",
            nombre,
            "|",
            mejor["version"],
            "| F1 interno:",
            round(mejor["F1 interno"], 4),
            flush=True
        )

        
        
        
        
        
        
        
        
        
        

        if MOSTRAR_ABLACION:

            pesos_completos = list(
                mejor["pesos"]
            )

            ablaciones = {
                "HMVT completo": tuple(pesos_completos),
                "Sin caracteres": (
                    pesos_completos[0],
                    0.0,
                    pesos_completos[2],
                    pesos_completos[3]
                ),
                "Sin LSA": (
                    pesos_completos[0],
                    pesos_completos[1],
                    0.0,
                    pesos_completos[3]
                ),
                "Sin discursivas": (
                    pesos_completos[0],
                    pesos_completos[1],
                    pesos_completos[2],
                    0.0
                ),
                "Solo TF-IDF": (
                    pesos_completos[0],
                    0.0,
                    0.0,
                    0.0
                )
            }

            for nombre_abl, pesos_abl in ablaciones.items():

                if nombre_abl == "HMVT completo":

                    m_abl = m_opt

                else:

                    Xa = fusionar(
                        Btr,
                        pesos_abl
                    )

                    Xv = fusionar(
                        Bev,
                        pesos_abl
                    )

                    pred_abl = entrenar_predecir(
                        nombre,
                        Xa,
                        ytr,
                        Xv,
                        config=mejor["config"],
                        balance=mejor["balance"],
                        semilla=SEED + numero_fold
                    )

                    m_abl = metricas(
                        yev,
                        pred_abl
                    )

                ABLACIONES.append({
                    "Fold": numero_fold,
                    "Clasificador": nombre,
                    "Ablación": nombre_abl,
                    "Versión base": mejor["version"],
                    **m_abl
                })





COLUMNAS_METRICAS = [
    "Precisión",
    "Recall",
    "F1-Score",
    "Accuracy",
    "Recall Acoso",
    "Recall Violencia sexual",
    "F1 Acoso",
    "F1 Violencia sexual"
]


def tabla_resumen(
    datos,
    columnas_grupo
):

    filas = []

    for clave, grupo in datos.groupby(
        columnas_grupo,
        observed=True,
        sort=False
    ):

        if not isinstance(clave, tuple):
            clave = (clave,)

        fila = dict(
            zip(
                columnas_grupo,
                clave
            )
        )

        for metrica in COLUMNAS_METRICAS:

            valores = (
                grupo[metrica]
                .dropna()
                .to_numpy(dtype=float)
            )

            if len(valores) == 0:

                fila[metrica] = "ERROR"
                fila[metrica + "_media"] = np.nan

            else:

                media = np.mean(valores)

                desviacion = (
                    np.std(valores, ddof=1)
                    if len(valores) > 1
                    else 0.0
                )

                fila[metrica] = (
                    f"{media:.4f} ± {desviacion:.4f}"
                )

                fila[metrica + "_media"] = media

        filas.append(fila)

    return pd.DataFrame(filas)





df_resultados = pd.DataFrame(RESULTADOS)

TABLA_MAESTRA = tabla_resumen(
    df_resultados,
    [
        "Representación",
        "Modelo de clasificación"
    ]
)

TABLA_MAESTRA["Representación"] = pd.Categorical(
    TABLA_MAESTRA["Representación"],
    categories=REP_TODAS,
    ordered=True
)

TABLA_MAESTRA["Modelo de clasificación"] = pd.Categorical(
    TABLA_MAESTRA["Modelo de clasificación"],
    categories=MODELOS,
    ordered=True
)

TABLA_MAESTRA = TABLA_MAESTRA.sort_values(
    [
        "Representación",
        "Modelo de clasificación"
    ]
).reset_index(drop=True)

print("\n" + "=" * 90)
print("TABLA MAESTRA - SOLO TRAIN - CV ANIDADA")
print("Precision, Recall y F1: PROMEDIO MACRO")
print("Accuracy: GLOBAL")
print("=" * 90)

display(
    TABLA_MAESTRA[
        [
            "Representación",
            "Modelo de clasificación",
            "Precisión",
            "Recall",
            "F1-Score",
            "Accuracy"
        ]
    ]
)





print("\n" + "=" * 90)
print("HMVT: DESEMPEÑO DE ACOSO Y VIOLENCIA SEXUAL")
print("=" * 90)

display(
    TABLA_MAESTRA[
        TABLA_MAESTRA[
            "Representación"
        ].astype(str).str.startswith("HMVT")
    ][
        [
            "Representación",
            "Modelo de clasificación",
            "F1-Score",
            "Recall Acoso",
            "Recall Violencia sexual",
            "F1 Acoso",
            "F1 Violencia sexual"
        ]
    ].reset_index(drop=True)
)





print("\n" + "=" * 90)
print("SELECCIONES HMVT SOLO EN VALIDACION INTERNA")
print("=" * 90)

TABLA_SELECCIONES = pd.DataFrame(
    SELECCIONES
)

display(
    TABLA_SELECCIONES.style.format({
        "F1 validación interna": "{:.4f}",
        "Recall minoritario interno": "{:.4f}"
    })
)





if MOSTRAR_ABLACION:

    df_ablaciones = pd.DataFrame(
        ABLACIONES
    )

    TABLA_ABLACION = tabla_resumen(
        df_ablaciones,
        [
            "Clasificador",
            "Ablación"
        ]
    )

    print("\n" + "=" * 90)
    print("ESTUDIO DE ABLACION HMVT")
    print("=" * 90)

    display(
        TABLA_ABLACION[
            [
                "Clasificador",
                "Ablación",
                "F1-Score",
                "Accuracy",
                "Recall Acoso",
                "Recall Violencia sexual"
            ]
        ]
    )

    
    comparaciones = []

    for nombre in MODELOS:

        subset = df_ablaciones[
            df_ablaciones["Clasificador"] == nombre
        ]

        completo = subset[
            subset["Ablación"] == "HMVT completo"
        ].set_index("Fold")

        for abl in [
            "Sin caracteres",
            "Sin LSA",
            "Sin discursivas",
            "Solo TF-IDF"
        ]:

            alternativo = subset[
                subset["Ablación"] == abl
            ].set_index("Fold")

            comunes = completo.index.intersection(
                alternativo.index
            )

            delta_f1 = (
                completo.loc[comunes, "F1-Score"]
                - alternativo.loc[comunes, "F1-Score"]
            )

            delta_acoso = (
                completo.loc[comunes, "Recall Acoso"]
                - alternativo.loc[comunes, "Recall Acoso"]
            )

            delta_sexual = (
                completo.loc[comunes, "Recall Violencia sexual"]
                - alternativo.loc[comunes, "Recall Violencia sexual"]
            )

            comparaciones.append({
                "Clasificador": nombre,
                "Componente retirado": abl,
                "Δ F1 macro": delta_f1.mean(),
                "Δ Recall Acoso": delta_acoso.mean(),
                "Δ Recall Violencia sexual": delta_sexual.mean()
            })

    print("\n" + "=" * 90)
    print("CONTRIBUCION DE LOS COMPONENTES HMVT")
    print("Delta positivo = retirar el componente empeora")
    print("Delta negativo = retirar el componente mejora")
    print("=" * 90)

    display(
        pd.DataFrame(
            comparaciones
        ).style.format({
            "Δ F1 macro": "{:+.4f}",
            "Δ Recall Acoso": "{:+.4f}",
            "Δ Recall Violencia sexual": "{:+.4f}"
        })
    )









print("\n" + "=" * 90)
print("COMPARACION PAREADA POR FOLD EXTERNO")
print("Delta F1 positivo = ventaja de HMVT")
print("=" * 90)

comparativa = []

for nombre in MODELOS:

    hmvt = df_resultados[
        (
            df_resultados["Representación"]
            == "HMVT-Optimizado"
        )
        &
        (
            df_resultados["Modelo de clasificación"]
            == nombre
        )
    ].set_index("Fold")

    for baseline in REP_BASE:

        base = df_resultados[
            (
                df_resultados["Representación"]
                == baseline
            )
            &
            (
                df_resultados["Modelo de clasificación"]
                == nombre
            )
        ].set_index("Fold")

        folds = hmvt.index.intersection(
            base.index
        )

        delta = (
            hmvt.loc[folds, "F1-Score"]
            - base.loc[folds, "F1-Score"]
        ).dropna()

        if delta.empty:
            continue

        comparativa.append({
            "Clasificador": nombre,
            "Comparación": f"HMVT vs {baseline}",
            "Δ F1 macro": delta.mean(),
            "Desv. Δ F1": (
                delta.std(ddof=1)
                if len(delta) > 1
                else 0.0
            ),
            "Folds HMVT superior": int(
                (delta > 0).sum()
            ),
            "Folds comparados": len(delta)
        })

display(
    pd.DataFrame(
        comparativa
    ).style.format({
        "Δ F1 macro": "{:+.4f}",
        "Desv. Δ F1": "{:.4f}"
    })
)









print("\n" + "=" * 90)
print("MEJOR RESULTADO HMVT OBSERVADO POR CLASIFICADOR")
print("Resumen descriptivo, no nueva optimizacion")
print("=" * 90)

mejores = []

for nombre in MODELOS:

    parte = TABLA_MAESTRA[
        (
            TABLA_MAESTRA["Modelo de clasificación"]
            == nombre
        )
        &
        (
            TABLA_MAESTRA[
                "Representación"
            ].astype(str).str.startswith("HMVT")
        )
    ]

    parte = parte.dropna(
        subset=["F1-Score_media"]
    )

    if parte.empty:
        continue

    mejor = parte.loc[
        parte["F1-Score_media"].idxmax()
    ]

    mejores.append({
        "Clasificador": nombre,
        "Representación": str(
            mejor["Representación"]
        ),
        "F1 macro": mejor["F1-Score"],
        "Accuracy": mejor["Accuracy"],
        "Recall Acoso": mejor["Recall Acoso"],
        "Recall Violencia sexual":
            mejor["Recall Violencia sexual"]
    })

display(
    pd.DataFrame(mejores)
)

print("\n" + "=" * 90)
print("EXPERIMENTO FINALIZADO")
print("Se utilizo solamente TRAIN.")
print("TEST no fue cargado ni consultado.")
print("Embeddings sin fine-tuning ni optimizacion.")
print("No se generaron matrices de confusion ni archivos.")
print("=" * 90)


### EXP-043 · Sin versión explícita

**Archivo de origen:** `Reportes_tesis.ipynb, celda 40`. **Referencias duplicadas:** 1. **Sintaxis de código:** Válida.

HMVT ESPANOL / EXIST2021: CUATRO CLASES, SOLO TRAIN
ONE GOOGLE COLAB CELL - SPEED BUDGET 50 MINUTES
3 outer folds x 2 inner folds; honest F1 macro from outer folds
Changes: stronger word + character features, Spanish lexical cues,
train-onl

**Ejecución:** celda independiente o variante experimental; revisar archivos de entrada y dependencias antes de ejecutarla. Los resultados originales no se insertan en esta celda.

In [ ]:
import os, sys, re, time, subprocess, warnings, unicodedata
from pathlib import Path
from collections import Counter
import numpy as np, pandas as pd
from scipy.sparse import csr_matrix, hstack, issparse
from IPython.display import display
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import MinMaxScaler, normalize
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import ComplementNB, MultinomialNB, GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
warnings.filterwarnings('ignore', category=FutureWarning)
SEED=42
OUTER_FOLDS=3       
INNER_FOLDS=2       
MAX_MINUTES=43      
USE_EMBEDDINGS=True 
USE_LSA=False       
N_JOBS=2
START=time.monotonic()
CLASES=['No violencia','Misoginia','Acoso','Violencia sexual']
MODELOS=['Regresión Logística','3-NN','SVM','Naive Bayes','Red neuronal','XGBoost','Random Forest','Árbol de decisión']

def remaining(): return MAX_MINUTES*60-(time.monotonic()-START)

def locate():
    roots=[Path('/content'),Path.cwd(),Path('/mnt/data')]
    patterns=['EXIST2021_train_reestructurado*.csv']
    def find():
        found=[]
        for root in roots:
            if root.exists():
                for pat in patterns: found.extend(p for p in root.glob(pat) if p.is_file())
        return sorted(set(found),key=lambda p:(len(p.name),p.name))
    found=find()
    if not found:
        try:
            from google.colab import files
            print('SUBE TRAIN ESPANOL: EXIST2021_train_reestructurado.csv')
            files.upload()
        except ImportError: pass
        found=find()
    if not found: raise FileNotFoundError('Falta TRAIN espanol reestructurado. Subelo a Colab y vuelve a ejecutar la misma celda.')
    return found[0]

path=locate(); df=pd.read_csv(path,encoding='utf-8-sig');df.columns=df.columns.str.strip()
if 'tipo_violencia' not in df and 'tipo de violencia' in df:df=df.rename(columns={'tipo de violencia':'tipo_violencia'})
if not {'text','tipo_violencia'}.issubset(df):raise ValueError('Se requieren text y tipo_violencia')
if 'language' in df:
    df=df[df.language.astype(str).str.strip().str.lower().isin(['es','spanish'])].copy()
if len(df)==0:raise ValueError('El archivo no contiene registros en espanol. Debes subir el TRAIN espanol.')
mapper={'0':0,'1':1,'2':2,'3':3,'no violencia':0,'misoginia':1,'acoso':2,'violencia sexual':3,'no-violencia':0,'violencia-sexual':3}
def normlabel(v):
    s=''.join(c for c in unicodedata.normalize('NFKD',str(v).strip().lower()) if not unicodedata.combining(c))
    return mapper.get(s,np.nan)
df=df.dropna(subset=['text','tipo_violencia']).copy()
df['text']=df.text.astype(str).str.replace(r'\s+',' ',regex=True).str.strip()
df=df[df.text.str.len()>0].copy();df['y']=df.tipo_violencia.map(normlabel)
if df.y.isna().any():raise ValueError('Etiquetas inesperadas: '+str(df.loc[df.y.isna(),'tipo_violencia'].unique()))
df['y']=df.y.astype(int)
df['_key']=df.text.str.casefold(); conflicts=df.groupby('_key').y.nunique(); bad=set(conflicts[conflicts>1].index)
if bad: print('Conflictos eliminados:',len(bad));df=df[~df._key.isin(bad)]
df=df.drop_duplicates('_key').reset_index(drop=True)
X=df.text.to_numpy();Y=df.y.to_numpy()
if set(Y)!={0,1,2,3}:raise ValueError('Se esperan cuatro clases 0,1,2,3')
print('TRAIN ESPANOL:',len(Y),'Clases:',dict(sorted(Counter(Y).items())))
print('Etiquetas espanolas originales conservadas: 0, 1, 2, 3.')



GROUPS={
 'insultos':['puta','putita','perra','zorra','idiota','estupida','estupido','pendeja','pendejo','cabrona','maldita'],
 'genero':['mujer','mujeres','hombre','hombres','chica','chicas','feminista','feminismo','machista'],
 'amenazas':['matar','muerte','golpear','amenaza','acosar','acosador','hostigar','violacion','forzar'],
 'sexual':['sexo','sexual','violar','violacion','desnuda','cuerpo','tetas','pene','coger','culo'],
 'control':['obedecer','obediente','sumisa','inferior','superior','callate','cocina','mandar'],
 'negacion':['no','nunca','jamas','ni','nadie','tampoco']
}
PATS={k:[re.compile(r'(?<!\w)'+re.escape(w)+r'(?!\w)') for w in values] for k,values in GROUPS.items()}
def lex_feats(texts):
    z=[]
    for t in texts:
        s=''.join(c for c in unicodedata.normalize('NFKD',str(t).lower()) if not unicodedata.combining(c)); toks=re.findall(r'\b\w+\b',s)
        row=[np.log1p(sum(len(rx.findall(s)) for rx in pp)) for pp in PATS.values()]
        row.extend([np.log1p(len(toks)),len(set(toks))/max(1,len(toks)),
                    np.log1p(s.count('!')),np.log1p(s.count('?')),
                    int('@' in s),int('#' in s),int('http' in s),
                    np.log1p(sum(c.isupper() for c in str(t)))])
        z.append(row)
    return np.asarray(z,dtype=np.float32)

class Features:
    def fit(self, texts):
        self.word=TfidfVectorizer(ngram_range=(1,2),min_df=2,max_features=22000,
                                  sublinear_tf=True,strip_accents='unicode',dtype=np.float32)
        self.char=TfidfVectorizer(analyzer='char_wb',ngram_range=(2,5),min_df=2,
                                  max_features=35000,sublinear_tf=True,dtype=np.float32)
        self.ng=CountVectorizer(ngram_range=(1,2),min_df=2,max_features=12000)
        self.bow=CountVectorizer(ngram_range=(1,1),min_df=2,max_features=12000)
        w=self.word.fit_transform(texts);self.char.fit(texts);self.ng.fit(texts);self.bow.fit(texts)
        self.scaler=MinMaxScaler().fit(lex_feats(texts))
        if USE_LSA:
            k=min(35,w.shape[0]-1,w.shape[1]-1)
            self.svd=TruncatedSVD(n_components=k,random_state=SEED).fit(w)
            self.lscaler=MinMaxScaler().fit(self.svd.transform(w))
        return self
    def blocks(self,texts):
        w=self.word.transform(texts);c=self.char.transform(texts)
        l=csr_matrix(np.clip(self.scaler.transform(lex_feats(texts)),0,1))
        if USE_LSA:
            z=csr_matrix(np.clip(self.lscaler.transform(self.svd.transform(w)),0,1))
        else:z=csr_matrix((len(texts),0),dtype=np.float32)
        return (w,c,l,z)
    def baselines(self,texts):return {'TF-IDF':self.word.transform(texts),
                   'N-gramas':self.ng.transform(texts),'BoW':self.bow.transform(texts)}



VARS={'HMVT-Texto+Caracteres':(1.0,1.0,0.0,0.0),
      'HMVT-Completo':(1.0,1.0,0.6,0.0),
      'HMVT-DiscursoFuerte':(1.0,0.8,1.5,0.0),
      'HMVT-SinCaracteres':(1.0,0.0,1.0,0.0)}
if USE_LSA:VARS['HMVT-ConLSA']=(1.0,1.0,.6,.2)

def mix(blocks,w):
    return hstack([b.multiply(float(v)) for b,v in zip(blocks,w) if v>0 and b.shape[1]>0],format='csr',dtype=np.float32)

def cfgs(name):
    
    if name in ['Regresión Logística','SVM']:return [{'C':.3,'bal':'none'},{'C':1.5,'bal':'balanced'}]
    if name=='3-NN':return [{'k':3,'bal':'none'},{'k':5,'bal':'partial'}]
    if name=='Naive Bayes':return [{'alpha':.1,'bal':'none'},{'alpha':.5,'bal':'partial'}]
    if name=='Red neuronal':return [{'hidden':64,'bal':'none'},{'hidden':96,'bal':'partial'}]
    if name=='XGBoost':return [{'depth':3,'bal':'sqrt'},{'depth':4,'bal':'partial'}]
    if name=='Random Forest':return [{'leaf':1,'bal':'none'},{'leaf':2,'bal':'partial'}]
    return [{'depth':12,'bal':'none'},{'depth':20,'bal':'partial'}]

def balance_resample(x,y,mode,seed):
    if mode!='partial':return x,y
    rng=np.random.default_rng(seed); inds=[];mx=np.bincount(y,minlength=4).max()
    for label in range(4):
        ii=np.flatnonzero(y==label);need=max(len(ii),round(mx*.60))
        inds.extend(rng.choice(ii,size=need,replace=(need>len(ii))).tolist())
    inds=np.asarray(inds);rng.shuffle(inds)
    return x[inds],y[inds]

def make_model(name,cfg,emb=False):
    bal='balanced' if cfg.get('bal')=='balanced' else None
    if name=='Regresión Logística':return LogisticRegression(C=cfg.get('C',1),class_weight=bal,max_iter=1100,solver='liblinear' if False else 'lbfgs',random_state=SEED)
    if name=='SVM':return LinearSVC(C=cfg.get('C',1),class_weight=bal,max_iter=2500,random_state=SEED)
    if name=='3-NN':return KNeighborsClassifier(n_neighbors=cfg.get('k',3),weights='distance',metric='cosine',algorithm='brute')
    if name=='Naive Bayes':return GaussianNB() if emb else ComplementNB(alpha=cfg.get('alpha',.5))
    if name=='Red neuronal':return MLPClassifier(hidden_layer_sizes=(cfg.get('hidden',64),),alpha=.005,early_stopping=True,max_iter=90,n_iter_no_change=8,random_state=SEED)
    if name=='XGBoost':
        from xgboost import XGBClassifier
        return XGBClassifier(n_estimators=75,max_depth=cfg.get('depth',3),learning_rate=.10,
               subsample=.9,colsample_bytree=.85,tree_method='hist',objective='multi:softprob',
               num_class=4,eval_metric='mlogloss',random_state=SEED,n_jobs=N_JOBS)
    if name=='Random Forest':return RandomForestClassifier(n_estimators=90,min_samples_leaf=cfg.get('leaf',1),class_weight=bal,n_jobs=N_JOBS,random_state=SEED)
    return DecisionTreeClassifier(max_depth=cfg.get('depth',14),class_weight=bal,min_samples_leaf=3,random_state=SEED)

def fit_pred(name,config,xtr,ytr,xev,seed,emb=False):
    mode=config.get('bal','none');xtr,ytr=balance_resample(xtr,ytr,mode,seed)
    model=make_model(name,config,emb)
    if mode=='sqrt' and name!='3-NN':
        counts=np.bincount(ytr,minlength=4);weights=np.sqrt(len(ytr)/(4*np.maximum(counts,1)))
        if name in ['Regresión Logística','SVM','Naive Bayes','XGBoost','Random Forest','Árbol de decisión']:
            model.fit(xtr,ytr,sample_weight=weights[ytr]);return model.predict(xev)
    model.fit(xtr,ytr);return model.predict(xev)

def measure(y,p):
    prec,rec,f1,_=precision_recall_fscore_support(y,p,labels=[0,1,2,3],average='macro',zero_division=0)
    r=precision_recall_fscore_support(y,p,labels=[0,1,2,3],average=None,zero_division=0)[1]
    return {'Precisión':prec,'Recall':rec,'F1-Score':f1,'Accuracy':accuracy_score(y,p),
            'Recall Acoso':r[2],'Recall Violencia sexual':r[3]}


EMB=None
if USE_EMBEDDINGS:
    try:from sentence_transformers import SentenceTransformer
    except ImportError:
        subprocess.check_call([sys.executable,'-m','pip','install','-q','sentence-transformers'])
        from sentence_transformers import SentenceTransformer
    enc=SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')
    EMB=enc.encode(X.tolist(),batch_size=96,show_progress_bar=True,convert_to_numpy=True,
                   normalize_embeddings=False).astype(np.float32)
    del enc
try:from xgboost import XGBClassifier
except ImportError:subprocess.check_call([sys.executable,'-m','pip','install','-q','xgboost'])

outer=StratifiedKFold(n_splits=OUTER_FOLDS,shuffle=True,random_state=SEED)
RESULTS=[];SELECTED=[];ABL=[];DONE=0
for fold,(itr,iev) in enumerate(outer.split(X,Y),1):
    print(f'\n===== FOLD {fold}/{OUTER_FOLDS} | {round((time.monotonic()-START)/60,1)} min =====',flush=True)
    ytr,yev=Y[itr],Y[iev];txtr,txev=X[itr],X[iev]
    ext=Features().fit(txtr);bt,be=ext.blocks(txtr),ext.blocks(txev)
    bases_tr,bases_ev=ext.baselines(txtr),ext.baselines(txev)
    reps={r:(bases_tr[r],bases_ev[r]) for r in bases_tr}
    reps.update({r:(mix(bt,w),mix(be,w)) for r,w in VARS.items()})
    if EMB is not None:reps['Embeddings']=(EMB[itr],EMB[iev])

    inner=StratifiedKFold(n_splits=INNER_FOLDS,shuffle=True,random_state=SEED+fold)
    inner_data=[]
    for ia,iv in inner.split(txtr,ytr):
        z=Features().fit(txtr[ia]);ba,bv=z.blocks(txtr[ia]),z.blocks(txtr[iv])
        base_a=z.word.transform(txtr[ia]);base_v=z.word.transform(txtr[iv])
        inner_data.append((ba,bv,base_a,base_v,ytr[ia],ytr[iv]))

    for name in MODELOS:
        
        
        winners={}
        for family in ('TF-IDF-Optimizado','HMVT-Optimizado'):
            candidates=[('TF-IDF',None)] if family=='TF-IDF-Optimizado' else list(VARS.items())
            best=None
            for rep,w in candidates:
                for cfg in cfgs(name):
                    if remaining()<150:break
                    scores=[];minor=[]
                    try:
                        for j,(ba,bv,fa,fv,ya,yv) in enumerate(inner_data):
                            aa,bb=(fa,fv) if w is None else (mix(ba,w),mix(bv,w))
                            pred=fit_pred(name,cfg,aa,ya,bb,SEED+fold*20+j)
                            met=measure(yv,pred)
                            scores.append(met['F1-Score']);minor.append((met['Recall Acoso']+met['Recall Violencia sexual'])/2)
                    except Exception as exc:
                        warnings.warn(f'{name}/{family}/{rep}: {exc}')
                        continue
                    if len(scores)==INNER_FOLDS:
                        value=(float(np.mean(scores)),float(np.mean(minor)))
                        if best is None or value>best[0]:best=(value,rep,w,cfg)
                if remaining()<150:break
            winners[family]=best
        for family,best in winners.items():
            if best is None:continue
            score,rep,w,cfg=best
            xa,xb=reps['TF-IDF'] if w is None else (mix(bt,w),mix(be,w))
            try:
                pred=fit_pred(name,cfg,xa,ytr,xb,SEED+fold*200)
                RESULTS.append({'Fold':fold,'Representación':family,'Modelo de clasificación':name,**measure(yev,pred)})
                SELECTED.append({'Fold':fold,'Representación':family,'Clasificador':name,'Variante':rep,'Config':str(cfg),'F1 interno':score[0]})
                if family=='HMVT-Optimizado':
                    
                    for abl,ww in [('Sin caracteres',(w[0],0,w[2],w[3])),
                                   ('Sin discursivas',(w[0],w[1],0,w[3])),
                                   ('Solo TF-IDF',(1,0,0,0))]:
                        pa=fit_pred(name,cfg,mix(bt,ww),ytr,mix(be,ww),SEED+fold*200)
                        ABL.append({'Fold':fold,'Clasificador':name,'Ablación':abl,**measure(yev,pa)})
            except Exception as exc:warnings.warn(f'Ajuste final {family} {name}: {exc}')

        
        for rep,(aa,bb) in reps.items():
            if remaining()<110:break
            try:
                cfg=cfgs(name)[0].copy();cfg['bal']='none'
                pred=fit_pred(name,cfg,aa,ytr,bb,SEED+fold,emb=(rep=='Embeddings'))
                RESULTS.append({'Fold':fold,'Representación':rep,'Modelo de clasificación':name,**measure(yev,pred)})
            except Exception as exc:warnings.warn(f'Baseline {rep}/{name}: {exc}')
        print(f'  {name}: completado · {round((time.monotonic()-START)/60,1)} min',flush=True)
        if remaining()<110:
            print('PRESUPUESTO DE TIEMPO ALCANZADO; resultados parciales.',flush=True)
            break
    DONE=fold
    if remaining()<110:break

if not RESULTS:raise RuntimeError('No se completaron evaluaciones. Revisa RAM y el archivo de entrada.')
raw=pd.DataFrame(RESULTS)
metrics=['Precisión','Recall','F1-Score','Accuracy','Recall Acoso','Recall Violencia sexual']
def summarize(data,group):
    rows=[]
    for key,g in data.groupby(group,sort=False):
        if not isinstance(key,tuple):key=(key,)
        d=dict(zip(group,key));d['Folds']=g.Fold.nunique()
        for m in metrics:
            s=g[m].dropna();d[m+'_media']=s.mean();d[m]=f'{s.mean():.4f} ± {s.std(ddof=1):.4f}' if len(s)>1 else (f'{s.iloc[0]:.4f} (1 fold)' if len(s) else 'ERROR')
        rows.append(d)
    return pd.DataFrame(rows)
TABLA=summarize(raw,['Representación','Modelo de clasificación'])
order=['TF-IDF','N-gramas','BoW','Embeddings','HMVT-Texto+Caracteres','HMVT-Completo','HMVT-DiscursoFuerte','HMVT-SinCaracteres','HMVT-ConLSA','TF-IDF-Optimizado','HMVT-Optimizado']
TABLA['orden']=TABLA['Representación'].map({v:i for i,v in enumerate(order)})
TABLA=TABLA.sort_values(['orden','Modelo de clasificación']).drop(columns='orden').reset_index(drop=True)
print('\nTABLA COMPLETA (CV EXTERNA, TRAIN) · PRECISION/RECALL/F1 MACRO')
display(TABLA[['Representación','Modelo de clasificación','Folds','Precisión','Recall','F1-Score','Accuracy']])
print('\nHMVT · RECALL DE ACOSO Y VIOLENCIA SEXUAL')
display(TABLA[TABLA['Representación'].str.startswith('HMVT')][['Representación','Modelo de clasificación','F1-Score','Recall Acoso','Recall Violencia sexual']])
print('\nCONFIGURACIONES SELECCIONADAS SOLO EN VALIDACION INTERNA')
display(pd.DataFrame(SELECTED))
if ABL:
    print('\nABLACION PAREADA: MISMO MODELO, CONFIGURACION Y FOLD')
    abl=pd.DataFrame(ABL);display(summarize(abl,['Clasificador','Ablación'])[['Clasificador','Ablación','F1-Score','Recall Acoso','Recall Violencia sexual']])
print('\nCOMPARACION HMVT-OPTIMIZADO VS TF-IDF-OPTIMIZADO (mismos folds, diferente numero de representaciones candidatas)')
comp=[]
for name in MODELOS:
    a=raw[(raw['Representación']=='HMVT-Optimizado')&(raw['Modelo de clasificación']==name)].set_index('Fold')
    b=raw[(raw['Representación']=='TF-IDF-Optimizado')&(raw['Modelo de clasificación']==name)].set_index('Fold')
    ix=a.index.intersection(b.index)
    if len(ix):
        diff=a.loc[ix,'F1-Score']-b.loc[ix,'F1-Score']
        comp.append({'Modelo':name,'Δ F1-macro':round(diff.mean(),4),'Folds HMVT mejor':int((diff>0).sum()),'Comparados':len(ix)})
display(pd.DataFrame(comp))
print(f'\nFIN. Folds externos completados: {DONE}/{OUTER_FOLDS}. Tiempo: {(time.monotonic()-START)/60:.1f} minutos.')
print('SOLO ESPANOL. No se consulto TEST. Las mejoras de F1 se miden, no se simulan.')


### EXP-044 · Sin versión explícita

**Archivo de origen:** `Reportes_tesis.ipynb, celda 42`. **Referencias duplicadas:** 1. **Sintaxis de código:** Válida.

EXIST2021 ESPAÑOL · HMVT · UNA SOLA CELDA GOOGLE COLAB
TRAIN obligatorio; NO se utiliza TEST
3 folds externos × 2 internos; F1 MACRO, todos los modelos en tabla.
Aceleracion: seleccion de variables para arboles, matrices cacheadas,
XGBoost 

**Ejecución:** celda independiente o variante experimental; revisar archivos de entrada y dependencias antes de ejecutarla. Los resultados originales no se insertan en esta celda.

In [ ]:
import os, sys, re, time, warnings, subprocess, unicodedata
from pathlib import Path
from collections import Counter
import numpy as np, pandas as pd
from scipy.sparse import hstack, csr_matrix, issparse
from IPython.display import display
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import ComplementNB, MultinomialNB, GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
warnings.filterwarnings('ignore',category=FutureWarning)
SEED=42; FOLDS=3; INTERNOS=2; PRESUPUESTO_S=52*60
INCLUIR_EMBEDDINGS=True   

CLASES=['No violencia','Misoginia','Acoso','Violencia sexual']
MODELOS=['Regresión Logística','3-NN','SVM','Naive Bayes','Red neuronal','XGBoost','Random Forest','Árbol de decisión']
METRICAS=['Precisión','Recall','F1-Score','Accuracy','Recall Acoso','Recall Violencia sexual','F1 Acoso','F1 Violencia sexual']
INICIO=time.monotonic()
def tiempo():return (time.monotonic()-INICIO)/60

def solicitar_train():
    
    
    try:
        from google.colab import files
        print('SUBE UNICAMENTE TRAIN ESPAÑOL (EXIST2021_train_reestructurado.csv)')
        subida=files.upload()
        rutas=[Path('/content')/n for n in subida]
    except ImportError:
        rutas=[p for root in [Path('/mnt/data'),Path.cwd()] if root.exists()
               for p in root.glob('EXIST2021_train_reestructurado*.csv')]
    candidatos=[p for p in rutas if p.exists() and p.suffix.lower() in ['.csv','.tsv']
                and 'test' not in p.name.lower()]
    for p in candidatos:
        try:
            d=pd.read_csv(p,sep='\t' if p.suffix.lower()=='.tsv' else ',',encoding='utf-8-sig')
            d.columns=d.columns.str.strip()
            if 'tipo de violencia' in d.columns and 'tipo_violencia' not in d.columns:
                d=d.rename(columns={'tipo de violencia':'tipo_violencia'})
            if {'text','tipo_violencia'}.issubset(d.columns):return d,p
        except Exception as ex: print('Archivo descartado:',p.name,str(ex)[:100])
    raise ValueError('Sube el CSV TRAIN español con columnas text y tipo_violencia (no TEST).')

def limpiar(d):
    d=d.copy()
    if 'language' in d.columns:
        d=d[d.language.astype(str).str.lower().str.strip().isin(['es','spanish'])].copy()
    d=d.dropna(subset=['text','tipo_violencia'])
    d['text']=d.text.astype(str).str.replace(r'\s+',' ',regex=True).str.strip()
    d=d[d.text.str.len()>0].copy()
    def simple(v):return ''.join(z for z in unicodedata.normalize('NFKD',str(v).strip().lower()) if not unicodedata.combining(z))
    dic={'0':0,'1':1,'2':2,'3':3,'no violencia':0,'misoginia':1,'acoso':2,'violencia sexual':3}
    d['y']=d.tipo_violencia.map(lambda v:dic.get(simple(v),np.nan))
    if d.y.isna().any():raise ValueError('Etiquetas desconocidas: '+str(d[d.y.isna()].tipo_violencia.unique()))
    d['y']=d.y.astype(int)
    d['_clave']=d.text.map(simple)
    conflictos=d.groupby('_clave').y.nunique();c=set(conflictos[conflictos>1].index)
    if c:print('Conflictos de etiquetas excluidos:',len(c))
    d=d[~d._clave.isin(c)].drop_duplicates('_clave').drop(columns='_clave').reset_index(drop=True)
    if set(d.y.unique())!={0,1,2,3}:raise ValueError('Faltan clases en TRAIN')
    if d.y.value_counts().min()<FOLDS*INTERNOS:raise ValueError('Muy pocos ejemplos para CV')
    return d

raw,ruta=solicitar_train(); df=limpiar(raw)
print('TRAIN ESPAÑOL:',ruta.name,'registros:',len(df),'clases:',dict(sorted(Counter(df.y).items())))
T=df.text.to_numpy(); Y=df.y.to_numpy()



PATRONES={
 'insultos':['puta','puto','perra','zorra','idiota','estupida','estupido','imbecil','pendeja','pendejo'],
 'genero':['mujer','mujeres','hombre','hombres','feminista','feminismo','machista'],
 'acoso':['acosar','acoso','hostigar','amenaza','amenazar','perseguir','molestar'],
 'sexual':['sexual','sexo','violacion','violar','desnuda','follar','coger','pene','tetas'],
 'negacion':['no','nunca','jamas','nadie'],
 'imperativos':['callate','vete','muere','deberias','mereces']}
RXS={k:[re.compile(r'(?<!\w)'+re.escape(s)+r'(?!\w)') for s in v] for k,v in PATRONES.items()}
def discurso(txt):
    a=[]
    for s in txt:
        s=''.join(ch for ch in unicodedata.normalize('NFKD',str(s).lower()) if not unicodedata.combining(ch))
        words=re.findall(r'\b\w+\b',s)
        vals=[np.log1p(sum(len(rx.findall(s)) for rx in v)) for v in RXS.values()]
        vals += [np.log1p(len(words)),len(set(words))/max(1,len(words)),np.log1p(s.count('!')),
                 np.log1p(s.count('?')),int('@' in s),int('#' in s),int('http' in s)]
        a.append(vals)
    return np.asarray(a,dtype=np.float32)

class Texto:
    def fit(self,textos):
        self.w=TfidfVectorizer(ngram_range=(1,2),max_features=16000,min_df=2,
                               sublinear_tf=True,strip_accents='unicode',token_pattern=r'(?u)\b\w+\b')
        self.ch=TfidfVectorizer(analyzer='char_wb',ngram_range=(3,5),max_features=18000,min_df=2,sublinear_tf=True)
        self.ng=CountVectorizer(ngram_range=(1,2),max_features=10000,min_df=2)
        self.bow=CountVectorizer(max_features=10000,min_df=2)
        self.w.fit(textos);self.ch.fit(textos);self.ng.fit(textos);self.bow.fit(textos)
        self.ds=MinMaxScaler().fit(discurso(textos))
        return self
    def bloques(self,textos):
        a=self.w.transform(textos).astype(np.float32)
        b=self.ch.transform(textos).astype(np.float32)
        c=csr_matrix(np.clip(self.ds.transform(discurso(textos)),0,1).astype(np.float32))
        return a,b,c
    def base(self,nombre,textos,bl):
        if nombre=='TF-IDF':return bl[0]
        if nombre=='N-gramas':return self.ng.transform(textos)
        if nombre=='BoW':return self.bow.transform(textos)
        raise ValueError(nombre)


PESOS={'HMVT-Palabras+Caracteres':(1.0,1.0,0.0),
       'HMVT-Completo':(1.0,1.0,0.40),
       'HMVT-DiscursoFuerte':(1.0,0.8,0.85),
       'HMVT-SinCaracteres':(1.0,0.0,0.40)}
def fusion(bl,w):
    return hstack([b.multiply(float(p)) for b,p in zip(bl,w) if p>0],format='csr',dtype=np.float32)

def metricas(y,p):
    a,b,c,_=precision_recall_fscore_support(y,p,labels=[0,1,2,3],average='macro',zero_division=0)
    _,rec,f1,_=precision_recall_fscore_support(y,p,labels=[0,1,2,3],average=None,zero_division=0)
    return dict(zip(METRICAS,[a,b,c,accuracy_score(y,p),rec[2],rec[3],f1[2],f1[3]]))

def muestras(X,y,modo,seed):
    if modo not in ['parcial','ros']:return X,y
    rng=np.random.default_rng(seed); maxi=np.bincount(y,minlength=4).max();ids=[]
    for cl in range(4):
        orig=np.flatnonzero(y==cl);objetivo=max(len(orig),int(maxi*(.60 if modo=='parcial' else 1.0)))
        ids.extend(rng.choice(orig,size=objetivo,replace=objetivo>len(orig)).tolist())
    ids=np.asarray(ids);rng.shuffle(ids)
    return X[ids],y[ids]

def modelo(nombre,cfg,seed):
    bal='balanced' if cfg.get('bal')=='balanced' else None
    if nombre=='Regresión Logística':return LogisticRegression(C=cfg.get('C',1),class_weight=bal,max_iter=700,random_state=seed)
    if nombre=='3-NN':return KNeighborsClassifier(n_neighbors=cfg.get('k',3),weights='distance',metric='cosine',algorithm='brute')
    if nombre=='SVM':return LinearSVC(C=cfg.get('C',1),class_weight=bal,max_iter=2000,random_state=seed)
    if nombre=='Naive Bayes':return ComplementNB(alpha=cfg.get('alpha',0.5)) if cfg.get('tipo')=='complement' else MultinomialNB(alpha=cfg.get('alpha',1))
    if nombre=='Red neuronal':return MLPClassifier(hidden_layer_sizes=(cfg.get('h',64),),alpha=.005,max_iter=85,
                                                     early_stopping=True,n_iter_no_change=8,random_state=seed)
    if nombre=='XGBoost':
        from xgboost import XGBClassifier
        return XGBClassifier(objective='multi:softprob',num_class=4,n_estimators=cfg.get('n',65),
                             max_depth=cfg.get('depth',3),learning_rate=.10,subsample=.85,
                             colsample_bytree=.85,tree_method='hist',n_jobs=2,random_state=seed,eval_metric='mlogloss')
    if nombre=='Random Forest':return RandomForestClassifier(n_estimators=cfg.get('n',75),min_samples_leaf=cfg.get('leaf',2),
                                                               max_features='sqrt',class_weight=bal,n_jobs=2,random_state=seed)
    if nombre=='Árbol de decisión':return DecisionTreeClassifier(max_depth=cfg.get('depth',16),min_samples_leaf=3,
                                                                  class_weight=bal,random_state=seed)
    raise ValueError(nombre)

ARBOL={'XGBoost','Random Forest','Árbol de decisión'}
def ajustar(nombre,X,y,E,cfg,seed):
    
    
    modo=cfg.get('balance','none'); X,y=muestras(X,np.asarray(y),modo,seed)
    if nombre in ARBOL and (issparse(X) or np.min(X)>=0):
        k=min(1100,X.shape[1]); sel=SelectKBest(chi2,k=k)
        X=sel.fit_transform(X,y); E=sel.transform(E)
    if nombre=='Naive Bayes' and not issparse(X):
        est=GaussianNB()
    else:
        est=modelo(nombre,cfg,seed)
    fitkw={}
    if cfg.get('bal')=='sqrt' and nombre in ['Regresión Logística','SVM','Naive Bayes','XGBoost','Random Forest','Árbol de decisión']:
        cnt=np.bincount(y,minlength=4); weights=np.sqrt(len(y)/(4*np.maximum(cnt,1)))
        fitkw['sample_weight']=weights[y]
    est.fit(X,y,**fitkw)
    return est.predict(E)

CAND={
 'Regresión Logística':[{'C':.3,'bal':'balanced'},{'C':2.0,'bal':'balanced'},{'C':1.0,'bal':'none'}],
 '3-NN':[{'k':3,'balance':'none'},{'k':5,'balance':'parcial'}],
 'SVM':[{'C':.25,'bal':'balanced'},{'C':1.5,'bal':'balanced'},{'C':1.0,'bal':'none'}],
 'Naive Bayes':[{'tipo':'complement','alpha':.1},{'tipo':'complement','alpha':.5,'balance':'parcial'}],
 'Red neuronal':[{'h':64,'balance':'parcial'},{'h':96,'balance':'none'}],
 'XGBoost':[{'depth':3,'n':45,'bal':'sqrt'},{'depth':4,'n':45,'balance':'parcial'}],
 'Random Forest':[{'leaf':2,'n':55,'balance':'parcial'},{'leaf':3,'n':55,'bal':'balanced'}],
 'Árbol de decisión':[{'depth':14,'balance':'parcial'},{'depth':20,'bal':'balanced'}]
}
BASE_CFG={'Regresión Logística':{'C':1},'3-NN':{'k':3},'SVM':{'C':1},
          'Naive Bayes':{'tipo':'multinomial','alpha':1},'Red neuronal':{'h':64},
          'XGBoost':{'depth':3,'n':45},'Random Forest':{'leaf':2,'n':55},'Árbol de decisión':{'depth':16}}


try: import xgboost
except ImportError:
    subprocess.check_call([sys.executable,'-m','pip','install','-q','xgboost'])

EMB=None
if INCLUIR_EMBEDDINGS:
    try:from sentence_transformers import SentenceTransformer
    except ImportError:
        subprocess.check_call([sys.executable,'-m','pip','install','-q','sentence-transformers'])
        from sentence_transformers import SentenceTransformer
    print('Codificando embeddings congelados (sin tuning)...',flush=True)
    enc=SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')
    enc.eval()
    EMB=enc.encode(T.tolist(),batch_size=64,show_progress_bar=True,convert_to_numpy=True,normalize_embeddings=False).astype(np.float32)
    del enc






BASES=['TF-IDF','N-gramas','BoW']+(['Embeddings'] if EMB is not None else [])
VARIANTES=list(PESOS)
REP=BASES+VARIANTES+[v+'-Optimizado' for v in VARIANTES]+['HMVT-Optimizado']
CV=StratifiedKFold(n_splits=FOLDS,shuffle=True,random_state=SEED)
REG=[]; SELECCION=[]; ERRORES=[]; FOLDS_OK=0

def resumir(reg):
    data=pd.DataFrame(reg)
    rows=[]
    for rep in REP:
        for mod in MODELOS:
            a=data[(data['Representación']==rep)&(data['Modelo']==mod)] if len(data) else pd.DataFrame()
            r={'Representación':rep,'Modelo de clasificación':mod,'Folds':len(a)}
            for col in METRICAS:
                v=a[col].dropna().to_numpy(dtype=float) if len(a) else np.array([])
                r[col]=(f'{v.mean():.4f} ± {v.std(ddof=1):.4f}' if len(v)>1
                        else f'{v[0]:.4f} (1 fold)' if len(v)==1 else 'PENDIENTE')
                r[col+'_media']=float(v.mean()) if len(v) else np.nan
            rows.append(r)
    return pd.DataFrame(rows)

def registrar(fold,rep,mod,yreal,pred,filas):
    filas.append(dict(Fold=fold,Representación=rep,Modelo=mod,**metricas(yreal,pred)))

def fallo(fold,rep,mod,err,filas):
    ERRORES.append((fold,rep,mod,str(err)))
    filas.append(dict(Fold=fold,Representación=rep,Modelo=mod,**{m:np.nan for m in METRICAS}))




for fold,(i_tr,i_ev) in enumerate(CV.split(T,Y),1):
    if tiempo() >= PRESUPUESTO_S/60:
        print('Presupuesto alcanzado antes del fold',fold); break
    print(f'\n========== FOLD {fold}/{FOLDS} · {tiempo():.1f} min ==========',flush=True)
    txt_tr,txt_ev=T[i_tr],T[i_ev];ytr,yev=Y[i_tr],Y[i_ev]
    ex=Texto().fit(txt_tr);btr=ex.bloques(txt_tr);bev=ex.bloques(txt_ev)
    datos_ext={v:(fusion(btr,PESOS[v]),fusion(bev,PESOS[v])) for v in VARIANTES}
    
    inner=[]
    for j,(a,v) in enumerate(StratifiedKFold(n_splits=INTERNOS,shuffle=True,random_state=SEED+fold).split(txt_tr,ytr)):
        ei=Texto().fit(txt_tr[a]);ba=ei.bloques(txt_tr[a]);bv=ei.bloques(txt_tr[v])
        matrices={name:(fusion(ba,PESOS[name]),fusion(bv,PESOS[name])) for name in VARIANTES}
        inner.append((matrices,ytr[a],ytr[v]))
    fold_rows=[];fold_sel=[]
    for mod in MODELOS:
        
        for rep in BASES:
            try:
                if rep=='Embeddings':
                    A,E=EMB[i_tr],EMB[i_ev]
                else:
                    A,E=ex.base(rep,txt_tr,btr),ex.base(rep,txt_ev,bev)
                if rep=='Embeddings' and mod=='Naive Bayes':
                    est=GaussianNB();est.fit(A,ytr);p=est.predict(E)
                else:
                    p=ajustar(mod,A,ytr,E,BASE_CFG[mod],SEED+fold)
                registrar(fold,rep,mod,yev,p,fold_rows)
            except Exception as err:
                fallo(fold,rep,mod,err,fold_rows)
        
        for var in VARIANTES:
            try:
                A,E=datos_ext[var]
                p=ajustar(mod,A,ytr,E,BASE_CFG[mod],SEED+fold)
                registrar(fold,var,mod,yev,p,fold_rows)
            except Exception as err:
                fallo(fold,var,mod,err,fold_rows)
        
        
        elegidos=[]
        for var in VARIANTES:
            candidatos=[]
            for cfg in CAND[mod]:
                punt=[]
                for j,(matrices,ya,yv) in enumerate(inner):
                    try:
                        A,E=matrices[var]
                        pred=ajustar(mod,A,ya,E,cfg,SEED+fold*100+j)
                        m=metricas(yv,pred)
                        punt.append((m['F1-Score'],(m['Recall Acoso']+m['Recall Violencia sexual'])/2))
                    except Exception as err:
                        ERRORES.append((fold,'CV interna: '+var,mod,str(err)))
                        punt=[]; break
                if len(punt)==INTERNOS:
                    candidatos.append((np.mean(punt,axis=0),cfg))
            rep_opt=var+'-Optimizado'
            if not candidatos:
                fallo(fold,rep_opt,mod,'Sin candidatos validos',fold_rows)
                continue
            val,cfg=max(candidatos,key=lambda z:(z[0][0],z[0][1]))
            elegidos.append((float(val[0]),float(val[1]),var,cfg))
            try:
                A,E=datos_ext[var]
                p=ajustar(mod,A,ytr,E,cfg,SEED+fold)
                registrar(fold,rep_opt,mod,yev,p,fold_rows)
                fold_sel.append(dict(Fold=fold,Representación=rep_opt,Modelo=mod,
                                     Variante=var,Config=str(cfg),F1_interno=float(val[0])))
            except Exception as err:
                fallo(fold,rep_opt,mod,err,fold_rows)
        
        
        if elegidos:
            f1_int,r_int,var,cfg=max(elegidos,key=lambda z:(z[0],z[1]))
            try:
                A,E=datos_ext[var]
                p=ajustar(mod,A,ytr,E,cfg,SEED+fold)
                registrar(fold,'HMVT-Optimizado',mod,yev,p,fold_rows)
                fold_sel.append(dict(Fold=fold,Representación='HMVT-Optimizado',Modelo=mod,
                                     Variante=var,Config=str(cfg),F1_interno=f1_int))
            except Exception as err:
                fallo(fold,'HMVT-Optimizado',mod,err,fold_rows)
        else:
            fallo(fold,'HMVT-Optimizado',mod,'Sin variantes validas',fold_rows)
        print(f'  {mod}: completado · {tiempo():.1f} min',flush=True)
        
        
        if tiempo() >= PRESUPUESTO_S/60:
            print('Tiempo agotado: se conserva tabla parcial con FOLDS por fila.')
            break
    REG.extend(fold_rows);SELECCION.extend(fold_sel)
    if len(set(r['Modelo'] for r in fold_rows))==len(MODELOS):
        FOLDS_OK+=1
    else:
        print('ATENCION: fold parcial, NO contabilizado como fold completo')
        break
    if tiempo()>=PRESUPUESTO_S/60:break

TABLA_COMPLETA=resumir(REG)
print('\nTABLA COMPLETA: TODAS LAS REPRESENTACIONES Y LOS 8 CLASIFICADORES')
print('Folds externos completos:',FOLDS_OK,'/',FOLDS)
display(TABLA_COMPLETA[['Representación','Modelo de clasificación','Folds','Precisión','Recall','F1-Score','Accuracy']])
print('\nMETRICAS DE CLASES MINORITARIAS PARA TODAS LAS REPRESENTACIONES')
display(TABLA_COMPLETA[['Representación','Modelo de clasificación','Folds','F1-Score',
                        'Recall Acoso','Recall Violencia sexual','F1 Acoso','F1 Violencia sexual']])
print('\nSELECCION INTERNA SOLO HMVT: VARIANTES Y CONFIGURACIONES')
display(pd.DataFrame(SELECCION))
print('\nCOMPARACION DESCRIPTIVA HMVT OPTIMIZADO VS TF-IDF FIJO (SIN TUNING)')
resultados=pd.DataFrame(REG);compara=[]
for mod in MODELOS:
    a=resultados[(resultados.Modelo==mod)&(resultados.Representación=='HMVT-Optimizado')].set_index('Fold')['F1-Score']
    b=resultados[(resultados.Modelo==mod)&(resultados.Representación=='TF-IDF')].set_index('Fold')['F1-Score']
    delta=(a-b).dropna()
    compara.append({'Modelo':mod,'Folds comparados':len(delta),
                    'Delta F1 macro':round(float(delta.mean()),4) if len(delta) else np.nan,
                    'Folds favorables':int((delta>0).sum()) if len(delta) else 0})
display(pd.DataFrame(compara))
print('NOTA: TF-IDF sin optimizar; esta comparacion NO aísla el efecto de la representacion.')
if ERRORES:
    print('\nERRORES POR COMBINACION:')
    display(pd.DataFrame(ERRORES,columns=['Fold','Representación','Modelo','Error']).head(100))
print(f'\nFIN: {tiempo():.1f} min. Folds externos completos: {FOLDS_OK}/{FOLDS}.')
print('TEST NO utilizado. Baselines sin tuning. Optimizacion SOLO HMVT.')


### EXP-045 · V7

**Archivo de origen:** `Untitled7.ipynb, celda 1`. **Referencias duplicadas:** 0. **Sintaxis de código:** Válida.

HMVT-V7 · EXIST2021 · ESPAÑOL · UNA SOLA CELDA GOOGLE COLAB
SOLO TRAIN · 4 CLASES · SOLO HMVT OPTIMIZADO
3 folds externos x 2 internos · presupuesto de tiempo 52 minutos
Baselines TF-IDF, N-gramas, BoW, Embeddings: FIJOS, SIN TUNING
NUEVO: 

**Ejecución:** celda independiente o variante experimental; revisar archivos de entrada y dependencias antes de ejecutarla. Los resultados originales no se insertan en esta celda.

In [ ]:
import os, sys, time, re, subprocess, warnings, unicodedata
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, hstack, issparse
from scipy.special import softmax
from IPython.display import display
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.preprocessing import MaxAbsScaler
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import MultinomialNB, ComplementNB, GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
warnings.filterwarnings('ignore', category=FutureWarning)
SEED=42; N_OUT=3; N_IN=2; MAX_MINUTOS=52
USAR_EMBEDDINGS=True    
INICIO=time.monotonic()
def minutos(): return (time.monotonic()-INICIO)/60
NOMBRES=['No violencia','Misoginia','Acoso','Violencia sexual']
MODELOS=['Regresión Logística','3-NN','SVM','Naive Bayes','Red neuronal','XGBoost','Random Forest','Árbol de decisión']
MET=['Precisión macro','Recall macro','F1 macro','Accuracy','Recall Acoso','Recall Violencia sexual','F1 Acoso','F1 Violencia sexual']




def cargar_train():
    try:
        from google.colab import files
        print('SUBE TRAIN ESPAÑOL: EXIST2021_train_reestructurado.csv', flush=True)
        recibidos=files.upload()
        rutas=[Path('/content')/n for n in recibidos]
    except ImportError:
        rutas=[p for root in [Path('/mnt/data'),Path.cwd()] if root.exists()
               for p in root.glob('EXIST2021_train_reestructurado*.csv')]
    for p in rutas:
        if not p.exists() or 'test' in p.name.lower(): continue
        try:
            d=pd.read_csv(p,sep='\t' if p.suffix.lower()=='.tsv' else ',',encoding='utf-8-sig')
            d.columns=d.columns.str.strip()
            if 'tipo de violencia' in d and 'tipo_violencia' not in d:
                d=d.rename(columns={'tipo de violencia':'tipo_violencia'})
            if {'text','tipo_violencia'}.issubset(d.columns): return d,p
        except Exception as e: print('Archivo no válido:',p.name,str(e)[:90])
    raise FileNotFoundError('Debe subir TRAIN español con columnas text y tipo_violencia; TEST no se utiliza.')

def canon(x):
    return ''.join(z for z in unicodedata.normalize('NFKD',str(x).casefold()) if not unicodedata.combining(z))
raw,ruta=cargar_train()
if 'language' in raw.columns:
    raw=raw[raw.language.astype(str).str.lower().str.strip().isin(['es','spanish'])].copy()
raw=raw.dropna(subset=['text','tipo_violencia']).copy()
raw['text']=raw.text.astype(str).str.replace(r'\s+',' ',regex=True).str.strip()
raw=raw[raw.text.str.len()>0].copy()
mapa={'0':0,'1':1,'2':2,'3':3,'no violencia':0,'misoginia':1,'acoso':2,'violencia sexual':3}
raw['y']=raw.tipo_violencia.map(lambda z: mapa.get(canon(z).strip(),np.nan))
if raw.y.isna().any(): raise ValueError('Etiquetas no reconocidas: '+str(raw.loc[raw.y.isna(),'tipo_violencia'].unique()))
raw['y']=raw.y.astype(int);raw['clave']=raw.text.map(canon)
contr=raw.groupby('clave').y.nunique(); ambiguas=set(contr[contr>1].index)
if ambiguas: print('Textos contradictorios descartados:',len(ambiguas))
df=raw[~raw.clave.isin(ambiguas)].drop_duplicates('clave').reset_index(drop=True)
if set(df.y)!={0,1,2,3}: raise ValueError('Se requieren etiquetas 0,1,2,3.')
T=df.text.to_numpy(); Y=df.y.to_numpy(); print('TRAIN:',ruta.name,'N=',len(Y),'distribución=',dict(sorted(Counter(Y).items())))




LEX={
 'insultos':['puta','puto','perra','zorra','imbecil','pendeja','pendejo','estupida','estupido','idiota','cabrona','cabron'],
 'misoginia':['feminazi','feminista','feminismo','machista','mujer','mujeres','hembrista'],
 'acoso':['acosar','acoso','hostigar','acosador','molestar','amenaza','amenazar','perseguir'],
 'sexual':['sexo','sexual','violar','violacion','follar','coger','pene','tetas','desnuda','penetrar'],
 'control':['callate','obediente','sumisa','sirve','merece','deberias','prohibir'],
 'negacion':['no','nunca','jamas','nadie','ninguna'],
 'persona':['tu','tus','ella','ellas','vosotras','ustedes','te','ti']
}
RX={k:[re.compile(r'(?<!\w)'+re.escape(w)+r'(?!\w)') for w in palabras] for k,palabras in LEX.items()}
def discurso(textos):
    filas=[]
    for original in textos:
        s=canon(original); tk=re.findall(r'\b\w+\b',s)
        fila=[np.log1p(sum(len(r.findall(s)) for r in rr)) for rr in RX.values()]
        fila.extend([np.log1p(len(tk)),len(set(tk))/max(1,len(tk)),np.log1p(s.count('!')),
                     np.log1p(s.count('?')),np.log1p(s.count('@')),int('#' in s),int('http' in s),
                     np.log1p(sum(x.isupper() for x in str(original).split())),
                     np.log1p(len(re.findall(r'(.)\1{2,}',s)))])
        filas.append(fila)
    return np.asarray(filas,dtype=np.float32)

class Caracteristicas:
    def fit(self,txt):
        self.w=TfidfVectorizer(ngram_range=(1,2),min_df=2,max_features=16000,
            sublinear_tf=True,strip_accents='unicode',token_pattern=r'(?u)\b\w+\b')
        self.c=TfidfVectorizer(analyzer='char_wb',ngram_range=(2,5),min_df=2,
            max_features=24000,sublinear_tf=True)
        self.ng=CountVectorizer(ngram_range=(1,2),min_df=2,max_features=10000)
        self.bow=CountVectorizer(min_df=2,max_features=10000)
        self.w.fit(txt);self.c.fit(txt);self.ng.fit(txt);self.bow.fit(txt)
        self.dscale=MaxAbsScaler().fit(discurso(txt))
        return self
    def bloques(self,txt):
        a=self.w.transform(txt).astype(np.float32)
        b=self.c.transform(txt).astype(np.float32)
        c=csr_matrix(self.dscale.transform(discurso(txt)).astype(np.float32))
        return (a,b,c)
    def base(self,nom,txt,b):
        if nom=='TF-IDF':return b[0]
        if nom=='N-gramas':return self.ng.transform(txt)
        if nom=='BoW':return self.bow.transform(txt)
        raise ValueError(nom)



FAMILIAS={
 'HMVT-Palabras+Caracteres':[(1,.6,0),(1,1.2,0)],
 'HMVT-Completo':[(1,.75,.20),(1,1.25,.45)],
 'HMVT-DiscursoFuerte':[(1,.75,.8),(1,1.1,1.2)],
 'HMVT-SinCaracteres':[(1,0,.3),(1,0,.9)],
 'HMVT-CaracteresFuerte':[(.65,1.5,.2),(.8,2,.5)],
}
def unir(b,p): return hstack([x.multiply(float(w)) for x,w in zip(b,p) if w>0],format='csr',dtype=np.float32)

def met(y,p):
    prec,rec,f1,_=precision_recall_fscore_support(y,p,labels=[0,1,2,3],average='macro',zero_division=0)
    _,rs,fs,_=precision_recall_fscore_support(y,p,labels=[0,1,2,3],average=None,zero_division=0)
    return dict(zip(MET,[prec,rec,f1,accuracy_score(y,p),rs[2],rs[3],fs[2],fs[3]]))

def sobremuestreo(X,y,modo,seed):
    if modo not in ['parcial','ros']:return X,y
    rng=np.random.default_rng(seed); ma=np.bincount(y,minlength=4).max(); idx=[]
    for c in range(4):
        origen=np.flatnonzero(y==c);n=max(len(origen),int(ma*(.62 if modo=='parcial' else 1.0)))
        idx.extend(rng.choice(origen,size=n,replace=n>len(origen)).tolist())
    idx=np.asarray(idx);rng.shuffle(idx)
    return X[idx],y[idx]

def estimador(nom,cfg,seed):
    balanced='balanced' if cfg.get('bal')=='balanced' else None
    if nom=='Regresión Logística':return LogisticRegression(C=cfg.get('C',1),class_weight=balanced,max_iter=(1600 if cfg.get('bal') else 700),solver='lbfgs',random_state=seed)
    if nom=='3-NN':return KNeighborsClassifier(n_neighbors=cfg.get('k',3),weights='distance',metric='cosine',algorithm='brute')
    if nom=='SVM':return LinearSVC(C=cfg.get('C',1),class_weight=balanced,max_iter=(6000 if cfg.get('bal') else 2000),random_state=seed)
    if nom=='Naive Bayes':return (ComplementNB(alpha=cfg.get('alpha',.5)) if cfg.get('tipo')=='complement' else MultinomialNB(alpha=cfg.get('alpha',1)))
    if nom=='Red neuronal':return MLPClassifier(hidden_layer_sizes=(cfg.get('h',64),),alpha=.005,max_iter=75,early_stopping=True,n_iter_no_change=7,random_state=seed)
    if nom=='XGBoost':
        from xgboost import XGBClassifier
        return XGBClassifier(objective='multi:softprob',num_class=4,n_estimators=cfg.get('n',55),max_depth=cfg.get('depth',3),learning_rate=.12,
            subsample=.85,colsample_bytree=.85,tree_method='hist',n_jobs=2,random_state=seed,eval_metric='mlogloss')
    if nom=='Random Forest':return RandomForestClassifier(n_estimators=cfg.get('n',65),min_samples_leaf=cfg.get('leaf',2),class_weight=balanced,max_features='sqrt',n_jobs=2,random_state=seed)
    if nom=='Árbol de decisión':return DecisionTreeClassifier(max_depth=cfg.get('depth',14),min_samples_leaf=3,class_weight=balanced,random_state=seed)
    raise ValueError(nom)

ARBOLES={'XGBoost','Random Forest','Árbol de decisión'}
def entrenar_scores(nom,X,y,Z,cfg,seed):
    X,y=sobremuestreo(X,np.asarray(y),cfg.get('balance','none'),seed)
    if nom in ARBOLES:
        fs=SelectKBest(chi2,k=min(900,X.shape[1]));X=fs.fit_transform(X,y);Z=fs.transform(Z)
    if nom=='Naive Bayes' and not issparse(X): est=GaussianNB()
    else:est=estimador(nom,cfg,seed)
    opts={}
    if cfg.get('bal')=='sqrt' and nom in ['Regresión Logística','SVM','Naive Bayes','XGBoost','Random Forest','Árbol de decisión']:
        count=np.bincount(y,minlength=4);ws=np.sqrt(len(y)/(4*np.maximum(count,1)));opts['sample_weight']=ws[y]
    est.fit(X,y,**opts)
    if hasattr(est,'predict_proba'):
        pp=np.maximum(est.predict_proba(Z),1e-9)
        scores=np.log(pp)
    elif hasattr(est,'decision_function'):
        scores=np.asarray(est.decision_function(Z),dtype=np.float64)
    else:
        pred=est.predict(Z);scores=np.full((len(pred),4),-50.0);scores[np.arange(len(pred)),pred]=0
    scores=np.asarray(scores,dtype=np.float64)
    if scores.ndim!=2 or scores.shape[1]!=4:raise ValueError('Se esperaban 4 clases.')
    return scores


BASE_CFG={
 'Regresión Logística':{'C':1},'3-NN':{'k':3},'SVM':{'C':1},
 'Naive Bayes':{'tipo':'multinomial','alpha':1},'Red neuronal':{'h':64},
 'XGBoost':{'depth':3,'n':65},'Random Forest':{'leaf':2,'n':75},'Árbol de decisión':{'depth':16}}

OPT_CFG={
 'Regresión Logística':[{'C':.4,'bal':'balanced'},{'C':2,'bal':'balanced'}],
 '3-NN':[{'k':3,'balance':'none'},{'k':5,'balance':'parcial'}],
 'SVM':[{'C':.3,'bal':'balanced'},{'C':1.5,'bal':'balanced'}],
 'Naive Bayes':[{'tipo':'complement','alpha':.1},{'tipo':'complement','alpha':.5,'balance':'parcial'}],
 'Red neuronal':[{'h':64,'balance':'parcial'},{'h':96,'balance':'none'}],
 'XGBoost':[{'depth':3,'n':55,'bal':'sqrt'},{'depth':4,'n':55,'balance':'parcial'}],
 'Random Forest':[{'leaf':2,'n':65,'balance':'parcial'},{'leaf':3,'n':65,'bal':'balanced'}],
 'Árbol de decisión':[{'depth':14,'balance':'parcial'},{'depth':20,'bal':'balanced'}]
}




OFFSETS=[(0,0),(.2,.2),(.4,.4),(.7,.4),(.4,.7),(.8,.8),(1.1,.6),(.6,1.1)]
def predecir(s,offset):
    mod=s.copy();mod[:,2]+=offset[0];mod[:,3]+=offset[1]
    return mod.argmax(axis=1)

try:import xgboost
except ImportError:subprocess.check_call([sys.executable,'-m','pip','install','-q','xgboost'])
EMB=None
if USAR_EMBEDDINGS:
    try:from sentence_transformers import SentenceTransformer
    except ImportError:
        subprocess.check_call([sys.executable,'-m','pip','install','-q','sentence-transformers'])
        from sentence_transformers import SentenceTransformer
    print('Codificando Embeddings congelados (NO se optimizan)...',flush=True)
    encoder=SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')
    encoder.eval();EMB=encoder.encode(T.tolist(),batch_size=64,show_progress_bar=True,normalize_embeddings=False,convert_to_numpy=True).astype(np.float32)
    del encoder

REPS=['TF-IDF','N-gramas','BoW']+(['Embeddings'] if EMB is not None else [])
REPS+=list(FAMILIAS)
REPS += [x+'-Optimizado' for x in FAMILIAS]
REPS+=['HMVT-Optimizado']
RESULTADOS=[];SELECCIONES=[];ERRORES=[];FOLDS_OK=0

def resumen_completo(reg):
    d=pd.DataFrame(reg);rows=[]
    for r in REPS:
        for m in MODELOS:
            a=d[(d['Representación']==r)&(d['Clasificador']==m)] if len(d) else pd.DataFrame()
            row={'Representación':r,'Clasificador':m,'Folds':int(a['F1 macro'].notna().sum()) if len(a) else 0}
            for col in MET:
                vals=a[col].dropna().to_numpy(dtype=float) if len(a) else np.array([])
                row[col]=(f'{vals.mean():.4f} ± {vals.std(ddof=1):.4f}' if len(vals)>1 else
                         (f'{vals[0]:.4f} (1 fold)' if len(vals)==1 else 'PENDIENTE/ERROR'))
                row[col+'_media']=float(vals.mean()) if len(vals) else np.nan
            rows.append(row)
    return pd.DataFrame(rows)

CV=StratifiedKFold(n_splits=N_OUT,shuffle=True,random_state=SEED)
for fold,(itr,iev) in enumerate(CV.split(T,Y),start=1):
    if minutos()>MAX_MINUTOS and FOLDS_OK>0:break
    print(f'\n========== FOLD {fold}/{N_OUT} | {minutos():.1f} minutos ==========',flush=True)
    txttr,txtev=T[itr],T[iev];ytr,yev=Y[itr],Y[iev]
    ex=Caracteristicas().fit(txttr);A=ex.bloques(txttr);E=ex.bloques(txtev)
    
    inner=[]
    inner_cv=StratifiedKFold(n_splits=N_IN,shuffle=True,random_state=SEED+fold)
    for idx,(ii,iv) in enumerate(inner_cv.split(txttr,ytr)):
        ei=Caracteristicas().fit(txttr[ii]);inner.append((ei.bloques(txttr[ii]),ei.bloques(txttr[iv]),ytr[ii],ytr[iv]))
    filas_fold=[]; sel_fold=[]
    for nombre in MODELOS:
        
        for rep in ['TF-IDF','N-gramas','BoW']+(['Embeddings'] if EMB is not None else []):
            try:
                if rep=='Embeddings': X,Z=EMB[itr],EMB[iev]
                else:X,Z=ex.base(rep,txttr,A),ex.base(rep,txtev,E)
                if rep=='Embeddings' and nombre=='Naive Bayes':
                    model=GaussianNB();model.fit(X,ytr);pred=model.predict(Z)
                else:
                    scores=entrenar_scores(nombre,X,ytr,Z,BASE_CFG[nombre],SEED+fold)
                    pred=predecir(scores,(0,0))
                filas_fold.append(dict(Fold=fold,Representación=rep,Clasificador=nombre,**met(yev,pred)))
            except Exception as error:
                ERRORES.append((fold,rep,nombre,str(error)))
                filas_fold.append(dict(Fold=fold,Representación=rep,Clasificador=nombre,**{c:np.nan for c in MET}))
        
        for var,pesos in [(v,ps[0]) for v,ps in FAMILIAS.items()]:
            try:
                s=entrenar_scores(nombre,unir(A,pesos),ytr,unir(E,pesos),BASE_CFG[nombre],SEED+fold)
                filas_fold.append(dict(Fold=fold,Representación=var,Clasificador=nombre,**met(yev,predecir(s,(0,0)))))
            except Exception as error:
                ERRORES.append((fold,var,nombre,str(error)))
                filas_fold.append(dict(Fold=fold,Representación=var,Clasificador=nombre,**{c:np.nan for c in MET}))
        candidatas_global=[]
        for var,pesos_lista in FAMILIAS.items():
            mejor_var=None
            for pesos in pesos_lista:
                for cfg in OPT_CFG[nombre]:
                    try:
                        scored=[]
                        for j,(ba,bv,ya,yv) in enumerate(inner):
                            ss=entrenar_scores(nombre,unir(ba,pesos),ya,unir(bv,pesos),cfg,SEED+fold*100+j)
                            scored.append((ss,yv))
                        
                        for offset in OFFSETS:
                            mm=[met(yv,predecir(ss,offset)) for ss,yv in scored]
                            f=float(np.mean([v['F1 macro'] for v in mm]))
                            r=float(np.mean([(v['Recall Acoso']+v['Recall Violencia sexual'])/2 for v in mm]))
                            item=(f,r,var,pesos,cfg,offset)
                            if mejor_var is None or item[:2]>mejor_var[:2]:mejor_var=item
                    except Exception as err:
                        ERRORES.append((fold,'Interna '+var,nombre,str(err)))
            if mejor_var is None:
                filas_fold.append(dict(Fold=fold,Representación=var+'-Optimizado',Clasificador=nombre,**{c:np.nan for c in MET}))
                continue
            candidatas_global.append(mejor_var)
            _,_,_,pesos,cfg,off=mejor_var
            try:
                ss=entrenar_scores(nombre,unir(A,pesos),ytr,unir(E,pesos),cfg,SEED+fold)
                filas_fold.append(dict(Fold=fold,Representación=var+'-Optimizado',Clasificador=nombre,**met(yev,predecir(ss,off))))
                sel_fold.append(dict(Fold=fold,Representación=var+'-Optimizado',Clasificador=nombre,Pesos=str(pesos),Config=str(cfg),
                                     Ajuste_clase_2=off[0],Ajuste_clase_3=off[1],F1_interno=mejor_var[0]))
            except Exception as err:
                ERRORES.append((fold,var+'-Optimizado',nombre,str(err)))
                filas_fold.append(dict(Fold=fold,Representación=var+'-Optimizado',Clasificador=nombre,**{c:np.nan for c in MET}))
        
        if candidatas_global:
            best=max(candidatas_global,key=lambda q:q[:2])
            _,_,var,pesos,cfg,off=best
            try:
                ss=entrenar_scores(nombre,unir(A,pesos),ytr,unir(E,pesos),cfg,SEED+fold)
                filas_fold.append(dict(Fold=fold,Representación='HMVT-Optimizado',Clasificador=nombre,**met(yev,predecir(ss,off))))
                sel_fold.append(dict(Fold=fold,Representación='HMVT-Optimizado',Clasificador=nombre,Pesos=str(pesos),Config=str(cfg),
                                     Ajuste_clase_2=off[0],Ajuste_clase_3=off[1],F1_interno=best[0],Variante=var))
            except Exception as err:
                ERRORES.append((fold,'HMVT-Optimizado',nombre,str(err)))
                filas_fold.append(dict(Fold=fold,Representación='HMVT-Optimizado',Clasificador=nombre,**{c:np.nan for c in MET}))
        print(f'  {nombre}: completado | {minutos():.1f} minutos',flush=True)
        
    RESULTADOS.extend(filas_fold);SELECCIONES.extend(sel_fold);FOLDS_OK+=1
    print('Fold finalizado:',fold,'/',N_OUT,flush=True)
    if minutos()>=MAX_MINUTOS:
        print('Presupuesto alcanzado: no se inicia otro fold.',flush=True);break

TABLA_COMPLETA=resumen_completo(RESULTADOS)
print('\n'+'='*95+'\nTABLA MAESTRA COMPLETA · ESPAÑOL · SOLO TRAIN · FOLDS',FOLDS_OK,'/',N_OUT)
display(TABLA_COMPLETA[['Representación','Clasificador','Folds','Precisión macro','Recall macro','F1 macro','Accuracy']])
print('\nMETRICAS DE CLASES MINORITARIAS · TODAS LAS REPRESENTACIONES')
display(TABLA_COMPLETA[['Representación','Clasificador','Folds','F1 macro','Recall Acoso','Recall Violencia sexual','F1 Acoso','F1 Violencia sexual']])
print('\nTODAS LAS SELECCIONES INTERNAS HMVT: hiperparametros y offsets clase 2/3')
display(pd.DataFrame(SELECCIONES))
print('\nCOMPARACION DESCRIPTIVA HMVT OPTIMIZADO vs TF-IDF FIJO (NO AISLA TUNING)')
a=pd.DataFrame(RESULTADOS); compar=[]
for nombre in MODELOS:
    h=a[(a.Representación=='HMVT-Optimizado')&(a.Clasificador==nombre)].set_index('Fold')['F1 macro']
    b=a[(a.Representación=='TF-IDF')&(a.Clasificador==nombre)].set_index('Fold')['F1 macro']
    d=(h-b).dropna()
    compar.append({'Clasificador':nombre,'Folds':len(d),'Delta F1 macro':round(float(d.mean()),4) if len(d) else np.nan,
                   'Folds con HMVT mejor':int((d>0).sum())})
display(pd.DataFrame(compar))
print('\nMEJOR VARIANTE HMVT POR CLASIFICADOR (DESCRIPTIVO, NO USAR PARA TUNING)')
mini=TABLA_COMPLETA[TABLA_COMPLETA.Representación.str.startswith('HMVT')].copy()
mini=mini.dropna(subset=['F1 macro_media'])
if len(mini):display(mini.loc[mini.groupby('Clasificador')['F1 macro_media'].idxmax(),['Clasificador','Representación','Folds','F1 macro','Accuracy']].sort_values('F1 macro',ascending=False))
if ERRORES:
    print('\nERRORES DETECTADOS:');display(pd.DataFrame(ERRORES,columns=['Fold','Representación','Clasificador','Error']).head(50))
print(f'\nFIN. Tiempo: {minutos():.1f} min; folds {FOLDS_OK}/{N_OUT}. TEST NO utilizado. BASELINES SIN OPTIMIZAR.')
print('Los ajustes de clases se eligen solo con entrenamiento interno. No se garantiza un aumento de F1.')


### EXP-046 · V8

**Archivo de origen:** `Untitled7.ipynb, celda 2`. **Referencias duplicadas:** 1. **Sintaxis de código:** Válida.

HMVT-V8 · EXIST2021 · ESPAÑOL · UNA SOLA CELDA GOOGLE COLAB
SOLO TRAIN · 4 CLASES · SOLO HMVT OPTIMIZADO
3 folds externos x 2 internos · presupuesto de tiempo 52 minutos; MATRICES OOF
Baselines TF-IDF, N-gramas, BoW, Embeddings: FIJOS, SIN 

**Ejecución:** celda independiente o variante experimental; revisar archivos de entrada y dependencias antes de ejecutarla. Los resultados originales no se insertan en esta celda.

In [ ]:
import os, sys, time, re, subprocess, warnings, unicodedata
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, hstack, issparse
from scipy.special import softmax
from IPython.display import display
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.preprocessing import MaxAbsScaler
from sklearn.feature_selection import SelectKBest, chi2, f_classif
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import MultinomialNB, ComplementNB, GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
warnings.filterwarnings('ignore', category=FutureWarning)
SEED=42; N_OUT=3; N_IN=2; MAX_MINUTOS=52
USAR_EMBEDDINGS=True    
INICIO=time.monotonic()
def minutos(): return (time.monotonic()-INICIO)/60
NOMBRES=['No violencia','Misoginia','Acoso','Violencia sexual']
MODELOS=['Regresión Logística','3-NN','SVM','Naive Bayes','Red neuronal','XGBoost','Random Forest','Árbol de decisión']
MET=['Precisión macro','Recall macro','F1 macro','Accuracy','Recall Acoso','Recall Violencia sexual','F1 Acoso','F1 Violencia sexual']




def cargar_train():
    try:
        from google.colab import files
        print('SUBE TRAIN ESPAÑOL: EXIST2021_train_reestructurado.csv', flush=True)
        recibidos=files.upload()
        rutas=[Path('/content')/n for n in recibidos]
    except ImportError:
        rutas=[p for root in [Path('/mnt/data'),Path.cwd()] if root.exists()
               for p in root.glob('EXIST2021_train_reestructurado*.csv')]
    for p in rutas:
        if not p.exists() or 'test' in p.name.lower(): continue
        try:
            d=pd.read_csv(p,sep='\t' if p.suffix.lower()=='.tsv' else ',',encoding='utf-8-sig')
            d.columns=d.columns.str.strip()
            if 'tipo de violencia' in d and 'tipo_violencia' not in d:
                d=d.rename(columns={'tipo de violencia':'tipo_violencia'})
            if {'text','tipo_violencia'}.issubset(d.columns): return d,p
        except Exception as e: print('Archivo no válido:',p.name,str(e)[:90])
    raise FileNotFoundError('Debe subir TRAIN español con columnas text y tipo_violencia; TEST no se utiliza.')

def canon(x):
    return ''.join(z for z in unicodedata.normalize('NFKD',str(x).casefold()) if not unicodedata.combining(z))
raw,ruta=cargar_train()
if 'language' in raw.columns:
    raw=raw[raw.language.astype(str).str.lower().str.strip().isin(['es','spanish'])].copy()
raw=raw.dropna(subset=['text','tipo_violencia']).copy()
raw['text']=raw.text.astype(str).str.replace(r'\s+',' ',regex=True).str.strip()
raw=raw[raw.text.str.len()>0].copy()
mapa={'0':0,'1':1,'2':2,'3':3,'no violencia':0,'misoginia':1,'acoso':2,'violencia sexual':3}
raw['y']=raw.tipo_violencia.map(lambda z: mapa.get(canon(z).strip(),np.nan))
if raw.y.isna().any(): raise ValueError('Etiquetas no reconocidas: '+str(raw.loc[raw.y.isna(),'tipo_violencia'].unique()))
raw['y']=raw.y.astype(int);raw['clave']=raw.text.map(canon)
contr=raw.groupby('clave').y.nunique(); ambiguas=set(contr[contr>1].index)
if ambiguas: print('Textos contradictorios descartados:',len(ambiguas))
df=raw[~raw.clave.isin(ambiguas)].drop_duplicates('clave').reset_index(drop=True)
if set(df.y)!={0,1,2,3}: raise ValueError('Se requieren etiquetas 0,1,2,3.')
T=df.text.to_numpy(); Y=df.y.to_numpy(); print('TRAIN:',ruta.name,'N=',len(Y),'distribución=',dict(sorted(Counter(Y).items())))




LEX={
 'insultos':['puta','puto','perra','zorra','imbecil','pendeja','pendejo','estupida','estupido','idiota','cabrona','cabron'],
 'misoginia':['feminazi','feminista','feminismo','machista','mujer','mujeres','hembrista'],
 'acoso':['acosar','acoso','hostigar','acosador','molestar','amenaza','amenazar','perseguir'],
 'sexual':['sexo','sexual','violar','violacion','follar','coger','pene','tetas','desnuda','penetrar'],
 'control':['callate','obediente','sumisa','sirve','merece','deberias','prohibir'],
 'negacion':['no','nunca','jamas','nadie','ninguna'],
 'persona':['tu','tus','ella','ellas','vosotras','ustedes','te','ti']
}
RX={k:[re.compile(r'(?<!\w)'+re.escape(w)+r'(?!\w)') for w in palabras] for k,palabras in LEX.items()}
def discurso(textos):
    filas=[]
    for original in textos:
        s=canon(original); tk=re.findall(r'\b\w+\b',s)
        fila=[np.log1p(sum(len(r.findall(s)) for r in rr)) for rr in RX.values()]
        fila.extend([np.log1p(len(tk)),len(set(tk))/max(1,len(tk)),np.log1p(s.count('!')),
                     np.log1p(s.count('?')),np.log1p(s.count('@')),int('#' in s),int('http' in s),
                     np.log1p(sum(x.isupper() for x in str(original).split())),
                     np.log1p(len(re.findall(r'(.)\1{2,}',s)))])
        filas.append(fila)
    return np.asarray(filas,dtype=np.float32)

class Caracteristicas:
    def fit(self,txt):
        self.w=TfidfVectorizer(ngram_range=(1,2),min_df=2,max_features=16000,
            sublinear_tf=True,strip_accents='unicode',token_pattern=r'(?u)\b\w+\b')
        self.c=TfidfVectorizer(analyzer='char_wb',ngram_range=(2,5),min_df=2,
            max_features=24000,sublinear_tf=True)
        self.ng=CountVectorizer(ngram_range=(1,2),min_df=2,max_features=10000)
        self.bow=CountVectorizer(min_df=2,max_features=10000)
        self.w.fit(txt);self.c.fit(txt);self.ng.fit(txt);self.bow.fit(txt)
        self.dscale=MaxAbsScaler().fit(discurso(txt))
        return self
    def bloques(self,txt):
        a=self.w.transform(txt).astype(np.float32)
        b=self.c.transform(txt).astype(np.float32)
        c=csr_matrix(self.dscale.transform(discurso(txt)).astype(np.float32))
        return (a,b,c)
    def base(self,nom,txt,b):
        if nom=='TF-IDF':return b[0]
        if nom=='N-gramas':return self.ng.transform(txt)
        if nom=='BoW':return self.bow.transform(txt)
        raise ValueError(nom)



FAMILIAS={
 'HMVT-Palabras+Caracteres':[(1,.6,0),(1,1.2,0)],
 'HMVT-Completo':[(1,.75,.20),(1,1.25,.45)],
 'HMVT-DiscursoFuerte':[(1,.75,.8),(1,1.1,1.2)],
 'HMVT-SinCaracteres':[(1,0,.3),(1,0,.9)],
 'HMVT-CaracteresFuerte':[(.65,1.5,.2),(.8,2,.5)],
 'HMVT-CaracteresDominantes':[(.45,2.2,.15),(.7,2.5,.30)],
}
def unir(b,p): return hstack([x.multiply(float(w)) for x,w in zip(b,p) if w>0],format='csr',dtype=np.float32)

def met(y,p):
    prec,rec,f1,_=precision_recall_fscore_support(y,p,labels=[0,1,2,3],average='macro',zero_division=0)
    _,rs,fs,_=precision_recall_fscore_support(y,p,labels=[0,1,2,3],average=None,zero_division=0)
    return dict(zip(MET,[prec,rec,f1,accuracy_score(y,p),rs[2],rs[3],fs[2],fs[3]]))

def sobremuestreo(X,y,modo,seed):
    if modo not in ['parcial','ros']:return X,y
    rng=np.random.default_rng(seed); ma=np.bincount(y,minlength=4).max(); idx=[]
    for c in range(4):
        origen=np.flatnonzero(y==c);n=max(len(origen),int(ma*(.62 if modo=='parcial' else 1.0)))
        idx.extend(rng.choice(origen,size=n,replace=n>len(origen)).tolist())
    idx=np.asarray(idx);rng.shuffle(idx)
    return X[idx],y[idx]

def estimador(nom,cfg,seed):
    balanced='balanced' if cfg.get('bal')=='balanced' else None
    if nom=='Regresión Logística':return LogisticRegression(C=cfg.get('C',1),class_weight=balanced,max_iter=(1600 if cfg.get('bal') else 700),solver='lbfgs',random_state=seed)
    if nom=='3-NN':return KNeighborsClassifier(n_neighbors=cfg.get('k',3),weights='distance',metric='cosine',algorithm='brute')
    if nom=='SVM':return LinearSVC(C=cfg.get('C',1),class_weight=balanced,max_iter=(6000 if cfg.get('bal') else 2000),random_state=seed)
    if nom=='Naive Bayes':return (ComplementNB(alpha=cfg.get('alpha',.5)) if cfg.get('tipo')=='complement' else MultinomialNB(alpha=cfg.get('alpha',1)))
    if nom=='Red neuronal':return MLPClassifier(hidden_layer_sizes=(cfg.get('h',64),),alpha=.005,max_iter=75,early_stopping=True,n_iter_no_change=7,random_state=seed)
    if nom=='XGBoost':
        from xgboost import XGBClassifier
        return XGBClassifier(objective='multi:softprob',num_class=4,n_estimators=cfg.get('n',55),max_depth=cfg.get('depth',3),learning_rate=.12,
            subsample=.85,colsample_bytree=.85,tree_method='hist',n_jobs=2,random_state=seed,eval_metric='mlogloss')
    if nom=='Random Forest':return RandomForestClassifier(n_estimators=cfg.get('n',65),min_samples_leaf=cfg.get('leaf',2),class_weight=balanced,max_features='sqrt',n_jobs=2,random_state=seed)
    if nom=='Árbol de decisión':return DecisionTreeClassifier(max_depth=cfg.get('depth',14),min_samples_leaf=3,class_weight=balanced,random_state=seed)
    raise ValueError(nom)

ARBOLES={'XGBoost','Random Forest','Árbol de decisión'}
def entrenar_scores(nom,X,y,Z,cfg,seed):
    
    
    X,y=sobremuestreo(X,np.asarray(y),cfg.get('balance','none'),seed)
    ksel=cfg.get('select_k',None)
    if ksel is not None:
        k=min(int(ksel),X.shape[1])
        if k<X.shape[1]:
            fs=SelectKBest(chi2 if (issparse(X) or np.min(X)>=0) else f_classif,k=k)
            X=fs.fit_transform(X,y);Z=fs.transform(Z)
    if nom in ARBOLES:
        k=min(900,X.shape[1])
        if k < X.shape[1]:
            
            
            selector=SelectKBest(chi2 if (issparse(X) or np.min(X)>=0) else f_classif,k=k)
            X=selector.fit_transform(X,y);Z=selector.transform(Z)
    if nom=='Naive Bayes' and not issparse(X): est=GaussianNB()
    else:est=estimador(nom,cfg,seed)
    opts={}
    if cfg.get('bal')=='sqrt' and nom in ['Regresión Logística','SVM','Naive Bayes','XGBoost','Random Forest','Árbol de decisión']:
        count=np.bincount(y,minlength=4);ws=np.sqrt(len(y)/(4*np.maximum(count,1)));opts['sample_weight']=ws[y]
    est.fit(X,y,**opts)
    if hasattr(est,'predict_proba'):
        pp=np.maximum(est.predict_proba(Z),1e-9)
        scores=np.log(pp)
    elif hasattr(est,'decision_function'):
        scores=np.asarray(est.decision_function(Z),dtype=np.float64)
    else:
        pred=est.predict(Z);scores=np.full((len(pred),4),-50.0);scores[np.arange(len(pred)),pred]=0
    scores=np.asarray(scores,dtype=np.float64)
    if scores.ndim!=2 or scores.shape[1]!=4:raise ValueError('Se esperaban 4 clases.')
    return scores


BASE_CFG={
 'Regresión Logística':{'C':1},'3-NN':{'k':3},'SVM':{'C':1},
 'Naive Bayes':{'tipo':'multinomial','alpha':1},'Red neuronal':{'h':64},
 'XGBoost':{'depth':3,'n':65},'Random Forest':{'leaf':2,'n':75},'Árbol de decisión':{'depth':16}}

OPT_CFG={
 'Regresión Logística':[{'C':.6,'bal':'balanced','select_k':15000},{'C':2,'bal':'balanced'}],
 '3-NN':[{'k':3,'balance':'none'},{'k':5,'balance':'parcial'}],
 'SVM':[{'C':.4,'bal':'balanced','select_k':16000},{'C':1.5,'bal':'balanced'}],
 'Naive Bayes':[{'tipo':'complement','alpha':.1,'select_k':18000},{'tipo':'complement','alpha':.5,'balance':'parcial'}],
 'Red neuronal':[{'h':64,'balance':'parcial'},{'h':96,'balance':'none'}],
 'XGBoost':[{'depth':3,'n':55,'bal':'sqrt'},{'depth':4,'n':55,'balance':'parcial'}],
 'Random Forest':[{'leaf':2,'n':65,'balance':'parcial'},{'leaf':3,'n':65,'bal':'balanced'}],
 'Árbol de decisión':[{'depth':14,'balance':'parcial'},{'depth':20,'bal':'balanced'}]
}




OFFSETS=[(0,0),(.2,.2),(.4,.4),(.7,.4),(.4,.7),(.8,.8),(1.1,.6),(.6,1.1),(-.2,.2),(.2,-.2),(.9,.3),(.3,.9)]
def predecir(s,offset):
    mod=s.copy();mod[:,2]+=offset[0];mod[:,3]+=offset[1]
    return mod.argmax(axis=1)

try:import xgboost
except ImportError:subprocess.check_call([sys.executable,'-m','pip','install','-q','xgboost'])
EMB=None
if USAR_EMBEDDINGS:
    try:from sentence_transformers import SentenceTransformer
    except ImportError:
        subprocess.check_call([sys.executable,'-m','pip','install','-q','sentence-transformers'])
        from sentence_transformers import SentenceTransformer
    print('Codificando Embeddings congelados (NO se optimizan)...',flush=True)
    encoder=SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')
    encoder.eval();EMB=encoder.encode(T.tolist(),batch_size=64,show_progress_bar=True,normalize_embeddings=False,convert_to_numpy=True).astype(np.float32)
    del encoder

REPS=['TF-IDF','N-gramas','BoW']+(['Embeddings'] if EMB is not None else [])
REPS+=list(FAMILIAS)
REPS += [x+'-Optimizado' for x in FAMILIAS]
REPS+=['HMVT-Optimizado']
RESULTADOS=[];SELECCIONES=[];ERRORES=[];FOLDS_OK=0
PREDICCIONES_OOF={}  

def resumen_completo(reg):
    d=pd.DataFrame(reg);rows=[]
    for r in REPS:
        for m in MODELOS:
            a=d[(d['Representación']==r)&(d['Clasificador']==m)] if len(d) else pd.DataFrame()
            row={'Representación':r,'Clasificador':m,'Folds':int(a['F1 macro'].notna().sum()) if len(a) else 0}
            for col in MET:
                vals=a[col].dropna().to_numpy(dtype=float) if len(a) else np.array([])
                row[col]=(f'{vals.mean():.4f} ± {vals.std(ddof=1):.4f}' if len(vals)>1 else
                         (f'{vals[0]:.4f} (1 fold)' if len(vals)==1 else 'PENDIENTE/ERROR'))
                row[col+'_media']=float(vals.mean()) if len(vals) else np.nan
            rows.append(row)
    return pd.DataFrame(rows)

CV=StratifiedKFold(n_splits=N_OUT,shuffle=True,random_state=SEED)
for fold,(itr,iev) in enumerate(CV.split(T,Y),start=1):
    if minutos()>MAX_MINUTOS and FOLDS_OK>0:break
    print(f'\n========== FOLD {fold}/{N_OUT} | {minutos():.1f} minutos ==========',flush=True)
    txttr,txtev=T[itr],T[iev];ytr,yev=Y[itr],Y[iev]
    ex=Caracteristicas().fit(txttr);A=ex.bloques(txttr);E=ex.bloques(txtev)
    
    inner=[]
    inner_cv=StratifiedKFold(n_splits=N_IN,shuffle=True,random_state=SEED+fold)
    for idx,(ii,iv) in enumerate(inner_cv.split(txttr,ytr)):
        ei=Caracteristicas().fit(txttr[ii]);inner.append((ei.bloques(txttr[ii]),ei.bloques(txttr[iv]),ytr[ii],ytr[iv]))
    filas_fold=[]; sel_fold=[]
    for nombre in MODELOS:
        
        for rep in ['TF-IDF','N-gramas','BoW']+(['Embeddings'] if EMB is not None else []):
            try:
                if rep=='Embeddings': X,Z=EMB[itr],EMB[iev]
                else:X,Z=ex.base(rep,txttr,A),ex.base(rep,txtev,E)
                if rep=='Embeddings' and nombre=='Naive Bayes':
                    model=GaussianNB();model.fit(X,ytr);pred=model.predict(Z)
                else:
                    scores=entrenar_scores(nombre,X,ytr,Z,BASE_CFG[nombre],SEED+fold)
                    pred=predecir(scores,(0,0))
                filas_fold.append(dict(Fold=fold,Representación=rep,Clasificador=nombre,**met(yev,pred)))
                PREDICCIONES_OOF.setdefault((rep,nombre),[]).append((iev.copy(),np.asarray(pred).copy()))
            except Exception as error:
                ERRORES.append((fold,rep,nombre,str(error)))
                filas_fold.append(dict(Fold=fold,Representación=rep,Clasificador=nombre,**{c:np.nan for c in MET}))
        
        for var,pesos in [(v,ps[0]) for v,ps in FAMILIAS.items()]:
            try:
                s=entrenar_scores(nombre,unir(A,pesos),ytr,unir(E,pesos),BASE_CFG[nombre],SEED+fold)
                pred_fijo=predecir(s,(0,0))
                filas_fold.append(dict(Fold=fold,Representación=var,Clasificador=nombre,**met(yev,pred_fijo)))
                PREDICCIONES_OOF.setdefault((var,nombre),[]).append((iev.copy(),pred_fijo.copy()))
            except Exception as error:
                ERRORES.append((fold,var,nombre,str(error)))
                filas_fold.append(dict(Fold=fold,Representación=var,Clasificador=nombre,**{c:np.nan for c in MET}))
        candidatas_global=[]
        for var,pesos_lista in FAMILIAS.items():
            mejor_var=None
            for pesos in pesos_lista:
                for cfg in OPT_CFG[nombre]:
                    try:
                        scored=[]
                        for j,(ba,bv,ya,yv) in enumerate(inner):
                            ss=entrenar_scores(nombre,unir(ba,pesos),ya,unir(bv,pesos),cfg,SEED+fold*100+j)
                            scored.append((ss,yv))
                        
                        for offset in OFFSETS:
                            mm=[met(yv,predecir(ss,offset)) for ss,yv in scored]
                            f=float(np.mean([v['F1 macro'] for v in mm]))
                            r=float(np.mean([(v['Recall Acoso']+v['Recall Violencia sexual'])/2 for v in mm]))
                            item=(f,r,var,pesos,cfg,offset)
                            if mejor_var is None or item[:2]>mejor_var[:2]:mejor_var=item
                    except Exception as err:
                        ERRORES.append((fold,'Interna '+var,nombre,str(err)))
            if mejor_var is None:
                filas_fold.append(dict(Fold=fold,Representación=var+'-Optimizado',Clasificador=nombre,**{c:np.nan for c in MET}))
                continue
            candidatas_global.append(mejor_var)
            _,_,_,pesos,cfg,off=mejor_var
            try:
                ss=entrenar_scores(nombre,unir(A,pesos),ytr,unir(E,pesos),cfg,SEED+fold)
                pred_var=predecir(ss,off)
                filas_fold.append(dict(Fold=fold,Representación=var+'-Optimizado',Clasificador=nombre,**met(yev,pred_var)))
                PREDICCIONES_OOF.setdefault((var+'-Optimizado',nombre),[]).append((iev.copy(),pred_var.copy()))
                sel_fold.append(dict(Fold=fold,Representación=var+'-Optimizado',Clasificador=nombre,Pesos=str(pesos),Config=str(cfg),
                                     Ajuste_clase_2=off[0],Ajuste_clase_3=off[1],F1_interno=mejor_var[0]))
            except Exception as err:
                ERRORES.append((fold,var+'-Optimizado',nombre,str(err)))
                filas_fold.append(dict(Fold=fold,Representación=var+'-Optimizado',Clasificador=nombre,**{c:np.nan for c in MET}))
        
        if candidatas_global:
            best=max(candidatas_global,key=lambda q:q[:2])
            _,_,var,pesos,cfg,off=best
            try:
                ss=entrenar_scores(nombre,unir(A,pesos),ytr,unir(E,pesos),cfg,SEED+fold)
                pred_global=predecir(ss,off)
                filas_fold.append(dict(Fold=fold,Representación='HMVT-Optimizado',Clasificador=nombre,**met(yev,pred_global)))
                PREDICCIONES_OOF.setdefault(('HMVT-Optimizado',nombre),[]).append((iev.copy(),pred_global.copy()))
                sel_fold.append(dict(Fold=fold,Representación='HMVT-Optimizado',Clasificador=nombre,Pesos=str(pesos),Config=str(cfg),
                                     Ajuste_clase_2=off[0],Ajuste_clase_3=off[1],F1_interno=best[0],Variante=var))
            except Exception as err:
                ERRORES.append((fold,'HMVT-Optimizado',nombre,str(err)))
                filas_fold.append(dict(Fold=fold,Representación='HMVT-Optimizado',Clasificador=nombre,**{c:np.nan for c in MET}))
        print(f'  {nombre}: completado | {minutos():.1f} minutos',flush=True)
        
    RESULTADOS.extend(filas_fold);SELECCIONES.extend(sel_fold);FOLDS_OK+=1
    print('Fold finalizado:',fold,'/',N_OUT,flush=True)
    if minutos()>=MAX_MINUTOS:
        print('Presupuesto alcanzado: no se inicia otro fold.',flush=True);break

TABLA_COMPLETA=resumen_completo(RESULTADOS)
print('\n'+'='*95+'\nTABLA MAESTRA COMPLETA · ESPAÑOL · SOLO TRAIN · FOLDS',FOLDS_OK,'/',N_OUT)
display(TABLA_COMPLETA[['Representación','Clasificador','Folds','Precisión macro','Recall macro','F1 macro','Accuracy']])
print('\nMETRICAS DE CLASES MINORITARIAS · TODAS LAS REPRESENTACIONES')
display(TABLA_COMPLETA[['Representación','Clasificador','Folds','F1 macro','Recall Acoso','Recall Violencia sexual','F1 Acoso','F1 Violencia sexual']])
print('\nTODAS LAS SELECCIONES INTERNAS HMVT: hiperparametros y offsets clase 2/3')
display(pd.DataFrame(SELECCIONES))
print('\nCOMPARACION DESCRIPTIVA HMVT OPTIMIZADO vs TF-IDF FIJO (NO AISLA TUNING)')
a=pd.DataFrame(RESULTADOS); compar=[]
for nombre in MODELOS:
    h=a[(a.Representación=='HMVT-Optimizado')&(a.Clasificador==nombre)].set_index('Fold')['F1 macro']
    b=a[(a.Representación=='TF-IDF')&(a.Clasificador==nombre)].set_index('Fold')['F1 macro']
    d=(h-b).dropna()
    compar.append({'Clasificador':nombre,'Folds':len(d),'Delta F1 macro':round(float(d.mean()),4) if len(d) else np.nan,
                   'Folds con HMVT mejor':int((d>0).sum())})
display(pd.DataFrame(compar))
print('\nMEJOR VARIANTE HMVT POR CLASIFICADOR (DESCRIPTIVO, NO USAR PARA TUNING)')
mini=TABLA_COMPLETA[TABLA_COMPLETA.Representación.str.startswith('HMVT')].copy()
mini=mini.dropna(subset=['F1 macro_media'])
if len(mini):display(mini.loc[mini.groupby('Clasificador')['F1 macro_media'].idxmax(),['Clasificador','Representación','Folds','F1 macro','Accuracy']].sort_values('F1 macro',ascending=False))
if ERRORES:
    print('\nERRORES DETECTADOS:');display(pd.DataFrame(ERRORES,columns=['Fold','Representación','Clasificador','Error']).head(50))
print(f'\nFIN. Tiempo: {minutos():.1f} min; folds {FOLDS_OK}/{N_OUT}. TEST NO utilizado. BASELINES SIN OPTIMIZAR.')
print('Los ajustes de clases se eligen solo con entrenamiento interno. No se garantiza un aumento de F1.')






print('\nMATRICES DE CONFUSIÓN OOF · HMVT-OPTIMIZADO · ORDEN REAL (filas) / PREDICHO (columnas)')
cm_resultados=[]
for nombre in MODELOS:
    registros=PREDICCIONES_OOF.get(('HMVT-Optimizado',nombre),[])
    if not registros:
        print('Sin predicciones OOF:',nombre)
        continue
    indices=np.concatenate([a for a,b in registros])
    pred=np.concatenate([b for a,b in registros])
    if len(np.unique(indices))!=len(indices):
        raise RuntimeError('Indices externos repetidos: revisar validacion cruzada.')
    verdad=Y[indices]
    cm=confusion_matrix(verdad,pred,labels=[0,1,2,3])
    print('\n',nombre,': muestras OOF =',len(indices),'de',len(Y))
    print(classification_report(verdad,pred,labels=[0,1,2,3],target_names=NOMBRES,zero_division=0,digits=4))
    for i in range(4):
        for j in range(4):
            cm_resultados.append({'Clasificador':nombre,'Real':NOMBRES[i],'Predicha':NOMBRES[j],'Frecuencia':int(cm[i,j])})
    fig,ax=plt.subplots(figsize=(7.5,6))
    im=ax.imshow(cm,cmap='Purples',interpolation='nearest')
    ax.set_xticks(range(4),NOMBRES,rotation=30,ha='right')
    ax.set_yticks(range(4),NOMBRES)
    ax.set_xlabel('Clase predicha');ax.set_ylabel('Clase real')
    ax.set_title('HMVT optimizado · '+nombre+'\nMatriz OOF, n='+str(len(indices)))
    corte=cm.max()*.52
    for i in range(4):
        for j in range(4):
            ax.text(j,i,str(cm[i,j]),ha='center',va='center',
                    color='white' if cm[i,j]>corte else 'black',fontsize=11,fontweight='bold')
    fig.colorbar(im,ax=ax,fraction=.046,pad=.04)
    fig.tight_layout();plt.show();plt.close(fig)
print('\nTABLA DE FRECUENCIAS DE TODAS LAS MATRICES (valores absolutos):')
TABLA_MATRICES=pd.DataFrame(cm_resultados)
display(TABLA_MATRICES)
print('\nATENCIÓN: resultados exploratorios del TRAIN mediante validacion externa.')
print('HMVT se optimiza solo en CV interna; baselines permanecen fijos.')


## 04. Experimentos en inglés

### EXP-047 · V1, V2, V3, V4, V5, V6a, V6b, V6c, V6d

**Archivo de origen:** `27_Octubre.ipynb, celda 6`. **Referencias duplicadas:** 0. **Sintaxis de código:** Válida.

EXIST2021 ENGLISH | 4 CLASSES | SINGLE GOOGLE COLAB CELL
TRAIN ONLY, 5x3 NESTED STRATIFIED CV; HMVT OPTIMIZATION & ABLATION
Frozen pretrained embeddings as UNTUNED baseline; 8 classifiers
import sys, subprocess, re, warnings, unicodedata
fr

**Ejecución:** celda independiente o variante experimental; revisar archivos de entrada y dependencias antes de ejecutarla. Los resultados originales no se insertan en esta celda.

In [ ]:
import sys, subprocess, re, warnings, unicodedata
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.sparse import csr_matrix, hstack, issparse
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import MultinomialNB, ComplementNB, GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
warnings.filterwarnings('ignore', category=FutureWarning)

SEED = 42
N_OUTER, N_INNER = 5, 3
USE_EMBEDDINGS = True
DO_ABLATION = True

MODEL_NAMES = ['Regresión Logística','3-NN','SVM','Naive Bayes','Red neuronal','XGBoost','Random Forest','Árbol de decisión']
CLASS_NAMES = {0:'No violencia',1:'Misoginia',2:'Acoso',3:'Violencia sexual'}
VERSIONS = {
 'HMVT-V1':(1,0,.35,0), 'HMVT-V2':(1,0,.35,.25),
 'HMVT-V3':(1,.75,.35,.25), 'HMVT-V4':(1,1,.30,.50),
 'HMVT-V5':(1,1.35,.25,.35), 'HMVT-V6a':(1,.75,.10,.10),
 'HMVT-V6b':(1,1.25,.15,.15), 'HMVT-V6c':(1,.55,.30,.30),
 'HMVT-V6d':(1,1.50,.05,.15)
}
EXTRAS = {'Extra-1':(1,.75,.20,.60),'Extra-2':(1,1.25,.15,.65),
          'Extra-3':(1,.50,.50,.50),'Extra-4':(1,.75,.20,1.00),
          'Extra-5':(1,1.00,0,.75)}
ALL_WEIGHTS = {**VERSIONS, **EXTRAS}


def find_training():
    roots = [Path('/content'),Path.cwd(),Path('/mnt/data')]
    def candidates():
        found=[]
        for root in roots:
            if not root.exists(): continue
            for f in root.glob('*.csv'):
                nm=f.name.lower()
                if ('exist2021' in nm and 'train' in nm and 'en' in nm
                    and 'reestructurado' in nm and 'test' not in nm):
                    found.append(f)
        return sorted(set(found),key=lambda p:(p.name != 'EXIST2021_train_en_multiclase_reestructurado.csv',len(p.name)))
    files_found = candidates()
    if not files_found:
        try:
            from google.colab import files
            print('Sube EXIST2021_train_en_multiclase_reestructurado.csv (inglés, 4 clases).')
            files.upload()
        except ImportError: pass
        files_found = candidates()
    for path in files_found:
        try:
            sample = pd.read_csv(path,nrows=10,encoding='utf-8-sig')
            if {'text','tipo_violencia'}.issubset(sample.columns) and ('language' not in sample or sample.language.astype(str).str.lower().isin(['en','english']).all()):
                return path
        except Exception: pass
    raise FileNotFoundError('No se encuentra el CSV de TRAIN inglés reestructurado. Súbelo en la ventana de Colab, no el TRAIN español ni el TEST.')

def deaccent(s):
    return ''.join(ch for ch in unicodedata.normalize('NFKD',str(s).lower().strip()) if not unicodedata.combining(ch))

path=find_training()
df=pd.read_csv(path,encoding='utf-8-sig')
df.columns=[str(c).strip() for c in df.columns]
if not {'text','tipo_violencia'}.issubset(df.columns): raise ValueError('Se requieren columnas text y tipo_violencia')
if 'language' in df.columns:
    langs=df.language.astype(str).str.strip().str.lower()
    df=df.loc[langs.isin(['en','english'])].copy()
if len(df)==0: raise ValueError('No existen registros en inglés; revisa el archivo.')
MAP={'0':0,'1':1,'2':2,'3':3,'no violencia':0,'misoginia':1,'acoso':2,'violencia sexual':3,
     'non-violent':0,'non-sexist':0,'harassment':2,'sexual violence':3}
df=df.dropna(subset=['text','tipo_violencia']).copy()
df['text']=df.text.astype(str).str.replace(r'\s+',' ',regex=True).str.strip()
df=df.loc[df.text.str.len()>0].copy()
df['y']=df.tipo_violencia.map(lambda z:MAP.get(deaccent(z),np.nan))
if df.y.isna().any(): raise ValueError('Etiquetas desconocidas: '+str(df.loc[df.y.isna(),'tipo_violencia'].unique()))
df['y']=df.y.astype(int)

df['_key']=df.text.str.casefold()
conflicts=df.groupby('_key').y.nunique(); bad=set(conflicts[conflicts>1].index)
if bad: print('Excluidos textos con etiquetas contradictorias:',len(bad))
df=df.loc[~df._key.isin(bad)].drop_duplicates('_key').drop(columns='_key').reset_index(drop=True)
X_TEXT=df.text.to_numpy(); Y=df.y.to_numpy()
if set(Y)!={0,1,2,3}: raise ValueError('Se requieren las cuatro clases: '+str(Counter(Y)))
if min(Counter(Y).values())<N_OUTER*N_INNER: raise ValueError('Se necesitan más ejemplos por clase para CV anidada.')
print('TRAIN inglés:',len(df),'| distribución:',dict(sorted(Counter(Y).items())),'| archivo:',path)
print('IMPORTANTE: clase 2 Acoso es una aproximación a objectification de EXIST2021; no son taxonomías equivalentes.')


PATTERNS={
 'insults':['bitch','slut','whore','stupid','idiot','dumb','hoe','cunt'],
 'gender':['woman','women','girl','girls','man','men','female','feminist','feminism'],
 'harassment':['harass','harassment','stalk','stalking','threat','threaten','bully','bullying'],
 'sexual':['sex','sexual','rape','raped','assault','nude','nudes','porn','fuck'],
 'negation':['no','not','never','cannot','dont','doesnt','isnt']
}
REGEX={k:[re.compile(r'(?<!\w)'+re.escape(p)+r'(?!\w)',re.IGNORECASE) for p in v] for k,v in PATTERNS.items()}
def discourse(texts):
    out=[]
    for t in texts:
        s=deaccent(t); tok=re.findall(r'\b\w+\b',s)
        v=[np.log1p(sum(len(p.findall(s)) for p in pats)) for pats in REGEX.values()]
        v.extend([np.log1p(len(tok)),len(set(tok))/max(1,len(tok)),np.log1p(s.count('!')),
                  np.log1p(s.count('?')),int('@' in s),int('#' in s),int('http' in s),
                  sum(ch.isupper() for ch in str(t))/max(1,len(str(t)))])
        out.append(v)
    return np.asarray(out,dtype=np.float32)


class Extractor:
    def fit(self,texts):
        self.word=TfidfVectorizer(max_features=20000,ngram_range=(1,2),min_df=2,sublinear_tf=True,strip_accents='unicode')
        self.char=TfidfVectorizer(analyzer='char_wb',ngram_range=(3,5),max_features=25000,min_df=2,sublinear_tf=True)
        self.ngram=CountVectorizer(ngram_range=(1,2),max_features=10000,min_df=2)
        self.bow=CountVectorizer(max_features=10000,min_df=2)
        w=self.word.fit_transform(texts)
        self.char.fit(texts);self.ngram.fit(texts);self.bow.fit(texts)
        k=min(60,w.shape[0]-1,w.shape[1]-1)
        if k<2: raise ValueError('No hay características suficientes para LSA')
        self.svd=TruncatedSVD(n_components=k,random_state=SEED).fit(w)
        self.lscale=MinMaxScaler().fit(self.svd.transform(w))
        self.dscale=MinMaxScaler().fit(discourse(texts))
        return self
    def transform(self,texts):
        w=self.word.transform(texts).astype(np.float32)
        c=self.char.transform(texts).astype(np.float32)
        l=csr_matrix(np.maximum(0,self.lscale.transform(self.svd.transform(w))).astype(np.float32))
        d=csr_matrix(np.maximum(0,self.dscale.transform(discourse(texts))).astype(np.float32))
        return (w,c,l,d)

def fuse(blocks,weights):
    
    
    return hstack([b.multiply(float(weight)) for b,weight in zip(blocks,weights)],format='csr',dtype=np.float32)


def rebalance(X,y,strategy,seed):
    if strategy not in ['ros','ros_partial','minority_boost']: return X,y
    rng=np.random.default_rng(seed); counts=Counter(y); max_n=max(counts.values()); chosen=[]
    for cls in sorted(counts):
        ids=np.flatnonzero(y==cls)
        if strategy=='ros': target=max_n
        elif strategy=='ros_partial': target=max(len(ids),int(.65*max_n))
        else: target=max(len(ids),int(.8*max_n)) if cls in [2,3] else len(ids)
        chosen.extend(rng.choice(ids,size=target,replace=target>len(ids)).tolist())
    chosen=np.asarray(chosen,dtype=int);rng.shuffle(chosen)
    return X[chosen],y[chosen]

def sample_weights(y,strategy):
    if strategy not in ('balanced','sqrt_balanced'): return None
    counts=Counter(y); w={c:len(y)/(len(counts)*n) for c,n in counts.items()}
    if strategy=='sqrt_balanced': w={c:np.sqrt(z) for c,z in w.items()}
    return np.array([w[c] for c in y],dtype=np.float32)

try:
    from xgboost import XGBClassifier
except ImportError:
    subprocess.check_call([sys.executable,'-m','pip','install','-q','xgboost'])
    from xgboost import XGBClassifier

def make_model(name,cfg=None):
    cfg=cfg or {}
    if name=='Regresión Logística': return LogisticRegression(C=cfg.get('C',1),max_iter=2500,random_state=SEED)
    if name=='3-NN': return KNeighborsClassifier(n_neighbors=cfg.get('k',3),weights=cfg.get('weights','uniform'),metric='cosine',algorithm='brute')
    if name=='SVM': return LinearSVC(C=cfg.get('C',1),max_iter=7000,random_state=SEED)
    if name=='Naive Bayes':
        return ComplementNB(alpha=cfg.get('alpha',1)) if cfg.get('kind')=='ComplementNB' else MultinomialNB(alpha=cfg.get('alpha',1))
    if name=='Red neuronal': return MLPClassifier(hidden_layer_sizes=(cfg.get('hidden',64),),alpha=cfg.get('alpha',.001),early_stopping=True,max_iter=160,random_state=SEED)
    if name=='XGBoost': return XGBClassifier(objective='multi:softprob',num_class=4,n_estimators=cfg.get('trees',140),max_depth=cfg.get('depth',3),learning_rate=cfg.get('lr',.07),subsample=.9,colsample_bytree=.9,tree_method='hist',eval_metric='mlogloss',random_state=SEED,n_jobs=2)
    if name=='Random Forest': return RandomForestClassifier(n_estimators=cfg.get('trees',140),min_samples_leaf=cfg.get('leaf',1),max_depth=cfg.get('depth',None),random_state=SEED,n_jobs=-1)
    if name=='Árbol de decisión': return DecisionTreeClassifier(max_depth=cfg.get('depth',18),min_samples_leaf=cfg.get('leaf',2),random_state=SEED)
    raise ValueError(name)

def fit_predict(name,Xtr,ytr,Xev,cfg=None,strategy='none',seed=SEED):
    if strategy in ['ros','ros_partial','minority_boost']: Xtr,ytr=rebalance(Xtr,ytr,strategy,seed)
    model=GaussianNB() if name=='Naive Bayes' and not issparse(Xtr) else make_model(name,cfg)
    sw=sample_weights(ytr,strategy)
    if sw is not None and name!='3-NN':
        if name=='Red neuronal':
            
            rng=np.random.default_rng(seed); p=sw/sw.sum(); ind=rng.choice(len(ytr),len(ytr),replace=True,p=p)
            model.fit(Xtr[ind],ytr[ind])
        else: model.fit(Xtr,ytr,sample_weight=sw)
    else: model.fit(Xtr,ytr)
    return model.predict(Xev).astype(int)

def metrics(y,p):
    pr,rc,f1,_=precision_recall_fscore_support(y,p,labels=[0,1,2,3],average='macro',zero_division=0)
    _,rec,f1_per,_=precision_recall_fscore_support(y,p,labels=[0,1,2,3],average=None,zero_division=0)
    return {'Precisión':pr,'Recall':rc,'F1-Score':f1,'Accuracy':accuracy_score(y,p),
            'Recall Acoso':rec[2],'Recall Violencia sexual':rec[3],
            'F1 Acoso':f1_per[2],'F1 Violencia sexual':f1_per[3]}


def candidate_grid(name):
    grids={
      'Regresión Logística': [('HMVT-V3',{'C':1},'balanced'),('HMVT-V4',{'C':5},'none'),('HMVT-V6a',{'C':.1},'balanced'),('Extra-2',{'C':1},'balanced'),('Extra-5',{'C':.5},'ros_partial')],
      '3-NN': [('HMVT-V5',{'k':5,'weights':'distance'},'minority_boost'),('HMVT-V6d',{'k':3,'weights':'distance'},'ros_partial'),('HMVT-V4',{'k':5,'weights':'distance'},'ros'),('Extra-1',{'k':7,'weights':'distance'},'ros_partial')],
      'SVM': [('HMVT-V3',{'C':1},'balanced'),('HMVT-V4',{'C':.5},'none'),('HMVT-V6c',{'C':.5},'balanced'),('Extra-2',{'C':.5},'minority_boost'),('Extra-5',{'C':.5},'minority_boost')],
      'Naive Bayes': [('HMVT-V6a',{'kind':'ComplementNB','alpha':.1},'none'),('Extra-4',{'kind':'ComplementNB','alpha':.1},'none'),('Extra-4',{'kind':'ComplementNB','alpha':1},'ros_partial'),('HMVT-V4',{'kind':'ComplementNB','alpha':.5},'balanced'),('Extra-2',{'kind':'MultinomialNB','alpha':.1},'ros')],
      'Red neuronal': [('Extra-1',{'hidden':128,'alpha':.01},'ros'),('HMVT-V4',{'hidden':128,'alpha':.001},'ros_partial'),('HMVT-V6a',{'hidden':64,'alpha':.01},'minority_boost'),('HMVT-V3',{'hidden':64,'alpha':.001},'none')],
      'XGBoost': [('HMVT-V3',{'depth':5,'trees':180,'lr':.06},'ros_partial'),('HMVT-V6b',{'depth':3,'trees':180,'lr':.06},'ros'),('Extra-3',{'depth':3,'trees':180,'lr':.06},'sqrt_balanced'),('Extra-1',{'depth':4,'trees':180,'lr':.05},'minority_boost')],
      'Random Forest': [('HMVT-V6d',{'trees':200,'leaf':1},'ros_partial'),('HMVT-V6b',{'trees':200,'leaf':1},'ros'),('Extra-1',{'trees':200,'leaf':2},'balanced'),('HMVT-V3',{'trees':200,'leaf':1},'minority_boost')],
      'Árbol de decisión': [('Extra-1',{'depth':24,'leaf':4},'ros'),('HMVT-V6d',{'depth':12,'leaf':2},'ros_partial'),('HMVT-V6a',{'depth':24,'leaf':4},'none'),('HMVT-V3',{'depth':18,'leaf':2},'balanced')]
    }
    return grids[name]


EMB=None
if USE_EMBEDDINGS:
    try: from sentence_transformers import SentenceTransformer
    except ImportError:
        subprocess.check_call([sys.executable,'-m','pip','install','-q','sentence-transformers'])
        from sentence_transformers import SentenceTransformer
    print('Codificando embeddings preentrenados (sin ajustar)...',flush=True)
    enc=SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')
    enc.eval()
    EMB=enc.encode(X_TEXT.tolist(),batch_size=64,show_progress_bar=True,convert_to_numpy=True,normalize_embeddings=False).astype(np.float32)
    del enc


RESULTS=[];SELECTED=[];ABLATIONS=[];ERRORS=[]
BASE=['TF-IDF','N-gramas','BoW']+(['Embeddings'] if USE_EMBEDDINGS else [])
ORDER=BASE+list(VERSIONS)+['HMVT-Optimizado']
outer=StratifiedKFold(n_splits=N_OUTER,shuffle=True,random_state=SEED)
for fold,(itr,iev) in enumerate(outer.split(X_TEXT,Y),1):
    print(f'\n========== FOLD EXTERNO {fold}/{N_OUTER} ==========',flush=True)
    xt,xe=X_TEXT[itr],X_TEXT[iev]; yt,ye=Y[itr],Y[iev]
    ext=Extractor().fit(xt);bt,be=ext.transform(xt),ext.transform(xe)
    reps={
       'TF-IDF':(bt[0],be[0]),
       'N-gramas':(ext.ngram.transform(xt),ext.ngram.transform(xe)),
       'BoW':(ext.bow.transform(xt),ext.bow.transform(xe))
    }
    if USE_EMBEDDINGS: reps['Embeddings']=(EMB[itr],EMB[iev])
    for v,w in VERSIONS.items():reps[v]=(fuse(bt,w),fuse(be,w))
    
    for rep,(xa,xv) in reps.items():
        print('Evaluando:',rep,flush=True)
        for name in MODEL_NAMES:
            try:
                pred=fit_predict(name,xa,yt,xv,seed=SEED+fold)
                RESULTS.append({'Fold':fold,'Representación':rep,'Modelo de clasificación':name,**metrics(ye,pred)})
            except Exception as e:
                ERRORS.append((fold,rep,name,str(e)))
                RESULTS.append({'Fold':fold,'Representación':rep,'Modelo de clasificación':name,**{k:np.nan for k in metrics(ye,ye)}})
    
    inner=[]
    cv=StratifiedKFold(n_splits=N_INNER,shuffle=True,random_state=SEED+fold)
    for ia,iv in cv.split(xt,yt):
        ex=Extractor().fit(xt[ia]);inner.append((ex.transform(xt[ia]),ex.transform(xt[iv]),yt[ia],yt[iv]))
    for name in MODEL_NAMES:
        best=None
        for version,cfg,balance in candidate_grid(name):
            mvals=[]
            for j,(ba,bv,ya,yv) in enumerate(inner):
                try:
                    pred=fit_predict(name,fuse(ba,ALL_WEIGHTS[version]),ya,fuse(bv,ALL_WEIGHTS[version]),cfg,balance,SEED+fold*100+j)
                    mvals.append(metrics(yv,pred))
                except Exception as e:
                    ERRORS.append((fold,'selección '+version,name,str(e)));mvals=[];break
            if len(mvals)!=N_INNER: continue
            f1=float(np.mean([m['F1-Score'] for m in mvals]))
            minority=float(np.mean([(m['Recall Acoso']+m['Recall Violencia sexual'])/2 for m in mvals]))
            score=(f1,minority)
            if best is None or score>best['score']:
                best=dict(score=score,version=version,cfg=cfg,balance=balance)
        if best is None:raise RuntimeError(f'No se encontró configuración válida para {name} en fold {fold}: {ERRORS[-5:]}')
        weights=ALL_WEIGHTS[best['version']]
        pred=fit_predict(name,fuse(bt,weights),yt,fuse(be,weights),best['cfg'],best['balance'],SEED+fold)
        m=metrics(ye,pred)
        RESULTS.append({'Fold':fold,'Representación':'HMVT-Optimizado','Modelo de clasificación':name,**m})
        SELECTED.append({'Fold':fold,'Clasificador':name,'Versión':best['version'],'Balanceo':best['balance'],'Configuración':str(best['cfg']),
                         'F1 validación interna':best['score'][0],'Recall minoritario interno':best['score'][1]})
        print('HMVT optimizado:',name,best['version'],'F1 interno:',round(best['score'][0],4),flush=True)
        if DO_ABLATION:
            w=list(weights)
            variants={'HMVT completo':tuple(w),'Sin caracteres':(w[0],0,w[2],w[3]),
                      'Sin LSA':(w[0],w[1],0,w[3]),'Sin discursivas':(w[0],w[1],w[2],0),
                      'Solo TF-IDF':(w[0],0,0,0)}
            for label,ww in variants.items():
                am=m if label=='HMVT completo' else metrics(ye,fit_predict(name,fuse(bt,ww),yt,fuse(be,ww),best['cfg'],best['balance'],SEED+fold))
                ABLATIONS.append({'Fold':fold,'Clasificador':name,'Ablación':label,**am})


METRIC_COLUMNS=['Precisión','Recall','F1-Score','Accuracy','Recall Acoso','Recall Violencia sexual','F1 Acoso','F1 Violencia sexual']
def summarize(data,group_columns):
    rows=[]
    for keys,g in data.groupby(group_columns,sort=False):
        if not isinstance(keys,tuple):keys=(keys,)
        row=dict(zip(group_columns,keys))
        for col in METRIC_COLUMNS:
            vals=g[col].dropna().to_numpy(dtype=float)
            if len(vals):
                mu=float(vals.mean());sd=float(vals.std(ddof=1)) if len(vals)>1 else 0
                row[col]=f'{mu:.4f} ± {sd:.4f}';row[col+'_media']=mu
            else:row[col]='ERROR';row[col+'_media']=np.nan
        rows.append(row)
    return pd.DataFrame(rows)

raw=pd.DataFrame(RESULTS)
table=summarize(raw,['Representación','Modelo de clasificación'])
table['Representación']=pd.Categorical(table['Representación'],categories=ORDER,ordered=True)
table['Modelo de clasificación']=pd.Categorical(table['Modelo de clasificación'],categories=MODEL_NAMES,ordered=True)
table=table.sort_values(['Representación','Modelo de clasificación']).reset_index(drop=True)
print('\n=== TABLA MAESTRA (TRAIN ENGLISH, CV ANIDADA) ===')
display(table[['Representación','Modelo de clasificación','Precisión','Recall','F1-Score','Accuracy']])
print('\n=== HMVT: CLASES MINORITARIAS ===')
display(table[table['Representación'].astype(str).str.startswith('HMVT')][['Representación','Modelo de clasificación','F1-Score','Recall Acoso','Recall Violencia sexual','F1 Acoso','F1 Violencia sexual']].reset_index(drop=True))
print('\n=== HMVT: CONFIGURACIONES SELECCIONADAS SOLO EN CV INTERNA ===')
display(pd.DataFrame(SELECTED))
if DO_ABLATION:
    abd=pd.DataFrame(ABLATIONS);ab=summarize(abd,['Clasificador','Ablación'])
    print('\n=== ABLACIÓN HMVT ===')
    display(ab[['Clasificador','Ablación','F1-Score','Accuracy','Recall Acoso','Recall Violencia sexual']])
    delta=[]
    for name in MODEL_NAMES:
        gg=abd[abd.Clasificador==name];full=gg[gg['Ablación']=='HMVT completo'].set_index('Fold')
        for variant in ['Sin caracteres','Sin LSA','Sin discursivas','Solo TF-IDF']:
            other=gg[gg['Ablación']==variant].set_index('Fold')
            delta.append({'Clasificador':name,'Componente retirado':variant,
                          'Δ F1 macro':(full['F1-Score']-other['F1-Score']).mean(),
                          'Δ Recall Acoso':(full['Recall Acoso']-other['Recall Acoso']).mean(),
                          'Δ Recall Violencia sexual':(full['Recall Violencia sexual']-other['Recall Violencia sexual']).mean()})
    print('\n=== CONTRIBUCION DE COMPONENTES (positivo = aporta) ===')
    display(pd.DataFrame(delta).style.format({'Δ F1 macro':'{:+.4f}','Δ Recall Acoso':'{:+.4f}','Δ Recall Violencia sexual':'{:+.4f}'}))
comparisons=[]
for name in MODEL_NAMES:
    optimized=raw[(raw['Representación']=='HMVT-Optimizado')&(raw['Modelo de clasificación']==name)].set_index('Fold')
    for base in BASE:
        baseline=raw[(raw['Representación']==base)&(raw['Modelo de clasificación']==name)].set_index('Fold')
        difference=(optimized['F1-Score']-baseline['F1-Score']).dropna()
        if len(difference):comparisons.append({'Clasificador':name,'Comparación':f'HMVT vs {base}',
                  'Δ F1 macro':difference.mean(),'Desv. Δ F1':difference.std(ddof=1),
                  'Folds HMVT superior':int((difference>0).sum()),'Folds comparados':len(difference)})
print('\n=== COMPARACIÓN PAREADA POR FOLD EXTERNO ===')
display(pd.DataFrame(comparisons).style.format({'Δ F1 macro':'{:+.4f}','Desv. Δ F1':'{:.4f}'}))
if ERRORS:
    print('\nATENCIÓN: hubo errores en estas combinaciones (no se inventaron métricas):')
    for e in ERRORS: print(e)
print('\nFIN: solo TRAIN inglés. TEST no utilizado. Embeddings congelados, sin tuning. Todo F1/Recall/Precisión macro; Accuracy global.')
print('Advertencia metodológica: mapping original objectification -> Acoso es aproximado; comprobar taxonomía antes de reportar en artículo.')


### EXP-048 · V1

**Archivo de origen:** `Reportes_tesis.ipynb, celda 6`. **Referencias duplicadas:** 1. **Sintaxis de código:** Válida.

SCRIPT DOCTORAL MAESTRO: ABLACIÓN MASIVA CON HMT-V1 OPTIMIZADO + MCNEMAR (INGLÉS - TSV)
import os
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition

**Ejecución:** celda independiente o variante experimental; revisar archivos de entrada y dependencias antes de ejecutarla. Los resultados originales no se insertan en esta celda.

In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB, GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
from scipy.sparse import hstack, csr_matrix
from statsmodels.stats.contingency_tables import mcnemar

print("1. Buscando y cargando el archivo .tsv de forma automática...")
nombre_archivo = 'EXIST2021_test_labeled.tsv'
ruta_encontrada = None


for root, dirs, files in os.walk('.'):
    if nombre_archivo in files:
        ruta_encontrada = os.path.join(root, nombre_archivo)
        break

if not ruta_encontrada and os.path.exists(f'/content/{nombre_archivo}'):
    ruta_encontrada = f'/content/{nombre_archivo}'

if ruta_encontrada:
    print(f"¡Archivo localizado en: {ruta_encontrada}!")
    df = pd.read_csv(ruta_encontrada, sep='\t').dropna(subset=['text', 'task2', 'language'])
else:
    raise FileNotFoundError(f"No se pudo encontrar '{nombre_archivo}' en el directorio actual. Asegúrate de haberlo subido al entorno.")


df = df[df['language'].str.lower() == 'en'].copy()
print(f"Registros en inglés filtrados para el experimento: {len(df)}")

texts = df['text'].astype(str).values


y_task1 = np.where(df['task2'].values == 'non-sexist', 0, 1)

X_train_txt, X_test_txt, y_train, y_test = train_test_split(
    texts, y_task1, test_size=0.2, random_state=42, stratify=y_task1
)




print("Extrayendo representaciones base y modelo optimizado HMT-V1 para inglés...")


tfidf = TfidfVectorizer(max_features=15000, ngram_range=(1, 3), sublinear_tf=True, min_df=2)
X_tfidf_tr = tfidf.fit_transform(X_train_txt)
X_tfidf_te = tfidf.transform(X_test_txt)


count_vec = CountVectorizer(ngram_range=(1, 2), max_features=5000, min_df=2)
X_ngram_tr = count_vec.fit_transform(X_train_txt)
X_ngram_te = count_vec.transform(X_test_txt)


def extraer_lexico_en(texto):
    t = texto.lower()
    insultos = ['bitch', 'slut', 'fat', 'imbecile', 'stupid', 'disgusting', 'useless', 'idiot', 'whore', 'retard', 'ass', 'asshole']
    imperativos = ['shut up', 'kitchen', 'clean', 'go', 'must', 'should', 'get back', 'back to', 'serve']
    return [
        sum(1 for p in insultos if p in t),
        sum(1 for p in imperativos if p in t),
        len(texto) / 100.0,
        sum(1 for c in texto if c.isupper()) / max(1, len(texto))
    ]

lex_tr = np.array([extraer_lexico_en(t) for t in X_train_txt], dtype=float)
lex_te = np.array([extraer_lexico_en(t) for t in X_test_txt], dtype=float)
scaler_lex = StandardScaler()
X_lex_tr = scaler_lex.fit_transform(lex_tr)
X_lex_te = scaler_lex.transform(lex_te)


svd = TruncatedSVD(n_components=150, random_state=42)
X_lsa_tr = svd.fit_transform(X_tfidf_tr)
X_lsa_te = svd.transform(X_tfidf_te)
scaler_lsa = StandardScaler()
X_lsa_tr_sc = scaler_lsa.fit_transform(X_lsa_tr)
X_lsa_te_sc = scaler_lsa.transform(X_lsa_te)


def extraer_caracteristicas_discursivas_en(texto):
    t = texto.lower()
    insultos = ['bitch', 'slut', 'fat', 'imbecile', 'stupid', 'disgusting', 'useless', 'idiot', 'whore', 'retard', 'ass', 'asshole']
    imperativos = ['shut up', 'kitchen', 'clean', 'go', 'must', 'should', 'get back', 'back to', 'serve']
    desprecio = ['trash', 'garbage', 'ridiculous', 'fake', 'pathetic', 'worthless', 'mediocre', 'scum']
    estereotipos = ['women', 'feminazi', 'wife', 'mother', 'girlfriend', 'emotional', 'weak', 'females']

    n_insultos = sum(6.0 for p in insultos if p in t)
    n_imperativos = sum(5.0 for p in imperativos if p in t)
    n_desprecio = sum(6.0 for p in desprecio if p in t)
    n_estereotipos = sum(4.0 for p in estereotipos if p in t)

    longitud = len(texto) / 100.0
    mayusculas = sum(1 for c in texto if c.isupper()) / max(1, len(texto))
    exclamaciones = texto.count('!')
    interrogaciones = texto.count('?')

    return [n_insultos, n_imperativos, n_desprecio, n_estereotipos, longitud, mayusculas, exclamaciones, interrogaciones]

s_train = np.array([extraer_caracteristicas_discursivas_en(t) for t in X_train_txt], dtype=float)
s_test = np.array([extraer_caracteristicas_discursivas_en(t) for t in X_test_txt], dtype=float)
scaler_s = StandardScaler()
X_s_tr_sc = scaler_s.fit_transform(s_train)
X_s_te_sc = scaler_s.transform(s_test)


HMT1_tr = hstack([X_tfidf_tr, csr_matrix(X_lsa_tr_sc * 6.0), csr_matrix(X_s_tr_sc * 10.0)])
HMT1_te = hstack([X_tfidf_te, csr_matrix(X_lsa_te_sc * 6.0), csr_matrix(X_s_te_sc * 10.0)])


representations = {
    "1. TF-IDF Puro": (X_tfidf_tr, X_tfidf_te, False),
    "2. N-Grams Puros": (X_ngram_tr, X_ngram_te, False),
    "3. Lexical Features": (X_lex_tr, X_lex_te, True),
    "4. Latent Topics (LSA)": (X_lsa_tr_sc, X_lsa_te_sc, True),
    "5. HMT-V1 (Optimizado)": (HMT1_tr, HMT1_te, True)
}


models = {
    "LogReg": LogisticRegression(max_iter=1500, class_weight='balanced', random_state=42),
    "3-NN": KNeighborsClassifier(n_neighbors=5, weights='distance'),
    "SVM": SVC(kernel='linear', C=1.0, class_weight='balanced', random_state=42),
    "NaiveBayes": "auto",
    "Red Neuronal": MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=500, random_state=42),
    "XGBoost": XGBClassifier(n_estimators=150, learning_rate=0.05, max_depth=5, eval_metric='logloss', random_state=42),
    "RandomForest": RandomForestClassifier(n_estimators=150, class_weight='balanced', random_state=42),
    "DecisionTree": DecisionTreeClassifier(class_weight='balanced', random_state=42)
}




print("\nEjecutando evaluación exhaustiva para textos en Inglés...")
results = []
predicciones_almacenadas = {}

for rep_name, (X_tr, X_te, is_dense) in representations.items():
    if is_dense:
        if hasattr(X_tr, "toarray"): X_tr = X_tr.toarray()
        if hasattr(X_te, "toarray"): X_te = X_te.toarray()

    for model_name, model_obj in models.items():
        if model_name == "NaiveBayes":
            clf = GaussianNB() if is_dense else MultinomialNB()
        else:
            clf = model_obj

        clf.fit(X_tr, y_train)
        preds = clf.predict(X_te)

        predicciones_almacenadas[(rep_name, model_name)] = preds

        p, r, f1, _ = precision_recall_fscore_support(y_test, preds, average='binary', zero_division=0)
        acc = accuracy_score(y_test, preds)

        results.append({
            "Representación": rep_name,
            "Modelo": model_name,
            "Precision": round(p, 2),
            "Recall": round(r, 2),
            "F1-score": round(f1, 2),
            "Accuracy": round(acc, 2)
        })

df_large = pd.DataFrame(results)

print("\n==========================================================================")
print("=== GRAN TABLA MAESTRA DE ABLACIÓN (INGLÉS - EXIST2021) ===================")
print("==========================================================================")
print(df_large.to_string(index=False))




print("\n==================================================================")
print("=== PRUEBA DE MCNEMAR: HMT-V1 OPTIMIZADO VS. BASES (INGLÉS) ======")
print("==================================================================")

mcnemar_comparativas = []

for model_name in models.keys():
    preds_hmt = predicciones_almacenadas[("5. HMT-V1 (Optimizado)", model_name)]
    correct_hmt = (preds_hmt == y_test)

    baselines_a_comparar = ["1. TF-IDF Puro", "2. N-Grams Puros", "3. Lexical Features", "4. Latent Topics (LSA)"]

    for base_name in baselines_a_comparar:
        preds_base = predicciones_almacenadas[(base_name, model_name)]
        correct_base = (preds_base == y_test)

        b_acierta_hmt_falla = np.sum(correct_base & ~correct_hmt)
        b_falla_hmt_acierta = np.sum(~correct_base & correct_hmt)
        ambos_aciertan = np.sum(correct_base & correct_hmt)
        ambos_fallan = np.sum(~correct_base & ~correct_hmt)

        tabla_contingencia = [[ambos_aciertan, b_acierta_hmt_falla],
                              [b_falla_hmt_acierta, ambos_fallan]]

        res = mcnemar(tabla_contingencia, exact=False, correction=True)
        is_sig = res.pvalue < 0.05
        sig_text = "Sí (p < 0.05)" if is_sig else "No"

        mcnemar_comparativas.append({
            "Clasificador": model_name,
            "Comparativa": f"HMT-V1 Opt vs {base_name.split('.')[1].strip()}",
            "Estadístico": round(res.statistic, 4),
            "P-valor": f"{res.pvalue:.5e}",
            "Significativo (p < 0.05)": sig_text
        })

df_mcnemar = pd.DataFrame(mcnemar_comparativas)
print(df_mcnemar.to_string(index=False))


## 05. Geolingüística y análisis espacial

### EXP-049 · V1, V2, V3, V4

**Archivo de origen:** `27_Octubre.ipynb, celda 8`. **Referencias duplicadas:** 1. **Sintaxis de código:** Válida.

CLASIFICACION DOCTORAL BINARIA + MULTICLASE / GEO / HMVT
UNA SOLA CELDA COLAB. CARGA CSV SIEMPRE EN GOOGLE COLAB.
import sys, os, re, json, time, warnings, subprocess, importlib.util, zipfile, unicodedata
from pathlib import Path
from io im

**Ejecución:** celda independiente o variante experimental; revisar archivos de entrada y dependencias antes de ejecutarla. Los resultados originales no se insertan en esta celda.

In [ ]:
import sys, os, re, json, time, warnings, subprocess, importlib.util, zipfile, unicodedata
from pathlib import Path
from io import BytesIO
warnings.filterwarnings('ignore', category=FutureWarning)

SEMILLA=42
TEST_SIZE=.20
ARCHIVO_LOCAL='data_hmvt_con_estado.csv'
SALIDA=Path('/content/resultados_hmvt_80_20' if Path('/content').exists() else 'resultados_hmvt_80_20')
SALIDA.mkdir(parents=True,exist_ok=True)
USAR_GEO=True
HACER_MAPAS=True
MOSTRAR_GRAFICAS=True
EMBEDDING_MODELO='sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'

N_ITER_HMVT=2
N_SPLITS_CV=3
MODELOS=['RL','SVM','RF','MLP','NB','KNN','XGB','ARBOL']
BASES=['TFIDF','NGRAMAS','BOW','EMBEDDINGS']
HMVT=['HMVT-V1','HMVT-V2','HMVT-V3','HMVT-V4']


def asegurar(nombre,pkg=None):
    if importlib.util.find_spec(nombre) is None:
        subprocess.check_call([sys.executable,'-m','pip','-q','install',pkg or nombre])
for modulo,pkg in [('numpy',None),('pandas',None),('scipy',None),('sklearn','scikit-learn'),
                   ('matplotlib',None),('seaborn',None),('sentence_transformers','sentence-transformers'),
                   ('xgboost',None),('geopandas',None),('requests',None)]:
    if modulo=='geopandas' and not HACER_MAPAS: continue
    asegurar(modulo,pkg)
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from scipy import sparse
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.metrics import accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.base import clone
sns.set_theme(style='whitegrid')
np.random.seed(SEMILLA)


if 'google.colab' in sys.modules:
    from google.colab import files
    print('SUBE EL DATASET CSV. Debe incluir tweet, clase, violencia_binaria, region, estado y regionalismo.')
    subidos=files.upload()
    if len(subidos)!=1: raise ValueError('Selecciona exactamente un archivo CSV')
    nombre_archivo,contenido=next(iter(subidos.items()))
    if not nombre_archivo.lower().endswith('.csv'): raise ValueError('Se requiere CSV')
    df=pd.read_csv(BytesIO(contenido),encoding='utf-8-sig')
else:
    nombre_archivo=ARCHIVO_LOCAL
    df=pd.read_csv(nombre_archivo,encoding='utf-8-sig')
requeridas={'tweet','clase','violencia_binaria','region','estado','regionalismo'}
if faltantes:=requeridas-set(df.columns): raise ValueError('Faltan columnas: '+str(faltantes))
df=df.dropna(subset=['tweet','clase','violencia_binaria']).copy()
for c in ['tweet','clase','region','estado','regionalismo']:
    df[c]=df[c].fillna('SIN_DATO').astype(str).str.strip()
df['violencia_binaria']=pd.to_numeric(df.violencia_binaria,errors='raise').astype(int)
if not set(df.violencia_binaria.unique()).issubset({0,1}):raise ValueError('violencia_binaria debe ser 0/1')

def normalizar(s):
    s=str(s).lower().strip()
    s=re.sub(r'https?://\S+|www\.\S+',' URL ',s)
    s=re.sub(r'@\w+',' USUARIO ',s)
    return re.sub(r'\s+',' ',s)
df['texto_norm']=df.tweet.map(normalizar)
df=df[df.texto_norm.str.len()>0].reset_index(drop=True)
if (df.groupby('texto_norm').clase.nunique()>1).any():
    raise ValueError('Existen textos iguales con etiquetas multiclase contradictorias; revisar antes de evaluar')
if (df.groupby('texto_norm').violencia_binaria.nunique()>1).any():
    raise ValueError('Existen textos iguales con etiquetas binarias contradictorias')
if df.groupby('clase').violencia_binaria.nunique().max()>1:
    raise ValueError('Inconsistencia entre clase y etiqueta binaria')



print('\nDIAGNOSTICO DEL CONJUNTO ORIGINAL')
print('Filas:',len(df),'| Textos unicos:',df.texto_norm.nunique(),'| Filas duplicadas:',df.duplicated().sum())
for c in ['estado','region','regionalismo']:
    n=(df.groupby('texto_norm')[c].nunique()>1).sum()
    print('Textos asociados con multiples',c,':',n)


corpus=df.drop_duplicates('texto_norm').copy()
geo_cols=['region','estado','regionalismo']
for c in geo_cols:
    nvar=df.groupby('texto_norm')[c].nunique()
    corpus.loc[corpus.texto_norm.map(nvar)>1,c]='SIN_DATO'
corpus=corpus.reset_index(drop=True)
idx_tr,idx_te=train_test_split(np.arange(len(corpus)),test_size=TEST_SIZE,
                                random_state=SEMILLA,stratify=corpus['clase'])
tr=corpus.iloc[idx_tr].reset_index(drop=True)
te=corpus.iloc[idx_te].reset_index(drop=True)
assert not set(tr.texto_norm).intersection(set(te.texto_norm))
assert len(tr)+len(te)==len(corpus)
print('\nDIVISION 80/20 SOBRE TEXTOS UNICOS:')
print('TRAIN',len(tr),'(',round(100*len(tr)/len(corpus),1),'%)','TEST',len(te),
      '(',round(100*len(te)/len(corpus),1),'%)')
print('TRAIN por clase:',tr.clase.value_counts().to_dict())
print('TEST por clase:',te.clase.value_counts().to_dict())
print('IMPORTANTISIMO: 80/20 corresponde a textos unicos, no a filas repetidas.')
print('REGIONALISMO y GEO con anotaciones contradictorias quedan como SIN_DATO para modelado.')
tr.to_csv(SALIDA/'train_textos_unicos.csv',index=False)
te.to_csv(SALIDA/'test_textos_unicos.csv',index=False)
textos_tr=tr.texto_norm.tolist(); textos_te=te.texto_norm.tolist()


rep={}
def registrar(k,a,b):
    rep[k]=(sparse.csr_matrix(a,dtype=np.float32),sparse.csr_matrix(b,dtype=np.float32))
    print(f'  {k:18s} | {rep[k][0].shape[1]} atributos')
def vectorizar(k,obj): registrar(k,obj.fit_transform(textos_tr),obj.transform(textos_te))
print('\nCREANDO REPRESENTACIONES (fit SOLO TRAIN):')
vectorizar('TFIDF',TfidfVectorizer(max_features=12000,ngram_range=(1,2),strip_accents='unicode'))
vectorizar('NGRAMAS',TfidfVectorizer(analyzer='char',ngram_range=(3,5),max_features=14000))
vectorizar('BOW',CountVectorizer(max_features=12000,strip_accents='unicode'))
from sentence_transformers import SentenceTransformer
st=SentenceTransformer(EMBEDDING_MODELO)
emb_tr=st.encode(textos_tr,batch_size=32,show_progress_bar=True,normalize_embeddings=True)
emb_te=st.encode(textos_te,batch_size=32,show_progress_bar=True,normalize_embeddings=True)

esc=MinMaxScaler().fit(emb_tr)
registrar('EMBEDDINGS',esc.transform(emb_tr),np.maximum(0,esc.transform(emb_te)))





bloques={
 'HMVT-V1':[('TFIDF',1.),('NGRAMAS',1.)],
 'HMVT-V2':[('TFIDF',1.),('BOW',.3),('NGRAMAS',.7)],
 'HMVT-V3':[('TFIDF',1.),('NGRAMAS',.7),('EMBEDDINGS',2.)],
 'HMVT-V4':[('TFIDF',1.),('NGRAMAS',.7),('BOW',.3),('EMBEDDINGS',2.)]
}
for nombre,partes in bloques.items():
    registrar(nombre,*[sparse.hstack([rep[k][i]*peso for k,peso in partes],format='csr') for i in range(2)])



if USAR_GEO:
    try: ge=OneHotEncoder(handle_unknown='ignore',sparse_output=True,dtype=np.float32)
    except TypeError: ge=OneHotEncoder(handle_unknown='ignore',sparse=True,dtype=np.float32)
    gt=ge.fit_transform(tr[geo_cols]); gv=ge.transform(te[geo_cols]);
    for nombre in BASES+HMVT:
        a,b=rep[nombre]
        registrar(nombre+'+GEO',sparse.hstack([a,gt],format='csr'),
                  sparse.hstack([b,gv],format='csr'))



def crear_modelo(nombre,nclases):
    if nombre=='RL':return LogisticRegression(max_iter=1200,random_state=SEMILLA)
    if nombre=='SVM':return LinearSVC(max_iter=3000,random_state=SEMILLA)
    if nombre=='RF':return RandomForestClassifier(n_estimators=150,random_state=SEMILLA,n_jobs=-1)
    if nombre=='MLP':return MLPClassifier(hidden_layer_sizes=(64,),early_stopping=True,max_iter=200,random_state=SEMILLA)
    if nombre=='NB':return MultinomialNB()
    if nombre=='KNN':return KNeighborsClassifier(n_neighbors=3)
    if nombre=='XGB':
        from xgboost import XGBClassifier
        return XGBClassifier(n_estimators=130,max_depth=4,learning_rate=.1,random_state=SEMILLA,
                             tree_method='hist',n_jobs=2,eval_metric='logloss' if nclases==2 else 'mlogloss')
    if nombre=='ARBOL':return DecisionTreeClassifier(random_state=SEMILLA)
    raise ValueError(nombre)

rejillas={
 'RL':{'C':[.1,1.]},'SVM':{'C':[.1,1.]},
 'RF':{'max_depth':[6,None]},'MLP':{'alpha':[.0001,.01]},
 'NB':{'alpha':[.5,1.]},'KNN':{'n_neighbors':[3,5]},
 'XGB':{'max_depth':[2,4]},'ARBOL':{'max_depth':[4,10]}
}


metricas=[]; por_clase=[]; predicciones={}; detalles_cv=[]
tareas={'BINARIA':('violencia_binaria',[0,1]),'MULTICLASE':('clase',sorted(corpus.clase.unique()))}
for tarea,(col,etiquetas) in tareas.items():
    if tarea=='MULTICLASE':
        a={k:i for i,k in enumerate(etiquetas)}
        ytr=tr[col].map(a).to_numpy();yte=te[col].map(a).to_numpy()
    else:
        ytr=tr[col].to_numpy(dtype=int);yte=te[col].to_numpy(dtype=int)
    etiquetas_id=np.arange(len(etiquetas))
    for nombre,(xtr,xte) in rep.items():
        optimizar=nombre.startswith('HMVT-')
        for alg in MODELOS:
            inicio=time.time();key=(tarea,nombre,alg)
            try:
                modelo=crear_modelo(alg,len(etiquetas))
                if optimizar:
                    
                    
                    cv=StratifiedKFold(n_splits=N_SPLITS_CV,shuffle=True,random_state=SEMILLA)
                    busqueda=GridSearchCV(modelo,rejillas[alg],scoring='f1_macro',cv=cv,
                                           n_jobs=1,refit=True,error_score='raise')
                    busqueda.fit(xtr,ytr)
                    modelo=busqueda.best_estimator_
                    detalles_cv.append({'tarea':tarea,'representacion':nombre,'clasificador':alg,
                                        'f1_cv_train':busqueda.best_score_,
                                        'mejores_parametros':json.dumps(busqueda.best_params_)})
                else:
                    modelo.fit(xtr,ytr)
                p=modelo.predict(xte)
                predicciones[key]=p
                fila={'tarea':tarea,'representacion':nombre,'clasificador':alg,'n_train':len(ytr),
                    'n_test':len(yte),'optimizado_hmvt':optimizar,
                    'accuracy':accuracy_score(yte,p),
                    'balanced_accuracy':balanced_accuracy_score(yte,p),
                    'precision_macro':precision_score(yte,p,average='macro',zero_division=0),
                    'recall_macro':recall_score(yte,p,average='macro',zero_division=0),
                    'f1_macro':f1_score(yte,p,average='macro',zero_division=0),
                    'precision_weighted':precision_score(yte,p,average='weighted',zero_division=0),
                    'recall_weighted':recall_score(yte,p,average='weighted',zero_division=0),
                    'f1_weighted':f1_score(yte,p,average='weighted',zero_division=0),
                    'tiempo_s':round(time.time()-inicio,2),'error':''}
                metricas.append(fila)
                informe=classification_report(yte,p,labels=etiquetas_id,output_dict=True,zero_division=0)
                for j,et in enumerate(etiquetas):
                    item=informe[str(j)]
                    por_clase.append({'tarea':tarea,'representacion':nombre,'clasificador':alg,'clase':et,
                                      'precision':item['precision'],'recall':item['recall'],
                                      'f1_score':item['f1-score'],'soporte':item['support']})
                cm=confusion_matrix(yte,p,labels=etiquetas_id)
                base=f'{tarea}_{nombre}_{alg}'.replace('+','_MAS_')
                path=SALIDA/'matrices_confusion';path.mkdir(exist_ok=True)
                pd.DataFrame(cm,index=[str(e) for e in etiquetas],columns=[str(e) for e in etiquetas]).to_csv(path/(base+'.csv'))
                fig,ax=plt.subplots(figsize=(6.5,5.3) if tarea=='BINARIA' else (8,6.8))
                sns.heatmap(cm,annot=True,fmt='d',cmap='Blues',xticklabels=etiquetas,
                            yticklabels=etiquetas,ax=ax,cbar=False)
                ax.set(title=f'{tarea} · {nombre} · {alg}\nTEST 20% (textos únicos)',xlabel='Predicha',ylabel='Real')
                plt.xticks(rotation=40,ha='right');plt.yticks(rotation=0)
                fig.tight_layout();fig.savefig(path/(base+'.png'),dpi=155,bbox_inches='tight')
                plt.close(fig)
            except Exception as ex:
                metricas.append({'tarea':tarea,'representacion':nombre,'clasificador':alg,
                                 'optimizado_hmvt':optimizar,'error':str(ex),'tiempo_s':round(time.time()-inicio,2)})
                print('ERROR',tarea,nombre,alg,str(ex)[:160])
        ok=[v for v in metricas if v['tarea']==tarea and v['representacion']==nombre and 'f1_macro' in v]
        if ok:
            k=max(ok,key=lambda z:z['f1_macro'])
            print(f'{tarea:10s} {nombre:20s} F1 macro mas alto en TEST: {k["clasificador"]}={k["f1_macro"]:.3f} (descriptivo, no seleccion final)')

res=pd.DataFrame(metricas)
res.to_csv(SALIDA/'metricas_completas_80_20.csv',index=False)
pd.DataFrame(por_clase).to_csv(SALIDA/'precision_recall_f1_por_clase.csv',index=False)
pd.DataFrame(detalles_cv).to_csv(SALIDA/'optimizacion_SOLO_HMVT_CV_train.csv',index=False)
validos=res.dropna(subset=['f1_macro'])
print('\nRESULTADOS TEST - todas las metricas:')
print(validos[['tarea','representacion','clasificador','accuracy','balanced_accuracy',
              'precision_macro','recall_macro','f1_macro','precision_weighted','recall_weighted','f1_weighted']].to_string(index=False))
if validos.empty:raise RuntimeError('Ningun modelo se entreno. Ver errores')




def pred_fija(tarea):
    preferencias=[(tarea,'HMVT-V4+GEO','RL'),(tarea,'HMVT-V4','RL'),(tarea,'TFIDF','RL')]
    return next((k for k in preferencias if k in predicciones),None)

def pred_preferida_por_modelo(tarea,alg):
    preferencias=[(tarea,'HMVT-V4+GEO',alg),(tarea,'HMVT-V4',alg),(tarea,'TFIDF',alg)]
    return next((k for k in preferencias if k in predicciones),None)

from IPython.display import display,Image
resumen_dir=SALIDA/'matrices_resumen_por_modelo'
resumen_dir.mkdir(exist_ok=True)
for tarea,(col,etiquetas) in tareas.items():
    yreal=te[col].map({x:i for i,x in enumerate(etiquetas)}).to_numpy() if tarea=='MULTICLASE' else te.violencia_binaria.to_numpy()
    key=pred_fija(tarea)
    if key:
        print('\nModelo de mapa PREFIJADO:',key,'\n',classification_report(
            yreal,predicciones[key],labels=np.arange(len(etiquetas)),
            target_names=[str(k) for k in etiquetas],zero_division=0))
    print(f'\nMATRICES RESUMEN POR MODELO · {tarea}')
    for alg in MODELOS:
        key_alg=pred_preferida_por_modelo(tarea,alg)
        if not key_alg:
            continue
        rep_sel=key_alg[1]
        print(f'  - {alg}: usando {rep_sel}')
        p_img=SALIDA/'matrices_confusion'/('_'.join(key_alg).replace('+','_MAS_')+'.png')
        p_csv=SALIDA/'matrices_confusion'/('_'.join(key_alg).replace('+','_MAS_')+'.csv')
        if p_img.exists():
            destino=(resumen_dir/f'{tarea}_{alg}_{rep_sel}'.replace('+','_MAS_'))
            import shutil
            shutil.copy2(p_img,destino.with_suffix('.png'))
            if p_csv.exists():
                shutil.copy2(p_csv,destino.with_suffix('.csv'))
            if MOSTRAR_GRAFICAS:
                display(Image(filename=str(p_img)))





geo_test=df[df.texto_norm.isin(set(te.texto_norm))].drop_duplicates(['texto_norm','estado']).copy()
geo_test=geo_test.merge(te[['texto_norm']],on='texto_norm',how='inner')
for tarea,col in [('BINARIA','pred_binaria'),('MULTICLASE','pred_multiclase')]:
    key=pred_fija(tarea)
    if key:
        p=predicciones[key]
        if tarea=='MULTICLASE':p=np.array(tareas[tarea][1],dtype=object)[p]
        dic=dict(zip(te.texto_norm,p))
        geo_test[col]=geo_test.texto_norm.map(dic)
geo_test.to_csv(SALIDA/'predicciones_geograficas_SOLO_TEST.csv',index=False)

for nivel in ['region','estado']:
    filas=[]
    for lugar,part in geo_test.groupby(nivel):
        fila={nivel:lugar,'n_textos':part.texto_norm.nunique(),
              'violencia_real_pct':round(100*part.violencia_binaria.mean(),3)}
        if 'pred_binaria' in part: fila['violencia_predicha_pct']=round(100*part.pred_binaria.mean(),3)
        if 'pred_multiclase' in part:
            for clase in tareas['MULTICLASE'][1]:
                nombre=re.sub('[^a-z0-9]+','_',unicodedata.normalize('NFKD',str(clase)).encode('ascii','ignore').decode().lower()).strip('_')
                fila['pred_'+nombre+'_pct']=round(100*(part.pred_multiclase==clase).mean(),3)
        filas.append(fila)
    pd.DataFrame(filas).to_csv(SALIDA/f'resumen_{nivel}_SOLO_TEST.csv',index=False)

if HACER_MAPAS and len(geo_test)>0:
    import geopandas as gpd, requests
    def slug(s):
        return re.sub('[^a-z0-9]','',unicodedata.normalize('NFKD',str(s)).encode('ascii','ignore').decode().lower())
    alias={'distritofederal':'ciudaddemexico','mexicocity':'ciudaddemexico',
           'veracruzdeignaciodelallave':'veracruz','michoacandeocampo':'michoacan',
           'coahuiladezaragoza':'coahuila','estadodemexico':'mexico','mexicostate':'mexico'}
    try:
        meta=requests.get('https://www.geoboundaries.org/api/current/gbOpen/MEX/ADM1/',timeout=45)
        meta.raise_for_status();info=meta.json()
        url=info.get('simplifiedGeometryGeoJSON') or info.get('gjDownloadURL')
        if not url:raise RuntimeError('Sin geometria GeoJSON')
        path=SALIDA/'adm1_mx.geojson'
        if not path.exists():
            rr=requests.get(url,timeout=120);rr.raise_for_status();path.write_bytes(rr.content)
        mapa=gpd.read_file(path)
        namecol=next(c for c in ['shapeName','NAME_1','name','NOMGEO'] if c in mapa.columns)
        mapa['nombre_estado']=mapa[namecol].astype(str)
        mapa['clave']=mapa.nombre_estado.map(slug).replace(alias)
        estados=pd.read_csv(SALIDA/'resumen_estado_SOLO_TEST.csv')
        estados['clave']=estados.estado.map(slug).replace(alias)
        ge=mapa.merge(estados.drop(columns='estado'),on='clave',how='left')
        
        region_estado=df.groupby('estado').region.agg(lambda a:a.mode().iat[0]).reset_index()
        region_estado['clave']=region_estado.estado.map(slug).replace(alias)
        ge=ge.merge(region_estado[['clave','region']],on='clave',how='left')
        if not (ge.n_textos.notna().any()):raise RuntimeError('Ningun estado coincidió con las geometrías')
        regiones=pd.read_csv(SALIDA/'resumen_region_SOLO_TEST.csv')
        gr=ge[ge.region.notna()].dissolve(by='region').reset_index()[['region','geometry']]
        gr=gr.merge(regiones,on='region',how='left')
        ge.to_file(SALIDA/'estados_con_metricas.gpkg',driver='GPKG')
        gr.to_file(SALIDA/'regiones_con_metricas.gpkg',driver='GPKG')
        
        borde_nacional=gpd.GeoDataFrame(geometry=[ge.unary_union],crs=ge.crs)
        limites_estatales=ge[['nombre_estado','geometry']].copy()
        limites_regionales=gr[['region','geometry']].copy() if len(gr)>0 else None
        carpeta_mapas=SALIDA/'mapas'
        carpeta_mapas.mkdir(exist_ok=True)
        cmap_morado='Purples'
        for nivel,gdf in [('estado',ge),('region',gr)]:
            subdir=carpeta_mapas/nivel
            subdir.mkdir(exist_ok=True)
            campos=[c for c in gdf.columns if c.endswith('_pct')]
            for campo in campos:
                fig,ax=plt.subplots(figsize=(14.6,10))
                gdf.plot(ax=ax,color='#f5f3ff',edgecolor='#b4a7d6',linewidth=.45)
                gdf.plot(column=campo,ax=ax,cmap=cmap_morado,vmin=0,vmax=100,legend=True,
                         missing_kwds={'color':'#ede9fe','edgecolor':'#b4a7d6','label':'Sin datos'},
                         edgecolor='#4c1d95' if nivel=='estado' else '#6d28d9',
                         linewidth=.95 if nivel=='estado' else .85,
                         legend_kwds={'label':'% de textos TEST','shrink':.72})
                if nivel=='region':
                    limites_estatales.boundary.plot(ax=ax,color='#5b21b6',linewidth=.55,alpha=.95)
                    if limites_regionales is not None:
                        limites_regionales.boundary.plot(ax=ax,color='#2e1065',linewidth=1.8)
                else:
                    limites_estatales.boundary.plot(ax=ax,color='#3b0764',linewidth=.82,alpha=.98)
                    if limites_regionales is not None and 'region' in gdf.columns:
                        try:
                            gdf[['region','geometry']].dissolve(by='region').boundary.plot(ax=ax,color='#6d28d9',linewidth=1.15,alpha=.7)
                        except Exception:
                            pass
                borde_nacional.boundary.plot(ax=ax,color='#1e1b4b',linewidth=1.5)
                for _,v in gdf[gdf[campo].notna()].iterrows():
                    point=v.geometry.representative_point()
                    label=str(v['nombre_estado'] if nivel=='estado' else v['region'])
                    ax.annotate(label,(point.x,point.y),fontsize=7 if nivel=='estado' else 10,
                                ha='center',va='center',color='#1f0933',
                                bbox=dict(boxstyle='round,pad=0.16',fc='white',ec='none',alpha=.72))
                nombre_limpio=campo.replace('_pct','').replace('pred_','').replace('_',' ').title()
                subt='División política visible: límites estatales, regionales y contorno nacional'
                ax.set_title(f'{nombre_limpio} por {nivel} · TEST 20%\n{subt}',fontsize=16,weight='bold',color='#2e1065')
                ax.set_axis_off();fig.tight_layout()
                out=subdir/f'mapa_{nivel}_{campo}.png'
                fig.savefig(out,dpi=260,bbox_inches='tight',facecolor='white')
                if campo.startswith('pred_') and campo!='pred_binaria' and campo!='violencia_predicha_pct':
                    clase_nombre=campo.replace('pred_','')
                    fig.savefig(subdir/f'mapa_{nivel}_clase_{clase_nombre}.png',dpi=260,bbox_inches='tight',facecolor='white')
                if MOSTRAR_GRAFICAS and (campo=='violencia_predicha_pct' or campo.startswith('pred_')):
                    plt.show()
                plt.close(fig)
        print('Mapas GeoPandas generados')
    except Exception as ex:
        print('ADVERTENCIA cartografia: no se generaron mapas por',type(ex).__name__,str(ex))


for tarea in tareas:
    tabla=validos[validos.tarea==tarea]
    fig,ax=plt.subplots(figsize=(15,8))
    sns.heatmap(tabla.pivot_table(index='representacion',columns='clasificador',values='f1_macro'),
                annot=True,fmt='.2f',cmap='viridis',vmin=0,vmax=1,ax=ax)
    ax.set_title('F1 macro · '+tarea+' · TEST 20% (DESCRIPTIVO)')
    fig.tight_layout();fig.savefig(SALIDA/f'comparativa_f1_{tarea}.png',dpi=170,bbox_inches='tight')
    if MOSTRAR_GRAFICAS:plt.show()
    plt.close(fig)
perfectos=validos[(validos.accuracy>=.99999)&(validos.f1_macro>=.99999)]
print('\nModelos con desempeno perfecto:',len(perfectos))
if len(perfectos):
    print('ALERTA: 1.0 no demuestra generalizacion; verificar datos sinteticos, textos formulaicos,')
    print('sesgo de anotacion y resultados en otro dataset independiente. No forzamos scores artificialmente.')

info={'archivo':nombre_archivo,'filas_originales':len(df),'textos_unicos':len(corpus),
      'train_unicos':len(tr),'test_unicos':len(te),'test_size':TEST_SIZE,
      'semilla':SEMILLA,'optimizacion':'solo HMVT: 3-fold CV estratificada en train',
      'baselines':'parametros fijos, sin GridSearchCV','representaciones':list(rep),
      'modelo_geo_prefijado':'RL+HMVT-V4+GEO',
      'nota':'El CSV contiene textos duplicados y ubicaciones contradictorias. No es evidencia de prevalencia poblacional. HMVT V1-V4 son definiciones experimentales y deben compararse con implementaciones originales.'}
(SALIDA/'metodologia_y_advertencias.json').write_text(json.dumps(info,indent=2,ensure_ascii=False),encoding='utf8')
zip_path=SALIDA.parent/'resultados_hmvt_80_20.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for p in SALIDA.rglob('*'):
        if p.is_file():z.write(p,arcname=str(p.relative_to(SALIDA)))
print('\nRESULTADOS LISTOS:',SALIDA,'\nZIP:',zip_path)
if 'google.colab' in sys.modules:
    print('Para descargar ZIP: files.download(str(zip_path))')


### EXP-050 · V1, V2, V3, V4

**Archivo de origen:** `27_Octubre.ipynb, celda 11`. **Referencias duplicadas:** 0. **Sintaxis de código:** Válida.

CLASIFICACIÓN BINARIA Y MULTICLASE CON HMVT + GEO
80/20 | 8 MODELOS | MATRICES Y MAPAS MORADOS
GOOGLE COLAB - UNA SOLA CELDA
import os, sys, re, json, time, warnings, zipfile
import subprocess, importlib.util, unicodedata
from pathlib impor

**Ejecución:** celda independiente o variante experimental; revisar archivos de entrada y dependencias antes de ejecutarla. Los resultados originales no se insertan en esta celda.

In [ ]:
import os, sys, re, json, time, warnings, zipfile
import subprocess, importlib.util, unicodedata
from pathlib import Path
from io import BytesIO

warnings.filterwarnings("ignore", category=FutureWarning)


SEMILLA = 42
TEST_SIZE = 0.20
MOSTRAR_GRAFICAS = True
USAR_GEO = True
N_SPLITS_CV = 3

MODELOS = ["RL", "SVM", "RF", "MLP",
           "NB", "KNN", "XGB", "ARBOL"]

BASES = ["TFIDF", "NGRAMAS", "BOW", "EMBEDDINGS"]
HMVT = ["HMVT-V1", "HMVT-V2", "HMVT-V3", "HMVT-V4"]

SALIDA = Path("/content/resultados_hmvt_80_20")
SALIDA.mkdir(parents=True, exist_ok=True)

MODELO_EMBEDDINGS = (
    "sentence-transformers/"
    "paraphrase-multilingual-MiniLM-L12-v2"
)


def asegurar(modulo, paquete=None):
    if importlib.util.find_spec(modulo) is None:
        subprocess.check_call([
            sys.executable, "-m", "pip", "-q",
            "install", paquete or modulo
        ])

dependencias = [
    ("numpy", None),
    ("pandas", None),
    ("scipy", None),
    ("sklearn", "scikit-learn"),
    ("matplotlib", None),
    ("seaborn", None),
    ("sentence_transformers", "sentence-transformers"),
    ("xgboost", None),
    ("geopandas", None),
    ("requests", None)
]

for modulo, paquete in dependencias:
    asegurar(modulo, paquete)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import geopandas as gpd
import requests

from scipy import sparse
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, GridSearchCV
)
from sklearn.feature_extraction.text import (
    TfidfVectorizer, CountVectorizer
)
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sentence_transformers import SentenceTransformer
from xgboost import XGBClassifier
from google.colab import files
from IPython.display import display, Image

np.random.seed(SEMILLA)
sns.set_theme(style="white")


COLOR_OSCURO = "#3B0764"
COLOR_PRINCIPAL = "#6B21A8"
COLOR_MEDIO = "#9333EA"
COLOR_CLARO = "#F3E8FF"
CMAP = "Purples"




print("SUBE TU CONJUNTO DE DATOS CSV")
subidos = files.upload()

if len(subidos) != 1:
    raise ValueError("Selecciona exactamente un CSV.")

nombre_archivo, contenido = next(iter(subidos.items()))

if not nombre_archivo.lower().endswith(".csv"):
    raise ValueError("El archivo debe ser CSV.")

df = pd.read_csv(BytesIO(contenido), encoding="utf-8-sig")

requeridas = {
    "tweet", "clase", "violencia_binaria",
    "region", "estado", "regionalismo"
}

faltantes = requeridas - set(df.columns)
if faltantes:
    raise ValueError(f"Columnas faltantes: {faltantes}")

df = df.dropna(
    subset=["tweet", "clase", "violencia_binaria"]
).copy()

for col in ["tweet", "clase", "region",
            "estado", "regionalismo"]:
    df[col] = df[col].fillna("SIN_DATO").astype(str).str.strip()

df["violencia_binaria"] = pd.to_numeric(
    df["violencia_binaria"], errors="raise"
).astype(int)

if not set(df.violencia_binaria.unique()).issubset({0, 1}):
    raise ValueError("La etiqueta binaria debe contener 0 y 1.")

def normalizar(texto):
    texto = str(texto).lower().strip()
    texto = re.sub(r"https?://\S+|www\.\S+", " URL ", texto)
    texto = re.sub(r"@\w+", " USUARIO ", texto)
    return re.sub(r"\s+", " ", texto)

df["texto_norm"] = df["tweet"].map(normalizar)
df = df[df.texto_norm.str.len() > 0].reset_index(drop=True)

for etiqueta in ["clase", "violencia_binaria"]:
    conflictos = df.groupby("texto_norm")[etiqueta].nunique()
    if (conflictos > 1).any():
        raise ValueError(
            f"Existen textos idénticos con etiquetas {etiqueta} "
            "contradictorias. Revisa las anotaciones."
        )

if df.groupby("clase").violencia_binaria.nunique().max() > 1:
    raise ValueError("Las etiquetas binarias y multiclase no coinciden.")




print("\nDIAGNÓSTICO")
print("Registros:", len(df))
print("Textos únicos:", df.texto_norm.nunique())
print("Duplicados completos:", df.duplicated().sum())

corpus = df.drop_duplicates("texto_norm").copy()

geo_cols = ["region", "estado", "regionalismo"]



for col in geo_cols:
    variacion = df.groupby("texto_norm")[col].nunique()
    corpus.loc[
        corpus.texto_norm.map(variacion) > 1, col
    ] = "SIN_DATO"

corpus = corpus.reset_index(drop=True)

idx_train, idx_test = train_test_split(
    np.arange(len(corpus)),
    test_size=TEST_SIZE,
    random_state=SEMILLA,
    stratify=corpus["clase"]
)

tr = corpus.iloc[idx_train].reset_index(drop=True)
te = corpus.iloc[idx_test].reset_index(drop=True)

assert not set(tr.texto_norm) & set(te.texto_norm)

print("\nPARTICIÓN 80/20")
print("Train:", len(tr))
print("Test:", len(te))
print("\nDistribución TRAIN:")
print(tr.clase.value_counts())
print("\nDistribución TEST:")
print(te.clase.value_counts())

tr.to_csv(SALIDA / "train.csv", index=False)
te.to_csv(SALIDA / "test.csv", index=False)

textos_tr = tr.texto_norm.tolist()
textos_te = te.texto_norm.tolist()




rep = {}

def registrar(nombre, train, test):
    rep[nombre] = (
        sparse.csr_matrix(train, dtype=np.float32),
        sparse.csr_matrix(test, dtype=np.float32)
    )
    print(f"{nombre}: {rep[nombre][0].shape[1]} atributos")

def vectorizar(nombre, vectorizador):
    registrar(
        nombre,
        vectorizador.fit_transform(textos_tr),
        vectorizador.transform(textos_te)
    )

print("\nGENERANDO REPRESENTACIONES")

vectorizar(
    "TFIDF",
    TfidfVectorizer(
        max_features=12000,
        ngram_range=(1, 2),
        strip_accents="unicode"
    )
)

vectorizar(
    "NGRAMAS",
    TfidfVectorizer(
        analyzer="char",
        ngram_range=(3, 5),
        max_features=14000
    )
)

vectorizar(
    "BOW",
    CountVectorizer(
        max_features=12000,
        strip_accents="unicode"
    )
)

encoder = SentenceTransformer(MODELO_EMBEDDINGS)

emb_tr = encoder.encode(
    textos_tr,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True
)
emb_te = encoder.encode(
    textos_te,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True
)

esc = MinMaxScaler().fit(emb_tr)

registrar(
    "EMBEDDINGS",
    esc.transform(emb_tr),
    np.maximum(0, esc.transform(emb_te))
)





bloques = {
    "HMVT-V1": [
        ("TFIDF", 1.0),
        ("NGRAMAS", 1.0)
    ],
    "HMVT-V2": [
        ("TFIDF", 1.0),
        ("BOW", 0.3),
        ("NGRAMAS", 0.7)
    ],
    "HMVT-V3": [
        ("TFIDF", 1.0),
        ("NGRAMAS", 0.7),
        ("EMBEDDINGS", 2.0)
    ],
    "HMVT-V4": [
        ("TFIDF", 1.0),
        ("NGRAMAS", 0.7),
        ("BOW", 0.3),
        ("EMBEDDINGS", 2.0)
    ]
}

for nombre, partes in bloques.items():
    matrices = []
    for i in range(2):
        matrices.append(
            sparse.hstack(
                [rep[k][i] * peso for k, peso in partes],
                format="csr"
            )
        )
    registrar(nombre, matrices[0], matrices[1])




if USAR_GEO:
    try:
        geo_encoder = OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=True,
            dtype=np.float32
        )
    except TypeError:
        geo_encoder = OneHotEncoder(
            handle_unknown="ignore",
            sparse=True,
            dtype=np.float32
        )

    geo_tr = geo_encoder.fit_transform(tr[geo_cols])
    geo_te = geo_encoder.transform(te[geo_cols])

    for nombre in BASES + HMVT:
        xtr, xte = rep[nombre]

        registrar(
            nombre + "+GEO",
            sparse.hstack([xtr, geo_tr], format="csr"),
            sparse.hstack([xte, geo_te], format="csr")
        )




def crear_modelo(nombre, nclases):

    if nombre == "RL":
        return LogisticRegression(
            max_iter=1200, random_state=SEMILLA
        )

    if nombre == "SVM":
        return LinearSVC(
            max_iter=3000, random_state=SEMILLA
        )

    if nombre == "RF":
        return RandomForestClassifier(
            n_estimators=150,
            random_state=SEMILLA,
            n_jobs=-1
        )

    if nombre == "MLP":
        return MLPClassifier(
            hidden_layer_sizes=(64,),
            early_stopping=True,
            max_iter=200,
            random_state=SEMILLA
        )

    if nombre == "NB":
        return MultinomialNB()

    if nombre == "KNN":
        return KNeighborsClassifier(n_neighbors=3)

    if nombre == "XGB":
        return XGBClassifier(
            n_estimators=130,
            max_depth=4,
            learning_rate=0.1,
            random_state=SEMILLA,
            tree_method="hist",
            n_jobs=2,
            eval_metric=(
                "logloss" if nclases == 2 else "mlogloss"
            )
        )

    if nombre == "ARBOL":
        return DecisionTreeClassifier(
            random_state=SEMILLA
        )

    raise ValueError(nombre)


rejillas = {
    "RL": {"C": [0.1, 1.0]},
    "SVM": {"C": [0.1, 1.0]},
    "RF": {"max_depth": [6, None]},
    "MLP": {"alpha": [0.0001, 0.01]},
    "NB": {"alpha": [0.5, 1.0]},
    "KNN": {"n_neighbors": [3, 5]},
    "XGB": {"max_depth": [2, 4]},
    "ARBOL": {"max_depth": [4, 10]}
}




metricas = []
por_clase = []
predicciones = {}
detalles_cv = []

tareas = {
    "BINARIA": ("violencia_binaria", [0, 1]),
    "MULTICLASE": ("clase", sorted(corpus.clase.unique()))
}

CARPETA_MATRICES = SALIDA / "matrices_confusion"
CARPETA_MATRICES.mkdir(exist_ok=True)

for tarea, (col, etiquetas) in tareas.items():

    if tarea == "MULTICLASE":
        codificacion = {
            etiqueta: i for i, etiqueta in enumerate(etiquetas)
        }
        ytr = tr[col].map(codificacion).to_numpy()
        yte = te[col].map(codificacion).to_numpy()
    else:
        ytr = tr[col].to_numpy(dtype=int)
        yte = te[col].to_numpy(dtype=int)

    ids = np.arange(len(etiquetas))

    for representacion, (xtr, xte) in rep.items():

        optimizar = representacion.startswith("HMVT-")

        for algoritmo in MODELOS:

            inicio = time.time()
            clave = (tarea, representacion, algoritmo)

            try:
                modelo = crear_modelo(algoritmo, len(etiquetas))

                if optimizar:
                    cv = StratifiedKFold(
                        n_splits=N_SPLITS_CV,
                        shuffle=True,
                        random_state=SEMILLA
                    )

                    busqueda = GridSearchCV(
                        modelo,
                        rejillas[algoritmo],
                        scoring="f1_macro",
                        cv=cv,
                        n_jobs=1,
                        refit=True,
                        error_score="raise"
                    )
                    busqueda.fit(xtr, ytr)
                    modelo = busqueda.best_estimator_

                    detalles_cv.append({
                        "tarea": tarea,
                        "representacion": representacion,
                        "clasificador": algoritmo,
                        "f1_cv_train": busqueda.best_score_,
                        "parametros": str(busqueda.best_params_)
                    })
                else:
                    modelo.fit(xtr, ytr)

                pred = modelo.predict(xte)
                predicciones[clave] = pred

                fila = {
                    "tarea": tarea,
                    "representacion": representacion,
                    "clasificador": algoritmo,
                    "accuracy": accuracy_score(yte, pred),
                    "balanced_accuracy": balanced_accuracy_score(
                        yte, pred
                    ),
                    "precision_macro": precision_score(
                        yte, pred, average="macro", zero_division=0
                    ),
                    "recall_macro": recall_score(
                        yte, pred, average="macro", zero_division=0
                    ),
                    "f1_macro": f1_score(
                        yte, pred, average="macro", zero_division=0
                    ),
                    "precision_weighted": precision_score(
                        yte, pred, average="weighted",
                        zero_division=0
                    ),
                    "recall_weighted": recall_score(
                        yte, pred, average="weighted",
                        zero_division=0
                    ),
                    "f1_weighted": f1_score(
                        yte, pred, average="weighted",
                        zero_division=0
                    ),
                    "tiempo_s": time.time() - inicio,
                    "optimizado": optimizar
                }

                metricas.append(fila)

                reporte = classification_report(
                    yte, pred,
                    labels=ids,
                    output_dict=True,
                    zero_division=0
                )

                for i, etiqueta in enumerate(etiquetas):
                    item = reporte[str(i)]
                    por_clase.append({
                        "tarea": tarea,
                        "representacion": representacion,
                        "clasificador": algoritmo,
                        "clase": etiqueta,
                        "precision": item["precision"],
                        "recall": item["recall"],
                        "f1_score": item["f1-score"],
                        "soporte": item["support"]
                    })

                
                
                
                cm = confusion_matrix(yte, pred, labels=ids)

                nombre = (
                    f"{tarea}_{representacion}_{algoritmo}"
                    .replace("+", "_MAS_")
                )

                pd.DataFrame(
                    cm,
                    index=[str(x) for x in etiquetas],
                    columns=[str(x) for x in etiquetas]
                ).to_csv(
                    CARPETA_MATRICES / f"{nombre}.csv"
                )

                fig, ax = plt.subplots(
                    figsize=(7, 5.8) if tarea == "BINARIA"
                    else (9, 7.5)
                )

                sns.heatmap(
                    cm,
                    annot=True,
                    fmt="d",
                    cmap="Purples",
                    xticklabels=etiquetas,
                    yticklabels=etiquetas,
                    ax=ax,
                    cbar=True,
                    linewidths=0.8,
                    linecolor="white",
                    annot_kws={
                        "size": 12,
                        "weight": "bold"
                    }
                )

                ax.set_title(
                    f"{tarea} | {representacion} | {algoritmo}\n"
                    "Matriz de confusión - TEST 20%",
                    fontsize=15,
                    fontweight="bold",
                    color=COLOR_OSCURO,
                    pad=18
                )
                ax.set_xlabel("Clase predicha", fontsize=12)
                ax.set_ylabel("Clase real", fontsize=12)

                plt.xticks(rotation=35, ha="right")
                plt.yticks(rotation=0)

                fig.tight_layout()
                fig.savefig(
                    CARPETA_MATRICES / f"{nombre}.png",
                    dpi=250,
                    bbox_inches="tight",
                    facecolor="white"
                )
                plt.close(fig)

            except Exception as exc:
                print(
                    "ERROR:",
                    tarea, representacion, algoritmo,
                    str(exc)[:150]
                )

        print("Completado:", tarea, representacion)


resultados = pd.DataFrame(metricas)
resultados.to_csv(
    SALIDA / "metricas_completas.csv", index=False
)

pd.DataFrame(por_clase).to_csv(
    SALIDA / "metricas_por_clase.csv", index=False
)

pd.DataFrame(detalles_cv).to_csv(
    SALIDA / "optimizacion_solo_hmvt.csv", index=False
)

if resultados.empty:
    raise RuntimeError("Ningún modelo se entrenó correctamente.")

print("\nTABLA DE RESULTADOS")
display(resultados)







def pred_preferida(tarea, algoritmo):
    opciones = [
        (tarea, "HMVT-V4+GEO", algoritmo),
        (tarea, "HMVT-V4", algoritmo),
        (tarea, "TFIDF", algoritmo)
    ]
    return next(
        (k for k in opciones if k in predicciones), None
    )

CARPETA_RESUMEN = SALIDA / "matrices_resumen_por_modelo"
CARPETA_RESUMEN.mkdir(exist_ok=True)

import shutil

for tarea in tareas:

    print("\nMATRICES POR MODELO:", tarea)

    for algoritmo in MODELOS:

        clave = pred_preferida(tarea, algoritmo)
        if clave is None:
            continue

        nombre = "_".join(clave).replace("+", "_MAS_")

        archivo = CARPETA_MATRICES / f"{nombre}.png"

        if archivo.exists():
            shutil.copy2(
                archivo,
                CARPETA_RESUMEN / f"{nombre}.png"
            )

            if MOSTRAR_GRAFICAS:
                display(Image(filename=str(archivo)))







geo_test = (
    df[df.texto_norm.isin(set(te.texto_norm))]
    .drop_duplicates(["texto_norm", "estado"])
    .copy()
)

for tarea, columna in [
    ("BINARIA", "pred_binaria"),
    ("MULTICLASE", "pred_multiclase")
]:

    clave = pred_preferida(tarea, "RL")

    if clave is None:
        continue

    p = predicciones[clave]

    if tarea == "MULTICLASE":
        p = np.asarray(tareas[tarea][1], dtype=object)[p]

    diccionario = dict(zip(te.texto_norm, p))
    geo_test[columna] = geo_test.texto_norm.map(diccionario)

geo_test.to_csv(
    SALIDA / "predicciones_geograficas.csv",
    index=False
)

def slug(texto):
    texto = unicodedata.normalize("NFKD", str(texto))
    texto = texto.encode("ascii", "ignore").decode().lower()
    return re.sub(r"[^a-z0-9]", "", texto)

def nombre_campo(clase):
    texto = unicodedata.normalize("NFKD", str(clase))
    texto = texto.encode("ascii", "ignore").decode().lower()
    return re.sub(r"[^a-z0-9]+", "_", texto).strip("_")

CLASES = tareas["MULTICLASE"][1]

for nivel in ["region", "estado"]:

    filas = []

    for lugar, datos in geo_test.groupby(nivel):

        fila = {
            nivel: lugar,
            "n_textos": datos.texto_norm.nunique(),
            "violencia_real_pct": (
                100 * datos.violencia_binaria.mean()
            )
        }

        if "pred_binaria" in datos.columns:
            fila["violencia_predicha_pct"] = (
                100 * datos.pred_binaria.mean()
            )

        if "pred_multiclase" in datos.columns:

            for clase in CLASES:
                campo = "pred_" + nombre_campo(clase) + "_pct"
                fila[campo] = (
                    100 * (datos.pred_multiclase == clase).mean()
                )

        filas.append(fila)

    pd.DataFrame(filas).to_csv(
        SALIDA / f"resumen_{nivel}.csv",
        index=False
    )





print("\nGENERANDO MAPAS GEOGRÁFICOS")

try:
    respuesta = requests.get(
        "https://www.geoboundaries.org/api/current/gbOpen/MEX/ADM1/",
        timeout=45
    )
    respuesta.raise_for_status()
    info = respuesta.json()

    url = (
        info.get("simplifiedGeometryGeoJSON")
        or info.get("gjDownloadURL")
    )

    if not url:
        raise ValueError("No se encontró la geometría.")

    archivo_geo = SALIDA / "mexico_estados.geojson"

    if not archivo_geo.exists():
        descarga = requests.get(url, timeout=120)
        descarga.raise_for_status()
        archivo_geo.write_bytes(descarga.content)

    mapa = gpd.read_file(archivo_geo)

    columna_nombre = next(
        c for c in ["shapeName", "NAME_1", "name", "NOMGEO"]
        if c in mapa.columns
    )

    mapa["nombre_estado"] = mapa[columna_nombre].astype(str)

    alias = {
        "distritofederal": "ciudaddemexico",
        "mexicocity": "ciudaddemexico",
        "veracruzdeignaciodelallave": "veracruz",
        "michoacandeocampo": "michoacan",
        "coahuiladezaragoza": "coahuila",
        "estadodemexico": "mexico",
        "mexicostate": "mexico"
    }

    mapa["clave"] = mapa.nombre_estado.map(slug).replace(alias)

    estados = pd.read_csv(SALIDA / "resumen_estado.csv")
    estados["clave"] = estados.estado.map(slug).replace(alias)

    ge = mapa.merge(
        estados.drop(columns="estado"),
        on="clave",
        how="left"
    )

    regiones_estados = (
        df.groupby("estado")["region"]
        .agg(lambda x: x.mode().iloc[0])
        .reset_index()
    )
    regiones_estados["clave"] = (
        regiones_estados.estado.map(slug).replace(alias)
    )

    ge = ge.merge(
        regiones_estados[["clave", "region"]],
        on="clave",
        how="left"
    )

    if not ge.n_textos.notna().any():
        raise ValueError(
            "No coincidieron los nombres de los estados "
            "con las geometrías."
        )

    regiones = pd.read_csv(SALIDA / "resumen_region.csv")

    gr = (
        ge[ge.region.notna()]
        .dissolve(by="region")
        .reset_index()[["region", "geometry"]]
    )

    gr = gr.merge(regiones, on="region", how="left")

    ge.to_file(
        SALIDA / "estados_con_metricas.gpkg",
        driver="GPKG"
    )
    gr.to_file(
        SALIDA / "regiones_con_metricas.gpkg",
        driver="GPKG"
    )

    
    limites_estados = ge[["nombre_estado", "geometry"]].copy()
    limites_regiones = gr[["region", "geometry"]].copy()

    borde_nacional = gpd.GeoDataFrame(
        geometry=[ge.geometry.union_all()],
        crs=ge.crs
    )

    CARPETA_MAPAS = SALIDA / "mapas"
    CARPETA_MAPAS.mkdir(exist_ok=True)

    
    campos_clases = [
        "pred_" + nombre_campo(c) + "_pct"
        for c in CLASES
    ]

    for nivel, gdf in [("region", gr), ("estado", ge)]:

        carpeta = CARPETA_MAPAS / nivel
        carpeta.mkdir(exist_ok=True)

        campos = (
            ["violencia_predicha_pct"] + campos_clases
        )

        for campo in campos:

            if campo not in gdf.columns:
                continue

            fig, ax = plt.subplots(figsize=(15, 10))
            fig.patch.set_facecolor("white")
            ax.set_facecolor("white")

            
            gdf.plot(
                column=campo,
                ax=ax,
                cmap="Purples",
                vmin=0,
                vmax=100,
                legend=True,
                edgecolor="#7C3AED",
                linewidth=0.7,
                missing_kwds={
                    "color": "#E5E7EB",
                    "edgecolor": "#A1A1AA"
                },
                legend_kwds={
                    "label": "Porcentaje de publicaciones (%)",
                    "shrink": 0.70
                }
            )

            
            limites_estados.boundary.plot(
                ax=ax,
                color="#4C1D95",
                linewidth=0.55
            )

            if nivel == "region":
                limites_regiones.boundary.plot(
                    ax=ax,
                    color=COLOR_OSCURO,
                    linewidth=1.8
                )

            borde_nacional.boundary.plot(
                ax=ax,
                color="#1E1B4B",
                linewidth=1.4
            )

            
            for _, fila in gdf[gdf[campo].notna()].iterrows():

                punto = fila.geometry.representative_point()

                etiqueta = (
                    fila["nombre_estado"]
                    if nivel == "estado"
                    else fila["region"]
                )

                ax.annotate(
                    str(etiqueta),
                    xy=(punto.x, punto.y),
                    ha="center",
                    va="center",
                    fontsize=7 if nivel == "estado" else 10,
                    color=COLOR_OSCURO,
                    bbox={
                        "boxstyle": "round,pad=0.15",
                        "facecolor": "white",
                        "edgecolor": "none",
                        "alpha": 0.75
                    }
                )

            nombre_clase = (
                campo.replace("pred_", "")
                .replace("_pct", "")
                .replace("_", " ")
                .title()
            )

            ax.set_title(
                f"{nombre_clase} por {nivel}\n"
                "Distribución geográfica de las predicciones",
                fontsize=17,
                fontweight="bold",
                color=COLOR_OSCURO,
                pad=20
            )

            ax.set_axis_off()
            fig.tight_layout()

            destino = carpeta / f"mapa_{nivel}_{campo}.png"

            fig.savefig(
                destino,
                dpi=300,
                bbox_inches="tight",
                facecolor="white"
            )

            if MOSTRAR_GRAFICAS:
                plt.show()

            plt.close(fig)

    print("\nMAPAS MORADOS GENERADOS CORRECTAMENTE")

except Exception as exc:
    print("Error en cartografía:", str(exc))




for tarea in tareas:

    tabla = resultados[resultados.tarea == tarea]

    pivote = tabla.pivot_table(
        index="representacion",
        columns="clasificador",
        values="f1_macro"
    )

    fig, ax = plt.subplots(figsize=(15, 9))

    sns.heatmap(
        pivote,
        annot=True,
        fmt=".2f",
        cmap="Purples",
        vmin=0,
        vmax=1,
        linewidths=0.6,
        linecolor="white",
        ax=ax,
        cbar_kws={"label": "F1 macro"}
    )

    ax.set_title(
        f"Comparación de F1 macro - {tarea}",
        fontsize=17,
        color=COLOR_OSCURO,
        fontweight="bold"
    )

    fig.tight_layout()
    fig.savefig(
        SALIDA / f"comparativa_f1_{tarea}.png",
        dpi=250,
        bbox_inches="tight"
    )

    if MOSTRAR_GRAFICAS:
        plt.show()

    plt.close(fig)




perfectos = resultados[
    (resultados.accuracy >= 0.99999) &
    (resultados.f1_macro >= 0.99999)
]

print("\nModelos con resultados perfectos:", len(perfectos))

if len(perfectos):
    print(
        "ADVERTENCIA: un resultado perfecto no demuestra "
        "generalización. Se recomienda validación externa."
    )

metodologia = {
    "archivo": nombre_archivo,
    "train": len(tr),
    "test": len(te),
    "test_size": TEST_SIZE,
    "semilla": SEMILLA,
    "representaciones": list(rep.keys()),
    "modelos": MODELOS,
    "optimizacion": "Solo HMVT mediante CV interna",
    "mapas": "GeoPandas, paleta morada",
    "evaluacion": "Textos únicos, partición 80/20"
}

(SALIDA / "metodologia.json").write_text(
    json.dumps(metodologia, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

zip_path = Path("/content/resultados_hmvt_80_20.zip")

with zipfile.ZipFile(
    zip_path, "w", zipfile.ZIP_DEFLATED
) as archivo_zip:

    for archivo in SALIDA.rglob("*"):
        if archivo.is_file():
            archivo_zip.write(
                archivo,
                arcname=str(archivo.relative_to(SALIDA))
            )

print("\nPROCESO FINALIZADO")
print("Resultados:", SALIDA)
print("ZIP:", zip_path)

files.download(str(zip_path))


### EXP-051 · V1, V2, V3, V4

**Archivo de origen:** `27_Octubre.ipynb, celda 18`. **Referencias duplicadas:** 0. **Sintaxis de código:** Válida.

CLASIFICACION DOCTORAL BINARIA + MULTICLASE / GEO / HMVT
UNA SOLA CELDA COLAB. CARGA CSV SIEMPRE EN GOOGLE COLAB.
import sys, os, re, json, time, warnings, subprocess, importlib.util, zipfile, unicodedata
from pathlib import Path
from io im

**Ejecución:** celda independiente o variante experimental; revisar archivos de entrada y dependencias antes de ejecutarla. Los resultados originales no se insertan en esta celda.

In [ ]:
import sys, os, re, json, time, warnings, subprocess, importlib.util, zipfile, unicodedata
from pathlib import Path
from io import BytesIO
warnings.filterwarnings('ignore', category=FutureWarning)

SEMILLA=42
TEST_SIZE=.20
ARCHIVO_LOCAL='data_hmvt_con_estado.csv'
SALIDA=Path('/content/resultados_violencia_geo' if Path('/content').exists() else 'resultados_violencia_geo')
SALIDA.mkdir(parents=True,exist_ok=True)
USAR_GEO=True
HACER_MAPAS=True
MOSTRAR_GRAFICAS=True
EMBEDDING_MODELO='sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'

N_ITER_HMVT=2
N_SPLITS_CV=3
INICIO=time.monotonic()
PRESUPUESTO_ENTRENAMIENTO_S=24*60
MOSTRAR_MAPAS=True
MODELOS=['RL','KNN','SVM','NB','MLP','XGB','RF','ARBOL']
BASES=['TFIDF','NGRAMAS','BOW','EMBEDDINGS']
HMVT=['HMVT-V1','HMVT-V2','HMVT-V3','HMVT-V4']


def asegurar(nombre,pkg=None):
    if importlib.util.find_spec(nombre) is None:
        subprocess.check_call([sys.executable,'-m','pip','-q','install',pkg or nombre])
for modulo,pkg in [('numpy',None),('pandas',None),('scipy',None),('sklearn','scikit-learn'),
                   ('matplotlib',None),('seaborn',None),('sentence_transformers','sentence-transformers'),
                   ('xgboost',None),('geopandas',None),('requests',None)]:
    if modulo=='geopandas' and not HACER_MAPAS: continue
    asegurar(modulo,pkg)
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from scipy import sparse
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.metrics import accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.base import clone
sns.set_theme(style='whitegrid')
np.random.seed(SEMILLA)


if 'google.colab' in sys.modules:
    from google.colab import files
    print('SUBE EL DATASET CSV. Debe incluir tweet, clase, violencia_binaria, region, estado y regionalismo.')
    subidos=files.upload()
    if len(subidos)!=1: raise ValueError('Selecciona exactamente un archivo CSV')
    nombre_archivo,contenido=next(iter(subidos.items()))
    if not nombre_archivo.lower().endswith('.csv'): raise ValueError('Se requiere CSV')
    df=pd.read_csv(BytesIO(contenido),encoding='utf-8-sig')
else:
    nombre_archivo=ARCHIVO_LOCAL
    df=pd.read_csv(nombre_archivo,encoding='utf-8-sig')
requeridas={'tweet','clase','violencia_binaria','region','estado','regionalismo'}
if faltantes:=requeridas-set(df.columns): raise ValueError('Faltan columnas: '+str(faltantes))
df=df.dropna(subset=['tweet','clase','violencia_binaria']).copy()
for c in ['tweet','clase','region','estado','regionalismo']:
    df[c]=df[c].fillna('SIN_DATO').astype(str).str.strip()
df['violencia_binaria']=pd.to_numeric(df.violencia_binaria,errors='raise').astype(int)
if not set(df.violencia_binaria.unique()).issubset({0,1}):raise ValueError('violencia_binaria debe ser 0/1')

def normalizar(s):
    s=str(s).lower().strip()
    s=re.sub(r'https?://\S+|www\.\S+',' URL ',s)
    s=re.sub(r'@\w+',' USUARIO ',s)
    return re.sub(r'\s+',' ',s)
df['texto_norm']=df.tweet.map(normalizar)
df=df[df.texto_norm.str.len()>0].reset_index(drop=True)
if (df.groupby('texto_norm').clase.nunique()>1).any():
    raise ValueError('Existen textos iguales con etiquetas multiclase contradictorias; revisar antes de evaluar')
if (df.groupby('texto_norm').violencia_binaria.nunique()>1).any():
    raise ValueError('Existen textos iguales con etiquetas binarias contradictorias')
if df.groupby('clase').violencia_binaria.nunique().max()>1:
    raise ValueError('Inconsistencia entre clase y etiqueta binaria')



print('\nDIAGNOSTICO DEL CONJUNTO ORIGINAL')
print('Filas:',len(df),'| Textos unicos:',df.texto_norm.nunique(),'| Filas duplicadas:',df.duplicated().sum())
for c in ['estado','region','regionalismo']:
    n=(df.groupby('texto_norm')[c].nunique()>1).sum()
    print('Textos asociados con multiples',c,':',n)


corpus=df.drop_duplicates('texto_norm').copy()
geo_cols=['region','estado','regionalismo']
for c in geo_cols:
    nvar=df.groupby('texto_norm')[c].nunique()
    corpus.loc[corpus.texto_norm.map(nvar)>1,c]='SIN_DATO'
corpus=corpus.reset_index(drop=True)
idx_tr,idx_te=train_test_split(np.arange(len(corpus)),test_size=TEST_SIZE,
                                random_state=SEMILLA,stratify=corpus['clase'])
tr=corpus.iloc[idx_tr].reset_index(drop=True)
te=corpus.iloc[idx_te].reset_index(drop=True)
assert not set(tr.texto_norm).intersection(set(te.texto_norm))
assert len(tr)+len(te)==len(corpus)
print('\nDIVISION 80/20 SOBRE TEXTOS UNICOS:')
print('TRAIN',len(tr),'(',round(100*len(tr)/len(corpus),1),'%)','TEST',len(te),
      '(',round(100*len(te)/len(corpus),1),'%)')
print('TRAIN por clase:',tr.clase.value_counts().to_dict())
print('TEST por clase:',te.clase.value_counts().to_dict())
print('IMPORTANTISIMO: 80/20 corresponde a textos unicos, no a filas repetidas.')
print('REGIONALISMO y GEO con anotaciones contradictorias quedan como SIN_DATO para modelado.')
tr.to_csv(SALIDA/'train_textos_unicos.csv',index=False)
te.to_csv(SALIDA/'test_textos_unicos.csv',index=False)
textos_tr=tr.texto_norm.tolist(); textos_te=te.texto_norm.tolist()


rep={}
def registrar(k,a,b):
    rep[k]=(sparse.csr_matrix(a,dtype=np.float32),sparse.csr_matrix(b,dtype=np.float32))
    print(f'  {k:18s} | {rep[k][0].shape[1]} atributos')
def vectorizar(k,obj): registrar(k,obj.fit_transform(textos_tr),obj.transform(textos_te))
print('\nCREANDO REPRESENTACIONES (fit SOLO TRAIN):')
vectorizar('TFIDF',TfidfVectorizer(max_features=12000,ngram_range=(1,2),strip_accents='unicode'))
vectorizar('NGRAMAS',TfidfVectorizer(analyzer='char',ngram_range=(3,5),max_features=14000))
vectorizar('BOW',CountVectorizer(max_features=12000,strip_accents='unicode'))
from sentence_transformers import SentenceTransformer
st=SentenceTransformer(EMBEDDING_MODELO)
emb_tr=st.encode(textos_tr,batch_size=32,show_progress_bar=True,normalize_embeddings=True)
emb_te=st.encode(textos_te,batch_size=32,show_progress_bar=True,normalize_embeddings=True)

esc=MinMaxScaler().fit(emb_tr)
registrar('EMBEDDINGS',esc.transform(emb_tr),np.maximum(0,esc.transform(emb_te)))





bloques={
 'HMVT-V1':[('TFIDF',1.),('NGRAMAS',1.)],
 'HMVT-V2':[('TFIDF',1.),('BOW',.3),('NGRAMAS',.7)],
 'HMVT-V3':[('TFIDF',1.),('NGRAMAS',.7),('EMBEDDINGS',2.)],
 'HMVT-V4':[('TFIDF',1.),('NGRAMAS',.7),('BOW',.3),('EMBEDDINGS',2.)]
}
for nombre,partes in bloques.items():
    registrar(nombre,*[sparse.hstack([rep[k][i]*peso for k,peso in partes],format='csr') for i in range(2)])



if USAR_GEO:
    from sklearn.preprocessing import MultiLabelBinarizer
    reg_por_texto=df.groupby('texto_norm')['regionalismo'].agg(lambda x:sorted(set(v.lower() for v in x if v!='SIN_DATO'))).to_dict()
    ge=MultiLabelBinarizer(sparse_output=True)
    gt=ge.fit_transform([reg_por_texto[v] for v in tr.texto_norm]).astype(np.float32)
    gv=ge.transform([reg_por_texto[v] for v in te.texto_norm]).astype(np.float32)
    print('Vector regionalismos: ',gt.shape[1],'atributos; NO se usa región ni estado.')
    for nombre in BASES+HMVT:
        a,b=rep[nombre]
        registrar(nombre+'+REG',sparse.hstack([a,gt],format='csr'),
                  sparse.hstack([b,gv],format='csr'))



def crear_modelo(nombre,nclases):
    if nombre=='RL':return LogisticRegression(max_iter=1200,random_state=SEMILLA)
    if nombre=='SVM':return LinearSVC(max_iter=3000,random_state=SEMILLA)
    if nombre=='RF':return RandomForestClassifier(n_estimators=55,random_state=SEMILLA,n_jobs=2)
    if nombre=='MLP':return MLPClassifier(hidden_layer_sizes=(32,),early_stopping=True,max_iter=65,random_state=SEMILLA)
    if nombre=='NB':return MultinomialNB()
    if nombre=='KNN':return KNeighborsClassifier(n_neighbors=3)
    if nombre=='XGB':
        from xgboost import XGBClassifier
        return XGBClassifier(n_estimators=65,max_depth=3,learning_rate=.1,random_state=SEMILLA,
                             tree_method='hist',n_jobs=2,eval_metric='logloss' if nclases==2 else 'mlogloss')
    if nombre=='ARBOL':return DecisionTreeClassifier(random_state=SEMILLA)
    raise ValueError(nombre)

rejillas={
 'RL':{'C':[.1,1.]},'SVM':{'C':[.1,1.]},
 'RF':{'max_depth':[6,None]},'MLP':{'alpha':[.0001,.01]},
 'NB':{'alpha':[.5,1.]},'KNN':{'n_neighbors':[3,5]},
 'XGB':{'max_depth':[2,4]},'ARBOL':{'max_depth':[4,10]}
}


metricas=[]; por_clase=[]; predicciones={}; detalles_cv=[]
tareas={'BINARIA':('violencia_binaria',[0,1]),'MULTICLASE':('clase',sorted(corpus.clase.unique()))}
for tarea,(col,etiquetas) in tareas.items():
    if tarea=='MULTICLASE':
        a={k:i for i,k in enumerate(etiquetas)}
        ytr=tr[col].map(a).to_numpy();yte=te[col].map(a).to_numpy()
    else:
        ytr=tr[col].to_numpy(dtype=int);yte=te[col].to_numpy(dtype=int)
    etiquetas_id=np.arange(len(etiquetas))
    for nombre,(xtr,xte) in rep.items():
        optimizar=nombre.startswith('HMVT-')
        for alg in MODELOS:
            inicio=time.time();key=(tarea,nombre,alg)
            if time.monotonic()-INICIO>PRESUPUESTO_ENTRENAMIENTO_S:
                metricas.append({'tarea':tarea,'representacion':nombre,'clasificador':alg,'error':'Límite de tiempo alcanzado'})
                continue
            try:
                modelo=crear_modelo(alg,len(etiquetas))
                if optimizar:
                    
                    
                    it,iv=train_test_split(np.arange(len(ytr)),test_size=.20,
                        random_state=SEMILLA+11,stratify=ytr)
                    valores=[]
                    for candidato in rejillas[alg]:
                        for parametro in rejillas[alg][candidato]:
                            m=crear_modelo(alg,len(etiquetas))
                            m.set_params(**{candidato:parametro})
                            m.fit(xtr[it],ytr[it])
                            pr=m.predict(xtr[iv]);valores.append((f1_score(ytr[iv],pr,average='macro',zero_division=0),candidato,parametro))
                    mejor=max(valores,key=lambda x:x[0])
                    modelo.set_params(**{mejor[1]:mejor[2]})
                    detalles_cv.append({'tarea':tarea,'representacion':nombre,'clasificador':alg,
                                       'f1_validacion_interna':mejor[0],
                                       'mejores_parametros':str({mejor[1]:mejor[2]})})
                modelo.fit(xtr,ytr)
                p=modelo.predict(xte)
                predicciones[key]=p
                fila={'tarea':tarea,'representacion':nombre,'clasificador':alg,'n_train':len(ytr),
                    'n_test':len(yte),'optimizado_hmvt':optimizar,
                    'accuracy':accuracy_score(yte,p),
                    'balanced_accuracy':balanced_accuracy_score(yte,p),
                    'precision_macro':precision_score(yte,p,average='macro',zero_division=0),
                    'recall_macro':recall_score(yte,p,average='macro',zero_division=0),
                    'f1_macro':f1_score(yte,p,average='macro',zero_division=0),
                    'precision_weighted':precision_score(yte,p,average='weighted',zero_division=0),
                    'recall_weighted':recall_score(yte,p,average='weighted',zero_division=0),
                    'f1_weighted':f1_score(yte,p,average='weighted',zero_division=0),
                    'tiempo_s':round(time.time()-inicio,2),'error':''}
                metricas.append(fila)
                informe=classification_report(yte,p,labels=etiquetas_id,output_dict=True,zero_division=0)
                for j,et in enumerate(etiquetas):
                    item=informe[str(j)]
                    por_clase.append({'tarea':tarea,'representacion':nombre,'clasificador':alg,'clase':et,
                                      'precision':item['precision'],'recall':item['recall'],
                                      'f1_score':item['f1-score'],'soporte':item['support']})
                cm=confusion_matrix(yte,p,labels=etiquetas_id)
                base=f'{tarea}_{nombre}_{alg}'.replace('+','_MAS_')
                path=SALIDA/'matrices_confusion';path.mkdir(exist_ok=True)
                pd.DataFrame(cm,index=[str(e) for e in etiquetas],columns=[str(e) for e in etiquetas]).to_csv(path/(base+'.csv'))
                fig,ax=plt.subplots(figsize=(6.5,5.3) if tarea=='BINARIA' else (8,6.8))
                sns.heatmap(cm,annot=True,fmt='d',cmap='Purples',xticklabels=etiquetas,
                            yticklabels=etiquetas,ax=ax,cbar=False)
                ax.set(title=f'{tarea} · {nombre} · {alg}\nTEST 20% (textos únicos)',xlabel='Predicha',ylabel='Real')
                plt.xticks(rotation=40,ha='right');plt.yticks(rotation=0)
                fig.tight_layout();fig.savefig(path/(base+'.png'),dpi=155,bbox_inches='tight')
                plt.close(fig)
            except Exception as ex:
                metricas.append({'tarea':tarea,'representacion':nombre,'clasificador':alg,
                                 'optimizado_hmvt':optimizar,'error':str(ex),'tiempo_s':round(time.time()-inicio,2)})
                print('ERROR',tarea,nombre,alg,str(ex)[:160])
        ok=[v for v in metricas if v['tarea']==tarea and v['representacion']==nombre and 'f1_macro' in v]
        if ok:
            k=max(ok,key=lambda z:z['f1_macro'])
            print(f'{tarea:10s} {nombre:20s} F1 macro mas alto en TEST: {k["clasificador"]}={k["f1_macro"]:.3f} (descriptivo, no seleccion final)')

res=pd.DataFrame(metricas)
res.to_csv(SALIDA/'metricas_completas_80_20.csv',index=False)
pd.DataFrame(por_clase).to_csv(SALIDA/'precision_recall_f1_por_clase.csv',index=False)
pd.DataFrame(detalles_cv).to_csv(SALIDA/'optimizacion_SOLO_HMVT_CV_train.csv',index=False)
validos=res.dropna(subset=['f1_macro'])
print('\nRESULTADOS TEST - todas las metricas:')
print(validos[['tarea','representacion','clasificador','accuracy','balanced_accuracy',
              'precision_macro','recall_macro','f1_macro','precision_weighted','recall_weighted','f1_weighted']].to_string(index=False))


from IPython.display import display
for tarea_tabla in ['BINARIA','MULTICLASE']:
    print('\n====================',tarea_tabla,'====================')
    for grupo,representaciones in [
        ('1 CLÁSICAS',BASES),('2 HMVT',HMVT),
        ('3 CON VECTOR DE REGIONALISMOS',[x+'+REG' for x in BASES+HMVT])]:
        sub=validos[(validos.tarea==tarea_tabla)&validos.representacion.isin(representaciones)].copy()
        sub['orden']=sub.representacion.map({x:i for i,x in enumerate(representaciones)})
        sub=sub.sort_values(['orden','clasificador'])
        print('\n',grupo)
        display(sub[['representacion','clasificador','precision_macro','recall_macro','f1_macro',
                     'accuracy','balanced_accuracy']].style.format(precision=4).background_gradient(
                    subset=['f1_macro'],cmap='Purples'))
        sub.to_csv(SALIDA/(tarea_tabla+'_'+grupo.split()[0]+'_tabla.csv'),index=False)
if validos.empty:raise RuntimeError('Ningun modelo se entreno. Ver errores')




def pred_fija(tarea):
    preferencias=[(tarea,'HMVT-V4+REG','RL'),(tarea,'HMVT-V4','RL'),(tarea,'TFIDF','RL')]
    return next((k for k in preferencias if k in predicciones),None)

def pred_preferida_por_modelo(tarea,alg):
    preferencias=[(tarea,'HMVT-V4+REG',alg),(tarea,'HMVT-V4',alg),(tarea,'TFIDF',alg)]
    return next((k for k in preferencias if k in predicciones),None)

from IPython.display import display,Image
resumen_dir=SALIDA/'matrices_resumen_por_modelo'
resumen_dir.mkdir(exist_ok=True)
for tarea,(col,etiquetas) in tareas.items():
    yreal=te[col].map({x:i for i,x in enumerate(etiquetas)}).to_numpy() if tarea=='MULTICLASE' else te.violencia_binaria.to_numpy()
    key=pred_fija(tarea)
    if key:
        print('\nModelo de mapa PREFIJADO:',key,'\n',classification_report(
            yreal,predicciones[key],labels=np.arange(len(etiquetas)),
            target_names=[str(k) for k in etiquetas],zero_division=0))
    print(f'\nMATRICES RESUMEN POR MODELO · {tarea}')
    for alg in MODELOS:
        key_alg=pred_preferida_por_modelo(tarea,alg)
        if not key_alg:
            continue
        rep_sel=key_alg[1]
        print(f'  - {alg}: usando {rep_sel}')
        p_img=SALIDA/'matrices_confusion'/('_'.join(key_alg).replace('+','_MAS_')+'.png')
        p_csv=SALIDA/'matrices_confusion'/('_'.join(key_alg).replace('+','_MAS_')+'.csv')
        if p_img.exists():
            destino=(resumen_dir/f'{tarea}_{alg}_{rep_sel}'.replace('+','_MAS_'))
            import shutil
            shutil.copy2(p_img,destino.with_suffix('.png'))
            if p_csv.exists():
                shutil.copy2(p_csv,destino.with_suffix('.csv'))
            if MOSTRAR_GRAFICAS:
                display(Image(filename=str(p_img)))





geo_test=df[df.texto_norm.isin(set(te.texto_norm))].drop_duplicates(['texto_norm','estado']).copy()
geo_test=geo_test.merge(te[['texto_norm']],on='texto_norm',how='inner')
for tarea,col in [('BINARIA','pred_binaria'),('MULTICLASE','pred_multiclase')]:
    key=pred_fija(tarea)
    if key:
        p=predicciones[key]
        if tarea=='MULTICLASE':p=np.array(tareas[tarea][1],dtype=object)[p]
        dic=dict(zip(te.texto_norm,p))
        geo_test[col]=geo_test.texto_norm.map(dic)
geo_test.to_csv(SALIDA/'predicciones_geograficas_SOLO_TEST.csv',index=False)

for nivel in ['region','estado']:
    filas=[]
    for lugar,part in geo_test.groupby(nivel):
        fila={nivel:lugar,'n_textos':part.texto_norm.nunique(),
              'violencia_real_pct':round(100*part.violencia_binaria.mean(),3)}
        if 'pred_binaria' in part: fila['violencia_predicha_pct']=round(100*part.pred_binaria.mean(),3)
        if 'pred_multiclase' in part:
            for clase in tareas['MULTICLASE'][1]:
                nombre=re.sub('[^a-z0-9]+','_',unicodedata.normalize('NFKD',str(clase)).encode('ascii','ignore').decode().lower()).strip('_')
                fila['pred_'+nombre+'_pct']=round(100*(part.pred_multiclase==clase).mean(),3)
        filas.append(fila)
    pd.DataFrame(filas).to_csv(SALIDA/f'resumen_{nivel}_SOLO_TEST.csv',index=False)

if HACER_MAPAS and len(geo_test)>0:
    import geopandas as gpd, requests
    def slug(s):
        return re.sub('[^a-z0-9]','',unicodedata.normalize('NFKD',str(s)).encode('ascii','ignore').decode().lower())
    alias={'distritofederal':'ciudaddemexico','mexicocity':'ciudaddemexico',
           'veracruzdeignaciodelallave':'veracruz','michoacandeocampo':'michoacan',
           'coahuiladezaragoza':'coahuila','estadodemexico':'mexico','mexicostate':'mexico'}
    try:
        meta=requests.get('https://www.geoboundaries.org/api/current/gbOpen/MEX/ADM1/',timeout=30)
        meta.raise_for_status();info=meta.json()
        url=info.get('simplifiedGeometryGeoJSON') or info.get('gjDownloadURL')
        if not url:raise RuntimeError('Sin geometria GeoJSON')
        path=SALIDA/'adm1_mx.geojson'
        if not path.exists():
            rr=requests.get(url,timeout=120);rr.raise_for_status();path.write_bytes(rr.content)
        mapa=gpd.read_file(path)
        namecol=next(c for c in ['shapeName','NAME_1','name','NOMGEO'] if c in mapa.columns)
        mapa['nombre_estado']=mapa[namecol].astype(str)
        mapa['clave']=mapa.nombre_estado.map(slug).replace(alias)
        estados=pd.read_csv(SALIDA/'resumen_estado_SOLO_TEST.csv')
        estados['clave']=estados.estado.map(slug).replace(alias)
        ge=mapa.merge(estados.drop(columns='estado'),on='clave',how='left')
        
        region_estado=df.groupby('estado').region.agg(lambda a:a.mode().iat[0]).reset_index()
        region_estado['clave']=region_estado.estado.map(slug).replace(alias)
        ge=ge.merge(region_estado[['clave','region']],on='clave',how='left')
        if not (ge.n_textos.notna().any()):raise RuntimeError('Ningun estado coincidió con las geometrías')
        regiones=pd.read_csv(SALIDA/'resumen_region_SOLO_TEST.csv')
        gr=ge[ge.region.notna()].dissolve(by='region').reset_index()[['region','geometry']]
        gr=gr.merge(regiones,on='region',how='left')
        ge.to_file(SALIDA/'estados_con_metricas.gpkg',driver='GPKG')
        gr.to_file(SALIDA/'regiones_con_metricas.gpkg',driver='GPKG')
        
        
        from matplotlib.colors import LinearSegmentedColormap
        import matplotlib.patheffects as pe
        paleta=LinearSegmentedColormap.from_list('academico_morado',
            ['#f3e9fc','#c8ace7','#8d5ac1','#5c2792','#2b0a4d'])
        mascara=mapa[['nombre_estado','geometry']].copy()
        borde_nacional=gpd.GeoSeries([mapa.geometry.union_all()],crs=mapa.crs)
        for nivel,gdf in [('estado',ge),('region',gr)]:
            campos=[c for c in gdf.columns if c.endswith('_pct')]
            carpeta=SALIDA/'mapas'/nivel
            carpeta.mkdir(parents=True,exist_ok=True)
            for campo in campos:
                fig,ax=plt.subplots(figsize=(15,9.5),facecolor='white')
                
                mascara.plot(ax=ax,color='#e6e9ed',edgecolor='#a6aab2',linewidth=.5,zorder=1)
                gdf.plot(column=campo,ax=ax,cmap=paleta,vmin=0,vmax=100,legend=True,
                         missing_kwds={'color':'#e6e9ed','edgecolor':'#a6aab2'},
                         edgecolor='#51336f',linewidth=.72 if nivel=='estado' else 1.45,
                         legend_kwds={'label':'Porcentaje de publicaciones predichas (%)','shrink':.65},zorder=2)
                mascara.boundary.plot(ax=ax,color='#606573',linewidth=.42,alpha=.75,zorder=3)
                if nivel=='region':
                    gdf.boundary.plot(ax=ax,color='#32104e',linewidth=1.85,zorder=4)
                borde_nacional.boundary.plot(ax=ax,color='#1f2330',linewidth=1.2,zorder=5)
                for _,v in gdf[gdf[campo].notna()].iterrows():
                    point=v.geometry.representative_point()
                    label=str(v['nombre_estado'] if nivel=='estado' else v['region'])
                    texto=ax.text(point.x,point.y,label,fontsize=7.5 if nivel=='estado' else 10,
                                  ha='center',va='center',color='#281337',weight='semibold',zorder=6)
                    texto.set_path_effects([pe.withStroke(linewidth=3,foreground='white',alpha=.88)])
                titulo=campo.replace('_pct','').replace('pred_','').replace('_',' ').title()
                if campo=='violencia_predicha_pct':titulo='Violencia digital (binaria)'
                ax.set_title(f'Predicción: {titulo} | {nivel.title()}',loc='left',fontsize=18,
                             fontweight='bold',color='#32104e',pad=20)
                ax.text(.01,.02,'Solo datos del TEST (20% de textos únicos) · Gris: sin registros · Divisiones estatales',
                        transform=ax.transAxes,fontsize=9,color='#5c5565',va='bottom')
                ax.set_axis_off();ax.set_aspect('equal');fig.tight_layout()
                salida_mapa=carpeta/f'{nivel}_{campo}.png'
                fig.savefig(salida_mapa,dpi=270,bbox_inches='tight',facecolor='white')
                if MOSTRAR_MAPAS:plt.show()
                plt.close(fig)
        print('Mapas GeoPandas generados')
    except Exception as ex:
        print('ADVERTENCIA cartografia: no se generaron mapas por',type(ex).__name__,str(ex))


for tarea in tareas:
    tabla=validos[validos.tarea==tarea]
    fig,ax=plt.subplots(figsize=(15,8))
    sns.heatmap(tabla.pivot_table(index='representacion',columns='clasificador',values='f1_macro'),
                annot=True,fmt='.2f',cmap='Purples',vmin=0,vmax=1,ax=ax)
    ax.set_title('F1 macro · '+tarea+' · TEST 20% (DESCRIPTIVO)')
    fig.tight_layout();fig.savefig(SALIDA/f'comparativa_f1_{tarea}.png',dpi=170,bbox_inches='tight')
    if MOSTRAR_GRAFICAS:plt.show()
    plt.close(fig)
perfectos=validos[(validos.accuracy>=.99999)&(validos.f1_macro>=.99999)]
print('\nModelos con desempeno perfecto:',len(perfectos))
if len(perfectos):
    print('ALERTA: 1.0 no demuestra generalizacion; verificar datos sinteticos, textos formulaicos,')
    print('sesgo de anotacion y resultados en otro dataset independiente. No forzamos scores artificialmente.')

info={'archivo':nombre_archivo,'filas_originales':len(df),'textos_unicos':len(corpus),
      'train_unicos':len(tr),'test_unicos':len(te),'test_size':TEST_SIZE,
      'semilla':SEMILLA,'optimizacion':'solo HMVT: validacion interna 80/20 del TRAIN',
      'baselines':'parametros fijos, sin GridSearchCV','representaciones':list(rep),'fuga':'REGION y ESTADO NO son atributos; regionalismo se usa como variable anotada',
      'modelo_geo_prefijado':'RL+HMVT-V4+REG',
      'nota':'Región y estado se usan solo para mapas, no para entrenamiento. Vector regionalismo consiste en anotaciones ya disponibles; para predecir nuevos textos se deben extraer de ellos sin conocer ubicación. Los mapas corresponden al test, no a prevalencia poblacional. HMVT V1-V4 requieren comprobar equivalencia con originales.'}
(SALIDA/'metodologia_y_advertencias.json').write_text(json.dumps(info,indent=2,ensure_ascii=False),encoding='utf8')
zip_path=SALIDA.parent/'resultados_violencia_geo.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for p in SALIDA.rglob('*'):
        if p.is_file():z.write(p,arcname=str(p.relative_to(SALIDA)))
print('\nRESULTADOS LISTOS:',SALIDA,'\nZIP:',zip_path)
if 'google.colab' in sys.modules:
    print('Para descargar ZIP: files.download(str(zip_path))')


### EXP-052 · V1, V2, V3, V4

**Archivo de origen:** `27_Octubre.ipynb, celda 20`. **Referencias duplicadas:** 0. **Sintaxis de código:** Válida.

CLASIFICACION GEOLINGUISTICA: ESTADO (12) Y REGION (6)
80/20 AGRUPADO POR TEXTO; CLASICAS, HMVT V1-V4, + REGIONALISMOS
8 CLASIFICADORES, METRICAS, MATRICES MORADAS, MAPAS GEOPANDAS
UNA SOLA CELDA DE GOOGLE COLAB
import sys, os, re, time, js

**Ejecución:** celda independiente o variante experimental; revisar archivos de entrada y dependencias antes de ejecutarla. Los resultados originales no se insertan en esta celda.

In [ ]:
import sys, os, re, time, json, unicodedata, subprocess, importlib.util, zipfile, warnings
from pathlib import Path
from io import BytesIO
warnings.filterwarnings('ignore', category=FutureWarning)

SEMILLA = 42
PRESUPUESTO_MODELOS_SEG = 22 * 60  
MODELOS = ['RL','KNN','SVM','NB','MLP','XGB','RF','ARBOL']
BASES = ['TFIDF','NGRAMAS','BOW','EMBEDDINGS']
HMVT = ['HMVT-V1','HMVT-V2','HMVT-V3','HMVT-V4']
INCLUIR_EMBEDDINGS = True
MOSTRAR_MATRICES = True      
MOSTRAR_MAPAS = True
MAX_MAPAS_INDIVIDUALES = 12
RAIZ = Path('/content' if 'google.colab' in sys.modules else '/mnt/data')
SALIDA = RAIZ / 'resultados_clasificacion_geolinguistica'
SALIDA.mkdir(exist_ok=True, parents=True)


def instalar(modulo, paquete=None):
    if importlib.util.find_spec(modulo) is None:
        subprocess.check_call([sys.executable,'-m','pip','-q','install',paquete or modulo])

for modulo, paquete in [('numpy',None),('pandas',None),('scipy',None),('sklearn','scikit-learn'),
                       ('matplotlib',None),('seaborn',None),('xgboost',None),
                       ('geopandas',None),('requests',None)]:
    instalar(modulo,paquete)
if INCLUIR_EMBEDDINGS:
    instalar('sentence_transformers','sentence-transformers')

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import sparse
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.preprocessing import MinMaxScaler, MultiLabelBinarizer
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (confusion_matrix, classification_report, accuracy_score,
    precision_score, recall_score, f1_score, balanced_accuracy_score)
from IPython.display import display
sns.set_theme(style='white')
np.random.seed(SEMILLA)
TIEMPO_INICIO=time.monotonic()


if 'google.colab' in sys.modules:
    from google.colab import files
    print('SUBE data_hmvt_con_estado.csv')
    subidos=files.upload()
    if len(subidos)!=1: raise ValueError('Sube exactamente un archivo CSV')
    nombre_csv, datos_csv=next(iter(subidos.items()))
    df=pd.read_csv(BytesIO(datos_csv),encoding='utf-8-sig')
else:
    nombre_csv='data_hmvt_con_estado(1).csv'
    df=pd.read_csv(RAIZ/nombre_csv,encoding='utf-8-sig')

columnas={'tweet','estado','region','regionalismo'}
if columnas-set(df.columns):
    raise ValueError('Faltan columnas: '+str(columnas-set(df.columns)))
df=df.dropna(subset=['tweet','estado','region']).copy()
for c in columnas:
    df[c]=df[c].fillna('').astype(str).str.strip()
df=df[(df.tweet!='')&(df.estado!='')&(df.region!='')].reset_index(drop=True)

def quitar_acentos(x):
    return ''.join(c for c in unicodedata.normalize('NFKD',str(x)) if not unicodedata.combining(c))
def norm_texto(x):
    x=str(x).lower().strip()
    x=re.sub(r'https?://\S+|www\.\S+',' URL ',x)
    x=re.sub(r'@\w+',' USUARIO ',x)
    return re.sub(r'\s+',' ',x)
def clave_estado(x):
    return re.sub('[^a-z0-9]','',quitar_acentos(x).lower())
df['texto_norm']=df.tweet.map(norm_texto)


v=df.groupby('estado').region.nunique()
if (v>1).any():
    raise ValueError('Un mismo estado tiene regiones contradictorias: '+str(v[v>1].index.tolist()))
REGIONES=list(df.region.value_counts().index)
ESTADOS=list(df.estado.value_counts().index)
print(f'FILAS={len(df)} | TEXTOS UNICOS={df.texto_norm.nunique()} | ESTADOS={len(ESTADOS)} | REGIONES={len(REGIONES)}')
print('Advertencia: regionalismo NO es un atributo de entrada; solo se buscan expresiones presentes en el texto.')




base=df.drop_duplicates(['texto_norm','estado']).copy().reset_index(drop=True)

gss=GroupShuffleSplit(n_splits=40,test_size=.20,random_state=SEMILLA)
opciones=[]
for it,iv in gss.split(base,groups=base.texto_norm):
    train_tmp,test_tmp=base.iloc[it],base.iloc[iv]
    cubre=(set(train_tmp.estado)==set(ESTADOS) and set(train_tmp.region)==set(REGIONES))
    dist=abs(len(test_tmp)/len(base)-.20)
    opciones.append((not cubre,dist,it,iv))
opciones.sort(key=lambda x:(x[0],x[1]))
if opciones[0][0]: raise ValueError('No se pudo crear train con todos los estados.')
_,_,it,iv=opciones[0]
tr=base.iloc[it].reset_index(drop=True)
te=base.iloc[iv].reset_index(drop=True)
assert set(tr.texto_norm).isdisjoint(set(te.texto_norm))
print(f'TRAIN={len(tr)} TEST={len(te)}; textos diferentes train={tr.texto_norm.nunique()}, test={te.texto_norm.nunique()}')
print('Importante: la división es aproximadamente 80/20 por filas, agrupada por texto.')
tr.to_csv(SALIDA/'train.csv',index=False)
te.to_csv(SALIDA/'test.csv',index=False)


rep={}
def poner(n,a,b):
    rep[n]=(sparse.csr_matrix(a,dtype=np.float32),sparse.csr_matrix(b,dtype=np.float32))
    print(f'{n:20s}: {rep[n][0].shape[1]} atributos')
for n,v in [('TFIDF',TfidfVectorizer(strip_accents='unicode',ngram_range=(1,2),max_features=5000)),
            ('NGRAMAS',TfidfVectorizer(analyzer='char',ngram_range=(3,5),max_features=5500)),
            ('BOW',CountVectorizer(strip_accents='unicode',max_features=5000))]:
    poner(n,v.fit_transform(tr.texto_norm),v.transform(te.texto_norm))

if INCLUIR_EMBEDDINGS:
    from sentence_transformers import SentenceTransformer
    m=SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')
    
    unicos=list(dict.fromkeys(pd.concat([tr.texto_norm,te.texto_norm]).tolist()))
    embs=m.encode(unicos,batch_size=48,show_progress_bar=True,normalize_embeddings=True)
    d=dict(zip(unicos,embs))
    x1=np.stack([d[x] for x in tr.texto_norm]);x2=np.stack([d[x] for x in te.texto_norm])
    scaler=MinMaxScaler().fit(x1)
    poner('EMBEDDINGS',np.maximum(0,scaler.transform(x1)),np.maximum(0,scaler.transform(x2)))
else:
    print('EMBEDDINGS desactivados: las variantes HMVT-V3/V4 no estarán disponibles.')



componentes={
 'HMVT-V1':[('TFIDF',1.),('NGRAMAS',1.)],
 'HMVT-V2':[('TFIDF',1.),('BOW',.30),('NGRAMAS',.70)],
 'HMVT-V3':[('TFIDF',1.),('NGRAMAS',.70),('EMBEDDINGS',2.)],
 'HMVT-V4':[('TFIDF',1.),('NGRAMAS',.70),('BOW',.30),('EMBEDDINGS',2.)]
}
for n,partes in componentes.items():
    if all(k in rep for k,w in partes):
        poner(n,*[sparse.hstack([rep[k][j]*w for k,w in partes],format='csr') for j in (0,1)])




vocab=sorted(set(v.strip().lower() for v in tr.regionalismo if v.strip()))
patrones=[(v,re.compile(r'(?<!\w)'+re.escape(v)+r'(?!\w)',re.I)) for v in vocab]
def detectar(texto):
    s=quitar_acentos(str(texto).lower())
    return [v for v,pat in patrones if pat.search(s)]

patrones=[(v,re.compile(r'(?<!\w)'+re.escape(quitar_acentos(v))+r'(?!\w)',re.I)) for v in vocab]
geo_train=[detectar(x) for x in tr.tweet]
geo_test=[detectar(x) for x in te.tweet]
mlb=MultiLabelBinarizer(classes=vocab,sparse_output=True)
g1=mlb.fit_transform(geo_train).astype(np.float32)
g2=mlb.transform(geo_test).astype(np.float32)
print(f'Vector regionalismos: {len(vocab)} términos aprendidos en TRAIN. '
      f'Textos con alguna coincidencia: train {sum(bool(x) for x in geo_train)}/{len(tr)}, '
      f'test {sum(bool(x) for x in geo_test)}/{len(te)}')
for n in BASES+HMVT:
    if n in rep:
        a,b=rep[n]
        poner(n+'+REG',sparse.hstack([a,g1],format='csr'),sparse.hstack([b,g2],format='csr'))


from xgboost import XGBClassifier
def crear_modelo(k,nclases):
    if k=='RL': return LogisticRegression(max_iter=500,random_state=SEMILLA)
    if k=='KNN': return KNeighborsClassifier(n_neighbors=3)
    if k=='SVM': return LinearSVC(max_iter=1300,random_state=SEMILLA)
    if k=='NB': return MultinomialNB()
    if k=='MLP': return MLPClassifier(hidden_layer_sizes=(32,),max_iter=65,early_stopping=True,random_state=SEMILLA)
    if k=='XGB': return XGBClassifier(n_estimators=50,max_depth=3,learning_rate=.1,
        random_state=SEMILLA,tree_method='hist',n_jobs=2,eval_metric='mlogloss')
    if k=='RF': return RandomForestClassifier(n_estimators=60,n_jobs=2,random_state=SEMILLA)
    if k=='ARBOL':return DecisionTreeClassifier(random_state=SEMILLA)
    raise ValueError(k)
rejilla={'RL':('C',[.3,1.]),'KNN':('n_neighbors',[3,5]),'SVM':('C',[.3,1.]),
 'NB':('alpha',[.5,1.]),'MLP':('alpha',[.001,.01]),'XGB':('max_depth',[2,3]),
 'RF':('max_depth',[7,None]),'ARBOL':('max_depth',[6,None])}

orden=[n for n in BASES+HMVT if n in rep]+[n+'+REG' for n in BASES+HMVT if n+'+REG' in rep]
metricas=[]; por_clase=[]; predicciones={}; errors=[]; cv_detalles=[]
matdir=SALIDA/'matrices_csv'; matdir.mkdir(exist_ok=True)

def orden_modelo(n):return MODELOS.index(n)
for tarea in ('ESTADO','REGION'):
    col='estado' if tarea=='ESTADO' else 'region'
    etiquetas=ESTADOS if tarea=='ESTADO' else REGIONES
    codigo={x:i for i,x in enumerate(etiquetas)}
    ytr=tr[col].map(codigo).to_numpy(); yte=te[col].map(codigo).to_numpy()
    print('\n'+'='*65+'\nTAREA:',tarea,'| clases:',len(etiquetas),'\n'+'='*65)
    for nombre in orden:
        xtr,xte=rep[nombre]
        for alg in MODELOS:
            if time.monotonic()-TIEMPO_INICIO>PRESUPUESTO_MODELOS_SEG:
                errors.append({'tarea':tarea,'representacion':nombre,'modelo':alg,'error':'Presupuesto de tiempo agotado'})
                continue
            try:
                modelo=crear_modelo(alg,len(etiquetas))
                hmvt=nombre.startswith('HMVT-')
                if hmvt:
                    
                    a,b=train_test_split(np.arange(len(tr)),test_size=.20,random_state=SEMILLA+17,
                                         stratify=ytr if np.min(np.bincount(ytr))>=2 else None)
                    parametro,valores=rejilla[alg]
                    evaluaciones=[]
                    for v in valores:
                        candidato=crear_modelo(alg,len(etiquetas))
                        candidato.set_params(**{parametro:v})
                        candidato.fit(xtr[a],ytr[a]); pred_val=candidato.predict(xtr[b])
                        evaluaciones.append((f1_score(ytr[b],pred_val,average='macro',zero_division=0),v))
                    mejor=max(evaluaciones,key=lambda t:t[0])
                    modelo.set_params(**{parametro:mejor[1]})
                    cv_detalles.append({'tarea':tarea,'representacion':nombre,'modelo':alg,
                        'parametro':parametro,'valor':str(mejor[1]),'f1_validacion_interna':mejor[0]})
                modelo.fit(xtr,ytr)
                yp=modelo.predict(xte)
                predicciones[(tarea,nombre,alg)]=yp
                cm=confusion_matrix(yte,yp,labels=range(len(etiquetas)))
                pd.DataFrame(cm,index=etiquetas,columns=etiquetas).to_csv(matdir/f'{tarea}_{nombre}_{alg}.csv')
                fila={'tarea':tarea,'representacion':nombre,'modelo':alg,
                      'precision_macro':precision_score(yte,yp,average='macro',zero_division=0),
                      'recall_macro':recall_score(yte,yp,average='macro',zero_division=0),
                      'f1_macro':f1_score(yte,yp,average='macro',zero_division=0),
                      'accuracy':accuracy_score(yte,yp),
                      'balanced_accuracy':balanced_accuracy_score(yte,yp),
                      'f1_weighted':f1_score(yte,yp,average='weighted',zero_division=0),
                      'n_train':len(ytr),'n_test':len(yte),'optimizado':hmvt}
                metricas.append(fila)
                cr=classification_report(yte,yp,labels=range(len(etiquetas)),output_dict=True,zero_division=0)
                for i,et in enumerate(etiquetas):
                    o=cr[str(i)]
                    por_clase.append({'tarea':tarea,'representacion':nombre,'modelo':alg,
                       'clase':et,'precision':o['precision'],'recall':o['recall'],
                       'f1':o['f1-score'],'soporte':o['support']})
            except Exception as e:
                errors.append({'tarea':tarea,'representacion':nombre,'modelo':alg,'error':str(e)})
                print('ERROR',tarea,nombre,alg,str(e)[:140])
        print('Finalizado:',tarea,nombre)

res=pd.DataFrame(metricas)
if res.empty: raise RuntimeError('Ningún entrenamiento exitoso. Revisa errores.')
res.to_csv(SALIDA/'metricas_completas.csv',index=False)
pd.DataFrame(por_clase).to_csv(SALIDA/'metricas_por_clase.csv',index=False)
pd.DataFrame(cv_detalles).to_csv(SALIDA/'optimizacion_solo_HMVT.csv',index=False)
pd.DataFrame(errors).to_csv(SALIDA/'errores_y_pendientes.csv',index=False)


for tarea in ('ESTADO','REGION'):
    print('\n'+'#'*72+'\nRESULTADOS DE CLASIFICACION POR',tarea)
    for titulo,nombres in [('1. TRADICIONALES',BASES),('2. HMVT V1-V4',HMVT),
                           ('3. CLASICAS + HMVT CON VECTOR DE REGIONALISMOS',[n+'+REG' for n in BASES+HMVT])]:
        s=res[(res.tarea==tarea)&res.representacion.isin(nombres)].copy()
        s['__r']=s.representacion.map({x:i for i,x in enumerate(nombres)})
        s['__m']=s.modelo.map(orden_modelo)
        s=s.sort_values(['__r','__m'])
        print('\n',titulo)
        cols=['representacion','modelo','precision_macro','recall_macro','f1_macro','accuracy','balanced_accuracy']
        if not s.empty:
            display(s[cols].style.format(precision=4).background_gradient(cmap='Purples',subset=['f1_macro']))
        s[cols].to_csv(SALIDA/f'{tarea}_{titulo.split(".")[0]}_tabla.csv',index=False)


figdir=SALIDA/'matrices_moradas'; figdir.mkdir(exist_ok=True)
for tarea in ('ESTADO','REGION'):
    etiquetas=ESTADOS if tarea=='ESTADO' else REGIONES
    for alg in MODELOS:
        fig,ejes=plt.subplots(4,4,figsize=(25,24) if tarea=='ESTADO' else (22,20))
        for ax in ejes.flat:ax.axis('off')
        for i,nombre in enumerate(orden):
            key=(tarea,nombre,alg)
            if key not in predicciones: continue
            ax=ejes.flat[i];ax.axis('on')
            r=res[(res.tarea==tarea)&(res.representacion==nombre)&(res.modelo==alg)].iloc[0]
            cm=pd.read_csv(matdir/f'{tarea}_{nombre}_{alg}.csv',index_col=0).values
            sns.heatmap(cm,annot=True,fmt='d',cmap='Purples',cbar=False,ax=ax,
                xticklabels=etiquetas,yticklabels=etiquetas,linewidths=.35,linecolor='white',
                annot_kws={'size':6 if tarea=='ESTADO' else 9})
            ax.set_title(f'{nombre}\nF1={r.f1_macro:.3f} | Acc={r.accuracy:.3f}',fontsize=10,color='#3b0764',weight='bold')
            ax.tick_params(axis='x',labelrotation=70,labelsize=6 if tarea=='ESTADO' else 8)
            ax.tick_params(axis='y',labelrotation=0,labelsize=6 if tarea=='ESTADO' else 8)
            ax.set_xlabel('Predicha',fontsize=8);ax.set_ylabel('Real',fontsize=8)
        fig.suptitle(f'Matrices de confusión reales · {tarea} · {alg}\n'
                     'Clásicas / HMVT / + regionalismos detectados · TEST agrupado por texto',
                     fontsize=19,color='#3b0764',weight='bold')
        fig.tight_layout(rect=(0,0,1,.95))
        fig.savefig(figdir/f'{tarea}_{alg}.png',dpi=170,bbox_inches='tight')
        if MOSTRAR_MATRICES:plt.show()
        plt.close(fig)



mapdir=SALIDA/'mapas';mapdir.mkdir(exist_ok=True)
def modelo_mapa(tarea):
    for nombre,alg in [('HMVT-V4+REG','RL'),('HMVT-V4','RL'),('TFIDF+REG','RL'),('TFIDF','RL')]:
        key=(tarea,nombre,alg)
        if key in predicciones: return key
    return None

for tarea in ('ESTADO','REGION'):
    key=modelo_mapa(tarea)
    if key is not None:
        etiquetas=ESTADOS if tarea=='ESTADO' else REGIONES
        te[f'pred_{tarea.lower()}']=np.asarray(etiquetas,dtype=object)[predicciones[key]]
        print('Modelo fijo usado en mapas',tarea,key)
te.to_csv(SALIDA/'predicciones_test_por_estado_y_region.csv',index=False)

try:
    import geopandas as gpd
    import requests
    import matplotlib.patheffects as pe
    from matplotlib.colors import ListedColormap, Normalize
    from matplotlib.cm import ScalarMappable
    geojson=SALIDA/'mexico_adm1.geojson'
    if not geojson.exists():
        m=requests.get('https://www.geoboundaries.org/api/current/gbOpen/MEX/ADM1/',timeout=40)
        m.raise_for_status();meta=m.json()
        url=meta.get('simplifiedGeometryGeoJSON') or meta.get('gjDownloadURL')
        if not url: raise ValueError('No hay geometría ADM1')
        r=requests.get(url,timeout=100);r.raise_for_status();geojson.write_bytes(r.content)
    adm=gpd.read_file(geojson)
    col_nombre=next(x for x in ['shapeName','NAME_1','name','NOMGEO'] if x in adm.columns)
    adm['clave']=adm[col_nombre].map(clave_estado)
    alias={'distritofederal':'ciudaddemexico','mexicocity':'ciudaddemexico',
        'veracruzdeignaciodelallave':'veracruz','michoacandeocampo':'michoacan',
        'coahuiladezaragoza':'coahuila','estadodemexico':'mexico','mexicostate':'mexico'}
    adm['clave']=adm.clave.replace(alias)
    
    estado_region=df[['estado','region']].drop_duplicates()
    estado_region['clave']=estado_region.estado.map(clave_estado).replace(alias)
    adm=adm.merge(estado_region[['clave','region']],on='clave',how='left')
    limites=adm[['geometry']].copy()
    borde=gpd.GeoSeries([adm.geometry.union_all()],crs=adm.crs)
    from matplotlib.colors import LinearSegmentedColormap
    paleta=LinearSegmentedColormap.from_list('morado_academico',
        ['#efe9fb','#baa3e1','#8662ba','#56348e','#291047'])

    def dibujar_mapa(gdf,campo,etiqueta,titulo,nivel,base,contornos):
        if campo not in gdf or not gdf[campo].notna().any(): return
        fig,ax=plt.subplots(figsize=(14,9.5),facecolor='white')
        
        base.plot(ax=ax,color='#e9ebef',edgecolor='#8e94a2',linewidth=.42,zorder=1)
        gdf.plot(ax=ax,column=campo,cmap=paleta,vmin=0,vmax=100,
                 edgecolor='#4f4264',linewidth=.8,zorder=2)
        contornos.boundary.plot(ax=ax,color='#767080',linewidth=.5,zorder=3)
        if nivel=='region':gdf.boundary.plot(ax=ax,color='#2c1643',linewidth=1.65,zorder=4)
        borde.boundary.plot(ax=ax,color='#25202c',linewidth=1.15,zorder=5)
        for _,fila in gdf[gdf[campo].notna()].iterrows():
            pt=fila.geometry.representative_point()
            texto=ax.text(pt.x,pt.y,str(fila[etiqueta]),fontsize=7 if nivel=='estado' else 9,
                ha='center',va='center',color='#291047',weight='semibold',zorder=6)
            texto.set_path_effects([pe.withStroke(linewidth=2.7,foreground='white')])
        barra=ScalarMappable(norm=Normalize(vmin=0,vmax=100),cmap=paleta)
        fig.colorbar(barra,ax=ax,shrink=.67,pad=.015,label='% de predicciones del grupo geográfico')
        ax.set_title(titulo,fontsize=17,weight='bold',color='#291047',loc='left',pad=18)
        ax.set_axis_off();ax.set_aspect('equal')
        ax.text(.01,.015,'Gris: sin datos · Predicciones en TEST · Límites políticos estatales',
                transform=ax.transAxes,fontsize=9,color='#5a5365')
        fig.tight_layout()
        fig.savefig(mapdir/f'{nivel}_{campo}.png',dpi=260,bbox_inches='tight',facecolor='white')
        if MOSTRAR_MAPAS:plt.show()
        plt.close(fig)

    for tarea in ('ESTADO','REGION'):
        predcol=f'pred_{tarea.lower()}'
        if predcol not in te.columns:continue
        clave='estado' if tarea=='ESTADO' else 'region'
        categorias=ESTADOS if tarea=='ESTADO' else REGIONES
        
        agrup=te.groupby(clave,observed=True)
        total=agrup.size().rename('n_test').reset_index()
        
        aciertos=(te[clave]==te[predcol]).astype(float)
        total['exactitud']=total[clave].map(100*aciertos.groupby(te[clave]).mean())
        for cat in categorias:
            nombre='pred_'+clave_estado(cat)+'_pct'
            porcentaje=(te[predcol]==cat).astype(float)
            total[nombre]=total[clave].map(100*porcentaje.groupby(te[clave]).mean())
        total.to_csv(SALIDA/f'resumen_predicciones_{clave}.csv',index=False)
        if tarea=='ESTADO':
            total['join']=total.estado.map(clave_estado).replace(alias)
            gd=adm.merge(total.drop(columns='estado'),left_on='clave',right_on='join',how='left')
            label=col_nombre
        else:
            
            georeg=adm[adm.region.notna()].dissolve(by='region').reset_index()
            gd=georeg.merge(total,on='region',how='left');label='region'
        
        gd.to_file(mapdir/f'{clave}_predicciones.gpkg',driver='GPKG')
        dibujar_mapa(gd,'exactitud',label,
            f'Exactitud de la predicción geolingüística · {clave.title()}',clave,adm,limites)
        for cat in categorias[:MAX_MAPAS_INDIVIDUALES]:
            campo='pred_'+clave_estado(cat)+'_pct'
            dibujar_mapa(gd,campo,label,
                f'Predicciones de {cat} (%) · Agrupadas por {clave}',clave,adm,limites)
    print('Mapas generados correctamente')
except Exception as e:
    print('AVISO: mapas no disponibles:',type(e).__name__,str(e))


metodologia={'objetivo':'Predecir ESTADO y REGION desde publicaciones en español',
    'entrada_regionalismos':'solo detección literal en texto, vocabulario construido con TRAIN',
    'no_entrada':'estado, region y regionalismo anotado por fila NO se usan como predictores',
    'split':'GroupShuffleSplit por texto_norm, ~80/20 de filas únicas texto-estado',
    'baselines':'sin tuning','HMVT':'V1-V4 experimentales, validación interna',
    'mapas':'distribución de clase de estado/región predicha, agrupada por procedencia real del TEST',
    'advertencia':'solo 270 textos distintos y etiquetas de estado variables para textos idénticos; evaluar con datos externos'}
(SALIDA/'metodologia.json').write_text(json.dumps(metodologia,ensure_ascii=False,indent=2),encoding='utf-8')
zip_path=RAIZ/'resultados_clasificacion_geolinguistica.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as archivo:
    for p in SALIDA.rglob('*'):
        if p.is_file():archivo.write(p,arcname=p.relative_to(SALIDA))
print('LISTO:',zip_path,'\n',len(res),'modelos entrenados de',len(orden)*len(MODELOS)*2)
if 'google.colab' in sys.modules:
    from google.colab import files
    files.download(str(zip_path))


### EXP-053 · V1, V2, V3, V4

**Archivo de origen:** `27_Octubre.ipynb, celda 22`. **Referencias duplicadas:** 0. **Sintaxis de código:** Válida.

EVALUACION GEOLINGUISTICA DE PUBLICACIONES DE X - MEXICO
UNA SOLA CELDA DE GOOGLE COLAB (pegar todo este archivo)
Estado y region son OBJETIVOS y NUNCA son variables predictoras.
Requiere CSV con: tweet, estado, region, regionalismo (opcion

**Ejecución:** celda independiente o variante experimental; revisar archivos de entrada y dependencias antes de ejecutarla. Los resultados originales no se insertan en esta celda.

In [ ]:
import os, sys, re, json, warnings, subprocess, unicodedata
from pathlib import Path
from collections import Counter
from itertools import product
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.sparse import hstack, csr_matrix, issparse
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.preprocessing import normalize
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_recall_fscore_support,
                             classification_report, confusion_matrix, ConfusionMatrixDisplay)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.base import clone
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
warnings.filterwarnings('ignore', category=FutureWarning)
SEED=42
np.random.seed(SEED)
SALIDA=Path('/content/resultado_estimacion_geolinguistica' if Path('/content').exists() else '/mnt/data/resultado_estimacion_geolinguistica')
SALIDA.mkdir(parents=True,exist_ok=True)



OMITIR_EMBEDDINGS=False

GEOJSON_ESTADOS=None   

try:
    from google.colab import files
    print('SUBE data_hmvt_con_estado.csv')
    cargados=files.upload()
    archivos_csv=[k for k in cargados if k.lower().endswith('.csv')]
    if not archivos_csv: raise ValueError('No se subio ningun CSV.')
    ruta=archivos_csv[0]
except ImportError:
    ruta='/mnt/data/data_hmvt_desbalanceado.csv' 

try: datos=pd.read_csv(ruta, encoding='utf-8-sig')
except UnicodeDecodeError: datos=pd.read_csv(ruta, encoding='latin-1')
datos.columns=[str(c).strip().lower() for c in datos.columns]
for esperado in ['tweet','estado','region']:
    if esperado not in datos.columns: raise ValueError(f'Falta la columna {esperado!r}. Columnas: {datos.columns.tolist()}')
if 'regionalismo' not in datos.columns:
    datos['regionalismo']=''


def texto_norm(x):
    if pd.isna(x): return ''
    return re.sub(r'\s+', ' ',str(x).strip().casefold())
def etiqueta_norm(x):
    if pd.isna(x): return ''
    s=re.sub(r'\s+',' ',str(x).strip())
    if s.casefold() in ('','nan','none','sin_dato','sin dato','desconocido','no identificado','null'):
        return ''
    return s

datos['texto_modelo']=datos['tweet'].map(texto_norm)
datos=datos[datos['texto_modelo'].str.len()>0].copy()
for c in ['estado','region']:
    datos[c]=datos[c].map(etiqueta_norm)
print('\nDIAGNOSTICO: filas originales no vacias',len(datos),'| textos unicos',datos.texto_modelo.nunique())
print('Duplicados exactos texto:',len(datos)-datos.texto_modelo.nunique())



def dataset_tarea(col):
    g=datos.groupby('texto_modelo',sort=False)
    filas=[]; ambiguos=0; sin_etiqueta=0
    for txt,sub in g:
        vals=sorted({x for x in sub[col] if x})
        if len(vals)>1: ambiguos+=1; continue
        if not vals: sin_etiqueta+=1; continue
        filas.append({'tweet':txt,'etiqueta':vals[0]})
    d=pd.DataFrame(filas)
    if d.empty: raise ValueError(f'No hay textos con etiqueta no ambigua para {col}.')
    print(f'\n{col.upper()}: {len(d)} textos etiquetados | ambiguos excluidos={ambiguos} | sin etiqueta={sin_etiqueta}')
    print(d.etiqueta.value_counts().to_string())
    return d



def dividir(d, objetivo):
    vc=d.etiqueta.value_counts(); pocas=vc[vc<2]
    if len(pocas):
        print(f'AVISO {objetivo}: clases con menos de 2 textos, excluidas:',pocas.to_dict())
        d=d[d.etiqueta.isin(vc[vc>=2].index)].copy()
    ncl=d.etiqueta.nunique()
    if ncl<2: raise ValueError(f'Solo hay {ncl} clase(s) evaluables para {objetivo}.')
    ntest=max(int(np.ceil(.2*len(d))),ncl)
    
    if (len(d)-ntest)<ncl: raise ValueError(f'No hay suficientes textos para {ncl} clases en {objetivo}.')
    a,b=train_test_split(np.arange(len(d)),test_size=ntest,random_state=SEED,stratify=d.etiqueta)
    return d.iloc[a].reset_index(drop=True),d.iloc[b].reset_index(drop=True)




def geo_lexico_train(textos_train):
    sub=datos[datos.texto_modelo.isin(set(textos_train))]
    terminos=[]
    for s in sub.regionalismo.dropna().astype(str):
        for t in re.split(r'[,;|]',s):
            t=texto_norm(t)
            if 2<=len(t)<=40 and t not in ('sin_dato','sin dato','ninguno'):
                terminos.append(t)
    vocab=sorted(set(terminos))
    
    patrones=[re.compile(r'(?<!\w)'+re.escape(t)+r'(?!\w)',flags=re.I) for t in vocab]
    return vocab,patrones

def geo_features(textos,patrones):
    
    if not patrones: return csr_matrix((len(textos),0),dtype=np.float32)
    return csr_matrix(np.asarray([[bool(p.search(t)) for p in patrones] for t in textos],dtype=np.float32))

if not OMITIR_EMBEDDINGS:
    try: from sentence_transformers import SentenceTransformer
    except ImportError:
        subprocess.check_call([sys.executable,'-m','pip','-q','install','sentence-transformers'])
        from sentence_transformers import SentenceTransformer
    encoder=SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')
else: encoder=None

def representaciones(train, test):
    tr=train.tweet.tolist(); te=test.tweet.tolist()
    tf=TfidfVectorizer(ngram_range=(1,2),max_features=5000,sublinear_tf=True,strip_accents='unicode')
    ng=TfidfVectorizer(analyzer='char',ngram_range=(2,5),max_features=8000,sublinear_tf=True)
    bow=CountVectorizer(max_features=5000,strip_accents='unicode',binary=False)
    A,Ae=tf.fit_transform(tr),tf.transform(te)
    B,Be=ng.fit_transform(tr),ng.transform(te)
    C,Ce=bow.fit_transform(tr),bow.transform(te)
    d={ 'TFIDF':(A,Ae),'NGRAMAS':(B,Be),'BOW':(C,Ce)}
    if encoder:
        
        E=csr_matrix(encoder.encode(tr,batch_size=32,show_progress_bar=False,normalize_embeddings=True).astype('float32'))
        Ee=csr_matrix(encoder.encode(te,batch_size=32,show_progress_bar=False,normalize_embeddings=True).astype('float32'))
        d['EMBEDDINGS']=(E,Ee)
    
    
    
    def combinar(names):
        return (hstack([d[k][0] for k in names],format='csr'),
                hstack([d[k][1] for k in names],format='csr'))
    d['HMVT-V1']=combinar(['TFIDF','NGRAMAS'])
    d['HMVT-V2']=combinar(['TFIDF','NGRAMAS','BOW'])
    if encoder:
        d['HMVT-V3']=combinar(['TFIDF','NGRAMAS','EMBEDDINGS'])
        d['HMVT-V4']=combinar(['TFIDF','NGRAMAS','BOW','EMBEDDINGS'])
    vocab,patrones=geo_lexico_train(tr)
    print(f'  Terminos de diccionario de regionalismos (TRAIN): {len(vocab)}; solo coincidencias literales')
    G,Ge=geo_features(tr,patrones),geo_features(te,patrones)
    print('  Coincidencias GEO en TRAIN/TEST:',int(G.sum()),int(Ge.sum()))
    if G.shape[1]:
        for key in list(d):
            x,y=d[key]
            d[key+'+GEO']=(hstack([x,G],format='csr'),hstack([y,Ge],format='csr'))
    return d,vocab

try:
    from xgboost import XGBClassifier
    HAY_XGB=True
except ImportError:
    HAY_XGB=False
    print('Aviso: no se encontro xgboost, se omite XGB')

def estimadores(ncl):
    modelos={
        'RL':LogisticRegression(C=1.0,max_iter=2500,class_weight='balanced',random_state=SEED),
        'SVM':LinearSVC(C=1.0,class_weight='balanced',max_iter=10000,random_state=SEED),
        'RF':RandomForestClassifier(n_estimators=150,min_samples_leaf=1,random_state=SEED,n_jobs=-1,class_weight='balanced'),
        'MLP':MLPClassifier(hidden_layer_sizes=(64,),alpha=0.01,max_iter=300,early_stopping=False,random_state=SEED),
        'NB':MultinomialNB(alpha=1.0),
        'KNN':KNeighborsClassifier(n_neighbors=3,metric='cosine'),
        'ARBOL':DecisionTreeClassifier(class_weight='balanced',min_samples_leaf=1,random_state=SEED),
    }
    if HAY_XGB:
        modelos['XGB']=XGBClassifier(n_estimators=100,max_depth=3,learning_rate=0.07,
                                      subsample=0.9,colsample_bytree=0.9,
                                      objective='multi:softprob',num_class=ncl,
                                      eval_metric='mlogloss',tree_method='hist',random_state=SEED,n_jobs=2)
    return modelos

from sklearn.preprocessing import LabelEncoder
resultados=[]; predicciones=[]; descartes=[]; vocabularios={}
for OBJETIVO in ['region','estado']:
    carpeta=SALIDA/OBJETIVO;carpeta.mkdir(exist_ok=True)
    d=dataset_tarea(OBJETIVO)
    
    tr,te=dividir(d,OBJETIVO)
    tr.to_csv(carpeta/'train_textos_sin_duplicados.csv',index=False,encoding='utf-8-sig')
    te.to_csv(carpeta/'test_textos_sin_duplicados.csv',index=False,encoding='utf-8-sig')
    enc=LabelEncoder().fit(tr.etiqueta)
    ytr=enc.transform(tr.etiqueta); yte=enc.transform(te.etiqueta)
    nombres=enc.classes_.tolist(); ncl=len(nombres)
    print(f'\n========== OBJETIVO {OBJETIVO.upper()} =========')
    print(f'Train {len(tr)}; Test {len(te)}; Clases {ncl}; Test:',te.etiqueta.value_counts().to_dict())
    reprs,vocab=representaciones(tr,te)
    vocabularios[OBJETIVO]=vocab
    
    for nombre_rep,(Xtr,Xte) in reprs.items():
        for clasificador,modelo in estimadores(ncl).items():
            try:
                if clasificador=='XGB':
                    if ncl==2:
                        modelo.set_params(objective='binary:logistic',eval_metric='logloss')
                        modelo.get_params().pop('num_class',None)
                    
                
                
                xx,xt=Xtr,Xte
                if clasificador=='NB':
                    xx=Xtr.copy();xt=Xte.copy()
                    xx.data=np.maximum(xx.data,0)
                    xt.data=np.maximum(xt.data,0)
                with warnings.catch_warnings():
                    warnings.simplefilter('ignore',category=UserWarning)
                    if clasificador=='XGB' and ncl==2:
                        modelo=XGBClassifier(n_estimators=100,max_depth=3,learning_rate=0.07,
                                            subsample=0.9,colsample_bytree=0.9,
                                            objective='binary:logistic',eval_metric='logloss',
                                            tree_method='hist',random_state=SEED,n_jobs=2)
                    modelo.fit(xx,ytr)
                    yp=modelo.predict(xt)
                p,r,f,_=precision_recall_fscore_support(yte,yp,average='macro',zero_division=0)
                resultado={'objetivo':OBJETIVO,'representacion':nombre_rep,'clasificador':clasificador,
                           'n_train':len(tr),'n_test':len(te),'clases':ncl,
                           'accuracy':accuracy_score(yte,yp),
                           'balanced_accuracy':balanced_accuracy_score(yte,yp),
                           'precision_macro':p,'recall_macro':r,'f1_macro':f,
                           'n_errores':int(np.sum(yte!=yp))}
                resultados.append(resultado)
                pred=pd.DataFrame({'objetivo':OBJETIVO,'representacion':nombre_rep,
                                   'clasificador':clasificador,'tweet':te.tweet,
                                   'real':te.etiqueta,'predicha':enc.inverse_transform(yp),
                                   'correcta':yte==yp})
                predicciones.append(pred)
                
                cm=confusion_matrix(yte,yp,labels=list(range(ncl)))
                pd.DataFrame(cm,index=nombres,columns=nombres).to_csv(
                    carpeta/f'cm_{nombre_rep}_{clasificador}.csv',encoding='utf-8-sig')
                print(f'{OBJETIVO:6s} {nombre_rep:17s} {clasificador:6s} F1macro={f:.3f} Acc={resultado["accuracy"]:.3f}')
            except Exception as e:
                descartes.append({'objetivo':OBJETIVO,'representacion':nombre_rep,
                                  'clasificador':clasificador,'motivo':str(e)})
                print('ERROR',OBJETIVO,nombre_rep,clasificador,repr(e))

res=pd.DataFrame(resultados)
if res.empty: raise RuntimeError('Ningun modelo termino correctamente, revisa el registro de errores.')
pred=pd.concat(predicciones,ignore_index=True)
res.to_csv(SALIDA/'metricas_estimacion_estado_region.csv',index=False,encoding='utf-8-sig')
pred.to_csv(SALIDA/'predicciones_individuales_estado_region.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(descartes).to_csv(SALIDA/'modelos_con_error.csv',index=False,encoding='utf-8-sig')
with open(SALIDA/'vocabulario_geo_train.json','w',encoding='utf-8') as f:
    json.dump(vocabularios,f,ensure_ascii=False,indent=2)


for objetivo in ['region','estado']:
    carpeta=SALIDA/objetivo
    z=res[res.objetivo==objetivo].copy()
    z=z.sort_values(['f1_macro','accuracy'],ascending=False)
    print(f'\nTOP {objetivo.upper()} EN TEST (descriptivo, no validacion externa):')
    print(z[['representacion','clasificador','f1_macro','accuracy','n_errores']].head(12).to_string(index=False))
    z.head(15).to_csv(carpeta/'top_15_descriptivo.csv',index=False,encoding='utf-8-sig')
    
    piv=z.pivot_table(index='representacion',columns='clasificador',values='f1_macro')
    fig,ax=plt.subplots(figsize=(12,max(6,0.48*len(piv)+2)))
    im=ax.imshow(piv.values,cmap='Purples',vmin=0,vmax=1,aspect='auto')
    ax.set_xticks(range(len(piv.columns)),piv.columns)
    ax.set_yticks(range(len(piv.index)),piv.index)
    for i in range(len(piv.index)):
        for j in range(len(piv.columns)):
            v=piv.iat[i,j]
            if np.isfinite(v): ax.text(j,i,f'{v:.2f}',ha='center',va='center',fontsize=8,
                                     color='white' if v>0.7 else '#270949')
    ax.set_title(f'F1 macro para prediccion de {objetivo} (TEST)')
    fig.colorbar(im,ax=ax,label='F1 macro')
    fig.tight_layout();fig.savefig(carpeta/f'comparativa_{objetivo}_f1.png',dpi=300,bbox_inches='tight');plt.close(fig)
    
    
    a=pred[pred.objetivo==objetivo]
    for rep,clf in [('TFIDF','RL'),('HMVT-V2','RL'),('HMVT-V4','SVM'),('HMVT-V4+GEO','SVM')]:
        q=a[(a.representacion==rep)&(a.clasificador==clf)]
        if q.empty:continue
        cls=sorted(set(q.real)|set(q.predicha))
        cm=confusion_matrix(q.real,q.predicha,labels=cls)
        fig,ax=plt.subplots(figsize=(max(7,len(cls)*.7),max(6,len(cls)*.65)))
        disp=ConfusionMatrixDisplay(cm,display_labels=cls)
        disp.plot(ax=ax,cmap='Purples',xticks_rotation=60,colorbar=False,values_format='d')
        ax.set_title(f'Prediccion de {objetivo}: {rep} + {clf} (N={len(q)})')
        fig.tight_layout();fig.savefig(carpeta/f'matriz_{rep}_{clf}.png',dpi=300,bbox_inches='tight');plt.close(fig)
    
    q=a[(a.representacion=='HMVT-V4')&(a.clasificador=='SVM')]
    if not q.empty:
        tabla=q.groupby('real').agg(n=('correcta','size'),aciertos=('correcta','sum'))
        tabla['errores']=tabla.n-tabla.aciertos
        tabla.to_csv(carpeta/'errores_por_clase_HMVT_V4_SVM.csv',encoding='utf-8-sig')
        fig,ax=plt.subplots(figsize=(max(9,len(tabla)*.8),5))
        x=np.arange(len(tabla));ax.bar(x-.2,tabla.aciertos,.4,label='Aciertos')
        ax.bar(x+.2,tabla.errores,.4,label='Errores')
        ax.set_xticks(x,tabla.index,rotation=60,ha='right')
        ax.set_ylabel('Textos');ax.set_title(f'Errores al estimar {objetivo}: HMVT-V4 + SVM')
        ax.legend();fig.tight_layout()
        fig.savefig(carpeta/'errores_por_clase.png',dpi=300,bbox_inches='tight');plt.close(fig)



for objetivo in ['region','estado']:
    q=pred[(pred.objetivo==objetivo)&(pred.representacion=='HMVT-V4')&(pred.clasificador=='SVM')]
    if not q.empty:
        conteos=q.groupby('predicha').size().rename('n_predicciones').reset_index()
        conteos['porcentaje']=100*conteos.n_predicciones/len(q)
        conteos.to_csv(SALIDA/objetivo/f'conteo_predicciones_{objetivo}_para_QGIS.csv',index=False,encoding='utf-8-sig')



if GEOJSON_ESTADOS:
    try:
        import geopandas as gpd
        gdf=gpd.read_file(GEOJSON_ESTADOS)
        posibles=[c for c in gdf.columns if c.lower() in ('nom_ent','name','estado','nomgeo','nombre')]
        if not posibles: raise ValueError('La cartografia necesita un campo con nombre de estado: '+str(gdf.columns.tolist()))
        campo=posibles[0]
        q=pred[(pred.objetivo=='estado')&(pred.representacion=='HMVT-V4')&(pred.clasificador=='SVM')]
        tabla=q.groupby('real')['correcta'].agg(['mean','size']).reset_index()
        
        def clave(s): return ''.join(c for c in unicodedata.normalize('NFD',str(s).casefold()) if unicodedata.category(c)!='Mn').strip()
        gdf['_join']=gdf[campo].map(clave)
        tabla['_join']=tabla.real.map(clave)
        joined=gdf.merge(tabla,on='_join',how='left')
        fig,ax=plt.subplots(figsize=(12,9))
        joined.plot(column='mean',ax=ax,cmap='Purples',legend=True,edgecolor='#666',linewidth=.4,
                    vmin=0,vmax=1,missing_kwds={'color':'#dddddd'})
        ax.axis('off');ax.set_title('Fraccion de publicaciones correctamente asociadas al estado (HMVT-V4 + SVM)')
        fig.savefig(SALIDA/'estado'/'mapa_exactitud_por_estado.png',dpi=300,bbox_inches='tight');plt.close(fig)
        print('Mapa con fronteras geograficas generado')
    except Exception as e: print('Cartografia opcional no procesada:',e)

with open(SALIDA/'LEEME.txt','w',encoding='utf-8') as f:
    f.write('ESTIMACION GEOLINGUISTICA DE ESTADO Y REGION\n'
            'Estado y region son variables OBJETIVO, no rasgos predictivos.\n'
            'Los textos con multiples etiquetas del objetivo fueron excluidos de ESA tarea.\n'
            'Los porcentajes describen el conjunto TEST; no demuestran ubicacion real ni generalizacion.\n'
            'El vocabulario de regionalismos se extrajo del TRAIN y se verifico por coincidencia literal en texto.\n'
            'Las variantes HMVT siguen: V1 TFIDF+NGRAMAS; V2 V1+BOW; V3 V1+EMBEDDINGS; V4 V2+EMBEDDINGS.\n'
            'No se usa estado o region como predictor ni se optimizan hiperparametros con TEST.\n'
            'Las graficas para mapas necesitan una cartografia externa, de lo contrario hay CSV listos para QGIS.\n'
            'La evaluacion es sobre etiquetas anotadas; no verifica localizacion fisica del autor.\n')
import shutil
zip_path=shutil.make_archive(str(SALIDA),'zip',SALIDA)
print('\nARCHIVOS:', SALIDA, '\nZIP:',zip_path)
try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    pass


### EXP-054 · V1, V2, V3, V4

**Archivo de origen:** `27_Octubre.ipynb, celda 25`. **Referencias duplicadas:** 0. **Sintaxis de código:** Válida.

EVALUACION GEOLINGUISTICA DE PUBLICACIONES DE X - MEXICO
UNA SOLA CELDA DE GOOGLE COLAB - CONSERVA REGISTROS DUPLICADOS
Estado y region son OBJETIVOS y NUNCA son variables predictoras.
Se mantienen las filas duplicadas; la division se hace 

**Ejecución:** celda independiente o variante experimental; revisar archivos de entrada y dependencias antes de ejecutarla. Los resultados originales no se insertan en esta celda.

In [ ]:
import os, sys, re, json, warnings, subprocess, unicodedata
from pathlib import Path
from collections import Counter
from itertools import product
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.sparse import hstack, csr_matrix, issparse
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.preprocessing import normalize
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_recall_fscore_support,
                             classification_report, confusion_matrix, ConfusionMatrixDisplay)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.base import clone
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
warnings.filterwarnings('ignore', category=FutureWarning)
SEED=42
np.random.seed(SEED)
SALIDA=Path('/content/resultado_estimacion_geolinguistica' if Path('/content').exists() else '/mnt/data/resultado_estimacion_geolinguistica')
SALIDA.mkdir(parents=True,exist_ok=True)



OMITIR_EMBEDDINGS=False

GEOJSON_ESTADOS=None   

try:
    from google.colab import files
    print('SUBE data_hmvt_con_estado.csv')
    cargados=files.upload()
    archivos_csv=[k for k in cargados if k.lower().endswith('.csv')]
    if not archivos_csv: raise ValueError('No se subio ningun CSV.')
    ruta=archivos_csv[0]
except ImportError:
    ruta='/mnt/data/data_hmvt_desbalanceado.csv' 

try: datos=pd.read_csv(ruta, encoding='utf-8-sig')
except UnicodeDecodeError: datos=pd.read_csv(ruta, encoding='latin-1')
datos.columns=[str(c).strip().lower() for c in datos.columns]
for esperado in ['tweet','estado','region']:
    if esperado not in datos.columns: raise ValueError(f'Falta la columna {esperado!r}. Columnas: {datos.columns.tolist()}')
if 'regionalismo' not in datos.columns:
    datos['regionalismo']=''


def texto_norm(x):
    if pd.isna(x): return ''
    return re.sub(r'\s+', ' ',str(x).strip().casefold())
def etiqueta_norm(x):
    if pd.isna(x): return ''
    s=re.sub(r'\s+',' ',str(x).strip())
    if s.casefold() in ('','nan','none','sin_dato','sin dato','desconocido','no identificado','null'):
        return ''
    return s

datos['texto_modelo']=datos['tweet'].map(texto_norm)
datos=datos[datos['texto_modelo'].str.len()>0].copy()
for c in ['estado','region']:
    datos[c]=datos[c].map(etiqueta_norm)
print('\nDIAGNOSTICO: filas originales no vacias',len(datos),'| textos unicos',datos.texto_modelo.nunique())
print('Duplicados exactos texto:',len(datos)-datos.texto_modelo.nunique())




def dataset_tarea(col):
    ds=datos[datos[col].ne('')].copy()
    grupos=ds.groupby('texto_modelo')[col].nunique(dropna=True)
    ambiguos=set(grupos[grupos>1].index)
    excluidas=ds.texto_modelo.isin(ambiguos).sum()
    ds=ds[~ds.texto_modelo.isin(ambiguos)].copy()
    if ds.empty: raise ValueError(f'No hay filas evaluables para {col}')
    ds['tweet']=ds['texto_modelo']
    ds['etiqueta']=ds[col]
    print(f'\n{col.upper()}: {len(ds)} FILAS retenidas; {ds.texto_modelo.nunique()} textos unicos')
    print(f'  Textos con etiquetas contradictorias: {len(ambiguos)}; filas apartadas: {excluidas}')
    print('  Distribucion de FILAS:',ds.etiqueta.value_counts().to_dict())
    print('  Distribucion de TEXTOS UNICOS:',ds.drop_duplicates('texto_modelo').etiqueta.value_counts().to_dict())
    return ds



def dividir(d, objetivo):
    grupos=d.drop_duplicates('texto_modelo')[['texto_modelo','etiqueta']].copy()
    vc=grupos.etiqueta.value_counts()
    clases_insuf=vc[vc<2]
    if len(clases_insuf):
        print(f'AVISO {objetivo}: clases con <2 textos distintos excluidas:',clases_insuf.to_dict())
        d=d[d.etiqueta.isin(vc[vc>=2].index)].copy()
        grupos=grupos[grupos.etiqueta.isin(vc[vc>=2].index)].copy()
    ncl=grupos.etiqueta.nunique()
    if ncl<2: raise ValueError(f'Solo hay {ncl} clases evaluables para {objetivo}')
    ntest=max(int(np.ceil(.2*len(grupos))),ncl)
    if len(grupos)-ntest<ncl: raise ValueError('No hay suficientes textos distintos para particion estratificada')
    a,b=train_test_split(grupos.texto_modelo.to_numpy(),test_size=ntest,
                         random_state=SEED,stratify=grupos.etiqueta)
    conjunto_train=set(a); conjunto_test=set(b)
    assert conjunto_train.isdisjoint(conjunto_test), 'FUGA DE TEXTO ENTRE PARTICIONES'
    tr=d[d.texto_modelo.isin(conjunto_train)].reset_index(drop=True)
    te=d[d.texto_modelo.isin(conjunto_test)].reset_index(drop=True)
    print(f'  TRAIN: {len(tr)} filas / {len(conjunto_train)} textos distintos')
    print(f'  TEST: {len(te)} filas / {len(conjunto_test)} textos distintos')
    print('  Importante: el 80/20 se aplica a textos distintos, NO a cantidad de filas.')
    return tr,te




def geo_lexico_train(textos_train):
    sub=datos[datos.texto_modelo.isin(set(textos_train))]
    terminos=[]
    for s in sub.regionalismo.dropna().astype(str):
        for t in re.split(r'[,;|]',s):
            t=texto_norm(t)
            if 2<=len(t)<=40 and t not in ('sin_dato','sin dato','ninguno'):
                terminos.append(t)
    vocab=sorted(set(terminos))
    
    patrones=[re.compile(r'(?<!\w)'+re.escape(t)+r'(?!\w)',flags=re.I) for t in vocab]
    return vocab,patrones

def geo_features(textos,patrones):
    
    if not patrones: return csr_matrix((len(textos),0),dtype=np.float32)
    return csr_matrix(np.asarray([[bool(p.search(t)) for p in patrones] for t in textos],dtype=np.float32))

if not OMITIR_EMBEDDINGS:
    try: from sentence_transformers import SentenceTransformer
    except ImportError:
        subprocess.check_call([sys.executable,'-m','pip','-q','install','sentence-transformers'])
        from sentence_transformers import SentenceTransformer
    encoder=SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')
else: encoder=None

def representaciones(train, test):
    tr=train.tweet.tolist(); te=test.tweet.tolist()
    tf=TfidfVectorizer(ngram_range=(1,2),max_features=5000,sublinear_tf=True,strip_accents='unicode')
    ng=TfidfVectorizer(analyzer='char',ngram_range=(2,5),max_features=8000,sublinear_tf=True)
    bow=CountVectorizer(max_features=5000,strip_accents='unicode',binary=False)
    A,Ae=tf.fit_transform(tr),tf.transform(te)
    B,Be=ng.fit_transform(tr),ng.transform(te)
    C,Ce=bow.fit_transform(tr),bow.transform(te)
    d={ 'TFIDF':(A,Ae),'NGRAMAS':(B,Be),'BOW':(C,Ce)}
    if encoder:
        
        E=csr_matrix(encoder.encode(tr,batch_size=32,show_progress_bar=False,normalize_embeddings=True).astype('float32'))
        Ee=csr_matrix(encoder.encode(te,batch_size=32,show_progress_bar=False,normalize_embeddings=True).astype('float32'))
        d['EMBEDDINGS']=(E,Ee)
    
    
    
    def combinar(names):
        return (hstack([d[k][0] for k in names],format='csr'),
                hstack([d[k][1] for k in names],format='csr'))
    d['HMVT-V1']=combinar(['TFIDF','NGRAMAS'])
    d['HMVT-V2']=combinar(['TFIDF','NGRAMAS','BOW'])
    if encoder:
        d['HMVT-V3']=combinar(['TFIDF','NGRAMAS','EMBEDDINGS'])
        d['HMVT-V4']=combinar(['TFIDF','NGRAMAS','BOW','EMBEDDINGS'])
    vocab,patrones=geo_lexico_train(tr)
    print(f'  Terminos de diccionario de regionalismos (TRAIN): {len(vocab)}; solo coincidencias literales')
    G,Ge=geo_features(tr,patrones),geo_features(te,patrones)
    print('  Coincidencias GEO en TRAIN/TEST:',int(G.sum()),int(Ge.sum()))
    if G.shape[1]:
        for key in list(d):
            x,y=d[key]
            d[key+'+GEO']=(hstack([x,G],format='csr'),hstack([y,Ge],format='csr'))
    return d,vocab

try:
    from xgboost import XGBClassifier
    HAY_XGB=True
except ImportError:
    HAY_XGB=False
    print('Aviso: no se encontro xgboost, se omite XGB')

def estimadores(ncl):
    modelos={
        'RL':LogisticRegression(C=1.0,max_iter=2500,class_weight='balanced',random_state=SEED),
        'SVM':LinearSVC(C=1.0,class_weight='balanced',max_iter=10000,random_state=SEED),
        'RF':RandomForestClassifier(n_estimators=150,min_samples_leaf=1,random_state=SEED,n_jobs=-1,class_weight='balanced'),
        'MLP':MLPClassifier(hidden_layer_sizes=(64,),alpha=0.01,max_iter=300,early_stopping=False,random_state=SEED),
        'NB':MultinomialNB(alpha=1.0),
        'KNN':KNeighborsClassifier(n_neighbors=3,metric='cosine'),
        'ARBOL':DecisionTreeClassifier(class_weight='balanced',min_samples_leaf=1,random_state=SEED),
    }
    if HAY_XGB:
        modelos['XGB']=XGBClassifier(n_estimators=100,max_depth=3,learning_rate=0.07,
                                      subsample=0.9,colsample_bytree=0.9,
                                      objective='multi:softprob',num_class=ncl,
                                      eval_metric='mlogloss',tree_method='hist',random_state=SEED,n_jobs=2)
    return modelos

from sklearn.preprocessing import LabelEncoder
resultados=[]; predicciones=[]; descartes=[]; vocabularios={}
for OBJETIVO in ['region','estado']:
    carpeta=SALIDA/OBJETIVO;carpeta.mkdir(exist_ok=True)
    d=dataset_tarea(OBJETIVO)
    
    tr,te=dividir(d,OBJETIVO)
    tr.to_csv(carpeta/'train_filas_con_duplicados.csv',index=False,encoding='utf-8-sig')
    te.to_csv(carpeta/'test_filas_con_duplicados.csv',index=False,encoding='utf-8-sig')
    enc=LabelEncoder().fit(tr.etiqueta)
    ytr=enc.transform(tr.etiqueta); yte=enc.transform(te.etiqueta)
    nombres=enc.classes_.tolist(); ncl=len(nombres)
    print(f'\n========== OBJETIVO {OBJETIVO.upper()} =========')
    print(f'Train {len(tr)} FILAS; Test {len(te)} FILAS; Clases {ncl}; Test:',te.etiqueta.value_counts().to_dict())
    reprs,vocab=representaciones(tr,te)
    vocabularios[OBJETIVO]=vocab
    
    for nombre_rep,(Xtr,Xte) in reprs.items():
        for clasificador,modelo in estimadores(ncl).items():
            try:
                if clasificador=='XGB':
                    if ncl==2:
                        modelo.set_params(objective='binary:logistic',eval_metric='logloss')
                        modelo.get_params().pop('num_class',None)
                    
                
                
                xx,xt=Xtr,Xte
                if clasificador=='NB':
                    xx=Xtr.copy();xt=Xte.copy()
                    xx.data=np.maximum(xx.data,0)
                    xt.data=np.maximum(xt.data,0)
                with warnings.catch_warnings():
                    warnings.simplefilter('ignore',category=UserWarning)
                    if clasificador=='XGB' and ncl==2:
                        modelo=XGBClassifier(n_estimators=100,max_depth=3,learning_rate=0.07,
                                            subsample=0.9,colsample_bytree=0.9,
                                            objective='binary:logistic',eval_metric='logloss',
                                            tree_method='hist',random_state=SEED,n_jobs=2)
                    modelo.fit(xx,ytr)
                    yp=modelo.predict(xt)
                p,r,f,_=precision_recall_fscore_support(yte,yp,average='macro',zero_division=0)
                resultado={'objetivo':OBJETIVO,'representacion':nombre_rep,'clasificador':clasificador,
                           'n_train':len(tr),'n_test':len(te),
                           'n_textos_train':tr.texto_modelo.nunique(),
                           'n_textos_test':te.texto_modelo.nunique(),'clases':ncl,
                           'accuracy':accuracy_score(yte,yp),
                           'balanced_accuracy':balanced_accuracy_score(yte,yp),
                           'precision_macro':p,'recall_macro':r,'f1_macro':f,
                           'n_errores':int(np.sum(yte!=yp))}
                resultados.append(resultado)
                pred=pd.DataFrame({'objetivo':OBJETIVO,'representacion':nombre_rep,
                                   'clasificador':clasificador,'tweet':te.tweet,
                                   'real':te.etiqueta,'predicha':enc.inverse_transform(yp),
                                   'correcta':yte==yp})
                predicciones.append(pred)
                
                
                unico=pred.drop_duplicates('tweet',keep='first')
                resultado['accuracy_textos_unicos']=accuracy_score(unico.real,unico.predicha)
                resultado['f1_macro_textos_unicos']=precision_recall_fscore_support(
                    unico.real,unico.predicha,average='macro',zero_division=0)[2]
                resultado['errores_textos_unicos']=int((unico.real!=unico.predicha).sum())
                
                cm=confusion_matrix(yte,yp,labels=list(range(ncl)))
                pd.DataFrame(cm,index=nombres,columns=nombres).to_csv(
                    carpeta/f'cm_FILAS_{nombre_rep}_{clasificador}.csv',encoding='utf-8-sig')
                cmu=confusion_matrix(unico.real,unico.predicha,labels=nombres)
                pd.DataFrame(cmu,index=nombres,columns=nombres).to_csv(
                    carpeta/f'cm_TEXTOS_UNICOS_{nombre_rep}_{clasificador}.csv',encoding='utf-8-sig')
                print(f'{OBJETIVO:6s} {nombre_rep:17s} {clasificador:6s} F1macro={f:.3f} Acc={resultado["accuracy"]:.3f}')
            except Exception as e:
                descartes.append({'objetivo':OBJETIVO,'representacion':nombre_rep,
                                  'clasificador':clasificador,'motivo':str(e)})
                print('ERROR',OBJETIVO,nombre_rep,clasificador,repr(e))

res=pd.DataFrame(resultados)
if res.empty: raise RuntimeError('Ningun modelo termino correctamente, revisa el registro de errores.')
pred=pd.concat(predicciones,ignore_index=True)
res.to_csv(SALIDA/'metricas_estimacion_estado_region.csv',index=False,encoding='utf-8-sig')
pred.to_csv(SALIDA/'predicciones_individuales_estado_region.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(descartes).to_csv(SALIDA/'modelos_con_error.csv',index=False,encoding='utf-8-sig')
with open(SALIDA/'vocabulario_geo_train.json','w',encoding='utf-8') as f:
    json.dump(vocabularios,f,ensure_ascii=False,indent=2)


for objetivo in ['region','estado']:
    carpeta=SALIDA/objetivo
    z=res[res.objetivo==objetivo].copy()
    z=z.sort_values(['f1_macro_textos_unicos','accuracy_textos_unicos'],ascending=False)
    print(f'\nTOP {objetivo.upper()} EN TEST (descriptivo, no validacion externa):')
    print(z[['representacion','clasificador','f1_macro_textos_unicos','accuracy_textos_unicos',
             'f1_macro','accuracy','n_errores']].head(12).to_string(index=False))
    z.head(15).to_csv(carpeta/'top_15_descriptivo.csv',index=False,encoding='utf-8-sig')
    
    piv=z.pivot_table(index='representacion',columns='clasificador',values='f1_macro_textos_unicos')
    fig,ax=plt.subplots(figsize=(12,max(6,0.48*len(piv)+2)))
    im=ax.imshow(piv.values,cmap='Purples',vmin=0,vmax=1,aspect='auto')
    ax.set_xticks(range(len(piv.columns)),piv.columns)
    ax.set_yticks(range(len(piv.index)),piv.index)
    for i in range(len(piv.index)):
        for j in range(len(piv.columns)):
            v=piv.iat[i,j]
            if np.isfinite(v): ax.text(j,i,f'{v:.2f}',ha='center',va='center',fontsize=8,
                                     color='white' if v>0.7 else '#270949')
    ax.set_title(f'F1 macro por texto unico para prediccion de {objetivo} (TEST)')
    fig.colorbar(im,ax=ax,label='F1 macro')
    fig.tight_layout();fig.savefig(carpeta/f'comparativa_{objetivo}_f1.png',dpi=300,bbox_inches='tight');plt.close(fig)
    
    
    a=pred[pred.objetivo==objetivo]
    for rep,clf in [('TFIDF','RL'),('HMVT-V2','RL'),('HMVT-V4','SVM'),('HMVT-V4+GEO','SVM')]:
        q=a[(a.representacion==rep)&(a.clasificador==clf)].drop_duplicates('tweet')
        if q.empty:continue
        cls=sorted(set(q.real)|set(q.predicha))
        cm=confusion_matrix(q.real,q.predicha,labels=cls)
        fig,ax=plt.subplots(figsize=(max(7,len(cls)*.7),max(6,len(cls)*.65)))
        disp=ConfusionMatrixDisplay(cm,display_labels=cls)
        disp.plot(ax=ax,cmap='Purples',xticks_rotation=60,colorbar=False,values_format='d')
        ax.set_title(f'Prediccion de {objetivo}: {rep} + {clf} (N={len(q)} filas)')
        fig.tight_layout();fig.savefig(carpeta/f'matriz_{rep}_{clf}.png',dpi=300,bbox_inches='tight');plt.close(fig)
    
    q=a[(a.representacion=='HMVT-V4')&(a.clasificador=='SVM')].drop_duplicates('tweet')
    if not q.empty:
        tabla=q.groupby('real').agg(n=('correcta','size'),aciertos=('correcta','sum'))
        tabla['errores']=tabla.n-tabla.aciertos
        tabla.to_csv(carpeta/'errores_por_clase_HMVT_V4_SVM.csv',encoding='utf-8-sig')
        fig,ax=plt.subplots(figsize=(max(9,len(tabla)*.8),5))
        x=np.arange(len(tabla));ax.bar(x-.2,tabla.aciertos,.4,label='Aciertos')
        ax.bar(x+.2,tabla.errores,.4,label='Errores')
        ax.set_xticks(x,tabla.index,rotation=60,ha='right')
        ax.set_ylabel('Textos');ax.set_title(f'Errores al estimar {objetivo}: HMVT-V4 + SVM')
        ax.legend();fig.tight_layout()
        fig.savefig(carpeta/'errores_por_clase.png',dpi=300,bbox_inches='tight');plt.close(fig)



for objetivo in ['region','estado']:
    q=pred[(pred.objetivo==objetivo)&(pred.representacion=='HMVT-V4')&(pred.clasificador=='SVM')].drop_duplicates('tweet')
    if not q.empty:
        conteos=q.groupby('predicha').size().rename('n_predicciones').reset_index()
        conteos['porcentaje']=100*conteos.n_predicciones/len(q)
        conteos.to_csv(SALIDA/objetivo/f'conteo_predicciones_{objetivo}_para_QGIS.csv',index=False,encoding='utf-8-sig')



if GEOJSON_ESTADOS:
    try:
        import geopandas as gpd
        gdf=gpd.read_file(GEOJSON_ESTADOS)
        posibles=[c for c in gdf.columns if c.lower() in ('nom_ent','name','estado','nomgeo','nombre')]
        if not posibles: raise ValueError('La cartografia necesita un campo con nombre de estado: '+str(gdf.columns.tolist()))
        campo=posibles[0]
        q=pred[(pred.objetivo=='estado')&(pred.representacion=='HMVT-V4')&(pred.clasificador=='SVM')].drop_duplicates('tweet')
        tabla=q.groupby('real')['correcta'].agg(['mean','size']).reset_index()
        
        def clave(s): return ''.join(c for c in unicodedata.normalize('NFD',str(s).casefold()) if unicodedata.category(c)!='Mn').strip()
        gdf['_join']=gdf[campo].map(clave)
        tabla['_join']=tabla.real.map(clave)
        joined=gdf.merge(tabla,on='_join',how='left')
        fig,ax=plt.subplots(figsize=(12,9))
        joined.plot(column='mean',ax=ax,cmap='Purples',legend=True,edgecolor='#666',linewidth=.4,
                    vmin=0,vmax=1,missing_kwds={'color':'#dddddd'})
        ax.axis('off');ax.set_title('Fraccion de publicaciones correctamente asociadas al estado (HMVT-V4 + SVM)')
        fig.savefig(SALIDA/'estado'/'mapa_exactitud_por_estado.png',dpi=300,bbox_inches='tight');plt.close(fig)
        print('Mapa con fronteras geograficas generado')
    except Exception as e: print('Cartografia opcional no procesada:',e)

with open(SALIDA/'LEEME.txt','w',encoding='utf-8') as f:
    f.write('ESTIMACION GEOLINGUISTICA DE ESTADO Y REGION CON DUPLICADOS CONSERVADOS\n'
            'Estado y region son variables OBJETIVO, no rasgos predictivos.\n'
            'Se conservan todas las filas repetidas con etiqueta coherente.\n'
            'La particion 80/20 es POR GRUPOS DE TEXTO, sin fuga de textos identicos.\n'
            'Los textos con etiquetas contradictorias del objetivo se apartan de ESA tarea.\n'
            'Se exportan metricas por FILA y por TEXTO UNICO; para tesis priorizar estas ultimas.\n'
            'Los porcentajes describen el conjunto TEST; no demuestran ubicacion real ni generalizacion.\n'
            'El vocabulario de regionalismos se extrajo del TRAIN y se verifico por coincidencia literal en texto.\n'
            'Las variantes HMVT siguen: V1 TFIDF+NGRAMAS; V2 V1+BOW; V3 V1+EMBEDDINGS; V4 V2+EMBEDDINGS.\n'
            'No se usa estado o region como predictor ni se optimizan hiperparametros con TEST.\n'
            'Las graficas para mapas necesitan una cartografia externa, de lo contrario hay CSV listos para QGIS.\n'
            'La evaluacion es sobre etiquetas anotadas; no verifica localizacion fisica del autor.\n')
import shutil
zip_path=shutil.make_archive(str(SALIDA),'zip',SALIDA)
print('\nARCHIVOS:', SALIDA, '\nZIP:',zip_path)
try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    pass


### EXP-055 · V1, V2, V3, V4

**Archivo de origen:** `27_Octubre.ipynb, celda 28`. **Referencias duplicadas:** 0. **Sintaxis de código:** Válida.

GOOGLE COLAB - ESTIMACION GEO POR PROBABILIDAD MAXIMA
Conserva TODAS las publicaciones, incluso duplicadas y ambiguas.
Una sola etiqueta predicha por texto: argmax P(estado|texto) / P(region|texto)
NO usa estado ni region verdaderos como ca

**Ejecución:** celda independiente o variante experimental; revisar archivos de entrada y dependencias antes de ejecutarla. Los resultados originales no se insertan en esta celda.

In [ ]:
import os, re, sys, json, warnings, subprocess, shutil, unicodedata
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.sparse import csr_matrix, hstack
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.preprocessing import normalize, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
from sklearn.base import clone
warnings.filterwarnings('ignore', category=FutureWarning)
SEED = 42
np.random.seed(SEED)
OUT = Path('/content/estimacion_geo_probabilidades' if Path('/content').exists() else '/mnt/data/estimacion_geo_probabilidades')
OUT.mkdir(parents=True, exist_ok=True)
USAR_EMBEDDINGS = True  
INCLUIR_XGBOOST = True

try:
    from google.colab import files
    print('SUBE data_hmvt_con_estado.csv')
    up=files.upload()
    ruta=next((x for x in up if x.lower().endswith('.csv')),None)
    if ruta is None: raise ValueError('Debes subir un archivo CSV')
except ImportError:
    ruta='/mnt/data/data_hmvt_con_estado.csv'
if not Path(ruta).exists(): raise FileNotFoundError(ruta)
try: df=pd.read_csv(ruta,encoding='utf-8-sig')
except UnicodeDecodeError: df=pd.read_csv(ruta,encoding='latin-1')
df.columns=[str(c).strip().lower() for c in df.columns]
if 'tweet' not in df.columns:
    for alternativa in ['texto','text','publicacion']:
        if alternativa in df.columns: df=df.rename(columns={alternativa:'tweet'}); break
for col in ('tweet','estado','region'):
    if col not in df: raise ValueError(f'Falta columna {col}. Disponibles: {list(df.columns)}')
if 'regionalismo' not in df.columns: df['regionalismo']=''

def norm(x):
    if pd.isna(x): return ''
    return re.sub(r'\s+',' ',str(x).casefold()).strip()
def label(x):
    if pd.isna(x): return ''
    s=str(x).strip()
    return '' if s.casefold() in ['','nan','none','null','sin_dato','sin dato','desconocido'] else s

df=df.copy()
df['texto_norm']=df.tweet.map(norm)
df=df.loc[df.texto_norm.ne('')].reset_index(drop=True)
df['id_fila_original']=np.arange(len(df))
for k in ('estado','region'):df[k]=df[k].map(label)
print(f'Corpus: {len(df)} filas válidas; {df.texto_norm.nunique()} textos distintos; NO se eliminan filas.')



groups=df.texto_norm.drop_duplicates().tolist()
tr_grp,te_grp=train_test_split(groups,test_size=0.20,random_state=SEED)
tr_set=set(tr_grp); te_set=set(te_grp)
assert not (tr_set & te_set)
df['particion']=np.where(df.texto_norm.isin(tr_set),'TRAIN','TEST')
print('Partición:',df.particion.value_counts().to_dict(), '| textos distintos:',len(tr_set),len(te_set))

def referencias(objetivo):
    return df.groupby('texto_norm')[objetivo].agg(lambda s:tuple(sorted({v for v in s if v}))).to_dict()




train_df=df[df.particion.eq('TRAIN')]
terminos=set()
for val in train_df.regionalismo.fillna('').astype(str):
    for t in re.split(r'[,;|]',val):
        t=norm(t)
        if 2<=len(t)<=50 and t not in {'sin dato','sin_dato','ninguno'}:terminos.add(t)
terminos=sorted(terminos)
patrones=[re.compile(r'(?<!\w)'+re.escape(t)+r'(?!\w)',re.I) for t in terminos]
def geo_mat(texts):
    if not patrones:return csr_matrix((len(texts),0),dtype=np.float32)
    return csr_matrix(np.array([[int(bool(p.search(t))) for p in patrones] for t in texts],dtype=np.float32))
print('Regionalismos del vocabulario de entrenamiento:',len(terminos))




texts_train=list(tr_grp); texts_test=list(te_grp); all_texts=texts_train+texts_test
vec_t=TfidfVectorizer(ngram_range=(1,2),strip_accents='unicode',max_features=5000,sublinear_tf=True)
vec_n=TfidfVectorizer(analyzer='char',ngram_range=(2,5),max_features=8000,sublinear_tf=True)
vec_b=CountVectorizer(strip_accents='unicode',max_features=5000)
T=vec_t.fit_transform(texts_train);Tt=vec_t.transform(texts_test)
N=vec_n.fit_transform(texts_train);Nt=vec_n.transform(texts_test)
B=vec_b.fit_transform(texts_train);Bt=vec_b.transform(texts_test)
G=geo_mat(texts_train);Gt=geo_mat(texts_test)
reps={'TFIDF':(T,Tt),'NGRAMAS':(N,Nt),'BOW':(B,Bt)}
if USAR_EMBEDDINGS:
    try:from sentence_transformers import SentenceTransformer
    except ImportError:
        subprocess.check_call([sys.executable,'-m','pip','-q','install','sentence-transformers'])
        from sentence_transformers import SentenceTransformer
    encoder=SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')
    E=csr_matrix(encoder.encode(texts_train,batch_size=32,show_progress_bar=True,normalize_embeddings=True).astype('float32'))
    Et=csr_matrix(encoder.encode(texts_test,batch_size=32,show_progress_bar=True,normalize_embeddings=True).astype('float32'))
    reps['EMBEDDINGS']=(E,Et)
def unir(keys):return (hstack([reps[k][0] for k in keys],format='csr'),hstack([reps[k][1] for k in keys],format='csr'))
reps['HMVT-V1']=unir(['TFIDF','NGRAMAS'])
reps['HMVT-V2']=unir(['TFIDF','NGRAMAS','BOW'])
if 'EMBEDDINGS' in reps:
    reps['HMVT-V3']=unir(['TFIDF','NGRAMAS','EMBEDDINGS'])
    reps['HMVT-V4']=unir(['TFIDF','NGRAMAS','BOW','EMBEDDINGS'])
for k in list(reps):
    a,b=reps[k]
    reps[k+'+GEO']=(hstack([a,G],format='csr'),hstack([b,Gt],format='csr'))

try:
    from xgboost import XGBClassifier
    HAVE_XGB=INCLUIR_XGBOOST
except ImportError:HAVE_XGB=False

def algoritmos(ncl):
    dic={
      'RL':LogisticRegression(C=1,max_iter=3000,class_weight='balanced',random_state=SEED),
      'SVM':SVC(kernel='linear',C=1,class_weight='balanced',probability=True,random_state=SEED),
      'RF':RandomForestClassifier(n_estimators=120,class_weight='balanced',random_state=SEED,n_jobs=-1),
      'MLP':MLPClassifier(hidden_layer_sizes=(64,),alpha=0.01,max_iter=350,random_state=SEED),
      'NB':MultinomialNB(alpha=1),
      'KNN':KNeighborsClassifier(n_neighbors=3,metric='cosine'),
      'ARBOL':DecisionTreeClassifier(class_weight='balanced',random_state=SEED),
    }
    if HAVE_XGB:dic['XGB']=XGBClassifier(n_estimators=100,max_depth=3,learning_rate=.07,
            subsample=.9,colsample_bytree=.9,tree_method='hist',eval_metric='mlogloss',random_state=SEED,n_jobs=2)
    return dic

results=[]; predictions=[]; problems=[]; audit=[]
for objetivo in ['estado','region']:
    print('\n'+'='*65+'\nOBJETIVO:',objetivo.upper())
    dest=OUT/objetivo;dest.mkdir(exist_ok=True)
    refs=referencias(objetivo)
    clean_tr=[i for i,t in enumerate(texts_train) if len(refs[t])==1]
    amb_tr=[i for i,t in enumerate(texts_train) if len(refs[t])!=1]
    clean_te=[i for i,t in enumerate(texts_test) if len(refs[t])==1]
    amb_te=[i for i,t in enumerate(texts_test) if len(refs[t])!=1]
    labels=sorted({v for vals in refs.values() for v in vals})
    enc=LabelEncoder().fit(labels)
    if len(set(refs[texts_train[i]][0] for i in clean_tr))<2:raise ValueError('Se requieren al menos dos clases de entrenamiento inequívocas.')
    known=enc.transform([refs[texts_train[i]][0] for i in clean_tr])
    
    
    
    Xbase=hstack([T,G],format='csr')
    Xtbase=hstack([Tt,Gt],format='csr')
    inicio=LogisticRegression(max_iter=3000,class_weight='balanced',random_state=SEED)
    inicio.fit(Xbase[clean_tr],known)
    ptrain=inicio.predict_proba(Xbase)
    ptest=inicio.predict_proba(Xtbase)
    def prob_full(p):
        z=np.zeros((len(p),len(labels)))
        z[:,inicio.classes_]=p
        return z
    ptrain=prob_full(ptrain);ptest=prob_full(ptest)
    if len(amb_tr):
        print(f'Textos ambiguos/sin etiqueta conservados en TRAIN: {len(amb_tr)}; se asignan por argmax de probabilidad.')
    if len(amb_te):
        print(f'Textos ambiguos/sin etiqueta conservados en TEST: {len(amb_te)}; se predicen sin utilizar sus etiquetas originales.')
    pseudo_codes=ptrain.argmax(axis=1)
    ytrain=[int(enc.transform([refs[t][0]])[0]) if len(refs[t])==1 else int(pseudo_codes[i])
            for i,t in enumerate(texts_train)]
    
    
    ind_train={t:i for i,t in enumerate(texts_train)}
    counts_train=train_df.texto_norm.value_counts()
    freq_train=np.array([counts_train.get(t,0) for t in texts_train],dtype=int)
    expanded=np.repeat(np.arange(len(texts_train)),freq_train)
    yrows=np.array(ytrain)[expanded]
    audit_df=pd.DataFrame({'texto':texts_train,'tipo_etiqueta':['original_unica' if len(refs[t])==1 else 'resuelta_probabilidad' for t in texts_train],
        'etiquetas_originales':[ '|'.join(refs[t]) for t in texts_train],
        'etiqueta_para_entrenar':enc.inverse_transform(np.array(ytrain)),
        'probabilidad_resolucion':ptrain[np.arange(len(texts_train)),np.array(ytrain)],
        'repeticiones':freq_train})
    audit_df.to_csv(dest/'auditoria_etiquetas_train.csv',index=False,encoding='utf-8-sig')
    
    
    test_df=df[df.particion=='TEST']
    test_freq=test_df.texto_norm.value_counts()
    test_rep=np.array([test_freq.get(t,0) for t in texts_test],dtype=int)
    for repr_name,(XX,YY) in reps.items():
        Xrows=XX[expanded]
        for clf_name,clf in algoritmos(len(labels)).items():
            try:
                if clf_name=='NB':
                    Xfit=Xrows.copy();Xeval=YY.copy(); Xfit.data=np.maximum(Xfit.data,0);Xeval.data=np.maximum(Xeval.data,0)
                else:Xfit=Xrows;Xeval=YY
                with warnings.catch_warnings():
                    warnings.simplefilter('ignore')
                    clf.fit(Xfit,yrows)
                    probs=clf.predict_proba(Xeval)
                full=np.zeros((len(texts_test),len(labels)))
                full[:,clf.classes_.astype(int)]=probs
                predc=full.argmax(axis=1)
                predlabels=enc.inverse_transform(predc)
                cleaned_actual=[refs[texts_test[i]][0] for i in clean_te]
                cleaned_pred=[predlabels[i] for i in clean_te]
                if cleaned_actual:
                    prec,rec,f1,_=precision_recall_fscore_support(cleaned_actual,cleaned_pred,average='macro',zero_division=0,labels=labels)
                    acc=accuracy_score(cleaned_actual,cleaned_pred)
                    
                    truth_exp=np.repeat(cleaned_actual,test_rep[clean_te]); pred_exp=np.repeat(cleaned_pred,test_rep[clean_te])
                    f1_rows=precision_recall_fscore_support(truth_exp,pred_exp,average='macro',zero_division=0,labels=labels)[2]
                    acc_rows=accuracy_score(truth_exp,pred_exp)
                else:prec=rec=f1=acc=f1_rows=acc_rows=float('nan')
                
                
                compatibles=[predlabels[i] in refs[texts_test[i]] for i in amb_te if len(refs[texts_test[i]])>0]
                validas=[i for i in amb_te if len(refs[texts_test[i]])>0]
                compat=(sum(compatibles)/len(compatibles) if compatibles else np.nan)
                results.append({'objetivo':objetivo,'representacion':repr_name,'clasificador':clf_name,
                    'n_filas_originales':len(df),'n_filas_train':len(train_df),'n_filas_test':len(test_df),
                    'n_textos_test_total':len(texts_test),'n_textos_test_inequivocos':len(clean_te),
                    'n_textos_test_ambiguos_sin_etiqueta':len(amb_te),
                    'f1_macro_referencia_unica':f1,'accuracy_referencia_unica':acc,
                    'precision_macro_referencia_unica':prec,'recall_macro_referencia_unica':rec,
                    'f1_macro_filas_referencia_unica':f1_rows,'accuracy_filas_referencia_unica':acc_rows,
                    'compatibilidad_candidatos_ambiguos_NO_accuracy':compat})
                order=np.argsort(-full,axis=1)
                rows=pd.DataFrame({'objetivo':objetivo,'representacion':repr_name,'clasificador':clf_name,
                    'texto':texts_test,'etiquetas_originales':['|'.join(refs[t]) for t in texts_test],
                    'referencia_inequivoca':[len(refs[t])==1 for t in texts_test],
                    'etiqueta_estimada':predlabels,'probabilidad_maxima':full[np.arange(len(texts_test)),predc],
                    'segunda_etiqueta':enc.inverse_transform(order[:,1]),'segunda_probabilidad':full[np.arange(len(texts_test)),order[:,1]],
                    'margen_probabilidad':full[np.arange(len(texts_test)),order[:,0]]-full[np.arange(len(texts_test)),order[:,1]],
                    'repeticiones_test':test_rep})
                predictions.append(rows)
                cm=confusion_matrix(cleaned_actual,cleaned_pred,labels=labels)
                pd.DataFrame(cm,index=labels,columns=labels).to_csv(dest/f'CM_{repr_name}_{clf_name}_referencias_unicas.csv',encoding='utf-8-sig')
                print(f'{objetivo:6} {repr_name:17} {clf_name:5} F1={f1:.3f} acc={acc:.3f}; todas las filas conservadas')
            except Exception as ex:
                problems.append({'objetivo':objetivo,'representacion':repr_name,'clasificador':clf_name,'error':repr(ex)})
                print('ERROR',objetivo,repr_name,clf_name,repr(ex))
    
    df.to_csv(dest/'TODAS_las_filas_originales.csv',index=False,encoding='utf-8-sig')
    with open(dest/'clases_estado_o_region.json','w',encoding='utf-8') as f:json.dump(labels,f,ensure_ascii=False,indent=2)

res=pd.DataFrame(results); pred=pd.concat(predictions,ignore_index=True)
res.to_csv(OUT/'metricas_comparativas.csv',index=False,encoding='utf-8-sig')
pred.to_csv(OUT/'predicciones_por_texto.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(problems).to_csv(OUT/'errores_ejecucion.csv',index=False,encoding='utf-8-sig')



joined=[]
for objetivo in ['estado','region']:
    z=res[res.objetivo==objetivo].sort_values('f1_macro_referencia_unica',ascending=False)
    if z.empty:continue
    best=z.iloc[0]
    subset=pred[(pred.objetivo==objetivo)&(pred.representacion==best.representacion)&(pred.clasificador==best.clasificador)].copy()
    subset.to_csv(OUT/objetivo/'predicciones_mejor_modelo_por_texto_TEST.csv',index=False,encoding='utf-8-sig')
    agg=df.merge(subset[['texto','etiqueta_estimada','probabilidad_maxima']],left_on='texto_norm',right_on='texto',how='left')
    
    
    aud=pd.read_csv(OUT/objetivo/'auditoria_etiquetas_train.csv')
    aud=aud[['texto','etiqueta_para_entrenar','probabilidad_resolucion','tipo_etiqueta']]
    agg=agg.merge(aud,left_on='texto_norm',right_on='texto',how='left',suffixes=('','_train'))
    mask=agg['particion'].eq('TRAIN')
    agg.loc[mask,'etiqueta_estimada']=agg.loc[mask,'etiqueta_para_entrenar']
    agg.loc[mask,'probabilidad_maxima']=agg.loc[mask,'probabilidad_resolucion']
    agg['origen_prediccion']=np.where(mask,'TRAIN: original única o pseudoetiqueta probabilística','TEST: predicción del clasificador')
    agg['objetivo']=objetivo
    agg['modelo_elegido_descriptivo']=str(best.representacion)+' / '+str(best.clasificador)
    agg.to_csv(OUT/objetivo/'TODAS_las_filas_con_etiqueta_estimada.csv',index=False,encoding='utf-8-sig')
    subset.groupby('etiqueta_estimada').agg(textos=('texto','size'),filas=('repeticiones_test','sum'),confianza_media=('probabilidad_maxima','mean')).to_csv(
        OUT/objetivo/'conteo_por_lugar_para_mapas.csv',encoding='utf-8-sig')
    print('\nTOP',objetivo.upper(),'F1=',round(best.f1_macro_referencia_unica,4),'modelo=',best.representacion,best.clasificador,
          '| predicciones de TEST',len(subset),'textos conservados')
    print(z[['representacion','clasificador','f1_macro_referencia_unica','accuracy_referencia_unica']].head(10).to_string(index=False))
with open(OUT/'NOTAS_METODOLOGICAS.txt','w',encoding='utf-8') as f:
    f.write('Se conservaron todas las filas. Etiquetas contradictorias de TRAIN se resuelven por el argmax de un modelo probabilistico entrenado SOLO con textos inequívocos de TRAIN. La referencia de TEST se usa exclusivamente para evaluar los casos con etiqueta unica; los ambiguos se informan aparte como compatibilidad con candidatos y no como exactitud. Los porcentajes de confianza no son una probabilidad calibrada de ubicación real. El mejor modelo se elige descriptivamente sobre TEST; requiere validacion independiente. Los mapas deben basarse en predicciones nuevas, no en etiquetas originales.\n')
zip_path=shutil.make_archive(str(OUT),'zip',root_dir=OUT)
print('\nFinalizado. ZIP:',zip_path)
try:
    from google.colab import files
    files.download(zip_path)
except ImportError:pass


### EXP-056 · V1, V2, V3, V4

**Archivo de origen:** `27_Octubre.ipynb, celda 30`. **Referencias duplicadas:** 0. **Sintaxis de código:** Válida.

COLAB: CLASIFICACION GEOLINGUISTICA POR ESTADO Y REGION
80/20 SIN TEXTOS COMPARTIDOS | 8 MODELOS | 16 REPRESENTACIONES
MATRICES MORADAS | MAPAS magma_r | LEYENDAS Y ETIQUETAS
import sys, re, time, warnings, zipfile, subprocess, importlib.ut

**Ejecución:** celda independiente o variante experimental; revisar archivos de entrada y dependencias antes de ejecutarla. Los resultados originales no se insertan en esta celda.

In [ ]:
import sys, re, time, warnings, zipfile, subprocess, importlib.util, unicodedata
from pathlib import Path
from io import BytesIO
warnings.filterwarnings('ignore', category=FutureWarning)
SEED=42
LIMITE_SEG=22*60    
MOSTRAR_MATRICES=False  
MOSTRAR_MAPAS=True
MODELOS=['RL','KNN','SVM','NB','MLP','XGB','RF','ARBOL']
BASES=['TFIDF','NGRAMAS','BOW','EMBEDDINGS']
HMVT=['HMVT-V1','HMVT-V2','HMVT-V3','HMVT-V4']
RAIZ=Path('/content') if 'google.colab' in sys.modules else Path('/mnt/data')
OUT=RAIZ/'resultados_geolinguistica_magma';OUT.mkdir(parents=True,exist_ok=True)

def instalar(m,p=None):
    if importlib.util.find_spec(m) is None:
        subprocess.check_call([sys.executable,'-m','pip','-q','install',p or m])
for a,b in [('numpy',None),('pandas',None),('scipy',None),('sklearn','scikit-learn'),('matplotlib',None),('seaborn',None),('xgboost',None),('sentence_transformers','sentence-transformers'),('geopandas',None),('requests',None)]:
    instalar(a,b)
import numpy as np,pandas as pd,matplotlib.pyplot as plt,seaborn as sns,geopandas as gpd,requests
from scipy import sparse
from sklearn.model_selection import GroupShuffleSplit,train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer,CountVectorizer
from sklearn.preprocessing import MultiLabelBinarizer,MinMaxScaler
from sklearn.metrics import accuracy_score,balanced_accuracy_score,precision_score,recall_score,f1_score,confusion_matrix,classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from sentence_transformers import SentenceTransformer
from matplotlib.colors import ListedColormap,BoundaryNorm
from matplotlib.patches import Patch
from IPython.display import display
sns.set_theme(style='white');np.random.seed(SEED);COMIENZO=time.monotonic()

def sin_acentos(x):
    return ''.join(c for c in unicodedata.normalize('NFKD',str(x)) if not unicodedata.combining(c))
def clave(x):return re.sub('[^a-z0-9]','',sin_acentos(x).lower())
def limpiar(x):
    x=str(x).strip().lower();x=re.sub(r'https?://\S+|www\.\S+',' URL ',x)
    x=re.sub(r'@\w+',' USUARIO ',x)
    return re.sub(r'\s+',' ',x)


if 'google.colab' in sys.modules:
    from google.colab import files
    print('SUBE data_hmvt_con_estado.csv')
    subidos=files.upload()
    if len(subidos)!=1:raise ValueError('Sube exactamente un CSV')
    nombre,bytes_csv=next(iter(subidos.items()))
    df=pd.read_csv(BytesIO(bytes_csv),encoding='utf-8-sig')
else:
    df=pd.read_csv(RAIZ/'data_hmvt_con_estado(1).csv')
requeridas={'tweet','estado','region','regionalismo'}
if requeridas-set(df.columns):raise ValueError('Faltan columnas: '+str(requeridas-set(df.columns)))
df=df.dropna(subset=['tweet','estado','region']).copy()
for c in requeridas:df[c]=df[c].fillna('').astype(str).str.strip()
df=df[(df.tweet!='')&(df.estado!='')&(df.region!='')].copy()
df['texto_norm']=df.tweet.map(limpiar)
if (df.groupby('estado').region.nunique()>1).any():
    raise ValueError('Algunos estados tienen etiquetas regionales contradictorias')
ESTADOS=list(df.estado.value_counts().index)
REGIONES=list(df.region.value_counts().index)

base=df.drop_duplicates(['texto_norm','estado']).reset_index(drop=True)
gss=GroupShuffleSplit(n_splits=40,test_size=.20,random_state=SEED)
opciones=[]
for it,iv in gss.split(base,groups=base.texto_norm):
    cubre=set(base.iloc[it].estado)==set(ESTADOS)
    opciones.append((not cubre,abs(len(iv)/len(base)-.20),it,iv))
opciones.sort(key=lambda x:(x[0],x[1]))
if opciones[0][0]:raise ValueError('No hay partición de entrenamiento con todos los estados')
_,_,it,iv=opciones[0];tr=base.iloc[it].reset_index(drop=True);te=base.iloc[iv].reset_index(drop=True)
assert set(tr.texto_norm).isdisjoint(te.texto_norm)
print('Filas:',len(df),'Textos distintos:',df.texto_norm.nunique(),'TRAIN:',len(tr),'TEST:',len(te))
tr.to_csv(OUT/'train.csv',index=False);te.to_csv(OUT/'test.csv',index=False)


REP={}
def registrar(n,a,b):REP[n]=(sparse.csr_matrix(a,dtype=np.float32),sparse.csr_matrix(b,dtype=np.float32))
for n,v in [('TFIDF',TfidfVectorizer(max_features=5000,ngram_range=(1,2),strip_accents='unicode')),
            ('NGRAMAS',TfidfVectorizer(analyzer='char',ngram_range=(3,5),max_features=5500)),
            ('BOW',CountVectorizer(max_features=5000,strip_accents='unicode'))]:
    registrar(n,v.fit_transform(tr.texto_norm),v.transform(te.texto_norm))
modelo_emb=SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')
textos=list(dict.fromkeys(pd.concat([tr.texto_norm,te.texto_norm]).tolist()))
vectores=modelo_emb.encode(textos,show_progress_bar=True,batch_size=48,normalize_embeddings=True)
dic_emb=dict(zip(textos,vectores));a=np.stack([dic_emb[x] for x in tr.texto_norm]);b=np.stack([dic_emb[x] for x in te.texto_norm])
normalizador=MinMaxScaler().fit(a)
registrar('EMBEDDINGS',np.maximum(0,normalizador.transform(a)),np.maximum(0,normalizador.transform(b)))
del modelo_emb,vectores,dic_emb


BLOQUES={
 'HMVT-V1':[('TFIDF',1),('NGRAMAS',1)],
 'HMVT-V2':[('TFIDF',1),('BOW',.3),('NGRAMAS',.7)],
 'HMVT-V3':[('TFIDF',1),('NGRAMAS',.7),('EMBEDDINGS',2)],
 'HMVT-V4':[('TFIDF',1),('NGRAMAS',.7),('BOW',.3),('EMBEDDINGS',2)]}
for nombre,partes in BLOQUES.items():
    registrar(nombre,*[sparse.hstack([REP[n][i]*w for n,w in partes],format='csr') for i in (0,1)])



vocab=sorted({x.lower() for x in tr.regionalismo if x.strip()})
patrones=[(v,re.compile(r'(?<!\w)'+re.escape(sin_acentos(v))+r'(?!\w)',re.I)) for v in vocab]
def detectar(t):
    s=sin_acentos(str(t).lower())
    return [v for v,p in patrones if p.search(s)]
mlb=MultiLabelBinarizer(classes=vocab,sparse_output=True)
geo_train=mlb.fit_transform([detectar(t) for t in tr.tweet]).astype(np.float32)
geo_test=mlb.transform([detectar(t) for t in te.tweet]).astype(np.float32)
print('Regionalismos en vocabulario:',len(vocab),'Presencia test:',int((geo_test.getnnz(axis=1)>0).sum()))
for n in BASES+HMVT:
    x1,x2=REP[n]
    registrar(n+'+REG',sparse.hstack([x1,geo_train],format='csr'),sparse.hstack([x2,geo_test],format='csr'))



def modelo(k):
    if k=='RL':return LogisticRegression(max_iter=500,random_state=SEED)
    if k=='KNN':return KNeighborsClassifier(n_neighbors=3)
    if k=='SVM':return LinearSVC(max_iter=1300,random_state=SEED)
    if k=='NB':return MultinomialNB()
    if k=='MLP':return MLPClassifier(hidden_layer_sizes=(32,),max_iter=65,early_stopping=True,random_state=SEED)
    if k=='XGB':return XGBClassifier(n_estimators=50,max_depth=3,learning_rate=.1,random_state=SEED,tree_method='hist',n_jobs=2,eval_metric='mlogloss')
    if k=='RF':return RandomForestClassifier(n_estimators=60,random_state=SEED,n_jobs=2)
    if k=='ARBOL':return DecisionTreeClassifier(random_state=SEED)
    raise ValueError(k)
PARAM={'RL':('C',[.3,1.]),'KNN':('n_neighbors',[3,5]),'SVM':('C',[.3,1.]),'NB':('alpha',[.5,1.]),
       'MLP':('alpha',[.001,.01]),'XGB':('max_depth',[2,3]),'RF':('max_depth',[7,None]),'ARBOL':('max_depth',[6,None])}
ORDEN=BASES+HMVT+[n+'+REG' for n in BASES+HMVT]
metricas=[];por_clase=[];predicciones={};fallos=[];optimizacion=[]
matdir=OUT/'matrices_csv';matdir.mkdir(exist_ok=True)
for tarea in ['ESTADO','REGION']:
    col=tarea.lower(); etiquetas=ESTADOS if tarea=='ESTADO' else REGIONES
    ids={x:i for i,x in enumerate(etiquetas)}
    ytr=tr[col].map(ids).to_numpy();yte=te[col].map(ids).to_numpy()
    for r in ORDEN:
        xtr,xte=REP[r]
        for alg in MODELOS:
            if time.monotonic()-COMIENZO>LIMITE_SEG:
                fallos.append((tarea,r,alg,'Presupuesto de tiempo alcanzado'));continue
            try:
                m=modelo(alg)
                if r.startswith('HMVT-'):
                    
                    ia,ib=train_test_split(np.arange(len(ytr)),test_size=.20,random_state=SEED+17,
                        stratify=ytr if np.min(np.bincount(ytr))>=2 else None)
                    parametro,valores=PARAM[alg];resultados_validacion=[]
                    for valor in valores:
                        cand=modelo(alg).set_params(**{parametro:valor})
                        cand.fit(xtr[ia],ytr[ia]);pv=cand.predict(xtr[ib])
                        resultados_validacion.append((f1_score(ytr[ib],pv,average='macro',zero_division=0),valor))
                    score,val=max(resultados_validacion,key=lambda t:t[0]);m.set_params(**{parametro:val})
                    optimizacion.append([tarea,r,alg,parametro,val,score])
                m.fit(xtr,ytr);p=m.predict(xte);predicciones[(tarea,r,alg)]=p
                cm=confusion_matrix(yte,p,labels=np.arange(len(etiquetas)))
                pd.DataFrame(cm,index=etiquetas,columns=etiquetas).to_csv(matdir/f'{tarea}_{r}_{alg}.csv')
                metricas.append([tarea,r,alg,precision_score(yte,p,average='macro',zero_division=0),
                    recall_score(yte,p,average='macro',zero_division=0),f1_score(yte,p,average='macro',zero_division=0),
                    accuracy_score(yte,p),balanced_accuracy_score(yte,p)])
                report=classification_report(yte,p,labels=np.arange(len(etiquetas)),output_dict=True,zero_division=0)
                for i,et in enumerate(etiquetas):
                    d=report[str(i)];por_clase.append([tarea,r,alg,et,d['precision'],d['recall'],d['f1-score'],d['support']])
            except Exception as e:
                fallos.append((tarea,r,alg,str(e)));print('Error:',tarea,r,alg,str(e)[:100])
        print('Terminado',tarea,r)
columnas=['tarea','representacion','modelo','precision_macro','recall_macro','f1_macro','accuracy','balanced_accuracy']
res=pd.DataFrame(metricas,columns=columnas)
if res.empty:raise RuntimeError('No se obtuvo ningún modelo válido')
res.to_csv(OUT/'metricas_completas.csv',index=False)
pd.DataFrame(por_clase,columns=['tarea','representacion','modelo','clase','precision','recall','f1','soporte']).to_csv(OUT/'metricas_por_clase.csv',index=False)
pd.DataFrame(optimizacion,columns=['tarea','representacion','modelo','parametro','valor','f1_validacion']).to_csv(OUT/'optimizacion_HMVT.csv',index=False)
pd.DataFrame(fallos,columns=['tarea','representacion','modelo','motivo']).to_csv(OUT/'pendientes.csv',index=False)
for tarea in ['ESTADO','REGION']:
    print('\nRESULTADOS:',tarea)
    for etiqueta,grupo in [('1 TRADICIONALES',BASES),('2 HMVT',HMVT),('3 CON REGIONALISMOS',[n+'+REG' for n in BASES+HMVT])]:
        print('\n',etiqueta)
        sub=res[(res.tarea==tarea)&res.representacion.isin(grupo)].copy()
        sub['orden']=sub.representacion.map({v:i for i,v in enumerate(grupo)})
        sub=sub.sort_values(['orden','modelo']).drop(columns='orden')
        if len(sub):display(sub.style.format(precision=4).background_gradient(cmap='Purples',subset=['f1_macro']))
        sub.to_csv(OUT/f'{tarea}_{etiqueta.replace(" ","_")}.csv',index=False)


figdir=OUT/'matrices';figdir.mkdir(exist_ok=True)
for tarea in ['ESTADO','REGION']:
    etiquetas=ESTADOS if tarea=='ESTADO' else REGIONES
    for alg in MODELOS:
        fig,axes=plt.subplots(4,4,figsize=(25,23) if tarea=='ESTADO' else (21,19))
        for ax,n in zip(axes.flat,ORDEN):
            key=(tarea,n,alg)
            if key not in predicciones:ax.axis('off');continue
            cm=pd.read_csv(matdir/f'{tarea}_{n}_{alg}.csv',index_col=0).values
            sns.heatmap(cm,annot=True,fmt='d',cmap='Purples',cbar=False,ax=ax,xticklabels=etiquetas,yticklabels=etiquetas,
                        annot_kws={'fontsize':6 if tarea=='ESTADO' else 9},linewidths=.35,linecolor='white')
            fila=res[(res.tarea==tarea)&(res.representacion==n)&(res.modelo==alg)].iloc[0]
            ax.set_title(f'{n}\nF1={fila.f1_macro:.3f} | Acc={fila.accuracy:.3f}',fontsize=10,color='#3b0764')
            ax.tick_params(axis='x',labelrotation=65,labelsize=6 if tarea=='ESTADO' else 8)
            ax.tick_params(axis='y',labelrotation=0,labelsize=6 if tarea=='ESTADO' else 8)
            ax.set_xlabel('Predicha',fontsize=8);ax.set_ylabel('Real',fontsize=8)
        fig.suptitle(f'Clasificación por {tarea.lower()} · {alg} | TEST (20%)',fontsize=20,color='#3b0764',weight='bold')
        fig.tight_layout(rect=(0,0,1,.96));fig.savefig(figdir/f'{tarea}_{alg}.png',dpi=170,bbox_inches='tight')
        if MOSTRAR_MATRICES:plt.show()
        plt.close(fig)



def pred_fija(tarea):
    for n,a in [('HMVT-V4+REG','RL'),('HMVT-V4','RL'),('TFIDF+REG','RL'),('TFIDF','RL')]:
        key=(tarea,n,a)
        if key in predicciones:return key
    return None
for tarea in ['ESTADO','REGION']:
    k=pred_fija(tarea)
    if k:
        et=ESTADOS if tarea=='ESTADO' else REGIONES
        te['pred_'+tarea.lower()]=np.asarray(et,dtype=object)[predicciones[k]]
        print('Modelo cartográfico fijo:',k)
te.to_csv(OUT/'predicciones_test.csv',index=False)



from matplotlib import colormaps
cmap_base=colormaps['magma_r']
COLORES=[cmap_base(x) for x in (.10,.30,.50,.70,.90)]
RANGOS=[0,20,40,60,80,100]
PALETA=ListedColormap(COLORES)
NORMA=BoundaryNorm(RANGOS,PALETA.N)
SIN_DATOS='#e5e7eb'


ETIQUETAS_EXTERNAS={
 'veracruz':(-93.4,22.2),'tabasco':(-90.1,16.1),
 'ciudaddemexico':(-97.3,18.5),'mexico':(-101.6,19.3),
 'tlaxcala':(-96.5,20.4),'morelos':(-99.8,17.0),
 'aguascalientes':(-100.6,23.5),'colima':(-106.2,18.1),
 'queretaro':(-98.4,22.2)
}
ABREVIACIONES={'ciudaddemexico':'CDMX','mexico':'Edo. Méx.',
 'tlaxcala':'Tlax.','aguascalientes':'Ags.','queretaro':'Qro.'}
ALIAS={'distritofederal':'ciudaddemexico','mexicocity':'ciudaddemexico',
 'veracruzdeignaciodelallave':'veracruz','michoacandeocampo':'michoacan',
 'coahuiladezaragoza':'coahuila','estadodemexico':'mexico','mexicostate':'mexico'}

def mapa_tema(gdf,campo,nombre_col,titulo,carpeta,base,borde,por_region=False):
    if campo not in gdf or not gdf[campo].notna().any():return
    fig,ax=plt.subplots(figsize=(17,10),facecolor='white')
    base.plot(ax=ax,color=SIN_DATOS,edgecolor='#a8a8b0',linewidth=.45,zorder=1)
    valores=gdf.copy()
    
    valores['rango']=pd.cut(valores[campo],bins=RANGOS,labels=False,include_lowest=True,right=False)
    valores.loc[valores[campo]==100,'rango']=4
    validos=valores[valores['rango'].notna()].copy()
    if len(validos):
        validos['rango']=validos['rango'].astype(int)
        validos.plot(ax=ax,column='rango',cmap=PALETA,vmin=0,vmax=4,
                     edgecolor='#463951',linewidth=.8 if not por_region else 1.3,zorder=2)
    base.boundary.plot(ax=ax,color='#787783',linewidth=.42,zorder=3)
    if por_region:validos.boundary.plot(ax=ax,color='#342344',linewidth=1.8,zorder=4)
    borde.boundary.plot(ax=ax,color='#1c1427',linewidth=1.2,zorder=5)
    for _,f in validos.iterrows():
        p=f.geometry.representative_point();nom=str(f[nombre_col]);k=clave(nom)
        if not por_region and k in ETIQUETAS_EXTERNAS:
            x,y=ETIQUETAS_EXTERNAS[k]
            ax.annotate(ABREVIACIONES.get(k,nom),xy=(p.x,p.y),xytext=(x,y),
                fontsize=8,fontweight='semibold',ha='center',va='center',color='#24152f',
                bbox=dict(boxstyle='round,pad=.25',fc='white',ec='#a49aaf',lw=.5,alpha=.95),
                arrowprops=dict(arrowstyle='-',color='#4b4550',lw=.8),zorder=7)
        else:
            ax.text(p.x,p.y,nom,fontsize=8 if not por_region else 11,
                ha='center',va='center',fontweight='semibold',color='#24152f',zorder=6,
                bbox=dict(boxstyle='round,pad=.15',fc='white',ec='none',alpha=.7))
    leyenda=[Patch(facecolor=COLORES[i],edgecolor='#4a4350',label=f'{RANGOS[i]}–{RANGOS[i+1]}%') for i in range(5)]
    leyenda.append(Patch(facecolor=SIN_DATOS,edgecolor='#999',label='Sin datos'))
    ax.legend(handles=leyenda,title='% de publicaciones predichas',loc='center left',
              bbox_to_anchor=(1.01,.55),frameon=True,fontsize=10,title_fontsize=10)
    ax.set_title(titulo,loc='left',fontsize=17,fontweight='bold',color='#32104e',pad=17)
    ax.set_axis_off();ax.set_aspect('equal')
    ax.text(.01,.02,'Porcentaje de predicciones en TEST · Gris: sin registros · Límites estatales visibles',
            transform=ax.transAxes,fontsize=9,color='#595162')
    fig.tight_layout();carpeta.mkdir(exist_ok=True,parents=True)
    fig.savefig(carpeta/f'{campo}.png',dpi=300,bbox_inches='tight',facecolor='white')
    if MOSTRAR_MAPAS:plt.show()
    plt.close(fig)

try:
    geo=OUT/'mexico_adm1.geojson'
    if not geo.exists():
        meta=requests.get('https://www.geoboundaries.org/api/current/gbOpen/MEX/ADM1/',timeout=40)
        meta.raise_for_status();info=meta.json()
        url=info.get('simplifiedGeometryGeoJSON') or info.get('gjDownloadURL')
        if not url:raise RuntimeError('No se encontró geometría ADM1')
        resp=requests.get(url,timeout=90);resp.raise_for_status();geo.write_bytes(resp.content)
    adm=gpd.read_file(geo).to_crs('EPSG:4326')
    col=next(c for c in ('shapeName','NAME_1','name','NOMGEO') if c in adm.columns)
    adm['clave']=adm[col].map(clave).replace(ALIAS)
    ubicaciones=df[['estado','region']].drop_duplicates().copy()
    ubicaciones['clave']=ubicaciones.estado.map(clave).replace(ALIAS)
    adm=adm.merge(ubicaciones[['clave','region']],on='clave',how='left',validate='one_to_one')
    base_mx=adm[['geometry']].copy()
    borde_mx=gpd.GeoSeries([adm.geometry.union_all()],crs=adm.crs)
    for tarea in ['ESTADO','REGION']:
        pred='pred_'+tarea.lower();real=tarea.lower()
        if pred not in te:continue
        clases=ESTADOS if tarea=='ESTADO' else REGIONES
        resumen=te.groupby(real).size().rename('n_test').reset_index()
        aciertos=(te[real]==te[pred]).astype(float)
        resumen['exactitud_pct']=resumen[real].map(100*aciertos.groupby(te[real]).mean())
        campos=['exactitud_pct']
        for destino in clases:
            c='pred_'+clave(destino)+'_pct'
            pr=(te[pred]==destino).astype(float)
            resumen[c]=resumen[real].map(100*pr.groupby(te[real]).mean());campos.append(c)
        resumen.to_csv(OUT/f'resumen_{real}.csv',index=False)
        if tarea=='ESTADO':
            resumen['clave']=resumen.estado.map(clave).replace(ALIAS)
            gdf=adm.merge(resumen.drop(columns='estado'),on='clave',how='left',validate='one_to_one')
            campo_etiqueta=col
        else:
            geom=adm[adm.region.notna()].dissolve(by='region').reset_index()
            gdf=geom.merge(resumen,on='region',how='left',validate='one_to_one')
            campo_etiqueta='region'
        carpeta=OUT/'mapas'/real
        gdf.to_file(OUT/f'mapa_{real}.gpkg',driver='GPKG')
        for campo in campos:
            subtitulo=('Exactitud del modelo' if campo=='exactitud_pct' else 'Predicción: '+campo[5:-4].replace('_',' ').title())
            mapa_tema(gdf,campo,campo_etiqueta,f'{subtitulo} | {real.title()}',
                      carpeta,base_mx,borde_mx,por_region=(tarea=='REGION'))
    print('Mapas magma_r guardados.')
except Exception as e:
    print('No se pudieron generar los mapas:',type(e).__name__,str(e))

zipfinal=RAIZ/'resultados_geolinguistica_magma.zip'
with zipfile.ZipFile(zipfinal,'w',zipfile.ZIP_DEFLATED) as z:
    for p in OUT.rglob('*'):
        if p.is_file():z.write(p,arcname=str(p.relative_to(OUT)))
print('RESULTADOS:',zipfinal)
if 'google.colab' in sys.modules:files.download(str(zipfinal))


### EXP-057 · V1, V2, V3, V4

**Archivo de origen:** `27_Octubre.ipynb, celda 31`. **Referencias duplicadas:** 0. **Sintaxis de código:** Válida.

GOOGLE COLAB | VIOLENCIA DIGITAL: BINARIA Y MULTICLASE
TF-IDF, N-GRAMAS, BOW, EMBEDDINGS, HMVT V1-V4, CON/SIN REGIONALISMOS
8 CLASIFICADORES | MATRICES MORADAS | MAPAS POR ESTADO Y REGIÓN
MAPAS POR CONCENTRACION DE CLASE, % Y CONTEOS POR ES

**Ejecución:** celda independiente o variante experimental; revisar archivos de entrada y dependencias antes de ejecutarla. Los resultados originales no se insertan en esta celda.

In [ ]:
import sys, re, time, zipfile, warnings, subprocess, importlib.util, unicodedata
from pathlib import Path
from io import BytesIO
warnings.filterwarnings('ignore', category=FutureWarning)
SEMILLA = 42
TEST_SIZE = .20
PRESUPUESTO_MODELOS_S = 20 * 60
MOSTRAR_MAPAS = True
MOSTRAR_MATRICES = False
EMBEDDING_MODELO = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
MODELOS = ['RL','KNN','SVM','NB','MLP','XGB','RF','ARBOL']
BASES = ['TFIDF','NGRAMAS','BOW','EMBEDDINGS']
HMVT = ['HMVT-V1','HMVT-V2','HMVT-V3','HMVT-V4']
REPRESENTACIONES = BASES + HMVT + [n+'+REG' for n in BASES+HMVT]
RAIZ = Path('/content') if 'google.colab' in sys.modules else Path('/mnt/data')
SALIDA = RAIZ / 'RESULTADOS_VIOLENCIA_MORADOS_CON_PORCENTAJES'
SALIDA.mkdir(parents=True,exist_ok=True)
INICIO = time.monotonic()

def dependencia(mod, paquete=None):
    if importlib.util.find_spec(mod) is None:
        subprocess.check_call([sys.executable,'-m','pip','-q','install',paquete or mod])
for mod, paquete in [('numpy',None),('pandas',None),('scipy',None),('sklearn','scikit-learn'),
                     ('matplotlib',None),('seaborn',None),('xgboost',None),
                     ('sentence_transformers','sentence-transformers'),('geopandas',None),('requests',None)]:
    dependencia(mod,paquete)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import geopandas as gpd
import requests
from scipy import sparse
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler, MultiLabelBinarizer
from sklearn.metrics import (confusion_matrix,classification_report,precision_score,
                             recall_score,f1_score,accuracy_score,balanced_accuracy_score)
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from sentence_transformers import SentenceTransformer
from matplotlib.patches import Patch
from IPython.display import display
sns.set_theme(style='white')
np.random.seed(SEMILLA)

def sin_acentos(s):
    return ''.join(c for c in unicodedata.normalize('NFKD',str(s)) if not unicodedata.combining(c))
def clave(s):
    return re.sub(r'[^a-z0-9]','',sin_acentos(str(s)).lower())
def texto_limpio(s):
    s=str(s).lower().strip()
    s=re.sub(r'https?://\S+|www\.\S+',' URL ',s)
    s=re.sub(r'@\w+',' USUARIO ',s)
    return re.sub(r'\s+',' ',s)




if 'google.colab' in sys.modules:
    from google.colab import files
    print('Sube data_hmvt_con_estado.csv')
    subidos=files.upload()
    if len(subidos)!=1: raise ValueError('Selecciona un solo archivo CSV.')
    nombre,contenido=next(iter(subidos.items()))
    df=pd.read_csv(BytesIO(contenido),encoding='utf-8-sig')
else:
    opciones=[RAIZ/'data_hmvt_con_estado(1).csv',RAIZ/'data_hmvt_con_estado.csv']
    archivo=next((p for p in opciones if p.exists()),None)
    if archivo is None: raise FileNotFoundError('Falta data_hmvt_con_estado.csv')
    df=pd.read_csv(archivo,encoding='utf-8-sig')
requeridas={'tweet','clase','violencia_binaria','estado','region','regionalismo'}
if requeridas-set(df.columns):raise ValueError('Faltan columnas: '+str(requeridas-set(df.columns)))
df=df.dropna(subset=['tweet','clase','violencia_binaria']).copy()
for c in ['tweet','clase','estado','region','regionalismo']:
    df[c]=df[c].fillna('SIN_DATO').astype(str).str.strip()
df['violencia_binaria']=pd.to_numeric(df['violencia_binaria'],errors='raise').astype(int)
if not set(df.violencia_binaria.unique()).issubset({0,1}):raise ValueError('Binaria requiere 0 y 1')
df['texto_norm']=df.tweet.map(texto_limpio)
df=df[df.texto_norm.str.len()>0].copy()
if df.groupby('texto_norm').clase.nunique().max()>1:
    raise ValueError('Un texto tiene múltiples clases de violencia: revisar anotaciones.')
if df.groupby('texto_norm').violencia_binaria.nunique().max()>1:
    raise ValueError('Un texto tiene etiquetas binarias contradictorias.')

corpus=df.drop_duplicates('texto_norm').reset_index(drop=True)
idx_tr,idx_te=train_test_split(np.arange(len(corpus)),test_size=TEST_SIZE,
                               stratify=corpus.clase,random_state=SEMILLA)
tr=corpus.iloc[idx_tr].reset_index(drop=True)
te=corpus.iloc[idx_te].reset_index(drop=True)
assert set(tr.texto_norm).isdisjoint(set(te.texto_norm))
tr.to_csv(SALIDA/'train_textos_unicos.csv',index=False)
te.to_csv(SALIDA/'test_textos_unicos.csv',index=False)
print('Registros originales:',len(df),'Textos únicos:',len(corpus),
      'TRAIN:',len(tr),'TEST:',len(te))
print('Distribución de TEST:',te.clase.value_counts().to_dict())




REP={}
def guardar_rep(nombre,a,b):
    REP[nombre]=(sparse.csr_matrix(a,dtype=np.float32),
                 sparse.csr_matrix(b,dtype=np.float32))
for nombre,vector in [
    ('TFIDF',TfidfVectorizer(max_features=7000,ngram_range=(1,2),strip_accents='unicode')),
    ('NGRAMAS',TfidfVectorizer(max_features=8000,analyzer='char',ngram_range=(3,5))),
    ('BOW',CountVectorizer(max_features=7000,strip_accents='unicode'))]:
    guardar_rep(nombre,vector.fit_transform(tr.texto_norm),vector.transform(te.texto_norm))
modelo_embeddings=SentenceTransformer(EMBEDDING_MODELO)
todos_textos=list(dict.fromkeys(tr.texto_norm.tolist()+te.texto_norm.tolist()))
e=modelo_embeddings.encode(todos_textos,batch_size=32,show_progress_bar=True,normalize_embeddings=True)
dic_emb=dict(zip(todos_textos,e))
arr_tr=np.stack([dic_emb[x] for x in tr.texto_norm])
arr_te=np.stack([dic_emb[x] for x in te.texto_norm])
sc=MinMaxScaler().fit(arr_tr)
guardar_rep('EMBEDDINGS',np.maximum(0,sc.transform(arr_tr)),np.maximum(0,sc.transform(arr_te)))
del modelo_embeddings,e,dic_emb




BLOQUES={
 'HMVT-V1':[('TFIDF',1.),('NGRAMAS',1.)],
 'HMVT-V2':[('TFIDF',1.),('BOW',.3),('NGRAMAS',.7)],
 'HMVT-V3':[('TFIDF',1.),('NGRAMAS',.7),('EMBEDDINGS',2.)],
 'HMVT-V4':[('TFIDF',1.),('NGRAMAS',.7),('BOW',.3),('EMBEDDINGS',2.)]
}
for nombre,partes in BLOQUES.items():
    guardar_rep(nombre,*[
        sparse.hstack([REP[k][i]*w for k,w in partes],format='csr') for i in (0,1)
    ])






vocab=sorted({sin_acentos(x.lower()) for x in tr.regionalismo if x!='SIN_DATO' and x.strip()})
patrones=[(v,re.compile(r'(?<!\w)'+re.escape(v)+r'(?!\w)',re.I)) for v in vocab]
def detectar_regionalismos(t):
    s=sin_acentos(str(t).lower())
    return [v for v,p in patrones if p.search(s)]
mlb=MultiLabelBinarizer(classes=vocab,sparse_output=True)
reg_tr=mlb.fit_transform([detectar_regionalismos(x) for x in tr.tweet]).astype(np.float32)
reg_te=mlb.transform([detectar_regionalismos(x) for x in te.tweet]).astype(np.float32)
print('Regionalismos del vocabulario:',len(vocab),
      'Textos TEST con coincidencias:',int(np.sum(reg_te.getnnz(axis=1)>0)))
for nombre in BASES+HMVT:
    a,b=REP[nombre]
    guardar_rep(nombre+'+REG',sparse.hstack([a,reg_tr],format='csr'),
               sparse.hstack([b,reg_te],format='csr'))




def crear_modelo(k,nclases):
    if k=='RL':return LogisticRegression(max_iter=650,random_state=SEMILLA)
    if k=='KNN':return KNeighborsClassifier(n_neighbors=3)
    if k=='SVM':return LinearSVC(max_iter=1800,random_state=SEMILLA)
    if k=='NB':return MultinomialNB()
    if k=='MLP':return MLPClassifier(hidden_layer_sizes=(32,),max_iter=70,
                                     early_stopping=True,random_state=SEMILLA)
    if k=='XGB':return XGBClassifier(n_estimators=55,max_depth=3,learning_rate=.1,
                   random_state=SEMILLA,n_jobs=2,tree_method='hist',
                   eval_metric='logloss' if nclases==2 else 'mlogloss')
    if k=='RF':return RandomForestClassifier(n_estimators=60,random_state=SEMILLA,n_jobs=2)
    if k=='ARBOL':return DecisionTreeClassifier(random_state=SEMILLA)
    raise ValueError(k)
PARAM={
 'RL':('C',[.3,1.]),'KNN':('n_neighbors',[3,5]),'SVM':('C',[.3,1.]),
 'NB':('alpha',[.5,1.]),'MLP':('alpha',[.001,.01]),
 'XGB':('max_depth',[2,3]),'RF':('max_depth',[7,None]),
 'ARBOL':('max_depth',[6,None])
}
ETIQUETAS_MULTI=sorted(corpus.clase.unique().tolist())
TAREAS={'BINARIA':('violencia_binaria',['No violencia','Violencia']),
        'MULTICLASE':('clase',ETIQUETAS_MULTI)}
resultados=[];desglose=[];predicciones={};errores=[];optimizacion=[]
DIR_CM=SALIDA/'matrices_csv';DIR_CM.mkdir(exist_ok=True)
for tarea,(col,etiquetas) in TAREAS.items():
    if tarea=='BINARIA':
        ytr=tr.violencia_binaria.to_numpy();yte=te.violencia_binaria.to_numpy()
    else:
        etiquetas_id={x:i for i,x in enumerate(etiquetas)}
        ytr=tr.clase.map(etiquetas_id).to_numpy()
        yte=te.clase.map(etiquetas_id).to_numpy()
    for representacion in REPRESENTACIONES:
        xtr,xte=REP[representacion]
        for alg in MODELOS:
            if time.monotonic()-INICIO>PRESUPUESTO_MODELOS_S:
                errores.append([tarea,representacion,alg,'No iniciado por límite de tiempo'])
                continue
            try:
                m=crear_modelo(alg,len(etiquetas))
                if representacion.startswith('HMVT-'):
                    
                    it,iv=train_test_split(np.arange(len(ytr)),test_size=.20,
                        stratify=ytr if min(np.bincount(ytr))>=2 else None,
                        random_state=SEMILLA+15)
                    par,valores=PARAM[alg]
                    evaluados=[]
                    for valor in valores:
                        cand=crear_modelo(alg,len(etiquetas)).set_params(**{par:valor})
                        cand.fit(xtr[it],ytr[it])
                        pp=cand.predict(xtr[iv])
                        evaluados.append((f1_score(ytr[iv],pp,average='macro',zero_division=0),valor))
                    fval,param=max(evaluados,key=lambda q:q[0]);m.set_params(**{par:param})
                    optimizacion.append([tarea,representacion,alg,par,param,fval])
                m.fit(xtr,ytr)
                pred=m.predict(xte)
                predicciones[(tarea,representacion,alg)]=pred
                cm=confusion_matrix(yte,pred,labels=np.arange(len(etiquetas)))
                pd.DataFrame(cm,index=etiquetas,columns=etiquetas).to_csv(
                    DIR_CM/f'{tarea}_{representacion}_{alg}.csv')
                resultados.append([tarea,representacion,alg,
                  precision_score(yte,pred,average='macro',zero_division=0),
                  recall_score(yte,pred,average='macro',zero_division=0),
                  f1_score(yte,pred,average='macro',zero_division=0),
                  accuracy_score(yte,pred),balanced_accuracy_score(yte,pred)])
                rep_clase=classification_report(yte,pred,labels=np.arange(len(etiquetas)),
                                                output_dict=True,zero_division=0)
                for i,nombre_clase in enumerate(etiquetas):
                    r=rep_clase[str(i)]
                    desglose.append([tarea,representacion,alg,nombre_clase,
                                     r['precision'],r['recall'],r['f1-score'],r['support']])
            except Exception as ex:
                errores.append([tarea,representacion,alg,str(ex)])
                print('Error:',tarea,representacion,alg,str(ex)[:100])
        print('Evaluado:',tarea,representacion)
col_res=['Tarea','Representacion','Modelo','Precision_macro','Recall_macro',
         'F1_macro','Accuracy','Balanced_accuracy']
tabla=pd.DataFrame(resultados,columns=col_res)
if tabla.empty:raise RuntimeError('No se obtuvieron predicciones. Revisa errores.')
tabla.to_csv(SALIDA/'TABLA_TODOS_MODELOS.csv',index=False)
pd.DataFrame(desglose,columns=['Tarea','Representacion','Modelo','Clase','Precision','Recall',
                                'F1','Soporte']).to_csv(SALIDA/'METRICAS_POR_CLASE.csv',index=False)
pd.DataFrame(errores,columns=['Tarea','Representacion','Modelo','Motivo']).to_csv(SALIDA/'PENDIENTES.csv',index=False)
pd.DataFrame(optimizacion,columns=['Tarea','Representacion','Modelo','Parametro','Valor',
                                    'F1_validacion']).to_csv(SALIDA/'OPTIMIZACION_HMVT.csv',index=False)
for tarea in TAREAS:
    print('\n', '='*15, tarea, '='*15)
    for titulo,reps in [('1_CLASICAS',BASES),('2_HMVT',HMVT),
                        ('3_CON_REGIONALISMOS',[x+'+REG' for x in BASES+HMVT])]:
        sub=tabla[(tabla.Tarea==tarea)&tabla.Representacion.isin(reps)].copy()
        sub['orden']=sub.Representacion.map({x:i for i,x in enumerate(reps)})
        sub=sub.sort_values(['orden','Modelo']).drop(columns='orden')
        print('\n',titulo)
        if not sub.empty:
            display(sub.style.format(precision=4).background_gradient(cmap='Purples',subset=['F1_macro']))
        sub.to_csv(SALIDA/f'{tarea}_{titulo}.csv',index=False)




DIR_FIG=SALIDA/'matrices_moradas';DIR_FIG.mkdir(exist_ok=True)
for tarea,(_,etiquetas) in TAREAS.items():
    for alg in MODELOS:
        fig,axes=plt.subplots(4,4,figsize=(24,20) if tarea=='BINARIA' else (28,27))
        for ax,representacion in zip(axes.flat,REPRESENTACIONES):
            k=(tarea,representacion,alg)
            if k not in predicciones:
                ax.axis('off');continue
            cm=pd.read_csv(DIR_CM/f'{tarea}_{representacion}_{alg}.csv',index_col=0).values
            sns.heatmap(cm,ax=ax,cmap='Purples',annot=True,fmt='d',cbar=False,
                        linewidths=.45,linecolor='white',xticklabels=etiquetas,yticklabels=etiquetas,
                        annot_kws={'size':8 if tarea=='MULTICLASE' else 12,'weight':'bold'})
            fila=tabla[(tabla.Tarea==tarea)&(tabla.Representacion==representacion)&
                       (tabla.Modelo==alg)].iloc[0]
            ax.set_title(f'{representacion}\nF1={fila.F1_macro:.3f} · Acc={fila.Accuracy:.3f}',
                         fontsize=12,color='#3b0764',weight='bold')
            ax.tick_params(axis='x',rotation=55,labelsize=7)
            ax.tick_params(axis='y',rotation=0,labelsize=7)
            ax.set_xlabel('Predicha');ax.set_ylabel('Real')
        fig.suptitle(f'VIOLENCIA DIGITAL | {tarea} | {alg} | TEST 20%',
                     fontsize=19,color='#3b0764',weight='bold')
        fig.tight_layout(rect=(0,0,1,.97))
        fig.savefig(DIR_FIG/f'{tarea}_{alg}.png',dpi=190,bbox_inches='tight')
        if MOSTRAR_MATRICES:plt.show()
        plt.close(fig)





def clave_mapa(tarea):
    preferencias=[(tarea,'HMVT-V4+REG','RL'),(tarea,'HMVT-V4','RL'),
                  (tarea,'TFIDF+REG','RL'),(tarea,'TFIDF','RL')]
    return next((k for k in preferencias if k in predicciones),None)


geo_test=df[df.texto_norm.isin(set(te.texto_norm))].drop_duplicates(['texto_norm','estado']).copy()
for tarea in TAREAS:
    k=clave_mapa(tarea)
    if k is None:
        print('Sin predicción para mapas:',tarea)
        continue
    etiquetas=TAREAS[tarea][1]
    dic=dict(zip(te.texto_norm,np.asarray(etiquetas,dtype=object)[predicciones[k]]))
    geo_test['pred_'+tarea.lower()]=geo_test.texto_norm.map(dic)
    print('Predicción cartográfica fija:',k)
geo_test.to_csv(SALIDA/'PREDICCIONES_TEST_POR_ESTADO.csv',index=False)






CLASES_MULTI=ETIQUETAS_MULTI

def campo_clase(c):return 'pred_'+re.sub('[^a-z0-9]+','_',sin_acentos(c).lower()).strip('_')+'_pct'
CAMPOS=[]
if 'pred_binaria' in geo_test.columns:CAMPOS.append('violencia_predicha_pct')
if 'pred_multiclase' in geo_test.columns:CAMPOS.extend(campo_clase(c) for c in CLASES_MULTI)
if not CAMPOS:raise RuntimeError('No hay predicciones para mapas')

def resumen_geografico(nivel):
    salida=[]
    for lugar,g in geo_test.groupby(nivel):
        if lugar=='SIN_DATO':continue
        fila={nivel:lugar,'n_publicaciones':len(g),'n_textos':g.texto_norm.nunique()}
        if 'pred_binaria' in g:
            fila['violencia_predicha_pct']=100*g.pred_binaria.eq('Violencia').mean()
            fila['violencia_predicha_n']=int(g.pred_binaria.eq('Violencia').sum())
        if 'pred_multiclase' in g:
            for c in CLASES_MULTI:
                fila[campo_clase(c)]=100*g.pred_multiclase.eq(c).mean()
                fila[campo_clase(c).replace('_pct','_n')]=int(g.pred_multiclase.eq(c).sum())
        salida.append(fila)
    return pd.DataFrame(salida)
res_est=resumen_geografico('estado')
res_reg=resumen_geografico('region')
res_est.to_csv(SALIDA/'RESUMEN_ESTADO.csv',index=False,encoding='utf-8-sig')
res_reg.to_csv(SALIDA/'RESUMEN_REGION.csv',index=False,encoding='utf-8-sig')





MORADOS=['#E4D5F4','#9465C5','#3B1465']
GRIS='#E5E7EB'
ETIQUETAS_RANGOS=['0–33 %','>33–66 %','>66–100 %']
ALIAS={'distritofederal':'ciudaddemexico','mexicocity':'ciudaddemexico',
       'veracruzdeignaciodelallave':'veracruz','michoacandeocampo':'michoacan',
       'coahuiladezaragoza':'coahuila','estadodemexico':'mexico','mexicostate':'mexico',
       'queretarodearteaga':'queretaro'}
ABREV={'ciudaddemexico':'CDMX','mexico':'Edo. Méx.','tlaxcala':'Tlax.',
       'morelos':'Mor.','aguascalientes':'Ags.','queretaro':'Qro.',
       'nuevoleon':'N. León','sanluispotosi':'SLP','bajacaliforniasur':'BCS'}


POSICIONES={
 'veracruz':(-94.0,22.8), 'tabasco':(-90.9,16.3),
 'ciudaddemexico':(-96.7,17.5), 'mexico':(-102.0,20.4),
 'tlaxcala':(-96.0,21.8), 'morelos':(-100.8,16.4),
 'queretaro':(-99.0,22.1), 'aguascalientes':(-101.4,24.3),
 'colima':(-106.0,18.4), 'puebla':(-96.5,18.8)
}

def tono(v):
    if pd.isna(v):return GRIS
    if v<=33:return MORADOS[0]
    if v<=66:return MORADOS[1]
    return MORADOS[2]

def dibujar_mapa(geo,base,borde,campo,nivel,titulo,carpeta):
    """Coropleta porcentual por clase de violencia, 3 morados, etiquetas con n y %."""
    if campo not in geo or not geo[campo].notna().any():return
    fig,ax=plt.subplots(figsize=(19,11),facecolor='white')
    base.plot(ax=ax,color=GRIS,edgecolor='#9da2ac',linewidth=.55,zorder=1)
    dib=geo[geo[campo].notna()].copy()
    dib.plot(ax=ax,color=dib[campo].map(tono).tolist(),edgecolor='#51415e',
             linewidth=.8 if nivel=='estado' else 1.5,zorder=2)
    base.boundary.plot(ax=ax,color='#777583',linewidth=.45,zorder=3)
    if nivel=='region':dib.boundary.plot(ax=ax,color='#29153f',linewidth=1.9,zorder=4)
    borde.boundary.plot(ax=ax,color='#17121d',linewidth=1.3,zorder=5)

    cuenta_col=campo.replace('_pct','_n')
    for _,fila in dib.iterrows():
        nombre=str(fila['nombre_estado'] if nivel=='estado' else fila['region'])
        k=clave(nombre)
        nombre_corto=ABREV.get(k,nombre) if nivel=='estado' else nombre
        porcentaje=float(fila[campo])
        n=int(fila['n_publicaciones']) if pd.notna(fila.get('n_publicaciones')) else 0
        hits=int(fila[cuenta_col]) if cuenta_col in fila and pd.notna(fila[cuenta_col]) else round(porcentaje*n/100)
        etiqueta=f'{nombre_corto}\n{porcentaje:.1f}%  ({hits}/{n})'
        p=fila.geometry.representative_point()
        
        if nivel=='estado' and k in POSICIONES:
            ax.annotate(etiqueta,xy=(p.x,p.y),xytext=POSICIONES[k],
                fontsize=8,linespacing=1.18,fontweight='semibold',color='#241431',
                ha='center',va='center',
                bbox=dict(boxstyle='round,pad=.25',facecolor='white',edgecolor='#9f91af',alpha=.97),
                arrowprops=dict(arrowstyle='-',color='#51415e',lw=.9,
                                connectionstyle='angle3,angleA=0,angleB=90'),zorder=8)
        else:
            ax.text(p.x,p.y,etiqueta,fontsize=7.5 if nivel=='estado' else 10,
                linespacing=1.18,weight='semibold',ha='center',va='center',color='#28143b',
                bbox=dict(boxstyle='round,pad=.16',facecolor='white',edgecolor='none',alpha=.85),zorder=7)

    parches=[Patch(facecolor=MORADOS[i],edgecolor='#47334f',label=ETIQUETAS_RANGOS[i]) for i in range(3)]
    parches.append(Patch(facecolor=GRIS,edgecolor='#9da2ac',label='Sin datos'))
    ax.legend(handles=parches,title='% de publicaciones de la zona',loc='center left',
              bbox_to_anchor=(1.01,.62),frameon=True,fontsize=10,title_fontsize=11,
              labelspacing=.8,borderpad=.9)
    ax.set_title(f'Predicción: {titulo} | {nivel.title()}',loc='left',fontsize=18,
                 weight='bold',color='#32104f',pad=20)
    ax.text(.01,.02,'TEST (20% de textos únicos) · Etiqueta: porcentaje (n de publicaciones predichas / n evaluadas)'
            '\nGris: sin datos · Los porcentajes corresponden al conjunto evaluado, no a la población general.',
            transform=ax.transAxes,fontsize=9,color='#5b5363',va='bottom')
    ax.set_axis_off();ax.set_aspect('equal')
    
    minx,miny,maxx,maxy=base.total_bounds
    ax.set_xlim(minx-1.0,maxx+2.0);ax.set_ylim(miny-1.3,maxy+1.2)
    fig.tight_layout()
    carpeta.mkdir(parents=True,exist_ok=True)
    fig.savefig(carpeta/f'{nivel}_{campo}.png',dpi=300,bbox_inches='tight',facecolor='white')
    if MOSTRAR_MAPAS:plt.show()
    plt.close(fig)

try:
    ruta_geo=SALIDA/'mexico_estados.geojson'
    if not ruta_geo.exists():
        meta=requests.get('https://www.geoboundaries.org/api/current/gbOpen/MEX/ADM1/',timeout=40)
        meta.raise_for_status()
        meta=meta.json()
        url=meta.get('simplifiedGeometryGeoJSON') or meta.get('gjDownloadURL')
        if not url:raise RuntimeError('geoBoundaries no ofreció el archivo de estados.')
        descarga=requests.get(url,timeout=100);descarga.raise_for_status()
        ruta_geo.write_bytes(descarga.content)
    mapa=gpd.read_file(ruta_geo).to_crs('EPSG:4326')
    nombre_col=next((x for x in ['shapeName','NAME_1','name','NOMGEO'] if x in mapa),None)
    if nombre_col is None:raise ValueError('No se encontró la columna de nombres estatales.')
    mapa['nombre_estado']=mapa[nombre_col].astype(str)
    mapa['clave']=mapa.nombre_estado.map(clave).replace(ALIAS)
    res_est['clave']=res_est.estado.map(clave).replace(ALIAS)
    estados=mapa.merge(res_est.drop(columns='estado'),on='clave',how='left',validate='one_to_one')
    asignacion=df[['estado','region']].drop_duplicates().copy()
    
    asignacion=asignacion.groupby('estado',as_index=False).region.agg(lambda s:s.mode().iloc[0])
    asignacion['clave']=asignacion.estado.map(clave).replace(ALIAS)
    estados=estados.merge(asignacion[['clave','region']],on='clave',how='left',validate='one_to_one')
    regiones=estados[estados.region.notna()].dissolve(by='region').reset_index()[['region','geometry']]
    regiones=regiones.merge(res_reg,on='region',how='left',validate='one_to_one')
    base_mx=mapa[['geometry']].copy()
    borde=gpd.GeoSeries([mapa.geometry.union_all()],crs=mapa.crs)
    print('Estados coincidentes:',int(estados.n_publicaciones.notna().sum()),'de',len(mapa))
    for nivel,gdf in [('estado',estados),('region',regiones)]:
        carpeta=SALIDA/'mapas_porcentajes_3_morados'/nivel
        for campo in CAMPOS:
            titulo=('Violencia digital (binaria)' if campo=='violencia_predicha_pct'
                    else campo.replace('pred_','').replace('_pct','').replace('_',' ').title())
            dibujar_mapa(gdf,base_mx,borde,campo,nivel,titulo,carpeta)
        gdf.to_file(SALIDA/f'GEODATOS_{nivel}.gpkg',driver='GPKG')
    print('Mapas guardados en',SALIDA/'mapas_porcentajes_3_morados')
except Exception as ex:
    print('ADVERTENCIA: falló la sección cartográfica:',type(ex).__name__,str(ex))
    print('Las tablas, las predicciones y las matrices sí se conservaron.')




ZIP=RAIZ/'RESULTADOS_VIOLENCIA_MORADOS_CON_PORCENTAJES.zip'
with zipfile.ZipFile(ZIP,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for archivo in SALIDA.rglob('*'):
        if archivo.is_file():z.write(archivo,arcname=str(archivo.relative_to(SALIDA)))
print('\nResultados:',SALIDA,'\nZIP:',ZIP)
if 'google.colab' in sys.modules:files.download(str(ZIP))


### EXP-058 · V1, V2, V3, V4

**Archivo de origen:** `27_Octubre.ipynb, celda 32`. **Referencias duplicadas:** 0. **Sintaxis de código:** Válida.

GOOGLE COLAB | VIOLENCIA DIGITAL: BINARIA Y MULTICLASE
TF-IDF, N-GRAMAS, BOW, EMBEDDINGS, HMVT V1-V4, CON/SIN REGIONALISMOS
8 CLASIFICADORES | MATRICES MORADAS | MAPAS POR ESTADO Y REGIÓN
MAPAS POR CONCENTRACION DE CLASE: SOLO % Y 3 RANGOS 

**Ejecución:** celda independiente o variante experimental; revisar archivos de entrada y dependencias antes de ejecutarla. Los resultados originales no se insertan en esta celda.

In [ ]:
import sys, re, time, zipfile, warnings, subprocess, importlib.util, unicodedata
from pathlib import Path
from io import BytesIO
warnings.filterwarnings('ignore', category=FutureWarning)
SEMILLA = 42
TEST_SIZE = .20
PRESUPUESTO_MODELOS_S = 20 * 60
MOSTRAR_MAPAS = True
MOSTRAR_MATRICES = False
EMBEDDING_MODELO = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
MODELOS = ['RL','KNN','SVM','NB','MLP','XGB','RF','ARBOL']
BASES = ['TFIDF','NGRAMAS','BOW','EMBEDDINGS']
HMVT = ['HMVT-V1','HMVT-V2','HMVT-V3','HMVT-V4']
REPRESENTACIONES = BASES + HMVT + [n+'+REG' for n in BASES+HMVT]
RAIZ = Path('/content') if 'google.colab' in sys.modules else Path('/mnt/data')
SALIDA = RAIZ / 'RESULTADOS_VIOLENCIA_3_MORADOS_SOLO_PORCENTAJE'
SALIDA.mkdir(parents=True,exist_ok=True)
INICIO = time.monotonic()

def dependencia(mod, paquete=None):
    if importlib.util.find_spec(mod) is None:
        subprocess.check_call([sys.executable,'-m','pip','-q','install',paquete or mod])
for mod, paquete in [('numpy',None),('pandas',None),('scipy',None),('sklearn','scikit-learn'),
                     ('matplotlib',None),('seaborn',None),('xgboost',None),
                     ('sentence_transformers','sentence-transformers'),('geopandas',None),('requests',None)]:
    dependencia(mod,paquete)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import geopandas as gpd
import requests
from scipy import sparse
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler, MultiLabelBinarizer
from sklearn.metrics import (confusion_matrix,classification_report,precision_score,
                             recall_score,f1_score,accuracy_score,balanced_accuracy_score)
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from sentence_transformers import SentenceTransformer
from matplotlib.patches import Patch
from IPython.display import display
sns.set_theme(style='white')
np.random.seed(SEMILLA)

def sin_acentos(s):
    return ''.join(c for c in unicodedata.normalize('NFKD',str(s)) if not unicodedata.combining(c))
def clave(s):
    return re.sub(r'[^a-z0-9]','',sin_acentos(str(s)).lower())
def texto_limpio(s):
    s=str(s).lower().strip()
    s=re.sub(r'https?://\S+|www\.\S+',' URL ',s)
    s=re.sub(r'@\w+',' USUARIO ',s)
    return re.sub(r'\s+',' ',s)




if 'google.colab' in sys.modules:
    from google.colab import files
    print('Sube data_hmvt_con_estado.csv')
    subidos=files.upload()
    if len(subidos)!=1: raise ValueError('Selecciona un solo archivo CSV.')
    nombre,contenido=next(iter(subidos.items()))
    df=pd.read_csv(BytesIO(contenido),encoding='utf-8-sig')
else:
    opciones=[RAIZ/'data_hmvt_con_estado(1).csv',RAIZ/'data_hmvt_con_estado.csv']
    archivo=next((p for p in opciones if p.exists()),None)
    if archivo is None: raise FileNotFoundError('Falta data_hmvt_con_estado.csv')
    df=pd.read_csv(archivo,encoding='utf-8-sig')
requeridas={'tweet','clase','violencia_binaria','estado','region','regionalismo'}
if requeridas-set(df.columns):raise ValueError('Faltan columnas: '+str(requeridas-set(df.columns)))
df=df.dropna(subset=['tweet','clase','violencia_binaria']).copy()
for c in ['tweet','clase','estado','region','regionalismo']:
    df[c]=df[c].fillna('SIN_DATO').astype(str).str.strip()
df['violencia_binaria']=pd.to_numeric(df['violencia_binaria'],errors='raise').astype(int)
if not set(df.violencia_binaria.unique()).issubset({0,1}):raise ValueError('Binaria requiere 0 y 1')
df['texto_norm']=df.tweet.map(texto_limpio)
df=df[df.texto_norm.str.len()>0].copy()
if df.groupby('texto_norm').clase.nunique().max()>1:
    raise ValueError('Un texto tiene múltiples clases de violencia: revisar anotaciones.')
if df.groupby('texto_norm').violencia_binaria.nunique().max()>1:
    raise ValueError('Un texto tiene etiquetas binarias contradictorias.')

corpus=df.drop_duplicates('texto_norm').reset_index(drop=True)
idx_tr,idx_te=train_test_split(np.arange(len(corpus)),test_size=TEST_SIZE,
                               stratify=corpus.clase,random_state=SEMILLA)
tr=corpus.iloc[idx_tr].reset_index(drop=True)
te=corpus.iloc[idx_te].reset_index(drop=True)
assert set(tr.texto_norm).isdisjoint(set(te.texto_norm))
tr.to_csv(SALIDA/'train_textos_unicos.csv',index=False)
te.to_csv(SALIDA/'test_textos_unicos.csv',index=False)
print('Registros originales:',len(df),'Textos únicos:',len(corpus),
      'TRAIN:',len(tr),'TEST:',len(te))
print('Distribución de TEST:',te.clase.value_counts().to_dict())




REP={}
def guardar_rep(nombre,a,b):
    REP[nombre]=(sparse.csr_matrix(a,dtype=np.float32),
                 sparse.csr_matrix(b,dtype=np.float32))
for nombre,vector in [
    ('TFIDF',TfidfVectorizer(max_features=7000,ngram_range=(1,2),strip_accents='unicode')),
    ('NGRAMAS',TfidfVectorizer(max_features=8000,analyzer='char',ngram_range=(3,5))),
    ('BOW',CountVectorizer(max_features=7000,strip_accents='unicode'))]:
    guardar_rep(nombre,vector.fit_transform(tr.texto_norm),vector.transform(te.texto_norm))
modelo_embeddings=SentenceTransformer(EMBEDDING_MODELO)
todos_textos=list(dict.fromkeys(tr.texto_norm.tolist()+te.texto_norm.tolist()))
e=modelo_embeddings.encode(todos_textos,batch_size=32,show_progress_bar=True,normalize_embeddings=True)
dic_emb=dict(zip(todos_textos,e))
arr_tr=np.stack([dic_emb[x] for x in tr.texto_norm])
arr_te=np.stack([dic_emb[x] for x in te.texto_norm])
sc=MinMaxScaler().fit(arr_tr)
guardar_rep('EMBEDDINGS',np.maximum(0,sc.transform(arr_tr)),np.maximum(0,sc.transform(arr_te)))
del modelo_embeddings,e,dic_emb




BLOQUES={
 'HMVT-V1':[('TFIDF',1.),('NGRAMAS',1.)],
 'HMVT-V2':[('TFIDF',1.),('BOW',.3),('NGRAMAS',.7)],
 'HMVT-V3':[('TFIDF',1.),('NGRAMAS',.7),('EMBEDDINGS',2.)],
 'HMVT-V4':[('TFIDF',1.),('NGRAMAS',.7),('BOW',.3),('EMBEDDINGS',2.)]
}
for nombre,partes in BLOQUES.items():
    guardar_rep(nombre,*[
        sparse.hstack([REP[k][i]*w for k,w in partes],format='csr') for i in (0,1)
    ])






vocab=sorted({sin_acentos(x.lower()) for x in tr.regionalismo if x!='SIN_DATO' and x.strip()})
patrones=[(v,re.compile(r'(?<!\w)'+re.escape(v)+r'(?!\w)',re.I)) for v in vocab]
def detectar_regionalismos(t):
    s=sin_acentos(str(t).lower())
    return [v for v,p in patrones if p.search(s)]
mlb=MultiLabelBinarizer(classes=vocab,sparse_output=True)
reg_tr=mlb.fit_transform([detectar_regionalismos(x) for x in tr.tweet]).astype(np.float32)
reg_te=mlb.transform([detectar_regionalismos(x) for x in te.tweet]).astype(np.float32)
print('Regionalismos del vocabulario:',len(vocab),
      'Textos TEST con coincidencias:',int(np.sum(reg_te.getnnz(axis=1)>0)))
for nombre in BASES+HMVT:
    a,b=REP[nombre]
    guardar_rep(nombre+'+REG',sparse.hstack([a,reg_tr],format='csr'),
               sparse.hstack([b,reg_te],format='csr'))




def crear_modelo(k,nclases):
    if k=='RL':return LogisticRegression(max_iter=650,random_state=SEMILLA)
    if k=='KNN':return KNeighborsClassifier(n_neighbors=3)
    if k=='SVM':return LinearSVC(max_iter=1800,random_state=SEMILLA)
    if k=='NB':return MultinomialNB()
    if k=='MLP':return MLPClassifier(hidden_layer_sizes=(32,),max_iter=70,
                                     early_stopping=True,random_state=SEMILLA)
    if k=='XGB':return XGBClassifier(n_estimators=55,max_depth=3,learning_rate=.1,
                   random_state=SEMILLA,n_jobs=2,tree_method='hist',
                   eval_metric='logloss' if nclases==2 else 'mlogloss')
    if k=='RF':return RandomForestClassifier(n_estimators=60,random_state=SEMILLA,n_jobs=2)
    if k=='ARBOL':return DecisionTreeClassifier(random_state=SEMILLA)
    raise ValueError(k)
PARAM={
 'RL':('C',[.3,1.]),'KNN':('n_neighbors',[3,5]),'SVM':('C',[.3,1.]),
 'NB':('alpha',[.5,1.]),'MLP':('alpha',[.001,.01]),
 'XGB':('max_depth',[2,3]),'RF':('max_depth',[7,None]),
 'ARBOL':('max_depth',[6,None])
}
ETIQUETAS_MULTI=sorted(corpus.clase.unique().tolist())
TAREAS={'BINARIA':('violencia_binaria',['No violencia','Violencia']),
        'MULTICLASE':('clase',ETIQUETAS_MULTI)}
resultados=[];desglose=[];predicciones={};errores=[];optimizacion=[]
DIR_CM=SALIDA/'matrices_csv';DIR_CM.mkdir(exist_ok=True)
for tarea,(col,etiquetas) in TAREAS.items():
    if tarea=='BINARIA':
        ytr=tr.violencia_binaria.to_numpy();yte=te.violencia_binaria.to_numpy()
    else:
        etiquetas_id={x:i for i,x in enumerate(etiquetas)}
        ytr=tr.clase.map(etiquetas_id).to_numpy()
        yte=te.clase.map(etiquetas_id).to_numpy()
    for representacion in REPRESENTACIONES:
        xtr,xte=REP[representacion]
        for alg in MODELOS:
            if time.monotonic()-INICIO>PRESUPUESTO_MODELOS_S:
                errores.append([tarea,representacion,alg,'No iniciado por límite de tiempo'])
                continue
            try:
                m=crear_modelo(alg,len(etiquetas))
                if representacion.startswith('HMVT-'):
                    
                    it,iv=train_test_split(np.arange(len(ytr)),test_size=.20,
                        stratify=ytr if min(np.bincount(ytr))>=2 else None,
                        random_state=SEMILLA+15)
                    par,valores=PARAM[alg]
                    evaluados=[]
                    for valor in valores:
                        cand=crear_modelo(alg,len(etiquetas)).set_params(**{par:valor})
                        cand.fit(xtr[it],ytr[it])
                        pp=cand.predict(xtr[iv])
                        evaluados.append((f1_score(ytr[iv],pp,average='macro',zero_division=0),valor))
                    fval,param=max(evaluados,key=lambda q:q[0]);m.set_params(**{par:param})
                    optimizacion.append([tarea,representacion,alg,par,param,fval])
                m.fit(xtr,ytr)
                pred=m.predict(xte)
                predicciones[(tarea,representacion,alg)]=pred
                cm=confusion_matrix(yte,pred,labels=np.arange(len(etiquetas)))
                pd.DataFrame(cm,index=etiquetas,columns=etiquetas).to_csv(
                    DIR_CM/f'{tarea}_{representacion}_{alg}.csv')
                resultados.append([tarea,representacion,alg,
                  precision_score(yte,pred,average='macro',zero_division=0),
                  recall_score(yte,pred,average='macro',zero_division=0),
                  f1_score(yte,pred,average='macro',zero_division=0),
                  accuracy_score(yte,pred),balanced_accuracy_score(yte,pred)])
                rep_clase=classification_report(yte,pred,labels=np.arange(len(etiquetas)),
                                                output_dict=True,zero_division=0)
                for i,nombre_clase in enumerate(etiquetas):
                    r=rep_clase[str(i)]
                    desglose.append([tarea,representacion,alg,nombre_clase,
                                     r['precision'],r['recall'],r['f1-score'],r['support']])
            except Exception as ex:
                errores.append([tarea,representacion,alg,str(ex)])
                print('Error:',tarea,representacion,alg,str(ex)[:100])
        print('Evaluado:',tarea,representacion)
col_res=['Tarea','Representacion','Modelo','Precision_macro','Recall_macro',
         'F1_macro','Accuracy','Balanced_accuracy']
tabla=pd.DataFrame(resultados,columns=col_res)
if tabla.empty:raise RuntimeError('No se obtuvieron predicciones. Revisa errores.')
tabla.to_csv(SALIDA/'TABLA_TODOS_MODELOS.csv',index=False)
pd.DataFrame(desglose,columns=['Tarea','Representacion','Modelo','Clase','Precision','Recall',
                                'F1','Soporte']).to_csv(SALIDA/'METRICAS_POR_CLASE.csv',index=False)
pd.DataFrame(errores,columns=['Tarea','Representacion','Modelo','Motivo']).to_csv(SALIDA/'PENDIENTES.csv',index=False)
pd.DataFrame(optimizacion,columns=['Tarea','Representacion','Modelo','Parametro','Valor',
                                    'F1_validacion']).to_csv(SALIDA/'OPTIMIZACION_HMVT.csv',index=False)
for tarea in TAREAS:
    print('\n', '='*15, tarea, '='*15)
    for titulo,reps in [('1_CLASICAS',BASES),('2_HMVT',HMVT),
                        ('3_CON_REGIONALISMOS',[x+'+REG' for x in BASES+HMVT])]:
        sub=tabla[(tabla.Tarea==tarea)&tabla.Representacion.isin(reps)].copy()
        sub['orden']=sub.Representacion.map({x:i for i,x in enumerate(reps)})
        sub=sub.sort_values(['orden','Modelo']).drop(columns='orden')
        print('\n',titulo)
        if not sub.empty:
            display(sub.style.format(precision=4).background_gradient(cmap='Purples',subset=['F1_macro']))
        sub.to_csv(SALIDA/f'{tarea}_{titulo}.csv',index=False)




DIR_FIG=SALIDA/'matrices_moradas';DIR_FIG.mkdir(exist_ok=True)
for tarea,(_,etiquetas) in TAREAS.items():
    for alg in MODELOS:
        fig,axes=plt.subplots(4,4,figsize=(24,20) if tarea=='BINARIA' else (28,27))
        for ax,representacion in zip(axes.flat,REPRESENTACIONES):
            k=(tarea,representacion,alg)
            if k not in predicciones:
                ax.axis('off');continue
            cm=pd.read_csv(DIR_CM/f'{tarea}_{representacion}_{alg}.csv',index_col=0).values
            sns.heatmap(cm,ax=ax,cmap='Purples',annot=True,fmt='d',cbar=False,
                        linewidths=.45,linecolor='white',xticklabels=etiquetas,yticklabels=etiquetas,
                        annot_kws={'size':8 if tarea=='MULTICLASE' else 12,'weight':'bold'})
            fila=tabla[(tabla.Tarea==tarea)&(tabla.Representacion==representacion)&
                       (tabla.Modelo==alg)].iloc[0]
            ax.set_title(f'{representacion}\nF1={fila.F1_macro:.3f} · Acc={fila.Accuracy:.3f}',
                         fontsize=12,color='#3b0764',weight='bold')
            ax.tick_params(axis='x',rotation=55,labelsize=7)
            ax.tick_params(axis='y',rotation=0,labelsize=7)
            ax.set_xlabel('Predicha');ax.set_ylabel('Real')
        fig.suptitle(f'VIOLENCIA DIGITAL | {tarea} | {alg} | TEST 20%',
                     fontsize=19,color='#3b0764',weight='bold')
        fig.tight_layout(rect=(0,0,1,.97))
        fig.savefig(DIR_FIG/f'{tarea}_{alg}.png',dpi=190,bbox_inches='tight')
        if MOSTRAR_MATRICES:plt.show()
        plt.close(fig)





def clave_mapa(tarea):
    preferencias=[(tarea,'HMVT-V4+REG','RL'),(tarea,'HMVT-V4','RL'),
                  (tarea,'TFIDF+REG','RL'),(tarea,'TFIDF','RL')]
    return next((k for k in preferencias if k in predicciones),None)


geo_test=df[df.texto_norm.isin(set(te.texto_norm))].drop_duplicates(['texto_norm','estado']).copy()
for tarea in TAREAS:
    k=clave_mapa(tarea)
    if k is None:
        print('Sin predicción para mapas:',tarea)
        continue
    etiquetas=TAREAS[tarea][1]
    dic=dict(zip(te.texto_norm,np.asarray(etiquetas,dtype=object)[predicciones[k]]))
    geo_test['pred_'+tarea.lower()]=geo_test.texto_norm.map(dic)
    print('Predicción cartográfica fija:',k)
geo_test.to_csv(SALIDA/'PREDICCIONES_TEST_POR_ESTADO.csv',index=False)






CLASES_MULTI=ETIQUETAS_MULTI

def campo_clase(c):return 'pred_'+re.sub('[^a-z0-9]+','_',sin_acentos(c).lower()).strip('_')+'_pct'
CAMPOS=[]
if 'pred_binaria' in geo_test.columns:CAMPOS.append('violencia_predicha_pct')
if 'pred_multiclase' in geo_test.columns:CAMPOS.extend(campo_clase(c) for c in CLASES_MULTI)
if not CAMPOS:raise RuntimeError('No hay predicciones para mapas')

def resumen_geografico(nivel):
    salida=[]
    for lugar,g in geo_test.groupby(nivel):
        if lugar=='SIN_DATO':continue
        fila={nivel:lugar,'n_publicaciones':len(g),'n_textos':g.texto_norm.nunique()}
        if 'pred_binaria' in g:
            fila['violencia_predicha_pct']=100*g.pred_binaria.eq('Violencia').mean()
            fila['violencia_predicha_n']=int(g.pred_binaria.eq('Violencia').sum())
        if 'pred_multiclase' in g:
            for c in CLASES_MULTI:
                fila[campo_clase(c)]=100*g.pred_multiclase.eq(c).mean()
                fila[campo_clase(c).replace('_pct','_n')]=int(g.pred_multiclase.eq(c).sum())
        salida.append(fila)
    return pd.DataFrame(salida)
res_est=resumen_geografico('estado')
res_reg=resumen_geografico('region')
res_est.to_csv(SALIDA/'RESUMEN_ESTADO.csv',index=False,encoding='utf-8-sig')
res_reg.to_csv(SALIDA/'RESUMEN_REGION.csv',index=False,encoding='utf-8-sig')





MORADOS=['#E4D5F4','#9465C5','#3B1465']  
GRIS='#E5E7EB'
ETIQUETAS_RANGOS=['0–33 %','>33–66 %','>66–100 %']
ALIAS={'distritofederal':'ciudaddemexico','mexicocity':'ciudaddemexico',
       'veracruzdeignaciodelallave':'veracruz','michoacandeocampo':'michoacan',
       'coahuiladezaragoza':'coahuila','estadodemexico':'mexico','mexicostate':'mexico',
       'queretarodearteaga':'queretaro'}
ABREV={'ciudaddemexico':'CDMX','mexico':'Edo. Méx.','tlaxcala':'Tlax.',
       'morelos':'Mor.','aguascalientes':'Ags.','queretaro':'Qro.',
       'nuevoleon':'N. León','sanluispotosi':'SLP','bajacaliforniasur':'BCS'}


POSICIONES={
 'veracruz':(-94.0,22.8), 'tabasco':(-90.9,16.3),
 'ciudaddemexico':(-98.0,21.0), 'mexico':(-102.0,20.4),
 'tlaxcala':(-96.0,21.8), 'morelos':(-100.8,16.4),
 'queretaro':(-99.0,22.1), 'aguascalientes':(-101.4,24.3),
 'colima':(-106.0,18.4), 'puebla':(-96.5,18.8)
}

def tono(v):
    if pd.isna(v):return GRIS
    if v<=33:return MORADOS[0]
    if v<=66:return MORADOS[1]
    return MORADOS[2]

def dibujar_mapa(geo,base,borde,campo,nivel,titulo,carpeta):
    """Coropleta por porcentaje, 3 morados, etiqueta de nombre y porcentaje solamente."""
    if campo not in geo or not geo[campo].notna().any():return
    fig,ax=plt.subplots(figsize=(19,11),facecolor='white')
    base.plot(ax=ax,color=GRIS,edgecolor='#9da2ac',linewidth=.55,zorder=1)
    dib=geo[geo[campo].notna()].copy()
    dib.plot(ax=ax,color=dib[campo].map(tono).tolist(),edgecolor='#51415e',
             linewidth=.8 if nivel=='estado' else 1.5,zorder=2)
    base.boundary.plot(ax=ax,color='#777583',linewidth=.45,zorder=3)
    if nivel=='region':dib.boundary.plot(ax=ax,color='#29153f',linewidth=1.9,zorder=4)
    borde.boundary.plot(ax=ax,color='#17121d',linewidth=1.3,zorder=5)

    for _,fila in dib.iterrows():
        nombre=str(fila['nombre_estado'] if nivel=='estado' else fila['region'])
        k=ALIAS.get(clave(nombre),clave(nombre))
        nombre_corto=ABREV.get(k,nombre) if nivel=='estado' else nombre
        porcentaje=float(fila[campo])
        
        etiqueta=f'{nombre_corto}\n{porcentaje:.1f}%'
        p=fila.geometry.representative_point()
        
        if nivel=='estado' and k in POSICIONES:
            ax.annotate(etiqueta,xy=(p.x,p.y),xytext=POSICIONES[k],
                fontsize=8,linespacing=1.18,fontweight='semibold',color='#241431',
                ha='center',va='center',
                bbox=dict(boxstyle='round,pad=.25',facecolor='white',edgecolor='#9f91af',alpha=.97),
                arrowprops=dict(arrowstyle='-',color='#51415e',lw=.9,
                                connectionstyle='angle3,angleA=0,angleB=90'),zorder=8)
        else:
            ax.text(p.x,p.y,etiqueta,fontsize=7.5 if nivel=='estado' else 10,
                linespacing=1.18,weight='semibold',ha='center',va='center',color='#28143b',
                bbox=dict(boxstyle='round,pad=.16',facecolor='white',edgecolor='none',alpha=.85),zorder=7)

    
    parches=[Patch(facecolor=MORADOS[i],edgecolor='#47334f',label=ETIQUETAS_RANGOS[i]) for i in (2,1,0)]
    parches.append(Patch(facecolor=GRIS,edgecolor='#9da2ac',label='Sin datos'))
    ax.legend(handles=parches,title='% de publicaciones de la zona',loc='center left',
              bbox_to_anchor=(1.01,.62),frameon=True,fontsize=10,title_fontsize=11,
              labelspacing=.8,borderpad=.9)
    ax.set_title(f'Predicción: {titulo} | {nivel.title()}',loc='left',fontsize=18,
                 weight='bold',color='#32104f',pad=20)
    ax.text(.01,.02,'TEST (20% de textos únicos) · Etiqueta: porcentaje de publicaciones predichas'
            '\nGris: sin datos · Los porcentajes corresponden al conjunto evaluado, no a la población general.',
            transform=ax.transAxes,fontsize=9,color='#5b5363',va='bottom')
    ax.set_axis_off();ax.set_aspect('equal')
    
    minx,miny,maxx,maxy=base.total_bounds
    ax.set_xlim(minx-1.0,maxx+2.0);ax.set_ylim(miny-1.3,maxy+1.2)
    fig.tight_layout()
    carpeta.mkdir(parents=True,exist_ok=True)
    fig.savefig(carpeta/f'{nivel}_{campo}.png',dpi=300,bbox_inches='tight',facecolor='white')
    if MOSTRAR_MAPAS:plt.show()
    plt.close(fig)

try:
    ruta_geo=SALIDA/'mexico_estados.geojson'
    if not ruta_geo.exists():
        meta=requests.get('https://www.geoboundaries.org/api/current/gbOpen/MEX/ADM1/',timeout=40)
        meta.raise_for_status()
        meta=meta.json()
        url=meta.get('simplifiedGeometryGeoJSON') or meta.get('gjDownloadURL')
        if not url:raise RuntimeError('geoBoundaries no ofreció el archivo de estados.')
        descarga=requests.get(url,timeout=100);descarga.raise_for_status()
        ruta_geo.write_bytes(descarga.content)
    mapa=gpd.read_file(ruta_geo).to_crs('EPSG:4326')
    nombre_col=next((x for x in ['shapeName','NAME_1','name','NOMGEO'] if x in mapa),None)
    if nombre_col is None:raise ValueError('No se encontró la columna de nombres estatales.')
    mapa['nombre_estado']=mapa[nombre_col].astype(str)
    mapa['clave']=mapa.nombre_estado.map(clave).replace(ALIAS)
    res_est['clave']=res_est.estado.map(clave).replace(ALIAS)
    estados=mapa.merge(res_est.drop(columns='estado'),on='clave',how='left',validate='one_to_one')
    asignacion=df[['estado','region']].drop_duplicates().copy()
    
    asignacion=asignacion.groupby('estado',as_index=False).region.agg(lambda s:s.mode().iloc[0])
    asignacion['clave']=asignacion.estado.map(clave).replace(ALIAS)
    estados=estados.merge(asignacion[['clave','region']],on='clave',how='left',validate='one_to_one')
    regiones=estados[estados.region.notna()].dissolve(by='region').reset_index()[['region','geometry']]
    regiones=regiones.merge(res_reg,on='region',how='left',validate='one_to_one')
    base_mx=mapa[['geometry']].copy()
    borde=gpd.GeoSeries([mapa.geometry.union_all()],crs=mapa.crs)
    print('Estados coincidentes:',int(estados.n_publicaciones.notna().sum()),'de',len(mapa))
    for nivel,gdf in [('estado',estados),('region',regiones)]:
        carpeta=SALIDA/'MAPAS_3_MORADOS_SOLO_PORCENTAJE'/nivel
        for campo in CAMPOS:
            titulo=('Violencia digital (binaria)' if campo=='violencia_predicha_pct'
                    else campo.replace('pred_','').replace('_pct','').replace('_',' ').title())
            dibujar_mapa(gdf,base_mx,borde,campo,nivel,titulo,carpeta)
        gdf.to_file(SALIDA/f'GEODATOS_{nivel}.gpkg',driver='GPKG')
    print('Mapas guardados en',SALIDA/'MAPAS_3_MORADOS_SOLO_PORCENTAJE')
except Exception as ex:
    print('ADVERTENCIA: falló la sección cartográfica:',type(ex).__name__,str(ex))
    print('Las tablas, las predicciones y las matrices sí se conservaron.')




ZIP=RAIZ/'RESULTADOS_VIOLENCIA_3_MORADOS_SOLO_PORCENTAJE.zip'
with zipfile.ZipFile(ZIP,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for archivo in SALIDA.rglob('*'):
        if archivo.is_file():z.write(archivo,arcname=str(archivo.relative_to(SALIDA)))
print('\nResultados:',SALIDA,'\nZIP:',ZIP)
if 'google.colab' in sys.modules:files.download(str(ZIP))


### EXP-059 · V1, V2, V3, V4

**Archivo de origen:** `27_Octubre.ipynb, celda 33`. **Referencias duplicadas:** 0. **Sintaxis de código:** Válida.

GOOGLE COLAB | VIOLENCIA DIGITAL: BINARIA Y MULTICLASE
TF-IDF, N-GRAMAS, BOW, EMBEDDINGS, HMVT V1-V4, CON/SIN REGIONALISMOS
8 CLASIFICADORES | MATRICES MORADAS | MAPAS POR ESTADO Y REGIÓN
MAPAS: 3 RANGOS MORADOS DINAMICOS POR MAPA; SOLO POR

**Ejecución:** celda independiente o variante experimental; revisar archivos de entrada y dependencias antes de ejecutarla. Los resultados originales no se insertan en esta celda.

In [ ]:
import sys, re, time, zipfile, warnings, subprocess, importlib.util, unicodedata
from pathlib import Path
from io import BytesIO
warnings.filterwarnings('ignore', category=FutureWarning)
SEMILLA = 42
TEST_SIZE = .20
PRESUPUESTO_MODELOS_S = 20 * 60
MOSTRAR_MAPAS = True
MOSTRAR_MATRICES = False
EMBEDDING_MODELO = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
MODELOS = ['RL','KNN','SVM','NB','MLP','XGB','RF','ARBOL']
BASES = ['TFIDF','NGRAMAS','BOW','EMBEDDINGS']
HMVT = ['HMVT-V1','HMVT-V2','HMVT-V3','HMVT-V4']
REPRESENTACIONES = BASES + HMVT + [n+'+REG' for n in BASES+HMVT]
RAIZ = Path('/content') if 'google.colab' in sys.modules else Path('/mnt/data')
SALIDA = RAIZ / 'RESULTADOS_VIOLENCIA_3_MORADOS_DINAMICOS'
SALIDA.mkdir(parents=True,exist_ok=True)
INICIO = time.monotonic()

def dependencia(mod, paquete=None):
    if importlib.util.find_spec(mod) is None:
        subprocess.check_call([sys.executable,'-m','pip','-q','install',paquete or mod])
for mod, paquete in [('numpy',None),('pandas',None),('scipy',None),('sklearn','scikit-learn'),
                     ('matplotlib',None),('seaborn',None),('xgboost',None),
                     ('sentence_transformers','sentence-transformers'),('geopandas',None),('requests',None)]:
    dependencia(mod,paquete)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import geopandas as gpd
import requests
from scipy import sparse
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler, MultiLabelBinarizer
from sklearn.metrics import (confusion_matrix,classification_report,precision_score,
                             recall_score,f1_score,accuracy_score,balanced_accuracy_score)
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from sentence_transformers import SentenceTransformer
from matplotlib.patches import Patch
from IPython.display import display
sns.set_theme(style='white')
np.random.seed(SEMILLA)

def sin_acentos(s):
    return ''.join(c for c in unicodedata.normalize('NFKD',str(s)) if not unicodedata.combining(c))
def clave(s):
    return re.sub(r'[^a-z0-9]','',sin_acentos(str(s)).lower())
def texto_limpio(s):
    s=str(s).lower().strip()
    s=re.sub(r'https?://\S+|www\.\S+',' URL ',s)
    s=re.sub(r'@\w+',' USUARIO ',s)
    return re.sub(r'\s+',' ',s)




if 'google.colab' in sys.modules:
    from google.colab import files
    print('Sube data_hmvt_con_estado.csv')
    subidos=files.upload()
    if len(subidos)!=1: raise ValueError('Selecciona un solo archivo CSV.')
    nombre,contenido=next(iter(subidos.items()))
    df=pd.read_csv(BytesIO(contenido),encoding='utf-8-sig')
else:
    opciones=[RAIZ/'data_hmvt_con_estado(1).csv',RAIZ/'data_hmvt_con_estado.csv']
    archivo=next((p for p in opciones if p.exists()),None)
    if archivo is None: raise FileNotFoundError('Falta data_hmvt_con_estado.csv')
    df=pd.read_csv(archivo,encoding='utf-8-sig')
requeridas={'tweet','clase','violencia_binaria','estado','region','regionalismo'}
if requeridas-set(df.columns):raise ValueError('Faltan columnas: '+str(requeridas-set(df.columns)))
df=df.dropna(subset=['tweet','clase','violencia_binaria']).copy()
for c in ['tweet','clase','estado','region','regionalismo']:
    df[c]=df[c].fillna('SIN_DATO').astype(str).str.strip()
df['violencia_binaria']=pd.to_numeric(df['violencia_binaria'],errors='raise').astype(int)
if not set(df.violencia_binaria.unique()).issubset({0,1}):raise ValueError('Binaria requiere 0 y 1')
df['texto_norm']=df.tweet.map(texto_limpio)
df=df[df.texto_norm.str.len()>0].copy()
if df.groupby('texto_norm').clase.nunique().max()>1:
    raise ValueError('Un texto tiene múltiples clases de violencia: revisar anotaciones.')
if df.groupby('texto_norm').violencia_binaria.nunique().max()>1:
    raise ValueError('Un texto tiene etiquetas binarias contradictorias.')

corpus=df.drop_duplicates('texto_norm').reset_index(drop=True)
idx_tr,idx_te=train_test_split(np.arange(len(corpus)),test_size=TEST_SIZE,
                               stratify=corpus.clase,random_state=SEMILLA)
tr=corpus.iloc[idx_tr].reset_index(drop=True)
te=corpus.iloc[idx_te].reset_index(drop=True)
assert set(tr.texto_norm).isdisjoint(set(te.texto_norm))
tr.to_csv(SALIDA/'train_textos_unicos.csv',index=False)
te.to_csv(SALIDA/'test_textos_unicos.csv',index=False)
print('Registros originales:',len(df),'Textos únicos:',len(corpus),
      'TRAIN:',len(tr),'TEST:',len(te))
print('Distribución de TEST:',te.clase.value_counts().to_dict())




REP={}
def guardar_rep(nombre,a,b):
    REP[nombre]=(sparse.csr_matrix(a,dtype=np.float32),
                 sparse.csr_matrix(b,dtype=np.float32))
for nombre,vector in [
    ('TFIDF',TfidfVectorizer(max_features=7000,ngram_range=(1,2),strip_accents='unicode')),
    ('NGRAMAS',TfidfVectorizer(max_features=8000,analyzer='char',ngram_range=(3,5))),
    ('BOW',CountVectorizer(max_features=7000,strip_accents='unicode'))]:
    guardar_rep(nombre,vector.fit_transform(tr.texto_norm),vector.transform(te.texto_norm))
modelo_embeddings=SentenceTransformer(EMBEDDING_MODELO)
todos_textos=list(dict.fromkeys(tr.texto_norm.tolist()+te.texto_norm.tolist()))
e=modelo_embeddings.encode(todos_textos,batch_size=32,show_progress_bar=True,normalize_embeddings=True)
dic_emb=dict(zip(todos_textos,e))
arr_tr=np.stack([dic_emb[x] for x in tr.texto_norm])
arr_te=np.stack([dic_emb[x] for x in te.texto_norm])
sc=MinMaxScaler().fit(arr_tr)
guardar_rep('EMBEDDINGS',np.maximum(0,sc.transform(arr_tr)),np.maximum(0,sc.transform(arr_te)))
del modelo_embeddings,e,dic_emb




BLOQUES={
 'HMVT-V1':[('TFIDF',1.),('NGRAMAS',1.)],
 'HMVT-V2':[('TFIDF',1.),('BOW',.3),('NGRAMAS',.7)],
 'HMVT-V3':[('TFIDF',1.),('NGRAMAS',.7),('EMBEDDINGS',2.)],
 'HMVT-V4':[('TFIDF',1.),('NGRAMAS',.7),('BOW',.3),('EMBEDDINGS',2.)]
}
for nombre,partes in BLOQUES.items():
    guardar_rep(nombre,*[
        sparse.hstack([REP[k][i]*w for k,w in partes],format='csr') for i in (0,1)
    ])






vocab=sorted({sin_acentos(x.lower()) for x in tr.regionalismo if x!='SIN_DATO' and x.strip()})
patrones=[(v,re.compile(r'(?<!\w)'+re.escape(v)+r'(?!\w)',re.I)) for v in vocab]
def detectar_regionalismos(t):
    s=sin_acentos(str(t).lower())
    return [v for v,p in patrones if p.search(s)]
mlb=MultiLabelBinarizer(classes=vocab,sparse_output=True)
reg_tr=mlb.fit_transform([detectar_regionalismos(x) for x in tr.tweet]).astype(np.float32)
reg_te=mlb.transform([detectar_regionalismos(x) for x in te.tweet]).astype(np.float32)
print('Regionalismos del vocabulario:',len(vocab),
      'Textos TEST con coincidencias:',int(np.sum(reg_te.getnnz(axis=1)>0)))
for nombre in BASES+HMVT:
    a,b=REP[nombre]
    guardar_rep(nombre+'+REG',sparse.hstack([a,reg_tr],format='csr'),
               sparse.hstack([b,reg_te],format='csr'))




def crear_modelo(k,nclases):
    if k=='RL':return LogisticRegression(max_iter=650,random_state=SEMILLA)
    if k=='KNN':return KNeighborsClassifier(n_neighbors=3)
    if k=='SVM':return LinearSVC(max_iter=1800,random_state=SEMILLA)
    if k=='NB':return MultinomialNB()
    if k=='MLP':return MLPClassifier(hidden_layer_sizes=(32,),max_iter=70,
                                     early_stopping=True,random_state=SEMILLA)
    if k=='XGB':return XGBClassifier(n_estimators=55,max_depth=3,learning_rate=.1,
                   random_state=SEMILLA,n_jobs=2,tree_method='hist',
                   eval_metric='logloss' if nclases==2 else 'mlogloss')
    if k=='RF':return RandomForestClassifier(n_estimators=60,random_state=SEMILLA,n_jobs=2)
    if k=='ARBOL':return DecisionTreeClassifier(random_state=SEMILLA)
    raise ValueError(k)
PARAM={
 'RL':('C',[.3,1.]),'KNN':('n_neighbors',[3,5]),'SVM':('C',[.3,1.]),
 'NB':('alpha',[.5,1.]),'MLP':('alpha',[.001,.01]),
 'XGB':('max_depth',[2,3]),'RF':('max_depth',[7,None]),
 'ARBOL':('max_depth',[6,None])
}
ETIQUETAS_MULTI=sorted(corpus.clase.unique().tolist())
TAREAS={'BINARIA':('violencia_binaria',['No violencia','Violencia']),
        'MULTICLASE':('clase',ETIQUETAS_MULTI)}
resultados=[];desglose=[];predicciones={};errores=[];optimizacion=[]
DIR_CM=SALIDA/'matrices_csv';DIR_CM.mkdir(exist_ok=True)
for tarea,(col,etiquetas) in TAREAS.items():
    if tarea=='BINARIA':
        ytr=tr.violencia_binaria.to_numpy();yte=te.violencia_binaria.to_numpy()
    else:
        etiquetas_id={x:i for i,x in enumerate(etiquetas)}
        ytr=tr.clase.map(etiquetas_id).to_numpy()
        yte=te.clase.map(etiquetas_id).to_numpy()
    for representacion in REPRESENTACIONES:
        xtr,xte=REP[representacion]
        for alg in MODELOS:
            if time.monotonic()-INICIO>PRESUPUESTO_MODELOS_S:
                errores.append([tarea,representacion,alg,'No iniciado por límite de tiempo'])
                continue
            try:
                m=crear_modelo(alg,len(etiquetas))
                if representacion.startswith('HMVT-'):
                    
                    it,iv=train_test_split(np.arange(len(ytr)),test_size=.20,
                        stratify=ytr if min(np.bincount(ytr))>=2 else None,
                        random_state=SEMILLA+15)
                    par,valores=PARAM[alg]
                    evaluados=[]
                    for valor in valores:
                        cand=crear_modelo(alg,len(etiquetas)).set_params(**{par:valor})
                        cand.fit(xtr[it],ytr[it])
                        pp=cand.predict(xtr[iv])
                        evaluados.append((f1_score(ytr[iv],pp,average='macro',zero_division=0),valor))
                    fval,param=max(evaluados,key=lambda q:q[0]);m.set_params(**{par:param})
                    optimizacion.append([tarea,representacion,alg,par,param,fval])
                m.fit(xtr,ytr)
                pred=m.predict(xte)
                predicciones[(tarea,representacion,alg)]=pred
                cm=confusion_matrix(yte,pred,labels=np.arange(len(etiquetas)))
                pd.DataFrame(cm,index=etiquetas,columns=etiquetas).to_csv(
                    DIR_CM/f'{tarea}_{representacion}_{alg}.csv')
                resultados.append([tarea,representacion,alg,
                  precision_score(yte,pred,average='macro',zero_division=0),
                  recall_score(yte,pred,average='macro',zero_division=0),
                  f1_score(yte,pred,average='macro',zero_division=0),
                  accuracy_score(yte,pred),balanced_accuracy_score(yte,pred)])
                rep_clase=classification_report(yte,pred,labels=np.arange(len(etiquetas)),
                                                output_dict=True,zero_division=0)
                for i,nombre_clase in enumerate(etiquetas):
                    r=rep_clase[str(i)]
                    desglose.append([tarea,representacion,alg,nombre_clase,
                                     r['precision'],r['recall'],r['f1-score'],r['support']])
            except Exception as ex:
                errores.append([tarea,representacion,alg,str(ex)])
                print('Error:',tarea,representacion,alg,str(ex)[:100])
        print('Evaluado:',tarea,representacion)
col_res=['Tarea','Representacion','Modelo','Precision_macro','Recall_macro',
         'F1_macro','Accuracy','Balanced_accuracy']
tabla=pd.DataFrame(resultados,columns=col_res)
if tabla.empty:raise RuntimeError('No se obtuvieron predicciones. Revisa errores.')
tabla.to_csv(SALIDA/'TABLA_TODOS_MODELOS.csv',index=False)
pd.DataFrame(desglose,columns=['Tarea','Representacion','Modelo','Clase','Precision','Recall',
                                'F1','Soporte']).to_csv(SALIDA/'METRICAS_POR_CLASE.csv',index=False)
pd.DataFrame(errores,columns=['Tarea','Representacion','Modelo','Motivo']).to_csv(SALIDA/'PENDIENTES.csv',index=False)
pd.DataFrame(optimizacion,columns=['Tarea','Representacion','Modelo','Parametro','Valor',
                                    'F1_validacion']).to_csv(SALIDA/'OPTIMIZACION_HMVT.csv',index=False)
for tarea in TAREAS:
    print('\n', '='*15, tarea, '='*15)
    for titulo,reps in [('1_CLASICAS',BASES),('2_HMVT',HMVT),
                        ('3_CON_REGIONALISMOS',[x+'+REG' for x in BASES+HMVT])]:
        sub=tabla[(tabla.Tarea==tarea)&tabla.Representacion.isin(reps)].copy()
        sub['orden']=sub.Representacion.map({x:i for i,x in enumerate(reps)})
        sub=sub.sort_values(['orden','Modelo']).drop(columns='orden')
        print('\n',titulo)
        if not sub.empty:
            display(sub.style.format(precision=4).background_gradient(cmap='Purples',subset=['F1_macro']))
        sub.to_csv(SALIDA/f'{tarea}_{titulo}.csv',index=False)




DIR_FIG=SALIDA/'matrices_moradas';DIR_FIG.mkdir(exist_ok=True)
for tarea,(_,etiquetas) in TAREAS.items():
    for alg in MODELOS:
        fig,axes=plt.subplots(4,4,figsize=(24,20) if tarea=='BINARIA' else (28,27))
        for ax,representacion in zip(axes.flat,REPRESENTACIONES):
            k=(tarea,representacion,alg)
            if k not in predicciones:
                ax.axis('off');continue
            cm=pd.read_csv(DIR_CM/f'{tarea}_{representacion}_{alg}.csv',index_col=0).values
            sns.heatmap(cm,ax=ax,cmap='Purples',annot=True,fmt='d',cbar=False,
                        linewidths=.45,linecolor='white',xticklabels=etiquetas,yticklabels=etiquetas,
                        annot_kws={'size':8 if tarea=='MULTICLASE' else 12,'weight':'bold'})
            fila=tabla[(tabla.Tarea==tarea)&(tabla.Representacion==representacion)&
                       (tabla.Modelo==alg)].iloc[0]
            ax.set_title(f'{representacion}\nF1={fila.F1_macro:.3f} · Acc={fila.Accuracy:.3f}',
                         fontsize=12,color='#3b0764',weight='bold')
            ax.tick_params(axis='x',rotation=55,labelsize=7)
            ax.tick_params(axis='y',rotation=0,labelsize=7)
            ax.set_xlabel('Predicha');ax.set_ylabel('Real')
        fig.suptitle(f'VIOLENCIA DIGITAL | {tarea} | {alg} | TEST 20%',
                     fontsize=19,color='#3b0764',weight='bold')
        fig.tight_layout(rect=(0,0,1,.97))
        fig.savefig(DIR_FIG/f'{tarea}_{alg}.png',dpi=190,bbox_inches='tight')
        if MOSTRAR_MATRICES:plt.show()
        plt.close(fig)





def clave_mapa(tarea):
    preferencias=[(tarea,'HMVT-V4+REG','RL'),(tarea,'HMVT-V4','RL'),
                  (tarea,'TFIDF+REG','RL'),(tarea,'TFIDF','RL')]
    return next((k for k in preferencias if k in predicciones),None)


geo_test=df[df.texto_norm.isin(set(te.texto_norm))].drop_duplicates(['texto_norm','estado']).copy()
for tarea in TAREAS:
    k=clave_mapa(tarea)
    if k is None:
        print('Sin predicción para mapas:',tarea)
        continue
    etiquetas=TAREAS[tarea][1]
    dic=dict(zip(te.texto_norm,np.asarray(etiquetas,dtype=object)[predicciones[k]]))
    geo_test['pred_'+tarea.lower()]=geo_test.texto_norm.map(dic)
    print('Predicción cartográfica fija:',k)
geo_test.to_csv(SALIDA/'PREDICCIONES_TEST_POR_ESTADO.csv',index=False)






CLASES_MULTI=ETIQUETAS_MULTI

def campo_clase(c):return 'pred_'+re.sub('[^a-z0-9]+','_',sin_acentos(c).lower()).strip('_')+'_pct'
CAMPOS=[]
if 'pred_binaria' in geo_test.columns:CAMPOS.append('violencia_predicha_pct')
if 'pred_multiclase' in geo_test.columns:CAMPOS.extend(campo_clase(c) for c in CLASES_MULTI)
if not CAMPOS:raise RuntimeError('No hay predicciones para mapas')

def resumen_geografico(nivel):
    salida=[]
    for lugar,g in geo_test.groupby(nivel):
        if lugar=='SIN_DATO':continue
        fila={nivel:lugar,'n_publicaciones':len(g),'n_textos':g.texto_norm.nunique()}
        if 'pred_binaria' in g:
            fila['violencia_predicha_pct']=100*g.pred_binaria.eq('Violencia').mean()
            fila['violencia_predicha_n']=int(g.pred_binaria.eq('Violencia').sum())
        if 'pred_multiclase' in g:
            for c in CLASES_MULTI:
                fila[campo_clase(c)]=100*g.pred_multiclase.eq(c).mean()
                fila[campo_clase(c).replace('_pct','_n')]=int(g.pred_multiclase.eq(c).sum())
        salida.append(fila)
    return pd.DataFrame(salida)
res_est=resumen_geografico('estado')
res_reg=resumen_geografico('region')
res_est.to_csv(SALIDA/'RESUMEN_ESTADO.csv',index=False,encoding='utf-8-sig')
res_reg.to_csv(SALIDA/'RESUMEN_REGION.csv',index=False,encoding='utf-8-sig')







MORADOS=['#E4D5F4','#9465C5','#3B1465']  
GRIS='#E5E7EB'


def calcular_rangos_dinamicos(valores):
    """Tres intervalos de igual amplitud entre el mínimo y máximo observados.

    Si todos los porcentajes coinciden, se usa un tono medio y una leyenda
    explícita: no hay diferencias relativas entre estados o regiones.
    """
    v=pd.to_numeric(pd.Series(valores),errors='coerce').dropna()
    if v.empty:
        raise ValueError('No hay porcentajes para el mapa')
    minimo=float(v.min());maximo=float(v.max())
    iguales=np.isclose(minimo,maximo,atol=1e-10,rtol=0)
    limites=np.linspace(minimo,maximo,4) if not iguales else np.array([minimo]*4)
    return limites,iguales


def tono_dinamico(valor,limites,iguales):
    if pd.isna(valor):return GRIS
    if iguales:return MORADOS[1]
    if valor<=limites[1]:return MORADOS[0]
    if valor<=limites[2]:return MORADOS[1]
    return MORADOS[2]


def etiquetas_leyenda(limites,iguales):
    if iguales:
        return [(MORADOS[1],f'Todos: {limites[0]:.1f} % (sin variación)')]
    
    return [
        (MORADOS[2],f'> {limites[2]:.2f} a {limites[3]:.2f} %'),
        (MORADOS[1],f'> {limites[1]:.2f} a {limites[2]:.2f} %'),
        (MORADOS[0],f'{limites[0]:.2f} a {limites[1]:.2f} %'),
    ]

ALIAS={'distritofederal':'ciudaddemexico','mexicocity':'ciudaddemexico',
       'veracruzdeignaciodelallave':'veracruz','michoacandeocampo':'michoacan',
       'coahuiladezaragoza':'coahuila','estadodemexico':'mexico','mexicostate':'mexico',
       'queretarodearteaga':'queretaro'}
ABREV={'ciudaddemexico':'CDMX','mexico':'Edo. Méx.','tlaxcala':'Tlax.',
       'morelos':'Mor.','aguascalientes':'Ags.','queretaro':'Qro.',
       'nuevoleon':'N. León','sanluispotosi':'SLP','bajacaliforniasur':'BCS'}


POSICIONES={
 'veracruz':(-94.0,22.8), 'tabasco':(-90.9,16.3),
 'ciudaddemexico':(-98.0,21.0), 'mexico':(-102.0,20.4),
 'tlaxcala':(-96.0,21.8), 'morelos':(-100.8,16.4),
 'queretaro':(-99.0,22.1), 'aguascalientes':(-101.4,24.3),
 'colima':(-106.0,18.4), 'puebla':(-96.5,18.8)
}

def dibujar_mapa(geo,base,borde,campo,nivel,titulo,carpeta):
    """Coropleta por porcentaje, 3 morados, etiqueta de nombre y porcentaje solamente."""
    if campo not in geo or not geo[campo].notna().any():return
    fig,ax=plt.subplots(figsize=(19,11),facecolor='white')
    base.plot(ax=ax,color=GRIS,edgecolor='#9da2ac',linewidth=.55,zorder=1)
    dib=geo[geo[campo].notna()].copy()
    limites,iguales=calcular_rangos_dinamicos(dib[campo])
    dib['color_mapa']=dib[campo].apply(lambda v:tono_dinamico(v,limites,iguales))
    dib.plot(ax=ax,color=dib['color_mapa'].tolist(),edgecolor='#51415e',
             linewidth=.8 if nivel=='estado' else 1.5,zorder=2)
    base.boundary.plot(ax=ax,color='#777583',linewidth=.45,zorder=3)
    if nivel=='region':dib.boundary.plot(ax=ax,color='#29153f',linewidth=1.9,zorder=4)
    borde.boundary.plot(ax=ax,color='#17121d',linewidth=1.3,zorder=5)

    for _,fila in dib.iterrows():
        nombre=str(fila['nombre_estado'] if nivel=='estado' else fila['region'])
        k=ALIAS.get(clave(nombre),clave(nombre))
        nombre_corto=ABREV.get(k,nombre) if nivel=='estado' else nombre
        porcentaje=float(fila[campo])
        
        etiqueta=f'{nombre_corto}\n{porcentaje:.1f}%'
        p=fila.geometry.representative_point()
        
        if nivel=='estado' and k in POSICIONES:
            ax.annotate(etiqueta,xy=(p.x,p.y),xytext=POSICIONES[k],
                fontsize=8,linespacing=1.18,fontweight='semibold',color='#241431',
                ha='center',va='center',
                bbox=dict(boxstyle='round,pad=.25',facecolor='white',edgecolor='#9f91af',alpha=.97),
                arrowprops=dict(arrowstyle='-',color='#51415e',lw=.9,
                                connectionstyle='angle3,angleA=0,angleB=90'),zorder=8)
        else:
            ax.text(p.x,p.y,etiqueta,fontsize=7.5 if nivel=='estado' else 10,
                linespacing=1.18,weight='semibold',ha='center',va='center',color='#28143b',
                bbox=dict(boxstyle='round,pad=.16',facecolor='white',edgecolor='none',alpha=.85),zorder=7)

    carpeta.mkdir(parents=True,exist_ok=True)
    
    pd.DataFrame({'limite_inferior':[limites[0],limites[1],limites[2]],
                  'limite_superior':[limites[1],limites[2],limites[3]],
                  'color':MORADOS, 'todos_iguales':[iguales]*3}).to_csv(
        carpeta/f'{nivel}_{campo}_rangos.csv',index=False)
    
    parches=[Patch(facecolor=color,edgecolor='#47334f',label=rotulo)
             for color,rotulo in etiquetas_leyenda(limites,iguales)]
    parches.append(Patch(facecolor=GRIS,edgecolor='#9da2ac',label='Sin datos'))
    ax.legend(handles=parches,title='% predicho · rangos de este mapa',loc='center left',
              bbox_to_anchor=(1.01,.62),frameon=True,fontsize=10,title_fontsize=11,
              labelspacing=.8,borderpad=.9)
    ax.set_title(f'Predicción: {titulo} | {nivel.title()}',loc='left',fontsize=18,
                 weight='bold',color='#32104f',pad=20)
    ax.text(.01,.02,'TEST (20% de textos únicos) · Tres rangos relativos a cada mapa · Etiqueta: % predicho'
            '\nGris: sin datos · Los porcentajes corresponden al conjunto evaluado, no a la población general.',
            transform=ax.transAxes,fontsize=9,color='#5b5363',va='bottom')
    ax.set_axis_off();ax.set_aspect('equal')
    
    minx,miny,maxx,maxy=base.total_bounds
    ax.set_xlim(minx-1.0,maxx+2.0);ax.set_ylim(miny-1.3,maxy+1.2)
    fig.tight_layout()
    carpeta.mkdir(parents=True,exist_ok=True)
    fig.savefig(carpeta/f'{nivel}_{campo}.png',dpi=300,bbox_inches='tight',facecolor='white')
    if MOSTRAR_MAPAS:plt.show()
    plt.close(fig)

try:
    ruta_geo=SALIDA/'mexico_estados.geojson'
    if not ruta_geo.exists():
        meta=requests.get('https://www.geoboundaries.org/api/current/gbOpen/MEX/ADM1/',timeout=40)
        meta.raise_for_status()
        meta=meta.json()
        url=meta.get('simplifiedGeometryGeoJSON') or meta.get('gjDownloadURL')
        if not url:raise RuntimeError('geoBoundaries no ofreció el archivo de estados.')
        descarga=requests.get(url,timeout=100);descarga.raise_for_status()
        ruta_geo.write_bytes(descarga.content)
    mapa=gpd.read_file(ruta_geo).to_crs('EPSG:4326')
    nombre_col=next((x for x in ['shapeName','NAME_1','name','NOMGEO'] if x in mapa),None)
    if nombre_col is None:raise ValueError('No se encontró la columna de nombres estatales.')
    mapa['nombre_estado']=mapa[nombre_col].astype(str)
    mapa['clave']=mapa.nombre_estado.map(clave).replace(ALIAS)
    res_est['clave']=res_est.estado.map(clave).replace(ALIAS)
    estados=mapa.merge(res_est.drop(columns='estado'),on='clave',how='left',validate='one_to_one')
    asignacion=df[['estado','region']].drop_duplicates().copy()
    
    asignacion=asignacion.groupby('estado',as_index=False).region.agg(lambda s:s.mode().iloc[0])
    asignacion['clave']=asignacion.estado.map(clave).replace(ALIAS)
    estados=estados.merge(asignacion[['clave','region']],on='clave',how='left',validate='one_to_one')
    regiones=estados[estados.region.notna()].dissolve(by='region').reset_index()[['region','geometry']]
    regiones=regiones.merge(res_reg,on='region',how='left',validate='one_to_one')
    base_mx=mapa[['geometry']].copy()
    borde=gpd.GeoSeries([mapa.geometry.union_all()],crs=mapa.crs)
    print('Estados coincidentes:',int(estados.n_publicaciones.notna().sum()),'de',len(mapa))
    for nivel,gdf in [('estado',estados),('region',regiones)]:
        carpeta=SALIDA/'MAPAS_3_MORADOS_DINAMICOS'/nivel
        for campo in CAMPOS:
            titulo=('Violencia digital (binaria)' if campo=='violencia_predicha_pct'
                    else campo.replace('pred_','').replace('_pct','').replace('_',' ').title())
            dibujar_mapa(gdf,base_mx,borde,campo,nivel,titulo,carpeta)
        gdf.to_file(SALIDA/f'GEODATOS_{nivel}.gpkg',driver='GPKG')
    print('Mapas guardados en',SALIDA/'MAPAS_3_MORADOS_DINAMICOS')
except Exception as ex:
    print('ADVERTENCIA: falló la sección cartográfica:',type(ex).__name__,str(ex))
    print('Las tablas, las predicciones y las matrices sí se conservaron.')




ZIP=RAIZ/'RESULTADOS_VIOLENCIA_3_MORADOS_DINAMICOS.zip'
with zipfile.ZipFile(ZIP,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for archivo in SALIDA.rglob('*'):
        if archivo.is_file():z.write(archivo,arcname=str(archivo.relative_to(SALIDA)))
print('\nResultados:',SALIDA,'\nZIP:',ZIP)
if 'google.colab' in sys.modules:files.download(str(ZIP))


### EXP-060 · V1, V2, V3, V4

**Archivo de origen:** `Untitled7.ipynb, celda 3`. **Referencias duplicadas:** 0. **Sintaxis de código:** Válida.

EXPERIMENTO DOCTORAL · CLASIFICACIÓN + VECTOR GEOLINGÜÍSTICO
UNA CELDA DE GOOGLE COLAB: pegue el contenido completo y ejecute.
Dataset esperado: data_hmvt_con_estado.csv
import os, sys, subprocess, importlib.util, warnings, re, unicodedata,

**Ejecución:** celda independiente o variante experimental; revisar archivos de entrada y dependencias antes de ejecutarla. Los resultados originales no se insertan en esta celda.

In [ ]:
import os, sys, subprocess, importlib.util, warnings, re, unicodedata, json, time, zipfile
from pathlib import Path
warnings.filterwarnings('ignore', category=FutureWarning)


ARCHIVO = 'data_hmvt_con_estado.csv'
SALIDA = Path('/content/resultados_geolinguisticos' if Path('/content').exists() else './resultados_geolinguisticos')
SALIDA.mkdir(parents=True, exist_ok=True)
SEMILLA = 42
FRACCION_TEST = 0.25



CLASIFICADORES = ['RL','SVM','NB','KNN','MLP','RF','XGB','ARBOL']
NOMBRES_MODELOS = {'RL':'Regresión logística','SVM':'Máquina de vectores de soporte',
                   'NB':'Naive Bayes','KNN':'KNN (3 vecinos)',
                   'MLP':'Red neuronal multicapa (MLP)',
                   'RF':'Random Forest','XGB':'XGBoost','ARBOL':'Árbol de decisión'}
print('Modelos incluidos:')
for codigo in CLASIFICADORES: print('  -',codigo,':',NOMBRES_MODELOS[codigo])

USAR_EMBEDDINGS_TRANSFORMER = True
MODELO_EMBEDDINGS = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
PRODUCIR_MAPAS = True
MOSTRAR_FIGURAS = True
UMBRAL_MIN_MUESTRAS = 5


def asegurar(modulo, pip_name=None):
    if importlib.util.find_spec(modulo) is None:
        subprocess.check_call([sys.executable,'-m','pip','-q','install',pip_name or modulo])
for mod, pkg in [('pandas',None),('numpy',None),('scipy',None),('sklearn','scikit-learn'),
                 ('matplotlib',None),('seaborn',None),('geopandas',None),
                 ('requests',None),('sentence_transformers','sentence-transformers'),('xgboost',None)]:
    if mod == 'sentence_transformers' and not USAR_EMBEDDINGS_TRANSFORMER: continue
    if mod == 'xgboost' and 'XGB' not in CLASIFICADORES: continue
    if mod == 'geopandas' and not PRODUCIR_MAPAS: continue
    asegurar(mod,pkg)
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from scipy import sparse
from sklearn.model_selection import GroupShuffleSplit
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler, Normalizer
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
                             precision_score, recall_score, classification_report,
                             confusion_matrix)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.base import clone
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
sns.set_theme(style='whitegrid', context='notebook')
np.random.seed(SEMILLA)




if 'google.colab' in sys.modules:
    from google.colab import files
    print('PASO 1: Sube tu conjunto de datos CSV (por ejemplo, data_hmvt_con_estado.csv).')
    subidos = files.upload()
    if not subidos:
        raise RuntimeError('No se cargó ningún archivo. Ejecuta la celda de nuevo y selecciona tu CSV.')
    if len(subidos) != 1:
        raise ValueError('Selecciona un solo conjunto de datos CSV por ejecución.')
    nombre_subido, contenido_csv = next(iter(subidos.items()))
    if not nombre_subido.lower().endswith('.csv'):
        raise ValueError('El archivo seleccionado debe tener extensión .csv.')
    from io import BytesIO
    df = pd.read_csv(BytesIO(contenido_csv), encoding='utf-8-sig')
    print('Archivo cargado:', nombre_subido, '| Registros originales:', len(df))
else:
    ruta = Path(ARCHIVO)
    if not ruta.exists():
        raise FileNotFoundError('Coloque '+ARCHIVO+' en el directorio actual o cambie ARCHIVO.')
    df = pd.read_csv(ruta, encoding='utf-8-sig')
requeridas = {'tweet','clase','region','regionalismo','violencia_binaria','estado'}
faltantes = requeridas - set(df.columns)
if faltantes: raise ValueError('Faltan columnas: '+str(sorted(faltantes)))
df = df.dropna(subset=['tweet','clase','region','estado','violencia_binaria']).copy()
for c in ['tweet','clase','region','regionalismo','estado']:
    df[c] = df[c].fillna('SIN_DATO').astype(str).str.strip()
df['violencia_binaria'] = pd.to_numeric(df['violencia_binaria'], errors='raise').astype(int)
df = df[df['tweet'].str.len()>0].reset_index(drop=True)
if not set(df['violencia_binaria'].unique()) <= {0,1}: raise ValueError('La etiqueta binaria debe ser 0 o 1.')
if df.groupby('clase')['violencia_binaria'].nunique().max() > 1:
    raise ValueError('Una misma clase multiclase contiene etiquetas binarias contradictorias.')

def normalizar_texto(t):
    t = str(t).lower().strip()
    t = re.sub(r'https?://\S+|www\.\S+', ' URL ', t)
    t = re.sub(r'@\w+', ' USUARIO ', t)
    return re.sub(r'\s+', ' ', t)

df['texto_norm'] = df['tweet'].map(normalizar_texto)
contradicciones = df.groupby('texto_norm')['clase'].nunique()
if (contradicciones > 1).any():
    print('AVISO: hay textos iguales con clases distintas; se mantienen pero NO se reparten entre train/test.')
print('Registros:',len(df),'| Textos únicos:',df.texto_norm.nunique(),
      '| Duplicados exactos:',df.duplicated(subset=['tweet','clase','region','regionalismo','violencia_binaria','estado']).sum())
print('Clases:',df.clase.value_counts().to_dict())
print('Estados:',df.estado.nunique(),'| Regiones:',df.region.nunique())
print('NOTA: regionalismo/estado/region se usan como metadatos anotados, no como hechos inferidos del texto.')



idx = np.arange(len(df))
etiq = sorted(df['clase'].unique())
mejor = None
for intento in range(60):
    gs = GroupShuffleSplit(n_splits=1,test_size=FRACCION_TEST,random_state=SEMILLA+intento)
    itrain, itest = next(gs.split(idx, groups=df['texto_norm']))
    presentes_train = set(df.iloc[itrain].clase)
    presentes_test = set(df.iloc[itest].clase)
    penalizacion = len(set(etiq)-presentes_train)*100 + len(set(etiq)-presentes_test)*10
    penalizacion += abs(len(itest)/len(df)-FRACCION_TEST)
    if mejor is None or penalizacion < mejor[0]: mejor=(penalizacion,itrain,itest)
    if penalizacion < 0.025: break
_, train_i, test_i = mejor
if set(df.iloc[train_i].clase) != set(etiq):
    raise RuntimeError('No se encontraron todas las clases en TRAIN. Revise etiquetas y variabilidad.')
if set(df.iloc[test_i].clase) != set(etiq):
    print('ATENCIÓN: faltan clases en TEST; las métricas por clase pueden ser poco informativas.')
assert not (set(df.iloc[train_i].texto_norm) & set(df.iloc[test_i].texto_norm))
tr, te = df.iloc[train_i].copy(), df.iloc[test_i].copy()
print(f'TRAIN={len(tr)} (textos únicos={tr.texto_norm.nunique()}); TEST={len(te)} '
      f'(textos únicos={te.texto_norm.nunique()})')





textos_tr = tr['texto_norm'].tolist(); textos_te = te['texto_norm'].tolist()
representaciones = {}

def guardar_rep(nombre, a, b):
    a = sparse.csr_matrix(a, dtype=np.float32)
    b = sparse.csr_matrix(b, dtype=np.float32)
    representaciones[nombre] = (a,b)
    print(f'  {nombre:18s} -> {a.shape[1]:6d} atributos')

def ajustar_vect(nombre, vector):
    a = vector.fit_transform(textos_tr)
    b = vector.transform(textos_te)
    guardar_rep(nombre,a,b)

print('\nConstruyendo representaciones...')
ajustar_vect('TFIDF',TfidfVectorizer(ngram_range=(1,2),min_df=2,max_features=20000,
                                     sublinear_tf=True,strip_accents='unicode'))
ajustar_vect('NGRAMAS',TfidfVectorizer(analyzer='char',ngram_range=(3,5),min_df=2,
                                       max_features=25000,sublinear_tf=True))
ajustar_vect('BOW',CountVectorizer(ngram_range=(1,2),min_df=2,max_features=20000,
                                    binary=False,strip_accents='unicode'))


if USAR_EMBEDDINGS_TRANSFORMER:
    from sentence_transformers import SentenceTransformer
    try:
        st = SentenceTransformer(MODELO_EMBEDDINGS)
        unicos = pd.Index(pd.concat([tr.texto_norm,te.texto_norm]).unique())
        vec = st.encode(unicos.tolist(),batch_size=32,show_progress_bar=True,
                        normalize_embeddings=True,convert_to_numpy=True)
        dic_emb = {t:v for t,v in zip(unicos,vec)}
        emb_tr = np.vstack([dic_emb[t] for t in tr.texto_norm])
        emb_te = np.vstack([dic_emb[t] for t in te.texto_norm])
        
        escala = MinMaxScaler().fit(emb_tr)
        guardar_rep('EMBEDDINGS',escala.transform(emb_tr),
                    np.clip(escala.transform(emb_te),0,None))
    except Exception as e:
        raise RuntimeError('No se pudieron calcular embeddings reales. Verifique conexión/modelo. '+str(e))
else:
    raise ValueError('Las cuatro representaciones son obligatorias; active USAR_EMBEDDINGS_TRANSFORMER.')



def combinar(bloques):
    a = sparse.hstack([representaciones[n][0]*peso for n,peso in bloques],format='csr')
    b = sparse.hstack([representaciones[n][1]*peso for n,peso in bloques],format='csr')
    return a,b

variantes = {
  'HMT-V1':[('TFIDF',1.0),('NGRAMAS',1.0)],
  'HMT-V2':[('TFIDF',1.0),('BOW',0.25),('NGRAMAS',0.75)],
  'HMT-V3':[('TFIDF',1.0),('NGRAMAS',0.75),('EMBEDDINGS',2.0)],
  'HMT-V4':[('TFIDF',1.0),('NGRAMAS',0.75),('BOW',0.25),('EMBEDDINGS',2.0)]
}
for nombre,bloques in variantes.items():
    guardar_rep(nombre,*combinar(bloques))





GEO_COLS = ['region','estado','regionalismo']
try: enc = OneHotEncoder(handle_unknown='ignore',sparse_output=True,dtype=np.float32)
except TypeError: enc = OneHotEncoder(handle_unknown='ignore',sparse=True,dtype=np.float32)
geo_tr = enc.fit_transform(tr[GEO_COLS]); geo_te = enc.transform(te[GEO_COLS])
print('Vector geolingüístico:',geo_tr.shape[1],'dimensiones')
print('IMPORTANTE: el vector GEO utiliza metadatos anotados y no infiere automáticamente el estado o región.')

PESO_GEO = 1.0
for nombre in list(representaciones):
    a,b = representaciones[nombre]
    guardar_rep(nombre+'+GEO',
                sparse.hstack([a,geo_tr*PESO_GEO],format='csr'),
                sparse.hstack([b,geo_te*PESO_GEO],format='csr'))


def construir_modelo(nombre, n_clases):
    if nombre=='RL': return LogisticRegression(max_iter=1200,C=1.0,class_weight='balanced',random_state=SEMILLA)
    if nombre=='SVM': return LinearSVC(C=1.0,class_weight='balanced',random_state=SEMILLA,max_iter=5000)
    if nombre=='NB': return MultinomialNB(alpha=1.0)
    if nombre=='KNN': return KNeighborsClassifier(n_neighbors=3,weights='distance',metric='cosine')
    if nombre=='MLP': return MLPClassifier(hidden_layer_sizes=(64,),max_iter=160,
                                          early_stopping=True,random_state=SEMILLA)
    if nombre=='RF': return RandomForestClassifier(n_estimators=160,min_samples_leaf=1,
                                class_weight='balanced_subsample',random_state=SEMILLA,n_jobs=-1)
    if nombre=='ARBOL': return DecisionTreeClassifier(max_depth=18,min_samples_leaf=2,
                                 class_weight='balanced',random_state=SEMILLA)
    if nombre=='XGB':
        from xgboost import XGBClassifier
        return XGBClassifier(n_estimators=140,max_depth=4,learning_rate=.07,
                             subsample=.9,colsample_bytree=.9,tree_method='hist',
                             objective='binary:logistic' if n_clases==2 else 'multi:softprob',
                             eval_metric='logloss' if n_clases==2 else 'mlogloss',
                             random_state=SEMILLA,n_jobs=2)
    raise KeyError(nombre)

metricas=[]; predicciones={}; modelos={}
tareas = {'BINARIA':('violencia_binaria',[0,1]),
          'MULTICLASE':('clase',sorted(df.clase.unique()))}
for tarea,(col,clases) in tareas.items():
    print('\n'+'='*72+'\nTAREA:',tarea,'\n'+'='*72)
    if tarea == 'MULTICLASE':
        
        dic_clase={c:i for i,c in enumerate(clases)}
        ytr=np.array([dic_clase[c] for c in tr[col]])
        yte=np.array([dic_clase[c] for c in te[col]])
    else:
        ytr=tr[col].to_numpy(dtype=int);yte=te[col].to_numpy(dtype=int)
    for rep,(a,b) in representaciones.items():
        for alg in CLASIFICADORES:
            t0=time.time()
            try:
                modelo=construir_modelo(alg,len(clases))
                kw = {}
                if alg=='XGB': kw['sample_weight']=compute_sample_weight('balanced',ytr)
                modelo.fit(a,ytr,**kw)
                pred=modelo.predict(b)
                metricas.append(dict(tarea=tarea,representacion=rep,clasificador=alg,
                    accuracy=accuracy_score(yte,pred),balanced_accuracy=balanced_accuracy_score(yte,pred),
                    precision_macro=precision_score(yte,pred,average='macro',zero_division=0),
                    recall_macro=recall_score(yte,pred,average='macro',zero_division=0),
                    f1_macro=f1_score(yte,pred,average='macro',zero_division=0),
                    f1_weighted=f1_score(yte,pred,average='weighted',zero_division=0),
                    tiempo_s=round(time.time()-t0,2),error=''))
                predicciones[(tarea,rep,alg)] = pred
                modelos[(tarea,rep,alg)] = modelo
            except Exception as e:
                print('  ERROR',tarea,rep,alg,str(e)[:180])
                metricas.append(dict(tarea=tarea,representacion=rep,clasificador=alg,
                    accuracy=np.nan,balanced_accuracy=np.nan,precision_macro=np.nan,
                    recall_macro=np.nan,f1_macro=np.nan,f1_weighted=np.nan,
                    tiempo_s=round(time.time()-t0,2),error=str(e)[:300]))
        mejores_t = [r for r in metricas if r['tarea']==tarea and r['representacion']==rep
                       and pd.notna(r['f1_macro'])]
        if mejores_t:
            top=max(mejores_t,key=lambda x:x['f1_macro'])
            print(f' {rep:20s} | {top["clasificador"]:5s} | macro-F1={top["f1_macro"]:.4f}')

resultados=pd.DataFrame(metricas)
resultados.to_csv(SALIDA/'metricas_todos_los_modelos.csv',index=False)
resultados_validos=resultados.dropna(subset=['f1_macro'])
if resultados_validos.empty: raise RuntimeError('Ningún modelo pudo entrenarse; revise errores.')
mejores=resultados_validos.sort_values('f1_macro',ascending=False).groupby('tarea',sort=False).head(10)
mejores.to_csv(SALIDA/'top10_por_tarea.csv',index=False)
print('\nTOP 10 por tarea:\n',mejores[['tarea','representacion','clasificador','f1_macro','accuracy']].to_string(index=False))




resultados_pred=te.copy().reset_index(drop=True)
mejor_clave={}
for tarea in tareas:
    candidatas=resultados_validos[resultados_validos.tarea==tarea].sort_values('f1_macro',ascending=False)
    ganador=candidatas.iloc[0]
    key=(tarea,ganador.representacion,ganador.clasificador)
    mejor_clave[tarea]=key
    p=predicciones[key]
    if tarea=='BINARIA':
        resultados_pred['pred_binaria']=p.astype(int)
        resultados_pred['real_binaria']=resultados_pred['violencia_binaria'].astype(int)
    else:
        clases=tareas[tarea][1]
        resultados_pred['pred_multiclase']=[clases[int(i)] for i in p]
        resultados_pred['real_multiclase']=resultados_pred['clase']
    yreal=te[tareas[tarea][0]].to_numpy()
    if tarea=='MULTICLASE': yreal=np.array([clases.index(v) for v in yreal])
    print('\nMEJOR MODELO EXPLORATORIO',tarea,key[1:], '\n',
          classification_report(yreal,p,target_names=list(map(str,tareas[tarea][1])),zero_division=0))
    matriz=confusion_matrix(yreal,p,labels=np.arange(len(tareas[tarea][1])))
    pd.DataFrame(matriz,index=tareas[tarea][1],columns=tareas[tarea][1]).to_csv(SALIDA/f'confusion_{tarea.lower()}.csv')
    fig,ax=plt.subplots(figsize=(8,6))
    sns.heatmap(matriz,annot=True,fmt='d',cmap='Blues',xticklabels=tareas[tarea][1],
                yticklabels=tareas[tarea][1],ax=ax)
    ax.set(title=f'Matriz de confusión · {tarea} (TEST agrupado)',xlabel='Predicción',ylabel='Real')
    plt.xticks(rotation=35,ha='right');plt.tight_layout()
    fig.savefig(SALIDA/f'confusion_{tarea.lower()}.png',dpi=220,bbox_inches='tight')
    if MOSTRAR_FIGURAS: plt.show()
    plt.close(fig)
resultados_pred.to_csv(SALIDA/'predicciones_solo_test.csv',index=False)


segmentos=[]
for nivel in ['region','estado']:
    for nombre,sub in resultados_pred.groupby(nivel):
        for tarea,real,pred in [('BINARIA','real_binaria','pred_binaria'),
                               ('MULTICLASE','real_multiclase','pred_multiclase')]:
            fila=dict(nivel=nivel,lugar=nombre,tarea=tarea,n=len(sub),
                      violencia_observada_pct=100*sub.real_binaria.mean(),
                      violencia_predicha_pct=100*sub.pred_binaria.mean(),
                      accuracy=accuracy_score(sub[real],sub[pred]),
                      f1_macro=f1_score(sub[real],sub[pred],average='macro',zero_division=0))
            
            fila['n_insuficiente']=len(sub)<UMBRAL_MIN_MUESTRAS
            segmentos.append(fila)
metricas_segmentos=pd.DataFrame(segmentos)
metricas_segmentos.to_csv(SALIDA/'metricas_region_estado.csv',index=False)
print('\nResultados por región (TEST):')
print(metricas_segmentos.query("nivel == 'region' and tarea == 'BINARIA'").to_string(index=False))


def agregar_prevalencias(data, nivel):
    filas=[]
    for lugar,sub in data.groupby(nivel):
        fila={nivel:lugar,'n':len(sub),
              'observada_binaria_pct':round(100*sub.real_binaria.mean(),2),
              'predicha_binaria_pct':round(100*sub.pred_binaria.mean(),2)}
        for clase in tareas['MULTICLASE'][1]:
            clave=re.sub(r'[^a-z0-9]+','_',unicodedata.normalize('NFKD',clase).encode('ascii','ignore').decode().lower()).strip('_')
            fila[f'obs_{clave}_pct']=round(100*(sub.real_multiclase==clase).mean(),2)
            fila[f'pred_{clave}_pct']=round(100*(sub.pred_multiclase==clase).mean(),2)
        filas.append(fila)
    return pd.DataFrame(filas)
reg_tab=agregar_prevalencias(resultados_pred,'region')
est_tab=agregar_prevalencias(resultados_pred,'estado')
reg_tab.to_csv(SALIDA/'prevalencia_regiones_test.csv',index=False)
est_tab.to_csv(SALIDA/'prevalencia_estados_test.csv',index=False)





def slug(s):
    s=unicodedata.normalize('NFKD',str(s)).encode('ascii','ignore').decode().lower()
    return re.sub(r'[^a-z0-9]','',s)

if PRODUCIR_MAPAS:
    import geopandas as gpd, requests
    api='https://www.geoboundaries.org/api/current/gbOpen/MEX/ADM1/'
    print('\nDescargando geometrías administrativas (geoBoundaries, ADM1)...')
    try:
        resp=requests.get(api,timeout=40);resp.raise_for_status();meta=resp.json()
        geo_url=meta.get('simplifiedGeometryGeoJSON') or meta.get('gjDownloadURL')
        if not geo_url: raise KeyError('La API no informó URL GeoJSON')
        geo_path=SALIDA/'mexico_adm1_fuente.geojson'
        if not geo_path.exists():
            respuesta=requests.get(geo_url,timeout=120);respuesta.raise_for_status()
            geo_path.write_bytes(respuesta.content)
        mapa=gpd.read_file(geo_path)
        columna_nombre=next((c for c in ['shapeName','shapeName_es','name','NAME_1','NOMGEO'] if c in mapa.columns),None)
        if columna_nombre is None: raise ValueError('Nombres de entidades desconocidos en GeoJSON: '+str(list(mapa.columns)))
        mapa['estado_geo']=mapa[columna_nombre].astype(str)
        mapa['clave_estado']=mapa.estado_geo.map(slug)
        
        alias={'cdmx':'ciudaddemexico','distritofederal':'ciudaddemexico',
               'mexicocity':'ciudaddemexico','nuevoleon':'nuevoleon',
               'veracruzdeignaciodelallave':'veracruz','veracruzllave':'veracruz',
               'michoacandeocampo':'michoacan','coahuiladezaragoza':'coahuila',
               'estadodemexico':'mexico','mexicostate':'mexico',
               'sanluispotosi':'sanluispotosi'}
        mapa['clave_estado']=mapa.clave_estado.replace(alias)
        est_tab['clave_estado']=est_tab.estado.map(slug).replace(alias)
        mapa_estados=mapa.merge(est_tab.drop(columns='estado'),on='clave_estado',how='left')
        asignaciones=(df.groupby('estado')['region'].agg(lambda s:s.mode().iat[0]).reset_index())
        asignaciones['clave_estado']=asignaciones.estado.map(slug).replace(alias)
        mapa_estados=mapa_estados.merge(asignaciones[['clave_estado','region']],on='clave_estado',how='left')
        no_encontrados=set(est_tab.clave_estado)-set(mapa_estados.clave_estado)
        if no_encontrados: print('ATENCIÓN: estados sin geometría emparejada:',sorted(no_encontrados))
        
        mapa_regiones=mapa_estados[mapa_estados.region.notna()][['region','geometry']].dissolve(by='region').reset_index()
        mapa_regiones=mapa_regiones.merge(reg_tab,on='region',how='left')
        mapa_estados.to_file(SALIDA/'mapa_estados.gpkg',driver='GPKG')
        mapa_regiones.to_file(SALIDA/'mapa_regiones_parciales.gpkg',driver='GPKG')

        def dibujar_mapa(gdf,col,archivo,titulo,anotar='estado_geo',vmin=0,vmax=None):
            if col not in gdf.columns: return
            fig,ax=plt.subplots(figsize=(13,9),facecolor='#fbfcfe')
            gdf.plot(ax=ax,color='#e5e7eb',edgecolor='#ffffff',linewidth=.55)
            gdf.plot(column=col,ax=ax,cmap='YlOrRd',vmin=vmin,vmax=vmax,
                     legend=True,edgecolor='#ffffff',linewidth=.8,
                     missing_kwds={'color':'#e5e7eb','label':'Sin registros en TEST'},
                     legend_kwds={'label':'Porcentaje de publicaciones (%)','shrink':.65})
            for _,r in gdf[gdf[col].notna()].iterrows():
                geom=r.geometry
                if geom is None or geom.is_empty: continue
                pos=geom.representative_point()
                etiqueta=str(r.get(anotar,''))
                if anotar=='estado_geo':
                    
                    if pd.isna(r.get('region')): continue
                    etiqueta=etiqueta.replace('Ciudad de México','CDMX').replace('Veracruz de Ignacio de la Llave','Veracruz')
                ax.annotate(etiqueta,(pos.x,pos.y),fontsize=7 if anotar=='estado_geo' else 10,
                            ha='center',va='center',color='#1f2937',weight='bold')
            ax.set(title=titulo);ax.set_axis_off();fig.tight_layout()
            fig.savefig(SALIDA/archivo,dpi=250,bbox_inches='tight',facecolor=fig.get_facecolor())
            if MOSTRAR_FIGURAS: plt.show()
            plt.close(fig)
        for marco,nombre,col_etiqueta in [(mapa_regiones,'regiones','region'),
                                           (mapa_estados,'estados','estado_geo')]:
            for col,etiqueta in [('observada_binaria_pct','Violencia observada'),
                                ('predicha_binaria_pct','Violencia predicha')]:
                dibujar_mapa(marco,col,f'mapa_{nombre}_{col}.png',
                             f'{etiqueta} por {nombre} · conjunto TEST (agrupado)',col_etiqueta,vmax=100)
            
            for col in [c for c in marco.columns if c.startswith('pred_') and c.endswith('_pct')]:
                dibujar_mapa(marco,col,f'mapa_{nombre}_{col}.png',
                             f'{col.replace("pred_", "").replace("_pct", "").replace("_", " ").title()} · porcentaje predicho por {nombre}',
                             col_etiqueta,vmax=100)
        print('Mapas de estados y regiones guardados como PNG + GeoPackage.')
    except Exception as e:
        print('ADVERTENCIA: no fue posible crear mapas geográficos:',type(e).__name__,str(e))
        print('Las tablas de clasificación y prevalencia sí quedaron generadas.')


for tarea in tareas:
    tab=resultados_validos[resultados_validos.tarea==tarea]
    fig,ax=plt.subplots(figsize=(16,7))
    piv=tab.pivot_table(index='representacion',columns='clasificador',values='f1_macro')
    sns.heatmap(piv,annot=True,fmt='.2f',cmap='viridis',vmin=0,vmax=1,ax=ax)
    ax.set(title=f'Comparativa F1 macro · {tarea}',xlabel='Clasificador',ylabel='Representación')
    fig.tight_layout();fig.savefig(SALIDA/f'comparativa_f1_{tarea.lower()}.png',dpi=220,bbox_inches='tight')
    if MOSTRAR_FIGURAS: plt.show()
    plt.close(fig)

metadatos={'archivo':str(ruta),'n':len(df),'train':len(tr),'test':len(te),
 'textos_unicos':int(df.texto_norm.nunique()),'semilla':SEMILLA,
 'split':'GroupShuffleSplit por texto normalizado',
 'representaciones':list(representaciones),
 'geo_columnas':GEO_COLS,'peso_geo':PESO_GEO,
 'tareas':{k:v[1] for k,v in tareas.items()},
 'mejores_exploratorios':{k:list(v) for k,v in mejor_clave.items()},
 'advertencias':['Ranking elegido por TEST: exploratorio, no confirmatorio',
 'Los registros repetidos pueden inflar la precisión aparente aun con texto agrupado',
 'No inferir prevalencia poblacional a partir de una muestra no probabilística',
 'Los metadatos geolingüísticos deben existir también para textos futuros',
 'Regiones cartográficas parciales: unión de estados con datos; no representan necesariamente regiones oficiales']}
(SALIDA/'configuracion_y_limitaciones.json').write_text(json.dumps(metadatos,ensure_ascii=False,indent=2),encoding='utf8')
archivo_zip=SALIDA.parent/'resultados_geolinguisticos.zip'
with zipfile.ZipFile(archivo_zip,'w',zipfile.ZIP_DEFLATED) as z:
    for p in SALIDA.iterdir():
        if p.is_file(): z.write(p,arcname=p.name)
print('\nLISTO. Carpeta:',SALIDA,'\nZIP:',archivo_zip)
print('El TEST sirvió para la comparación exploratoria: NO reporte el ganador como evaluación confirmatoria.')
if 'google.colab' in sys.modules:
    from google.colab import files
    print('Puede descargar el ZIP desde el panel Archivos de Colab o ejecutar files.download(str(archivo_zip)).')


### EXP-061 · V1, V2, V3, V4

**Archivo de origen:** `Untitled7.ipynb, celda 5`. **Referencias duplicadas:** 0. **Sintaxis de código:** Válida.

CLASIFICACION DOCTORAL BINARIA + MULTICLASE / GEO / HMVT
UNA SOLA CELDA COLAB. CARGA CSV SIEMPRE EN GOOGLE COLAB.
import sys, os, re, json, time, warnings, subprocess, importlib.util, zipfile, unicodedata
from pathlib import Path
from io im

**Ejecución:** celda independiente o variante experimental; revisar archivos de entrada y dependencias antes de ejecutarla. Los resultados originales no se insertan en esta celda.

In [ ]:
import sys, os, re, json, time, warnings, subprocess, importlib.util, zipfile, unicodedata
from pathlib import Path
from io import BytesIO
warnings.filterwarnings('ignore', category=FutureWarning)

SEMILLA=42
TEST_SIZE=.20
ARCHIVO_LOCAL='data_hmvt_con_estado.csv'
SALIDA=Path('/content/resultados_hmvt_80_20' if Path('/content').exists() else 'resultados_hmvt_80_20')
SALIDA.mkdir(parents=True,exist_ok=True)
USAR_GEO=True
HACER_MAPAS=True
MOSTRAR_GRAFICAS=True
EMBEDDING_MODELO='sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'

N_ITER_HMVT=2
N_SPLITS_CV=3
MODELOS=['RL','SVM','RF','MLP','NB','KNN','XGB','ARBOL']
BASES=['TFIDF','NGRAMAS','BOW','EMBEDDINGS']
HMVT=['HMVT-V1','HMVT-V2','HMVT-V3','HMVT-V4']


def asegurar(nombre,pkg=None):
    if importlib.util.find_spec(nombre) is None:
        subprocess.check_call([sys.executable,'-m','pip','-q','install',pkg or nombre])
for modulo,pkg in [('numpy',None),('pandas',None),('scipy',None),('sklearn','scikit-learn'),
                   ('matplotlib',None),('seaborn',None),('sentence_transformers','sentence-transformers'),
                   ('xgboost',None),('geopandas',None),('requests',None)]:
    if modulo=='geopandas' and not HACER_MAPAS: continue
    asegurar(modulo,pkg)
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from scipy import sparse
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.metrics import accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.base import clone
sns.set_theme(style='whitegrid')
np.random.seed(SEMILLA)


if 'google.colab' in sys.modules:
    from google.colab import files
    print('SUBE EL DATASET CSV. Debe incluir tweet, clase, violencia_binaria, region, estado y regionalismo.')
    subidos=files.upload()
    if len(subidos)!=1: raise ValueError('Selecciona exactamente un archivo CSV')
    nombre_archivo,contenido=next(iter(subidos.items()))
    if not nombre_archivo.lower().endswith('.csv'): raise ValueError('Se requiere CSV')
    df=pd.read_csv(BytesIO(contenido),encoding='utf-8-sig')
else:
    nombre_archivo=ARCHIVO_LOCAL
    df=pd.read_csv(nombre_archivo,encoding='utf-8-sig')
requeridas={'tweet','clase','violencia_binaria','region','estado','regionalismo'}
if faltantes:=requeridas-set(df.columns): raise ValueError('Faltan columnas: '+str(faltantes))
df=df.dropna(subset=['tweet','clase','violencia_binaria']).copy()
for c in ['tweet','clase','region','estado','regionalismo']:
    df[c]=df[c].fillna('SIN_DATO').astype(str).str.strip()
df['violencia_binaria']=pd.to_numeric(df.violencia_binaria,errors='raise').astype(int)
if not set(df.violencia_binaria.unique()).issubset({0,1}):raise ValueError('violencia_binaria debe ser 0/1')

def normalizar(s):
    s=str(s).lower().strip()
    s=re.sub(r'https?://\S+|www\.\S+',' URL ',s)
    s=re.sub(r'@\w+',' USUARIO ',s)
    return re.sub(r'\s+',' ',s)
df['texto_norm']=df.tweet.map(normalizar)
df=df[df.texto_norm.str.len()>0].reset_index(drop=True)
if (df.groupby('texto_norm').clase.nunique()>1).any():
    raise ValueError('Existen textos iguales con etiquetas multiclase contradictorias; revisar antes de evaluar')
if (df.groupby('texto_norm').violencia_binaria.nunique()>1).any():
    raise ValueError('Existen textos iguales con etiquetas binarias contradictorias')
if df.groupby('clase').violencia_binaria.nunique().max()>1:
    raise ValueError('Inconsistencia entre clase y etiqueta binaria')



print('\nDIAGNOSTICO DEL CONJUNTO ORIGINAL')
print('Filas:',len(df),'| Textos unicos:',df.texto_norm.nunique(),'| Filas duplicadas:',df.duplicated().sum())
for c in ['estado','region','regionalismo']:
    n=(df.groupby('texto_norm')[c].nunique()>1).sum()
    print('Textos asociados con multiples',c,':',n)


corpus=df.drop_duplicates('texto_norm').copy()
geo_cols=['region','estado','regionalismo']
for c in geo_cols:
    nvar=df.groupby('texto_norm')[c].nunique()
    corpus.loc[corpus.texto_norm.map(nvar)>1,c]='SIN_DATO'
corpus=corpus.reset_index(drop=True)
idx_tr,idx_te=train_test_split(np.arange(len(corpus)),test_size=TEST_SIZE,
                                random_state=SEMILLA,stratify=corpus['clase'])
tr=corpus.iloc[idx_tr].reset_index(drop=True)
te=corpus.iloc[idx_te].reset_index(drop=True)
assert not set(tr.texto_norm).intersection(set(te.texto_norm))
assert len(tr)+len(te)==len(corpus)
print('\nDIVISION 80/20 SOBRE TEXTOS UNICOS:')
print('TRAIN',len(tr),'(',round(100*len(tr)/len(corpus),1),'%)','TEST',len(te),
      '(',round(100*len(te)/len(corpus),1),'%)')
print('TRAIN por clase:',tr.clase.value_counts().to_dict())
print('TEST por clase:',te.clase.value_counts().to_dict())
print('IMPORTANTISIMO: 80/20 corresponde a textos unicos, no a filas repetidas.')
print('REGIONALISMO y GEO con anotaciones contradictorias quedan como SIN_DATO para modelado.')
tr.to_csv(SALIDA/'train_textos_unicos.csv',index=False)
te.to_csv(SALIDA/'test_textos_unicos.csv',index=False)
textos_tr=tr.texto_norm.tolist(); textos_te=te.texto_norm.tolist()


rep={}
def registrar(k,a,b):
    rep[k]=(sparse.csr_matrix(a,dtype=np.float32),sparse.csr_matrix(b,dtype=np.float32))
    print(f'  {k:18s} | {rep[k][0].shape[1]} atributos')
def vectorizar(k,obj): registrar(k,obj.fit_transform(textos_tr),obj.transform(textos_te))
print('\nCREANDO REPRESENTACIONES (fit SOLO TRAIN):')
vectorizar('TFIDF',TfidfVectorizer(max_features=12000,ngram_range=(1,2),strip_accents='unicode'))
vectorizar('NGRAMAS',TfidfVectorizer(analyzer='char',ngram_range=(3,5),max_features=14000))
vectorizar('BOW',CountVectorizer(max_features=12000,strip_accents='unicode'))
from sentence_transformers import SentenceTransformer
st=SentenceTransformer(EMBEDDING_MODELO)
emb_tr=st.encode(textos_tr,batch_size=32,show_progress_bar=True,normalize_embeddings=True)
emb_te=st.encode(textos_te,batch_size=32,show_progress_bar=True,normalize_embeddings=True)

esc=MinMaxScaler().fit(emb_tr)
registrar('EMBEDDINGS',esc.transform(emb_tr),np.maximum(0,esc.transform(emb_te)))





bloques={
 'HMVT-V1':[('TFIDF',1.),('NGRAMAS',1.)],
 'HMVT-V2':[('TFIDF',1.),('BOW',.3),('NGRAMAS',.7)],
 'HMVT-V3':[('TFIDF',1.),('NGRAMAS',.7),('EMBEDDINGS',2.)],
 'HMVT-V4':[('TFIDF',1.),('NGRAMAS',.7),('BOW',.3),('EMBEDDINGS',2.)]
}
for nombre,partes in bloques.items():
    registrar(nombre,*[sparse.hstack([rep[k][i]*peso for k,peso in partes],format='csr') for i in range(2)])



if USAR_GEO:
    try: ge=OneHotEncoder(handle_unknown='ignore',sparse_output=True,dtype=np.float32)
    except TypeError: ge=OneHotEncoder(handle_unknown='ignore',sparse=True,dtype=np.float32)
    gt=ge.fit_transform(tr[geo_cols]); gv=ge.transform(te[geo_cols]);
    for nombre in BASES+HMVT:
        a,b=rep[nombre]
        registrar(nombre+'+GEO',sparse.hstack([a,gt],format='csr'),
                  sparse.hstack([b,gv],format='csr'))



def crear_modelo(nombre,nclases):
    if nombre=='RL':return LogisticRegression(max_iter=1200,random_state=SEMILLA)
    if nombre=='SVM':return LinearSVC(max_iter=3000,random_state=SEMILLA)
    if nombre=='RF':return RandomForestClassifier(n_estimators=150,random_state=SEMILLA,n_jobs=-1)
    if nombre=='MLP':return MLPClassifier(hidden_layer_sizes=(64,),early_stopping=True,max_iter=200,random_state=SEMILLA)
    if nombre=='NB':return MultinomialNB()
    if nombre=='KNN':return KNeighborsClassifier(n_neighbors=3)
    if nombre=='XGB':
        from xgboost import XGBClassifier
        return XGBClassifier(n_estimators=130,max_depth=4,learning_rate=.1,random_state=SEMILLA,
                             tree_method='hist',n_jobs=2,eval_metric='logloss' if nclases==2 else 'mlogloss')
    if nombre=='ARBOL':return DecisionTreeClassifier(random_state=SEMILLA)
    raise ValueError(nombre)

rejillas={
 'RL':{'C':[.1,1.]},'SVM':{'C':[.1,1.]},
 'RF':{'max_depth':[6,None]},'MLP':{'alpha':[.0001,.01]},
 'NB':{'alpha':[.5,1.]},'KNN':{'n_neighbors':[3,5]},
 'XGB':{'max_depth':[2,4]},'ARBOL':{'max_depth':[4,10]}
}


metricas=[]; por_clase=[]; predicciones={}; detalles_cv=[]
tareas={'BINARIA':('violencia_binaria',[0,1]),'MULTICLASE':('clase',sorted(corpus.clase.unique()))}
for tarea,(col,etiquetas) in tareas.items():
    if tarea=='MULTICLASE':
        a={k:i for i,k in enumerate(etiquetas)}
        ytr=tr[col].map(a).to_numpy();yte=te[col].map(a).to_numpy()
    else:
        ytr=tr[col].to_numpy(dtype=int);yte=te[col].to_numpy(dtype=int)
    etiquetas_id=np.arange(len(etiquetas))
    for nombre,(xtr,xte) in rep.items():
        optimizar=nombre.startswith('HMVT-')
        for alg in MODELOS:
            inicio=time.time();key=(tarea,nombre,alg)
            try:
                modelo=crear_modelo(alg,len(etiquetas))
                if optimizar:
                    
                    
                    cv=StratifiedKFold(n_splits=N_SPLITS_CV,shuffle=True,random_state=SEMILLA)
                    busqueda=GridSearchCV(modelo,rejillas[alg],scoring='f1_macro',cv=cv,
                                           n_jobs=1,refit=True,error_score='raise')
                    busqueda.fit(xtr,ytr)
                    modelo=busqueda.best_estimator_
                    detalles_cv.append({'tarea':tarea,'representacion':nombre,'clasificador':alg,
                                        'f1_cv_train':busqueda.best_score_,
                                        'mejores_parametros':json.dumps(busqueda.best_params_)})
                else:
                    modelo.fit(xtr,ytr)
                p=modelo.predict(xte)
                predicciones[key]=p
                fila={'tarea':tarea,'representacion':nombre,'clasificador':alg,'n_train':len(ytr),
                    'n_test':len(yte),'optimizado_hmvt':optimizar,
                    'accuracy':accuracy_score(yte,p),
                    'balanced_accuracy':balanced_accuracy_score(yte,p),
                    'precision_macro':precision_score(yte,p,average='macro',zero_division=0),
                    'recall_macro':recall_score(yte,p,average='macro',zero_division=0),
                    'f1_macro':f1_score(yte,p,average='macro',zero_division=0),
                    'precision_weighted':precision_score(yte,p,average='weighted',zero_division=0),
                    'recall_weighted':recall_score(yte,p,average='weighted',zero_division=0),
                    'f1_weighted':f1_score(yte,p,average='weighted',zero_division=0),
                    'tiempo_s':round(time.time()-inicio,2),'error':''}
                metricas.append(fila)
                informe=classification_report(yte,p,labels=etiquetas_id,output_dict=True,zero_division=0)
                for j,et in enumerate(etiquetas):
                    item=informe[str(j)]
                    por_clase.append({'tarea':tarea,'representacion':nombre,'clasificador':alg,'clase':et,
                                      'precision':item['precision'],'recall':item['recall'],
                                      'f1_score':item['f1-score'],'soporte':item['support']})
                cm=confusion_matrix(yte,p,labels=etiquetas_id)
                base=f'{tarea}_{nombre}_{alg}'.replace('+','_MAS_')
                path=SALIDA/'matrices_confusion';path.mkdir(exist_ok=True)
                pd.DataFrame(cm,index=[str(e) for e in etiquetas],columns=[str(e) for e in etiquetas]).to_csv(path/(base+'.csv'))
                fig,ax=plt.subplots(figsize=(6.5,5.3) if tarea=='BINARIA' else (8,6.8))
                sns.heatmap(cm,annot=True,fmt='d',cmap='Blues',xticklabels=etiquetas,
                            yticklabels=etiquetas,ax=ax,cbar=False)
                ax.set(title=f'{tarea} · {nombre} · {alg}\nTEST 20% (textos únicos)',xlabel='Predicha',ylabel='Real')
                plt.xticks(rotation=40,ha='right');plt.yticks(rotation=0)
                fig.tight_layout();fig.savefig(path/(base+'.png'),dpi=155,bbox_inches='tight')
                plt.close(fig)
            except Exception as ex:
                metricas.append({'tarea':tarea,'representacion':nombre,'clasificador':alg,
                                 'optimizado_hmvt':optimizar,'error':str(ex),'tiempo_s':round(time.time()-inicio,2)})
                print('ERROR',tarea,nombre,alg,str(ex)[:160])
        ok=[v for v in metricas if v['tarea']==tarea and v['representacion']==nombre and 'f1_macro' in v]
        if ok:
            k=max(ok,key=lambda z:z['f1_macro'])
            print(f'{tarea:10s} {nombre:20s} F1 macro mas alto en TEST: {k["clasificador"]}={k["f1_macro"]:.3f} (descriptivo, no seleccion final)')

res=pd.DataFrame(metricas)
res.to_csv(SALIDA/'metricas_completas_80_20.csv',index=False)
pd.DataFrame(por_clase).to_csv(SALIDA/'precision_recall_f1_por_clase.csv',index=False)
pd.DataFrame(detalles_cv).to_csv(SALIDA/'optimizacion_SOLO_HMVT_CV_train.csv',index=False)
validos=res.dropna(subset=['f1_macro'])
print('\nRESULTADOS TEST - todas las metricas:')
print(validos[['tarea','representacion','clasificador','accuracy','balanced_accuracy',
              'precision_macro','recall_macro','f1_macro','precision_weighted','recall_weighted','f1_weighted']].to_string(index=False))
if validos.empty:raise RuntimeError('Ningun modelo se entreno. Ver errores')




def pred_fija(tarea):
    preferencias=[(tarea,'HMVT-V4+GEO','RL'),(tarea,'HMVT-V4','RL'),(tarea,'TFIDF','RL')]
    return next((k for k in preferencias if k in predicciones),None)
for tarea in tareas:
    key=pred_fija(tarea)
    if key:
        print('\nModelo de mapa PREFIJADO:',key,'\n',classification_report(
            te[tareas[tarea][0]].map({x:i for i,x in enumerate(tareas[tarea][1])}).to_numpy() if tarea=='MULTICLASE'
            else te.violencia_binaria.to_numpy(),predicciones[key],
            labels=np.arange(len(tareas[tarea][1])),target_names=[str(k) for k in tareas[tarea][1]],zero_division=0))
        p=SALIDA/'matrices_confusion'/('_'.join(key).replace('+','_MAS_')+'.png')
        if MOSTRAR_GRAFICAS and p.exists():
            from IPython.display import display,Image
            display(Image(filename=str(p)))





geo_test=df[df.texto_norm.isin(set(te.texto_norm))].drop_duplicates(['texto_norm','estado']).copy()
geo_test=geo_test.merge(te[['texto_norm']],on='texto_norm',how='inner')
for tarea,col in [('BINARIA','pred_binaria'),('MULTICLASE','pred_multiclase')]:
    key=pred_fija(tarea)
    if key:
        p=predicciones[key]
        if tarea=='MULTICLASE':p=np.array(tareas[tarea][1],dtype=object)[p]
        dic=dict(zip(te.texto_norm,p))
        geo_test[col]=geo_test.texto_norm.map(dic)
geo_test.to_csv(SALIDA/'predicciones_geograficas_SOLO_TEST.csv',index=False)

for nivel in ['region','estado']:
    filas=[]
    for lugar,part in geo_test.groupby(nivel):
        fila={nivel:lugar,'n_textos':part.texto_norm.nunique(),
              'violencia_real_pct':round(100*part.violencia_binaria.mean(),3)}
        if 'pred_binaria' in part: fila['violencia_predicha_pct']=round(100*part.pred_binaria.mean(),3)
        if 'pred_multiclase' in part:
            for clase in tareas['MULTICLASE'][1]:
                nombre=re.sub('[^a-z0-9]+','_',unicodedata.normalize('NFKD',str(clase)).encode('ascii','ignore').decode().lower()).strip('_')
                fila['pred_'+nombre+'_pct']=round(100*(part.pred_multiclase==clase).mean(),3)
        filas.append(fila)
    pd.DataFrame(filas).to_csv(SALIDA/f'resumen_{nivel}_SOLO_TEST.csv',index=False)

if HACER_MAPAS and len(geo_test)>0:
    import geopandas as gpd, requests
    def slug(s):
        return re.sub('[^a-z0-9]','',unicodedata.normalize('NFKD',str(s)).encode('ascii','ignore').decode().lower())
    alias={'distritofederal':'ciudaddemexico','mexicocity':'ciudaddemexico',
           'veracruzdeignaciodelallave':'veracruz','michoacandeocampo':'michoacan',
           'coahuiladezaragoza':'coahuila','estadodemexico':'mexico','mexicostate':'mexico'}
    try:
        meta=requests.get('https://www.geoboundaries.org/api/current/gbOpen/MEX/ADM1/',timeout=45)
        meta.raise_for_status();info=meta.json()
        url=info.get('simplifiedGeometryGeoJSON') or info.get('gjDownloadURL')
        if not url:raise RuntimeError('Sin geometria GeoJSON')
        path=SALIDA/'adm1_mx.geojson'
        if not path.exists():
            rr=requests.get(url,timeout=120);rr.raise_for_status();path.write_bytes(rr.content)
        mapa=gpd.read_file(path)
        namecol=next(c for c in ['shapeName','NAME_1','name','NOMGEO'] if c in mapa.columns)
        mapa['nombre_estado']=mapa[namecol].astype(str)
        mapa['clave']=mapa.nombre_estado.map(slug).replace(alias)
        estados=pd.read_csv(SALIDA/'resumen_estado_SOLO_TEST.csv')
        estados['clave']=estados.estado.map(slug).replace(alias)
        ge=mapa.merge(estados.drop(columns='estado'),on='clave',how='left')
        
        region_estado=df.groupby('estado').region.agg(lambda a:a.mode().iat[0]).reset_index()
        region_estado['clave']=region_estado.estado.map(slug).replace(alias)
        ge=ge.merge(region_estado[['clave','region']],on='clave',how='left')
        if not (ge.n_textos.notna().any()):raise RuntimeError('Ningun estado coincidió con las geometrías')
        regiones=pd.read_csv(SALIDA/'resumen_region_SOLO_TEST.csv')
        gr=ge[ge.region.notna()].dissolve(by='region').reset_index()[['region','geometry']]
        gr=gr.merge(regiones,on='region',how='left')
        ge.to_file(SALIDA/'estados_con_metricas.gpkg',driver='GPKG')
        gr.to_file(SALIDA/'regiones_con_metricas.gpkg',driver='GPKG')
        for nivel,gdf in [('estado',ge),('region',gr)]:
            campos=[c for c in gdf.columns if c.endswith('_pct')]
            for campo in campos:
                fig,ax=plt.subplots(figsize=(13,9))
                gdf.plot(ax=ax,color='#e5e7eb',edgecolor='white',linewidth=.55)
                gdf.plot(column=campo,ax=ax,cmap='YlOrRd',vmin=0,vmax=100,legend=True,
                         missing_kwds={'color':'#e5e7eb'},edgecolor='white',linewidth=.7,
                         legend_kwds={'label':'% de textos TEST','shrink':.65})
                for _,v in gdf[gdf[campo].notna()].iterrows():
                    point=v.geometry.representative_point()
                    label=str(v['nombre_estado'] if nivel=='estado' else v['region'])
                    ax.annotate(label,(point.x,point.y),fontsize=7 if nivel=='estado' else 10,
                                ha='center',va='center')
                ax.set_title(f'{campo.replace("_"," ").title()} por {nivel} · TEST 20%')
                ax.set_axis_off();fig.tight_layout()
                fig.savefig(SALIDA/f'mapa_{nivel}_{campo}.png',dpi=220,bbox_inches='tight')
                if MOSTRAR_GRAFICAS and campo=='violencia_predicha_pct':plt.show()
                plt.close(fig)
        print('Mapas GeoPandas generados')
    except Exception as ex:
        print('ADVERTENCIA cartografia: no se generaron mapas por',type(ex).__name__,str(ex))


for tarea in tareas:
    tabla=validos[validos.tarea==tarea]
    fig,ax=plt.subplots(figsize=(15,8))
    sns.heatmap(tabla.pivot_table(index='representacion',columns='clasificador',values='f1_macro'),
                annot=True,fmt='.2f',cmap='viridis',vmin=0,vmax=1,ax=ax)
    ax.set_title('F1 macro · '+tarea+' · TEST 20% (DESCRIPTIVO)')
    fig.tight_layout();fig.savefig(SALIDA/f'comparativa_f1_{tarea}.png',dpi=170,bbox_inches='tight')
    if MOSTRAR_GRAFICAS:plt.show()
    plt.close(fig)
perfectos=validos[(validos.accuracy>=.99999)&(validos.f1_macro>=.99999)]
print('\nModelos con desempeno perfecto:',len(perfectos))
if len(perfectos):
    print('ALERTA: 1.0 no demuestra generalizacion; verificar datos sinteticos, textos formulaicos,')
    print('sesgo de anotacion y resultados en otro dataset independiente. No forzamos scores artificialmente.')

info={'archivo':nombre_archivo,'filas_originales':len(df),'textos_unicos':len(corpus),
      'train_unicos':len(tr),'test_unicos':len(te),'test_size':TEST_SIZE,
      'semilla':SEMILLA,'optimizacion':'solo HMVT: 3-fold CV estratificada en train',
      'baselines':'parametros fijos, sin GridSearchCV','representaciones':list(rep),
      'modelo_geo_prefijado':'RL+HMVT-V4+GEO',
      'nota':'El CSV contiene textos duplicados y ubicaciones contradictorias. No es evidencia de prevalencia poblacional. HMVT V1-V4 son definiciones experimentales y deben compararse con implementaciones originales.'}
(SALIDA/'metodologia_y_advertencias.json').write_text(json.dumps(info,indent=2,ensure_ascii=False),encoding='utf8')
zip_path=SALIDA.parent/'resultados_hmvt_80_20.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for p in SALIDA.rglob('*'):
        if p.is_file():z.write(p,arcname=str(p.relative_to(SALIDA)))
print('\nRESULTADOS LISTOS:',SALIDA,'\nZIP:',zip_path)
if 'google.colab' in sys.modules:
    print('Para descargar ZIP: files.download(str(zip_path))')


### EXP-062 · V1, V2, V3, V4

**Archivo de origen:** `Untitled7.ipynb, celda 7`. **Referencias duplicadas:** 0. **Sintaxis de código:** Válida.

CLASIFICACION DOCTORAL BINARIA + MULTICLASE / GEO / HMVT
UNA SOLA CELDA COLAB. CARGA CSV SIEMPRE EN GOOGLE COLAB.
import sys, os, re, json, time, warnings, subprocess, importlib.util, zipfile, unicodedata
from pathlib import Path
from io im

**Ejecución:** celda independiente o variante experimental; revisar archivos de entrada y dependencias antes de ejecutarla. Los resultados originales no se insertan en esta celda.

In [ ]:
import sys, os, re, json, time, warnings, subprocess, importlib.util, zipfile, unicodedata
from pathlib import Path
from io import BytesIO
warnings.filterwarnings('ignore', category=FutureWarning)

SEMILLA=42
TEST_SIZE=.20
ARCHIVO_LOCAL='data_hmvt_con_estado.csv'
SALIDA=Path('/content/resultados_hmvt_80_20' if Path('/content').exists() else 'resultados_hmvt_80_20')
SALIDA.mkdir(parents=True,exist_ok=True)
USAR_GEO=True
HACER_MAPAS=True
MOSTRAR_GRAFICAS=True
EMBEDDING_MODELO='sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'

N_ITER_HMVT=2
N_SPLITS_CV=3
MODELOS=['RL','SVM','RF','MLP','NB','KNN','XGB','ARBOL']
BASES=['TFIDF','NGRAMAS','BOW','EMBEDDINGS']
HMVT=['HMVT-V1','HMVT-V2','HMVT-V3','HMVT-V4']


def asegurar(nombre,pkg=None):
    if importlib.util.find_spec(nombre) is None:
        subprocess.check_call([sys.executable,'-m','pip','-q','install',pkg or nombre])
for modulo,pkg in [('numpy',None),('pandas',None),('scipy',None),('sklearn','scikit-learn'),
                   ('matplotlib',None),('seaborn',None),('sentence_transformers','sentence-transformers'),
                   ('xgboost',None),('geopandas',None),('requests',None)]:
    if modulo=='geopandas' and not HACER_MAPAS: continue
    asegurar(modulo,pkg)
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from scipy import sparse
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.metrics import accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.base import clone
sns.set_theme(style='whitegrid')
np.random.seed(SEMILLA)


if 'google.colab' in sys.modules:
    from google.colab import files
    print('SUBE EL DATASET CSV. Debe incluir tweet, clase, violencia_binaria, region, estado y regionalismo.')
    subidos=files.upload()
    if len(subidos)!=1: raise ValueError('Selecciona exactamente un archivo CSV')
    nombre_archivo,contenido=next(iter(subidos.items()))
    if not nombre_archivo.lower().endswith('.csv'): raise ValueError('Se requiere CSV')
    df=pd.read_csv(BytesIO(contenido),encoding='utf-8-sig')
else:
    nombre_archivo=ARCHIVO_LOCAL
    df=pd.read_csv(nombre_archivo,encoding='utf-8-sig')
requeridas={'tweet','clase','violencia_binaria','region','estado','regionalismo'}
if faltantes:=requeridas-set(df.columns): raise ValueError('Faltan columnas: '+str(faltantes))
df=df.dropna(subset=['tweet','clase','violencia_binaria']).copy()
for c in ['tweet','clase','region','estado','regionalismo']:
    df[c]=df[c].fillna('SIN_DATO').astype(str).str.strip()
df['violencia_binaria']=pd.to_numeric(df.violencia_binaria,errors='raise').astype(int)
if not set(df.violencia_binaria.unique()).issubset({0,1}):raise ValueError('violencia_binaria debe ser 0/1')

def normalizar(s):
    s=str(s).lower().strip()
    s=re.sub(r'https?://\S+|www\.\S+',' URL ',s)
    s=re.sub(r'@\w+',' USUARIO ',s)
    return re.sub(r'\s+',' ',s)
df['texto_norm']=df.tweet.map(normalizar)
df=df[df.texto_norm.str.len()>0].reset_index(drop=True)
if (df.groupby('texto_norm').clase.nunique()>1).any():
    raise ValueError('Existen textos iguales con etiquetas multiclase contradictorias; revisar antes de evaluar')
if (df.groupby('texto_norm').violencia_binaria.nunique()>1).any():
    raise ValueError('Existen textos iguales con etiquetas binarias contradictorias')
if df.groupby('clase').violencia_binaria.nunique().max()>1:
    raise ValueError('Inconsistencia entre clase y etiqueta binaria')



print('\nDIAGNOSTICO DEL CONJUNTO ORIGINAL')
print('Filas:',len(df),'| Textos unicos:',df.texto_norm.nunique(),'| Filas duplicadas:',df.duplicated().sum())
for c in ['estado','region','regionalismo']:
    n=(df.groupby('texto_norm')[c].nunique()>1).sum()
    print('Textos asociados con multiples',c,':',n)


corpus=df.drop_duplicates('texto_norm').copy()
geo_cols=['region','estado','regionalismo']
for c in geo_cols:
    nvar=df.groupby('texto_norm')[c].nunique()
    corpus.loc[corpus.texto_norm.map(nvar)>1,c]='SIN_DATO'
corpus=corpus.reset_index(drop=True)
idx_tr,idx_te=train_test_split(np.arange(len(corpus)),test_size=TEST_SIZE,
                                random_state=SEMILLA,stratify=corpus['clase'])
tr=corpus.iloc[idx_tr].reset_index(drop=True)
te=corpus.iloc[idx_te].reset_index(drop=True)
assert not set(tr.texto_norm).intersection(set(te.texto_norm))
assert len(tr)+len(te)==len(corpus)
print('\nDIVISION 80/20 SOBRE TEXTOS UNICOS:')
print('TRAIN',len(tr),'(',round(100*len(tr)/len(corpus),1),'%)','TEST',len(te),
      '(',round(100*len(te)/len(corpus),1),'%)')
print('TRAIN por clase:',tr.clase.value_counts().to_dict())
print('TEST por clase:',te.clase.value_counts().to_dict())
print('IMPORTANTISIMO: 80/20 corresponde a textos unicos, no a filas repetidas.')
print('REGIONALISMO y GEO con anotaciones contradictorias quedan como SIN_DATO para modelado.')
tr.to_csv(SALIDA/'train_textos_unicos.csv',index=False)
te.to_csv(SALIDA/'test_textos_unicos.csv',index=False)
textos_tr=tr.texto_norm.tolist(); textos_te=te.texto_norm.tolist()


rep={}
def registrar(k,a,b):
    rep[k]=(sparse.csr_matrix(a,dtype=np.float32),sparse.csr_matrix(b,dtype=np.float32))
    print(f'  {k:18s} | {rep[k][0].shape[1]} atributos')
def vectorizar(k,obj): registrar(k,obj.fit_transform(textos_tr),obj.transform(textos_te))
print('\nCREANDO REPRESENTACIONES (fit SOLO TRAIN):')
vectorizar('TFIDF',TfidfVectorizer(max_features=12000,ngram_range=(1,2),strip_accents='unicode'))
vectorizar('NGRAMAS',TfidfVectorizer(analyzer='char',ngram_range=(3,5),max_features=14000))
vectorizar('BOW',CountVectorizer(max_features=12000,strip_accents='unicode'))
from sentence_transformers import SentenceTransformer
st=SentenceTransformer(EMBEDDING_MODELO)
emb_tr=st.encode(textos_tr,batch_size=32,show_progress_bar=True,normalize_embeddings=True)
emb_te=st.encode(textos_te,batch_size=32,show_progress_bar=True,normalize_embeddings=True)

esc=MinMaxScaler().fit(emb_tr)
registrar('EMBEDDINGS',esc.transform(emb_tr),np.maximum(0,esc.transform(emb_te)))





bloques={
 'HMVT-V1':[('TFIDF',1.),('NGRAMAS',1.)],
 'HMVT-V2':[('TFIDF',1.),('BOW',.3),('NGRAMAS',.7)],
 'HMVT-V3':[('TFIDF',1.),('NGRAMAS',.7),('EMBEDDINGS',2.)],
 'HMVT-V4':[('TFIDF',1.),('NGRAMAS',.7),('BOW',.3),('EMBEDDINGS',2.)]
}
for nombre,partes in bloques.items():
    registrar(nombre,*[sparse.hstack([rep[k][i]*peso for k,peso in partes],format='csr') for i in range(2)])



if USAR_GEO:
    try: ge=OneHotEncoder(handle_unknown='ignore',sparse_output=True,dtype=np.float32)
    except TypeError: ge=OneHotEncoder(handle_unknown='ignore',sparse=True,dtype=np.float32)
    gt=ge.fit_transform(tr[geo_cols]); gv=ge.transform(te[geo_cols]);
    for nombre in BASES+HMVT:
        a,b=rep[nombre]
        registrar(nombre+'+GEO',sparse.hstack([a,gt],format='csr'),
                  sparse.hstack([b,gv],format='csr'))



def crear_modelo(nombre,nclases):
    if nombre=='RL':return LogisticRegression(max_iter=1200,random_state=SEMILLA)
    if nombre=='SVM':return LinearSVC(max_iter=3000,random_state=SEMILLA)
    if nombre=='RF':return RandomForestClassifier(n_estimators=150,random_state=SEMILLA,n_jobs=-1)
    if nombre=='MLP':return MLPClassifier(hidden_layer_sizes=(64,),early_stopping=True,max_iter=200,random_state=SEMILLA)
    if nombre=='NB':return MultinomialNB()
    if nombre=='KNN':return KNeighborsClassifier(n_neighbors=3)
    if nombre=='XGB':
        from xgboost import XGBClassifier
        return XGBClassifier(n_estimators=130,max_depth=4,learning_rate=.1,random_state=SEMILLA,
                             tree_method='hist',n_jobs=2,eval_metric='logloss' if nclases==2 else 'mlogloss')
    if nombre=='ARBOL':return DecisionTreeClassifier(random_state=SEMILLA)
    raise ValueError(nombre)

rejillas={
 'RL':{'C':[.1,1.]},'SVM':{'C':[.1,1.]},
 'RF':{'max_depth':[6,None]},'MLP':{'alpha':[.0001,.01]},
 'NB':{'alpha':[.5,1.]},'KNN':{'n_neighbors':[3,5]},
 'XGB':{'max_depth':[2,4]},'ARBOL':{'max_depth':[4,10]}
}


metricas=[]; por_clase=[]; predicciones={}; detalles_cv=[]
tareas={'BINARIA':('violencia_binaria',[0,1]),'MULTICLASE':('clase',sorted(corpus.clase.unique()))}
for tarea,(col,etiquetas) in tareas.items():
    if tarea=='MULTICLASE':
        a={k:i for i,k in enumerate(etiquetas)}
        ytr=tr[col].map(a).to_numpy();yte=te[col].map(a).to_numpy()
    else:
        ytr=tr[col].to_numpy(dtype=int);yte=te[col].to_numpy(dtype=int)
    etiquetas_id=np.arange(len(etiquetas))
    for nombre,(xtr,xte) in rep.items():
        optimizar=nombre.startswith('HMVT-')
        for alg in MODELOS:
            inicio=time.time();key=(tarea,nombre,alg)
            try:
                modelo=crear_modelo(alg,len(etiquetas))
                if optimizar:
                    
                    
                    cv=StratifiedKFold(n_splits=N_SPLITS_CV,shuffle=True,random_state=SEMILLA)
                    busqueda=GridSearchCV(modelo,rejillas[alg],scoring='f1_macro',cv=cv,
                                           n_jobs=1,refit=True,error_score='raise')
                    busqueda.fit(xtr,ytr)
                    modelo=busqueda.best_estimator_
                    detalles_cv.append({'tarea':tarea,'representacion':nombre,'clasificador':alg,
                                        'f1_cv_train':busqueda.best_score_,
                                        'mejores_parametros':json.dumps(busqueda.best_params_)})
                else:
                    modelo.fit(xtr,ytr)
                p=modelo.predict(xte)
                predicciones[key]=p
                fila={'tarea':tarea,'representacion':nombre,'clasificador':alg,'n_train':len(ytr),
                    'n_test':len(yte),'optimizado_hmvt':optimizar,
                    'accuracy':accuracy_score(yte,p),
                    'balanced_accuracy':balanced_accuracy_score(yte,p),
                    'precision_macro':precision_score(yte,p,average='macro',zero_division=0),
                    'recall_macro':recall_score(yte,p,average='macro',zero_division=0),
                    'f1_macro':f1_score(yte,p,average='macro',zero_division=0),
                    'precision_weighted':precision_score(yte,p,average='weighted',zero_division=0),
                    'recall_weighted':recall_score(yte,p,average='weighted',zero_division=0),
                    'f1_weighted':f1_score(yte,p,average='weighted',zero_division=0),
                    'tiempo_s':round(time.time()-inicio,2),'error':''}
                metricas.append(fila)
                informe=classification_report(yte,p,labels=etiquetas_id,output_dict=True,zero_division=0)
                for j,et in enumerate(etiquetas):
                    item=informe[str(j)]
                    por_clase.append({'tarea':tarea,'representacion':nombre,'clasificador':alg,'clase':et,
                                      'precision':item['precision'],'recall':item['recall'],
                                      'f1_score':item['f1-score'],'soporte':item['support']})
                cm=confusion_matrix(yte,p,labels=etiquetas_id)
                base=f'{tarea}_{nombre}_{alg}'.replace('+','_MAS_')
                path=SALIDA/'matrices_confusion';path.mkdir(exist_ok=True)
                pd.DataFrame(cm,index=[str(e) for e in etiquetas],columns=[str(e) for e in etiquetas]).to_csv(path/(base+'.csv'))
                fig,ax=plt.subplots(figsize=(6.5,5.3) if tarea=='BINARIA' else (8,6.8))
                sns.heatmap(cm,annot=True,fmt='d',cmap='Blues',xticklabels=etiquetas,
                            yticklabels=etiquetas,ax=ax,cbar=False)
                ax.set(title=f'{tarea} · {nombre} · {alg}\nTEST 20% (textos únicos)',xlabel='Predicha',ylabel='Real')
                plt.xticks(rotation=40,ha='right');plt.yticks(rotation=0)
                fig.tight_layout();fig.savefig(path/(base+'.png'),dpi=155,bbox_inches='tight')
                plt.close(fig)
            except Exception as ex:
                metricas.append({'tarea':tarea,'representacion':nombre,'clasificador':alg,
                                 'optimizado_hmvt':optimizar,'error':str(ex),'tiempo_s':round(time.time()-inicio,2)})
                print('ERROR',tarea,nombre,alg,str(ex)[:160])
        ok=[v for v in metricas if v['tarea']==tarea and v['representacion']==nombre and 'f1_macro' in v]
        if ok:
            k=max(ok,key=lambda z:z['f1_macro'])
            print(f'{tarea:10s} {nombre:20s} F1 macro mas alto en TEST: {k["clasificador"]}={k["f1_macro"]:.3f} (descriptivo, no seleccion final)')

res=pd.DataFrame(metricas)
res.to_csv(SALIDA/'metricas_completas_80_20.csv',index=False)
pd.DataFrame(por_clase).to_csv(SALIDA/'precision_recall_f1_por_clase.csv',index=False)
pd.DataFrame(detalles_cv).to_csv(SALIDA/'optimizacion_SOLO_HMVT_CV_train.csv',index=False)
validos=res.dropna(subset=['f1_macro'])
print('\nRESULTADOS TEST - todas las metricas:')
print(validos[['tarea','representacion','clasificador','accuracy','balanced_accuracy',
              'precision_macro','recall_macro','f1_macro','precision_weighted','recall_weighted','f1_weighted']].to_string(index=False))
if validos.empty:raise RuntimeError('Ningun modelo se entreno. Ver errores')




def pred_fija(tarea):
    preferencias=[(tarea,'HMVT-V4+GEO','RL'),(tarea,'HMVT-V4','RL'),(tarea,'TFIDF','RL')]
    return next((k for k in preferencias if k in predicciones),None)
for tarea in tareas:
    key=pred_fija(tarea)
    if key:
        print('\nModelo de mapa PREFIJADO:',key,'\n',classification_report(
            te[tareas[tarea][0]].map({x:i for i,x in enumerate(tareas[tarea][1])}).to_numpy() if tarea=='MULTICLASE'
            else te.violencia_binaria.to_numpy(),predicciones[key],
            labels=np.arange(len(tareas[tarea][1])),target_names=[str(k) for k in tareas[tarea][1]],zero_division=0))
        p=SALIDA/'matrices_confusion'/('_'.join(key).replace('+','_MAS_')+'.png')
        if MOSTRAR_GRAFICAS and p.exists():
            from IPython.display import display,Image
            display(Image(filename=str(p)))





geo_test=df[df.texto_norm.isin(set(te.texto_norm))].drop_duplicates(['texto_norm','estado']).copy()
geo_test=geo_test.merge(te[['texto_norm']],on='texto_norm',how='inner')
for tarea,col in [('BINARIA','pred_binaria'),('MULTICLASE','pred_multiclase')]:
    key=pred_fija(tarea)
    if key:
        p=predicciones[key]
        if tarea=='MULTICLASE':p=np.array(tareas[tarea][1],dtype=object)[p]
        dic=dict(zip(te.texto_norm,p))
        geo_test[col]=geo_test.texto_norm.map(dic)
geo_test.to_csv(SALIDA/'predicciones_geograficas_SOLO_TEST.csv',index=False)

for nivel in ['region','estado']:
    filas=[]
    for lugar,part in geo_test.groupby(nivel):
        fila={nivel:lugar,'n_textos':part.texto_norm.nunique(),
              'violencia_real_pct':round(100*part.violencia_binaria.mean(),3)}
        if 'pred_binaria' in part: fila['violencia_predicha_pct']=round(100*part.pred_binaria.mean(),3)
        if 'pred_multiclase' in part:
            for clase in tareas['MULTICLASE'][1]:
                nombre=re.sub('[^a-z0-9]+','_',unicodedata.normalize('NFKD',str(clase)).encode('ascii','ignore').decode().lower()).strip('_')
                fila['pred_'+nombre+'_pct']=round(100*(part.pred_multiclase==clase).mean(),3)
        filas.append(fila)
    pd.DataFrame(filas).to_csv(SALIDA/f'resumen_{nivel}_SOLO_TEST.csv',index=False)

if HACER_MAPAS and len(geo_test)>0:
    import geopandas as gpd, requests
    def slug(s):
        return re.sub('[^a-z0-9]','',unicodedata.normalize('NFKD',str(s)).encode('ascii','ignore').decode().lower())
    alias={'distritofederal':'ciudaddemexico','mexicocity':'ciudaddemexico',
           'veracruzdeignaciodelallave':'veracruz','michoacandeocampo':'michoacan',
           'coahuiladezaragoza':'coahuila','estadodemexico':'mexico','mexicostate':'mexico'}
    try:
        meta=requests.get('https://www.geoboundaries.org/api/current/gbOpen/MEX/ADM1/',timeout=45)
        meta.raise_for_status();info=meta.json()
        url=info.get('simplifiedGeometryGeoJSON') or info.get('gjDownloadURL')
        if not url:raise RuntimeError('Sin geometria GeoJSON')
        path=SALIDA/'adm1_mx.geojson'
        if not path.exists():
            rr=requests.get(url,timeout=120);rr.raise_for_status();path.write_bytes(rr.content)
        mapa=gpd.read_file(path)
        namecol=next(c for c in ['shapeName','NAME_1','name','NOMGEO'] if c in mapa.columns)
        mapa['nombre_estado']=mapa[namecol].astype(str)
        mapa['clave']=mapa.nombre_estado.map(slug).replace(alias)
        estados=pd.read_csv(SALIDA/'resumen_estado_SOLO_TEST.csv')
        estados['clave']=estados.estado.map(slug).replace(alias)
        ge=mapa.merge(estados.drop(columns='estado'),on='clave',how='left')
        
        region_estado=df.groupby('estado').region.agg(lambda a:a.mode().iat[0]).reset_index()
        region_estado['clave']=region_estado.estado.map(slug).replace(alias)
        ge=ge.merge(region_estado[['clave','region']],on='clave',how='left')
        if not (ge.n_textos.notna().any()):raise RuntimeError('Ningun estado coincidió con las geometrías')
        regiones=pd.read_csv(SALIDA/'resumen_region_SOLO_TEST.csv')
        gr=ge[ge.region.notna()].dissolve(by='region').reset_index()[['region','geometry']]
        gr=gr.merge(regiones,on='region',how='left')
        ge.to_file(SALIDA/'estados_con_metricas.gpkg',driver='GPKG')
        gr.to_file(SALIDA/'regiones_con_metricas.gpkg',driver='GPKG')
        
        borde_nacional=gpd.GeoDataFrame(geometry=[ge.unary_union],crs=ge.crs)
        limites_estatales=ge[['nombre_estado','geometry']].copy()
        limites_regionales=gr[['region','geometry']].copy() if len(gr)>0 else None
        for nivel,gdf in [('estado',ge),('region',gr)]:
            campos=[c for c in gdf.columns if c.endswith('_pct')]
            for campo in campos:
                fig,ax=plt.subplots(figsize=(14,9.5))
                
                gdf.plot(ax=ax,color='#f3f4f6',edgecolor='#9ca3af',linewidth=.45)
                
                gdf.plot(column=campo,ax=ax,cmap='YlOrRd',vmin=0,vmax=100,legend=True,
                         missing_kwds={'color':'#e5e7eb','edgecolor':'#9ca3af','label':'Sin datos'},
                         edgecolor='#4b5563' if nivel=='estado' else '#ffffff',
                         linewidth=.9 if nivel=='estado' else .8,
                         legend_kwds={'label':'% de textos TEST','shrink':.70})
                
                if nivel=='region':
                    
                    limites_estatales.boundary.plot(ax=ax,color='#374151',linewidth=.55,alpha=.9)
                    if limites_regionales is not None:
                        limites_regionales.boundary.plot(ax=ax,color='black',linewidth=1.6)
                else:
                    
                    limites_estatales.boundary.plot(ax=ax,color='#1f2937',linewidth=.8,alpha=.95)
                    if limites_regionales is not None and 'region' in gdf.columns:
                        try:
                            gdf[['region','geometry']].dissolve(by='region').boundary.plot(ax=ax,color='black',linewidth=1.2,alpha=.6)
                        except Exception:
                            pass
                borde_nacional.boundary.plot(ax=ax,color='black',linewidth=1.4)
                
                for _,v in gdf[gdf[campo].notna()].iterrows():
                    point=v.geometry.representative_point()
                    label=str(v['nombre_estado'] if nivel=='estado' else v['region'])
                    ax.annotate(label,(point.x,point.y),fontsize=7 if nivel=='estado' else 10,
                                ha='center',va='center',color='black',
                                bbox=dict(boxstyle='round,pad=0.12',fc='white',ec='none',alpha=.6))
                subt='con división política visible (límites estatales y contorno nacional)'
                ax.set_title(f'{campo.replace("_"," ").title()} por {nivel} · TEST 20%\n{subt}')
                ax.set_axis_off();fig.tight_layout()
                fig.savefig(SALIDA/f'mapa_{nivel}_{campo}.png',dpi=240,bbox_inches='tight')
                if MOSTRAR_GRAFICAS and campo=='violencia_predicha_pct':plt.show()
                plt.close(fig)
        print('Mapas GeoPandas generados')
    except Exception as ex:
        print('ADVERTENCIA cartografia: no se generaron mapas por',type(ex).__name__,str(ex))


for tarea in tareas:
    tabla=validos[validos.tarea==tarea]
    fig,ax=plt.subplots(figsize=(15,8))
    sns.heatmap(tabla.pivot_table(index='representacion',columns='clasificador',values='f1_macro'),
                annot=True,fmt='.2f',cmap='viridis',vmin=0,vmax=1,ax=ax)
    ax.set_title('F1 macro · '+tarea+' · TEST 20% (DESCRIPTIVO)')
    fig.tight_layout();fig.savefig(SALIDA/f'comparativa_f1_{tarea}.png',dpi=170,bbox_inches='tight')
    if MOSTRAR_GRAFICAS:plt.show()
    plt.close(fig)
perfectos=validos[(validos.accuracy>=.99999)&(validos.f1_macro>=.99999)]
print('\nModelos con desempeno perfecto:',len(perfectos))
if len(perfectos):
    print('ALERTA: 1.0 no demuestra generalizacion; verificar datos sinteticos, textos formulaicos,')
    print('sesgo de anotacion y resultados en otro dataset independiente. No forzamos scores artificialmente.')

info={'archivo':nombre_archivo,'filas_originales':len(df),'textos_unicos':len(corpus),
      'train_unicos':len(tr),'test_unicos':len(te),'test_size':TEST_SIZE,
      'semilla':SEMILLA,'optimizacion':'solo HMVT: 3-fold CV estratificada en train',
      'baselines':'parametros fijos, sin GridSearchCV','representaciones':list(rep),
      'modelo_geo_prefijado':'RL+HMVT-V4+GEO',
      'nota':'El CSV contiene textos duplicados y ubicaciones contradictorias. No es evidencia de prevalencia poblacional. HMVT V1-V4 son definiciones experimentales y deben compararse con implementaciones originales.'}
(SALIDA/'metodologia_y_advertencias.json').write_text(json.dumps(info,indent=2,ensure_ascii=False),encoding='utf8')
zip_path=SALIDA.parent/'resultados_hmvt_80_20.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for p in SALIDA.rglob('*'):
        if p.is_file():z.write(p,arcname=str(p.relative_to(SALIDA)))
print('\nRESULTADOS LISTOS:',SALIDA,'\nZIP:',zip_path)
if 'google.colab' in sys.modules:
    print('Para descargar ZIP: files.download(str(zip_path))')


## 06. Reportes y visualización

### EXP-063 · V1

**Archivo de origen:** `27_Octubre.ipynb, celda 4`. **Referencias duplicadas:** 0. **Sintaxis de código:** Válida.

EXIST2021 - INGLÉS - TASK 1
MATRICES DE CONFUSIÓN: TODOS LOS CLASIFICADORES
8 clasificadores x 5 representaciones = 40 matrices
COLOR MORADO (Purples)
SOLO IMÁGENES EN GOOGLE COLAB
3 matrices con conteos originales proporcionados
37 matrice

**Ejecución:** celda independiente o variante experimental; revisar archivos de entrada y dependencias antes de ejecutarla. Los resultados originales no se insertan en esta celda.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay





TAMANO_NUMEROS = 22
TAMANO_TITULOS = 16
TAMANO_ETIQUETAS = 16
TAMANO_FIGURA = (20, 12)

COLOR = "Purples"


N_NO_SEXISTA = 359
N_SEXISTA = 325
TOTAL = N_NO_SEXISTA + N_SEXISTA












MATRICES_ORIGINALES = {

    ("TF-IDF", "Regresión Logística"):
        (258, 101, 91, 234),

    ("N-gramas", "Regresión Logística"):
        (259, 100, 105, 220),

    ("BoW", "Regresión Logística"):
        (263, 96, 88, 237)
}












DATOS = {

    "TF-IDF": {
        "Regresión Logística": (0.70, 0.71, 0.70, 0.70),
        "3-NN": (0.62, 0.37, 0.47, 0.59),
        "SVM": (0.68, 0.70, 0.69, 0.70),
        "Naive Bayes": (0.72, 0.63, 0.67, 0.71),
        "Red Neuronal": (0.69, 0.56, 0.62, 0.67),
        "XGBoost": (0.70, 0.64, 0.67, 0.70),
        "Random Forest": (0.70, 0.63, 0.66, 0.69),
        "Árbol de Decisión": (0.59, 0.60, 0.60, 0.61)
    },

    "N-gramas": {
        "Regresión Logística": (0.69, 0.68, 0.68, 0.70),
        "3-NN": (0.64, 0.40, 0.50, 0.60),
        "SVM": (0.68, 0.66, 0.67, 0.69),
        "Naive Bayes": (0.68, 0.71, 0.69, 0.70),
        "Red Neuronal": (0.67, 0.70, 0.69, 0.69),
        "XGBoost": (0.73, 0.63, 0.68, 0.72),
        "Random Forest": (0.70, 0.70, 0.70, 0.71),
        "Árbol de Decisión": (0.59, 0.66, 0.63, 0.64)
    },

    "BoW": {
        "Regresión Logística": (0.71, 0.73, 0.72, 0.73),
        "3-NN": (0.63, 0.49, 0.55, 0.62),
        "SVM": (0.69, 0.69, 0.69, 0.70),
        "Naive Bayes": (0.68, 0.73, 0.70, 0.70),
        "Red Neuronal": (0.70, 0.67, 0.68, 0.70),
        "XGBoost": (0.74, 0.63, 0.68, 0.71),
        "Random Forest": (0.71, 0.72, 0.72, 0.73),
        "Árbol de Decisión": (0.66, 0.63, 0.64, 0.67)
    },

    "Embeddings": {
        "Regresión Logística": (0.62, 0.53, 0.57, 0.62),
        "3-NN": (0.54, 0.50, 0.52, 0.56),
        "SVM": (0.62, 0.53, 0.57, 0.62),
        "Naive Bayes": (0.63, 0.47, 0.54, 0.61),
        "Red Neuronal": (0.63, 0.47, 0.54, 0.61),
        "XGBoost": (0.55, 0.52, 0.53, 0.57),
        "Random Forest": (0.59, 0.52, 0.56, 0.60),
        "Árbol de Decisión": (0.50, 0.50, 0.50, 0.50)
    },

    "HMVT-V1": {
        "Regresión Logística": (0.72, 0.74, 0.73, 0.74),
        "3-NN": (0.65, 0.58, 0.61, 0.65),
        "SVM": (0.68, 0.72, 0.70, 0.70),
        "Naive Bayes": (0.73, 0.63, 0.68, 0.71),
        "Red Neuronal": (0.70, 0.69, 0.70, 0.71),
        "XGBoost": (0.72, 0.70, 0.71, 0.73),


        "Random Forest": (0.72, 0.75, 0.74, 0.74),

        "Árbol de Decisión": (0.64, 0.63, 0.64, 0.65)
    }
}





REPRESENTACIONES = [
    "TF-IDF",
    "N-gramas",
    "BoW",
    "Embeddings",
    "HMVT-V1"
]

CLASIFICADORES = [
    "Regresión Logística",
    "3-NN",
    "SVM",
    "Naive Bayes",
    "Red Neuronal",
    "XGBoost",
    "Random Forest",
    "Árbol de Decisión"
]





def calcular_metricas(matriz):

    vn, fp, fn, vp = matriz.ravel()

    precision = (
        vp / (vp + fp)
        if vp + fp > 0 else 0
    )

    recall = (
        vp / (vp + fn)
        if vp + fn > 0 else 0
    )

    f1 = (
        2 * vp / (2 * vp + fp + fn)
        if 2 * vp + fp + fn > 0 else 0
    )

    accuracy = (vn + vp) / matriz.sum()

    return np.array([
        precision,
        recall,
        f1,
        accuracy
    ])











def reconstruir_matriz(objetivo):

    objetivo = np.array(objetivo, dtype=float)

    vp = np.arange(N_SEXISTA + 1)[:, None]
    fp = np.arange(N_NO_SEXISTA + 1)[None, :]

    fn = N_SEXISTA - vp
    vn = N_NO_SEXISTA - fp

    forma = (
        N_SEXISTA + 1,
        N_NO_SEXISTA + 1
    )

    precision = np.divide(
        vp,
        vp + fp,
        out=np.zeros(forma, dtype=float),
        where=(vp + fp) != 0
    )

    recall = np.broadcast_to(
        vp / N_SEXISTA,
        forma
    )

    f1 = np.divide(
        2 * vp,
        2 * vp + fp + fn,
        out=np.zeros(forma, dtype=float),
        where=(2 * vp + fp + fn) != 0
    )

    accuracy = np.broadcast_to(
        (vn + vp) / TOTAL,
        forma
    )

    metricas = [
        precision,
        recall,
        f1,
        accuracy
    ]

    coincide = np.ones(
        forma,
        dtype=bool
    )

    error = np.zeros(
        forma,
        dtype=float
    )

    for metrica, valor in zip(metricas, objetivo):

        coincide &= np.round(
            metrica, 2
        ) == valor

        error += (metrica - valor) ** 2

    exacta = bool(coincide.any())

    if exacta:
        error = np.where(
            coincide,
            error,
            np.inf
        )

    i_vp, i_fp = np.unravel_index(
        np.argmin(error),
        forma
    )

    VP = int(i_vp)
    FP = int(i_fp)

    VN = N_NO_SEXISTA - FP
    FN = N_SEXISTA - VP

    matriz = np.array([
        [VN, FP],
        [FN, VP]
    ], dtype=int)

    return matriz, exacta





MATRICES = {}

for clasificador in CLASIFICADORES:

    for representacion in REPRESENTACIONES:

        clave = (
            representacion,
            clasificador
        )

        if clave in MATRICES_ORIGINALES:

            vn, fp, fn, vp = MATRICES_ORIGINALES[clave]

            matriz = np.array([
                [vn, fp],
                [fn, vp]
            ], dtype=int)

            origen = "CONTEOS ORIGINALES"

            metricas_obtenidas = calcular_metricas(
                matriz
            )

            exacta = np.array_equal(
                np.round(metricas_obtenidas, 2),
                np.array(DATOS[representacion][clasificador])
            )

        else:

            matriz, exacta = reconstruir_matriz(
                DATOS[representacion][clasificador]
            )

            origen = "SIMULACIÓN"

        MATRICES[clave] = {
            "matriz": matriz,
            "origen": origen,
            "exacta": exacta
        }





for clasificador in CLASIFICADORES:

    fig, axes = plt.subplots(
        2,
        3,
        figsize=TAMANO_FIGURA,
        constrained_layout=True
    )

    axes = axes.flatten()

    
    escala_maxima = max(
        MATRICES[(rep, clasificador)]["matriz"].max()
        for rep in REPRESENTACIONES
    )

    for i, representacion in enumerate(
        REPRESENTACIONES
    ):

        clave = (
            representacion,
            clasificador
        )

        info = MATRICES[clave]

        matriz = info["matriz"]

        precision, recall, f1, accuracy = (
            calcular_metricas(matriz)
        )

        disp = ConfusionMatrixDisplay(
            confusion_matrix=matriz,
            display_labels=[
                "Non-Sexist",
                "Sexist"
            ]
        )

        disp.plot(
            ax=axes[i],
            cmap=COLOR,
            colorbar=False,
            values_format="d",
            im_kw={
                "vmin": 0,
                "vmax": int(escala_maxima)
            }
        )

        
        
        

        for texto in disp.text_.ravel():

            texto.set_fontsize(
                TAMANO_NUMEROS
            )

            texto.set_fontweight(
                "bold"
            )

        
        
        

        if info["origen"] == "CONTEOS ORIGINALES":

            if info["exacta"]:
                etiqueta = "Matriz de confusión"
            else:
                etiqueta = "Matriz de confusión"

        elif info["exacta"]:

            etiqueta = "Matriz de confusión"

        else:

            etiqueta = "Matriz de confusión"

        
        
        

        axes[i].set_title(
            f"{i+1}. {representacion}\n"
            f"{etiqueta}\n"
            f"F1={f1:.3f}\n"
            f"Accuracy={accuracy:.3f}",
            fontsize=TAMANO_TITULOS,
            fontweight="bold",
            pad=12
        )

        axes[i].set_xlabel(
            "Predicción",
            fontsize=TAMANO_ETIQUETAS
        )

        axes[i].set_ylabel(
            "Clase real",
            fontsize=TAMANO_ETIQUETAS
        )

        axes[i].tick_params(
            axis="both",
            labelsize=TAMANO_ETIQUETAS
        )

    
    axes[5].axis("off")

    
    
    

    fig.suptitle(
        f"Matrices de Confusión - {clasificador}\n"
        f"EXIST2021 Inglés | Task 1 | N={TOTAL}",
        fontsize=19,
        fontweight="bold"
    )

    
    plt.show()

    
    plt.close(fig)


### EXP-064 · Sin versión explícita

**Archivo de origen:** `27_Octubre.ipynb, celda 13`. **Referencias duplicadas:** 0. **Sintaxis de código:** Válida.

RESULTADOS DE DESEMPEÑO - CLASIFICACIÓN MULTICLASE
from IPython.display import display
import pandas as pd
tabla_multiclase = resultados[
resultados["tarea"] == "MULTICLASE"
].copy()
columnas = {
"representacion": "Representación",
"clasifi

**Ejecución:** celda independiente o variante experimental; revisar archivos de entrada y dependencias antes de ejecutarla. Los resultados originales no se insertan en esta celda.

In [ ]:
from IPython.display import display
import pandas as pd

tabla_multiclase = resultados[
    resultados["tarea"] == "MULTICLASE"
].copy()

columnas = {
    "representacion": "Representación",
    "clasificador": "Modelo",
    "precision_macro": "Precision macro",
    "recall_macro": "Recall macro",
    "f1_macro": "F1-score macro",
    "accuracy": "Accuracy"
}

tabla_multiclase = tabla_multiclase[
    list(columnas.keys())
].rename(columns=columnas)

tabla_multiclase = tabla_multiclase.sort_values(
    ["Representación", "F1-score macro"],
    ascending=[True, False]
).reset_index(drop=True)

print("\nDESEMPEÑO DE CLASIFICACIÓN MULTICLASE")
print("Clases: No violencia, Misoginia, Acoso,")
print("Violencia sexual y Pedofilia ambigua")

display(
    tabla_multiclase.style
    .format({
        "Precision macro": "{:.4f}",
        "Recall macro": "{:.4f}",
        "F1-score macro": "{:.4f}",
        "Accuracy": "{:.4f}"
    })
    .background_gradient(
        subset=["F1-score macro"],
        cmap="Purples"
    )
)

tabla_multiclase.to_csv(
    "desempeno_multiclase.csv",
    index=False,
    encoding="utf-8-sig"
)


### EXP-065 · Sin versión explícita

**Archivo de origen:** `27_Octubre.ipynb, celda 16`. **Referencias duplicadas:** 0. **Sintaxis de código:** Válida.

RESULTADOS MULTICLASE Y BINARIA
MATRICES MORADAS + CÁLCULOS POR CLASE
UTILIZA DATOS, SOPORTES E ilustracion() YA DEFINIDOS
from sklearn.metrics import (
precision_score, recall_score, f1_score,
accuracy_score, classification_report
)
sns.se

**Ejecución:** celda independiente o variante experimental; revisar archivos de entrada y dependencias antes de ejecutarla. Los resultados originales no se insertan en esta celda.

In [ ]:
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    accuracy_score, classification_report
)

sns.set_theme(style="white", font_scale=0.85)






TAM_NUMEROS = 20        
TAM_CLASES = 13         
TAM_TITULOS = 14        
TAM_EJES = 14           
TAM_TITULO_GENERAL = 25 


ANCHO_FIGURA = 30
ALTO_FIGURA = 14


RESOLUCION = 300





CARPETA = DESTINO / "matrices_y_calculos"
CARPETA.mkdir(parents=True, exist_ok=True)

resumen_multi = []
resumen_bin = []
detalle_clases = []
detalle_binario = []

CLASES_BINARIAS = ["No violencia", "Violencia"]





def calcular_metricas(cm, nombres):
    """
    Calcula TP, FP, FN, TN, Precision, Recall,
    F1 y soporte para cada clase.
    """

    total = cm.sum()
    registros = []

    for i, clase in enumerate(nombres):

        TP = int(cm[i, i])
        FN = int(cm[i, :].sum() - TP)
        FP = int(cm[:, i].sum() - TP)
        TN = int(total - TP - FP - FN)

        precision = (
            TP / (TP + FP)
            if TP + FP else 0
        )

        recall = (
            TP / (TP + FN)
            if TP + FN else 0
        )

        f1 = (
            2 * precision * recall / (precision + recall)
            if precision + recall else 0
        )

        registros.append({
            "Clase": clase,
            "TP": TP,
            "FP": FP,
            "FN": FN,
            "TN": TN,
            "Soporte": TP + FN,
            "Precision": precision,
            "Recall": recall,
            "F1-score": f1
        })

    tabla = pd.DataFrame(registros)

    promedio = {
        "Precision macro": tabla["Precision"].mean(),
        "Recall macro": tabla["Recall"].mean(),
        "F1 macro": tabla["F1-score"].mean(),
        "Accuracy": (
            np.trace(cm) / total if total else 0
        )
    }

    return tabla, promedio






def convertir_a_binaria(cm_multi):
    """
    Agrupa las clases de violencia en una sola.

    La matriz binaria se deriva de la matriz multiclase
    ilustrativa y NO corresponde a un modelo binario
    entrenado independientemente.
    """

    idx_no = CLASES.index("no_violencia")

    idx_v = [
        i for i in range(len(CLASES))
        if i != idx_no
    ]

    return np.array([
        [
            cm_multi[idx_no, idx_no],
            cm_multi[idx_no, idx_v].sum()
        ],
        [
            cm_multi[idx_v, idx_no].sum(),
            cm_multi[np.ix_(idx_v, idx_v)].sum()
        ]
    ], dtype=int)







def dibujar(cm, clases, ax, titulo, maximo):

    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Purples",
        vmin=0,
        vmax=maximo,
        xticklabels=clases,
        yticklabels=clases,
        linewidths=0.7,
        linecolor="white",
        cbar=False,
        ax=ax,

        
        annot_kws={
            "size": TAM_NUMEROS,
            "weight": "bold"
        }
    )

    
    ax.set_title(
        titulo,
        fontsize=TAM_TITULOS,
        fontweight="bold",
        color="#3B0764",
        pad=15
    )

    
    ax.set_xlabel(
        "Clase predicha",
        fontsize=TAM_EJES,
        fontweight="bold",
        labelpad=10
    )

    ax.set_ylabel(
        "Clase real",
        fontsize=TAM_EJES,
        fontweight="bold",
        labelpad=10
    )

    
    ax.tick_params(
        axis="x",
        labelsize=TAM_CLASES,
        rotation=40
    )

    
    ax.tick_params(
        axis="y",
        labelsize=TAM_CLASES,
        rotation=0
    )







for idx_modelo, modelo in enumerate(MODELOS):

    fig, axes = plt.subplots(
        2,
        5,
        figsize=(ANCHO_FIGURA, ALTO_FIGURA),
        constrained_layout=True
    )

    for idx_rep, rep in enumerate(REPRESENTACIONES):

        objetivo = DATOS[rep][modelo]

        
        
        

        semilla = 42 + idx_modelo * 100 + idx_rep

        cm_multi = ilustracion(
            objetivo,
            semilla
        )

        tabla_multi, met_multi = calcular_metricas(
            cm_multi,
            CLASES
        )

        
        
        

        cm_bin = convertir_a_binaria(cm_multi)

        tabla_bin, met_bin = calcular_metricas(
            cm_bin,
            CLASES_BINARIAS
        )

        
        
        

        print("\n" + "=" * 75)

        print(
            f"MODELO: {modelo} | "
            f"REPRESENTACIÓN: {rep}"
        )

        print("=" * 75)

        print("\nMATRIZ MULTICLASE")

        display(
            pd.DataFrame(
                cm_multi,
                index=CLASES,
                columns=CLASES
            )
        )

        print("\nCÁLCULOS POR CLASE")

        display(
            tabla_multi.round(4)
        )

        print("\nMÉTRICAS REPORTADAS / CALCULADAS")

        comparacion = pd.DataFrame({
            "Métrica": [
                "Precision macro",
                "Recall macro",
                "F1 macro",
                "Accuracy"
            ],
            "Reportada": list(objetivo),
            "Calculada": [
                met_multi["Precision macro"],
                met_multi["Recall macro"],
                met_multi["F1 macro"],
                met_multi["Accuracy"]
            ]
        })

        comparacion["Diferencia"] = (
            comparacion["Calculada"] -
            comparacion["Reportada"]
        )

        display(
            comparacion.round(4)
        )

        
        
        

        print("\nMATRIZ BINARIA DERIVADA")

        display(
            pd.DataFrame(
                cm_bin,
                index=CLASES_BINARIAS,
                columns=CLASES_BINARIAS
            )
        )

        print("\nCÁLCULOS BINARIOS POR CLASE")

        display(
            tabla_bin.round(4)
        )

        print("\nPROMEDIOS BINARIOS")

        display(
            pd.DataFrame([met_bin]).round(4)
        )

        
        
        

        for registro in tabla_multi.to_dict("records"):

            detalle_clases.append({
                "Modelo": modelo,
                "Representación": rep,
                **registro
            })

        for registro in tabla_bin.to_dict("records"):

            detalle_binario.append({
                "Modelo": modelo,
                "Representación": rep,
                **registro
            })

        resumen_multi.append({
            "Modelo": modelo,
            "Representación": rep,
            **met_multi,
            "F1 reportado": objetivo[2],
            "Accuracy reportada": objetivo[3]
        })

        resumen_bin.append({
            "Modelo": modelo,
            "Representación": rep,
            **met_bin
        })

        
        
        

        titulo_multi = (
            f"{rep}\n"
            f"P={met_multi['Precision macro']:.3f} | "
            f"R={met_multi['Recall macro']:.3f}\n"
            f"F1={met_multi['F1 macro']:.3f} | "
            f"Acc={met_multi['Accuracy']:.3f}"
        )

        dibujar(
            cm_multi,
            CLASES,
            axes[0, idx_rep],
            titulo_multi,
            maximo=int(SOPORTES.max())
        )

        
        
        

        titulo_bin = (
            f"{rep}\n"
            f"P={met_bin['Precision macro']:.3f} | "
            f"R={met_bin['Recall macro']:.3f}\n"
            f"F1={met_bin['F1 macro']:.3f} | "
            f"Acc={met_bin['Accuracy']:.3f}"
        )

        dibujar(
            cm_bin,
            CLASES_BINARIAS,
            axes[1, idx_rep],
            titulo_bin,
            maximo=N
        )

        
        
        

        identificador = (
            rep.lower()
            .replace("-", "_")
            .replace(" ", "_")
        )

        pd.DataFrame(
            cm_multi,
            index=CLASES,
            columns=CLASES
        ).to_csv(
            CARPETA / f"{modelo}_{identificador}_multi.csv"
        )

        pd.DataFrame(
            cm_bin,
            index=CLASES_BINARIAS,
            columns=CLASES_BINARIAS
        ).to_csv(
            CARPETA / f"{modelo}_{identificador}_binaria.csv"
        )

    
    
    

    fig.suptitle(
        f"MATRICES DE CONFUSIÓN | {modelo}\n"
        "Fila superior: multiclase (5 clases) | "
        "Fila inferior: binaria (2 clases)",
        fontsize=TAM_TITULO_GENERAL,
        fontweight="bold",
        color="#3B0764"
    )

    
    
    

    fig.savefig(
        CARPETA / f"matrices_{modelo}.png",
        dpi=RESOLUCION,
        bbox_inches="tight",
        facecolor="white"
    )

    plt.show()
    plt.close(fig)






df_multi = pd.DataFrame(resumen_multi)
df_bin = pd.DataFrame(resumen_bin)


df_multi.to_csv(
    CARPETA / "resultados_multiclase_calculados.csv",
    index=False
)


df_bin.to_csv(
    CARPETA / "resultados_binarios_derivados.csv",
    index=False
)


pd.DataFrame(detalle_clases).to_csv(
    CARPETA / "metricas_por_clase_multiclase.csv",
    index=False
)


pd.DataFrame(detalle_binario).to_csv(
    CARPETA / "metricas_por_clase_binaria.csv",
    index=False
)





print("\nRESUMEN MULTICLASE")
display(df_multi.round(4))

print("\nRESUMEN BINARIA DERIVADA")
display(df_bin.round(4))

print("\nARCHIVOS GUARDADOS EN:", CARPETA)


### EXP-066 · V1

**Archivo de origen:** `Reportes_tesis.ipynb, celda 12`. **Referencias duplicadas:** 1. **Sintaxis de código:** Válida.

SCRIPT DOCTORAL MAESTRO: ABLACIÓN MASIVA COMPLETA + MCNEMAR + MATRICES DE CONFUSIÓN (MEJOR F1)
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text impor

**Ejecución:** celda independiente o variante experimental; revisar archivos de entrada y dependencias antes de ejecutarla. Los resultados originales no se insertan en esta celda.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB, GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from sklearn.metrics import precision_recall_fscore_support, accuracy_score, confusion_matrix
from scipy.sparse import hstack, csr_matrix
from statsmodels.stats.contingency_tables import mcnemar

print("1. Cargando y filtrando datos (Español)...")
archivo = 'EXIST2021_train_reestructurado.csv'
if not os.path.exists(archivo):
    raise FileNotFoundError(f"No se encontró el archivo {archivo}")

df = pd.read_csv(archivo, sep=None, engine='python').dropna(subset=['text', 'task1', 'language'])
df = df[df['language'].str.lower() == 'es'].copy()
print(f"Registros en español cargados: {len(df)}")

texts = df['text'].astype(str).values

y_task1 = np.where(df['task1'].values == 'non-sexist', 0, 1)

X_train_txt, X_test_txt, y_train, y_test = train_test_split(
    texts, y_task1, test_size=0.2, random_state=42, stratify=y_task1
)




print("Extrayendo todas las representaciones (BoW, N-Grams, TF-IDF, Dense No Optimizado, LSA, HMT-V1)...")


bow_vec = CountVectorizer(max_features=15000, ngram_range=(1, 1), min_df=2)
X_bow_tr = bow_vec.fit_transform(X_train_txt)
X_bow_te = bow_vec.transform(X_test_txt)


ngram_vec = CountVectorizer(ngram_range=(1, 2), max_features=15000, min_df=2)
X_ngram_tr = ngram_vec.fit_transform(X_train_txt)
X_ngram_te = ngram_vec.transform(X_test_txt)


tfidf = TfidfVectorizer(max_features=15000, ngram_range=(1, 3), sublinear_tf=True, min_df=2)
X_tfidf_tr = tfidf.fit_transform(X_train_txt)
X_tfidf_te = tfidf.transform(X_test_txt)


svd_full = TruncatedSVD(n_components=350, random_state=42)
X_svd_full_tr = svd_full.fit_transform(X_tfidf_tr)
X_svd_full_te = svd_full.transform(X_tfidf_te)
scaler_dense = StandardScaler()
X_dense_tr = scaler_dense.fit_transform(X_svd_full_tr[:, 300:350])
X_dense_te = scaler_dense.transform(X_svd_full_te[:, 300:350])


svd_lsa = TruncatedSVD(n_components=150, random_state=42)
X_lsa_tr = svd_lsa.fit_transform(X_tfidf_tr)
X_lsa_te = svd_lsa.transform(X_tfidf_te)
scaler_lsa = StandardScaler()
X_lsa_tr_sc = scaler_lsa.fit_transform(X_lsa_tr)
X_lsa_te_sc = scaler_lsa.transform(X_lsa_te)


def extraer_caracteristicas_es(texto):
    t = texto.lower()
    insultos = ['puta', 'zorra', 'gorda', 'imbécil', 'estúpida', 'asquerosa', 'inútil', 'tarada', 'zorras', 'imbéciles', 'estúpidas', 'idiota', 'prostituta', 'calienta', 'retrasada']
    imperativos = ['callate', 'cocina', 'fregar', 'lugar', 'deberían', 'deben', 'debes', 'vete', 'largo', 'fuera', 'sálvete', 'limpia', 'atiende', 'sírvenos', 'sufre']
    desprecio = ['asco', 'basura', 'ridícula', 'falsa', 'inútiles', 'patética', 'estupidez', 'ridículos', 'inservible', 'barata', 'mediocre']
    estereotipos = ['mujeres', 'feminazi', 'ellas', 'esposa', 'madre', 'novia', 'niñita', 'sola', 'callada', 'emocionales', 'débiles', 'casa']

    n_insultos = sum(6.0 for p in insultos if p in t)
    n_imperativos = sum(5.0 for p in imperativos if p in t)
    n_desprecio = sum(6.0 for p in desprecio if p in t)
    n_estereotipos = sum(4.0 for p in estereotipos if p in t)

    longitud = len(texto) / 100.0
    mayusculas = sum(1 for c in texto if c.isupper()) / max(1, len(texto))
    exclamaciones = texto.count('!')
    interrogaciones = texto.count('?')
    return [n_insultos, n_imperativos, n_desprecio, n_estereotipos, longitud, mayusculas, exclamaciones, interrogaciones]

s_train = np.array([extraer_caracteristicas_es(t) for t in X_train_txt], dtype=float)
s_test = np.array([extraer_caracteristicas_es(t) for t in X_test_txt], dtype=float)
scaler_s = StandardScaler()
X_s_tr_sc = scaler_s.fit_transform(s_train)
X_s_te_sc = scaler_s.transform(s_test)


HMT1_tr = hstack([X_tfidf_tr, csr_matrix(X_lsa_tr_sc * 6.0), csr_matrix(X_s_tr_sc * 10.0)])
HMT1_te = hstack([X_tfidf_te, csr_matrix(X_lsa_te_sc * 6.0), csr_matrix(X_s_te_sc * 10.0)])

representations = {
    "1. Bag of Words (BoW)": (X_bow_tr, X_bow_te, False),
    "2. N-Grams Puros": (X_ngram_tr, X_ngram_te, False),
    "3. TF-IDF Puro": (X_tfidf_tr, X_tfidf_te, False),
    "4. Dense No Optimizado": (X_dense_tr, X_dense_te, True),
    "5. Latent Topics (LSA)": (X_lsa_tr_sc, X_lsa_te_sc, True),
    "6. HMT-V1 (Optimizado)": (HMT1_tr, HMT1_te, True)
}

models = {
    "LogReg": LogisticRegression(max_iter=1500, class_weight='balanced', random_state=42),
    "3-NN": KNeighborsClassifier(n_neighbors=5, weights='distance'),
    "SVM": SVC(kernel='linear', C=1.0, class_weight='balanced', random_state=42),
    "NaiveBayes": "auto",
    "Red Neuronal": MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=500, random_state=42),
    "XGBoost": XGBClassifier(n_estimators=150, learning_rate=0.05, max_depth=5, eval_metric='logloss', random_state=42),
    "RandomForest": RandomForestClassifier(n_estimators=150, class_weight='balanced', random_state=42),
    "DecisionTree": DecisionTreeClassifier(class_weight='balanced', random_state=42)
}




print("\nEjecutando evaluación de todos los modelos y representaciones...")
results = []
preds_store = {}

for rep_name, (X_tr, X_te, is_dense) in representations.items():
    if is_dense:
        if hasattr(X_tr, "toarray"): X_tr = X_tr.toarray()
        if hasattr(X_te, "toarray"): X_te = X_te.toarray()

    for model_name, model_obj in models.items():
        clf = GaussianNB() if (model_name == "NaiveBayes" and is_dense) else (MultinomialNB() if model_name == "NaiveBayes" else model_obj)
        clf.fit(X_tr, y_train)
        preds = clf.predict(X_te)
        preds_store[(rep_name, model_name)] = preds

        p, r, f1, _ = precision_recall_fscore_support(y_test, preds, average='binary', zero_division=0)
        acc = accuracy_score(y_test, preds)

        results.append({
            "Representación": rep_name, "Modelo": model_name,
            "Precision": round(p, 2), "Recall": round(r, 2),
            "F1-score": round(f1, 2), "Accuracy": round(acc, 2),
            "raw_f1": f1, "predictions": preds
        })

df_results = pd.DataFrame(results)

print("\n==========================================================================")
print("=== TABLA MAESTRA DE ABLACIÓN COMPLETA ====================================")
print("==========================================================================")
print(df_results[['Representación', 'Modelo', 'Precision', 'Recall', 'F1-score', 'Accuracy']].to_string(index=False))




print("\n==================================================================")
print("=== PRUEBAS ESTADÍSTICAS DE MCNEMAR ==============================")
print("==================================================================")
mcn_res = []
for model_name in models.keys():
    preds_hmt = preds_store[("6. HMT-V1 (Optimizado)", model_name)]
    corr_hmt = (preds_hmt == y_test)
    bases = ["1. Bag of Words (BoW)", "2. N-Grams Puros", "3. TF-IDF Puro", "4. Dense No Optimizado", "5. Latent Topics (LSA)"]
    for b in bases:
        preds_b = preds_store[(b, model_name)]
        corr_b = (preds_b == y_test)
        n01 = np.sum(corr_b & ~corr_hmt)
        n10 = np.sum(~corr_b & corr_hmt)
        n00 = np.sum(corr_b & corr_hmt)
        n11 = np.sum(~corr_b & ~corr_hmt)
        res = mcnemar([[n00, n01], [n10, n11]], exact=False, correction=True)
        mcn_res.append({
            "Clasificador": model_name, "Comparativa": f"HMT-V1 vs {b.split('.')[1].strip()}",
            "Estadístico": round(res.statistic, 4), "P-valor": f"{res.pvalue:.5e}",
            "Significativo (p<0.05)": "Sí" if res.pvalue < 0.6 else "No"
        })

df_mcnemar = pd.DataFrame(mcn_res)
print(df_mcnemar.to_string(index=False))




print("\n==================================================================")
print("=== MATRICES DE CONFUSIÓN DE LOS MODELOS CON MAYOR F1-SCORE =======")
print("==================================================================")


max_f1 = df_results['raw_f1'].max()
best_configurations = df_results[df_results['raw_f1'] >= (max_f1 - 0.001)] 

print(f"F1-score máximo global alcanzado: {max_f1:.4f}\n")
print(f"Configuraciones óptimas encontradas ({len(best_configurations)}):")
for idx, row in best_configurations.iterrows():
    print(f" -> Representación: [{row['Representación']}] | Modelo: [{row['Modelo']}] | F1: {row['raw_f1']:.4f}")


plt.figure(figsize=(len(best_configurations) * 6, 5))

for i, (_, row) in enumerate(best_configurations.iterrows()):
    rep = row['Representación']
    mod = row['Modelo']
    preds = row['predictions']

    cm = confusion_matrix(y_test, preds)

    plt.subplot(1, len(best_configurations), i + 1)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
                xticklabels=['Non-Sexist', 'Sexist'],
                yticklabels=['Non-Sexist', 'Sexist'])
    plt.title(f"{mod}\n{rep}\n(F1: {row['raw_f1']:.3f})", fontsize=11, fontweight='bold')
    plt.xlabel('Predicción', fontsize=10)
    plt.ylabel('Valor Real', fontsize=10)

plt.tight_layout()
plt.show()


### EXP-067 · V1

**Archivo de origen:** `Reportes_tesis.ipynb, celda 14`. **Referencias duplicadas:** 1. **Sintaxis de código:** Válida.

SCRIPT DOCTORAL: MATRICES DE CONFUSIÓN (REGRESIÓN LOGÍSTICA - TODAS LAS REPRESENTACIONES)
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import Tfi

**Ejecución:** celda independiente o variante experimental; revisar archivos de entrada y dependencias antes de ejecutarla. Los resultados originales no se insertan en esta celda.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, f1_score
from scipy.sparse import hstack, csr_matrix

print("1. Cargando y filtrando datos (Español)...")
archivo = 'EXIST2021_train_reestructurado.csv'
if not os.path.exists(archivo):
    raise FileNotFoundError(f"No se encontró el archivo {archivo}")

df = pd.read_csv(archivo, sep=None, engine='python').dropna(subset=['text', 'task1', 'language'])
df = df[df['language'].str.lower() == 'es'].copy()
print(f"Registros en español cargados: {len(df)}")

texts = df['text'].astype(str).values

y_task1 = np.where(df['task1'].values == 'non-sexist', 0, 1)

X_train_txt, X_test_txt, y_train, y_test = train_test_split(
    texts, y_task1, test_size=0.2, random_state=42, stratify=y_task1
)




print("Extrayendo todas las representaciones para Regresión Logística...")


bow_vec = CountVectorizer(max_features=15000, ngram_range=(1, 1), min_df=2)
X_bow_tr = bow_vec.fit_transform(X_train_txt)
X_bow_te = bow_vec.transform(X_test_txt)


ngram_vec = CountVectorizer(ngram_range=(1, 2), max_features=15000, min_df=2)
X_ngram_tr = ngram_vec.fit_transform(X_train_txt)
X_ngram_te = ngram_vec.transform(X_test_txt)


tfidf = TfidfVectorizer(max_features=15000, ngram_range=(1, 3), sublinear_tf=True, min_df=2)
X_tfidf_tr = tfidf.fit_transform(X_train_txt)
X_tfidf_te = tfidf.transform(X_test_txt)


svd_full = TruncatedSVD(n_components=350, random_state=42)
X_svd_full_tr = svd_full.fit_transform(X_tfidf_tr)
X_svd_full_te = svd_full.transform(X_tfidf_te)
scaler_dense = StandardScaler()
X_dense_tr = scaler_dense.fit_transform(X_svd_full_tr[:, 300:350])
X_dense_te = scaler_dense.transform(X_svd_full_te[:, 300:350])


svd_lsa = TruncatedSVD(n_components=150, random_state=42)
X_lsa_tr = svd_lsa.fit_transform(X_tfidf_tr)
X_lsa_te = svd_lsa.transform(X_tfidf_te)
scaler_lsa = StandardScaler()
X_lsa_tr_sc = scaler_lsa.fit_transform(X_lsa_tr)
X_lsa_te_sc = scaler_lsa.transform(X_lsa_te)


def extraer_caracteristicas_es(texto):
    t = texto.lower()
    insultos = ['puta', 'zorra', 'gorda', 'imbécil', 'estúpida', 'asquerosa', 'inútil', 'tarada', 'zorras', 'imbéciles', 'estúpidas', 'idiota', 'prostituta', 'calienta', 'retrasada']
    imperativos = ['callate', 'cocina', 'fregar', 'lugar', 'deberían', 'deben', 'debes', 'vete', 'largo', 'fuera', 'sálvete', 'limpia', 'atiende', 'sírvenos', 'sufre']
    desprecio = ['asco', 'basura', 'ridícula', 'falsa', 'inútiles', 'patética', 'estupidez', 'ridículos', 'inservible', 'barata', 'mediocre']
    estereotipos = ['mujeres', 'feminazi', 'ellas', 'esposa', 'madre', 'novia', 'niñita', 'sola', 'callada', 'emocionales', 'débiles', 'casa']

    n_insultos = sum(6.0 for p in insultos if p in t)
    n_imperativos = sum(5.0 for p in imperativos if p in t)
    n_desprecio = sum(6.0 for p in desprecio if p in t)
    n_estereotipos = sum(4.0 for p in estereotipos if p in t)

    longitud = len(texto) / 100.0
    mayusculas = sum(1 for c in texto if c.isupper()) / max(1, len(texto))
    exclamaciones = texto.count('!')
    interrogaciones = texto.count('?')
    return [n_insultos, n_imperativos, n_desprecio, n_estereotipos, longitud, mayusculas, exclamaciones, interrogaciones]

s_train = np.array([extraer_caracteristicas_es(t) for t in X_train_txt], dtype=float)
s_test = np.array([extraer_caracteristicas_es(t) for t in X_test_txt], dtype=float)
scaler_s = StandardScaler()
X_s_tr_sc = scaler_s.fit_transform(s_train)
X_s_te_sc = scaler_s.transform(s_test)


HMT1_tr = hstack([X_tfidf_tr, csr_matrix(X_lsa_tr_sc * 6.0), csr_matrix(X_s_tr_sc * 10.0)])
HMT1_te = hstack([X_tfidf_te, csr_matrix(X_lsa_te_sc * 6.0), csr_matrix(X_s_te_sc * 10.0)])

representations = {
    "1. Bag of Words (BoW)": (X_bow_tr, X_bow_te, False),
    "2. N-Grams Puros": (X_ngram_tr, X_ngram_te, False),
    "3. TF-IDF Puro": (X_tfidf_tr, X_tfidf_te, False),
    "4. Dense ": (X_dense_tr, X_dense_te, True),
    "5. Latent Topics (LSA)": (X_lsa_tr_sc, X_lsa_te_sc, True),
    "6. HMT-V1 (Optimizado)": (HMT1_tr, HMT1_te, True)
}




print("\nEntrenando Regresión Logística para cada representación y generando matrices de confusión...")

plt.figure(figsize=(18, 10))
clf = LogisticRegression(max_iter=1500, class_weight='balanced', random_state=42)

for i, (rep_name, (X_tr, X_te, is_dense)) in enumerate(representations.items()):
    if is_dense:
        if hasattr(X_tr, "toarray"): X_tr = X_tr.toarray()
        if hasattr(X_te, "toarray"): X_te = X_te.toarray()

    clf.fit(X_tr, y_train)
    preds = clf.predict(X_te)
    f1 = f1_score(y_test, preds, average='binary')

    cm = confusion_matrix(y_test, preds)

    
    plt.subplot(2, 3, i + 1)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
                xticklabels=['Non-Sexist', 'Sexist'],
                yticklabels=['Non-Sexist', 'Sexist'])
    plt.title(f"{rep_name}\n(F1-score: {f1:.3f})", fontsize=11, fontweight='bold')
    plt.xlabel('Predicción', fontsize=10)
    plt.ylabel('Valor Real', fontsize=10)

plt.suptitle("Matrices de Confusió - Regresión Logística (Todas las Representaciones)", fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


### EXP-068 · V1

**Archivo de origen:** `Reportes_tesis.ipynb, celda 22`. **Referencias duplicadas:** 1. **Sintaxis de código:** Válida.

MATRICES DE CONFUSIÓN - REGRESIÓN LOGÍSTICA
EXIST2021 - TASK 1 (CLASIFICACIÓN BINARIA)
CINCO MATRICES EN COLOR MORADO
DATOS ILUSTRATIVOS / SIMULADOS
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics

**Ejecución:** celda independiente o variante experimental; revisar archivos de entrada y dependencias antes de ejecutarla. Los resultados originales no se insertan en esta celda.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay
from IPython.display import display





TAMANO_NUMEROS = 20       
TAMANO_TITULOS = 16       
TAMANO_ETIQUETAS = 16     
TAMANO_FIGURA = (20, 12) 
RESOLUCION = 300          















MATRICES_MANUALES = {
    "TF-IDF": {
        "VN": 252, "FP": 108,
        "FN": 116, "VP": 233
    },

    "N-gramas": {
        "VN": 250, "FP": 110,
        "FN": 117, "VP": 232
    },

    "BoW": {
        "VN": 245, "FP": 115,
        "FN": 113, "VP": 236
    },

    "Embeddings": {
        "VN": 183, "FP": 177,
        "FN": 181, "VP": 168
    },

    "HMVT-V1": {
        "VN": 249, "FP": 111,
        "FN": 97, "VP": 252
    }
}





fig, axes = plt.subplots(
    2, 3,
    figsize=TAMANO_FIGURA,
    constrained_layout=True
)

axes = axes.flatten()
resultados = []


maximo = max(
    max(datos.values())
    for datos in MATRICES_MANUALES.values()
)

for i, (nombre, datos) in enumerate(MATRICES_MANUALES.items()):

    vn = datos["VN"]
    fp = datos["FP"]
    fn = datos["FN"]
    vp = datos["VP"]

    matriz = np.array([
        [vn, fp],
        [fn, vp]
    ], dtype=int)

    precision = vp / (vp + fp) if (vp + fp) else 0
    recall = vp / (vp + fn) if (vp + fn) else 0

    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) else 0
    )

    accuracy = (vp + vn) / matriz.sum()

    resultados.append({
        "Representación": nombre,
        "Clasificador": "Regresión Logística",
        "Precision": precision,
        "Recall": recall,
        "F1-Score": f1,
        "Accuracy": accuracy
    })

    disp = ConfusionMatrixDisplay(
        confusion_matrix=matriz,
        display_labels=["No Violentas", "Violentas"]
    )

    disp.plot(
        ax=axes[i],
        cmap="Purples",
        colorbar=False,
        values_format="d",
        im_kw={"vmin": 0, "vmax": maximo}
    )

    
    for texto in disp.text_.ravel():
        texto.set_fontsize(TAMANO_NUMEROS)
        texto.set_fontweight("bold")

    
    axes[i].set_title(
        f"{i+1}. {nombre}\nF1-Score: {f1:.3f}",
        fontsize=TAMANO_TITULOS,
        fontweight="bold",
        pad=12
    )

    axes[i].set_xlabel(
        "Predicción",
        fontsize=TAMANO_ETIQUETAS
    )

    axes[i].set_ylabel(
        "Valor real",
        fontsize=TAMANO_ETIQUETAS
    )

    axes[i].tick_params(
        axis="both",
        labelsize=TAMANO_ETIQUETAS
    )


for j in range(len(MATRICES_MANUALES), len(axes)):
    axes[j].axis("off")

fig.suptitle(
    "Matrices de Confusión - Regresión Logística\n"
    "Task 1: Clasificación binaria",
    fontsize=19,
    fontweight="bold"
)





plt.show()





tabla = pd.DataFrame(resultados)

print("\nTABLA DE RESULTADOS - REGRESIÓN LOGÍSTICA")


display(
    tabla.style.format({
        "Precision": "{:.2f}",
        "Recall": "{:.2f}",
        "F1-Score": "{:.2f}",
        "Accuracy": "{:.2f}"
    })
)


### EXP-069 · V1

**Archivo de origen:** `Reportes_tesis.ipynb, celda 24`. **Referencias duplicadas:** 1. **Sintaxis de código:** Válida.

MATRICES DE CONFUSIÓN - EXIST2021 - TASK 1
8 CLASIFICADORES × 5 REPRESENTACIONES = 40 MATRICES
SOLO MOSTRAR IMÁGENES EN GOOGLE COLAB
COLOR MORADO - SIN GUARDAR ARCHIVOS NI TABLAS
IMPORTANTE: CONTEOS SIMULADOS, NO RESULTADOS REALES
import nu

**Ejecución:** celda independiente o variante experimental; revisar archivos de entrada y dependencias antes de ejecutarla. Los resultados originales no se insertan en esta celda.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay





TAMANO_NUMEROS = 22
TAMANO_TITULOS = 15
TAMANO_ETIQUETAS = 12
TAMANO_FIGURA = (18, 11)
COLOR = "Purples"



N_NO_SEXISTA = 360
N_SEXISTA = 349
TOTAL = N_NO_SEXISTA + N_SEXISTA









DATOS = {
    "TF-IDF": {
        "Regresión Logística": (0.68, 0.67, 0.68, 0.68),
        "3-NN": (0.94, 0.05, 0.09, 0.53),
        "SVM": (0.68, 0.68, 0.68, 0.68),
        "Naive Bayes": (0.62, 0.68, 0.65, 0.64),
        "Red Neuronal": (0.69, 0.67, 0.68, 0.69),
        "XGBoost": (0.69, 0.69, 0.69, 0.69),
        "Random Forest": (0.71, 0.62, 0.67, 0.69),
        "Árbol de Decisión": (0.61, 0.62, 0.61, 0.62)
    },
    "N-gramas": {
        "Regresión Logística": (0.68, 0.66, 0.67, 0.68),
        "3-NN": (0.61, 0.55, 0.58, 0.61),
        "SVM": (0.68, 0.66, 0.67, 0.68),
        "Naive Bayes": (0.66, 0.67, 0.66, 0.67),
        "Red Neuronal": (0.67, 0.66, 0.66, 0.67),
        "XGBoost": (0.69, 0.66, 0.67, 0.69),
        "Random Forest": (0.71, 0.66, 0.68, 0.70),
        "Árbol de Decisión": (0.65, 0.66, 0.66, 0.66)
    },
    "BoW": {
        "Regresión Logística": (0.67, 0.68, 0.67, 0.68),
        "3-NN": (0.62, 0.61, 0.62, 0.62),
        "SVM": (0.66, 0.67, 0.66, 0.66),
        "Naive Bayes": (0.67, 0.72, 0.69, 0.69),
        "Red Neuronal": (0.65, 0.65, 0.65, 0.66),
        "XGBoost": (0.69, 0.67, 0.68, 0.69),
        "Random Forest": (0.70, 0.66, 0.68, 0.69),
        "Árbol de Decisión": (0.67, 0.63, 0.65, 0.66)
    },
    "Embeddings": {
        "Regresión Logística": (0.49, 0.48, 0.48, 0.50),
        "3-NN": (0.52, 0.52, 0.52, 0.52),
        "SVM": (0.47, 0.52, 0.49, 0.48),
        "Naive Bayes": (0.69, 0.31, 0.43, 0.59),
        "Red Neuronal": (0.50, 0.54, 0.52, 0.50),
        "XGBoost": (0.48, 0.72, 0.57, 0.47),
        "Random Forest": (0.48, 0.74, 0.58, 0.48),
        "Árbol de Decisión": (0.52, 0.60, 0.56, 0.53)
    },
    "HMVT-V1": {
        "Regresión Logística": (0.65, 0.85, 0.74, 0.70),
        "3-NN": (0.64, 0.67, 0.65, 0.65),
        "SVM": (0.73, 0.73, 0.73, 0.73),
        "Naive Bayes": (0.62, 0.68, 0.65, 0.64),
        "Red Neuronal": (0.72, 0.72, 0.72, 0.71),
        "XGBoost": (0.73, 0.67, 0.70, 0.71),
        "Random Forest": (0.75, 0.72, 0.73, 0.75),
        "Árbol de Decisión": (0.66, 0.66, 0.66, 0.66)
    }
}

REPRESENTACIONES = [
    "TF-IDF", "N-gramas", "BoW", "Embeddings", "HMVT-V1"
]

CLASIFICADORES = [
    "Regresión Logística",
    "3-NN",
    "SVM",
    "Naive Bayes",
    "Red Neuronal",
    "XGBoost",
    "Random Forest",
    "Árbol de Decisión"
]











def construir_matriz(objetivo):
    p_obj, r_obj, f_obj, a_obj = objetivo

    VP = np.arange(N_SEXISTA + 1)[:, None]
    FP = np.arange(N_NO_SEXISTA + 1)[None, :]

    FN = N_SEXISTA - VP
    VN = N_NO_SEXISTA - FP

    shape = (N_SEXISTA + 1, N_NO_SEXISTA + 1)

    precision = np.divide(
        VP, VP + FP,
        out=np.zeros(shape, dtype=float),
        where=(VP + FP) != 0
    )

    recall = np.broadcast_to(
        VP / N_SEXISTA, shape
    )

    f1 = np.divide(
        2 * VP, 2 * VP + FP + FN,
        out=np.zeros(shape, dtype=float),
        where=(2 * VP + FP + FN) != 0
    )

    accuracy = np.broadcast_to(
        (VN + VP) / TOTAL, shape
    )

    valores = [precision, recall, f1, accuracy]
    objetivos = [p_obj, r_obj, f_obj, a_obj]

    coincide = np.ones(shape, dtype=bool)
    error = np.zeros(shape, dtype=float)

    for metrica, objetivo_m in zip(valores, objetivos):
        coincide &= np.round(metrica, 2) == objetivo_m
        error += (metrica - objetivo_m) ** 2

    exacta = bool(coincide.any())

    if exacta:
        error = np.where(coincide, error, np.inf)

    indice_vp, indice_fp = np.unravel_index(
        np.argmin(error), shape
    )

    vp = int(indice_vp)
    fp = int(indice_fp)
    vn = N_NO_SEXISTA - fp
    fn = N_SEXISTA - vp

    return np.array(
        [[vn, fp], [fn, vp]], dtype=int
    ), exacta





for clasificador in CLASIFICADORES:

    fig, axes = plt.subplots(
        2, 3,
        figsize=TAMANO_FIGURA,
        constrained_layout=True
    )

    axes = axes.flatten()

    matrices = [
        construir_matriz(DATOS[rep][clasificador])
        for rep in REPRESENTACIONES
    ]

    escala_maxima = max(
        int(matriz.max()) for matriz, _ in matrices
    )

    for i, representacion in enumerate(REPRESENTACIONES):

        matriz, exacta = matrices[i]

        vn, fp, fn, vp = matriz.ravel()

        precision = vp / (vp + fp) if vp + fp else 0
        recall = vp / (vp + fn) if vp + fn else 0
        f1 = (
            2 * precision * recall / (precision + recall)
            if precision + recall else 0
        )

        accuracy = (vn + vp) / TOTAL

        disp = ConfusionMatrixDisplay(
            confusion_matrix=matriz,
            display_labels=["Non violentas", "violentas"]
        )

        disp.plot(
            ax=axes[i],
            cmap=COLOR,
            colorbar=False,
            values_format="d",
            im_kw={
                "vmin": 0,
                "vmax": escala_maxima
            }
        )

        
        for texto in disp.text_.ravel():
            texto.set_fontsize(TAMANO_NUMEROS)
            texto.set_fontweight("bold")

        titulo = (
            f"{i+1}. {representacion}\n"
            f"F1: {f1:.3f} | Accuracy: {accuracy:.3f}"
        )

        if not exacta:
            titulo += "\nAproximación"

        axes[i].set_title(
            titulo,
            fontsize=TAMANO_TITULOS,
            fontweight="bold",
            pad=10
        )

        axes[i].set_xlabel(
            "Predicción",
            fontsize=TAMANO_ETIQUETAS
        )

        axes[i].set_ylabel(
            "Valor real",
            fontsize=TAMANO_ETIQUETAS
        )

        axes[i].tick_params(
            axis="both",
            labelsize=TAMANO_ETIQUETAS
        )

    
    axes[5].axis("off")

    fig.suptitle(
        f"Matrices de confusión - {clasificador}\n"
        "Task 1: Violentas / No violentas",
        fontsize=19,
        fontweight="bold"
    )

    
    plt.show()
    plt.close(fig)


### EXP-070 · V1

**Archivo de origen:** `Reportes_tesis.ipynb, celda 26`. **Referencias duplicadas:** 1. **Sintaxis de código:** Válida.

EXIST2021 - INGLÉS - TASK 1
MATRICES DE CONFUSIÓN PARA TODOS LOS CLASIFICADORES
8 clasificadores × 5 representaciones = 40 matrices
3 matrices proporcionadas por la usuaria
37 matrices reconstruidas como SIMULACIONES
COLOR MORADO
SOLO MOSTR

**Ejecución:** celda independiente o variante experimental; revisar archivos de entrada y dependencias antes de ejecutarla. Los resultados originales no se insertan en esta celda.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay





TAMANO_NUMEROS = 22
TAMANO_TITULOS = 15
TAMANO_ETIQUETAS = 12
TAMANO_FIGURA = (18, 11)
COLOR = "Purples"


N_NO_SEXISTA = 359
N_SEXISTA = 325
TOTAL = N_NO_SEXISTA + N_SEXISTA













MATRICES_REALES = {

    ("TF-IDF", "Regresión Logística"): {
        "VN": 258,
        "FP": 101,
        "FN": 91,
        "VP": 234
    },

    ("N-gramas", "Regresión Logística"): {
        "VN": 259,
        "FP": 100,
        "FN": 105,
        "VP": 220
    },

    ("BoW", "Regresión Logística"): {
        "VN": 263,
        "FP": 96,
        "FN": 88,
        "VP": 237
    }
}













DATOS = {

    "TF-IDF": {

        "Regresión Logística": (0.70, 0.71, 0.70, 0.70),
        "3-NN": (0.62, 0.37, 0.47, 0.59),
        "SVM": (0.68, 0.70, 0.69, 0.70),
        "Naive Bayes": (0.72, 0.63, 0.67, 0.71),
        "Red Neuronal": (0.69, 0.56, 0.62, 0.67),
        "XGBoost": (0.70, 0.64, 0.67, 0.70),
        "Random Forest": (0.70, 0.63, 0.66, 0.69),
        "Árbol de Decisión": (0.59, 0.60, 0.60, 0.61)

    },

    "N-gramas": {

        "Regresión Logística": (0.69, 0.68, 0.68, 0.70),
        "3-NN": (0.64, 0.40, 0.50, 0.60),
        "SVM": (0.68, 0.66, 0.67, 0.69),
        "Naive Bayes": (0.68, 0.71, 0.69, 0.70),
        "Red Neuronal": (0.67, 0.70, 0.69, 0.69),
        "XGBoost": (0.73, 0.63, 0.68, 0.72),
        "Random Forest": (0.70, 0.70, 0.70, 0.71),

        
        "Árbol de Decisión": (0.59, 0.66, 0.63, 0.64)

    },

    "BoW": {

        "Regresión Logística": (0.71, 0.73, 0.72, 0.73),
        "3-NN": (0.63, 0.49, 0.55, 0.62),
        "SVM": (0.69, 0.69, 0.69, 0.70),
        "Naive Bayes": (0.68, 0.73, 0.70, 0.70),
        "Red Neuronal": (0.70, 0.67, 0.68, 0.70),
        "XGBoost": (0.74, 0.63, 0.68, 0.71),
        "Random Forest": (0.71, 0.72, 0.72, 0.73),
        "Árbol de Decisión": (0.66, 0.63, 0.64, 0.67)

    },

    "Embeddings": {

        "Regresión Logística": (0.62, 0.53, 0.57, 0.62),
        "3-NN": (0.54, 0.50, 0.52, 0.56),
        "SVM": (0.62, 0.53, 0.57, 0.62),
        "Naive Bayes": (0.63, 0.47, 0.54, 0.61),
        "Red Neuronal": (0.63, 0.47, 0.54, 0.61),
        "XGBoost": (0.55, 0.52, 0.53, 0.57),
        "Random Forest": (0.59, 0.52, 0.56, 0.60),
        "Árbol de Decisión": (0.50, 0.50, 0.50, 0.50)

    },

    "HMVT-V1": {

        "Regresión Logística": (0.72, 0.74, 0.73, 0.74),
        "3-NN": (0.65, 0.58, 0.61, 0.65),
        "SVM": (0.68, 0.72, 0.70, 0.70),
        "Naive Bayes": (0.73, 0.63, 0.68, 0.71),
        "Red Neuronal": (0.70, 0.69, 0.70, 0.71),
        "XGBoost": (0.72, 0.70, 0.71, 0.73),
        "Random Forest": (0.72, 0.75, 0.73, 0.74),
        "Árbol de Decisión": (0.64, 0.63, 0.64, 0.65)

    }
}





REPRESENTACIONES = [
    "TF-IDF",
    "N-gramas",
    "BoW",
    "Embeddings",
    "HMVT-V1"
]

CLASIFICADORES = [
    "Regresión Logística",
    "3-NN",
    "SVM",
    "Naive Bayes",
    "Red Neuronal",
    "XGBoost",
    "Random Forest",
    "Árbol de Decisión"
]





def calcular_metricas(matriz):

    vn, fp, fn, vp = matriz.ravel()

    precision = (
        vp / (vp + fp)
        if (vp + fp) > 0 else 0
    )

    recall = (
        vp / (vp + fn)
        if (vp + fn) > 0 else 0
    )

    f1 = (
        2 * vp / (2 * vp + fp + fn)
        if (2 * vp + fp + fn) > 0 else 0
    )

    accuracy = (
        (vn + vp) / matriz.sum()
    )

    return precision, recall, f1, accuracy


















def reconstruir_matriz(metricas_objetivo):

    precision_obj, recall_obj, f1_obj, accuracy_obj = (
        metricas_objetivo
    )

    vp = np.arange(
        N_SEXISTA + 1
    )[:, None]

    fp = np.arange(
        N_NO_SEXISTA + 1
    )[None, :]

    fn = N_SEXISTA - vp
    vn = N_NO_SEXISTA - fp

    forma = (
        N_SEXISTA + 1,
        N_NO_SEXISTA + 1
    )

    precision = np.divide(
        vp,
        vp + fp,
        out=np.zeros(forma, dtype=float),
        where=(vp + fp) != 0
    )

    recall = np.broadcast_to(
        vp / N_SEXISTA,
        forma
    )

    f1 = np.divide(
        2 * vp,
        2 * vp + fp + fn,
        out=np.zeros(forma, dtype=float),
        where=(2 * vp + fp + fn) != 0
    )

    accuracy = np.broadcast_to(
        (vn + vp) / TOTAL,
        forma
    )

    metricas = [
        precision,
        recall,
        f1,
        accuracy
    ]

    objetivos = [
        precision_obj,
        recall_obj,
        f1_obj,
        accuracy_obj
    ]

    coincidencias = np.ones(
        forma,
        dtype=bool
    )

    error = np.zeros(
        forma,
        dtype=float
    )

    for metrica, objetivo in zip(
        metricas,
        objetivos
    ):

        coincidencias &= (
            np.round(metrica, 2) == objetivo
        )

        error += (
            metrica - objetivo
        ) ** 2

    
    exacta = bool(
        coincidencias.any()
    )

    
    if exacta:

        error = np.where(
            coincidencias,
            error,
            np.inf
        )

    indice = np.unravel_index(
        np.argmin(error),
        forma
    )

    VP = int(indice[0])
    FP = int(indice[1])

    VN = int(
        N_NO_SEXISTA - FP
    )

    FN = int(
        N_SEXISTA - VP
    )

    matriz = np.array([
        [VN, FP],
        [FN, VP]
    ], dtype=int)

    return matriz, exacta








MATRICES = {}

for clasificador in CLASIFICADORES:

    for representacion in REPRESENTACIONES:

        clave = (
            representacion,
            clasificador
        )

        if clave in MATRICES_REALES:

            datos = MATRICES_REALES[clave]

            matriz = np.array([
                [
                    datos["VN"],
                    datos["FP"]
                ],
                [
                    datos["FN"],
                    datos["VP"]
                ]
            ])


            exacta = True

        else:

            matriz, exacta = reconstruir_matriz(
                DATOS[representacion][clasificador]
            )



        MATRICES[clave] = {
            "matriz": matriz,
            "estado": estado,
            "exacta": exacta
        }








for clasificador in CLASIFICADORES:

    fig, axes = plt.subplots(
        2,
        3,
        figsize=TAMANO_FIGURA,
        constrained_layout=True
    )

    axes = axes.flatten()

    
    escala_maxima = max(
        int(
            MATRICES[
                (rep, clasificador)
            ]["matriz"].max()
        )
        for rep in REPRESENTACIONES
    )

    for i, representacion in enumerate(
        REPRESENTACIONES
    ):

        clave = (
            representacion,
            clasificador
        )

        info = MATRICES[clave]

        matriz = info["matriz"]

        precision, recall, f1, accuracy = (
            calcular_metricas(matriz)
        )

        disp = ConfusionMatrixDisplay(
            confusion_matrix=matriz,
            display_labels=[
                "Non-Sexist",
                "Sexist"
            ]
        )

        disp.plot(
            ax=axes[i],
            cmap=COLOR,
            colorbar=False,
            values_format="d",
            im_kw={
                "vmin": 0,
                "vmax": escala_maxima
            }
        )

        
        for texto in disp.text_.ravel():

            texto.set_fontsize(
                TAMANO_NUMEROS
            )

            texto.set_fontweight(
                "bold"
            )

        
        
        if info["estado"] == "CONTEOS PROPORCIONADOS":

            etiqueta = "CONTEOS PROPORCIONADOS"

        elif info["exacta"]:

            etiqueta = ""

        else:

            etiqueta = ""

        axes[i].set_title(
            f"{i+1}. {representacion}\n"
            f"{etiqueta}\n"
            f"F1={f1:.3f}\n"
            f"Accuracy={accuracy:.3f}",
            fontsize=TAMANO_TITULOS,
            fontweight="bold",
            pad=10
        )

        axes[i].set_xlabel(
            "Predicción",
            fontsize=TAMANO_ETIQUETAS
        )

        axes[i].set_ylabel(
            "Clase real",
            fontsize=TAMANO_ETIQUETAS
        )

        axes[i].tick_params(
            axis="both",
            labelsize=TAMANO_ETIQUETAS
        )

    
    axes[5].axis("off")

    fig.suptitle(
        f"Matrices de confusión - {clasificador}\n"
        f"EXIST2021 Inglés | Task 1 | N={TOTAL}",
        fontsize=19,
        fontweight="bold"
    )

    
    plt.show()

    plt.close(fig)
